In [2]:

from pathlib import Path

# Main AMC project directory
PROJECT_DIR = Path.cwd().parent

# Raw data directory
RAW_DATA_DIR = PROJECT_DIR / "01_RAW_DATA"

print("Project directory:", PROJECT_DIR)
print("Raw data directory:", RAW_DATA_DIR)
print("Raw data folder exists:", RAW_DATA_DIR.exists())


Project directory: c:\Users\rocks\OneDrive\Desktop\AMC Project Data
Raw data directory: c:\Users\rocks\OneDrive\Desktop\AMC Project Data\01_RAW_DATA
Raw data folder exists: True


In [3]:

from pathlib import Path

amc_folders = [
    "HDFC",
    "ICICI_Prudential",
    "SBI",
    "Nippon_India",
    "Kotak",
]

for amc in amc_folders:
    folder = RAW_DATA_DIR / amc

    print("\n" + "=" * 50)
    print(f"AMC: {amc}")
    print("=" * 50)

    if not folder.exists():
        print("Folder not found:", folder)
        continue

    files = sorted(
        file for file in folder.iterdir()
        if file.is_file()
    )

    print("Total files:", len(files))

    for file in files:
        print(f"{file.name} | Extension: {file.suffix}")



AMC: HDFC
Total files: 33
Monthly HDFC Multi Asset Allocation Fund - 30 September 2026_0.xlsx | Extension: .xlsx
Monthly HDFC Multi Asset Allocation Fund - 31 August 2026 (1).xlsx | Extension: .xlsx
Monthly HDFC Multi-Asset Allocation Fund - 28 February 2026.xlsx | Extension: .xlsx
Monthly HDFC Multi-Asset Allocation Fund - 30 April 2026.xlsx | Extension: .xlsx
Monthly HDFC Multi-Asset Allocation Fund - 30 June 2026.xlsx | Extension: .xlsx
Monthly HDFC Multi-Asset Allocation Fund - 31 December 2025.xlsx | Extension: .xlsx
Monthly HDFC Multi-Asset Allocation Fund - 31 January 2026.xlsx | Extension: .xlsx
Monthly HDFC Multi-Asset Allocation Fund - 31 July 2026.xlsx | Extension: .xlsx
Monthly HDFC Multi-Asset Allocation Fund - 31 March 2026.xlsx | Extension: .xlsx
Monthly HDFC Multi-Asset Allocation Fund - 31 May 2026.xlsx | Extension: .xlsx
Monthly HDFC Multi-Asset Fund - 28 February 2025.xlsx | Extension: .xlsx
Monthly HDFC Multi-Asset Fund - 29 February 2024.xlsx | Extension: .xlsx
Mo

In [4]:

from collections import Counter

for amc in amc_folders:
    folder = RAW_DATA_DIR / amc

    print(f"\n{amc}")

    if not folder.exists():
        print("Folder not found")
        continue

    files = [f for f in folder.iterdir() if f.is_file()]
    extension_counts = Counter(
        f.suffix.lower() or "(no extension)" for f in files
    )

    print("Total files:", len(files))
    print("File types:", dict(extension_counts))



HDFC
Total files: 33
File types: {'.xlsx': 33}

ICICI_Prudential
Total files: 33
File types: {'.xlsx': 33}

SBI
Total files: 33
File types: {'.xlsx': 33}

Nippon_India
Total files: 33
File types: {'.xls': 33}

Kotak
Total files: 33
File types: {'.xls': 23, '.xlsx': 10}


In [5]:

from datetime import datetime

EXPECTED_MONTHS = {
    f"{year}-{month:02d}"
    for year in range(2024, 2027)
    for month in range(1, 13)
    if (year, month) <= (2026, 9)
}

print("Expected reporting months:", len(EXPECTED_MONTHS))
print("Study period: January 2024 to September 2026")


Expected reporting months: 33
Study period: January 2024 to September 2026


In [6]:

hdfc_folder = RAW_DATA_DIR / "HDFC"

for file in sorted(hdfc_folder.iterdir()):
    if file.is_file():
        print(file.name)


Monthly HDFC Multi Asset Allocation Fund - 30 September 2026_0.xlsx
Monthly HDFC Multi Asset Allocation Fund - 31 August 2026 (1).xlsx
Monthly HDFC Multi-Asset Allocation Fund - 28 February 2026.xlsx
Monthly HDFC Multi-Asset Allocation Fund - 30 April 2026.xlsx
Monthly HDFC Multi-Asset Allocation Fund - 30 June 2026.xlsx
Monthly HDFC Multi-Asset Allocation Fund - 31 December 2025.xlsx
Monthly HDFC Multi-Asset Allocation Fund - 31 January 2026.xlsx
Monthly HDFC Multi-Asset Allocation Fund - 31 July 2026.xlsx
Monthly HDFC Multi-Asset Allocation Fund - 31 March 2026.xlsx
Monthly HDFC Multi-Asset Allocation Fund - 31 May 2026.xlsx
Monthly HDFC Multi-Asset Fund - 28 February 2025.xlsx
Monthly HDFC Multi-Asset Fund - 29 February 2024.xlsx
Monthly HDFC Multi-Asset Fund - 30 April 2024.xlsx
Monthly HDFC Multi-Asset Fund - 30 April 2025.xlsx
Monthly HDFC Multi-Asset Fund - 30 June 2024.xlsx
Monthly HDFC Multi-Asset Fund - 30 June 2025.xlsx
Monthly HDFC Multi-Asset Fund - 30 November 2024.xlsx
M

In [7]:

from pathlib import Path
import re
from collections import Counter

PROJECT_DIR = Path.cwd().parent
RAW_DATA_DIR = PROJECT_DIR / "01_RAW_DATA"

amc_folders = [
    "HDFC",
    "ICICI_Prudential",
    "SBI",
    "Nippon_India",
    "Kotak",
]

expected_months = {
    (year, month)
    for year in range(2024, 2027)
    for month in range(1, 13)
    if (year, month) <= (2026, 9)
}

for amc in amc_folders:
    folder = RAW_DATA_DIR / amc
    print(f"\n{'=' * 55}")
    print(f"AMC: {amc}")

    if not folder.exists():
        print("Folder not found:", folder)
        continue

    files = [
        f for f in folder.iterdir()
        if f.is_file() and not f.name.startswith("~$")
    ]

    found_months = set()

    for file in files:
        # Find month names such as January 2024 or Sep 2026.
        match = re.search(
            r"(January|February|March|April|May|June|July|"
            r"August|September|October|November|December|"
            r"Jan|Feb|Mar|Apr|Jun|Jul|Aug|Sep|Oct|Nov|Dec)"
            r"[\s_-]+(\d{4})",
            file.name,
            re.IGNORECASE,
        )

        if match:
            month_text, year_text = match.groups()
            from datetime import datetime
            month_num = datetime.strptime(
                month_text[:3].title(), "%b"
            ).month
            found_months.add((int(year_text), month_num))

        else:
            # Also recognise filenames beginning with YYYY-MM.
            match = re.search(r"\b(2024|2025|2026)[-_](0?[1-9]|1[0-2])\b",
                              file.name)
            if match:
                found_months.add(
                    (int(match.group(1)), int(match.group(2)))
                )

    missing = sorted(expected_months - found_months)
    print("Files found:", len(files))
    print("Distinct months identified:", len(found_months))
    print("Missing months:", [
        f"{year}-{month:02d}" for year, month in missing
    ])



AMC: HDFC
Files found: 33
Distinct months identified: 33
Missing months: []

AMC: ICICI_Prudential
Files found: 33
Distinct months identified: 33
Missing months: []

AMC: SBI
Files found: 33
Distinct months identified: 33
Missing months: []

AMC: Nippon_India
Files found: 33
Distinct months identified: 3
Missing months: ['2024-02', '2024-03', '2024-05', '2024-06', '2024-07', '2024-08', '2024-09', '2024-10', '2024-12', '2025-01', '2025-02', '2025-03', '2025-04', '2025-05', '2025-06', '2025-07', '2025-08', '2025-09', '2025-10', '2025-11', '2025-12', '2026-01', '2026-02', '2026-03', '2026-04', '2026-05', '2026-06', '2026-07', '2026-08', '2026-09']

AMC: Kotak
Files found: 33
Distinct months identified: 0
Missing months: ['2024-01', '2024-02', '2024-03', '2024-04', '2024-05', '2024-06', '2024-07', '2024-08', '2024-09', '2024-10', '2024-11', '2024-12', '2025-01', '2025-02', '2025-03', '2025-04', '2025-05', '2025-06', '2025-07', '2025-08', '2025-09', '2025-10', '2025-11', '2025-12', '2026-0

In [8]:

# Import Path so we can work with folders and file paths.
from pathlib import Path

# Locate the raw-data directory inside the AMC project.
PROJECT_DIR = Path.cwd().parent
RAW_DATA_DIR = PROJECT_DIR / "01_RAW_DATA"

# Inspect only the three AMC folders that need further checking.
amcs_to_inspect = ["SBI", "Nippon_India", "Kotak"]

for amc in amcs_to_inspect:
    # Build the path to the current AMC's folder.
    folder = RAW_DATA_DIR / amc

    print("\n" + "=" * 60)
    print(f"AMC: {amc}")
    print("=" * 60)

    # Check whether the expected folder exists.
    if not folder.exists():
        print(f"Folder not found: {folder}")
        continue

    # List files, excluding temporary Excel lock files.
    files = sorted(
        file for file in folder.iterdir()
        if file.is_file() and not file.name.startswith("~$")
    )

    print(f"Total files: {len(files)}")
    print("\nFirst 8 filenames:")

    # Display up to eight filenames to inspect their patterns.
    for file in files[:8]:
        print(file.name)



AMC: SBI
Total files: 33

First 8 filenames:
SBI-MULTI-ASSET-ALLOCATION-FUND---MARCH-2024.xlsx
SBI-MULTI-ASSET-ALLOCATION-FUND-feb-2024.xlsx
SBI-MULTI-ASSET-ALLOCATION-FUND-jan-2024.xlsx
SBI-MULTI-ASSET-ALLOCATION-FUND-MONTHLY-PORTFOLIO---APRIL-2024.xlsx
SBI-MULTI-ASSET-ALLOCATION-FUND-MONTHLY-PORTFOLIO---APRIL-2025.xlsx
SBI-MULTI-ASSET-ALLOCATION-FUND-MONTHLY-PORTFOLIO---APRIL-2026.xlsx
SBI-MULTI-ASSET-ALLOCATION-FUND-MONTHLY-PORTFOLIO---AUGUST-2024.xlsx
SBI-MULTI-ASSET-ALLOCATION-FUND-MONTHLY-PORTFOLIO---AUGUST-2025.xlsx

AMC: Nippon_India
Total files: 33

First 8 filenames:
MONTHLY-PORTFOLIO-JAN-2024.xls
MONTHLY-PORTFOLIO-REPORT-FEB-24.xls
NIMF-MONTHLY-PORTFOLIO-28-Feb-25.xls
NIMF-MONTHLY-PORTFOLIO-28-Feb-26.xls
NIMF-MONTHLY-PORTFOLIO-30-April-25.xls
NIMF-MONTHLY-PORTFOLIO-30-April-26.xls
NIMF-MONTHLY-PORTFOLIO-30-Jun-26.xls
NIMF-MONTHLY-PORTFOLIO-30-June-25.xls

AMC: Kotak
Total files: 33

First 8 filenames:
ConsolidatedSebiPortfolioApril2024.xls
ConsolidatedSEBIPortfolioApril2025

In [9]:

from pathlib import Path
from datetime import datetime
import re

# Locate the AMC project's raw-data folder.
PROJECT_DIR = Path.cwd().parent
RAW_DATA_DIR = PROJECT_DIR / "01_RAW_DATA"

# List the five AMC folders to inspect.
amc_folders = [
    "HDFC",
    "ICICI_Prudential",
    "SBI",
    "Nippon_India",
    "Kotak",
]

# Generate all 33 expected months from January 2024 to September 2026.
expected_months = {
    (year, month)
    for year in range(2024, 2027)
    for month in range(1, 13)
    if (year, month) <= (2026, 9)
}

# Match full or abbreviated month names followed by a year.
month_pattern = re.compile(
    r"(January|February|March|April|May|June|July|August|"
    r"September|October|November|December|"
    r"Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)"
    r"[\s_-]*(\d{2,4})",
    re.IGNORECASE,
)

# Check each AMC folder independently.
for amc in amc_folders:
    folder = RAW_DATA_DIR / amc

    print("\n" + "=" * 55)
    print(f"AMC: {amc}")
    print("=" * 55)

    # Stop checking this AMC if its folder cannot be found.
    if not folder.exists():
        print("Folder not found:", folder)
        continue

    # Ignore temporary Excel files created while workbooks are open.
    files = sorted(
        file for file in folder.iterdir()
        if file.is_file() and not file.name.startswith("~$")
    )

    found_months = set()
    unrecognised_files = []

    # Extract month and year information from each filename.
    for file in files:
        filename = file.stem
        match = month_pattern.search(filename)

        if match:
            month_text, year_text = match.groups()

            # Convert the month name to a number from 1 to 12.
            month_number = datetime.strptime(
                month_text[:3].title(), "%b"
            ).month

            # Convert two-digit years such as 24, 25, and 26.
            year_number = int(year_text)
            if len(year_text) == 2:
                year_number += 2000

            found_months.add((year_number, month_number))

        else:
            # Record filenames that need manual inspection.
            unrecognised_files.append(file.name)

    # Compare the detected months with the expected study period.
    missing_months = sorted(expected_months - found_months)

    print("Files found:", len(files))
    print("Distinct months identified:", len(found_months))
    print("Missing months:", [
        f"{year}-{month:02d}" for year, month in missing_months
    ])

    # Display filenames the code could not interpret.
    print("Unrecognised filenames:", len(unrecognised_files))
    for filename in unrecognised_files[:5]:
        print(" -", filename)



AMC: HDFC
Files found: 33
Distinct months identified: 33
Missing months: []
Unrecognised filenames: 0

AMC: ICICI_Prudential
Files found: 33
Distinct months identified: 33
Missing months: []
Unrecognised filenames: 0

AMC: SBI
Files found: 33
Distinct months identified: 33
Missing months: []
Unrecognised filenames: 0

AMC: Nippon_India
Files found: 33
Distinct months identified: 33
Missing months: []
Unrecognised filenames: 0

AMC: Kotak
Files found: 33
Distinct months identified: 33
Missing months: []
Unrecognised filenames: 0


In [10]:
# STEP 1: Check the notebook's current working directory.
# This helps us identify the correct paths to the AMC project folders.

from pathlib import Path

CURRENT_DIR = Path.cwd()

print("Current working directory:", CURRENT_DIR)
print("Parent directory:", CURRENT_DIR.parent)
print("Grandparent directory:", CURRENT_DIR.parent.parent)

Current working directory: c:\Users\rocks\OneDrive\Desktop\AMC Project Data\08_NOTEBOOKS
Parent directory: c:\Users\rocks\OneDrive\Desktop\AMC Project Data
Grandparent directory: c:\Users\rocks\OneDrive\Desktop


In [11]:
# STEP 2: Verify that all five AMC raw-data folders exist.
# The AMC folders should be directly inside the 01_RAW_DATA folder.

from pathlib import Path

PROJECT_DIR = Path.cwd().parent
RAW_DATA_DIR = PROJECT_DIR / "01_RAW_DATA"

amc_folders = [
    "HDFC",
    "ICICI_Prudential",
    "SBI",
    "Nippon_India",
    "Kotak"
]

for amc in amc_folders:
    folder_path = RAW_DATA_DIR / amc

    print(f"{amc}: {'Found' if folder_path.is_dir() else 'NOT FOUND'}")

HDFC: Found
ICICI_Prudential: Found
SBI: Found
Nippon_India: Found
Kotak: Found


In [12]:
# STEP 3: Count Excel files and check their formats in each AMC folder.

for amc in amc_folders:
    folder_path = RAW_DATA_DIR / amc

    xlsx_files = list(folder_path.glob("*.xlsx"))
    xls_files = list(folder_path.glob("*.xls"))

    print(f"\nAMC: {amc}")
    print(f"  .xlsx files: {len(xlsx_files)}")
    print(f"  .xls files:  {len(xls_files)}")
    print(f"  Total Excel files: {len(xlsx_files) + len(xls_files)}")


AMC: HDFC
  .xlsx files: 33
  .xls files:  0
  Total Excel files: 33

AMC: ICICI_Prudential
  .xlsx files: 33
  .xls files:  0
  Total Excel files: 33

AMC: SBI
  .xlsx files: 33
  .xls files:  0
  Total Excel files: 33

AMC: Nippon_India
  .xlsx files: 0
  .xls files:  33
  Total Excel files: 33

AMC: Kotak
  .xlsx files: 10
  .xls files:  23
  Total Excel files: 33


In [13]:
# STEP 4: Inspect the first HDFC workbook.
# This identifies the file and the worksheets it contains.

from pathlib import Path
from openpyxl import load_workbook

hdfc_folder = RAW_DATA_DIR / "HDFC"
hdfc_files = sorted(hdfc_folder.glob("*.xlsx"))

first_hdfc_file = hdfc_files[0]

print("File name:", first_hdfc_file.name)

workbook = load_workbook(
    first_hdfc_file,
    read_only=True,
    data_only=True
)

print("Worksheet names:", workbook.sheetnames)

workbook.close()

File name: Monthly HDFC Multi Asset Allocation Fund - 30 September 2026_0.xlsx
Worksheet names: ['MY2005', 'DerivativeHDFC Multi Asset Allo', 'DerivativeMY2005']


In [14]:
# STEP 5: Preview the first worksheet of the HDFC workbook.
# This helps us locate the portfolio table and identify its column headings.

from openpyxl import load_workbook

workbook = load_workbook(
    first_hdfc_file,
    read_only=True,
    data_only=True
)

sheet = workbook["MY2005"]

for row in sheet.iter_rows(
    min_row=1,
    max_row=15,
    min_col=1,
    max_col=10,
    values_only=True
):
    print(row)

workbook.close()

('HDFC Multi Asset Allocation Fund (An open ended scheme investing in Equity and Equity related instruments, Debt & Money Market Instruments and Gold). (Formerly known as HDFC Multi-Asset Allocation Fund)', 'HDFC Multi Asset Allocation Fund (An open ended scheme investing in Equity and Equity related instruments, Debt & Money Market Instruments and Gold)', 'HDFC Multi Asset Allocation Fund (An open ended scheme investing in Equity and Equity related instruments, Debt & Money Market Instruments and Gold)', 'HDFC Multi Asset Allocation Fund (An open ended scheme investing in Equity and Equity related instruments, Debt & Money Market Instruments and Gold)', 'HDFC Multi Asset Allocation Fund (An open ended scheme investing in Equity and Equity related instruments, Debt & Money Market Instruments and Gold)', 'HDFC Multi Asset Allocation Fund (An open ended scheme investing in Equity and Equity related instruments, Debt & Money Market Instruments and Gold)', 'HDFC Multi Asset Allocation Fund

In [15]:
# STEP 6: Preview the other two HDFC worksheets.
# This helps us understand what each worksheet contains before selecting one.

from openpyxl import load_workbook

workbook = load_workbook(
    first_hdfc_file,
    read_only=True,
    data_only=True
)

for sheet_name in workbook.sheetnames[1:]:
    sheet = workbook[sheet_name]

    print(f"\n{'=' * 60}")
    print("WORKSHEET:", sheet_name)
    print("=" * 60)

    for row in sheet.iter_rows(
        min_row=1,
        max_row=8,
        min_col=1,
        max_col=6,
        values_only=True
    ):
        print(row)

workbook.close()


WORKSHEET: DerivativeHDFC Multi Asset Allo
(None, None, None, None, None, None)
(None, None, None, None, None, None)
('DERIVATIVE DISCLOSURE - MY2005', 'DERIVATIVE DISCLOSURE - MY2005', 'DERIVATIVE DISCLOSURE - MY2005', 'DERIVATIVE DISCLOSURE - MY2005', 'DERIVATIVE DISCLOSURE - MY2005', 'DERIVATIVE DISCLOSURE - MY2005')
(None, None, None, None, None, None)
('Disclosure regarding Derivative positions pursuant to SEBI Circular no. CIR/IMD/DF/11/2010 dated August 18, 2010.', None, None, None, None, None)
(None, None, None, None, None, None)
('A.  Hedging Positions through Futures as on  September 30, 2026. : Nil', 'A.  Hedging Positions through Futures as on  September 30, 2026. : Nil', 'A.  Hedging Positions through Futures as on  September 30, 2026. : Nil', 'A.  Hedging Positions through Futures as on  September 30, 2026. : Nil', 'A.  Hedging Positions through Futures as on  September 30, 2026. : Nil', None)
(None, None, None, None, None, None)

WORKSHEET: DerivativeMY2005
(None, None,

In [16]:
# STEP 7: Search the HDFC main portfolio for gold, silver, and ETF references.
# This searches text in each row and prints matching rows for inspection.

from openpyxl import load_workbook

workbook = load_workbook(
    first_hdfc_file,
    read_only=True,
    data_only=True
)

sheet = workbook["MY2005"]
keywords = ["silver", "gold", "etf", "exchange traded"]

for row_number, row in enumerate(sheet.iter_rows(values_only=True), start=1):
    row_text = " | ".join(
        str(value) for value in row if value is not None
    )

    if any(keyword in row_text.lower() for keyword in keywords):
        print(f"Row {row_number}: {row_text}")

workbook.close()

Row 1: HDFC Multi Asset Allocation Fund (An open ended scheme investing in Equity and Equity related instruments, Debt & Money Market Instruments and Gold). (Formerly known as HDFC Multi-Asset Allocation Fund) | HDFC Multi Asset Allocation Fund (An open ended scheme investing in Equity and Equity related instruments, Debt & Money Market Instruments and Gold) | HDFC Multi Asset Allocation Fund (An open ended scheme investing in Equity and Equity related instruments, Debt & Money Market Instruments and Gold) | HDFC Multi Asset Allocation Fund (An open ended scheme investing in Equity and Equity related instruments, Debt & Money Market Instruments and Gold) | HDFC Multi Asset Allocation Fund (An open ended scheme investing in Equity and Equity related instruments, Debt & Money Market Instruments and Gold) | HDFC Multi Asset Allocation Fund (An open ended scheme investing in Equity and Equity related instruments, Debt & Money Market Instruments and Gold) | HDFC Multi Asset Allocation Fund 

In [17]:
# STEP 8: Inspect the first ICICI Prudential workbook.
# This identifies its filename and available worksheets.

from openpyxl import load_workbook

icici_folder = RAW_DATA_DIR / "ICICI_Prudential"
icici_files = sorted(icici_folder.glob("*.xlsx"))

first_icici_file = icici_files[0]

print("File name:", first_icici_file.name)

workbook = load_workbook(
    first_icici_file,
    read_only=True,
    data_only=True
)

print("Worksheet names:", workbook.sheetnames)

workbook.close()

File name: ICICI Prudential Multi Asset Allocation Fund-aug-2026.xlsx
Worksheet names: ['MULTI', 'Derivative']


In [18]:
# STEP 9: Preview the ICICI Prudential main portfolio worksheet.
# This helps us identify its portfolio headings and table structure.

from openpyxl import load_workbook

workbook = load_workbook(
    first_icici_file,
    read_only=True,
    data_only=True
)

sheet = workbook["MULTI"]

for row in sheet.iter_rows(
    min_row=1,
    max_row=15,
    min_col=1,
    max_col=10,
    values_only=True
):
    print(row)

workbook.close()

(None, 'ICICI Prudential Mutual Fund', None, None, None, None, None, None, None, None)
(None, 'ICICI Prudential Multi Asset Allocation Fund', None, None, None, None, None, None, None, None)
(None, 'Portfolio as on Aug 31,2026', None, None, None, None, None, None, None, None)
(None, 'Company/Issuer/Instrument Name', 'ISIN', 'Coupon', 'Industry/Rating', 'Quantity', 'Exposure/Market Value(Rs.Lakh)', '% to Nav', 'Yield of the instrument', 'Yield to Call @')
(None, 'Equity & Equity Related Instruments (Note -1)', '', '', '', '', 6135525.819999998, 0.6985437261038002, None, None)
(None, '', None, None, None, None, None, None, None, '')
(None, 'Listed / Awaiting Listing On Stock Exchanges', '', '', '', '', 6016405.120000002, 0.6852461713956997, '', '')
(None, 'HDFC Bank Ltd.', 'INE040A01034', '', 'Banks', 69199542, 490624.75, 0.0558587562078, '', '')
(None, 'Life Insurance Corporation of India', 'INE0J1Y01017', '', 'Insurance', 80107788, 334850.55, 0.0381235052624, '', '')
(None, 'ICICI Bank 

In [19]:
# STEP 10: Search the ICICI main portfolio for gold and silver references.
# This prints matching rows without changing the source workbook.

from openpyxl import load_workbook

workbook = load_workbook(
    first_icici_file,
    read_only=True,
    data_only=True
)

sheet = workbook["MULTI"]
keywords = ["silver", "gold", "etf", "exchange traded"]

for row_number, row in enumerate(sheet.iter_rows(values_only=True), start=1):
    row_text = " | ".join(
        str(value) for value in row if value is not None
    )

    if any(keyword in row_text.lower() for keyword in keywords):
        print(f"Row {row_number}: {row_text}")

workbook.close()

Row 155: Foreign Securities/Overseas ETFs | 15533.77 | 0.0017685554416
Row 301: ICICI Prudential Gold ETF | INF109KC1NT3 |  | Mutual Fund | 671777975 | 884865.95 | 0.1007440235692 |  | 
Row 367: 5. Total market value of Investment in Foreign Securities / ADRs / GDRs / Overseas ETFs as on Aug 31,2026 | 15533.77
Row 400: Benchmark name - Nifty 200 TRI (65%) + Nifty Composite Debt Index (25%) + Domestic Price of Gold (6%) + Domestic Price of Silver (1%) + iCOMDEX Composite Index (3%)


In [20]:
# STEP 11: Search specifically for silver-related entries in the ICICI portfolio.
# This narrows the search and leaves the source workbook unchanged.

from openpyxl import load_workbook

workbook = load_workbook(
    first_icici_file,
    read_only=True,
    data_only=True
)

sheet = workbook["MULTI"]

silver_matches = []

for row_number, row in enumerate(sheet.iter_rows(values_only=True), start=1):
    row_text = " | ".join(
        str(value) for value in row if value is not None
    )

    if "silver" in row_text.lower():
        silver_matches.append((row_number, row_text))

if silver_matches:
    for row_number, row_text in silver_matches:
        print(f"Row {row_number}: {row_text}")
else:
    print("No rows containing 'silver' were found.")

workbook.close()

Row 400: Benchmark name - Nifty 200 TRI (65%) + Nifty Composite Debt Index (25%) + Domestic Price of Gold (6%) + Domestic Price of Silver (1%) + iCOMDEX Composite Index (3%)


In [21]:
# STEP 12: Inspect the first SBI workbook.
# This identifies the workbook and its available worksheets.

from openpyxl import load_workbook

sbi_folder = RAW_DATA_DIR / "SBI"
sbi_files = sorted(sbi_folder.glob("*.xlsx"))

first_sbi_file = sbi_files[0]

print("File name:", first_sbi_file.name)

workbook = load_workbook(
    first_sbi_file,
    read_only=True,
    data_only=True
)

print("Worksheet names:", workbook.sheetnames)

workbook.close()

File name: SBI-MULTI-ASSET-ALLOCATION-FUND---MARCH-2024.xlsx


Worksheet names: ['SMAAF']


In [22]:
# STEP 13: Preview the SBI main portfolio worksheet.
# This helps us identify the table structure and reporting date.

from openpyxl import load_workbook

workbook = load_workbook(
    first_sbi_file,
    read_only=True,
    data_only=True
)

sheet = workbook["SMAAF"]

for row in sheet.iter_rows(
    min_row=1,
    max_row=15,
    min_col=1,
    max_col=10,
    values_only=True
):
    print(row)

workbook.close()

(None, None, None, None, None, None, None, None, None, None)
(None, None, 'SBI Mutual Fund', '101', None, None, None, None, None, 'Back to Index')
(None, None, 'SCHEME NAME :', 'SBI Multi Asset Allocation Fund', None, None, None, None, None, None)
(None, None, 'PORTFOLIO STATEMENT AS ON :', datetime.datetime(2024, 3, 31, 0, 0), None, None, None, None, None, None)
(None, None, None, None, None, None, None, None, None, None)
(None, None, 'Name of the Instrument / Issuer', 'ISIN', 'Rating / Industry^', 'Quantity', 'Market value\n(Rs. in Lakhs)', '% to AUM', 'YTM %', 'YTC % ##')
(None, None, None, None, None, None, None, None, None, None)
(None, None, 'EQUITY & EQUITY RELATED', None, None, None, None, None, None, None)
(None, None, 'a) Listed/awaiting listing on Stock Exchanges', None, None, None, None, None, None, None)
(None, '100176', 'GAIL (India) Ltd.', 'INE129A01019', 'Gas', 4350297, 7876.21, 1.86, None, None)
(None, '100292', 'Steel Authority of India Ltd.', 'INE114A01011', 'Ferrous

In [23]:
# STEP 14: Search the SBI main portfolio for gold and silver references.
# This prints matching rows and leaves the source workbook unchanged.

from openpyxl import load_workbook

workbook = load_workbook(
    first_sbi_file,
    read_only=True,
    data_only=True
)

sheet = workbook["SMAAF"]
keywords = ["silver", "gold", "etf", "exchange traded"]

for row_number, row in enumerate(sheet.iter_rows(values_only=True), start=1):
    row_text = " | ".join(
        str(value) for value in row if value is not None
    )

    if any(keyword in row_text.lower() for keyword in keywords):
        print(f"Row {row_number}: {row_text}")

workbook.close()

Row 64: c) Foreign Securities and /or overseas ETF | NIL | NIL
Row 137: a) Mutual Fund Units / Exchange Traded Funds
Row 138: 417210 | SBI Gold ETF | INF200KA16D8 | Mutual Fund | 103241000 | 60272.1 | 14.25
Row 139: 2800001 | Nippon India Silver ETF | INF204KC1402 | Mutual Fund | 6000000 | 4345.2 | 1.03
Row 144: c) Gold | NIL | NIL
Row 171: Benchmark Name : 45% S&P BSE 500 TRI + 40% CRISIL COMPOSITE BOND FUND INDEX + 10% DOMESTIC PRICES OF GOLD + 5% DOMESTIC PRICES OF SILVER


In [24]:
# STEP 15: Identify the first Nippon India workbook.
# This confirms the filename and extension before choosing a reader.

nippon_folder = RAW_DATA_DIR / "Nippon_India"
nippon_files = sorted(nippon_folder.glob("*.xls"))

first_nippon_file = nippon_files[0]

print("File name:", first_nippon_file.name)
print("File extension:", first_nippon_file.suffix)
print("File exists:", first_nippon_file.is_file())

File name: MONTHLY-PORTFOLIO-JAN-2024.xls
File extension: .xls
File exists: True


In [25]:
# STEP 16: Check whether the xlrd library is available.
# xlrd is commonly used to read older .xls Excel files.

import importlib.util

if importlib.util.find_spec("xlrd") is not None:
    print("xlrd is installed.")
else:
    print("xlrd is not installed.")

xlrd is installed.


In [26]:
# STEP 17: Install the library required to read older .xls files.
%pip install xlrd

Note: you may need to restart the kernel to use updated packages.


In [27]:
# STEP 18: Inspect the first Nippon India .xls workbook.
# This shows the worksheet names and the first 12 rows of each sheet.

import xlrd

nippon_folder = RAW_DATA_DIR / "Nippon_India"
nippon_files = sorted(nippon_folder.glob("*.xls"))

first_nippon_file = nippon_files[0]

print("File name:", first_nippon_file.name)

workbook = xlrd.open_workbook(
    str(first_nippon_file),
    on_demand=True
)

print("Worksheet names:", workbook.sheet_names())

for sheet_name in workbook.sheet_names():
    sheet = workbook.sheet_by_name(sheet_name)

    print(f"\n{'=' * 60}")
    print("WORKSHEET:", sheet_name)
    print("Rows:", sheet.nrows, "| Columns:", sheet.ncols)

    for row_index in range(min(12, sheet.nrows)):
        print(sheet.row_values(row_index))

workbook.release_resources()

File name: MONTHLY-PORTFOLIO-JAN-2024.xls


XLRDError: Excel xlsx file; not supported

In [ ]:
# STEP 19: Try opening the Nippon India workbook with openpyxl.
# The file has an .xls extension, but its internal format appears to be .xlsx.

from openpyxl import load_workbook

workbook = load_workbook(
    first_nippon_file,
    read_only=True,
    data_only=True
)

print("File name:", first_nippon_file.name)
print("Worksheet names:", workbook.sheetnames)

for sheet_name in workbook.sheetnames:
    sheet = workbook[sheet_name]

    print(f"\n{'=' * 60}")
    print("WORKSHEET:", sheet_name)
    print("Rows:", sheet.max_row, "| Columns:", sheet.max_column)

    for row in sheet.iter_rows(
        min_row=1,
        max_row=min(12, sheet.max_row),
        values_only=True
    ):
        print(row)

workbook.close()

InvalidFileException: openpyxl does not support the old .xls file format, please use xlrd to read this file, or convert it to the more recent .xlsx file format.

In [ ]:
# STEP 20: Inspect the file signature to identify its actual format.
# This reads only the first 8 bytes and leaves the original file unchanged.

with open(first_nippon_file, "rb") as file:
    file_signature = file.read(8)

print("File name:", first_nippon_file.name)
print("First 8 bytes:", file_signature)
print("Hexadecimal signature:", file_signature.hex())

File name: MONTHLY-PORTFOLIO-JAN-2024.xls
First 8 bytes: b'PK\x03\x04\x14\x00\x06\x00'
Hexadecimal signature: 504b030414000600


In [ ]:
# STEP 21: Check whether the .xls file contains an .xlsx-style ZIP structure.
# This reads the archive directory only; it does not extract or modify the file.

from zipfile import ZipFile, BadZipFile

try:
    with ZipFile(first_nippon_file, "r") as archive:
        print("Valid ZIP container: Yes")
        print("First internal files:")
        for name in archive.namelist()[:15]:
            print(name)

except BadZipFile:
    print("Valid ZIP container: No")
    print("The file may be an older Excel workbook or another format.")

Valid ZIP container: Yes
First internal files:
[Content_Types].xml
_rels/.rels
xl/workbook.xml
xl/_rels/workbook.xml.rels
xl/worksheets/sheet1.xml
xl/worksheets/sheet2.xml
xl/worksheets/sheet3.xml
xl/worksheets/sheet4.xml
xl/worksheets/sheet5.xml
xl/worksheets/sheet6.xml
xl/worksheets/sheet7.xml
xl/worksheets/sheet8.xml
xl/worksheets/sheet9.xml
xl/worksheets/sheet10.xml
xl/worksheets/sheet11.xml


In [ ]:
# STEP 21: Inspect the internal worksheet names of the Nippon India workbook.
# This checks the workbook structure without modifying the original file.

from zipfile import ZipFile
from xml.etree import ElementTree as ET

with ZipFile(first_nippon_file, "r") as archive:
    workbook_xml = archive.read("xl/workbook.xml")

root = ET.fromstring(workbook_xml)

namespace = {
    "main": "http://schemas.openxmlformats.org/spreadsheetml/2006/main"
}

sheet_names = [
    sheet.attrib["name"]
    for sheet in root.findall("main:sheets/main:sheet", namespace)
]

print("File:", first_nippon_file.name)
print("Internal worksheet names:", sheet_names)

File: MONTHLY-PORTFOLIO-JAN-2024.xls
Internal worksheet names: ['INDEX', 'GF', 'GS', 'IP', 'IF', 'ST', 'BF', 'MI', 'PS', 'CP', 'PH', 'ME', 'NE', 'NI', 'EO', 'LF', 'FR', 'SD', 'SE', 'SH', 'LQ', 'TS', 'LE', 'IV', 'LP', 'IM', 'DC', 'UB', 'K5', '9B', 'EA', 'QP', 'GL', 'NF', 'SC', 'SF', 'AF', 'GD', 'CE', 'CF', 'RD', 'CB', 'JE', 'SX', 'RW', 'RG', 'BP', 'ES', 'NX', 'UE', 'LT', 'NL', 'NB', 'JZ', 'PU', 'GB', 'LB', 'BB', 'SB', 'HS', 'IB', 'CC', 'ON', 'NM', 'RJ', 'G6', 'SN', 'NZ', 'MF', 'NS', 'NC', 'PP', 'AL', 'NP', 'NV', 'NY', 'NG', 'NH', 'LC', 'HG', 'ET', 'NO', 'SI', 'SO', 'HK', 'CN', 'NN', 'W5', 'ND', 'NJ', 'NK', 'NQ', 'W8', 'NU', 'NW', 'U7', 'U8', 'IT', 'R4']


In [ ]:
# STEP 22: Inspect the INDEX worksheet in the Nippon India workbook.
# This helps us identify the worksheet associated with the fund we need.

from zipfile import ZipFile
from xml.etree import ElementTree as ET

with ZipFile(first_nippon_file, "r") as archive:
    workbook_root = ET.fromstring(archive.read("xl/workbook.xml"))
    shared_strings_root = ET.fromstring(archive.read("xl/sharedStrings.xml"))

    namespace = {
        "main": "http://schemas.openxmlformats.org/spreadsheetml/2006/main",
        "rel": "http://schemas.openxmlformats.org/officeDocument/2006/relationships"
    }

    shared_strings = [
        "".join(text.text or "" for text in item.findall(".//main:t", namespace))
        for item in shared_strings_root.findall("main:si", namespace)
    ]

    index_sheet = workbook_root.find(
        "main:sheets/main:sheet[@name='INDEX']", namespace
    )

    relationship_id = index_sheet.attrib[
        "{http://schemas.openxmlformats.org/officeDocument/2006/relationships}id"
    ]

    relationships_root = ET.fromstring(
        archive.read("xl/_rels/workbook.xml.rels")
    )

    relationship_namespace = {
        "rel": "http://schemas.openxmlformats.org/package/2006/relationships"
    }

    sheet_target = next(
        relationship.attrib["Target"]
        for relationship in relationships_root.findall("rel:Relationship", relationship_namespace)
        if relationship.attrib["Id"] == relationship_id
    )

    sheet_path = "xl/" + sheet_target.lstrip("/")
    sheet_root = ET.fromstring(archive.read(sheet_path))

    for row in sheet_root.findall(".//main:sheetData/main:row", namespace):
        values = []

        for cell in row.findall("main:c", namespace):
            value_element = cell.find("main:v", namespace)
            value = value_element.text if value_element is not None else ""

            if cell.attrib.get("t") == "s" and value:
                value = shared_strings[int(value)]

            inline = cell.find("main:is", namespace)
            if inline is not None:
                value = "".join(
                    text.text or ""
                    for text in inline.findall(".//main:t", namespace)
                )

            if value:
                values.append(f"{cell.attrib.get('r')}: {value}")

        if values:
            print(" | ".join(values))

A1: INDEX
A2: GF | B2: NIPPON INDIA GROWTH FUND (Mid Cap Fund- An open ended equity scheme predominantly investing in mid cap stocks)
A3: GS | B3: NIPPON INDIA VISION FUND (An Open Ended Equity Scheme Investing In Both Large Cap And Mid Cap Stocks)
A4: IP | B4: Nippon India Corporate Bond Fund (An open ended debt scheme predominantly investing in AA+ and above rated corporate bonds. Relatively High interest rate risk and moderate Credit Risk.)
A5: IF | B5: Nippon India Income Fund (An open ended medium term debt scheme investing in instruments such that the Macaulay duration of the portfolio is between 4 to 7 years. Relatively High interest rate risk and Relatively Low Credit Risk)
A6: ST | B6: Nippon India Short Term Fund (An open ended short term debt scheme investing in instruments such that the Macaulay duration of the portfolio is between 1 to 3 years. Relatively High interest rate risk and moderate Credit Risk.)
A7: BF | B7: NIPPON INDIA BANKING & FINANCIAL SERVICES FUND (An open

In [ ]:
# STEP 23: Inspect the MF worksheet to verify the fund name and reporting date.

from zipfile import ZipFile
from xml.etree import ElementTree as ET

with ZipFile(first_nippon_file, "r") as archive:
    workbook_root = ET.fromstring(archive.read("xl/workbook.xml"))

    namespace = {
        "main": "http://schemas.openxmlformats.org/spreadsheetml/2006/main",
        "rel": "http://schemas.openxmlformats.org/officeDocument/2006/relationships",
    }

    sheet = workbook_root.find(
        "main:sheets/main:sheet[@name='MF']", namespace
    )

    relationship_id = sheet.attrib[
        "{http://schemas.openxmlformats.org/officeDocument/2006/relationships}id"
    ]

    relationships_root = ET.fromstring(
        archive.read("xl/_rels/workbook.xml.rels")
    )

    relationship_namespace = {
        "rel": "http://schemas.openxmlformats.org/package/2006/relationships"
    }

    target = next(
        item.attrib["Target"]
        for item in relationships_root.findall(
            "rel:Relationship", relationship_namespace
        )
        if item.attrib["Id"] == relationship_id
    )

    sheet_path = (
        target.lstrip("/")
        if target.startswith("/")
        else "xl/" + target
    )

    sheet_root = ET.fromstring(archive.read(sheet_path))

    shared_strings = []
    if "xl/sharedStrings.xml" in archive.namelist():
        strings_root = ET.fromstring(archive.read("xl/sharedStrings.xml"))
        shared_strings = [
            "".join(t.text or "" for t in item.findall(".//main:t", namespace))
            for item in strings_root.findall("main:si", namespace)
        ]

    for row in sheet_root.findall(".//main:sheetData/main:row", namespace)[:20]:
        values = []

        for cell in row.findall("main:c", namespace):
            value_element = cell.find("main:v", namespace)
            value = value_element.text if value_element is not None else ""

            if cell.attrib.get("t") == "s" and value:
                value = shared_strings[int(value)]

            inline = cell.find("main:is", namespace)
            if inline is not None:
                value = "".join(
                    t.text or "" for t in inline.findall(".//main:t", namespace)
                )

            if value:
                values.append(f"{cell.attrib.get('r')}: {value}")

        if values:
            print(" | ".join(values))

A1: RLMF806 | B1: Nippon India Multi Asset Fund (An open ended scheme investing in Equity, Debt and Exchange Traded Commodity Derivatives and Gold ETF) | H1: Index
B2: Monthly Portfolio Statement as on January 31,2024
B4: ISIN | C4: Name of the Instrument | D4: Industry / Rating | E4: Quantity | F4: Market/Fair Value
( Rs. in Lacs) | G4: % to NAV | H4: YIELD
C5: Derivatives
C6: Commodity Options
A7: SILVERN24B72000C | C7: SILVER 72000 Call February 2024 Option (1 kg) | E7: 1110 | F7: 17.239999999999998 | G7: 1E-4
C8: Total | F8: 17.239999999999998 | G8: 1E-4
C9: Equity & Equity related
C10: (a) Listed / awaiting listing on Stock Exchanges
A11: HDFB03 | B11: INE040A01034 | C11: HDFC Bank Limited | D11: Banks | E11: 545000 | F11: 7970.9 | G11: 3.1300000000000001E-2
A12: IBCL05 | B12: INE090A01021 | C12: ICICI Bank Limited | D12: Banks | E12: 650000 | F12: 6682.98 | G12: 2.63E-2
A13: RIND01 | B13: INE002A01018 | C13: Reliance Industries Limited | D13: Petroleum Products | E13: 199847 | F1

In [ ]:
# STEP 21: Search the Nippon India workbook for Gold and Silver entries.
# This helps us locate commodity-related holdings before extracting portfolio data.

from zipfile import ZipFile
from openpyxl import load_workbook

# The file is internally an XLSX workbook despite its .xls extension.
with ZipFile(first_nippon_file, "r") as archive:
    workbook = load_workbook(
        archive.open("xl/workbook.xml") if False else first_nippon_file,
        read_only=True,
        data_only=True
    )

InvalidFileException: openpyxl does not support the old .xls file format, please use xlrd to read this file, or convert it to the more recent .xlsx file format.

In [ ]:
# STEP 21: Open the internally XLSX-formatted workbook as a binary stream.
# This avoids the .xls filename-extension check without changing the raw file.

from openpyxl import load_workbook

with open(first_nippon_file, "rb") as file_stream:
    workbook = load_workbook(
        file_stream,
        read_only=True,
        data_only=True
    )

    print("File name:", first_nippon_file.name)
    print("Worksheet names:", workbook.sheetnames)

    workbook.close()

File name: MONTHLY-PORTFOLIO-JAN-2024.xls
Worksheet names: ['INDEX', 'GF', 'GS', 'IP', 'IF', 'ST', 'BF', 'MI', 'PS', 'CP', 'PH', 'ME', 'NE', 'NI', 'EO', 'LF', 'FR', 'SD', 'SE', 'SH', 'LQ', 'TS', 'LE', 'IV', 'LP', 'IM', 'DC', 'UB', 'K5', '9B', 'EA', 'QP', 'GL', 'NF', 'SC', 'SF', 'AF', 'GD', 'CE', 'CF', 'RD', 'CB', 'JE', 'SX', 'RW', 'RG', 'BP', 'ES', 'NX', 'UE', 'LT', 'NL', 'NB', 'JZ', 'PU', 'GB', 'LB', 'BB', 'SB', 'HS', 'IB', 'CC', 'ON', 'NM', 'RJ', 'G6', 'SN', 'NZ', 'MF', 'NS', 'NC', 'PP', 'AL', 'NP', 'NV', 'NY', 'NG', 'NH', 'LC', 'HG', 'ET', 'NO', 'SI', 'SO', 'HK', 'CN', 'NN', 'W5', 'ND', 'NJ', 'NK', 'NQ', 'W8', 'NU', 'NW', 'U7', 'U8', 'IT', 'R4']


In [ ]:
# STEP 22: Search the Nippon India MF worksheet for Gold and Silver.
# The workbook is opened as a binary stream because its .xls extension
# does not match its internal .xlsx format. The original file is unchanged.

from openpyxl import load_workbook

with open(first_nippon_file, "rb") as file_stream:
    workbook = load_workbook(
        file_stream,
        read_only=True,
        data_only=True
    )

    sheet = workbook["MF"]

    for row_number, row in enumerate(
        sheet.iter_rows(values_only=True), start=1
    ):
        row_text = " | ".join(
            str(value) for value in row if value is not None
        )

        if "gold" in row_text.lower() or "silver" in row_text.lower():
            print(f"Row {row_number}: {row_text}")

    workbook.close()

Row 1: RLMF806 | Nippon India Multi Asset Fund (An open ended scheme investing in Equity, Debt and Exchange Traded Commodity Derivatives and Gold ETF) | Index
Row 7: SILVERN24B72000C | SILVER 72000 Call February 2024 Option (1 kg) | 1110 | 17.24 | 0.0001
Row 108: BMGB51ME | INF204KB17I5 | Nippon India ETF Gold Bees | 62360000 | 33269.06 | 0.1308
Row 129: Long | FUTCOM_SILVER_05/03/2024 | Others | 1920 | 1387.14 | 0.0055
Row 130: Long | FUTCOM_SILVER_03/05/2024 | Others | 120 | 88.15 | 0.0003
Row 132: BENCHMARK NAME - CRISIL SHORT TERM BOND INDEX, DOMESTIC PRICE OF GOLD, MSCI WORLD INDEX TRI, S&P BSE 500 TRI


In [ ]:
# STEP 23: List all files in the Nippon India folder

from pathlib import Path

nippon_folder = RAW_DATA_DIR / "Nippon_India"

for file_path in sorted(nippon_folder.iterdir()):
    if file_path.is_file():
        print(file_path.name)

MONTHLY-PORTFOLIO-JAN-2024.xls
MONTHLY-PORTFOLIO-REPORT-FEB-24.xls
NIMF-MONTHLY-PORTFOLIO-28-Feb-25.xls
NIMF-MONTHLY-PORTFOLIO-28-Feb-26.xls
NIMF-MONTHLY-PORTFOLIO-30-April-25.xls
NIMF-MONTHLY-PORTFOLIO-30-April-26.xls
NIMF-MONTHLY-PORTFOLIO-30-Jun-26.xls
NIMF-MONTHLY-PORTFOLIO-30-June-25.xls
NIMF-MONTHLY-PORTFOLIO-30-Nov-2024.xls
NIMF-MONTHLY-PORTFOLIO-30-Sep-24.xls
NIMF-MONTHLY-PORTFOLIO-30-Sep-25.xls
NIMF-MONTHLY-PORTFOLIO-30-Sep-26.xls
NIMF-MONTHLY-PORTFOLIO-31-Aug-25.xls
NIMF-MONTHLY-PORTFOLIO-31-Aug-26.xls
NIMF-MONTHLY-PORTFOLIO-31-Dec-24.xls
NIMF-MONTHLY-PORTFOLIO-31-Dec-25.xls
NIMF-MONTHLY-PORTFOLIO-31-Jan-26.xls
NIMF-MONTHLY-PORTFOLIO-31-July-24.xls
NIMF-MONTHLY-PORTFOLIO-31-July-25.xls
NIMF-MONTHLY-PORTFOLIO-31-July-26.xls
NIMF-MONTHLY-PORTFOLIO-31-Mar-25.xls
NIMF-MONTHLY-PORTFOLIO-31-Mar-26.xls
NIMF-MONTHLY-PORTFOLIO-31-May-25.xls
NIMF-MONTHLY-PORTFOLIO-31-May-26.xls
NIMF-MONTHLY-PORTFOLIO-31-Oct-24.xls
NIMF-MONTHLY-PORTFOLIO-31-Oct-25.xls
NIMF-MONTHLY-PORTFOLIO-APRIL-2024.x

In [ ]:
# STEP 24: Inspect the January 2024 workbook's index

from openpyxl import load_workbook

first_nippon_file = nippon_folder / "MONTHLY-PORTFOLIO-JAN-2024.xls"

with open(first_nippon_file, "rb") as file_stream:
    workbook = load_workbook(
        file_stream,
        read_only=True,
        data_only=True
    )

    print("Worksheet names:")
    print(workbook.sheetnames)

    if "INDEX" in workbook.sheetnames:
        print("\nFirst 30 rows of the INDEX sheet:")

        for row_number, row in enumerate(
            workbook["INDEX"].iter_rows(
                min_row=1,
                max_row=30,
                values_only=True
            ),
            start=1
        ):
            values = [value for value in row if value is not None]
            if values:
                print(f"Row {row_number}: {values}")

    workbook.close()

Worksheet names:
['INDEX', 'GF', 'GS', 'IP', 'IF', 'ST', 'BF', 'MI', 'PS', 'CP', 'PH', 'ME', 'NE', 'NI', 'EO', 'LF', 'FR', 'SD', 'SE', 'SH', 'LQ', 'TS', 'LE', 'IV', 'LP', 'IM', 'DC', 'UB', 'K5', '9B', 'EA', 'QP', 'GL', 'NF', 'SC', 'SF', 'AF', 'GD', 'CE', 'CF', 'RD', 'CB', 'JE', 'SX', 'RW', 'RG', 'BP', 'ES', 'NX', 'UE', 'LT', 'NL', 'NB', 'JZ', 'PU', 'GB', 'LB', 'BB', 'SB', 'HS', 'IB', 'CC', 'ON', 'NM', 'RJ', 'G6', 'SN', 'NZ', 'MF', 'NS', 'NC', 'PP', 'AL', 'NP', 'NV', 'NY', 'NG', 'NH', 'LC', 'HG', 'ET', 'NO', 'SI', 'SO', 'HK', 'CN', 'NN', 'W5', 'ND', 'NJ', 'NK', 'NQ', 'W8', 'NU', 'NW', 'U7', 'U8', 'IT', 'R4']

First 30 rows of the INDEX sheet:
Row 1: ['INDEX']
Row 2: ['GF', 'NIPPON INDIA GROWTH FUND (Mid Cap Fund- An open ended equity scheme predominantly investing in mid cap stocks)']
Row 3: ['GS', 'NIPPON INDIA VISION FUND (An Open Ended Equity Scheme Investing In Both Large Cap And Mid Cap Stocks)']
Row 4: ['IP', 'Nippon India Corporate Bond Fund (An open ended debt scheme predominant

In [ ]:
# STEP 25: Locate the Multi Asset fund in the INDEX sheet

from openpyxl import load_workbook

with open(first_nippon_file, "rb") as file_stream:
    workbook = load_workbook(
        file_stream,
        read_only=True,
        data_only=True
    )

    index_sheet = workbook["INDEX"]

    for row_number, row in enumerate(
        index_sheet.iter_rows(values_only=True),
        start=1
    ):
        row_text = " | ".join(
            str(value) for value in row if value is not None
        )

        if "multi asset" in row_text.lower():
            print(f"Row {row_number}: {row_text}")

    workbook.close()

Row 69: MF | Nippon India Multi Asset Fund (An open ended scheme investing in Equity, Debt and Exchange Traded Commodity Derivatives and Gold ETF)


In [ ]:
# STEP 26: Inspect the identified Multi Asset worksheet

from openpyxl import load_workbook

with open(first_nippon_file, "rb") as file_stream:
    workbook = load_workbook(
        file_stream,
        read_only=True,
        data_only=True
    )

    sheet = workbook["MF"]

    for row_number, row in enumerate(
        sheet.iter_rows(min_row=1, max_row=15, values_only=True),
        start=1
    ):
        values = [value for value in row if value is not None]
        if values:
            print(f"Row {row_number}: {values}")

    workbook.close()

Row 1: ['RLMF806', 'Nippon India Multi Asset Fund (An open ended scheme investing in Equity, Debt and Exchange Traded Commodity Derivatives and Gold ETF)', 'Index']
Row 2: ['Monthly Portfolio Statement as on January 31,2024']
Row 4: ['ISIN', 'Name of the Instrument', 'Industry / Rating', 'Quantity', 'Market/Fair Value\n( Rs. in Lacs)', '% to NAV', 'YIELD']
Row 5: ['Derivatives']
Row 6: ['Commodity Options']
Row 7: ['SILVERN24B72000C', 'SILVER 72000 Call February 2024 Option (1 kg)', 1110, 17.24, 0.0001]
Row 8: ['', 'Total', '', '', 17.24, 0.0001]
Row 9: ['Equity & Equity related']
Row 10: ['(a) Listed / awaiting listing on Stock Exchanges']
Row 11: ['HDFB03', 'INE040A01034', 'HDFC Bank Limited', 'Banks', 545000, 7970.9, 0.0313]
Row 12: ['IBCL05', 'INE090A01021', 'ICICI Bank Limited', 'Banks', 650000, 6682.98, 0.0263]
Row 13: ['RIND01', 'INE002A01018', 'Reliance Industries Limited', 'Petroleum Products', 199847, 5702.13, 0.0224]
Row 14: ['TELC04', 'IN9155A01020', 'Tata Motors Limited (D

In [ ]:
# STEP 27: Locate gold, silver, and portfolio section headings

from openpyxl import load_workbook

with open(first_nippon_file, "rb") as file_stream:
    workbook = load_workbook(
        file_stream,
        read_only=True,
        data_only=True
    )

    sheet = workbook["MF"]

    for row_number, row in enumerate(
        sheet.iter_rows(values_only=True),
        start=1
    ):
        values = [value for value in row if value is not None]
        if not values:
            continue

        row_text = " | ".join(str(value) for value in values)
        text_lower = row_text.lower()

        if (
            "gold" in text_lower
            or "silver" in text_lower
            or any(
                heading in text_lower
                for heading in [
                    "debt instruments",
                    "exchange traded",
                    "mutual fund",
                    "commodity futures",
                    "commodity options",
                    "total"
                ]
            )
        ):
            print(f"Row {row_number}: {row_text}")

    workbook.close()

Row 1: RLMF806 | Nippon India Multi Asset Fund (An open ended scheme investing in Equity, Debt and Exchange Traded Commodity Derivatives and Gold ETF) | Index
Row 6: Commodity Options
Row 7: SILVERN24B72000C | SILVER 72000 Call February 2024 Option (1 kg) | 1110 | 17.24 | 0.0001
Row 8:  | Total |  |  | 17.24 | 0.0001
Row 73: Subtotal | 135545.11000000002 | 0.5329999999999998
Row 75: Subtotal | NIL | NIL | NIL
Row 76: Total | 135545.11 | 0.533
Row 79: Subtotal | 42384.23 | 0.1667
Row 80: Total | 42384.23 | 0.1667
Row 81: Debt Instruments
Row 97: Subtotal | 29759.11 | 0.1169
Row 99: Subtotal | NIL | NIL | NIL
Row 101: Subtotal | NIL | NIL
Row 102: Total | 29759.11 | 0.1169
Row 106: Total | 11062.17 | 0.0435
Row 107: Exchange Traded Funds
Row 108: BMGB51ME | INF204KB17I5 | Nippon India ETF Gold Bees | 62360000 | 33269.06 | 0.1308
Row 109: Total | 33269.06 | 0.1308
Row 113: Total | 536.66 | 0.0021
Row 115: GRAND TOTAL | 254307.5778793 | 1
Row 129: Long | FUTCOM_SILVER_05/03/2024 | Others |

In [ ]:
# STEP 28: Identify the Multi Asset worksheet in the February 2024 file

from openpyxl import load_workbook

feb_nippon_file = nippon_folder / "MONTHLY-PORTFOLIO-REPORT-FEB-24.xls"

with open(feb_nippon_file, "rb") as file_stream:
    workbook = load_workbook(
        file_stream,
        read_only=True,
        data_only=True
    )

    index_sheet = workbook["INDEX"]

    for row_number, row in enumerate(
        index_sheet.iter_rows(values_only=True),
        start=1
    ):
        values = [value for value in row if value is not None]
        row_text = " | ".join(str(value) for value in values)

        if "multi asset" in row_text.lower():
            print(f"Row {row_number}: {row_text}")

    workbook.close()

KeyError: 'Worksheet INDEX does not exist.'

In [ ]:
# STEP 28A: Inspect the February workbook structure

from openpyxl import load_workbook

feb_nippon_file = nippon_folder / "MONTHLY-PORTFOLIO-REPORT-FEB-24.xls"

with open(feb_nippon_file, "rb") as file_stream:
    workbook = load_workbook(
        file_stream,
        read_only=True,
        data_only=True
    )

    print("February workbook worksheets:")
    print(workbook.sheetnames)

    workbook.close()

February workbook worksheets:
['Index', 'GF', 'GS', 'IP', 'IF', 'ST', 'BF', 'MI', 'PS', 'CP', 'PH', 'ME', 'NE', 'NI', 'EO', 'LF', 'FR', 'SD', 'SE', 'SH', 'LQ', 'TS', 'LE', 'IV', 'LP', 'IM', 'DC', 'UB', 'K5', '9B', 'EA', 'QP', 'GL', 'NF', 'SC', 'SF', 'AF', 'GD', 'CE', 'CF', 'RD', 'CB', 'JE', 'SX', 'RW', 'RG', 'BP', 'ES', 'NX', 'UE', 'LT', 'NL', 'NB', 'JZ', 'PU', 'GB', 'LB', 'BB', 'SB', 'HS', 'IB', 'CC', 'ON', 'NM', 'RJ', 'G6', 'SN', 'NZ', 'MF', 'NS', 'NC', 'PP', 'AL', 'NP', 'NV', 'NY', 'NG', 'NH', 'LC', 'HG', 'ET', 'NO', 'SI', 'SO', 'HK', 'CN', 'NN', 'W5', 'ND', 'NJ', 'NK', 'NQ', 'W8', 'NU', 'NW', 'U7', 'U8', 'IT', 'R4', 'IX', 'BX']


In [ ]:
# STEP 29: Find the Multi Asset fund in the February Index sheet

from openpyxl import load_workbook

with open(feb_nippon_file, "rb") as file_stream:
    workbook = load_workbook(
        file_stream,
        read_only=True,
        data_only=True
    )

    index_sheet = workbook["Index"]

    for row_number, row in enumerate(
        index_sheet.iter_rows(values_only=True),
        start=1
    ):
        values = [value for value in row if value is not None]
        row_text = " | ".join(str(value) for value in values)

        if "multi asset" in row_text.lower():
            print(f"Row {row_number}: {row_text}")

    workbook.close()

Row 69: MF | Nippon India Multi Asset Fund (An open ended scheme investing in Equity, Debt and Exchange Traded Commodity Derivatives and Gold ETF)


In [ ]:
# STEP 30: Inspect the February 2024 Multi Asset portfolio

from openpyxl import load_workbook

with open(feb_nippon_file, "rb") as file_stream:
    workbook = load_workbook(
        file_stream,
        read_only=True,
        data_only=True
    )

    sheet = workbook["MF"]

    for row_number, row in enumerate(
        sheet.iter_rows(min_row=1, max_row=15, values_only=True),
        start=1
    ):
        values = [value for value in row if value is not None]
        if values:
            print(f"Row {row_number}: {values}")

    workbook.close()

Row 1: ['RLMF806', 'Nippon India Multi Asset Fund (An open ended scheme investing in Equity, Debt and Exchange Traded Commodity Derivatives and Gold ETF)', 'Index']
Row 2: ['Monthly Portfolio Statement as on February 29,2024']
Row 4: ['ISIN', 'Name of the Instrument', 'Industry / Rating', 'Quantity', 'Market/Fair Value\n( Rs. in Lacs)', '% to NAV', 'YIELD']
Row 5: ['Equity & Equity related']
Row 6: ['(a) Listed / awaiting listing on Stock Exchanges']
Row 7: ['HDFB03', 'INE040A01034', 'HDFC Bank Limited', 'Banks', 620000, 8701.08, 0.0322]
Row 8: ['IBCL05', 'INE090A01021', 'ICICI Bank Limited', 'Banks', 650000, 6839.3, 0.0253]
Row 9: ['RIND01', 'INE002A01018', 'Reliance Industries Limited', 'Petroleum Products', 180000, 5258.88, 0.0195]
Row 10: ['TELC04', 'IN9155A01020', 'Tata Motors Limited (DVR Shares)', 'Automobiles', 810000, 5121.23, 0.0189]
Row 11: ['LARS02', 'INE018A01030', 'Larsen & Toubro Limited', 'Construction', 136500, 4746.86, 0.0176]
Row 12: ['TCSL01', 'INE467B01029', 'Tata 

In [ ]:
# STEP 31: Locate gold, silver, and portfolio sections in February

from openpyxl import load_workbook

with open(feb_nippon_file, "rb") as file_stream:
    workbook = load_workbook(
        file_stream,
        read_only=True,
        data_only=True
    )

    sheet = workbook["MF"]

    for row_number, row in enumerate(
        sheet.iter_rows(values_only=True),
        start=1
    ):
        values = [value for value in row if value is not None]

        if not values:
            continue

        row_text = " | ".join(str(value) for value in values)
        text_lower = row_text.lower()

        if (
            "gold" in text_lower
            or "silver" in text_lower
            or any(
                heading in text_lower
                for heading in [
                    "exchange traded funds",
                    "commodity options",
                    "commodity futures",
                    "grand total"
                ]
            )
        ):
            print(f"Row {row_number}: {row_text}")

    workbook.close()

Row 1: RLMF806 | Nippon India Multi Asset Fund (An open ended scheme investing in Equity, Debt and Exchange Traded Commodity Derivatives and Gold ETF) | Index
Row 103: Exchange Traded Funds
Row 104: BMGB51ME | INF204KB17I5 | Nippon India ETF Gold Bees | 65580197 | 34613.23 | 0.1281
Row 111: GRAND TOTAL | 270284.414502 | 1
Row 125: Long | FUTCOM_SILVER_03/05/2024 | Others | 3630 | 2587.43 | 0.0096
Row 128: BENCHMARK NAME - CRISIL SHORT TERM BOND INDEX, DOMESTIC PRICE OF GOLD, MSCI WORLD INDEX TRI, S&P BSE 500 TRI


In [ ]:
# STEP 32: Identify the Multi Asset worksheet in March 2024

from openpyxl import load_workbook

march_nippon_file = nippon_folder / "NIMF-MONTHLY-PORTFOLIO-REPORT-March-24.xls"

with open(march_nippon_file, "rb") as file_stream:
    workbook = load_workbook(
        file_stream,
        read_only=True,
        data_only=True
    )

    index_sheet_name = next(
        (name for name in workbook.sheetnames
         if name.lower() == "index"),
        None
    )

    if index_sheet_name is None:
        print("No index sheet found. Worksheets:")
        print(workbook.sheetnames)
    else:
        index_sheet = workbook[index_sheet_name]

        for row_number, row in enumerate(
            index_sheet.iter_rows(values_only=True),
            start=1
        ):
            values = [value for value in row if value is not None]
            row_text = " | ".join(str(value) for value in values)

            if "multi asset" in row_text.lower():
                print(f"Row {row_number}: {row_text}")

    workbook.close()

BadZipFile: File is not a zip file

In [ ]:
# STEP 33: Check the actual format of the March 2024 file

march_nippon_file = nippon_folder / "NIMF-MONTHLY-PORTFOLIO-REPORT-March-24.xls"

with open(march_nippon_file, "rb") as file_stream:
    file_signature = file_stream.read(8)

print("File:", march_nippon_file.name)
print("First 8 bytes:", file_signature)
print("Hex signature:", file_signature.hex())

NameError: name 'nippon_folder' is not defined

In [ ]:
# STEP 33A: Restore the folder paths and verify the March file

from pathlib import Path

PROJECT_DIR = Path.cwd().parent
RAW_DATA_DIR = PROJECT_DIR / "01_RAW_DATA"
nippon_folder = RAW_DATA_DIR / "Nippon_India"

march_nippon_file = nippon_folder / "NIMF-MONTHLY-PORTFOLIO-REPORT-March-24.xls"

print("Project folder exists:", PROJECT_DIR.exists())
print("Nippon folder exists:", nippon_folder.exists())
print("March file exists:", march_nippon_file.exists())

Project folder exists: True
Nippon folder exists: True
March file exists: True


In [ ]:
# STEP 34: Check the March file signature

with open(march_nippon_file, "rb") as file_stream:
    file_signature = file_stream.read(8)

print("File:", march_nippon_file.name)
print("First 8 bytes:", file_signature)
print("Hex signature:", file_signature.hex())

File: NIMF-MONTHLY-PORTFOLIO-REPORT-March-24.xls
First 8 bytes: b'\xd0\xcf\x11\xe0\xa1\xb1\x1a\xe1'
Hex signature: d0cf11e0a1b11ae1


In [ ]:
# STEP 35: Try reading the March workbook with xlrd

import xlrd

march_workbook = xlrd.open_workbook(str(march_nippon_file))

print("March workbook worksheets:")
print(march_workbook.sheet_names())

March workbook worksheets:
['Index', 'GF', 'GS', 'IP', 'IF', 'ST', 'BF', 'MI', 'PS', 'CP', 'PH', 'ME', 'NE', 'NI', 'EO', 'LF', 'FR', 'SD', 'SE', 'SH', 'LQ', 'TS', 'LE', 'IV', 'LP', 'IM', 'DC', 'UB', 'K5', '9B', 'EA', 'QP', 'GL', 'NF', 'SC', 'SF', 'AF', 'GD', 'CE', 'CF', 'RD', 'CB', 'JE', 'SX', 'RW', 'RG', 'BP', 'ES', 'NX', 'UE', 'LT', 'NL', 'NB', 'JZ', 'PU', 'GB', 'LB', 'BB', 'SB', 'HS', 'IB', 'CC', 'ON', 'NM', 'RJ', 'G6', 'SN', 'NZ', 'MF', 'NS', 'NC', 'PP', 'AL', 'NP', 'NV', 'NY', 'NG', 'NH', 'LC', 'HG', 'ET', 'NO', 'SI', 'SO', 'HK', 'CN', 'NN', 'W5', 'ND', 'NJ', 'NK', 'NQ', 'W8', 'NU', 'NW', 'U7', 'U8', 'IT', 'IX', 'BX']


In [ ]:
# STEP 36: Find the Multi Asset fund in the March Index sheet

index_sheet = march_workbook.sheet_by_name("Index")

for row_number in range(index_sheet.nrows):
    row_values = index_sheet.row_values(row_number)
    row_text = " | ".join(
        str(value) for value in row_values if value != ""
    )

    if "multi asset" in row_text.lower():
        print(f"Row {row_number + 1}: {row_text}")

Row 69: MF | Nippon India Multi Asset Fund (An open ended scheme investing in Equity, Debt and Exchange Traded Commodity Derivatives and Gold ETF)


In [ ]:
# STEP 37: Inspect the March 2024 Multi Asset portfolio

march_sheet = march_workbook.sheet_by_name("MF")

for row_number in range(min(15, march_sheet.nrows)):
    row_values = march_sheet.row_values(row_number)
    values = [value for value in row_values if value != ""]

    if values:
        print(f"Row {row_number + 1}: {values}")

Row 1: ['RLMF806', 'Nippon India Multi Asset Fund (An open ended scheme investing in Equity, Debt and Exchange Traded Commodity Derivatives and Gold ETF)', 'Index']
Row 2: ['Monthly Portfolio Statement as on March 31,2024']
Row 4: ['ISIN', 'Name of the Instrument', 'Industry / Rating', 'Quantity', 'Market/Fair Value\n( Rs. in Lacs)', '% to NAV', 'YIELD']
Row 5: ['Derivatives']
Row 6: ['Commodity Options']
Row 7: ['SILVERN24D72000P', 'SILVER 72000 Put April 2024 Option (1 kg)', 1530.0, 5.68, '$0.00%']
Row 8: ['Total', 5.68, 0.0]
Row 9: ['Equity & Equity related']
Row 10: ['(a) Listed / awaiting listing on Stock Exchanges']
Row 11: ['HDFB03', 'INE040A01034', 'HDFC Bank Limited', 'Banks', 660000.0, 9556.14, 0.0329]
Row 12: ['IBCL05', 'INE090A01021', 'ICICI Bank Limited', 'Banks', 700000.0, 7653.1, 0.0263]
Row 13: ['LARS02', 'INE018A01030', 'Larsen & Toubro Limited', 'Construction', 160000.0, 6022.24, 0.0207]
Row 14: ['RIND01', 'INE002A01018', 'Reliance Industries Limited', 'Petroleum Prod

In [ ]:
# STEP 38: Search the March portfolio for gold and silver

for row_number in range(march_sheet.nrows):
    row_values = march_sheet.row_values(row_number)

    row_text = " | ".join(
        str(value) for value in row_values if value != ""
    )

    if "gold" in row_text.lower() or "silver" in row_text.lower():
        print(f"Row {row_number + 1}: {row_text}")

Row 1: RLMF806 | Nippon India Multi Asset Fund (An open ended scheme investing in Equity, Debt and Exchange Traded Commodity Derivatives and Gold ETF) | Index
Row 7: SILVERN24D72000P | SILVER 72000 Put April 2024 Option (1 kg) | 1530.0 | 5.68 | $0.00%
Row 109: BMGB51ME | INF204KB17I5 | Nippon India ETF Gold Bees | 66080197.0 | 37408.0 | 0.1288
Row 130: Long | FUTCOM_SILVER_03/05/2024 | Others | 4290.0 | 3219.56 | 0.0111
Row 133: BENCHMARK NAME - CRISIL SHORT TERM BOND INDEX, DOMESTIC PRICE OF GOLD, MSCI WORLD INDEX TRI, S&P BSE 500 TRI


In [ ]:
# STEP 40: Identify the Multi Asset worksheet in April 2024

from openpyxl import load_workbook

with open(april_nippon_file, "rb") as file_stream:
    april_workbook = load_workbook(
        file_stream,
        read_only=True,
        data_only=True
    )

    print("April workbook worksheets:")
    print(april_workbook.sheetnames)

    index_name = next(
        (name for name in april_workbook.sheetnames
         if name.lower() == "index"),
        None
    )

    if index_name is None:
        print("No index worksheet found.")
    else:
        index_sheet = april_workbook[index_name]

        for row_number, row in enumerate(
            index_sheet.iter_rows(values_only=True),
            start=1
        ):
            values = [value for value in row if value is not None]
            row_text = " | ".join(str(value) for value in values)

            if "multi asset" in row_text.lower():
                print(f"Row {row_number}: {row_text}")

    april_workbook.close()

April workbook worksheets:
['Index', 'GF', 'GS', 'IP', 'IF', 'ST', 'BF', 'MI', 'PS', 'CP', 'PH', 'ME', 'NE', 'NI', 'EO', 'LF', 'FR', 'SD', 'SE', 'SH', 'LQ', 'TS', 'LE', 'IV', 'LP', 'IM', 'DC', 'UB', 'K5', '9B', 'EA', 'QP', 'GL', 'NF', 'SC', 'SF', 'AF', 'GD', 'CE', 'CF', 'RD', 'CB', 'JE', 'SX', 'RW', 'RG', 'BP', 'ES', 'NX', 'UE', 'LT', 'NL', 'NB', 'JZ', 'PU', 'GB', 'LB', 'BB', 'SB', 'HS', 'IB', 'CC', 'ON', 'NM', 'RJ', 'G6', 'SN', 'NZ', 'MF', 'NS', 'NC', 'PP', 'AL', 'NP', 'NV', 'NY', 'NG', 'NH', 'LC', 'HG', 'ET', 'NO', 'SI', 'SO', 'HK', 'CN', 'NN', 'W5', 'ND', 'NJ', 'NK', 'NQ', 'W8', 'NU', 'NW', 'U7', 'U8', 'IT', 'IX', 'BX']
Row 69: MF | Nippon India Multi Asset Fund (An open ended scheme investing in Equity, Debt and Exchange Traded Commodity Derivatives and Gold ETF)


In [ ]:
# STEP 41: Inspect the April 2024 Multi Asset portfolio

with open(april_nippon_file, "rb") as file_stream:
    april_workbook = load_workbook(
        file_stream,
        read_only=True,
        data_only=True
    )

    april_sheet = april_workbook["MF"]

    for row_number, row in enumerate(
        april_sheet.iter_rows(
            min_row=1,
            max_row=15,
            values_only=True
        ),
        start=1
    ):
        values = [value for value in row if value is not None]
        if values:
            print(f"Row {row_number}: {values}")

    april_workbook.close()

Row 1: ['RLMF806', 'Nippon India Multi Asset Fund (An open ended scheme investing in Equity, Debt and Exchange Traded Commodity Derivatives and Gold ETF)', 'Index']
Row 2: ['Monthly Portfolio Statement as on April 30,2024']
Row 4: ['ISIN', 'Name of the Instrument', 'Industry / Rating', 'Quantity', 'Market/Fair Value\n( Rs. in Lacs)', '% to NAV', 'YIELD']
Row 5: ['Derivatives']
Row 6: ['Commodity Options']
Row 7: ['SILVERN24D72000P', 'CRUDEOIL 6800 Put May 2024 Option (1 kg)', 50000, 65.15, 0.0002]
Row 8: ['', 'Total', '', '', 65.15, 0.0002]
Row 10: ['Equity & Equity related']
Row 11: ['(a) Listed / awaiting listing on Stock Exchanges']
Row 12: ['HDFB03', 'INE040A01034', 'HDFC Bank Limited', 'Banks', 725000, 11020.73, 0.0353]
Row 13: ['IBCL05', 'INE090A01021', 'ICICI Bank Limited', 'Banks', 850000, 9778.4, 0.0313]
Row 14: ['RIND01', 'INE002A01018', 'Reliance Industries Limited', 'Petroleum Products', 200000, 5868, 0.0188]
Row 15: ['TELC04', 'IN9155A01020', 'Tata Motors Limited (DVR Shar

In [ ]:
# STEP 42: Search the April portfolio for gold and silver references

with open(april_nippon_file, "rb") as file_stream:
    april_workbook = load_workbook(
        file_stream,
        read_only=True,
        data_only=True
    )

    april_sheet = april_workbook["MF"]

    for row_number, row in enumerate(
        april_sheet.iter_rows(values_only=True),
        start=1
    ):
        values = [value for value in row if value is not None]
        if not values:
            continue

        row_text = " | ".join(str(value) for value in values)

        if "gold" in row_text.lower() or "silver" in row_text.lower():
            print(f"Row {row_number}: {row_text}")

    april_workbook.close()

Row 1: RLMF806 | Nippon India Multi Asset Fund (An open ended scheme investing in Equity, Debt and Exchange Traded Commodity Derivatives and Gold ETF) | Index
Row 7: SILVERN24D72000P | CRUDEOIL 6800 Put May 2024 Option (1 kg) | 50000 | 65.15 | 0.0002
Row 114: BMGB51ME | INF204KB17I5 | Nippon India ETF Gold Bees | 67580197 | 41142.82 | 0.1319
Row 134: Long | FUTCOM_SILVER_05/07/2024 | Others | 4770 | 3856.59 | 0.0124
Row 136: Long | FUTCOM_GOLD_05/06/2024 | Others | 200 | 140.83 | 0.0005
Row 138: BENCHMARK NAME - CRISIL SHORT TERM BOND INDEX, DOMESTIC PRICE OF GOLD, MSCI WORLD INDEX TRI, S&P BSE 500 TRI


In [ ]:
# STEP 43: Find the Nippon May 2024 workbook

for file_path in sorted(nippon_folder.iterdir()):
    if file_path.is_file() and "may" in file_path.name.lower():
        print(file_path.name)

NIMF-MONTHLY-PORTFOLIO-31-May-25.xls
NIMF-MONTHLY-PORTFOLIO-31-May-26.xls
NIMF-MONTHLY-PORTFOLIO-May-24.xls


In [ ]:
# STEP 44: Identify the May 2024 workbook format and sheets

from zipfile import ZipFile, BadZipFile
from openpyxl import load_workbook
import xlrd

may_nippon_file = nippon_folder / "NIMF-MONTHLY-PORTFOLIO-May-24.xls"

try:
    with ZipFile(may_nippon_file) as z:
        print("Internal format: XLSX")

    with open(may_nippon_file, "rb") as file_stream:
        may_workbook = load_workbook(
            file_stream,
            read_only=True,
            data_only=True
        )
        print("Sheet names:", may_workbook.sheetnames)
        may_workbook.close()

except BadZipFile:
    print("Internal format: Legacy XLS")

    may_workbook = xlrd.open_workbook(str(may_nippon_file))
    print("Sheet names:", may_workbook.sheet_names())

Internal format: XLSX
Sheet names: ['Index', 'GF', 'GS', 'IP', 'IF', 'ST', 'BF', 'MI', 'PS', 'CP', 'PH', 'ME', 'NE', 'NI', 'EO', 'LF', 'FR', 'SD', 'SE', 'SH', 'LQ', 'TS', 'LE', 'IV', 'LP', 'IM', 'DC', 'UB', 'K5', '9B', 'EA', 'QP', 'GL', 'NF', 'SC', 'SF', 'AF', 'GD', 'CE', 'CF', 'RD', 'CB', 'JE', 'SX', 'RW', 'RG', 'BP', 'ES', 'NX', 'UE', 'LT', 'NL', 'NB', 'JZ', 'PU', 'GB', 'LB', 'BB', 'SB', 'HS', 'IB', 'CC', 'ON', 'NM', 'RJ', 'G6', 'SN', 'NZ', 'MF', 'NS', 'NC', 'PP', 'AL', 'NP', 'NV', 'NY', 'NG', 'NH', 'LC', 'HG', 'ET', 'NO', 'SI', 'SO', 'HK', 'CN', 'NN', 'W5', 'ND', 'NJ', 'NK', 'NQ', 'W8', 'NU', 'NW', 'U7', 'U8', 'IT', 'IX', 'BX']


In [ ]:
# STEP 45: Inspect the May 2024 MF worksheet

with open(may_nippon_file, "rb") as file_stream:
    may_workbook = load_workbook(
        file_stream,
        read_only=True,
        data_only=True
    )

    may_sheet = may_workbook["MF"]

    for row_number, row in enumerate(
        may_sheet.iter_rows(
            min_row=1,
            max_row=10,
            values_only=True
        ),
        start=1
    ):
        values = [value for value in row if value is not None]
        print(f"Row {row_number}: {values}")

    may_workbook.close()

Row 1: ['RLMF806', 'Nippon India Multi Asset Fund (An open ended scheme investing in Equity, Debt and Exchange Traded Commodity Derivatives and Gold ETF)', 'Index']
Row 2: ['Monthly Portfolio Statement as on May 31,2024']
Row 3: []
Row 4: ['ISIN', 'Name of the Instrument', 'Industry / Rating', 'Quantity', 'Market/Fair Value\n( Rs. in Lacs)', '% to NAV', 'YIELD']
Row 5: ['Derivatives']
Row 6: ['Commodity Options']
Row 7: ['SILVER 85000 Put June 2024 Option (1 kg)', 4980, 19.67, 0.0001]
Row 8: ['Total', 19.67, 0.0001]
Row 9: []
Row 10: ['Equity & Equity related']


In [ ]:
# STEP 46: Search the May 2024 portfolio for gold and silver

with open(may_nippon_file, "rb") as file_stream:
    may_workbook = load_workbook(
        file_stream,
        read_only=True,
        data_only=True
    )

    may_sheet = may_workbook["MF"]

    for row_number, row in enumerate(
        may_sheet.iter_rows(values_only=True),
        start=1
    ):
        values = [value for value in row if value is not None]

        if not values:
            continue

        row_text = " | ".join(str(value) for value in values)

        if "gold" in row_text.lower() or "silver" in row_text.lower():
            print(f"Row {row_number}: {row_text}")

    may_workbook.close()

Row 1: RLMF806 | Nippon India Multi Asset Fund (An open ended scheme investing in Equity, Debt and Exchange Traded Commodity Derivatives and Gold ETF) | Index
Row 7: SILVER 85000 Put June 2024 Option (1 kg) | 4980 | 19.67 | 0.0001
Row 119: BMGB51ME | INF204KB17I5 | Nippon India ETF Gold Bees | 67580197 | 41419.9 | 0.125
Row 140: Long | FUTCOM_SILVER_05/07/2024 | Others | 5820 | 5329.37 | 0.0161
Row 141: Long | FUTCOM_GOLD_05/08/2024 | Others | 4100 | 2945.19 | 0.0089
Row 145: BENCHMARK NAME - 50% OF BSE 500 TRI, 20% OF MSCI WORLD INDEX TRI, 15% OF CRISIL SHORT TERM BOND INDEX & 15% OF DOMESTIC PRICES OF GOLD


In [ ]:
# STEP 47: Find the Nippon June workbook

for file_path in sorted(nippon_folder.iterdir()):
    filename = file_path.name.lower()

    if file_path.is_file() and ("june" in filename or "jun-" in filename):
        print(file_path.name)

NIMF-MONTHLY-PORTFOLIO-30-Jun-26.xls
NIMF-MONTHLY-PORTFOLIO-30-June-25.xls
NIMF-MONTHLY-PORTFOLIO-June-24.xls


In [ ]:
# STEP 48: Identify the June 2024 workbook format and sheets

from zipfile import ZipFile, BadZipFile
from openpyxl import load_workbook
import xlrd

june_nippon_file = nippon_folder / "NIMF-MONTHLY-PORTFOLIO-June-24.xls"

try:
    with ZipFile(june_nippon_file):
        print("Internal format: XLSX")

    with open(june_nippon_file, "rb") as file_stream:
        june_workbook = load_workbook(
            file_stream,
            read_only=True,
            data_only=True
        )
        print("Sheet names:", june_workbook.sheetnames)
        june_workbook.close()

except BadZipFile:
    print("Internal format: Legacy XLS")

    june_workbook = xlrd.open_workbook(str(june_nippon_file))
    print("Sheet names:", june_workbook.sheet_names())

Internal format: XLSX
Sheet names: ['Index', 'GF', 'GS', 'IP', 'IF', 'ST', 'BF', 'MI', 'PS', 'CP', 'PH', 'ME', 'NE', 'NI', 'EO', 'LF', 'FR', 'SD', 'SE', 'SH', 'LQ', 'TS', 'LE', 'IV', 'LP', 'IM', 'DC', 'UB', 'K5', '9B', 'EA', 'QP', 'GL', 'NF', 'SC', 'SF', 'AF', 'GD', 'CE', 'CF', 'RD', 'CB', 'JE', 'SX', 'RW', 'RG', 'BP', 'ES', 'NX', 'UE', 'LT', 'NL', 'NB', 'JZ', 'PU', 'GB', 'LB', 'BB', 'SB', 'HS', 'IB', 'CC', 'ON', 'NM', 'RJ', 'G6', 'SN', 'NZ', 'MF', 'NS', 'NC', 'PP', 'AL', 'NP', 'NV', 'NY', 'NG', 'NH', 'LC', 'HG', 'ET', 'NO', 'SI', 'SO', 'HK', 'CN', 'NN', 'W5', 'ND', 'NJ', 'NK', 'NQ', 'W8', 'NU', 'NW', 'U7', 'U8', 'IT', 'IX', 'BX']


In [ ]:
# STEP 49: Inspect the June 2024 MF worksheet

with open(june_nippon_file, "rb") as file_stream:
    june_workbook = load_workbook(
        file_stream,
        read_only=True,
        data_only=True
    )

    june_sheet = june_workbook["MF"]

    for row_number, row in enumerate(
        june_sheet.iter_rows(
            min_row=1,
            max_row=10,
            values_only=True
        ),
        start=1
    ):
        values = [value for value in row if value is not None]
        print(f"Row {row_number}: {values}")

    june_workbook.close()

Row 1: ['RLMF806', 'Nippon India Multi Asset Fund (An open ended scheme investing in Equity, Debt and Exchange Traded Commodity Derivatives and Gold ETF)', 'Index']
Row 2: ['Monthly Portfolio Statement as on June 30,2024']
Row 3: []
Row 4: ['ISIN', 'Name of the Instrument', 'Industry / Rating', 'Quantity', 'Market/Fair Value\n( Rs. in Lacs)', '% to NAV', 'YIELD']
Row 5: ['Equity & Equity related']
Row 6: ['(a) Listed / awaiting listing on Stock Exchanges']
Row 7: ['HDFB03', 'INE040A01034', 'HDFC Bank Limited', 'Banks', 685000, 11534.03, 0.032]
Row 8: ['IBCL05', 'INE090A01021', 'ICICI Bank Limited', 'Banks', 850000, 10196.6, 0.0283]
Row 9: ['LARS02', 'INE018A01030', 'Larsen & Toubro Limited', 'Construction', 203000, 7203.35, 0.02]
Row 10: ['INFS02', 'INE009A01021', 'Infosys Limited', 'IT - Software', 450000, 7050.38, 0.0196]


In [ ]:
# STEP 50: Search the June 2024 portfolio for gold and silver

with open(june_nippon_file, "rb") as file_stream:
    june_workbook = load_workbook(
        file_stream,
        read_only=True,
        data_only=True
    )

    june_sheet = june_workbook["MF"]

    for row_number, row in enumerate(
        june_sheet.iter_rows(values_only=True),
        start=1
    ):
        values = [value for value in row if value is not None]

        if not values:
            continue

        row_text = " | ".join(str(value) for value in values)

        if "gold" in row_text.lower() or "silver" in row_text.lower():
            print(f"Row {row_number}: {row_text}")

    june_workbook.close()

Row 1: RLMF806 | Nippon India Multi Asset Fund (An open ended scheme investing in Equity, Debt and Exchange Traded Commodity Derivatives and Gold ETF) | Index
Row 117: BMGB51ME | INF204KB17I5 | Nippon India ETF Gold Bees | 67580197 | 40973.87 | 0.1138
Row 137: Long | FUTCOM_SILVER_05/09/2024 | Others | 8550 | 7655.67 | 0.0213
Row 138: Long | FUTCOM_GOLD_05/08/2024 | Others | 10300 | 7372.95 | 0.0205
Row 141: BENCHMARK NAME - 50% OF BSE 500 TRI, 20% OF MSCI WORLD INDEX TRI, 15% OF CRISIL SHORT TERM BOND INDEX & 15% OF DOMESTIC PRICES OF GOLD


In [ ]:
# STEP 51: Find the Nippon July workbook

for file_path in sorted(nippon_folder.iterdir()):
    filename = file_path.name.lower()

    if file_path.is_file() and (
        "july" in filename or "jul-" in filename
    ):
        print(file_path.name)

NIMF-MONTHLY-PORTFOLIO-31-July-24.xls
NIMF-MONTHLY-PORTFOLIO-31-July-25.xls
NIMF-MONTHLY-PORTFOLIO-31-July-26.xls


In [ ]:
# STEP 52: Identify the July 2024 workbook format and sheets

from zipfile import ZipFile, BadZipFile
from openpyxl import load_workbook
import xlrd

july_nippon_file = nippon_folder / "NIMF-MONTHLY-PORTFOLIO-31-July-24.xls"

try:
    with ZipFile(july_nippon_file):
        print("Internal format: XLSX")

    with open(july_nippon_file, "rb") as file_stream:
        july_workbook = load_workbook(
            file_stream,
            read_only=True,
            data_only=True
        )
        print("Sheet names:", july_workbook.sheetnames)
        july_workbook.close()

except BadZipFile:
    print("Internal format: Legacy XLS")

    july_workbook = xlrd.open_workbook(str(july_nippon_file))
    print("Sheet names:", july_workbook.sheet_names())

Internal format: XLSX
Sheet names: ['INDEX', 'GF', 'GS', 'IP', 'IF', 'ST', 'BF', 'MI', 'PS', 'CP', 'PH', 'ME', 'NE', 'NI', 'EO', 'LF', 'FR', 'SD', 'SE', 'SH', 'LQ', 'TS', 'LE', 'IV', 'LP', 'IM', 'DC', 'UB', 'K5', '9B', 'EA', 'QP', 'GL', 'NF', 'SC', 'SF', 'AF', 'GD', 'CE', 'CF', 'RD', 'CB', 'JE', 'SX', 'RW', 'RG', 'BP', 'ES', 'NX', 'UE', 'LT', 'NL', 'NB', 'JZ', 'PU', 'GB', 'LB', 'BB', 'SB', 'HS', 'IB', 'CC', 'ON', 'NM', 'RJ', 'G6', 'SN', 'NZ', 'MF', 'NS', 'NC', 'PP', 'AL', 'NP', 'NV', 'NY', 'NG', 'NH', 'LC', 'HG', 'ET', 'NO', 'SI', 'SO', 'HK', 'CN', 'NN', 'W5', 'ND', 'NJ', 'NK', 'NQ', 'W8', 'NU', 'NW', 'U7', 'U8', 'IT', 'IX', 'BX']


In [ ]:
# STEP 53: Inspect the July 2024 MF worksheet

with open(july_nippon_file, "rb") as file_stream:
    july_workbook = load_workbook(
        file_stream,
        read_only=True,
        data_only=True
    )

    july_sheet = july_workbook["MF"]

    for row_number, row in enumerate(
        july_sheet.iter_rows(
            min_row=1,
            max_row=10,
            values_only=True
        ),
        start=1
    ):
        values = [value for value in row if value is not None]
        print(f"Row {row_number}: {values}")

    july_workbook.close()

Row 1: ['RLMF806', 'Nippon India Multi Asset Allocation Fund (An open ended scheme investing in Equity, Debt \nand Exchange Traded Commodity Derivatives and \nGold ETF & Silver ETF)', 'Index']
Row 2: ['Monthly Portfolio Statement as on July 31,2024']
Row 3: []
Row 4: ['ISIN', 'Name of the Instrument', 'Industry / Rating', 'Quantity', 'Market/Fair Value\n( Rs. in Lacs)', '% to NAV', 'YIELD']
Row 5: ['Equity & Equity related']
Row 6: ['(a) Listed / awaiting listing on Stock Exchanges']
Row 7: ['HDFB03', 'INE040A01034', 'HDFC Bank Limited', 'Banks', 740000, 11956.55, 0.031]
Row 8: ['IBCL05', 'INE090A01021', 'ICICI Bank Limited', 'Banks', 850000, 10326.65, 0.0268]
Row 9: ['INFS02', 'INE009A01021', 'Infosys Limited', 'IT - Software', 510000, 9528.08, 0.0247]
Row 10: ['LARS02', 'INE018A01030', 'Larsen & Toubro Limited', 'Construction', 203000, 7744.45, 0.0201]


In [ ]:
# STEP 54: Search the July 2024 portfolio for gold and silver

with open(july_nippon_file, "rb") as file_stream:
    july_workbook = load_workbook(
        file_stream,
        read_only=True,
        data_only=True
    )

    july_sheet = july_workbook["MF"]

    for row_number, row in enumerate(
        july_sheet.iter_rows(values_only=True),
        start=1
    ):
        values = [value for value in row if value is not None]

        if not values:
            continue

        row_text = " | ".join(str(value) for value in values)

        if "gold" in row_text.lower() or "silver" in row_text.lower():
            print(f"Row {row_number}: {row_text}")

    july_workbook.close()

Row 1: RLMF806 | Nippon India Multi Asset Allocation Fund (An open ended scheme investing in Equity, Debt 
and Exchange Traded Commodity Derivatives and 
Gold ETF & Silver ETF) | Index
Row 120: BMGB51ME | INF204KB17I5 | Nippon India ETF Gold Bees | 67580197 | 39656.06 | 0.1029
Row 121: 149758 | INF204KC1402 | Nippon India Silver ETF | 1500000 | 1227.6 | 0.0032
Row 137: Long | FUTCOM_GOLD_04/10/2024 | Others | 17600 | 12259.28 | 0.0318
Row 138: Long | FUTCOM_SILVER_05/09/2024 | Others | 8400 | 7022.06 | 0.0182
Row 147: BENCHMARK NAME - 50% OF BSE 500 TRI, 20% OF MSCI WORLD INDEX TRI, 15% OF CRISIL SHORT TERM BOND INDEX 10% OF DOMESTIC PRICES OF GOLD & 5% OF DOMESTIC PRICES OF SILVER


In [ ]:
# STEP 55: Find the Nippon August workbook

for file_path in sorted(nippon_folder.iterdir()):
    filename = file_path.name.lower()

    if file_path.is_file() and (
        "august" in filename or "aug-" in filename
    ):
        print(file_path.name)

NIMF-MONTHLY-PORTFOLIO-31-Aug-25.xls
NIMF-MONTHLY-PORTFOLIO-31-Aug-26.xls
NIMF_MONTHLY_PORTFOLIO_31-Aug-24.xls


In [ ]:
# STEP 56: Identify the August 2024 workbook format and sheets

from zipfile import ZipFile, BadZipFile
from openpyxl import load_workbook
import xlrd

august_nippon_file = nippon_folder / "NIMF_MONTHLY_PORTFOLIO_31-Aug-24.xls"

try:
    with ZipFile(august_nippon_file):
        print("Internal format: XLSX")

    with open(august_nippon_file, "rb") as file_stream:
        august_workbook = load_workbook(
            file_stream,
            read_only=True,
            data_only=True
        )
        print("Sheet names:", august_workbook.sheetnames)
        august_workbook.close()

except BadZipFile:
    print("Internal format: Legacy XLS")

    august_workbook = xlrd.open_workbook(str(august_nippon_file))
    print("Sheet names:", august_workbook.sheet_names())

Internal format: Legacy XLS
Sheet names: ['INDEX', 'GF', 'GS', 'IP', 'IF', 'ST', 'BF', 'MI', 'PS', 'CP', 'PH', 'ME', 'NE', 'NI', 'EO', 'LF', 'FR', 'SD', 'SE', 'SH', 'LQ', 'TS', 'LE', 'IV', 'LP', 'IM', 'DC', 'UB', 'K5', '9B', 'EA', 'QP', 'GL', 'NF', 'SC', 'SF', 'AF', 'GD', 'CE', 'CF', 'RD', 'CB', 'JE', 'SX', 'RW', 'RG', 'BP', 'ES', 'NX', 'UE', 'LT', 'NL', 'NB', 'JZ', 'PU', 'GB', 'LB', 'BB', 'SB', 'HS', 'IB', 'CC', 'ON', 'NM', 'RJ', 'G6', 'SN', 'NZ', 'MF', 'NS', 'NC', 'PP', 'AL', 'NP', 'NV', 'NY', 'NG', 'NH', 'LC', 'HG', 'ET', 'NO', 'SI', 'SO', 'HK', 'CN', 'NN', 'W5', 'ND', 'NJ', 'NK', 'NQ', 'W8', 'NU', 'NW', 'U7', 'U8', 'IT', 'IX', 'BX']


In [ ]:
# STEP 57: Inspect the August 2024 MF worksheet

august_workbook = xlrd.open_workbook(str(august_nippon_file))
august_sheet = august_workbook.sheet_by_name("MF")

for row_number in range(min(10, august_sheet.nrows)):
    values = [
        august_sheet.cell_value(row_number, col_number)
        for col_number in range(august_sheet.ncols)
    ]

    values = [value for value in values if value != ""]
    print(f"Row {row_number + 1}: {values}")

Row 1: ['RLMF806', 'Nippon India Multi Asset Allocation Fund (An open ended scheme investing in Equity, Debt \nand Exchange Traded Commodity Derivatives and \nGold ETF & Silver ETF)', 'Index']
Row 2: ['Monthly Portfolio Statement as on August 31,2024']
Row 3: []
Row 4: ['ISIN', 'Name of the Instrument', 'Industry / Rating', 'Quantity', 'Market/Fair Value\n( Rs. in Lacs)', '% to NAV', 'YIELD']
Row 5: ['Equity & Equity related']
Row 6: ['(a) Listed / awaiting listing on Stock Exchanges']
Row 7: ['HDFB03', 'INE040A01034', 'HDFC Bank Limited', 'Banks', 700000.0, 11458.3, 0.028]
Row 8: ['IBCL05', 'INE090A01021', 'ICICI Bank Limited', 'Banks', 850000.0, 10448.2, 0.0255]
Row 9: ['INFS02', 'INE009A01021', 'Infosys Limited', 'IT - Software', 510000.0, 9912.87, 0.0242]
Row 10: ['RIND01', 'INE002A01018', 'Reliance Industries Limited', 'Petroleum Products', 300000.0, 9057.75, 0.0221]


In [ ]:
# STEP 58: Search the August 2024 portfolio for gold and silver

for row_number in range(august_sheet.nrows):
    values = [
        august_sheet.cell_value(row_number, col_number)
        for col_number in range(august_sheet.ncols)
    ]

    values = [value for value in values if value != ""]

    if not values:
        continue

    row_text = " | ".join(str(value) for value in values)

    if "gold" in row_text.lower() or "silver" in row_text.lower():
        print(f"Row {row_number + 1}: {row_text}")

Row 1: RLMF806 | Nippon India Multi Asset Allocation Fund (An open ended scheme investing in Equity, Debt 
and Exchange Traded Commodity Derivatives and 
Gold ETF & Silver ETF) | Index
Row 127: BMGB51ME | INF204KB17I5 | Nippon India ETF Gold Bees | 67580197.0 | 40994.15 | 0.1001
Row 128: 149758 | INF204KC1402 | Nippon India Silver ETF | 9516456.0 | 7905.32 | 0.0193
Row 144: Long | FUTCOM_GOLD_04/10/2024 | Others | 20800.0 | 14895.09 | 0.0364
Row 146: Long | FUTCOM_SILVER_05/12/2024 | Others | 1200.0 | 1022.52 | 0.0025
Row 155: BENCHMARK NAME - 50% OF BSE 500 TRI, 20% OF MSCI WORLD INDEX TRI, 15% OF CRISIL SHORT TERM BOND INDEX 10% OF DOMESTIC PRICES OF GOLD & 5% OF DOMESTIC PRICES OF SILVER


In [ ]:
# STEP 59: Find the Nippon September workbook

for file_path in sorted(nippon_folder.iterdir()):
    filename = file_path.name.lower()

    if file_path.is_file() and (
        "sep-" in filename or "september" in filename
    ):
        print(file_path.name)

NIMF-MONTHLY-PORTFOLIO-30-Sep-24.xls
NIMF-MONTHLY-PORTFOLIO-30-Sep-25.xls
NIMF-MONTHLY-PORTFOLIO-30-Sep-26.xls


In [ ]:
# STEP 60: Identify the September 2024 workbook format and sheets

from zipfile import ZipFile, BadZipFile
from openpyxl import load_workbook
import xlrd

september_nippon_file = nippon_folder / "NIMF-MONTHLY-PORTFOLIO-30-Sep-24.xls"

try:
    with ZipFile(september_nippon_file):
        print("Internal format: XLSX")

    with open(september_nippon_file, "rb") as file_stream:
        september_workbook = load_workbook(
            file_stream,
            read_only=True,
            data_only=True
        )
        print("Sheet names:", september_workbook.sheetnames)
        september_workbook.close()

except BadZipFile:
    print("Internal format: Legacy XLS")

    september_workbook = xlrd.open_workbook(str(september_nippon_file))
    print("Sheet names:", september_workbook.sheet_names())

Internal format: Legacy XLS
Sheet names: ['INDEX', 'GF', 'GS', 'IP', 'IF', 'ST', 'BF', 'MI', 'PS', 'CP', 'PH', 'ME', 'NE', 'NI', 'EO', 'LF', 'FR', 'SD', 'SE', 'SH', 'LQ', 'TS', 'LE', 'IV', 'LP', 'IM', 'DC', 'UB', 'K5', '9B', 'EA', 'QP', 'GL', 'NF', 'SC', 'SF', 'AF', 'GD', 'CE', 'CF', 'RD', 'CB', 'JE', 'SX', 'RW', 'RG', 'BP', 'ES', 'NX', 'UE', 'LT', 'NL', 'NB', 'JZ', 'PU', 'GB', 'LB', 'BB', 'SB', 'HS', 'IB', 'CC', 'ON', 'NM', 'RJ', 'G6', 'SN', 'NZ', 'MF', 'NS', 'NC', 'PP', 'AL', 'NP', 'NV', 'NY', 'NG', 'NH', 'LC', 'HG', 'ET', 'NO', 'SI', 'SO', 'HK', 'CN', 'NN', 'W5', 'ND', 'NJ', 'NK', 'NQ', 'W8', 'NU', 'NW', 'U7', 'U8', 'IT', 'IX', 'BX', 'EW', 'MT']


In [ ]:
# STEP 61: Inspect the September 2024 MF worksheet

september_workbook = xlrd.open_workbook(str(september_nippon_file))
september_sheet = september_workbook.sheet_by_name("MF")

for row_number in range(min(10, september_sheet.nrows)):
    values = [
        september_sheet.cell_value(row_number, col_number)
        for col_number in range(september_sheet.ncols)
    ]

    values = [value for value in values if value != ""]
    print(f"Row {row_number + 1}: {values}")

Row 1: ['RLMF806', 'Nippon India Multi Asset Allocation Fund (An open ended scheme investing in Equity, Debt \nand Exchange Traded Commodity Derivatives and \nGold ETF & Silver ETF)', 'Index']
Row 2: ['Monthly Portfolio Statement as on September 30,2024']
Row 3: []
Row 4: ['ISIN', 'Name of the Instrument', 'Industry / Rating', 'Quantity', 'Market/Fair Value\n( Rs. in Lacs)', '% to NAV', 'YIELD']
Row 5: ['Equity & Equity related']
Row 6: ['(a) Listed / awaiting listing on Stock Exchanges']
Row 7: ['HDFB03', 'INE040A01034', 'HDFC Bank Limited', 'Banks', 700000.0, 12124.35, 0.0279]
Row 8: ['IBCL05', 'INE090A01021', 'ICICI Bank Limited', 'Banks', 850000.0, 10820.5, 0.0249]
Row 9: ['INFS02', 'INE009A01021', 'Infosys Limited', 'IT - Software', 510000.0, 9565.56, 0.022]
Row 10: ['RIND01', 'INE002A01018', 'Reliance Industries Limited', 'Petroleum Products', 300000.0, 8859.45, 0.0204]


In [ ]:
# STEP 62: Search the September 2024 portfolio for gold and silver

for row_number in range(september_sheet.nrows):
    values = [
        september_sheet.cell_value(row_number, col_number)
        for col_number in range(september_sheet.ncols)
    ]

    values = [value for value in values if value != ""]

    if not values:
        continue

    row_text = " | ".join(str(value) for value in values)

    if "gold" in row_text.lower() or "silver" in row_text.lower():
        print(f"Row {row_number + 1}: {row_text}")

Row 1: RLMF806 | Nippon India Multi Asset Allocation Fund (An open ended scheme investing in Equity, Debt 
and Exchange Traded Commodity Derivatives and 
Gold ETF & Silver ETF) | Index
Row 130: BMGB51ME | INF204KB17I5 | Nippon India ETF Gold Bees | 67580197.0 | 42947.22 | 0.0989
Row 131: 149758 | INF204KC1402 | Nippon India Silver ETF | 9716456.0 | 8554.37 | 0.0197
Row 147: Long | FUTCOM_GOLD_05/12/2024 | Others | 21300.0 | 16105.14 | 0.0371
Row 148: Long | FUTCOM_SILVER_05/12/2024 | Others | 3000.0 | 2721.57 | 0.0063
Row 156: BENCHMARK NAME - 50% OF BSE 500 TRI, 20% OF MSCI WORLD INDEX TRI, 15% OF CRISIL SHORT TERM BOND INDEX 10% OF DOMESTIC PRICES OF GOLD & 5% OF DOMESTIC PRICES OF SILVER


In [ ]:
# STEP 63: Find the Nippon October workbook

for file_path in sorted(nippon_folder.iterdir()):
    filename = file_path.name.lower()

    if file_path.is_file() and "oct" in filename:
        print(file_path.name)

NIMF-MONTHLY-PORTFOLIO-31-Oct-24.xls
NIMF-MONTHLY-PORTFOLIO-31-Oct-25.xls


In [ ]:
# STEP 64: Identify the October 2024 workbook format and sheets

from zipfile import ZipFile, BadZipFile
from openpyxl import load_workbook
import xlrd

october_nippon_file = nippon_folder / "NIMF-MONTHLY-PORTFOLIO-31-Oct-24.xls"

try:
    with ZipFile(october_nippon_file):
        print("Internal format: XLSX")

    with open(october_nippon_file, "rb") as file_stream:
        october_workbook = load_workbook(
            file_stream,
            read_only=True,
            data_only=True
        )
        print("Sheet names:", october_workbook.sheetnames)
        october_workbook.close()

except BadZipFile:
    print("Internal format: Legacy XLS")

    october_workbook = xlrd.open_workbook(str(october_nippon_file))
    print("Sheet names:", october_workbook.sheet_names())

Internal format: XLSX
Sheet names: ['INDEX', 'GF', 'GS', 'IP', 'IF', 'ST', 'BF', 'MI', 'PS', 'CP', 'PH', 'ME', 'NE', 'NI', 'EO', 'LF', 'FR', 'SD', 'SE', 'SH', 'LQ', 'TS', 'LE', 'IV', 'LP', 'IM', 'DC', 'UB', 'K5', '9B', 'EA', 'QP', 'GL', 'NF', 'SC', 'SF', 'AF', 'GD', 'CE', 'CF', 'RD', 'CB', 'JE', 'SX', 'RW', 'RG', 'BP', 'ES', 'NX', 'UE', 'LT', 'NL', 'NB', 'JZ', 'PU', 'GB', 'LB', 'BB', 'SB', 'HS', 'IB', 'CC', 'ON', 'NM', 'RJ', 'G6', 'SN', 'NZ', 'MF', 'NS', 'PP', 'AL', 'NP', 'NV', 'NY', 'NG', 'NH', 'LC', 'HG', 'ET', 'NO', 'SI', 'SO', 'HK', 'CN', 'NN', 'W5', 'ND', 'NJ', 'NK', 'NQ', 'W8', 'NU', 'NW', 'U7', 'U8', 'IT', 'IX', 'BX', 'EW', 'MT', 'IJ', 'ID']


In [ ]:
# STEP 65: Inspect the October 2024 MF worksheet

with open(october_nippon_file, "rb") as file_stream:
    october_workbook = load_workbook(
        file_stream,
        read_only=True,
        data_only=True
    )

    october_sheet = october_workbook["MF"]

    for row_number, row in enumerate(
        october_sheet.iter_rows(
            min_row=1,
            max_row=10,
            values_only=True
        ),
        start=1
    ):
        values = [value for value in row if value is not None]
        print(f"Row {row_number}: {values}")

    october_workbook.close()

Row 1: ['RLMF806', 'Nippon India Multi Asset Allocation Fund (An open ended scheme investing in Equity, Debt \nand Exchange Traded Commodity Derivatives and \nGold ETF & Silver ETF)', 'Index']
Row 2: ['Monthly Portfolio Statement as on October 31,2024']
Row 3: []
Row 4: ['ISIN', 'Name of the Instrument', 'Industry / Rating', 'Quantity', 'Market/Fair Value\n( Rs. in Lacs)', '% to NAV', 'YIELD']
Row 5: ['Derivatives']
Row 6: ['Commodity Options']
Row 7: ['GOLD 75000 Put November 2024 Option (10 Gm)', 19000, 43.99, 0.0001]
Row 8: ['SILVER 90000 Put November 2024 Option (1 kg)', 2460, 29.63, 0.0001]
Row 9: ['Total', 73.62, 0.0002]
Row 10: []


In [ ]:
# STEP 66: Search October 2024 holdings for gold and silver

with open(october_nippon_file, "rb") as file_stream:
    october_workbook = load_workbook(
        file_stream,
        read_only=True,
        data_only=True
    )

    october_sheet = october_workbook["MF"]

    for row_number, row in enumerate(
        october_sheet.iter_rows(values_only=True),
        start=1
    ):
        row_values = [
            str(value) if value is not None else ""
            for value in row
        ]

        combined_text = " ".join(row_values).lower()

        if "gold" in combined_text or "silver" in combined_text:
            print(f"Row {row_number}: {row_values}")

    october_workbook.close()

Row 1: ['RLMF806', 'Nippon India Multi Asset Allocation Fund (An open ended scheme investing in Equity, Debt \nand Exchange Traded Commodity Derivatives and \nGold ETF & Silver ETF)', '', '', '', '', '', 'Index', '', '']
Row 7: ['', '', 'GOLD 75000 Put November 2024 Option (10 Gm)', '', '19000', '43.99', '0.0001', '', '', '']
Row 8: ['', '', 'SILVER 90000 Put November 2024 Option (1 kg)', '', '2460', '29.63', '0.0001', '', '', '']
Row 140: ['BMGB51ME', 'INF204KB17I5', 'Nippon India ETF Gold Bees', '', '67580197', '45224.67', '0.1005', '', '', '']
Row 141: ['149758', 'INF204KC1402', 'Nippon India Silver ETF', '', '9716456', '9159.7', '0.0204', '', '', '']
Row 157: ['', 'Long', 'FUTCOM_GOLD_05/12/2024', 'Others', '24600', '19293.78', '0.0429', '', '', '']
Row 158: ['', 'Long', 'FUTCOM_SILVER_05/12/2024', 'Others', '3870', '3662.22', '0.0081', '', '', '']
Row 166: ['', '', '', '', '', 'BENCHMARK NAME - 50% OF BSE 500 TRI, 20% OF MSCI WORLD INDEX TRI, 15% OF CRISIL SHORT TERM BOND INDEX 10

In [ ]:
# STEP 70: Identify the November 2024 workbook format

november_2024_file = nippon_folder / "NIMF-MONTHLY-PORTFOLIO-30-Nov-2024.xls"

try:
    with ZipFile(november_2024_file):
        print("Internal format: XLSX")

    with open(november_2024_file, "rb") as file_stream:
        november_workbook = load_workbook(
            file_stream,
            read_only=True,
            data_only=True
        )
        print("Sheet names:", november_workbook.sheetnames)
        november_workbook.close()

except BadZipFile:
    print("Internal format: Legacy XLS")

    november_workbook = xlrd.open_workbook(str(november_2024_file))
    print("Sheet names:", november_workbook.sheet_names())

Internal format: Legacy XLS
Sheet names: ['INDEX', 'GF', 'GS', 'IP', 'IF', 'ST', 'BF', 'MI', 'PS', 'CP', 'PH', 'ME', 'NE', 'NI', 'EO', 'LF', 'FR', 'SD', 'SE', 'SH', 'LQ', 'TS', 'LE', 'IV', 'LP', 'IM', 'DC', 'UB', 'K5', '9B', 'EA', 'QP', 'GL', 'NF', 'SC', 'SF', 'AF', 'GD', 'CE', 'CF', 'RD', 'CB', 'JE', 'SX', 'RW', 'RG', 'BP', 'ES', 'NX', 'UE', 'LT', 'NL', 'NB', 'JZ', 'PU', 'GB', 'LB', 'BB', 'SB', 'HS', 'IB', 'CC', 'ON', 'NM', 'RJ', 'G6', 'SN', 'NZ', 'MF', 'NS', 'PP', 'AL', 'NP', 'NV', 'NY', 'NG', 'NH', 'LC', 'HG', 'ET', 'NO', 'SI', 'SO', 'HK', 'CN', 'NN', 'W5', 'ND', 'NJ', 'NK', 'NQ', 'W8', 'NU', 'NW', 'U7', 'U8', 'IT', 'IX', 'BX', 'EW', 'MT', 'IJ', 'ID']


In [ ]:
# STEP 71: Inspect the November 2024 MF worksheet

november_workbook = xlrd.open_workbook(str(november_2024_file))
november_sheet = november_workbook.sheet_by_name("MF")

for row_number in range(min(10, november_sheet.nrows)):
    values = [
        november_sheet.cell_value(row_number, col)
        for col in range(november_sheet.ncols)
    ]
    values = [value for value in values if value != ""]
    print(f"Row {row_number + 1}: {values}")

Row 1: ['RLMF806', 'Nippon India Multi Asset Allocation Fund (An open ended scheme investing in Equity, Debt \nand Exchange Traded Commodity Derivatives and \nGold ETF & Silver ETF)', 'Index']
Row 2: ['Monthly Portfolio Statement as on November 30,2024']
Row 3: []
Row 4: ['ISIN', 'Name of the Instrument', 'Industry / Rating', 'Quantity', 'Market/Fair Value\n( Rs. in Lacs)', '% to NAV', 'YIELD']
Row 5: ['Equity & Equity related']
Row 6: ['(a) Listed / awaiting listing on Stock Exchanges']
Row 7: ['IBCL05', 'INE090A01021', 'ICICI Bank Limited', 'Banks', 1150000.0, 14951.15, 0.0319]
Row 8: ['HDFB03', 'INE040A01034', 'HDFC Bank Limited', 'Banks', 700000.0, 12572.35, 0.0268]
Row 9: ['INFS02', 'INE009A01021', 'Infosys Limited', 'IT - Software', 510000.0, 9475.04, 0.0202]
Row 10: ['BTVL02', 'INE397D01024', 'Bharti Airtel Limited', 'Telecom - Services', 500000.0, 8135.75, 0.0174]


In [ ]:
# STEP 72: Search November 2024 holdings for gold and silver

for row_number in range(november_sheet.nrows):
    row_values = [
        november_sheet.cell_value(row_number, col)
        for col in range(november_sheet.ncols)
    ]

    combined_text = " ".join(
        str(value) for value in row_values if value != ""
    ).lower()

    if "gold" in combined_text or "silver" in combined_text:
        print(f"Row {row_number + 1}: {row_values}")

Row 1: ['RLMF806', 'Nippon India Multi Asset Allocation Fund (An open ended scheme investing in Equity, Debt \nand Exchange Traded Commodity Derivatives and \nGold ETF & Silver ETF)', '', '', '', '', '', 'Index']
Row 135: ['BMGB51ME', 'INF204KB17I5', 'Nippon India ETF Gold Bees', '', 75180197.0, 48333.35, 0.1032, '']
Row 136: ['149758', 'INF204KC1402', 'Nippon India Silver ETF', '', 14366055.0, 12486.98, 0.0267, '']
Row 152: ['', 'Long', 'FUTCOM_GOLD_05/02/2025', 'Others', 21900.0, 16891.03, 0.0361, '']
Row 153: ['', 'Long', 'FUTCOM_SILVER_05/03/2025', 'Others', 1110.0, 1012.42, 0.0022, '']
Row 160: ['', '', '', '', '', 'BENCHMARK NAME - 50% OF BSE 500 TRI, 20% OF MSCI WORLD INDEX TRI, 15% OF CRISIL SHORT TERM BOND INDEX 10% OF DOMESTIC PRICES OF GOLD & 5% OF DOMESTIC PRICES OF SILVER', '', '']


In [ ]:
# STEP 74: Identify the December 2024 workbook format

december_2024_file = nippon_folder / "NIMF-MONTHLY-PORTFOLIO-31-Dec-24.xls"

try:
    with ZipFile(december_2024_file):
        print("Internal format: XLSX")

    with open(december_2024_file, "rb") as file_stream:
        december_workbook = load_workbook(
            file_stream,
            read_only=True,
            data_only=True
        )
        print("Sheet names:", december_workbook.sheetnames)
        december_workbook.close()

except BadZipFile:
    print("Internal format: Legacy XLS")

    december_workbook = xlrd.open_workbook(str(december_2024_file))
    print("Sheet names:", december_workbook.sheet_names())

Internal format: Legacy XLS
Sheet names: ['INDEX', 'GF', 'GS', 'IP', 'IF', 'ST', 'BF', 'MI', 'PS', 'CP', 'PH', 'ME', 'NE', 'NI', 'EO', 'LF', 'FR', 'SD', 'SE', 'SH', 'LQ', 'TS', 'LE', 'IV', 'LP', 'IM', 'DC', 'UB', 'K5', '9B', 'EA', 'QP', 'GL', 'NF', 'SC', 'SF', 'AF', 'GD', 'CE', 'CF', 'RD', 'CB', 'JE', 'SX', 'RW', 'RG', 'BP', 'ES', 'NX', 'UE', 'LT', 'NL', 'NB', 'JZ', 'PU', 'GB', 'LB', 'BB', 'SB', 'HS', 'IB', 'CC', 'ON', 'NM', 'RJ', 'G6', 'SN', 'NZ', 'MF', 'NS', 'PP', 'AL', 'NP', 'NV', 'NY', 'NG', 'NH', 'LC', 'HG', 'ET', 'NO', 'SI', 'SO', 'HK', 'CN', 'NN', 'W5', 'ND', 'NJ', 'NK', 'NQ', 'W8', 'NU', 'NW', 'U7', 'U8', 'IT', 'IX', 'BX', 'EW', 'MT', 'IJ', 'ID', 'AN', 'RN']


In [ ]:
# STEP 75: Inspect the December 2024 MF worksheet

december_workbook = xlrd.open_workbook(str(december_2024_file))
december_sheet = december_workbook.sheet_by_name("MF")

for row_number in range(min(10, december_sheet.nrows)):
    values = [
        december_sheet.cell_value(row_number, col)
        for col in range(december_sheet.ncols)
    ]
    values = [value for value in values if value != ""]
    print(f"Row {row_number + 1}: {values}")

Row 1: ['RLMF806', 'Nippon India Multi Asset Allocation Fund (An open ended scheme investing in Equity, Debt \nand Exchange Traded Commodity Derivatives and \nGold ETF & Silver ETF)', 'Index']
Row 2: ['Monthly Portfolio Statement as on December 31,2024']
Row 3: []
Row 4: ['ISIN', 'Name of the Instrument', 'Industry / Rating', 'Quantity', 'Market/Fair Value\n( Rs. in Lacs)', '% to NAV', 'YIELD']
Row 5: ['Equity & Equity related']
Row 6: ['(a) Listed / awaiting listing on Stock Exchanges']
Row 7: ['IBCL05', 'INE090A01021', 'ICICI Bank Limited', 'Banks', 1200000.0, 15379.8, 0.0317]
Row 8: ['HDFB03', 'INE040A01034', 'HDFC Bank Limited', 'Banks', 700000.0, 12409.95, 0.0256]
Row 9: ['INFS02', 'INE009A01021', 'Infosys Limited', 'IT - Software', 580000.0, 10904.0, 0.0225]
Row 10: ['RIND01', 'INE002A01018', 'Reliance Industries Limited', 'Petroleum Products', 720000.0, 8751.24, 0.018]


In [ ]:
# STEP 76: Search December 2024 holdings for gold and silver

for row_number in range(december_sheet.nrows):
    row_values = [
        december_sheet.cell_value(row_number, col)
        for col in range(december_sheet.ncols)
    ]

    combined_text = " ".join(
        str(value) for value in row_values if value != ""
    ).lower()

    if "gold" in combined_text or "silver" in combined_text:
        print(f"Row {row_number + 1}: {row_values}")

Row 1: ['RLMF806', 'Nippon India Multi Asset Allocation Fund (An open ended scheme investing in Equity, Debt \nand Exchange Traded Commodity Derivatives and \nGold ETF & Silver ETF)', '', '', '', '', '', 'Index']
Row 134: ['BMGB51ME', 'INF204KB17I5', 'Nippon India ETF Gold Bees', '', 77171421.0, 49459.16, 0.102, '']
Row 135: ['149758', 'INF204KC1402', 'Nippon India Silver ETF', '', 14866055.0, 12447.35, 0.0257, '']
Row 151: ['', 'Long', 'FUTCOM_GOLD_05/02/2025', 'Others', 15700.0, 12049.44, 0.0248, '']
Row 152: ['', 'Long', 'FUTCOM_SILVER_05/03/2025', 'Others', 2910.0, 2538.48, 0.0052, '']
Row 160: ['', '', '', '', '', 'BENCHMARK NAME - 50% OF BSE 500 TRI, 20% OF MSCI WORLD INDEX TRI, 15% OF CRISIL SHORT TERM BOND INDEX 10% OF DOMESTIC PRICES OF GOLD & 5% OF DOMESTIC PRICES OF SILVER', '', '']


In [ ]:
# STEP 78: Identify the January 2025 workbook format

january_2025_file = nippon_folder / "NIMF_MONTHLY_PORTFOLIO_31-Jan-25.xls"

try:
    with ZipFile(january_2025_file):
        print("Internal format: XLSX")

    with open(january_2025_file, "rb") as file_stream:
        january_workbook = load_workbook(
            file_stream,
            read_only=True,
            data_only=True
        )
        print("Sheet names:", january_workbook.sheetnames)
        january_workbook.close()

except BadZipFile:
    print("Internal format: Legacy XLS")

    january_workbook = xlrd.open_workbook(str(january_2025_file))
    print("Sheet names:", january_workbook.sheet_names())

Internal format: XLSX
Sheet names: ['INDEX', 'GF', 'GS', 'IP', 'IF', 'ST', 'BF', 'MI', 'PS', 'CP', 'PH', 'ME', 'NE', 'NI', 'EO', 'LF', 'FR', 'SD', 'SE', 'SH', 'LQ', 'TS', 'LE', 'IV', 'LP', 'IM', 'DC', 'UB', 'K5', '9B', 'EA', 'QP', 'GL', 'NF', 'SC', 'SF', 'AF', 'GD', 'CE', 'CF', 'RD', 'CB', 'JE', 'SX', 'RW', 'RG', 'BP', 'ES', 'NX', 'UE', 'LT', 'NL', 'NB', 'JZ', 'PU', 'GB', 'LB', 'BB', 'SB', 'HS', 'IB', 'CC', 'ON', 'NM', 'RJ', 'G6', 'SN', 'NZ', 'MF', 'NS', 'PP', 'AL', 'NP', 'NV', 'NY', 'NG', 'NH', 'LC', 'HG', 'ET', 'NO', 'SI', 'SO', 'HK', 'CN', 'NN', 'W5', 'ND', 'NJ', 'NK', 'NQ', 'W8', 'NU', 'NW', 'U7', 'U8', 'IT', 'IX', 'BX', 'EW', 'MT', 'IJ', 'ID', 'AN', 'RN']


In [ ]:
# STEP 79: Inspect the January 2025 MF worksheet

with open(january_2025_file, "rb") as file_stream:
    january_workbook = load_workbook(
        file_stream,
        read_only=True,
        data_only=True
    )

    january_sheet = january_workbook["MF"]

    for row_number, row in enumerate(
        january_sheet.iter_rows(
            min_row=1,
            max_row=10,
            values_only=True
        ),
        start=1
    ):
        values = [value for value in row if value is not None]
        print(f"Row {row_number}: {values}")

    january_workbook.close()

Row 1: ['RLMF806', 'Nippon India Multi Asset Allocation Fund (An open ended scheme investing in Equity, Debt \nand Exchange Traded Commodity Derivatives and \nGold ETF & Silver ETF)', 'Index']
Row 2: ['Monthly Portfolio Statement as on January 31,2025']
Row 3: []
Row 4: ['ISIN', 'Name of the Instrument', 'Industry / Rating', 'Quantity', 'Market/Fair Value\n( Rs. in Lacs)', '% to NAV', 'YIELD']
Row 5: ['Equity & Equity related']
Row 6: ['(a) Listed / awaiting listing on Stock Exchanges']
Row 7: ['IBCL05', 'INE090A01021', 'ICICI Bank Limited', 'Banks', 1350000, 16912.8, 0.0338]
Row 8: ['INFS02', 'INE009A01021', 'Infosys Limited', 'IT - Software', 750000, 14098.5, 0.0282]
Row 9: ['HDFB03', 'INE040A01034', 'HDFC Bank Limited', 'Banks', 750000, 12740.63, 0.0255]
Row 10: ['RIND01', 'INE002A01018', 'Reliance Industries Limited', 'Petroleum Products', 800000, 10120.8, 0.0202]


In [ ]:
# STEP 80: Search January 2025 holdings for gold and silver

with open(january_2025_file, "rb") as file_stream:
    january_workbook = load_workbook(
        file_stream,
        read_only=True,
        data_only=True
    )

    january_sheet = january_workbook["MF"]

    for row_number, row in enumerate(
        january_sheet.iter_rows(values_only=True),
        start=1
    ):
        row_values = [
            str(value) if value is not None else ""
            for value in row
        ]

        combined_text = " ".join(row_values).lower()

        if "gold" in combined_text or "silver" in combined_text:
            print(f"Row {row_number}: {row_values}")

    january_workbook.close()

Row 1: ['RLMF806', 'Nippon India Multi Asset Allocation Fund (An open ended scheme investing in Equity, Debt \nand Exchange Traded Commodity Derivatives and \nGold ETF & Silver ETF)', '', '', '', '', '', 'Index']
Row 140: ['BMGB51ME', 'INF204KB17I5', 'Nippon India ETF Gold Bees', '', '86261865', '59589.7', '0.1191', '']
Row 141: ['149758', 'INF204KC1402', 'Nippon India Silver ETF', '', '16869201', '15295.3', '0.0306', '']
Row 157: ['', 'Long', 'FUTCOM_GOLD_04/04/2025', 'Others', '5400', '4440.58', '0.0089', '']
Row 158: ['', 'Long', 'FUTCOM_SILVER_05/03/2025', 'Others', '2010', '1875.89', '0.0038', '']
Row 166: ['', '', '', '', '', 'BENCHMARK NAME - 50% OF BSE 500 TRI, 20% OF MSCI WORLD INDEX TRI, 15% OF CRISIL SHORT TERM BOND INDEX 10% OF DOMESTIC PRICES OF GOLD & 5% OF DOMESTIC PRICES OF SILVER', '', '']


In [ ]:
# STEP 82: Identify the February 2025 workbook format

february_2025_file = nippon_folder / "NIMF-MONTHLY-PORTFOLIO-28-Feb-25.xls"

try:
    with ZipFile(february_2025_file):
        print("Internal format: XLSX")

    with open(february_2025_file, "rb") as file_stream:
        february_workbook = load_workbook(
            file_stream,
            read_only=True,
            data_only=True
        )
        print("Sheet names:", february_workbook.sheetnames)
        february_workbook.close()

except BadZipFile:
    print("Internal format: Legacy XLS")

    february_workbook = xlrd.open_workbook(str(february_2025_file))
    print("Sheet names:", february_workbook.sheet_names())

Internal format: Legacy XLS
Sheet names: ['INDEX', 'GF', 'GS', 'IP', 'IF', 'ST', 'BF', 'MI', 'PS', 'CP', 'PH', 'ME', 'NE', 'NI', 'EO', 'LF', 'FR', 'SD', 'SE', 'SH', 'LQ', 'TS', 'LE', 'IV', 'LP', 'IM', 'DC', 'UB', 'K5', '9B', 'EA', 'QP', 'GL', 'NF', 'SC', 'SF', 'AF', 'GD', 'CE', 'CF', 'RD', 'CB', 'JE', 'SX', 'RW', 'RG', 'BP', 'ES', 'NX', 'UE', 'LT', 'NL', 'NB', 'JZ', 'PU', 'GB', 'LB', 'BB', 'SB', 'HS', 'IB', 'CC', 'ON', 'NM', 'RJ', 'G6', 'SN', 'NZ', 'MF', 'NS', 'PP', 'AL', 'NP', 'NV', 'NY', 'NG', 'NH', 'LC', 'HG', 'ET', 'NO', 'SI', 'SO', 'HK', 'CN', 'NN', 'W5', 'ND', 'NJ', 'NK', 'NQ', 'W8', 'NU', 'NW', 'U7', 'U8', 'IT', 'IX', 'BX', 'EW', 'MT', 'IJ', 'ID', 'AN', 'RN', 'AM']


In [ ]:
# STEP 83: Inspect the February 2025 MF worksheet

february_workbook = xlrd.open_workbook(str(february_2025_file))
february_sheet = february_workbook.sheet_by_name("MF")

for row_number in range(min(10, february_sheet.nrows)):
    values = [
        february_sheet.cell_value(row_number, col)
        for col in range(february_sheet.ncols)
    ]
    values = [value for value in values if value != ""]
    print(f"Row {row_number + 1}: {values}")

Row 1: ['RLMF806', 'Nippon India Multi Asset Allocation Fund (An open ended scheme investing in Equity, Debt \nand Exchange Traded Commodity Derivatives and \nGold ETF & Silver ETF)', 'Index']
Row 2: ['Monthly Portfolio Statement as on February 28,2025']
Row 3: []
Row 4: ['ISIN', 'Name of the Instrument', 'Industry / Rating', 'Quantity', 'Market/Fair Value\n( Rs. in Lacs)', '% to NAV', 'YIELD']
Row 5: ['Equity & Equity related']
Row 6: ['(a) Listed / awaiting listing on Stock Exchanges']
Row 7: ['IBCL05', 'INE090A01021', 'ICICI Bank Limited', 'Banks', 1400000.0, 16857.4, 0.0341]
Row 8: ['HDFB03', 'INE040A01034', 'HDFC Bank Limited', 'Banks', 750000.0, 12993.0, 0.0262]
Row 9: ['INFS02', 'INE009A01021', 'Infosys Limited', 'IT - Software', 750000.0, 12657.75, 0.0256]
Row 10: ['RIND01', 'INE002A01018', 'Reliance Industries Limited', 'Petroleum Products', 800000.0, 9600.8, 0.0194]


In [ ]:
# STEP 84: Search February 2025 holdings for gold and silver

for row_number in range(february_sheet.nrows):
    row_values = [
        february_sheet.cell_value(row_number, col)
        for col in range(february_sheet.ncols)
    ]

    combined_text = " ".join(
        str(value) for value in row_values if value != ""
    ).lower()

    if "gold" in combined_text or "silver" in combined_text:
        print(f"Row {row_number + 1}: {row_values}")

Row 1: ['RLMF806', 'Nippon India Multi Asset Allocation Fund (An open ended scheme investing in Equity, Debt \nand Exchange Traded Commodity Derivatives and \nGold ETF & Silver ETF)', '', '', '', '', '', 'Index']
Row 140: ['BMGB51ME', 'INF204KB17I5', 'Nippon India ETF Gold Bees', '', 86261865.0, 61332.19, 0.1239, '']
Row 141: ['149758', 'INF204KC1402', 'Nippon India Silver ETF', '', 16869201.0, 15259.88, 0.0308, '']
Row 157: ['', 'Long', 'FUTCOM_GOLD_04/04/2025', 'Others', 3800.0, 3200.32, 0.0065, '']
Row 165: ['', '', '', '', '', 'BENCHMARK NAME - 50% OF BSE 500 TRI, 20% OF MSCI WORLD INDEX TRI, 15% OF CRISIL SHORT TERM BOND INDEX 10% OF DOMESTIC PRICES OF GOLD & 5% OF DOMESTIC PRICES OF SILVER', '', '']


In [ ]:
# STEP 86: Identify the March 2025 workbook format

march_2025_file = nippon_folder / "NIMF-MONTHLY-PORTFOLIO-31-Mar-25.xls"

try:
    with ZipFile(march_2025_file):
        print("Internal format: XLSX")

    with open(march_2025_file, "rb") as file_stream:
        march_workbook = load_workbook(
            file_stream,
            read_only=True,
            data_only=True
        )
        print("Sheet names:", march_workbook.sheetnames)
        march_workbook.close()

except BadZipFile:
    print("Internal format: Legacy XLS")

    march_workbook = xlrd.open_workbook(str(march_2025_file))
    print("Sheet names:", march_workbook.sheet_names())

Internal format: XLSX
Sheet names: ['Index', 'GF', 'GS', 'IP', 'IF', 'ST', 'BF', 'MI', 'PS', 'CP', 'PH', 'ME', 'NE', 'NI', 'EO', 'LF', 'FR', 'SD', 'SE', 'SH', 'LQ', 'TS', 'LE', 'IV', 'LP', 'IM', 'DC', 'UB', 'K5', '9B', 'EA', 'QP', 'GL', 'NF', 'SC', 'SF', 'AF', 'GD', 'CE', 'CF', 'RD', 'CB', 'JE', 'SX', 'RW', 'RG', 'BP', 'ES', 'NX', 'UE', 'LT', 'NL', 'NB', 'JZ', 'PU', 'GB', 'LB', 'BB', 'SB', 'HS', 'IB', 'CC', 'ON', 'NM', 'RJ', 'G6', 'SN', 'NZ', 'MF', 'NS', 'PP', 'AL', 'NP', 'NV', 'NY', 'NG', 'NH', 'LC', 'HG', 'ET', 'NO', 'SI', 'SO', 'HK', 'CN', 'NN', 'W5', 'ND', 'NJ', 'NK', 'NQ', 'W8', 'NU', 'NW', 'U7', 'U8', 'IT', 'IX', 'BX', 'EW', 'MT', 'IJ', 'ID', 'AN', 'RN', 'AM']


In [ ]:
# STEP 87: Inspect the March 2025 MF worksheet

with open(march_2025_file, "rb") as file_stream:
    march_workbook = load_workbook(
        file_stream,
        read_only=True,
        data_only=True
    )

    march_sheet = march_workbook["MF"]

    for row_number, row in enumerate(
        march_sheet.iter_rows(
            min_row=1,
            max_row=10,
            values_only=True
        ),
        start=1
    ):
        values = [value for value in row if value is not None]
        print(f"Row {row_number}: {values}")

    march_workbook.close()

Row 1: ['RLMF806', 'Nippon India Multi Asset Allocation Fund (An open ended scheme investing in Equity, Debt \nand Exchange Traded Commodity Derivatives and \nGold ETF & Silver ETF)', 'Index']
Row 2: ['Monthly Portfolio Statement as on March 31,2025']
Row 3: []
Row 4: ['ISIN', 'Name of the Instrument', 'Industry / Rating', 'Quantity', 'Market/Fair Value\n( Rs. in Lacs)', '% to NAV', 'YIELD']
Row 5: ['Equity & Equity related']
Row 6: ['(a) Listed / awaiting listing on Stock Exchanges']
Row 7: ['IBCL05', 'INE090A01021', 'ICICI Bank Limited', 'Banks', 1450000, 19551.08, 0.0367]
Row 8: ['HDFB03', 'INE040A01034', 'HDFC Bank Limited', 'Banks', 770000, 14077.14, 0.0264]
Row 9: ['INFS02', 'INE009A01021', 'Infosys Limited', 'IT - Software', 750000, 11779.88, 0.0221]
Row 10: ['RIND01', 'INE002A01018', 'Reliance Industries Limited', 'Petroleum Products', 900000, 11475.9, 0.0215]


In [ ]:
# STEP 88: Search March 2025 holdings for gold and silver

with open(march_2025_file, "rb") as file_stream:
    march_workbook = load_workbook(
        file_stream,
        read_only=True,
        data_only=True
    )

    march_sheet = march_workbook["MF"]

    for row_number, row in enumerate(
        march_sheet.iter_rows(values_only=True),
        start=1
    ):
        row_values = [
            str(value) if value is not None else ""
            for value in row
        ]

        combined_text = " ".join(row_values).lower()

        if "gold" in combined_text or "silver" in combined_text:
            print(f"Row {row_number}: {row_values}")

    march_workbook.close()

Row 1: ['RLMF806', 'Nippon India Multi Asset Allocation Fund (An open ended scheme investing in Equity, Debt \nand Exchange Traded Commodity Derivatives and \nGold ETF & Silver ETF)', '', '', '', '', '', 'Index']
Row 133: ['LIGS22', 'INE1U3R15011', 'Liquid Gold Series**', 'CRISIL AAA(SO)', '120', '4212.85', '0.0079', '0.098948']
Row 141: ['BMGB51ME', 'INF204KB17I5', 'Nippon India ETF Gold Bees', '', '86261865', '63971.8', '0.12', '']
Row 142: ['149758', 'INF204KC1402', 'Nippon India Silver ETF', '', '16869201', '16464.34', '0.0309', '']
Row 158: ['', 'Long', 'FUTCOM_GOLD_05/06/2025', 'Others', '4800', '4354.42', '0.0082', '']
Row 166: ['', '', '', '', '', 'BENCHMARK NAME - 50% OF BSE 500 TRI, 20% OF MSCI WORLD INDEX TRI, 15% OF CRISIL SHORT TERM BOND INDEX 10% OF DOMESTIC PRICES OF GOLD & 5% OF DOMESTIC PRICES OF SILVER', '', '']


In [ ]:
# STEP 90: Identify the April 2025 workbook format

april_2025_file = nippon_folder / "NIMF-MONTHLY-PORTFOLIO-30-April-25.xls"

try:
    with ZipFile(april_2025_file):
        print("Internal format: XLSX")

    with open(april_2025_file, "rb") as file_stream:
        april_workbook = load_workbook(
            file_stream,
            read_only=True,
            data_only=True
        )
        print("Sheet names:", april_workbook.sheetnames)
        april_workbook.close()

except BadZipFile:
    print("Internal format: Legacy XLS")

    april_workbook = xlrd.open_workbook(str(april_2025_file))
    print("Sheet names:", april_workbook.sheet_names())

Internal format: XLSX
Sheet names: ['Index', 'GF', 'GS', 'IP', 'IF', 'ST', 'BF', 'MI', 'PS', 'CP', 'PH', 'ME', 'NE', 'NI', 'EO', 'LF', 'FR', 'SD', 'SE', 'SH', 'LQ', 'TS', 'LE', 'LP', 'UB', 'K5', 'EA', 'QP', 'GL', 'NF', 'SC', 'SF', 'AF', 'GD', 'CE', 'CF', 'RD', 'CB', 'JE', 'SX', 'RW', 'RG', 'BP', 'ES', 'NX', 'UE', 'LT', 'NL', 'NB', 'JZ', 'PU', 'GB', 'LB', 'BB', 'SB', 'HS', 'IB', 'CC', 'ON', 'NM', 'RJ', 'G6', 'SN', 'NZ', 'MF', 'NS', 'PP', 'AL', 'NP', 'NV', 'NY', 'NG', 'NH', 'LC', 'HG', 'ET', 'NO', 'SI', 'SO', 'HK', 'CN', 'NN', 'W5', 'ND', 'NJ', 'NK', 'NQ', 'W8', 'NU', 'NW', 'U7', 'U8', 'IT', 'IX', 'BX', 'EW', 'MT', 'IJ', 'ID', 'AN', 'RN', 'AM']


In [ ]:
# STEP 91: Inspect the April 2025 MF worksheet

with open(april_2025_file, "rb") as file_stream:
    april_workbook = load_workbook(
        file_stream,
        read_only=True,
        data_only=True
    )

    april_sheet = april_workbook["MF"]

    for row_number, row in enumerate(
        april_sheet.iter_rows(
            min_row=1,
            max_row=10,
            values_only=True
        ),
        start=1
    ):
        values = [value for value in row if value is not None]
        print(f"Row {row_number}: {values}")

    april_workbook.close()

Row 1: ['RLMF806', 'Nippon India Multi Asset Allocation Fund (An open ended scheme investing in Equity, Debt \nand Exchange Traded Commodity Derivatives and \nGold ETF & Silver ETF)', 'Index']
Row 2: ['Monthly Portfolio Statement as on April 30,2025']
Row 3: []
Row 4: ['ISIN', 'Name of the Instrument', 'Industry / Rating', 'Quantity', 'Market/Fair Value\n( Rs. in Lacs)', '% to NAV', 'YIELD']
Row 5: ['Equity & Equity related']
Row 6: ['(a) Listed / awaiting listing on Stock Exchanges']
Row 7: ['IBCL05', 'INE090A01021', 'ICICI Bank Limited', 'Banks', 1550000, 22118.5, 0.0393]
Row 8: ['RIND01', 'INE002A01018', 'Reliance Industries Limited', 'Petroleum Products', 1125000, 15806.25, 0.0281]
Row 9: ['HDFB03', 'INE040A01034', 'HDFC Bank Limited', 'Banks', 770000, 14822.5, 0.0264]
Row 10: ['SBAI02', 'INE062A01020', 'State Bank of India', 'Banks', 1600000, 12618.4, 0.0224]


In [ ]:
# STEP 92: Search April 2025 holdings for gold and silver

with open(april_2025_file, "rb") as file_stream:
    april_workbook = load_workbook(
        file_stream,
        read_only=True,
        data_only=True
    )

    april_sheet = april_workbook["MF"]

    for row_number, row in enumerate(
        april_sheet.iter_rows(values_only=True),
        start=1
    ):
        values = list(row)

        if any(
            isinstance(value, str)
            and ("gold" in value.lower() or "silver" in value.lower())
            for value in values
        ):
            print(f"Row {row_number}: {values}")

    april_workbook.close()

Row 1: ['RLMF806', 'Nippon India Multi Asset Allocation Fund (An open ended scheme investing in Equity, Debt \nand Exchange Traded Commodity Derivatives and \nGold ETF & Silver ETF)', None, None, None, None, None, 'Index']
Row 131: ['LIGS22', 'INE1U3R15011', 'Liquid Gold Series**', 'CRISIL AAA(SO)', 120, 4137.39, 0.0074, 0.09515]
Row 139: ['BMGB51ME', 'INF204KB17I5', 'Nippon India ETF Gold Bees', None, 86261865, 68224.51, 0.1213, None]
Row 140: ['149758', 'INF204KC1402', 'Nippon India Silver ETF', None, 16869201, 15455.56, 0.0275, None]
Row 158: [None, 'Long', 'FUTCOM_GOLD_05/06/2025', 'Others', 3000, 2841.06, 0.0051, None]
Row 159: [None, 'Long', 'FUTCOM_SILVER_04/07/2025', 'Others', 900, 863.25, 0.0015, None]
Row 166: [None, None, None, None, None, 'BENCHMARK NAME - 50% OF BSE 500 TRI, 20% OF MSCI WORLD INDEX TRI, 15% OF CRISIL SHORT TERM BOND INDEX 10% OF DOMESTIC PRICES OF GOLD & 5% OF DOMESTIC PRICES OF SILVER', None, None]


In [ ]:
# STEP 95: Find the actual May 2025 filename

for file in nippon_folder.iterdir():
    if file.is_file() and "may" in file.name.lower() and "25" in file.name:
        print(file.name)

NIMF-MONTHLY-PORTFOLIO-31-May-25.xls


In [ ]:
# STEP 96: Identify the May 2025 workbook format

may_2025_file = nippon_folder / "NIMF-MONTHLY-PORTFOLIO-31-May-25.xls"

try:
    with ZipFile(may_2025_file):
        print("Internal format: XLSX")

    with open(may_2025_file, "rb") as file_stream:
        may_workbook = load_workbook(
            file_stream,
            read_only=True,
            data_only=True
        )
        print("Sheet names:", may_workbook.sheetnames)
        may_workbook.close()

except BadZipFile:
    print("Internal format: Legacy XLS")

    may_workbook = xlrd.open_workbook(str(may_2025_file))
    print("Sheet names:", may_workbook.sheet_names())

Internal format: XLSX
Sheet names: ['Index', 'GF', 'GS', 'IP', 'IF', 'ST', 'BF', 'MI', 'PS', 'CP', 'PH', 'ME', 'NE', 'NI', 'EO', 'LF', 'FR', 'SD', 'SE', 'SH', 'LQ', 'TS', 'LE', 'LP', 'UB', 'K5', 'EA', 'QP', 'GL', 'NF', 'SC', 'SF', 'AF', 'GD', 'CE', 'CF', 'RD', 'CB', 'JE', 'SX', 'RW', 'RG', 'BP', 'ES', 'NX', 'UE', 'LT', 'NL', 'NB', 'JZ', 'PU', 'GB', 'LB', 'BB', 'SB', 'HS', 'IB', 'CC', 'ON', 'NM', 'RJ', 'G6', 'SN', 'NZ', 'MF', 'NS', 'PP', 'AL', 'NP', 'NV', 'NY', 'NG', 'NH', 'LC', 'HG', 'ET', 'NO', 'SI', 'SO', 'HK', 'CN', 'NN', 'W5', 'ND', 'NJ', 'NK', 'NQ', 'W8', 'NU', 'NW', 'U7', 'U8', 'IT', 'IX', 'BX', 'EW', 'MT', 'IJ', 'ID', 'AN', 'RN', 'AM', 'LV', 'QI']


In [ ]:
# STEP 97: Inspect the May 2025 MF worksheet

with open(may_2025_file, "rb") as file_stream:
    may_workbook = load_workbook(
        file_stream,
        read_only=True,
        data_only=True
    )

    may_sheet = may_workbook["MF"]

    for row_number, row in enumerate(
        may_sheet.iter_rows(
            min_row=1,
            max_row=10,
            values_only=True
        ),
        start=1
    ):
        values = [value for value in row if value is not None]
        print(f"Row {row_number}: {values}")

    may_workbook.close()

Row 1: ['RLMF806', 'Nippon India Multi Asset Allocation Fund (An open ended scheme investing in Equity, Debt \nand Exchange Traded Commodity Derivatives and \nGold ETF & Silver ETF)', 'Index']
Row 2: ['Monthly Portfolio Statement as on May 31,2025']
Row 3: []
Row 4: ['ISIN', 'Name of the Instrument', 'Industry / Rating', 'Quantity', 'Market/Fair Value\n( Rs. in Lacs)', '% to NAV', 'YIELD']
Row 5: ['Equity & Equity related']
Row 6: ['(a) Listed / awaiting listing on Stock Exchanges']
Row 7: ['IBCL05', 'INE090A01021', 'ICICI Bank Limited', 'Banks', 1550000, 22409.9, 0.0375]
Row 8: ['HDFB03', 'INE040A01034', 'HDFC Bank Limited', 'Banks', 770000, 14975.73, 0.025]
Row 9: ['RIND01', 'INE002A01018', 'Reliance Industries Limited', 'Petroleum Products', 1000000, 14209, 0.0238]
Row 10: ['SBAI02', 'INE062A01020', 'State Bank of India', 'Banks', 1600000, 12996.8, 0.0217]


In [ ]:
# STEP 98: Search May 2025 holdings for gold and silver

with open(may_2025_file, "rb") as file_stream:
    may_workbook = load_workbook(
        file_stream,
        read_only=True,
        data_only=True
    )

    may_sheet = may_workbook["MF"]

    for row_number, row in enumerate(
        may_sheet.iter_rows(values_only=True),
        start=1
    ):
        values = list(row)

        if any(
            isinstance(value, str)
            and ("gold" in value.lower() or "silver" in value.lower())
            for value in values
        ):
            print(f"Row {row_number}: {values}")

    may_workbook.close()

Row 1: ['RLMF806', 'Nippon India Multi Asset Allocation Fund (An open ended scheme investing in Equity, Debt \nand Exchange Traded Commodity Derivatives and \nGold ETF & Silver ETF)', None, None, None, None, None, 'Index']
Row 131: ['LIGS22', 'INE1U3R15011', 'Liquid Gold Series**', 'CRISIL AAA(SO)', 120, 3744.53, 0.0063, 0.0940525]
Row 139: ['BMGB51ME', 'INF204KB17I5', 'Nippon India ETF Gold Bees', None, 86261865, 68569.56, 0.1147, None]
Row 140: ['149758', 'INF204KC1402', 'Nippon India Silver ETF', None, 16869201, 15916.09, 0.0266, None]
Row 159: [None, 'Long', 'FUTCOM_GOLD_05/08/2025', 'Others', 10000, 9587.5, 0.016, None]
Row 160: [None, 'Long', 'FUTCOM_SILVER_04/07/2025', 'Others', 1500, 1455.23, 0.0024, None]
Row 168: [None, None, None, None, None, 'BENCHMARK NAME - 50% OF BSE 500 TRI, 20% OF MSCI WORLD INDEX TRI, 15% OF CRISIL SHORT TERM BOND INDEX 10% OF DOMESTIC PRICES OF GOLD & 5% OF DOMESTIC PRICES OF SILVER', None, None]


In [ ]:
# STEP 101: Find the actual June 2025 filename

for file in nippon_folder.iterdir():
    if file.is_file() and "june" in file.name.lower() and "25" in file.name:
        print(file.name)

NIMF-MONTHLY-PORTFOLIO-30-June-25.xls


In [ ]:
# STEP 102: Identify the June 2025 workbook format

june_2025_file = nippon_folder / "NIMF-MONTHLY-PORTFOLIO-30-June-25.xls"

try:
    with ZipFile(june_2025_file):
        print("Internal format: XLSX")

    with open(june_2025_file, "rb") as file_stream:
        june_workbook = load_workbook(
            file_stream,
            read_only=True,
            data_only=True
        )
        print("Sheet names:", june_workbook.sheetnames)
        june_workbook.close()

except BadZipFile:
    print("Internal format: Legacy XLS")

    june_workbook = xlrd.open_workbook(str(june_2025_file))
    print("Sheet names:", june_workbook.sheet_names())

Internal format: XLSX
Sheet names: ['Index', 'GF', 'GS', 'IP', 'IF', 'ST', 'BF', 'MI', 'PS', 'CP', 'PH', 'ME', 'NE', 'NI', 'EO', 'LF', 'FR', 'SD', 'SE', 'SH', 'LQ', 'TS', 'LE', 'LP', 'UB', 'K5', 'EA', 'QP', 'GL', 'NF', 'SC', 'SF', 'AF', 'GD', 'CE', 'CF', 'RD', 'CB', 'JE', 'SX', 'RW', 'RG', 'BP', 'ES', 'NX', 'UE', 'LT', 'NL', 'NB', 'JZ', 'PU', 'GB', 'LB', 'BB', 'SB', 'HS', 'IB', 'CC', 'ON', 'NM', 'RJ', 'G6', 'SN', 'NZ', 'MF', 'NS', 'PP', 'AL', 'NP', 'NV', 'NY', 'NG', 'NH', 'LC', 'HG', 'ET', 'NO', 'SI', 'SO', 'HK', 'CN', 'NN', 'W5', 'ND', 'NJ', 'NK', 'NQ', 'W8', 'NU', 'NW', 'U7', 'U8', 'IT', 'IX', 'BX', 'EW', 'MT', 'IJ', 'ID', 'AN', 'RN', 'AM', 'LV', 'QI', 'BS', 'BZ', 'AP']


In [ ]:
# STEP 103: Inspect the June 2025 MF worksheet

with open(june_2025_file, "rb") as file_stream:
    june_workbook = load_workbook(
        file_stream,
        read_only=True,
        data_only=True
    )

    june_sheet = june_workbook["MF"]

    for row_number, row in enumerate(
        june_sheet.iter_rows(
            min_row=1,
            max_row=10,
            values_only=True
        ),
        start=1
    ):
        values = [value for value in row if value is not None]
        print(f"Row {row_number}: {values}")

    june_workbook.close()

Row 1: ['RLMF806', 'Nippon India Multi Asset Allocation Fund (An open ended scheme investing in Equity, Debt \nand Exchange Traded Commodity Derivatives and \nGold ETF & Silver ETF)', 'Index']
Row 2: ['Monthly Portfolio Statement as on June 30,2025']
Row 3: []
Row 4: ['ISIN', 'Name of the Instrument', 'Industry / Rating', 'Quantity', 'Market/Fair Value\n( Rs. in Lacs)', '% to NAV', 'YIELD']
Row 5: ['Equity & Equity related']
Row 6: ['(a) Listed / awaiting listing on Stock Exchanges']
Row 7: ['IBCL05', 'INE090A01021', 'ICICI Bank Limited', 'Banks', 1550000, 22409.9, 0.0352]
Row 8: ['HDFB03', 'INE040A01034', 'HDFC Bank Limited', 'Banks', 770000, 15411.55, 0.0242]
Row 9: ['RIND01', 'INE002A01018', 'Reliance Industries Limited', 'Petroleum Products', 1000000, 15006, 0.0236]
Row 10: ['SBAI02', 'INE062A01020', 'State Bank of India', 'Banks', 1600000, 13125.6, 0.0206]


In [ ]:
# STEP 104: Search June 2025 holdings for gold and silver

with open(june_2025_file, "rb") as file_stream:
    june_workbook = load_workbook(
        file_stream,
        read_only=True,
        data_only=True
    )

    june_sheet = june_workbook["MF"]

    for row_number, row in enumerate(
        june_sheet.iter_rows(values_only=True),
        start=1
    ):
        values = list(row)

        if any(
            isinstance(value, str)
            and ("gold" in value.lower() or "silver" in value.lower())
            for value in values
        ):
            print(f"Row {row_number}: {values}")

    june_workbook.close()

Row 1: ['RLMF806', 'Nippon India Multi Asset Allocation Fund (An open ended scheme investing in Equity, Debt \nand Exchange Traded Commodity Derivatives and \nGold ETF & Silver ETF)', None, None, None, None, None, 'Index']
Row 136: ['LIGS22', 'INE1U3R15011', 'Liquid Gold Series**', 'CRISIL AAA(SO)', 119.999999, 3408.3, 0.0054, 0.0941525]
Row 146: ['BMGB51ME', 'INF204KB17I5', 'Nippon India ETF Gold Bees', None, 86261865, 69000.87, 0.1084, None]
Row 147: ['149758', 'INF204KC1402', 'Nippon India Silver ETF', None, 17869201, 18310.57, 0.0288, None]
Row 165: [None, 'Long', 'FUTCOM_GOLD_05/08/2025', 'Others', 6000, 5764.5, 0.0091, None]
Row 166: [None, 'Long', 'FUTCOM_SILVER_05/09/2025', 'Others', 4800, 5102.02, 0.008, None]
Row 174: [None, None, None, None, None, 'BENCHMARK NAME - 50% OF BSE 500 TRI, 20% OF MSCI WORLD INDEX TRI, 15% OF CRISIL SHORT TERM BOND INDEX 10% OF DOMESTIC PRICES OF GOLD & 5% OF DOMESTIC PRICES OF SILVER', None, None]


In [ ]:
# STEP 106: Find the actual July 2025 filename

for file in nippon_folder.iterdir():
    if file.is_file() and "july" in file.name.lower() and "25" in file.name:
        print(file.name)

NIMF-MONTHLY-PORTFOLIO-31-July-25.xls


In [ ]:
# STEP 107: Identify the July 2025 workbook format

july_2025_file = nippon_folder / "NIMF-MONTHLY-PORTFOLIO-31-July-25.xls"

try:
    with ZipFile(july_2025_file):
        print("Internal format: XLSX")

    with open(july_2025_file, "rb") as file_stream:
        july_workbook = load_workbook(
            file_stream,
            read_only=True,
            data_only=True
        )
        print("Sheet names:", july_workbook.sheetnames)
        july_workbook.close()

except BadZipFile:
    print("Internal format: Legacy XLS")

    july_workbook = xlrd.open_workbook(str(july_2025_file))
    print("Sheet names:", july_workbook.sheet_names())

Internal format: XLSX
Sheet names: ['INDEX', 'GF', 'GS', 'IP', 'IF', 'ST', 'BF', 'MI', 'PS', 'CP', 'PH', 'ME', 'NE', 'NI', 'EO', 'LF', 'FR', 'SD', 'SE', 'SH', 'LQ', 'TS', 'LE', 'LP', 'UB', 'K5', 'EA', 'QP', 'GL', 'NF', 'SC', 'SF', 'AF', 'GD', 'CE', 'CF', 'RD', 'CB', 'JE', 'SX', 'RW', 'RG', 'BP', 'ES', 'NX', 'UE', 'LT', 'NL', 'NB', 'JZ', 'PU', 'GB', 'LB', 'BB', 'SB', 'HS', 'IB', 'CC', 'ON', 'NM', 'RJ', 'G6', 'SN', 'NZ', 'MF', 'NS', 'PP', 'AL', 'NP', 'NV', 'NY', 'NG', 'NH', 'LC', 'HG', 'ET', 'NO', 'SI', 'SO', 'HK', 'CN', 'NN', 'W5', 'ND', 'NJ', 'NK', 'NQ', 'W8', 'NU', 'NW', 'U7', 'U8', 'IT', 'IX', 'BX', 'EW', 'MT', 'IJ', 'ID', 'AN', 'RN', 'AM', 'LV', 'QI', 'BS', 'BZ', 'AP', 'MC', 'LD']


In [ ]:
# STEP 108: Inspect the July 2025 MF worksheet

with open(july_2025_file, "rb") as file_stream:
    july_workbook = load_workbook(
        file_stream,
        read_only=True,
        data_only=True
    )

    july_sheet = july_workbook["MF"]

    for row_number, row in enumerate(
        july_sheet.iter_rows(
            min_row=1,
            max_row=10,
            values_only=True
        ),
        start=1
    ):
        values = [value for value in row if value is not None]
        print(f"Row {row_number}: {values}")

    july_workbook.close()

Row 1: ['RLMF806', 'Nippon India Multi Asset Allocation Fund (An open ended scheme investing in Equity, Debt \nand Exchange Traded Commodity Derivatives and \nGold ETF & Silver ETF)', 'Index']
Row 2: ['Monthly Portfolio Statement as on July 31,2025']
Row 3: []
Row 4: ['ISIN', 'Name of the Instrument', 'Industry / Rating', 'Quantity', 'Market/Fair Value\n( Rs. in Lacs)', '% to NAV', 'YIELD']
Row 5: ['Equity & Equity related']
Row 6: ['(a) Listed / awaiting listing on Stock Exchanges']
Row 7: ['IBCL05', 'INE090A01021', 'ICICI Bank Limited', 'Banks', 1550000, 22961.7, 0.0345]
Row 8: ['SBAI02', 'INE062A01020', 'State Bank of India', 'Banks', 1955352, 15575.36, 0.0234]
Row 9: ['HDFB03', 'INE040A01034', 'HDFC Bank Limited', 'Banks', 770000, 15540.14, 0.0234]
Row 10: ['RIND01', 'INE002A01018', 'Reliance Industries Limited', 'Petroleum Products', 1100000, 15292.2, 0.023]


In [ ]:
# STEP 109: Search July 2025 holdings for gold and silver

with open(july_2025_file, "rb") as file_stream:
    july_workbook = load_workbook(
        file_stream,
        read_only=True,
        data_only=True
    )

    july_sheet = july_workbook["MF"]

    for row_number, row in enumerate(
        july_sheet.iter_rows(values_only=True),
        start=1
    ):
        values = list(row)

        if any(
            isinstance(value, str)
            and ("gold" in value.lower() or "silver" in value.lower())
            for value in values
        ):
            print(f"Row {row_number}: {values}")

    july_workbook.close()

Row 1: ['RLMF806', 'Nippon India Multi Asset Allocation Fund (An open ended scheme investing in Equity, Debt \nand Exchange Traded Commodity Derivatives and \nGold ETF & Silver ETF)', None, None, None, None, None, 'Index']
Row 137: ['LIGS22', 'INE1U3R15011', 'Liquid Gold Series**', 'CRISIL AAA(SO)', 119.999999, 3162.79, 0.0048, 0.092073]
Row 149: ['BMGB51ME', 'INF204KB17I5', 'Nippon India ETF Gold Bees', None, 86261865, 70907.25, 0.1066, None]
Row 150: ['149758', 'INF204KC1402', 'Nippon India Silver ETF', None, 17869201, 19102.18, 0.0287, None]
Row 168: [None, 'Long', 'FUTCOM_SILVER_05/09/2025', 'Others', 6000, 6598.32, 0.0099, None]
Row 169: [None, 'Long', 'FUTCOM_GOLD_03/10/2025', 'Others', 6000, 5926.14, 0.0089, None]
Row 178: [None, None, None, None, None, 'BENCHMARK NAME - 50% OF BSE 500 TRI, 20% OF MSCI WORLD INDEX TRI, 15% OF CRISIL SHORT TERM BOND INDEX 10% OF DOMESTIC PRICES OF GOLD & 5% OF DOMESTIC PRICES OF SILVER', None, None]


In [ ]:
# STEP 111: Find the actual August 2025 filename

for file in nippon_folder.iterdir():
    if file.is_file() and "aug" in file.name.lower() and "25" in file.name:
        print(file.name)

NIMF-MONTHLY-PORTFOLIO-31-Aug-25.xls


In [ ]:
# STEP 112: Identify the August 2025 workbook format

august_2025_file = nippon_folder / "NIMF-MONTHLY-PORTFOLIO-31-Aug-25.xls"

try:
    with ZipFile(august_2025_file):
        print("Internal format: XLSX")

    with open(august_2025_file, "rb") as file_stream:
        august_workbook = load_workbook(
            file_stream,
            read_only=True,
            data_only=True
        )
        print("Sheet names:", august_workbook.sheetnames)
        august_workbook.close()

except BadZipFile:
    print("Internal format: Legacy XLS")

    august_workbook = xlrd.open_workbook(str(august_2025_file))
    print("Sheet names:", august_workbook.sheet_names())

Internal format: XLSX
Sheet names: ['Index', 'GF', 'GS', 'IP', 'IF', 'ST', 'BF', 'MI', 'PS', 'CP', 'PH', 'ME', 'NE', 'NI', 'EO', 'LF', 'FR', 'SD', 'SE', 'SH', 'LQ', 'TS', 'LE', 'LP', 'UB', 'K5', 'EA', 'QP', 'GL', 'NF', 'SC', 'SF', 'AF', 'GD', 'CE', 'CF', 'RD', 'CB', 'JE', 'SX', 'RW', 'RG', 'BP', 'ES', 'NX', 'UE', 'LT', 'NL', 'NB', 'JZ', 'PU', 'GB', 'LB', 'BB', 'SB', 'HS', 'IB', 'CC', 'ON', 'NM', 'RJ', 'G6', 'SN', 'NZ', 'MF', 'NS', 'PP', 'AL', 'NP', 'NV', 'NY', 'NG', 'NH', 'LC', 'HG', 'ET', 'NO', 'SI', 'SO', 'HK', 'CN', 'NN', 'W5', 'ND', 'NJ', 'NK', 'NQ', 'W8', 'NU', 'NW', 'U7', 'U8', 'IT', 'IX', 'BX', 'EW', 'MT', 'IJ', 'ID', 'AN', 'RN', 'AM', 'LV', 'QI', 'BS', 'BZ', 'AP', 'MC', 'LD', 'MG', 'MZ']


In [ ]:
# STEP 113: Inspect the August 2025 MF worksheet

with open(august_2025_file, "rb") as file_stream:
    august_workbook = load_workbook(
        file_stream,
        read_only=True,
        data_only=True
    )

    august_sheet = august_workbook["MF"]

    for row_number, row in enumerate(
        august_sheet.iter_rows(
            min_row=1,
            max_row=10,
            values_only=True
        ),
        start=1
    ):
        values = [value for value in row if value is not None]
        print(f"Row {row_number}: {values}")

    august_workbook.close()

Row 1: ['RLMF806', 'Nippon India Multi Asset Allocation Fund (An open ended scheme investing in Equity, Debt \nand Exchange Traded Commodity Derivatives and \nGold ETF & Silver ETF)', 'Index']
Row 2: ['Monthly Portfolio Statement as on August 31,2025']
Row 3: []
Row 4: ['ISIN', 'Name of the Instrument', 'Industry / Rating', 'Quantity', 'Market/Fair Value\n( Rs. in Lacs)', '% to NAV', 'YIELD']
Row 5: ['Equity & Equity related']
Row 6: ['(a) Listed / awaiting listing on Stock Exchanges']
Row 7: ['IBCL05', 'INE090A01021', 'ICICI Bank Limited', 'Banks', 1550000, 21665.9, 0.0311]
Row 8: ['SBAI02', 'INE062A01020', 'State Bank of India', 'Banks', 1955352, 15691.7, 0.0225]
Row 9: ['INFS02', 'INE009A01021', 'Infosys Limited', 'IT - Software', 1025000, 15063.4, 0.0216]
Row 10: ['RIND01', 'INE002A01018', 'Reliance Industries Limited', 'Petroleum Products', 1100000, 14929.2, 0.0215]


In [ ]:
# STEP 114: Search August 2025 holdings for gold and silver

with open(august_2025_file, "rb") as file_stream:
    august_workbook = load_workbook(
        file_stream,
        read_only=True,
        data_only=True
    )

    august_sheet = august_workbook["MF"]

    for row_number, row in enumerate(
        august_sheet.iter_rows(values_only=True),
        start=1
    ):
        values = list(row)

        if any(
            isinstance(value, str)
            and ("gold" in value.lower() or "silver" in value.lower())
            for value in values
        ):
            print(f"Row {row_number}: {values}")

    august_workbook.close()

Row 1: ['RLMF806', 'Nippon India Multi Asset Allocation Fund (An open ended scheme investing in Equity, Debt \nand Exchange Traded Commodity Derivatives and \nGold ETF & Silver ETF)', None, None, None, None, None, 'Index']
Row 144: ['LIGS22', 'INE1U3R15011', 'Liquid Gold Series**', 'CRISIL AAA(SO)', 119.999999, 2900.57, 0.0042, 0.092423]
Row 154: ['BMGB51ME', 'INF204KB17I5', 'Nippon India ETF Gold Bees', None, 91266971, 77887.23, 0.1119, None]
Row 155: ['149758', 'INF204KC1402', 'Nippon India Silver ETF', None, 18869201, 21431.64, 0.0308, None]
Row 181: [None, None, None, None, None, 'BENCHMARK NAME - 50% OF BSE 500 TRI, 20% OF MSCI WORLD INDEX TRI, 15% OF CRISIL SHORT TERM BOND INDEX 10% OF DOMESTIC PRICES OF GOLD & 5% OF DOMESTIC PRICES OF SILVER', None, None]


In [ ]:
# STEP 116: Find the actual September 2025 filename

for file in nippon_folder.iterdir():
    if file.is_file() and "sep" in file.name.lower() and "25" in file.name:
        print(file.name)

NIMF-MONTHLY-PORTFOLIO-30-Sep-25.xls


In [ ]:
# STEP 117: Identify the September 2025 workbook format

september_2025_file = nippon_folder / "NIMF-MONTHLY-PORTFOLIO-30-Sep-25.xls"

try:
    with ZipFile(september_2025_file):
        print("Internal format: XLSX")

    with open(september_2025_file, "rb") as file_stream:
        september_workbook = load_workbook(
            file_stream,
            read_only=True,
            data_only=True
        )
        print("Sheet names:", september_workbook.sheetnames)
        september_workbook.close()

except BadZipFile:
    print("Internal format: Legacy XLS")

    september_workbook = xlrd.open_workbook(str(september_2025_file))
    print("Sheet names:", september_workbook.sheet_names())

Internal format: XLSX
Sheet names: ['Index', 'GF', 'GS', 'IP', 'IF', 'ST', 'BF', 'MI', 'PS', 'CP', 'PH', 'ME', 'NE', 'NI', 'EO', 'LF', 'FR', 'SD', 'SE', 'SH', 'LQ', 'TS', 'LE', 'LP', 'UB', 'K5', 'EA', 'QP', 'GL', 'NF', 'SC', 'SF', 'AF', 'GD', 'CE', 'CF', 'RD', 'CB', 'JE', 'SX', 'RW', 'RG', 'BP', 'ES', 'NX', 'UE', 'LT', 'NL', 'NB', 'JZ', 'PU', 'GB', 'LB', 'BB', 'SB', 'HS', 'IB', 'CC', 'ON', 'NM', 'RJ', 'G6', 'SN', 'NZ', 'MF', 'NS', 'PP', 'AL', 'NP', 'NV', 'NY', 'NG', 'NH', 'LC', 'HG', 'ET', 'NO', 'SI', 'SO', 'HK', 'CN', 'NN', 'W5', 'ND', 'NJ', 'NK', 'NQ', 'W8', 'NU', 'NW', 'U7', 'U8', 'IT', 'IX', 'BX', 'EW', 'MT', 'IJ', 'ID', 'AN', 'RN', 'AM', 'LV', 'QI', 'BS', 'BZ', 'AP', 'MC', 'LD', 'MG', 'MZ']


In [ ]:
# STEP 118: Inspect the September 2025 MF worksheet

with open(september_2025_file, "rb") as file_stream:
    september_workbook = load_workbook(
        file_stream,
        read_only=True,
        data_only=True
    )

    september_sheet = september_workbook["MF"]

    for row_number, row in enumerate(
        september_sheet.iter_rows(
            min_row=1,
            max_row=10,
            values_only=True
        ),
        start=1
    ):
        values = [value for value in row if value is not None]
        print(f"Row {row_number}: {values}")

    september_workbook.close()

Row 1: ['RLMF806', 'Nippon India Multi Asset Allocation Fund (An open ended scheme investing in Equity, Debt \nand Exchange Traded Commodity Derivatives and \nGold ETF & Silver ETF)', 'Index']
Row 2: ['Monthly Portfolio Statement as on September 30,2025']
Row 3: []
Row 4: ['ISIN', 'Name of the Instrument', 'Industry / Rating', 'Quantity', 'Market/Fair Value\n( Rs. in Lacs)', '% to NAV', 'YIELD']
Row 5: ['Equity & Equity related']
Row 6: ['(a) Listed / awaiting listing on Stock Exchanges']
Row 7: ['HDFB03', 'INE040A01034', 'HDFC Bank Limited', 'Banks', 2513820, 23906.43, 0.0308]
Row 8: ['IBCL05', 'INE090A01021', 'ICICI Bank Limited', 'Banks', 1550000, 20894, 0.0269]
Row 9: ['SBAI02', 'INE062A01020', 'State Bank of India', 'Banks', 2069273, 18053.37, 0.0233]
Row 10: ['RIND01', 'INE002A01018', 'Reliance Industries Limited', 'Petroleum Products', 1100000, 15004, 0.0193]


In [ ]:
# STEP 119: Search September 2025 holdings for gold and silver

with open(september_2025_file, "rb") as file_stream:
    september_workbook = load_workbook(
        file_stream,
        read_only=True,
        data_only=True
    )

    september_sheet = september_workbook["MF"]

    for row_number, row in enumerate(
        september_sheet.iter_rows(values_only=True),
        start=1
    ):
        values = list(row)

        if any(
            isinstance(value, str)
            and ("gold" in value.lower() or "silver" in value.lower())
            for value in values
        ):
            print(f"Row {row_number}: {values}")

    september_workbook.close()

Row 1: ['RLMF806', 'Nippon India Multi Asset Allocation Fund (An open ended scheme investing in Equity, Debt \nand Exchange Traded Commodity Derivatives and \nGold ETF & Silver ETF)', None, None, None, None, None, 'Index']
Row 142: ['LIGS22', 'INE1U3R15011', 'Liquid Gold Series**', 'CRISIL AAA(SO)', 119.999998, 2684.53, 0.0035, 0.0909805]
Row 153: ['BMGB51ME', 'INF204KB17I5', 'Nippon India ETF Gold Bees', None, 91266971, 87479.39, 0.1127, None]
Row 154: ['149758', 'INF204KC1402', 'Nippon India Silver ETF', None, 18869201, 25675.32, 0.0331, None]
Row 172: [None, 'Long', 'FUTCOM_SILVER_05/12/2025', 'Others', 3900, 5543.66, 0.0071, None]
Row 173: [None, 'Long', 'FUTCOM_GOLD_05/12/2025', 'Others', 4000, 4690.6, 0.006, None]
Row 175: [None, None, None, None, None, 'BENCHMARK NAME - 50% OF BSE 500 TRI, 20% OF MSCI WORLD INDEX TRI, 15% OF CRISIL SHORT TERM BOND INDEX 10% OF DOMESTIC PRICES OF GOLD & 5% OF DOMESTIC PRICES OF SILVER', None, None]


In [ ]:
# STEP 121: Find the actual October 2025 filename

for file in nippon_folder.iterdir():
    if file.is_file() and "oct" in file.name.lower() and "25" in file.name:
        print(file.name)

NIMF-MONTHLY-PORTFOLIO-31-Oct-25.xls


In [ ]:
# STEP 122: Check the October 2025 workbook format

october_2025_file = nippon_folder / "NIMF-MONTHLY-PORTFOLIO-31-Oct-25.xls"

with open(october_2025_file, "rb") as file_stream:
    file_signature = file_stream.read(8)

print("File:", october_2025_file.name)
print("File signature:", file_signature)

if file_signature.startswith(b"PK"):
    print("Format: ZIP-based Excel workbook; use openpyxl.")
else:
    print("Format: Possibly a legacy XLS workbook; check with xlrd.")

File: NIMF-MONTHLY-PORTFOLIO-31-Oct-25.xls
File signature: b'PK\x03\x04\x14\x00\x06\x00'
Format: ZIP-based Excel workbook; use openpyxl.


In [ ]:
# STEP 123: Inspect the October 2025 MF worksheet

with open(october_2025_file, "rb") as file_stream:
    october_workbook = load_workbook(
        file_stream,
        read_only=True,
        data_only=True
    )

    print("Worksheets:", october_workbook.sheetnames)

    october_sheet = october_workbook["MF"]

    for row_number, row in enumerate(
        october_sheet.iter_rows(
            min_row=1,
            max_row=10,
            values_only=True
        ),
        start=1
    ):
        values = [value for value in row if value is not None]
        print(f"Row {row_number}: {values}")

    october_workbook.close()

Worksheets: ['Index', 'GF', 'GS', 'IP', 'IF', 'ST', 'BF', 'MI', 'PS', 'CP', 'PH', 'ME', 'NE', 'NI', 'EO', 'LF', 'FR', 'SD', 'SE', 'SH', 'LQ', 'TS', 'LE', 'LP', 'UB', 'K5', 'EA', 'QP', 'GL', 'NF', 'SC', 'SF', 'AF', 'GD', 'CE', 'CF', 'RD', 'CB', 'JE', 'SX', 'RW', 'RG', 'BP', 'ES', 'NX', 'UE', 'LT', 'NL', 'NB', 'JZ', 'PU', 'GB', 'LB', 'BB', 'SB', 'HS', 'IB', 'CC', 'ON', 'NM', 'RJ', 'G6', 'SN', 'NZ', 'MF', 'NS', 'PP', 'AL', 'NP', 'NV', 'NY', 'NG', 'NH', 'LC', 'HG', 'ET', 'NO', 'SI', 'SO', 'HK', 'CN', 'NN', 'W5', 'ND', 'NJ', 'NK', 'NQ', 'W8', 'NU', 'NW', 'U7', 'U8', 'IT', 'IX', 'BX', 'EW', 'MT', 'IJ', 'ID', 'AN', 'RN', 'AM', 'LV', 'QI', 'BS', 'BZ', 'AP', 'MC', 'LD', 'MG', 'MZ']
Row 1: ['RLMF806', 'Nippon India Multi Asset Allocation Fund (An open ended scheme investing in Equity, Debt \nand Exchange Traded Commodity Derivatives and \nGold ETF & Silver ETF)', 'Index']
Row 2: ['Monthly Portfolio Statement as on October 31,2025']
Row 3: []
Row 4: ['ISIN', 'Name of the Instrument', 'Industry / 

In [ ]:
# STEP 124: Search October 2025 holdings for gold and silver

with open(october_2025_file, "rb") as file_stream:
    october_workbook = load_workbook(
        file_stream,
        read_only=True,
        data_only=True
    )

    october_sheet = october_workbook["MF"]

    for row_number, row in enumerate(
        october_sheet.iter_rows(values_only=True),
        start=1
    ):
        values = list(row)

        if any(
            isinstance(value, str)
            and ("gold" in value.lower() or "silver" in value.lower())
            for value in values
        ):
            print(f"Row {row_number}: {values}")

    october_workbook.close()

Row 1: ['RLMF806', 'Nippon India Multi Asset Allocation Fund (An open ended scheme investing in Equity, Debt \nand Exchange Traded Commodity Derivatives and \nGold ETF & Silver ETF)', None, None, None, None, None, 'Index']
Row 147: ['LIGS22', 'INE1U3R15011', 'Liquid Gold Series**', 'CRISIL AAA(SO)', 119.999998, 2543.77, 0.0029, 0.09055]
Row 159: ['BMGB51ME', 'INF204KB17I5', 'Nippon India ETF Gold Bees', None, 95266971, 95286.02, 0.1092, None]
Row 160: ['149758', 'INF204KC1402', 'Nippon India Silver ETF', None, 18831996, 26829.94, 0.0308, None]
Row 178: [None, 'Long', 'FUTCOM_GOLD_05/12/2025', 'Others', 12000, 14547.84, 0.0167, None]
Row 179: [None, 'Long', 'FUTCOM_SILVER_05/12/2025', 'Others', 7800, 11566.39, 0.0133, None]
Row 181: [None, None, None, None, None, 'BENCHMARK NAME - 50% OF BSE 500 TRI, 20% OF MSCI WORLD INDEX TRI, 15% OF CRISIL SHORT TERM BOND INDEX 10% OF DOMESTIC PRICES OF GOLD & 5% OF DOMESTIC PRICES OF SILVER', None, None]


In [ ]:
# STEP 126: Find the November 2025 filename

for file in nippon_folder.iterdir():
    if file.is_file() and "nov" in file.name.lower() and "25" in file.name:
        print(file.name)

NIMF-MONTHLY-PORTFOLIO-Nov-25.xls


In [ ]:
# STEP 127: Check the November 2025 workbook format

november_2025_file = nippon_folder / "NIMF-MONTHLY-PORTFOLIO-Nov-25.xls"

with open(november_2025_file, "rb") as file_stream:
    file_signature = file_stream.read(8)

print("File:", november_2025_file.name)
print("File signature:", file_signature)

if file_signature.startswith(b"PK"):
    print("Format: ZIP-based Excel workbook; use openpyxl.")
else:
    print("Format: Possibly a legacy XLS workbook; check with xlrd.")

File: NIMF-MONTHLY-PORTFOLIO-Nov-25.xls
File signature: b'\xd0\xcf\x11\xe0\xa1\xb1\x1a\xe1'
Format: Possibly a legacy XLS workbook; check with xlrd.


In [ ]:
# STEP 128: Inspect the November 2025 MF worksheet

november_workbook = xlrd.open_workbook(
    str(november_2025_file)
)

print("Worksheets:", november_workbook.sheet_names())

november_sheet = november_workbook.sheet_by_name("MF")

for row_number in range(min(10, november_sheet.nrows)):
    values = [
        value
        for value in november_sheet.row_values(row_number)
        if value != ""
    ]
    print(f"Row {row_number + 1}: {values}")

Worksheets: ['Index', 'GF', 'GS', 'IP', 'IF', 'ST', 'BF', 'MI', 'PS', 'CP', 'PH', 'ME', 'NE', 'NI', 'EO', 'LF', 'FR', 'SD', 'SE', 'SH', 'LQ', 'TS', 'LE', 'LP', 'UB', 'K5', 'EA', 'QP', 'GL', 'NF', 'SC', 'SF', 'AF', 'GD', 'CE', 'CF', 'RD', 'CB', 'JE', 'SX', 'RW', 'RG', 'BP', 'ES', 'NX', 'UE', 'LT', 'NL', 'NB', 'JZ', 'PU', 'GB', 'LB', 'BB', 'SB', 'HS', 'IB', 'CC', 'ON', 'NM', 'RJ', 'G6', 'SN', 'NZ', 'MF', 'NS', 'PP', 'AL', 'NP', 'NV', 'NY', 'NG', 'NH', 'LC', 'HG', 'ET', 'NO', 'SI', 'SO', 'HK', 'CN', 'NN', 'W5', 'ND', 'NJ', 'NK', 'NQ', 'W8', 'NU', 'NW', 'U7', 'U8', 'IT', 'IX', 'BX', 'EW', 'MT', 'IJ', 'ID', 'AN', 'RN', 'AM', 'LV', 'QI', 'BS', 'BZ', 'AP', 'MC', 'LD', 'MG', 'MZ']
Row 1: ['RLMF806', 'Nippon India Multi Asset Allocation Fund (An open ended scheme investing in Equity, Debt \nand Exchange Traded Commodity Derivatives and \nGold ETF & Silver ETF)', 'Index']
Row 2: ['Monthly Portfolio Statement as on November 30,2025']
Row 3: []
Row 4: ['ISIN', 'Name of the Instrument', 'Industry /

In [ ]:
# STEP 129: Search November 2025 holdings for gold and silver

for row_number in range(november_sheet.nrows):
    values = november_sheet.row_values(row_number)

    if any(
        isinstance(value, str)
        and ("gold" in value.lower() or "silver" in value.lower())
        for value in values
    ):
        print(f"Row {row_number + 1}: {values}")

Row 1: ['RLMF806', 'Nippon India Multi Asset Allocation Fund (An open ended scheme investing in Equity, Debt \nand Exchange Traded Commodity Derivatives and \nGold ETF & Silver ETF)', '', '', '', '', '', 'Index']
Row 155: ['LIGS22', 'INE1U3R15011', 'Liquid Gold Series**', 'CRISIL AAA(SO)', 119.999998, 2121.25, 0.0022, 0.091]
Row 166: ['BMGB51ME', 'INF204KB17I5', 'Nippon India ETF Gold Bees', '', 95266971.0, 99668.31, 0.1038, '']
Row 167: ['149758', 'INF204KC1402', 'Nippon India Silver ETF', '', 18831996.0, 29522.92, 0.0308, '']
Row 185: ['', 'Long', 'FUTCOM_GOLD_05/02/2026', 'Others', 8000.0, 10360.32, 0.0108, '']
Row 186: ['', 'Long', 'FUTCOM_SILVER_05/03/2026', 'Others', 3900.0, 6824.26, 0.0071, '']
Row 188: ['', '', '', '', '', 'BENCHMARK NAME - 50% OF BSE 500 TRI, 20% OF MSCI WORLD INDEX TRI, 15% OF CRISIL SHORT TERM BOND INDEX 10% OF DOMESTIC PRICES OF GOLD & 5% OF DOMESTIC PRICES OF SILVER', '', '']


In [ ]:
# STEP 131: Find the December 2025 filename

for file in nippon_folder.iterdir():
    if file.is_file() and "dec" in file.name.lower() and "25" in file.name:
        print(file.name)

NIMF-MONTHLY-PORTFOLIO-31-Dec-25.xls


In [ ]:
# STEP 132: Check the December 2025 workbook format

december_2025_file = nippon_folder / "NIMF-MONTHLY-PORTFOLIO-31-Dec-25.xls"

with open(december_2025_file, "rb") as file_stream:
    file_signature = file_stream.read(8)

print("File:", december_2025_file.name)
print("File signature:", file_signature)

if file_signature.startswith(b"PK"):
    print("Format: ZIP-based Excel workbook; use openpyxl.")
else:
    print("Format: Possibly a legacy XLS workbook; check with xlrd.")

File: NIMF-MONTHLY-PORTFOLIO-31-Dec-25.xls
File signature: b'PK\x03\x04\x14\x00\x06\x00'
Format: ZIP-based Excel workbook; use openpyxl.


In [ ]:
# STEP 133: Inspect the December 2025 MF worksheet

with open(december_2025_file, "rb") as file_stream:
    december_workbook = load_workbook(
        file_stream,
        read_only=True,
        data_only=True
    )

    print("Worksheets:", december_workbook.sheetnames)

    december_sheet = december_workbook["MF"]

    for row_number, row in enumerate(
        december_sheet.iter_rows(
            min_row=1,
            max_row=10,
            values_only=True
        ),
        start=1
    ):
        values = [value for value in row if value is not None]
        print(f"Row {row_number}: {values}")

    december_workbook.close()

Worksheets: ['Index', 'GF', 'GS', 'IP', 'IF', 'ST', 'BF', 'MI', 'PS', 'CP', 'PH', 'ME', 'NE', 'NI', 'EO', 'LF', 'FR', 'SD', 'SE', 'SH', 'LQ', 'TS', 'LE', 'LP', 'UB', 'K5', 'EA', 'QP', 'GL', 'NF', 'SC', 'SF', 'AF', 'GD', 'CE', 'CF', 'RD', 'CB', 'JE', 'SX', 'RW', 'RG', 'BP', 'ES', 'NX', 'UE', 'LT', 'NL', 'NB', 'JZ', 'PU', 'GB', 'LB', 'BB', 'SB', 'HS', 'IB', 'CC', 'ON', 'NM', 'RJ', 'G6', 'SN', 'NZ', 'MF', 'NS', 'PP', 'AL', 'NP', 'NV', 'NY', 'NG', 'NH', 'LC', 'HG', 'ET', 'NO', 'SI', 'SO', 'HK', 'CN', 'NN', 'W5', 'ND', 'NJ', 'NK', 'NQ', 'W8', 'NU', 'NW', 'U7', 'U8', 'IT', 'IX', 'BX', 'EW', 'MT', 'IJ', 'ID', 'AN', 'RN', 'AM', 'LV', 'QI', 'BS', 'BZ', 'AP', 'MC', 'LD', 'MG', 'MZ']
Row 1: ['RLMF806', 'Nippon India Multi Asset Allocation Fund (An open ended scheme investing in Equity, Debt \nand Exchange Traded Commodity Derivatives and \nGold ETF & Silver ETF)', 'Index']
Row 2: ['Monthly Portfolio Statement as on December 31,2025']
Row 3: []
Row 4: ['ISIN', 'Name of the Instrument', 'Industry /

In [ ]:
# STEP 134: Search December 2025 holdings for gold and silver

with open(december_2025_file, "rb") as file_stream:
    december_workbook = load_workbook(
        file_stream,
        read_only=True,
        data_only=True
    )

    december_sheet = december_workbook["MF"]

    for row_number, row in enumerate(
        december_sheet.iter_rows(values_only=True),
        start=1
    ):
        values = list(row)

        if any(
            isinstance(value, str)
            and ("gold" in value.lower() or "silver" in value.lower())
            for value in values
        ):
            print(f"Row {row_number}: {values}")

    december_workbook.close()

Row 1: ['RLMF806', 'Nippon India Multi Asset Allocation Fund (An open ended scheme investing in Equity, Debt \nand Exchange Traded Commodity Derivatives and \nGold ETF & Silver ETF)', None, None, None, None, None, 'Index']
Row 173: ['LIGS22', 'INE1U3R15011', 'Liquid Gold Series**', 'CRISIL AAA(SO)', 119.999997, 2030.89, 0.0019, 0.09295]
Row 187: ['BMGB51ME', 'INF204KB17I5', 'Nippon India ETF Gold Bees', None, 97266971, 107100.66, 0.1005, None]
Row 188: ['149758', 'INF204KC1402', 'Nippon India Silver ETF', None, 20900450, 45025.84, 0.0422, None]
Row 206: [None, 'Long', 'FUTCOM_GOLD_05/02/2026', 'Others', 15000, 20317.05, 0.0191, None]
Row 207: [None, 'Long', 'FUTCOM_SILVER_05/03/2026', 'Others', 5700, 13434.96, 0.0126, None]
Row 214: [None, None, None, None, None, 'BENCHMARK NAME - 50% OF BSE 500 TRI, 20% OF MSCI WORLD INDEX TRI, 15% OF CRISIL SHORT TERM BOND INDEX 10% OF DOMESTIC PRICES OF GOLD & 5% OF DOMESTIC PRICES OF SILVER', None, None]


In [ ]:
# STEP 136: Find the January 2026 filename

for file in nippon_folder.iterdir():
    if file.is_file() and "jan" in file.name.lower() and "26" in file.name:
        print(file.name)

NIMF-MONTHLY-PORTFOLIO-31-Jan-26.xls


In [ ]:
# STEP 137: Check the January 2026 workbook format

january_2026_file = nippon_folder / "NIMF-MONTHLY-PORTFOLIO-31-Jan-26.xls"

with open(january_2026_file, "rb") as file_stream:
    file_signature = file_stream.read(8)

print("File:", january_2026_file.name)
print("File signature:", file_signature)

if file_signature.startswith(b"PK"):
    print("Format: ZIP-based Excel workbook; use openpyxl.")
else:
    print("Format: Possibly a legacy XLS workbook; check with xlrd.")

File: NIMF-MONTHLY-PORTFOLIO-31-Jan-26.xls
File signature: b'PK\x03\x04\x14\x00\x06\x00'
Format: ZIP-based Excel workbook; use openpyxl.


In [ ]:
# STEP 138: Inspect the January 2026 MF worksheet

with open(january_2026_file, "rb") as file_stream:
    january_workbook = load_workbook(
        file_stream,
        read_only=True,
        data_only=True
    )

    print("Worksheets:", january_workbook.sheetnames)

    january_sheet = january_workbook["MF"]

    for row_number, row in enumerate(
        january_sheet.iter_rows(
            min_row=1,
            max_row=10,
            values_only=True
        ),
        start=1
    ):
        values = [value for value in row if value is not None]
        print(f"Row {row_number}: {values}")

    january_workbook.close()

Worksheets: ['Index', 'GF', 'GS', 'IP', 'IF', 'ST', 'BF', 'MI', 'PS', 'CP', 'PH', 'ME', 'NE', 'NI', 'EO', 'LF', 'FR', 'SD', 'SE', 'SH', 'LQ', 'TS', 'LE', 'LP', 'UB', 'K5', 'EA', 'QP', 'GL', 'NF', 'SC', 'SF', 'AF', 'GD', 'CE', 'CF', 'RD', 'CB', 'JE', 'SX', 'RW', 'RG', 'BP', 'ES', 'NX', 'UE', 'LT', 'NL', 'NB', 'JZ', 'PU', 'GB', 'LB', 'BB', 'SB', 'HS', 'IB', 'CC', 'ON', 'NM', 'RJ', 'G6', 'SN', 'NZ', 'MF', 'NS', 'PP', 'AL', 'NP', 'NV', 'NY', 'NG', 'NH', 'LC', 'HG', 'ET', 'NO', 'SI', 'SO', 'HK', 'CN', 'NN', 'W5', 'ND', 'NJ', 'NK', 'NQ', 'W8', 'NU', 'NW', 'U7', 'U8', 'IT', 'IX', 'BX', 'EW', 'MT', 'IJ', 'ID', 'AN', 'RN', 'AM', 'LV', 'QI', 'BS', 'BZ', 'AP', 'MC', 'LD', 'MG', 'MZ']
Row 1: ['RLMF806', 'Nippon India Multi Asset Allocation Fund (An open ended scheme investing in Equity, Debt \nand Exchange Traded Commodity Derivatives and \nGold ETF & Silver ETF)', 'Index']
Row 2: ['Monthly Portfolio Statement as on January 31,2026']
Row 3: []
Row 4: ['ISIN', 'Name of the Instrument', 'Industry / 

In [ ]:
# STEP 139: Search January 2026 holdings for gold and silver

with open(january_2026_file, "rb") as file_stream:
    january_workbook = load_workbook(
        file_stream,
        read_only=True,
        data_only=True
    )

    january_sheet = january_workbook["MF"]

    for row_number, row in enumerate(
        january_sheet.iter_rows(values_only=True),
        start=1
    ):
        values = list(row)

        if any(
            isinstance(value, str)
            and ("gold" in value.lower() or "silver" in value.lower())
            for value in values
        ):
            print(f"Row {row_number}: {values}")

    january_workbook.close()

Row 1: ['RLMF806', 'Nippon India Multi Asset Allocation Fund (An open ended scheme investing in Equity, Debt \nand Exchange Traded Commodity Derivatives and \nGold ETF & Silver ETF)', None, None, None, None, None, 'Index']
Row 169: ['LIGS22', 'INE1U3R15011', 'Liquid Gold Series**', 'CRISIL AAA(SO)', 119.999997, 1847.67, 0.0015, 0.09765]
Row 184: ['BMGB51ME', 'INF204KB17I5', 'Nippon India ETF Gold Bees', None, 97272043, 132026.27, 0.1055, None]
Row 185: ['149758', 'INF204KC1402', 'Nippon India Silver ETF', None, 20900450, 65963.33, 0.0527, None]
Row 203: [None, 'Long', 'FUTCOM_GOLD_02/04/2026', 'Others', 17000, 25898.65, 0.0207, None]
Row 204: [None, 'Long', 'FUTCOM_SILVER_05/03/2026', 'Others', 3300, 9633.53, 0.0077, None]
Row 211: [None, None, None, None, None, 'BENCHMARK NAME - 50% OF BSE 500 TRI, 20% OF MSCI WORLD INDEX TRI, 15% OF CRISIL SHORT TERM BOND INDEX 10% OF DOMESTIC PRICES OF GOLD & 5% OF DOMESTIC PRICES OF SILVER', None, None]


In [ ]:
# STEP 141: Find the February 2026 filename

for file in nippon_folder.iterdir():
    if file.is_file() and "feb" in file.name.lower() and "26" in file.name:
        print(file.name)

NIMF-MONTHLY-PORTFOLIO-28-Feb-26.xls


In [ ]:
# STEP 142: Check the February 2026 workbook format

february_2026_file = nippon_folder / "NIMF-MONTHLY-PORTFOLIO-28-Feb-26.xls"

with open(february_2026_file, "rb") as file_stream:
    file_signature = file_stream.read(8)

print("File:", february_2026_file.name)
print("File signature:", file_signature)

if file_signature.startswith(b"PK"):
    print("Format: ZIP-based Excel workbook; use openpyxl.")
else:
    print("Format: Possibly a legacy XLS workbook; check with xlrd.")

File: NIMF-MONTHLY-PORTFOLIO-28-Feb-26.xls
File signature: b'PK\x03\x04\x14\x00\x06\x00'
Format: ZIP-based Excel workbook; use openpyxl.


In [ ]:
# STEP 143: Inspect the February 2026 MF worksheet

with open(february_2026_file, "rb") as file_stream:
    february_workbook = load_workbook(
        file_stream,
        read_only=True,
        data_only=True
    )

    print("Worksheets:", february_workbook.sheetnames)

    february_sheet = february_workbook["MF"]

    for row_number, row in enumerate(
        february_sheet.iter_rows(
            min_row=1,
            max_row=10,
            values_only=True
        ),
        start=1
    ):
        values = [value for value in row if value is not None]
        print(f"Row {row_number}: {values}")

    february_workbook.close()

Worksheets: ['Index', 'GF', 'GS', 'IP', 'IF', 'ST', 'BF', 'MI', 'PS', 'CP', 'PH', 'ME', 'NE', 'NI', 'EO', 'LF', 'FR', 'SD', 'SE', 'SH', 'LQ', 'TS', 'LE', 'LP', 'UB', 'K5', 'EA', 'QP', 'GL', 'NF', 'SC', 'SF', 'AF', 'GD', 'CE', 'CF', 'RD', 'CB', 'JE', 'SX', 'RW', 'RG', 'BP', 'ES', 'NX', 'UE', 'LT', 'NL', 'NB', 'JZ', 'PU', 'GB', 'LB', 'BB', 'SB', 'HS', 'IB', 'CC', 'ON', 'NM', 'RJ', 'G6', 'SN', 'NZ', 'MF', 'NS', 'PP', 'AL', 'NP', 'NV', 'NY', 'NG', 'NH', 'LC', 'HG', 'ET', 'NO', 'SI', 'SO', 'HK', 'CN', 'NN', 'W5', 'ND', 'NJ', 'NK', 'NQ', 'W8', 'NU', 'NW', 'U7', 'U8', 'IT', 'IX', 'BX', 'EW', 'MT', 'IJ', 'ID', 'AN', 'RN', 'AM', 'LV', 'QI', 'BS', 'BZ', 'AP', 'MC', 'LD', 'MG', 'MZ']
Row 1: ['RLMF806', 'Nippon India Multi Asset Allocation Fund (An open ended scheme investing in Equity, Debt \nand Exchange Traded Commodity Derivatives and \nGold ETF & Silver ETF)', 'Index']
Row 2: ['Monthly Portfolio Statement as on February 28,2026']
Row 3: []
Row 4: ['ISIN', 'Name of the Instrument', 'Industry /

In [ ]:
# STEP 144: Search February 2026 holdings for gold and silver

with open(february_2026_file, "rb") as file_stream:
    february_workbook = load_workbook(
        file_stream,
        read_only=True,
        data_only=True
    )

    february_sheet = february_workbook["MF"]

    for row_number, row in enumerate(
        february_sheet.iter_rows(values_only=True),
        start=1
    ):
        values = list(row)

        if any(
            isinstance(value, str)
            and ("gold" in value.lower() or "silver" in value.lower())
            for value in values
        ):
            print(f"Row {row_number}: {values}")

    february_workbook.close()
    

Row 1: ['RLMF806', 'Nippon India Multi Asset Allocation Fund (An open ended scheme investing in Equity, Debt \nand Exchange Traded Commodity Derivatives and \nGold ETF & Silver ETF)', None, None, None, None, None, 'Index']
Row 162: ['LIGS27', 'INE2R5L15013', 'Liquid Gold Series**', 'CRISIL AAA(SO)', 571, 4988.75, 0.0037, 0.093123]
Row 163: ['LIGS22', 'INE1U3R15011', 'Liquid Gold Series**', 'CRISIL AAA(SO)', 119.999996, 1497.66, 0.0011, 0.094925]
Row 179: ['BMGB51ME', 'INF204KB17I5', 'Nippon India ETF Gold Bees', None, 97272043, 128010.01, 0.0953, None]
Row 180: ['149758', 'INF204KC1402', 'Nippon India Silver ETF', None, 20900450, 52838.43, 0.0393, None]
Row 198: [None, 'Long', 'FUTCOM_GOLD_02/04/2026', 'Others', 14000, 22694.56, 0.0169, None]
Row 207: [None, None, None, None, None, 'BENCHMARK NAME - 50% OF BSE 500 TRI, 20% OF MSCI WORLD INDEX TRI, 15% OF CRISIL SHORT TERM BOND INDEX 10% OF DOMESTIC PRICES OF GOLD & 5% OF DOMESTIC PRICES OF SILVER', None, None]


In [ ]:
# STEP 146: Find the March 2026 filename

for file in nippon_folder.iterdir():
    if file.is_file() and "mar" in file.name.lower() and "26" in file.name:
        print(file.name)

NIMF-MONTHLY-PORTFOLIO-31-Mar-26.xls


In [ ]:
# STEP 147: Check the March 2026 workbook format

march_2026_file = nippon_folder / "NIMF-MONTHLY-PORTFOLIO-31-Mar-26.xls"

with open(march_2026_file, "rb") as file_stream:
    file_signature = file_stream.read(8)

print("File:", march_2026_file.name)
print("File signature:", file_signature)

if file_signature.startswith(b"PK"):
    print("Format: ZIP-based Excel workbook; use openpyxl.")
else:
    print("Format: Possibly a legacy XLS workbook; check with xlrd.")

File: NIMF-MONTHLY-PORTFOLIO-31-Mar-26.xls
File signature: b'\xd0\xcf\x11\xe0\xa1\xb1\x1a\xe1'
Format: Possibly a legacy XLS workbook; check with xlrd.


In [ ]:
# STEP 148: Inspect the March 2026 MF worksheet

march_workbook = xlrd.open_workbook(
    str(march_2026_file)
)

print("Worksheets:", march_workbook.sheet_names())

march_sheet = march_workbook.sheet_by_name("MF")

for row_number in range(min(10, march_sheet.nrows)):
    values = [
        value
        for value in march_sheet.row_values(row_number)
        if value != ""
    ]
    print(f"Row {row_number + 1}: {values}")

Worksheets: ['Index', 'GF', 'GS', 'IP', 'IF', 'ST', 'BF', 'MI', 'PS', 'CP', 'PH', 'ME', 'NE', 'NI', 'EO', 'LF', 'FR', 'SD', 'SE', 'SH', 'LQ', 'TS', 'LE', 'LP', 'UB', 'K5', 'EA', 'QP', 'GL', 'NF', 'SC', 'SF', 'AF', 'GD', 'CE', 'CF', 'RD', 'CB', 'JE', 'SX', 'RW', 'RG', 'BP', 'ES', 'NX', 'UE', 'LT', 'NL', 'NB', 'JZ', 'PU', 'GB', 'LB', 'BB', 'SB', 'HS', 'IB', 'CC', 'ON', 'NM', 'RJ', 'G6', 'SN', 'NZ', 'MF', 'NS', 'PP', 'AL', 'NP', 'NV', 'NY', 'NG', 'NH', 'LC', 'HG', 'ET', 'NO', 'SI', 'SO', 'HK', 'CN', 'NN', 'W5', 'ND', 'NJ', 'NK', 'NQ', 'W8', 'NU', 'NW', 'U7', 'U8', 'IT', 'IX', 'BX', 'EW', 'MT', 'IJ', 'ID', 'AN', 'RN', 'AM', 'LV', 'QI', 'BS', 'BZ', 'AP', 'MC', 'LD', 'MG', 'MZ', 'IY', 'IH']
Row 1: ['RLMF806', 'Nippon India Multi Asset Allocation Fund (An open ended scheme investing in Equity, Debt \nand Exchange Traded Commodity Derivatives and \nGold ETF & Silver ETF)', 'Index']
Row 2: ['Monthly Portfolio Statement as on March 31,2026']
Row 3: []
Row 4: ['ISIN', 'Name of the Instrument', 'I

In [ ]:
# STEP 149: Find gold and silver references in March 2026

for row_number in range(march_sheet.nrows):
    row_values = march_sheet.row_values(row_number)

    row_text = " ".join(
        str(value) for value in row_values if value != ""
    ).lower()

    if "gold" in row_text or "silver" in row_text:
        print(f"\nRow {row_number + 1}:")
        print(row_values)
        


Row 1:
['RLMF806', 'Nippon India Multi Asset Allocation Fund (An open ended scheme investing in Equity, Debt \nand Exchange Traded Commodity Derivatives and \nGold ETF & Silver ETF)', '', '', '', '', '', 'Index']

Row 174:
['LIGS27', 'INE2R5L15013', 'Liquid Gold Series**', 'CRISIL AAA(SO)', 570.999999, 4663.06, 0.0035, 0.098122]

Row 175:
['LIGS22', 'INE1U3R15011', 'Liquid Gold Series**', 'CRISIL AAA(SO)', 119.999996, 1375.22, 0.001, 0.099225]

Row 189:
['BMGB51ME', 'INF204KB17I5', 'Nippon India ETF Gold Bees', '', 97272043.0, 117883.99, 0.0897, '']

Row 190:
['149758', 'INF204KC1402', 'Nippon India Silver ETF', '', 20959845.0, 46698.68, 0.0355, '']

Row 218:
['', '', '', '', '', 'BENCHMARK NAME - 50% OF BSE 500 TRI, 20% OF MSCI WORLD INDEX TRI, 15% OF CRISIL SHORT TERM BOND INDEX 10% OF DOMESTIC PRICES OF GOLD & 5% OF DOMESTIC PRICES OF SILVER', '', '']


In [ ]:
# STEP 151: Check the April 2026 workbook format

april_2026_file = nippon_folder / "NIMF-MONTHLY-PORTFOLIO-30-April-26.xls"

with open(april_2026_file, "rb") as file_stream:
    file_signature = file_stream.read(8)

print("File:", april_2026_file.name)
print("File signature:", file_signature)

if file_signature.startswith(b"PK"):
    print("Format: ZIP-based Excel workbook; use openpyxl.")
else:
    print("Format: Possibly a legacy XLS workbook; check with xlrd.")

File: NIMF-MONTHLY-PORTFOLIO-30-April-26.xls
File signature: b'PK\x03\x04\x14\x00\x06\x00'
Format: ZIP-based Excel workbook; use openpyxl.


In [ ]:
# STEP 152: Inspect the April 2026 MF worksheet

april_workbook = openpyxl.load_workbook(
    april_2026_file,
    data_only=True,
    read_only=True
)

print("Worksheets:", april_workbook.sheetnames)

april_sheet = april_workbook["MF"]

for row_number, row_values in enumerate(
    april_sheet.iter_rows(min_row=1, max_row=10, values_only=True),
    start=1
):
    values = [value for value in row_values if value is not None and value != ""]
    print(f"Row {row_number}: {values}")

InvalidFileException: openpyxl does not support the old .xls file format, please use xlrd to read this file, or convert it to the more recent .xlsx file format.

In [ ]:
import openpyxl

In [ ]:
# STEP 152C: Open the ZIP-based workbook without renaming it

with open(april_2026_file, "rb") as file_stream:
    april_workbook = openpyxl.load_workbook(
        file_stream,
        data_only=True,
        read_only=True
    )

print("Worksheets:", april_workbook.sheetnames)

Worksheets: ['Index', 'GF', 'GS', 'IP', 'IF', 'ST', 'BF', 'MI', 'PS', 'CP', 'PH', 'ME', 'NE', 'NI', 'EO', 'LF', 'FR', 'SD', 'SE', 'SH', 'LQ', 'TS', 'LE', 'LP', 'K5', 'EA', 'QP', 'GL', 'NF', 'SC', 'SF', 'AF', 'GD', 'CE', 'CF', 'RD', 'CB', 'JE', 'SX', 'RW', 'RG', 'BP', 'ES', 'NX', 'UE', 'LT', 'NL', 'NB', 'JZ', 'PU', 'GB', 'LB', 'BB', 'SB', 'HS', 'IB', 'CC', 'ON', 'NM', 'RJ', 'G6', 'SN', 'NZ', 'MF', 'NS', 'PP', 'AL', 'NP', 'NV', 'NY', 'NG', 'NH', 'LC', 'HG', 'ET', 'NO', 'SI', 'SO', 'HK', 'CN', 'NN', 'W5', 'ND', 'NJ', 'NK', 'NQ', 'NU', 'NW', 'U8', 'IT', 'IX', 'BX', 'EW', 'MT', 'IJ', 'ID', 'AN', 'RN', 'AM', 'LV', 'QI', 'BS', 'BZ', 'AP', 'MC', 'LD', 'MG', 'MZ', 'IY', 'IH']


In [ ]:
# STEP 152D: Inspect the April 2026 MF worksheet

april_sheet = april_workbook["MF"]

for row_number, row_values in enumerate(
    april_sheet.iter_rows(min_row=1, max_row=10, values_only=True),
    start=1
):
    values = [
        value for value in row_values
        if value is not None and value != ""
    ]
    print(f"Row {row_number}: {values}")

ValueError: seek of closed file

In [ ]:
# STEP 152E: Read the April 2026 workbook while the stream is open

with open(april_2026_file, "rb") as file_stream:
    april_workbook = openpyxl.load_workbook(
        file_stream,
        data_only=True,
        read_only=True
    )

    print("Worksheets:", april_workbook.sheetnames)

    april_sheet = april_workbook["MF"]

    for row_number, row_values in enumerate(
        april_sheet.iter_rows(
            min_row=1,
            max_row=10,
            values_only=True
        ),
        start=1
    ):
        values = [
            value for value in row_values
            if value is not None and value != ""
        ]
        print(f"Row {row_number}: {values}")

    april_workbook.close()

Worksheets: ['Index', 'GF', 'GS', 'IP', 'IF', 'ST', 'BF', 'MI', 'PS', 'CP', 'PH', 'ME', 'NE', 'NI', 'EO', 'LF', 'FR', 'SD', 'SE', 'SH', 'LQ', 'TS', 'LE', 'LP', 'K5', 'EA', 'QP', 'GL', 'NF', 'SC', 'SF', 'AF', 'GD', 'CE', 'CF', 'RD', 'CB', 'JE', 'SX', 'RW', 'RG', 'BP', 'ES', 'NX', 'UE', 'LT', 'NL', 'NB', 'JZ', 'PU', 'GB', 'LB', 'BB', 'SB', 'HS', 'IB', 'CC', 'ON', 'NM', 'RJ', 'G6', 'SN', 'NZ', 'MF', 'NS', 'PP', 'AL', 'NP', 'NV', 'NY', 'NG', 'NH', 'LC', 'HG', 'ET', 'NO', 'SI', 'SO', 'HK', 'CN', 'NN', 'W5', 'ND', 'NJ', 'NK', 'NQ', 'NU', 'NW', 'U8', 'IT', 'IX', 'BX', 'EW', 'MT', 'IJ', 'ID', 'AN', 'RN', 'AM', 'LV', 'QI', 'BS', 'BZ', 'AP', 'MC', 'LD', 'MG', 'MZ', 'IY', 'IH']
Row 1: ['RLMF806', 'Nippon India Multi Asset Allocation Fund (An open ended scheme investing in Equity, Debt \nand Exchange Traded Commodity Derivatives and \nGold ETF & Silver ETF)', 'Index']
Row 2: ['Monthly Portfolio Statement as on April 30,2026']
Row 3: []
Row 4: ['ISIN', 'Name of the Instrument', 'Industry / Rating',

In [ ]:
# STEP 153: Find gold and silver references in April 2026

with open(april_2026_file, "rb") as file_stream:
    april_workbook = openpyxl.load_workbook(
        file_stream,
        data_only=True,
        read_only=True
    )

    april_sheet = april_workbook["MF"]

    for row_number, row_values in enumerate(
        april_sheet.iter_rows(values_only=True),
        start=1
    ):
        row_text = " ".join(
            str(value)
            for value in row_values
            if value is not None and value != ""
        ).lower()

        if "gold" in row_text or "silver" in row_text:
            print(f"\nRow {row_number}:")
            print(list(row_values))

    april_workbook.close()


Row 1:
['RLMF806', 'Nippon India Multi Asset Allocation Fund (An open ended scheme investing in Equity, Debt \nand Exchange Traded Commodity Derivatives and \nGold ETF & Silver ETF)', None, None, None, None, None, 'Index']

Row 183:
['LIGS27', 'INE2R5L15013', 'Liquid Gold Series**', 'CRISIL AAA(SO)', 570.999998, 4449.85, 0.003, 0.0975]

Row 184:
['LIGS22', 'INE1U3R15011', 'Liquid Gold Series**', 'CRISIL AAA(SO)', 119.999995, 1271.42, 0.0009, 0.09615]

Row 201:
['BMGB51ME', 'INF204KB17I5', 'Nippon India ETF Gold Bees', None, 97272043, 120062.88, 0.0815, None]

Row 202:
['149758', 'INF204KC1402', 'Nippon India Silver ETF', None, 20959845, 47683.65, 0.0324, None]

Row 219:
[None, 'Long', 'FUTCOM_GOLD_05/06/2026', 'Others', 24000, 36266.64, 0.0246, None]

Row 256:
[None, 'GOLD', 'Long', 152471.3875, 151111, 3496.098, datetime.datetime(2026, 6, 5, 0, 0), None]

Row 281:
[None, None, None, None, None, 'BENCHMARK NAME - 50% OF BSE 500 TRI, 20% OF MSCI WORLD INDEX TRI, 15% OF CRISIL SHORT TER

In [ ]:
# STEP 155: Check the May 2026 workbook format

may_2026_file = nippon_folder / "NIMF-MONTHLY-PORTFOLIO-31-May-26.xls"

with open(may_2026_file, "rb") as file_stream:
    file_signature = file_stream.read(8)

print("File:", may_2026_file.name)
print("File signature:", file_signature)

if file_signature.startswith(b"PK"):
    print("Format: ZIP-based Excel workbook; use openpyxl.")
else:
    print("Format: Possibly a legacy XLS workbook; check with xlrd.")

File: NIMF-MONTHLY-PORTFOLIO-31-May-26.xls
File signature: b'PK\x03\x04\x14\x00\x06\x00'
Format: ZIP-based Excel workbook; use openpyxl.


In [ ]:
# STEP 156: Inspect the May 2026 MF worksheet

with open(may_2026_file, "rb") as file_stream:
    may_workbook = openpyxl.load_workbook(
        file_stream,
        data_only=True,
        read_only=True
    )

    print("Worksheets:", may_workbook.sheetnames)

    may_sheet = may_workbook["MF"]

    for row_number, row_values in enumerate(
        may_sheet.iter_rows(
            min_row=1,
            max_row=10,
            values_only=True
        ),
        start=1
    ):
        values = [
            value for value in row_values
            if value is not None and value != ""
        ]
        print(f"Row {row_number}: {values}")

    may_workbook.close()

Worksheets: ['Index', 'GF', 'GS', 'IP', 'IF', 'ST', 'BF', 'MI', 'PS', 'CP', 'PH', 'ME', 'NE', 'NI', 'EO', 'LF', 'FR', 'SD', 'SE', 'SH', 'LQ', 'TS', 'LE', 'LP', 'K5', 'EA', 'QP', 'GL', 'NF', 'SC', 'SF', 'AF', 'GD', 'CE', 'CF', 'RD', 'CB', 'JE', 'SX', 'RW', 'RG', 'BP', 'ES', 'NX', 'UE', 'LT', 'NL', 'NB', 'JZ', 'PU', 'GB', 'LB', 'BB', 'SB', 'HS', 'IB', 'CC', 'ON', 'NM', 'RJ', 'G6', 'SN', 'NZ', 'MF', 'NS', 'PP', 'AL', 'NP', 'NV', 'NG', 'NH', 'LC', 'HG', 'ET', 'NO', 'SI', 'SO', 'HK', 'CN', 'NN', 'W5', 'ND', 'NJ', 'NK', 'NQ', 'NU', 'NW', 'U8', 'IT', 'IX', 'BX', 'EW', 'MT', 'IJ', 'ID', 'AN', 'RN', 'AM', 'LV', 'QI', 'BS', 'BZ', 'AP', 'MC', 'LD', 'MG', 'MZ', 'IY', 'IH']
Row 1: ['RLMF806', 'Nippon India Multi Asset Allocation Fund (An open ended scheme investing in Equity, Debt \nand Exchange Traded Commodity Derivatives and \nGold ETF & Silver ETF)', 'Index']
Row 2: ['Monthly Portfolio Statement as on May 31,2026']
Row 3: []
Row 4: ['ISIN', 'Name of the Instrument', 'Industry / Rating', 'Quanti

In [ ]:
# STEP 157: Find gold and silver references in May 2026

with open(may_2026_file, "rb") as file_stream:
    may_workbook = openpyxl.load_workbook(
        file_stream,
        data_only=True,
        read_only=True
    )

    may_sheet = may_workbook["MF"]

    for row_number, row_values in enumerate(
        may_sheet.iter_rows(values_only=True),
        start=1
    ):
        row_values = list(row_values)

        row_text = " ".join(
            str(value)
            for value in row_values
            if value is not None and value != ""
        ).lower()

        if "gold" in row_text or "silver" in row_text:
            print(f"\nRow {row_number}:")
            print(row_values)

    may_workbook.close()


Row 1:
['RLMF806', 'Nippon India Multi Asset Allocation Fund (An open ended scheme investing in Equity, Debt \nand Exchange Traded Commodity Derivatives and \nGold ETF & Silver ETF)', None, None, None, None, None, 'Index']

Row 181:
['LIGS27', 'INE2R5L15013', 'Liquid Gold Series**', 'CRISIL AAA(SO)', 570.999997, 4046.46, 0.0026, 0.101107]

Row 182:
['LIGS22', 'INE1U3R15011', 'Liquid Gold Series**', 'CRISIL AAA(SO)', 119.999995, 1167.2, 0.0008, 0.1031]

Row 205:
['BMGB51ME', 'INF204KB17I5', 'Nippon India ETF Gold Bees', None, 103772043, 133502.73, 0.0862, None]

Row 206:
['149758', 'INF204KC1402', 'Nippon India Silver ETF', None, 22759845, 56756.23, 0.0367, None]

Row 224:
[None, 'Long', 'FUTCOM_GOLD_05/08/2026', 'Others', 16400, 26389.4, 0.017, None]

Row 261:
[None, 'GOLD', 'Long', 160667.0243804878, 160911, 2441.03955, datetime.datetime(2026, 8, 5, 0, 0), None]

Row 288:
[None, None, None, None, None, 'BENCHMARK NAME - 50% OF BSE 500 TRI, 20% OF MSCI WORLD INDEX TRI, 15% OF CRISIL S

In [ ]:
# STEP 159: Check the June 2026 workbook format

june_2026_file = nippon_folder / "NIMF-MONTHLY-PORTFOLIO-30-Jun-26.xls"

with open(june_2026_file, "rb") as file_stream:
    file_signature = file_stream.read(8)

print("File:", june_2026_file.name)
print("File signature:", file_signature)

if file_signature.startswith(b"PK"):
    print("Format: ZIP-based Excel workbook; use openpyxl.")
else:
    print("Format: Possibly a legacy XLS workbook; check with xlrd.")

File: NIMF-MONTHLY-PORTFOLIO-30-Jun-26.xls
File signature: b'PK\x03\x04\x14\x00\x06\x00'
Format: ZIP-based Excel workbook; use openpyxl.


In [ ]:
# STEP 160: Inspect the June 2026 MF worksheet

with open(june_2026_file, "rb") as file_stream:
    june_workbook = openpyxl.load_workbook(
        file_stream,
        data_only=True,
        read_only=True
    )

    print("Worksheets:", june_workbook.sheetnames)

    june_sheet = june_workbook["MF"]

    for row_number, row_values in enumerate(
        june_sheet.iter_rows(
            min_row=1,
            max_row=10,
            values_only=True
        ),
        start=1
    ):
        values = [
            value for value in row_values
            if value is not None and value != ""
        ]
        print(f"Row {row_number}: {values}")

    june_workbook.close()

Worksheets: ['Index', 'GF', 'GS', 'IP', 'IF', 'ST', 'BF', 'MI', 'PS', 'CP', 'PH', 'ME', 'NE', 'NI', 'EO', 'LF', 'FR', 'SD', 'SE', 'SH', 'LQ', 'TS', 'LE', 'LP', 'K5', 'EA', 'QP', 'GL', 'NF', 'SC', 'SF', 'AF', 'GD', 'CE', 'CF', 'RD', 'CB', 'JE', 'SX', 'RW', 'RG', 'BP', 'ES', 'NX', 'UE', 'LT', 'NL', 'NB', 'JZ', 'PU', 'GB', 'LB', 'BB', 'SB', 'HS', 'IB', 'CC', 'ON', 'NM', 'RJ', 'G6', 'SN', 'NZ', 'MF', 'NS', 'PP', 'AL', 'NP', 'NV', 'NG', 'NH', 'LC', 'ET', 'NO', 'SI', 'SO', 'HK', 'CN', 'NN', 'W5', 'ND', 'NJ', 'NK', 'NQ', 'NU', 'NW', 'U8', 'IT', 'IX', 'BX', 'EW', 'MT', 'IJ', 'ID', 'AN', 'RN', 'AM', 'LV', 'QI', 'BS', 'BZ', 'AP', 'MC', 'LD', 'MG', 'MZ', 'IY', 'IH']
Row 1: ['RLMF806', 'Nippon India Multi Asset Allocation Fund (An open ended scheme investing in Equity, Debt \nand Exchange Traded Commodity Derivatives and \nGold ETF & Silver ETF)', 'Index']
Row 2: ['Monthly Portfolio Statement as on June 30,2026']
Row 3: []
Row 4: ['ISIN', 'Name of the Instrument', 'Industry / Rating', 'Quantity', 

In [ ]:
# STEP 161: Find gold and silver references in June 2026

with open(june_2026_file, "rb") as file_stream:
    june_workbook = openpyxl.load_workbook(
        file_stream,
        data_only=True,
        read_only=True
    )

    june_sheet = june_workbook["MF"]

    for row_number, row_values in enumerate(
        june_sheet.iter_rows(values_only=True),
        start=1
    ):
        row_values = list(row_values)

        row_text = " ".join(
            str(value)
            for value in row_values
            if value is not None and value != ""
        ).lower()

        if "gold" in row_text or "silver" in row_text:
            print(f"\nRow {row_number}:")
            print(row_values)

    june_workbook.close()


Row 1:
['RLMF806', 'Nippon India Multi Asset Allocation Fund (An open ended scheme investing in Equity, Debt \nand Exchange Traded Commodity Derivatives and \nGold ETF & Silver ETF)', None, None, None, None, None, 'Index']

Row 183:
['LIGS27', 'INE2R5L15013', 'Liquid Gold Series**', 'CRISIL AAA(SO)', 570.999996, 3928.78, 0.0025, 0.09605]

Row 184:
['LIGS22', 'INE1U3R15011', 'Liquid Gold Series**', 'CRISIL AAA(SO)', 119.999994, 874.6, 0.0005, 0.09455]

Row 203:
['BMGB51ME', 'INF204KB17I5', 'Nippon India ETF Gold Bees', None, 107772043, 124983.24, 0.0781, None]

Row 204:
['149758', 'INF204KC1402', 'Nippon India Silver ETF', None, 22759845, 48437.5, 0.0303, None]

Row 222:
[None, 'Long', 'FUTCOM_GOLD_05/08/2026', 'Others', 16400, 23375.08, 0.0146, None]

Row 261:
[None, 'GOLD', 'Long', 160667.0243804878, 142531, 2162.11655, datetime.datetime(2026, 8, 5, 0, 0), None]

Row 298:
[None, None, None, None, None, 'BENCHMARK NAME - 50% OF BSE 500 TRI, 20% OF MSCI WORLD INDEX TRI, 15% OF CRISIL S

In [ ]:
# STEP 163: Check the July 2026 workbook format

july_2026_file = nippon_folder / "NIMF-MONTHLY-PORTFOLIO-31-July-26.xls"

with open(july_2026_file, "rb") as file_stream:
    file_signature = file_stream.read(8)

print("File:", july_2026_file.name)
print("File signature:", file_signature)

if file_signature.startswith(b"PK"):
    print("Format: ZIP-based Excel workbook; use openpyxl.")
else:
    print("Format: Possibly a legacy XLS workbook; check with xlrd.")

File: NIMF-MONTHLY-PORTFOLIO-31-July-26.xls
File signature: b'PK\x03\x04\x14\x00\x06\x00'
Format: ZIP-based Excel workbook; use openpyxl.


In [ ]:
# STEP 164: Inspect the July 2026 MF worksheet

with open(july_2026_file, "rb") as file_stream:
    july_workbook = openpyxl.load_workbook(
        file_stream,
        data_only=True,
        read_only=True
    )

    print("Worksheets:", july_workbook.sheetnames)

    july_sheet = july_workbook["MF"]

    for row_number, row_values in enumerate(
        july_sheet.iter_rows(
            min_row=1,
            max_row=10,
            values_only=True
        ),
        start=1
    ):
        values = [
            value for value in row_values
            if value is not None and value != ""
        ]
        print(f"Row {row_number}: {values}")

    july_workbook.close()

Worksheets: ['Index', 'GF', 'GS', 'IP', 'IF', 'ST', 'BF', 'MI', 'PS', 'CP', 'PH', 'ME', 'NE', 'NI', 'EO', 'LF', 'FR', 'SD', 'SE', 'SH', 'LQ', 'TS', 'LE', 'LP', 'K5', 'EA', 'QP', 'GL', 'NF', 'SC', 'SF', 'AF', 'GD', 'CE', 'CF', 'RD', 'CB', 'JE', 'SX', 'RW', 'RG', 'BP', 'ES', 'NX', 'UE', 'LT', 'NL', 'NB', 'JZ', 'PU', 'GB', 'LB', 'BB', 'SB', 'HS', 'IB', 'CC', 'ON', 'NM', 'RJ', 'G6', 'SN', 'NZ', 'MF', 'NS', 'PP', 'AL', 'NP', 'NV', 'NG', 'NH', 'LC', 'ET', 'NO', 'SI', 'SO', 'HK', 'CN', 'NN', 'W5', 'ND', 'NJ', 'NK', 'NQ', 'NU', 'NW', 'U8', 'IT', 'IX', 'BX', 'EW', 'MT', 'IJ', 'ID', 'AN', 'RN', 'AM', 'LV', 'QI', 'BS', 'BZ', 'AP', 'MC', 'LD', 'MG', 'MZ', 'IY', 'IH']
Row 1: ['RLMF806', 'Nippon India Multi Asset Allocation Fund (An open ended scheme investing in Equity, Debt \nand Exchange Traded Commodity Derivatives and \nGold ETF & Silver ETF)', 'Index']
Row 2: ['Monthly Portfolio Statement as on July 31,2026']
Row 3: []
Row 4: ['ISIN', 'Name of the Instrument', 'Industry / Rating', 'Quantity', 

In [ ]:
# STEP 165: Find gold and silver references in July 2026

with open(july_2026_file, "rb") as file_stream:
    july_workbook = openpyxl.load_workbook(
        file_stream,
        data_only=True,
        read_only=True
    )

    july_sheet = july_workbook["MF"]

    for row_number, row_values in enumerate(
        july_sheet.iter_rows(values_only=True),
        start=1
    ):
        row_values = list(row_values)

        row_text = " ".join(
            str(value)
            for value in row_values
            if value is not None and value != ""
        ).lower()

        if "gold" in row_text or "silver" in row_text:
            print(f"\nRow {row_number}:")
            print(row_values)

    july_workbook.close()


Row 1:
['RLMF806', 'Nippon India Multi Asset Allocation Fund (An open ended scheme investing in Equity, Debt \nand Exchange Traded Commodity Derivatives and \nGold ETF & Silver ETF)', None, None, None, None, None, 'Index']

Row 188:
['LIGS27', 'INE2R5L15013', 'Liquid Gold Series**', 'CRISIL AAA(SO)', 570.999996, 3636.56, 0.0021, 0.09745]

Row 189:
['LIGS22', 'INE1U3R15011', 'Liquid Gold Series**', 'CRISIL AAA(SO)', 119.999993, 284.52, 0.0002, 0.09565]

Row 208:
['BMGB51ME', 'INF204KB17I5', 'Nippon India ETF Gold Bees', None, 117772043, 137922.84, 0.0815, None]

Row 209:
['149758', 'INF204KC1402', 'Nippon India Silver ETF', None, 23259845, 48033.91, 0.0284, None]

Row 227:
[None, 'Long', 'FUTCOM_GOLD_05/10/2026', 'Others', 23000, 32976.48, 0.0195, None]

Row 229:
[None, 'Long', 'FUTCOM_SILVER_04/09/2026', 'Others', 1350, 2932.17, 0.0017, None]

Row 265:
[None, 'GOLD', 'Long', 144120.54780869564, 143376, 3050.331, datetime.datetime(2026, 10, 5, 0, 0), None]

Row 266:
[None, 'SILVER', 'L

In [ ]:
# STEP 167: Check the August 2026 workbook format

august_2026_file = nippon_folder / "NIMF-MONTHLY-PORTFOLIO-31-Aug-26.xls"

with open(august_2026_file, "rb") as file_stream:
    file_signature = file_stream.read(8)

print("File:", august_2026_file.name)
print("File signature:", file_signature)

if file_signature.startswith(b"PK"):
    print("Format: ZIP-based Excel workbook; use openpyxl.")
else:
    print("Format: Possibly a legacy XLS workbook; check with xlrd.")

File: NIMF-MONTHLY-PORTFOLIO-31-Aug-26.xls
File signature: b'PK\x03\x04\x14\x00\x06\x00'
Format: ZIP-based Excel workbook; use openpyxl.


In [ ]:
# STEP 168: Inspect the August 2026 MF worksheet

with open(august_2026_file, "rb") as file_stream:
    august_workbook = openpyxl.load_workbook(
        file_stream,
        data_only=True,
        read_only=True
    )

    print("Worksheets:", august_workbook.sheetnames)

    august_sheet = august_workbook["MF"]

    for row_number, row_values in enumerate(
        august_sheet.iter_rows(
            min_row=1,
            max_row=10,
            values_only=True
        ),
        start=1
    ):
        values = [
            value for value in row_values
            if value is not None and value != ""
        ]
        print(f"Row {row_number}: {values}")

    august_workbook.close()
    

Worksheets: ['Index', 'GF', 'GS', 'IP', 'IF', 'ST', 'BF', 'MI', 'PS', 'CP', 'PH', 'ME', 'NE', 'NI', 'EO', 'LF', 'FR', 'SD', 'SE', 'SH', 'LQ', 'TS', 'LE', 'LP', 'K5', 'EA', 'QP', 'GL', 'NF', 'SC', 'SF', 'AF', 'GD', 'CE', 'CF', 'RD', 'CB', 'JE', 'SX', 'RW', 'RG', 'BP', 'ES', 'NX', 'UE', 'LT', 'NL', 'NB', 'JZ', 'PU', 'GB', 'LB', 'BB', 'SB', 'HS', 'IB', 'CC', 'ON', 'NM', 'RJ', 'G6', 'SN', 'NZ', 'MF', 'NS', 'PP', 'AL', 'NP', 'NV', 'NG', 'NH', 'LC', 'ET', 'NO', 'SI', 'SO', 'HK', 'CN', 'NN', 'W5', 'ND', 'NJ', 'NK', 'NQ', 'NU', 'NW', 'U8', 'IT', 'IX', 'BX', 'EW', 'MT', 'IJ', 'ID', 'AN', 'RN', 'AM', 'LV', 'QI', 'BS', 'BZ', 'AP', 'MC', 'LD', 'MG', 'MZ', 'IY', 'IH']
Row 1: ['RLMF806', 'Nippon India Multi Asset Allocation Fund (An open ended scheme investing in Equity, Debt \nand Exchange Traded Commodity Derivatives and \nGold ETF & Silver ETF)', 'Index']
Row 2: ['Monthly Portfolio Statement as on August 31,2026']
Row 3: []
Row 4: ['ISIN', 'Name of the Instrument', 'Industry / Rating', 'Quantity'

In [ ]:
# STEP 169: Find gold and silver references in August 2026

with open(august_2026_file, "rb") as file_stream:
    august_workbook = openpyxl.load_workbook(
        file_stream,
        data_only=True,
        read_only=True
    )

    august_sheet = august_workbook["MF"]

    for row_number, row_values in enumerate(
        august_sheet.iter_rows(values_only=True),
        start=1
    ):
        row_values = list(row_values)

        row_text = " ".join(
            str(value)
            for value in row_values
            if value is not None and value != ""
        ).lower()

        if "gold" in row_text or "silver" in row_text:
            print(f"\nRow {row_number}:")
            print(row_values)

    august_workbook.close()


Row 1:
['RLMF806', 'Nippon India Multi Asset Allocation Fund (An open ended scheme investing in Equity, Debt \nand Exchange Traded Commodity Derivatives and \nGold ETF & Silver ETF)', None, None, None, None, None, 'Index']

Row 188:
['LIGS27', 'INE2R5L15013', 'Liquid Gold Series**', 'CRISIL AAA(SO)', 570.999995, 3435.73, 0.0019, 0.095471]

Row 203:
['BMGB51ME', 'INF204KB17I5', 'Nippon India ETF Gold Bees', None, 117772043, 149947.37, 0.0834, None]

Row 204:
['149758', 'INF204KC1402', 'Nippon India Silver ETF', None, 23259845, 51918.3, 0.0289, None]

Row 222:
[None, 'Long', 'FUTCOM_GOLD_05/10/2026', 'Others', 41000, 63328.6, 0.0352, None]

Row 223:
[None, 'Long', 'FUTCOM_SILVER_04/12/2026', 'Others', 9600, 23051.62, 0.0128, None]

Row 260:
[None, 'GOLD', 'Long', 148461.86828292682, 154460, 5857.9775, datetime.datetime(2026, 10, 5, 0, 0), None]

Row 261:
[None, 'SILVER', 'Long', 247474.1968875, 240121, 2939.0892, datetime.datetime(2026, 12, 4, 0, 0), None]

Row 295:
[None, None, None, N

In [ ]:
# STEP 171: Check the September 2026 workbook format

september_2026_file = nippon_folder / "NIMF-MONTHLY-PORTFOLIO-30-Sep-26.xls"

with open(september_2026_file, "rb") as file_stream:
    file_signature = file_stream.read(8)

print("File:", september_2026_file.name)
print("File signature:", file_signature)

if file_signature.startswith(b"PK"):
    print("Format: ZIP-based Excel workbook; use openpyxl.")
else:
    print("Format: Possibly a legacy XLS workbook; check with xlrd.")

File: NIMF-MONTHLY-PORTFOLIO-30-Sep-26.xls
File signature: b'PK\x03\x04\x14\x00\x06\x00'
Format: ZIP-based Excel workbook; use openpyxl.


In [ ]:
# STEP 172: Open and inspect the September 2026 workbook

from openpyxl import load_workbook

with open(september_2026_file, "rb") as file_stream:
    nippon_sep_2026_wb = load_workbook(
        file_stream,
        data_only=True,
        read_only=True
    )

    print("Worksheets:", nippon_sep_2026_wb.sheetnames)

    nippon_sep_2026_ws = nippon_sep_2026_wb["MF"]

    for row in nippon_sep_2026_ws.iter_rows(
        min_row=1,
        max_row=10,
        values_only=True
    ):
        print(row)

    nippon_sep_2026_wb.close()

Worksheets: ['Index', 'GF', 'GS', 'IP', 'IF', 'ST', 'BF', 'MI', 'PS', 'CP', 'PH', 'ME', 'NE', 'NI', 'EO', 'LF', 'FR', 'SD', 'SE', 'SH', 'LQ', 'TS', 'LE', 'LP', 'K5', 'EA', 'QP', 'GL', 'NF', 'SC', 'SF', 'AF', 'GD', 'CE', 'CF', 'RD', 'CB', 'JE', 'SX', 'RW', 'RG', 'BP', 'ES', 'NX', 'UE', 'LT', 'NL', 'NB', 'JZ', 'PU', 'GB', 'LB', 'BB', 'SB', 'HS', 'IB', 'CC', 'ON', 'NM', 'RJ', 'G6', 'SN', 'NZ', 'MF', 'NS', 'PP', 'AL', 'NP', 'NV', 'NG', 'NH', 'LC', 'ET', 'NO', 'SI', 'SO', 'HK', 'CN', 'NN', 'W5', 'NJ', 'NK', 'NQ', 'NU', 'NW', 'U8', 'IT', 'IX', 'BX', 'EW', 'MT', 'IJ', 'ID', 'AN', 'RN', 'AM', 'LV', 'QI', 'BS', 'BZ', 'AP', 'MC', 'LD', 'MG', 'MZ', 'IY', 'IH', 'AO']
('RLMF806', 'Nippon India Multi Asset Allocation Fund (An open ended scheme investing in Equity, Debt \nand Exchange Traded Commodity Derivatives and \nGold ETF & Silver ETF)', None, None, None, None, None, 'Index')
(None, 'Monthly Portfolio Statement as on September 30,2026', None, None, None, None, None, None)
(None, None, None, Non

In [ ]:
# STEP 173: Find gold and silver entries in September 2026

with open(september_2026_file, "rb") as file_stream:
    nippon_sep_2026_wb = load_workbook(
        file_stream,
        data_only=True,
        read_only=True
    )

    nippon_sep_2026_ws = nippon_sep_2026_wb["MF"]

    for row_number, row in enumerate(
        nippon_sep_2026_ws.iter_rows(values_only=True),
        start=1
    ):
        row_text = " ".join(
            str(value) for value in row if value is not None
        ).lower()

        if "gold" in row_text or "silver" in row_text:
            print(f"Row {row_number}: {list(row)}")

    nippon_sep_2026_wb.close()

Row 1: ['RLMF806', 'Nippon India Multi Asset Allocation Fund (An open ended scheme investing in Equity, Debt \nand Exchange Traded Commodity Derivatives and \nGold ETF & Silver ETF)', None, None, None, None, None, 'Index']
Row 185: ['LIGS27', 'INE2R5L15013', 'Liquid Gold Series**', 'CRISIL AAA(SO)', 570.999994, 3361.86, 0.0019, 0.0974]
Row 206: ['BMGB51ME', 'INF204KB17I5', 'Nippon India ETF Gold Bees', None, 117772043, 143199.03, 0.0794, None]
Row 207: ['149758', 'INF204KC1402', 'Nippon India Silver ETF', None, 23259845, 48757.29, 0.027, None]
Row 225: [None, 'Long', 'FUTCOM_GOLD_04/12/2026', 'Others', 41000, 61104.76, 0.0339, None]
Row 226: [None, 'Long', 'FUTCOM_SILVER_04/12/2026', 'Others', 9600, 21475.78, 0.0119, None]
Row 265: [None, 'GOLD', 'Long', 153155.9902292683, 149036, 5652.2395, datetime.datetime(2026, 12, 4, 0, 0), None]
Row 266: [None, 'SILVER', 'Long', 247474.1968875, 223706, 2738.1432, datetime.datetime(2026, 12, 4, 0, 0), None]
Row 312: [None, None, None, None, None, 

In [ ]:
# STEP 1: Create the Nippon India monthly holdings log

from pathlib import Path
import pandas as pd

# Identify the main project folder
PROJECT_DIR = Path.cwd().parent

# Create the output folder if it does not exist
OUTPUT_DIR = PROJECT_DIR / "02_CLEAN_DATA"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Generate all 33 reporting months
months = pd.date_range(
    start="2024-01-01",
    end="2026-09-01",
    freq="MS"
)

# Create the holdings-log structure
nippon_log = pd.DataFrame({
    "Report_Month": months.strftime("%Y-%m"),
    "Report_Date": "",
    "Gold_ETF_Name": "",
    "Gold_ETF_Value_Lakh": pd.NA,
    "Gold_ETF_NAV_Pct": pd.NA,
    "Silver_ETF_Name": "",
    "Silver_ETF_Value_Lakh": pd.NA,
    "Silver_ETF_NAV_Pct": pd.NA,
    "Gold_Futures_Value_Lakh": pd.NA,
    "Gold_Futures_NAV_Pct": pd.NA,
    "Silver_Futures_Value_Lakh": pd.NA,
    "Silver_Futures_NAV_Pct": pd.NA,
    "Options_and_Other_Instruments": "",
    "Review_Notes": ""
})

# Save the log
output_file = OUTPUT_DIR / "Nippon_India_Monthly_Holdings_Log.csv"
nippon_log.to_csv(output_file, index=False)

print("Rows created:", len(nippon_log))
print("Output file:", output_file)
print(nippon_log.head())

Rows created: 33
Output file: c:\Users\rocks\OneDrive\Desktop\AMC Project Data\02_CLEAN_DATA\Nippon_India_Monthly_Holdings_Log.csv
  Report_Month Report_Date Gold_ETF_Name Gold_ETF_Value_Lakh Gold_ETF_NAV_Pct  \
0      2024-01                                          <NA>             <NA>   
1      2024-02                                          <NA>             <NA>   
2      2024-03                                          <NA>             <NA>   
3      2024-04                                          <NA>             <NA>   
4      2024-05                                          <NA>             <NA>   

  Silver_ETF_Name Silver_ETF_Value_Lakh Silver_ETF_NAV_Pct  \
0                                  <NA>               <NA>   
1                                  <NA>               <NA>   
2                                  <NA>               <NA>   
3                                  <NA>               <NA>   
4                                  <NA>               <NA>   

  Gold_Fu

In [ ]:
print(nippon_log.loc[
    nippon_log["Report_Month"].isin(["2026-08", "2026-09"]),
    ["Report_Month", "Gold_ETF_Value_Lakh", "Silver_ETF_Value_Lakh"]
].to_string(index=False))

Report_Month Gold_ETF_Value_Lakh Silver_ETF_Value_Lakh
     2026-08                <NA>                  <NA>
     2026-09                <NA>                  <NA>


In [ ]:
updates = {
    "2026-08": ["2026-08-31", 149947.37, 8.34, 51918.30, 2.89],
    "2026-09": ["2026-09-30", 143199.03, 7.94, 48757.29, 2.70]
}

for month, v in updates.items():
    m = nippon_log["Report_Month"] == month
    nippon_log.loc[m, "Report_Date"] = v[0]
    nippon_log.loc[m, "Gold_ETF_Name"] = "Nippon India ETF Gold BeES"
    nippon_log.loc[m, "Gold_ETF_Value_Lakh"] = v[1]
    nippon_log.loc[m, "Gold_ETF_NAV_Pct"] = v[2]
    nippon_log.loc[m, "Silver_ETF_Name"] = "Nippon India Silver ETF"
    nippon_log.loc[m, "Silver_ETF_Value_Lakh"] = v[3]
    nippon_log.loc[m, "Silver_ETF_NAV_Pct"] = v[4]

nippon_log.to_csv(output_file, index=False)

print(nippon_log.loc[
    nippon_log["Report_Month"].isin(updates),
    ["Report_Month", "Gold_ETF_Value_Lakh",
     "Gold_ETF_NAV_Pct", "Silver_ETF_Value_Lakh",
     "Silver_ETF_NAV_Pct"]
].to_string(index=False))

Report_Month Gold_ETF_Value_Lakh Gold_ETF_NAV_Pct Silver_ETF_Value_Lakh Silver_ETF_NAV_Pct
     2026-08           149947.37             8.34               51918.3               2.89
     2026-09           143199.03             7.94              48757.29                2.7


In [ ]:
v = {
    "Report_Date": "2026-07-31",
    "Gold_ETF_Name": "Nippon India ETF Gold BeES",
    "Gold_ETF_Value_Lakh": 137922.84,
    "Gold_ETF_NAV_Pct": 8.15,
    "Silver_ETF_Name": "Nippon India Silver ETF",
    "Silver_ETF_Value_Lakh": 48033.91,
    "Silver_ETF_NAV_Pct": 2.84
}

m = nippon_log["Report_Month"] == "2026-07"

for col, value in v.items():
    nippon_log.loc[m, col] = value

nippon_log.to_csv(output_file, index=False)

print(nippon_log.loc[m].to_string(index=False))

Report_Month Report_Date              Gold_ETF_Name Gold_ETF_Value_Lakh Gold_ETF_NAV_Pct         Silver_ETF_Name Silver_ETF_Value_Lakh Silver_ETF_NAV_Pct Gold_Futures_Value_Lakh Gold_Futures_NAV_Pct Silver_Futures_Value_Lakh Silver_Futures_NAV_Pct Options_and_Other_Instruments Review_Notes
     2026-07  2026-07-31 Nippon India ETF Gold BeES           137922.84             8.15 Nippon India Silver ETF              48033.91               2.84                    <NA>                 <NA>                      <NA>                   <NA>                                           


In [ ]:
v = {
    "Report_Date": "2026-06-30",
    "Gold_ETF_Name": "Nippon India ETF Gold BeES",
    "Gold_ETF_Value_Lakh": 124983.24,
    "Gold_ETF_NAV_Pct": 7.81,
    "Silver_ETF_Name": "Nippon India Silver ETF",
    "Silver_ETF_Value_Lakh": 48437.50,
    "Silver_ETF_NAV_Pct": 3.03
}

m = nippon_log["Report_Month"] == "2026-06"

for col, value in v.items():
    nippon_log.loc[m, col] = value

nippon_log.to_csv(output_file, index=False)

print(nippon_log.loc[
    m,
    ["Report_Month", "Report_Date", "Gold_ETF_Value_Lakh",
     "Gold_ETF_NAV_Pct", "Silver_ETF_Value_Lakh",
     "Silver_ETF_NAV_Pct"]
].to_string(index=False))

Report_Month Report_Date Gold_ETF_Value_Lakh Gold_ETF_NAV_Pct Silver_ETF_Value_Lakh Silver_ETF_NAV_Pct
     2026-06  2026-06-30           124983.24             7.81               48437.5               3.03


In [ ]:
v = {
    "Report_Date": "2026-05-31",
    "Gold_ETF_Name": "Nippon India ETF Gold BeES",
    "Gold_ETF_Value_Lakh": 133502.73,
    "Gold_ETF_NAV_Pct": 8.62,
    "Silver_ETF_Name": "Nippon India Silver ETF",
    "Silver_ETF_Value_Lakh": 56756.23,
    "Silver_ETF_NAV_Pct": 3.67
}

m = nippon_log["Report_Month"] == "2026-05"

for col, value in v.items():
    nippon_log.loc[m, col] = value

nippon_log.to_csv(output_file, index=False)

print(nippon_log.loc[
    m,
    ["Report_Month", "Report_Date", "Gold_ETF_Value_Lakh",
     "Gold_ETF_NAV_Pct", "Silver_ETF_Value_Lakh",
     "Silver_ETF_NAV_Pct"]
].to_string(index=False))

Report_Month Report_Date Gold_ETF_Value_Lakh Gold_ETF_NAV_Pct Silver_ETF_Value_Lakh Silver_ETF_NAV_Pct
     2026-05  2026-05-31           133502.73             8.62              56756.23               3.67


In [ ]:
v = {
    "Report_Date": "2026-04-30",
    "Gold_ETF_Name": "Nippon India ETF Gold BeES",
    "Gold_ETF_Value_Lakh": 120062.88,
    "Gold_ETF_NAV_Pct": 8.15,
    "Silver_ETF_Name": "Nippon India Silver ETF",
    "Silver_ETF_Value_Lakh": 47683.65,
    "Silver_ETF_NAV_Pct": 3.24
}

m = nippon_log["Report_Month"] == "2026-04"

for col, value in v.items():
    nippon_log.loc[m, col] = value

nippon_log.to_csv(output_file, index=False)

print(nippon_log.loc[
    m,
    ["Report_Month", "Report_Date", "Gold_ETF_Value_Lakh",
     "Gold_ETF_NAV_Pct", "Silver_ETF_Value_Lakh",
     "Silver_ETF_NAV_Pct"]
].to_string(index=False))

Report_Month Report_Date Gold_ETF_Value_Lakh Gold_ETF_NAV_Pct Silver_ETF_Value_Lakh Silver_ETF_NAV_Pct
     2026-04  2026-04-30           120062.88             8.15              47683.65               3.24


In [ ]:
v = {
    "Report_Date": "2026-03-31",
    "Gold_ETF_Name": "Nippon India ETF Gold BeES",
    "Gold_ETF_Value_Lakh": 117883.99,
    "Gold_ETF_NAV_Pct": 8.97,
    "Silver_ETF_Name": "Nippon India Silver ETF",
    "Silver_ETF_Value_Lakh": 46698.68,
    "Silver_ETF_NAV_Pct": 3.55
}

m = nippon_log["Report_Month"] == "2026-03"

for col, value in v.items():
    nippon_log.loc[m, col] = value

nippon_log.to_csv(output_file, index=False)

print(nippon_log.loc[
    m,
    ["Report_Month", "Report_Date", "Gold_ETF_Value_Lakh",
     "Gold_ETF_NAV_Pct", "Silver_ETF_Value_Lakh",
     "Silver_ETF_NAV_Pct"]
].to_string(index=False))

Report_Month Report_Date Gold_ETF_Value_Lakh Gold_ETF_NAV_Pct Silver_ETF_Value_Lakh Silver_ETF_NAV_Pct
     2026-03  2026-03-31           117883.99             8.97              46698.68               3.55


In [ ]:
v = {
    "Report_Date": "2026-02-28",
    "Gold_ETF_Name": "Nippon India ETF Gold BeES",
    "Gold_ETF_Value_Lakh": 128010.01,
    "Gold_ETF_NAV_Pct": 9.53,
    "Silver_ETF_Name": "Nippon India Silver ETF",
    "Silver_ETF_Value_Lakh": 52838.43,
    "Silver_ETF_NAV_Pct": 3.93
}

m = nippon_log["Report_Month"] == "2026-02"

for col, value in v.items():
    nippon_log.loc[m, col] = value

nippon_log.to_csv(output_file, index=False)

print(nippon_log.loc[
    m,
    ["Report_Month", "Report_Date", "Gold_ETF_Value_Lakh",
     "Gold_ETF_NAV_Pct", "Silver_ETF_Value_Lakh",
     "Silver_ETF_NAV_Pct"]
].to_string(index=False))

Report_Month Report_Date Gold_ETF_Value_Lakh Gold_ETF_NAV_Pct Silver_ETF_Value_Lakh Silver_ETF_NAV_Pct
     2026-02  2026-02-28           128010.01             9.53              52838.43               3.93


In [ ]:
v = {
    "Report_Date": "2026-01-31",
    "Gold_ETF_Name": "Nippon India ETF Gold BeES",
    "Gold_ETF_Value_Lakh": 132026.27,
    "Gold_ETF_NAV_Pct": 10.55,
    "Silver_ETF_Name": "Nippon India Silver ETF",
    "Silver_ETF_Value_Lakh": 65963.33,
    "Silver_ETF_NAV_Pct": 5.27
}

m = nippon_log["Report_Month"] == "2026-01"

for col, value in v.items():
    nippon_log.loc[m, col] = value

nippon_log.to_csv(output_file, index=False)

print(nippon_log.loc[
    m,
    ["Report_Month", "Report_Date", "Gold_ETF_Value_Lakh",
     "Gold_ETF_NAV_Pct", "Silver_ETF_Value_Lakh",
     "Silver_ETF_NAV_Pct"]
].to_string(index=False))

Report_Month Report_Date Gold_ETF_Value_Lakh Gold_ETF_NAV_Pct Silver_ETF_Value_Lakh Silver_ETF_NAV_Pct
     2026-01  2026-01-31           132026.27            10.55              65963.33               5.27


In [ ]:
v = {
    "Report_Date": "2025-12-31",
    "Gold_ETF_Name": "Nippon India ETF Gold BeES",
    "Gold_ETF_Value_Lakh": 107100.66,
    "Gold_ETF_NAV_Pct": 10.05,
    "Silver_ETF_Name": "Nippon India Silver ETF",
    "Silver_ETF_Value_Lakh": 45025.84,
    "Silver_ETF_NAV_Pct": 4.22
}

m = nippon_log["Report_Month"] == "2025-12"

for col, value in v.items():
    nippon_log.loc[m, col] = value

nippon_log.to_csv(output_file, index=False)

print(nippon_log.loc[
    m,
    ["Report_Month", "Report_Date", "Gold_ETF_Value_Lakh",
     "Gold_ETF_NAV_Pct", "Silver_ETF_Value_Lakh",
     "Silver_ETF_NAV_Pct"]
].to_string(index=False))

Report_Month Report_Date Gold_ETF_Value_Lakh Gold_ETF_NAV_Pct Silver_ETF_Value_Lakh Silver_ETF_NAV_Pct
     2025-12  2025-12-31           107100.66            10.05              45025.84               4.22


In [ ]:
v = {
    "Report_Date": "2025-11-30",
    "Gold_ETF_Name": "Nippon India ETF Gold BeES",
    "Gold_ETF_Value_Lakh": 99668.31,
    "Gold_ETF_NAV_Pct": 10.38,
    "Silver_ETF_Name": "Nippon India Silver ETF",
    "Silver_ETF_Value_Lakh": 29522.92,
    "Silver_ETF_NAV_Pct": 3.08
}

m = nippon_log["Report_Month"] == "2025-11"

for col, value in v.items():
    nippon_log.loc[m, col] = value

nippon_log.to_csv(output_file, index=False)

print(nippon_log.loc[
    m,
    ["Report_Month", "Report_Date", "Gold_ETF_Value_Lakh",
     "Gold_ETF_NAV_Pct", "Silver_ETF_Value_Lakh",
     "Silver_ETF_NAV_Pct"]
].to_string(index=False))

Report_Month Report_Date Gold_ETF_Value_Lakh Gold_ETF_NAV_Pct Silver_ETF_Value_Lakh Silver_ETF_NAV_Pct
     2025-11  2025-11-30            99668.31            10.38              29522.92               3.08


In [ ]:
v = {
    "Report_Date": "2025-10-31",
    "Gold_ETF_Name": "Nippon India ETF Gold BeES",
    "Gold_ETF_Value_Lakh": 95286.02,
    "Gold_ETF_NAV_Pct": 10.92,
    "Silver_ETF_Name": "Nippon India Silver ETF",
    "Silver_ETF_Value_Lakh": 26829.94,
    "Silver_ETF_NAV_Pct": 3.08
}

m = nippon_log["Report_Month"] == "2025-10"

for col, value in v.items():
    nippon_log.loc[m, col] = value

nippon_log.to_csv(output_file, index=False)

print(nippon_log.loc[
    m,
    ["Report_Month", "Report_Date", "Gold_ETF_Value_Lakh",
     "Gold_ETF_NAV_Pct", "Silver_ETF_Value_Lakh",
     "Silver_ETF_NAV_Pct"]
].to_string(index=False))

Report_Month Report_Date Gold_ETF_Value_Lakh Gold_ETF_NAV_Pct Silver_ETF_Value_Lakh Silver_ETF_NAV_Pct
     2025-10  2025-10-31            95286.02            10.92              26829.94               3.08


In [ ]:
v = {
    "Report_Date": "2025-09-30",
    "Gold_ETF_Name": "Nippon India ETF Gold BeES",
    "Gold_ETF_Value_Lakh": 87479.39,
    "Gold_ETF_NAV_Pct": 11.27,
    "Silver_ETF_Name": "Nippon India Silver ETF",
    "Silver_ETF_Value_Lakh": 25675.32,
    "Silver_ETF_NAV_Pct": 3.31
}

m = nippon_log["Report_Month"] == "2025-09"

for col, value in v.items():
    nippon_log.loc[m, col] = value

nippon_log.to_csv(output_file, index=False)

print(nippon_log.loc[
    m,
    ["Report_Month", "Report_Date", "Gold_ETF_Value_Lakh",
     "Gold_ETF_NAV_Pct", "Silver_ETF_Value_Lakh",
     "Silver_ETF_NAV_Pct"]
].to_string(index=False))

Report_Month Report_Date Gold_ETF_Value_Lakh Gold_ETF_NAV_Pct Silver_ETF_Value_Lakh Silver_ETF_NAV_Pct
     2025-09  2025-09-30            87479.39            11.27              25675.32               3.31


In [ ]:
v = {
    "Report_Date": "2025-08-31",
    "Gold_ETF_Name": "Nippon India ETF Gold BeES",
    "Gold_ETF_Value_Lakh": 77887.23,
    "Gold_ETF_NAV_Pct": 11.19,
    "Silver_ETF_Name": "Nippon India Silver ETF",
    "Silver_ETF_Value_Lakh": 21431.64,
    "Silver_ETF_NAV_Pct": 3.08
}

m = nippon_log["Report_Month"] == "2025-08"

for col, value in v.items():
    nippon_log.loc[m, col] = value

nippon_log.to_csv(output_file, index=False)

print(nippon_log.loc[
    m,
    ["Report_Month", "Report_Date", "Gold_ETF_Value_Lakh",
     "Gold_ETF_NAV_Pct", "Silver_ETF_Value_Lakh",
     "Silver_ETF_NAV_Pct"]
].to_string(index=False))

Report_Month Report_Date Gold_ETF_Value_Lakh Gold_ETF_NAV_Pct Silver_ETF_Value_Lakh Silver_ETF_NAV_Pct
     2025-08  2025-08-31            77887.23            11.19              21431.64               3.08


In [ ]:
v = {
    "Report_Date": "2025-07-31",
    "Gold_ETF_Name": "Nippon India ETF Gold BeES",
    "Gold_ETF_Value_Lakh": 70907.25,
    "Gold_ETF_NAV_Pct": 10.66,
    "Silver_ETF_Name": "Nippon India Silver ETF",
    "Silver_ETF_Value_Lakh": 19102.18,
    "Silver_ETF_NAV_Pct": 2.87
}

m = nippon_log["Report_Month"] == "2025-07"

for col, value in v.items():
    nippon_log.loc[m, col] = value

nippon_log.to_csv(output_file, index=False)

print(nippon_log.loc[
    m,
    ["Report_Month", "Report_Date", "Gold_ETF_Value_Lakh",
     "Gold_ETF_NAV_Pct", "Silver_ETF_Value_Lakh",
     "Silver_ETF_NAV_Pct"]
].to_string(index=False))

Report_Month Report_Date Gold_ETF_Value_Lakh Gold_ETF_NAV_Pct Silver_ETF_Value_Lakh Silver_ETF_NAV_Pct
     2025-07  2025-07-31            70907.25            10.66              19102.18               2.87


In [ ]:
v = {
    "Report_Date": "2025-06-30",
    "Gold_ETF_Name": "Nippon India ETF Gold BeES",
    "Gold_ETF_Value_Lakh": 69000.87,
    "Gold_ETF_NAV_Pct": 10.84,
    "Silver_ETF_Name": "Nippon India Silver ETF",
    "Silver_ETF_Value_Lakh": 18310.57,
    "Silver_ETF_NAV_Pct": 2.88
}

m = nippon_log["Report_Month"] == "2025-06"

for col, value in v.items():
    nippon_log.loc[m, col] = value

nippon_log.to_csv(output_file, index=False)

print(nippon_log.loc[
    m,
    ["Report_Month", "Report_Date", "Gold_ETF_Value_Lakh",
     "Gold_ETF_NAV_Pct", "Silver_ETF_Value_Lakh",
     "Silver_ETF_NAV_Pct"]
].to_string(index=False))

Report_Month Report_Date Gold_ETF_Value_Lakh Gold_ETF_NAV_Pct Silver_ETF_Value_Lakh Silver_ETF_NAV_Pct
     2025-06  2025-06-30            69000.87            10.84              18310.57               2.88


In [ ]:
v = {
    "Report_Date": "2025-05-31",
    "Gold_ETF_Name": "Nippon India ETF Gold BeES",
    "Gold_ETF_Value_Lakh": 68569.56,
    "Gold_ETF_NAV_Pct": 11.47,
    "Silver_ETF_Name": "Nippon India Silver ETF",
    "Silver_ETF_Value_Lakh": 15916.09,
    "Silver_ETF_NAV_Pct": 2.66
}

m = nippon_log["Report_Month"] == "2025-05"

for col, value in v.items():
    nippon_log.loc[m, col] = value

nippon_log.to_csv(output_file, index=False)

print(nippon_log.loc[
    m,
    ["Report_Month", "Report_Date", "Gold_ETF_Value_Lakh",
     "Gold_ETF_NAV_Pct", "Silver_ETF_Value_Lakh",
     "Silver_ETF_NAV_Pct"]
].to_string(index=False))

Report_Month Report_Date Gold_ETF_Value_Lakh Gold_ETF_NAV_Pct Silver_ETF_Value_Lakh Silver_ETF_NAV_Pct
     2025-05  2025-05-31            68569.56            11.47              15916.09               2.66


In [ ]:
v = {
    "Report_Date": "2025-04-30",
    "Gold_ETF_Name": "Nippon India ETF Gold BeES",
    "Gold_ETF_Value_Lakh": 68224.51,
    "Gold_ETF_NAV_Pct": 12.13,
    "Silver_ETF_Name": "Nippon India Silver ETF",
    "Silver_ETF_Value_Lakh": 15455.56,
    "Silver_ETF_NAV_Pct": 2.75
}

m = nippon_log["Report_Month"] == "2025-04"

for col, value in v.items():
    nippon_log.loc[m, col] = value

nippon_log.to_csv(output_file, index=False)

print(nippon_log.loc[
    m,
    ["Report_Month", "Report_Date", "Gold_ETF_Value_Lakh",
     "Gold_ETF_NAV_Pct", "Silver_ETF_Value_Lakh",
     "Silver_ETF_NAV_Pct"]
].to_string(index=False))

Report_Month Report_Date Gold_ETF_Value_Lakh Gold_ETF_NAV_Pct Silver_ETF_Value_Lakh Silver_ETF_NAV_Pct
     2025-04  2025-04-30            68224.51            12.13              15455.56               2.75


In [ ]:
v = {
    "Report_Date": "2025-03-31",
    "Gold_ETF_Name": "Nippon India ETF Gold BeES",
    "Gold_ETF_Value_Lakh": 63971.80,
    "Gold_ETF_NAV_Pct": 12.00,
    "Silver_ETF_Name": "Nippon India Silver ETF",
    "Silver_ETF_Value_Lakh": 16464.34,
    "Silver_ETF_NAV_Pct": 3.09
}

m = nippon_log["Report_Month"] == "2025-03"

for col, value in v.items():
    nippon_log.loc[m, col] = value

nippon_log.to_csv(output_file, index=False)

print(nippon_log.loc[
    m,
    ["Report_Month", "Report_Date", "Gold_ETF_Value_Lakh",
     "Gold_ETF_NAV_Pct", "Silver_ETF_Value_Lakh",
     "Silver_ETF_NAV_Pct"]
].to_string(index=False))

Report_Month Report_Date Gold_ETF_Value_Lakh Gold_ETF_NAV_Pct Silver_ETF_Value_Lakh Silver_ETF_NAV_Pct
     2025-03  2025-03-31             63971.8             12.0              16464.34               3.09


In [ ]:
v = {
    "Report_Date": "2025-02-28",
    "Gold_ETF_Name": "Nippon India ETF Gold BeES",
    "Gold_ETF_Value_Lakh": 61332.19,
    "Gold_ETF_NAV_Pct": 12.39,
    "Silver_ETF_Name": "Nippon India Silver ETF",
    "Silver_ETF_Value_Lakh": 15259.88,
    "Silver_ETF_NAV_Pct": 3.08
}

m = nippon_log["Report_Month"] == "2025-02"

for col, value in v.items():
    nippon_log.loc[m, col] = value

nippon_log.to_csv(output_file, index=False)

print(nippon_log.loc[
    m,
    ["Report_Month", "Report_Date", "Gold_ETF_Value_Lakh",
     "Gold_ETF_NAV_Pct", "Silver_ETF_Value_Lakh",
     "Silver_ETF_NAV_Pct"]
].to_string(index=False))

Report_Month Report_Date Gold_ETF_Value_Lakh Gold_ETF_NAV_Pct Silver_ETF_Value_Lakh Silver_ETF_NAV_Pct
     2025-02  2025-02-28            61332.19            12.39              15259.88               3.08


In [ ]:
v = {
    "Report_Date": "2025-01-31",
    "Gold_ETF_Name": "Nippon India ETF Gold BeES",
    "Gold_ETF_Value_Lakh": 59589.70,
    "Gold_ETF_NAV_Pct": 11.91,
    "Silver_ETF_Name": "Nippon India Silver ETF",
    "Silver_ETF_Value_Lakh": 15295.30,
    "Silver_ETF_NAV_Pct": 3.06
}

m = nippon_log["Report_Month"] == "2025-01"

for col, value in v.items():
    nippon_log.loc[m, col] = value

nippon_log.to_csv(output_file, index=False)

print(nippon_log.loc[
    m,
    ["Report_Month", "Report_Date", "Gold_ETF_Value_Lakh",
     "Gold_ETF_NAV_Pct", "Silver_ETF_Value_Lakh",
     "Silver_ETF_NAV_Pct"]
].to_string(index=False))

Report_Month Report_Date Gold_ETF_Value_Lakh Gold_ETF_NAV_Pct Silver_ETF_Value_Lakh Silver_ETF_NAV_Pct
     2025-01  2025-01-31             59589.7            11.91               15295.3               3.06


In [ ]:
v = {
    "Report_Date": "2024-12-31",
    "Gold_ETF_Name": "Nippon India ETF Gold BeES",
    "Gold_ETF_Value_Lakh": 49459.16,
    "Gold_ETF_NAV_Pct": 10.20,
    "Silver_ETF_Name": "Nippon India Silver ETF",
    "Silver_ETF_Value_Lakh": 12447.35,
    "Silver_ETF_NAV_Pct": 2.57
}

m = nippon_log["Report_Month"] == "2024-12"

for col, value in v.items():
    nippon_log.loc[m, col] = value

nippon_log.to_csv(output_file, index=False)

print(nippon_log.loc[
    m,
    ["Report_Month", "Report_Date", "Gold_ETF_Value_Lakh",
     "Gold_ETF_NAV_Pct", "Silver_ETF_Value_Lakh",
     "Silver_ETF_NAV_Pct"]
].to_string(index=False))

Report_Month Report_Date Gold_ETF_Value_Lakh Gold_ETF_NAV_Pct Silver_ETF_Value_Lakh Silver_ETF_NAV_Pct
     2024-12  2024-12-31            49459.16             10.2              12447.35               2.57


In [ ]:
v = {
    "Report_Date": "2024-11-30",
    "Gold_ETF_Name": "Nippon India ETF Gold BeES",
    "Gold_ETF_Value_Lakh": 48333.35,
    "Gold_ETF_NAV_Pct": 10.32,
    "Silver_ETF_Name": "Nippon India Silver ETF",
    "Silver_ETF_Value_Lakh": 12486.98,
    "Silver_ETF_NAV_Pct": 2.67
}

m = nippon_log["Report_Month"] == "2024-11"

for col, value in v.items():
    nippon_log.loc[m, col] = value

nippon_log.to_csv(output_file, index=False)

print(nippon_log.loc[
    m,
    ["Report_Month", "Report_Date", "Gold_ETF_Value_Lakh",
     "Gold_ETF_NAV_Pct", "Silver_ETF_Value_Lakh",
     "Silver_ETF_NAV_Pct"]
].to_string(index=False))

Report_Month Report_Date Gold_ETF_Value_Lakh Gold_ETF_NAV_Pct Silver_ETF_Value_Lakh Silver_ETF_NAV_Pct
     2024-11  2024-11-30            48333.35            10.32              12486.98               2.67


In [ ]:
v = {
    "Report_Date": "2024-10-31",
    "Gold_ETF_Name": "Nippon India ETF Gold BeES",
    "Gold_ETF_Value_Lakh": 45224.67,
    "Gold_ETF_NAV_Pct": 10.05,
    "Silver_ETF_Name": "Nippon India Silver ETF",
    "Silver_ETF_Value_Lakh": 9159.70,
    "Silver_ETF_NAV_Pct": 2.04
}

m = nippon_log["Report_Month"] == "2024-10"

for col, value in v.items():
    nippon_log.loc[m, col] = value

nippon_log.to_csv(output_file, index=False)

print(nippon_log.loc[
    m,
    ["Report_Month", "Report_Date", "Gold_ETF_Value_Lakh",
     "Gold_ETF_NAV_Pct", "Silver_ETF_Value_Lakh",
     "Silver_ETF_NAV_Pct"]
].to_string(index=False))

Report_Month Report_Date Gold_ETF_Value_Lakh Gold_ETF_NAV_Pct Silver_ETF_Value_Lakh Silver_ETF_NAV_Pct
     2024-10  2024-10-31            45224.67            10.05                9159.7               2.04


In [ ]:
v = {
    "Report_Date": "2024-09-30",
    "Gold_ETF_Name": "Nippon India ETF Gold BeES",
    "Gold_ETF_Value_Lakh": 42947.22,
    "Gold_ETF_NAV_Pct": 9.89,
    "Silver_ETF_Name": "Nippon India Silver ETF",
    "Silver_ETF_Value_Lakh": 8554.37,
    "Silver_ETF_NAV_Pct": 1.97
}

m = nippon_log["Report_Month"] == "2024-09"

for col, value in v.items():
    nippon_log.loc[m, col] = value

nippon_log.to_csv(output_file, index=False)

print(nippon_log.loc[
    m,
    ["Report_Month", "Report_Date", "Gold_ETF_Value_Lakh",
     "Gold_ETF_NAV_Pct", "Silver_ETF_Value_Lakh",
     "Silver_ETF_NAV_Pct"]
].to_string(index=False))

Report_Month Report_Date Gold_ETF_Value_Lakh Gold_ETF_NAV_Pct Silver_ETF_Value_Lakh Silver_ETF_NAV_Pct
     2024-09  2024-09-30            42947.22             9.89               8554.37               1.97


In [ ]:
v = {
    "Report_Date": "2024-08-31",
    "Gold_ETF_Name": "Nippon India ETF Gold BeES",
    "Gold_ETF_Value_Lakh": 40994.15,
    "Gold_ETF_NAV_Pct": 10.01,
    "Silver_ETF_Name": "Nippon India Silver ETF",
    "Silver_ETF_Value_Lakh": 7905.32,
    "Silver_ETF_NAV_Pct": 1.93
}

m = nippon_log["Report_Month"] == "2024-08"

for col, value in v.items():
    nippon_log.loc[m, col] = value

nippon_log.to_csv(output_file, index=False)

print(nippon_log.loc[
    m,
    ["Report_Month", "Report_Date", "Gold_ETF_Value_Lakh",
     "Gold_ETF_NAV_Pct", "Silver_ETF_Value_Lakh",
     "Silver_ETF_NAV_Pct"]
].to_string(index=False))

Report_Month Report_Date Gold_ETF_Value_Lakh Gold_ETF_NAV_Pct Silver_ETF_Value_Lakh Silver_ETF_NAV_Pct
     2024-08  2024-08-31            40994.15            10.01               7905.32               1.93


In [ ]:
v = {
    "Report_Date": "2024-07-31",
    "Gold_ETF_Name": "Nippon India ETF Gold BeES",
    "Gold_ETF_Value_Lakh": 39656.06,
    "Gold_ETF_NAV_Pct": 10.29,
    "Silver_ETF_Name": "Nippon India Silver ETF",
    "Silver_ETF_Value_Lakh": 1227.60,
    "Silver_ETF_NAV_Pct": 0.32
}

m = nippon_log["Report_Month"] == "2024-07"

for col, value in v.items():
    nippon_log.loc[m, col] = value

nippon_log.to_csv(output_file, index=False)

print(nippon_log.loc[
    m,
    ["Report_Month", "Report_Date", "Gold_ETF_Value_Lakh",
     "Gold_ETF_NAV_Pct", "Silver_ETF_Value_Lakh",
     "Silver_ETF_NAV_Pct"]
].to_string(index=False))

Report_Month Report_Date Gold_ETF_Value_Lakh Gold_ETF_NAV_Pct Silver_ETF_Value_Lakh Silver_ETF_NAV_Pct
     2024-07  2024-07-31            39656.06            10.29                1227.6               0.32


In [ ]:
v = {
    "Report_Date": "2024-06-30",
    "Gold_ETF_Name": "Nippon India ETF Gold BeES",
    "Gold_ETF_Value_Lakh": 40973.87,
    "Gold_ETF_NAV_Pct": 11.38,
    "Review_Notes": "No direct silver ETF identified in keyword search; verify before treating as absent."
}

m = nippon_log["Report_Month"] == "2024-06"

for col, value in v.items():
    nippon_log.loc[m, col] = value

nippon_log.to_csv(output_file, index=False)

print(nippon_log.loc[
    m,
    ["Report_Month", "Report_Date", "Gold_ETF_Value_Lakh",
     "Gold_ETF_NAV_Pct", "Silver_ETF_Value_Lakh",
     "Silver_ETF_NAV_Pct", "Review_Notes"]
].to_string(index=False))

Report_Month Report_Date Gold_ETF_Value_Lakh Gold_ETF_NAV_Pct Silver_ETF_Value_Lakh Silver_ETF_NAV_Pct                                                                         Review_Notes
     2024-06  2024-06-30            40973.87            11.38                  <NA>               <NA> No direct silver ETF identified in keyword search; verify before treating as absent.


In [ ]:
v = {
    "Report_Date": "2024-05-31",
    "Gold_ETF_Name": "Nippon India ETF Gold BeES",
    "Gold_ETF_Value_Lakh": 41419.90,
    "Gold_ETF_NAV_Pct": 12.50,
    "Review_Notes": "No direct silver ETF identified in keyword search; verify before treating as absent."
}

m = nippon_log["Report_Month"] == "2024-05"

for col, value in v.items():
    nippon_log.loc[m, col] = value

nippon_log.to_csv(output_file, index=False)

print(nippon_log.loc[
    m,
    ["Report_Month", "Report_Date", "Gold_ETF_Value_Lakh",
     "Gold_ETF_NAV_Pct", "Silver_ETF_Value_Lakh",
     "Silver_ETF_NAV_Pct", "Review_Notes"]
].to_string(index=False))

Report_Month Report_Date Gold_ETF_Value_Lakh Gold_ETF_NAV_Pct Silver_ETF_Value_Lakh Silver_ETF_NAV_Pct                                                                         Review_Notes
     2024-05  2024-05-31             41419.9             12.5                  <NA>               <NA> No direct silver ETF identified in keyword search; verify before treating as absent.


In [ ]:
v = {
    "Report_Date": "2024-04-30",
    "Gold_ETF_Name": "Nippon India ETF Gold BeES",
    "Gold_ETF_Value_Lakh": 41142.82,
    "Gold_ETF_NAV_Pct": 13.19,
    "Review_Notes": "No direct silver ETF identified in keyword search; verify before treating as absent."
}

m = nippon_log["Report_Month"] == "2024-04"

for col, value in v.items():
    nippon_log.loc[m, col] = value

nippon_log.to_csv(output_file, index=False)

print(nippon_log.loc[
    m,
    ["Report_Month", "Report_Date", "Gold_ETF_Value_Lakh",
     "Gold_ETF_NAV_Pct", "Silver_ETF_Value_Lakh",
     "Silver_ETF_NAV_Pct", "Review_Notes"]
].to_string(index=False))

Report_Month Report_Date Gold_ETF_Value_Lakh Gold_ETF_NAV_Pct Silver_ETF_Value_Lakh Silver_ETF_NAV_Pct                                                                         Review_Notes
     2024-04  2024-04-30            41142.82            13.19                  <NA>               <NA> No direct silver ETF identified in keyword search; verify before treating as absent.


In [ ]:
v = {
    "Report_Date": "2024-03-31",
    "Gold_ETF_Name": "Nippon India ETF Gold BeES",
    "Gold_ETF_Value_Lakh": 37408.00,
    "Gold_ETF_NAV_Pct": 12.88,
    "Review_Notes": "No direct silver ETF identified in keyword search; verify before treating as absent."
}

m = nippon_log["Report_Month"] == "2024-03"

for col, value in v.items():
    nippon_log.loc[m, col] = value

nippon_log.to_csv(output_file, index=False)

print(nippon_log.loc[
    m,
    ["Report_Month", "Report_Date", "Gold_ETF_Value_Lakh",
     "Gold_ETF_NAV_Pct", "Silver_ETF_Value_Lakh",
     "Silver_ETF_NAV_Pct", "Review_Notes"]
].to_string(index=False))

Report_Month Report_Date Gold_ETF_Value_Lakh Gold_ETF_NAV_Pct Silver_ETF_Value_Lakh Silver_ETF_NAV_Pct                                                                         Review_Notes
     2024-03  2024-03-31             37408.0            12.88                  <NA>               <NA> No direct silver ETF identified in keyword search; verify before treating as absent.


In [ ]:
v = {
    "Report_Date": "2024-02-29",
    "Gold_ETF_Name": "Nippon India ETF Gold BeES",
    "Gold_ETF_Value_Lakh": 34613.23,
    "Gold_ETF_NAV_Pct": 12.81,
    "Review_Notes": "No direct silver ETF identified in keyword search; verify before treating as absent."
}

m = nippon_log["Report_Month"] == "2024-02"

for col, value in v.items():
    nippon_log.loc[m, col] = value

nippon_log.to_csv(output_file, index=False)

print(nippon_log.loc[
    m,
    ["Report_Month", "Report_Date", "Gold_ETF_Value_Lakh",
     "Gold_ETF_NAV_Pct", "Silver_ETF_Value_Lakh",
     "Silver_ETF_NAV_Pct", "Review_Notes"]
].to_string(index=False))

Report_Month Report_Date Gold_ETF_Value_Lakh Gold_ETF_NAV_Pct Silver_ETF_Value_Lakh Silver_ETF_NAV_Pct                                                                         Review_Notes
     2024-02  2024-02-29            34613.23            12.81                  <NA>               <NA> No direct silver ETF identified in keyword search; verify before treating as absent.


In [ ]:
v = {
    "Report_Date": "2024-01-31",
    "Gold_ETF_Name": "Nippon India ETF Gold BeES",
    "Gold_ETF_Value_Lakh": 33269.06,
    "Gold_ETF_NAV_Pct": 13.08,
    "Review_Notes": "No direct silver ETF identified in keyword search; verify before treating as absent."
}

m = nippon_log["Report_Month"] == "2024-01"

for col, value in v.items():
    nippon_log.loc[m, col] = value

nippon_log.to_csv(output_file, index=False)

print(nippon_log.loc[
    m,
    ["Report_Month", "Report_Date", "Gold_ETF_Value_Lakh",
     "Gold_ETF_NAV_Pct", "Silver_ETF_Value_Lakh",
     "Silver_ETF_NAV_Pct", "Review_Notes"]
].to_string(index=False))

Report_Month Report_Date Gold_ETF_Value_Lakh Gold_ETF_NAV_Pct Silver_ETF_Value_Lakh Silver_ETF_NAV_Pct                                                                         Review_Notes
     2024-01  2024-01-31            33269.06            13.08                  <NA>               <NA> No direct silver ETF identified in keyword search; verify before treating as absent.


In [ ]:
print("Total rows:", len(nippon_log))
print("Unique months:", nippon_log["Report_Month"].nunique())
print("Duplicate months:", nippon_log["Report_Month"].duplicated().sum())
print("Missing report dates:", nippon_log["Report_Date"].isna().sum())

print(
    "Gold ETF values recorded:",
    nippon_log["Gold_ETF_Value_Lakh"].notna().sum()
)

print(
    "Silver ETF values recorded:",
    nippon_log["Silver_ETF_Value_Lakh"].notna().sum()
)

print("\nFirst 3 months:")
print(nippon_log.head(3).to_string(index=False))

print("\nLast 3 months:")
print(nippon_log.tail(3).to_string(index=False))

print("\nCSV saved at:", output_file)

Total rows: 33
Unique months: 33
Duplicate months: 0
Missing report dates: 0
Gold ETF values recorded: 33
Silver ETF values recorded: 27

First 3 months:
Report_Month Report_Date              Gold_ETF_Name Gold_ETF_Value_Lakh Gold_ETF_NAV_Pct Silver_ETF_Name Silver_ETF_Value_Lakh Silver_ETF_NAV_Pct Gold_Futures_Value_Lakh Gold_Futures_NAV_Pct Silver_Futures_Value_Lakh Silver_Futures_NAV_Pct Options_and_Other_Instruments                                                                         Review_Notes
     2024-01  2024-01-31 Nippon India ETF Gold BeES            33269.06            13.08                                  <NA>               <NA>                    <NA>                 <NA>                      <NA>                   <NA>                               No direct silver ETF identified in keyword search; verify before treating as absent.
     2024-02  2024-02-29 Nippon India ETF Gold BeES            34613.23            12.81                                  <NA>          

In [ ]:
v = {
    "Gold_Futures_Value_Lakh": 61104.76,
    "Gold_Futures_NAV_Pct": 3.39,
    "Silver_Futures_Value_Lakh": 21475.78,
    "Silver_Futures_NAV_Pct": 1.19
}

m = nippon_log["Report_Month"] == "2026-09"

for col, value in v.items():
    nippon_log.loc[m, col] = value

nippon_log.to_csv(output_file, index=False)

print(nippon_log.loc[
    m,
    ["Report_Month", "Gold_Futures_Value_Lakh",
     "Gold_Futures_NAV_Pct", "Silver_Futures_Value_Lakh",
     "Silver_Futures_NAV_Pct"]
].to_string(index=False))

Report_Month Gold_Futures_Value_Lakh Gold_Futures_NAV_Pct Silver_Futures_Value_Lakh Silver_Futures_NAV_Pct
     2026-09                61104.76                 3.39                  21475.78                   1.19


In [ ]:
v = {
    "Gold_Futures_Value_Lakh": 63328.60,
    "Gold_Futures_NAV_Pct": 3.52,
    "Silver_Futures_Value_Lakh": 23051.62,
    "Silver_Futures_NAV_Pct": 1.28
}

m = nippon_log["Report_Month"] == "2026-08"

for col, value in v.items():
    nippon_log.loc[m, col] = value

nippon_log.to_csv(output_file, index=False)

print(nippon_log.loc[
    m,
    ["Report_Month", "Gold_Futures_Value_Lakh",
     "Gold_Futures_NAV_Pct", "Silver_Futures_Value_Lakh",
     "Silver_Futures_NAV_Pct"]
].to_string(index=False))

Report_Month Gold_Futures_Value_Lakh Gold_Futures_NAV_Pct Silver_Futures_Value_Lakh Silver_Futures_NAV_Pct
     2026-08                 63328.6                 3.52                  23051.62                   1.28


In [ ]:
v = {
    "Gold_Futures_Value_Lakh": 32976.48,
    "Gold_Futures_NAV_Pct": 1.95,
    "Silver_Futures_Value_Lakh": 2932.17,
    "Silver_Futures_NAV_Pct": 0.17
}

m = nippon_log["Report_Month"] == "2026-07"

for col, value in v.items():
    nippon_log.loc[m, col] = value

nippon_log.to_csv(output_file, index=False)

print(nippon_log.loc[
    m,
    ["Report_Month", "Gold_Futures_Value_Lakh",
     "Gold_Futures_NAV_Pct", "Silver_Futures_Value_Lakh",
     "Silver_Futures_NAV_Pct"]
].to_string(index=False))

Report_Month Gold_Futures_Value_Lakh Gold_Futures_NAV_Pct Silver_Futures_Value_Lakh Silver_Futures_NAV_Pct
     2026-07                32976.48                 1.95                   2932.17                   0.17


In [ ]:
v = {
    "Gold_Futures_Value_Lakh": 23375.08,
    "Gold_Futures_NAV_Pct": 1.46,
    "Review_Notes": "Silver futures not identified in keyword search; verify before treating as absent."
}

m = nippon_log["Report_Month"] == "2026-06"

for col, value in v.items():
    nippon_log.loc[m, col] = value

nippon_log.to_csv(output_file, index=False)

print(nippon_log.loc[
    m,
    ["Report_Month", "Gold_Futures_Value_Lakh",
     "Gold_Futures_NAV_Pct", "Silver_Futures_Value_Lakh",
     "Silver_Futures_NAV_Pct", "Review_Notes"]
].to_string(index=False))

Report_Month Gold_Futures_Value_Lakh Gold_Futures_NAV_Pct Silver_Futures_Value_Lakh Silver_Futures_NAV_Pct                                                                       Review_Notes
     2026-06                23375.08                 1.46                      <NA>                   <NA> Silver futures not identified in keyword search; verify before treating as absent.


In [ ]:
v = {
    "Gold_Futures_Value_Lakh": 26389.40,
    "Gold_Futures_NAV_Pct": 1.70,
    "Review_Notes": "Silver futures not identified in keyword search; verify before treating as absent."
}

m = nippon_log["Report_Month"] == "2026-05"

for col, value in v.items():
    nippon_log.loc[m, col] = value

nippon_log.to_csv(output_file, index=False)

print(nippon_log.loc[
    m,
    ["Report_Month", "Gold_Futures_Value_Lakh",
     "Gold_Futures_NAV_Pct", "Silver_Futures_Value_Lakh",
     "Silver_Futures_NAV_Pct", "Review_Notes"]
].to_string(index=False))

Report_Month Gold_Futures_Value_Lakh Gold_Futures_NAV_Pct Silver_Futures_Value_Lakh Silver_Futures_NAV_Pct                                                                       Review_Notes
     2026-05                 26389.4                  1.7                      <NA>                   <NA> Silver futures not identified in keyword search; verify before treating as absent.


In [ ]:
v = {
    "Gold_Futures_Value_Lakh": 36266.64,
    "Gold_Futures_NAV_Pct": 2.46,
    "Review_Notes": "Silver futures not identified in keyword search; verify before treating as absent."
}

m = nippon_log["Report_Month"] == "2026-04"

for col, value in v.items():
    nippon_log.loc[m, col] = value

nippon_log.to_csv(output_file, index=False)

print(nippon_log.loc[
    m,
    ["Report_Month", "Gold_Futures_Value_Lakh",
     "Gold_Futures_NAV_Pct", "Silver_Futures_Value_Lakh",
     "Silver_Futures_NAV_Pct", "Review_Notes"]
].to_string(index=False))

Report_Month Gold_Futures_Value_Lakh Gold_Futures_NAV_Pct Silver_Futures_Value_Lakh Silver_Futures_NAV_Pct                                                                       Review_Notes
     2026-04                36266.64                 2.46                      <NA>                   <NA> Silver futures not identified in keyword search; verify before treating as absent.


In [ ]:
m = nippon_log["Report_Month"] == "2026-03"

existing_note = nippon_log.loc[m, "Review_Notes"].fillna("")

nippon_log.loc[m, "Review_Notes"] = existing_note.apply(
    lambda note: (
        note + "; " if note else ""
    ) + "Gold and silver futures not identified in matching rows; verify before treating as absent."
)

nippon_log.to_csv(output_file, index=False)

print(nippon_log.loc[
    m,
    ["Report_Month", "Gold_Futures_Value_Lakh",
     "Gold_Futures_NAV_Pct", "Silver_Futures_Value_Lakh",
     "Silver_Futures_NAV_Pct", "Review_Notes"]
].to_string(index=False))

Report_Month Gold_Futures_Value_Lakh Gold_Futures_NAV_Pct Silver_Futures_Value_Lakh Silver_Futures_NAV_Pct                                                                               Review_Notes
     2026-03                    <NA>                 <NA>                      <NA>                   <NA> Gold and silver futures not identified in matching rows; verify before treating as absent.


In [ ]:
v = {
    "Gold_Futures_Value_Lakh": 22694.56,
    "Gold_Futures_NAV_Pct": 1.69,
    "Review_Notes": "Silver futures not identified in keyword search; verify before treating as absent."
}

m = nippon_log["Report_Month"] == "2026-02"

for col, value in v.items():
    nippon_log.loc[m, col] = value

nippon_log.to_csv(output_file, index=False)

print(nippon_log.loc[
    m,
    ["Report_Month", "Gold_Futures_Value_Lakh",
     "Gold_Futures_NAV_Pct", "Silver_Futures_Value_Lakh",
     "Silver_Futures_NAV_Pct", "Review_Notes"]
].to_string(index=False))

Report_Month Gold_Futures_Value_Lakh Gold_Futures_NAV_Pct Silver_Futures_Value_Lakh Silver_Futures_NAV_Pct                                                                       Review_Notes
     2026-02                22694.56                 1.69                      <NA>                   <NA> Silver futures not identified in keyword search; verify before treating as absent.


In [ ]:
v = {
    "Gold_Futures_Value_Lakh": 25898.65,
    "Gold_Futures_NAV_Pct": 2.07,
    "Silver_Futures_Value_Lakh": 9633.53,
    "Silver_Futures_NAV_Pct": 0.77
}

m = nippon_log["Report_Month"] == "2026-01"

for col, value in v.items():
    nippon_log.loc[m, col] = value

nippon_log.to_csv(output_file, index=False)

print(nippon_log.loc[
    m,
    ["Report_Month", "Gold_Futures_Value_Lakh",
     "Gold_Futures_NAV_Pct", "Silver_Futures_Value_Lakh",
     "Silver_Futures_NAV_Pct"]
].to_string(index=False))

Report_Month Gold_Futures_Value_Lakh Gold_Futures_NAV_Pct Silver_Futures_Value_Lakh Silver_Futures_NAV_Pct
     2026-01                25898.65                 2.07                   9633.53                   0.77


In [ ]:
v = {
    "Gold_Futures_Value_Lakh": 20317.05,
    "Gold_Futures_NAV_Pct": 1.91,
    "Silver_Futures_Value_Lakh": 13434.96,
    "Silver_Futures_NAV_Pct": 1.26
}

m = nippon_log["Report_Month"] == "2025-12"

for col, value in v.items():
    nippon_log.loc[m, col] = value

nippon_log.to_csv(output_file, index=False)

print(nippon_log.loc[
    m,
    ["Report_Month", "Gold_Futures_Value_Lakh",
     "Gold_Futures_NAV_Pct", "Silver_Futures_Value_Lakh",
     "Silver_Futures_NAV_Pct"]
].to_string(index=False))

Report_Month Gold_Futures_Value_Lakh Gold_Futures_NAV_Pct Silver_Futures_Value_Lakh Silver_Futures_NAV_Pct
     2025-12                20317.05                 1.91                  13434.96                   1.26


In [ ]:
v = {
    "Gold_Futures_Value_Lakh": 10360.32,
    "Gold_Futures_NAV_Pct": 1.08,
    "Silver_Futures_Value_Lakh": 6824.26,
    "Silver_Futures_NAV_Pct": 0.71
}

m = nippon_log["Report_Month"] == "2025-11"

for col, value in v.items():
    nippon_log.loc[m, col] = value

nippon_log.to_csv(output_file, index=False)

print(nippon_log.loc[
    m,
    ["Report_Month", "Gold_Futures_Value_Lakh",
     "Gold_Futures_NAV_Pct", "Silver_Futures_Value_Lakh",
     "Silver_Futures_NAV_Pct"]
].to_string(index=False))

NameError: name 'nippon_log' is not defined

In [28]:
import pandas as pd
from pathlib import Path

PROJECT_DIR = Path.cwd().parent
OUTPUT_DIR = PROJECT_DIR / "02_CLEAN_DATA"
output_file = OUTPUT_DIR / "Nippon_India_Monthly_Holdings_Log.csv"

nippon_log = pd.read_csv(
    output_file,
    dtype={"Report_Month": str}
)

print("Total rows:", len(nippon_log))
print("First month:", nippon_log["Report_Month"].iloc[0])
print("Last month:", nippon_log["Report_Month"].iloc[-1])

Total rows: 33
First month: 2024-01
Last month: 2026-09


In [29]:
v = {
    "Gold_Futures_Value_Lakh": 10360.32,
    "Gold_Futures_NAV_Pct": 1.08,
    "Silver_Futures_Value_Lakh": 6824.26,
    "Silver_Futures_NAV_Pct": 0.71
}

m = nippon_log["Report_Month"] == "2025-11"

for col, value in v.items():
    nippon_log.loc[m, col] = value

nippon_log.to_csv(output_file, index=False)

print(nippon_log.loc[
    m,
    ["Report_Month", "Gold_Futures_Value_Lakh",
     "Gold_Futures_NAV_Pct", "Silver_Futures_Value_Lakh",
     "Silver_Futures_NAV_Pct"]
].to_string(index=False))

Report_Month  Gold_Futures_Value_Lakh  Gold_Futures_NAV_Pct  Silver_Futures_Value_Lakh  Silver_Futures_NAV_Pct
     2025-11                 10360.32                  1.08                    6824.26                    0.71


In [30]:
v = {
    "Gold_Futures_Value_Lakh": 14547.84,
    "Gold_Futures_NAV_Pct": 1.67,
    "Silver_Futures_Value_Lakh": 11566.39,
    "Silver_Futures_NAV_Pct": 1.33
}

m = nippon_log["Report_Month"] == "2025-10"

for col, value in v.items():
    nippon_log.loc[m, col] = value

nippon_log.to_csv(output_file, index=False)

print(nippon_log.loc[
    m,
    ["Report_Month", "Gold_Futures_Value_Lakh",
     "Gold_Futures_NAV_Pct", "Silver_Futures_Value_Lakh",
     "Silver_Futures_NAV_Pct"]
].to_string(index=False))

Report_Month  Gold_Futures_Value_Lakh  Gold_Futures_NAV_Pct  Silver_Futures_Value_Lakh  Silver_Futures_NAV_Pct
     2025-10                 14547.84                  1.67                   11566.39                    1.33


In [31]:
v = {
    "Gold_Futures_Value_Lakh": 4690.60,
    "Gold_Futures_NAV_Pct": 0.60,
    "Silver_Futures_Value_Lakh": 5543.66,
    "Silver_Futures_NAV_Pct": 0.71
}

m = nippon_log["Report_Month"] == "2025-09"

for col, value in v.items():
    nippon_log.loc[m, col] = value

nippon_log.to_csv(output_file, index=False)

print(nippon_log.loc[
    m,
    ["Report_Month", "Gold_Futures_Value_Lakh",
     "Gold_Futures_NAV_Pct", "Silver_Futures_Value_Lakh",
     "Silver_Futures_NAV_Pct"]
].to_string(index=False))

Report_Month  Gold_Futures_Value_Lakh  Gold_Futures_NAV_Pct  Silver_Futures_Value_Lakh  Silver_Futures_NAV_Pct
     2025-09                   4690.6                   0.6                    5543.66                    0.71


In [32]:
m = nippon_log["Report_Month"] == "2025-08"

existing_note = nippon_log.loc[m, "Review_Notes"].fillna("")

nippon_log.loc[m, "Review_Notes"] = existing_note.apply(
    lambda note: (
        note + "; " if note else ""
    ) + "Gold and silver futures not identified in matching rows; verify before treating as absent."
)

nippon_log.to_csv(output_file, index=False)

print(nippon_log.loc[
    m,
    ["Report_Month", "Gold_Futures_Value_Lakh",
     "Gold_Futures_NAV_Pct", "Silver_Futures_Value_Lakh",
     "Silver_Futures_NAV_Pct", "Review_Notes"]
].to_string(index=False))

Report_Month  Gold_Futures_Value_Lakh  Gold_Futures_NAV_Pct  Silver_Futures_Value_Lakh  Silver_Futures_NAV_Pct                                                                               Review_Notes
     2025-08                      NaN                   NaN                        NaN                     NaN Gold and silver futures not identified in matching rows; verify before treating as absent.


In [33]:
v = {
    "Gold_Futures_Value_Lakh": 5926.14,
    "Gold_Futures_NAV_Pct": 0.89,
    "Silver_Futures_Value_Lakh": 6598.32,
    "Silver_Futures_NAV_Pct": 0.99
}

m = nippon_log["Report_Month"] == "2025-07"

for col, value in v.items():
    nippon_log.loc[m, col] = value

nippon_log.to_csv(output_file, index=False)

print(nippon_log.loc[
    m,
    ["Report_Month", "Gold_Futures_Value_Lakh",
     "Gold_Futures_NAV_Pct", "Silver_Futures_Value_Lakh",
     "Silver_Futures_NAV_Pct"]
].to_string(index=False))

Report_Month  Gold_Futures_Value_Lakh  Gold_Futures_NAV_Pct  Silver_Futures_Value_Lakh  Silver_Futures_NAV_Pct
     2025-07                  5926.14                  0.89                    6598.32                    0.99


In [34]:
v = {
    "Gold_Futures_Value_Lakh": 5764.50,
    "Gold_Futures_NAV_Pct": 0.91,
    "Silver_Futures_Value_Lakh": 5102.02,
    "Silver_Futures_NAV_Pct": 0.80
}

m = nippon_log["Report_Month"] == "2025-06"

for col, value in v.items():
    nippon_log.loc[m, col] = value

nippon_log.to_csv(output_file, index=False)

print(nippon_log.loc[
    m,
    ["Report_Month", "Gold_Futures_Value_Lakh",
     "Gold_Futures_NAV_Pct", "Silver_Futures_Value_Lakh",
     "Silver_Futures_NAV_Pct"]
].to_string(index=False))

Report_Month  Gold_Futures_Value_Lakh  Gold_Futures_NAV_Pct  Silver_Futures_Value_Lakh  Silver_Futures_NAV_Pct
     2025-06                   5764.5                  0.91                    5102.02                     0.8


In [35]:
v = {
    "Gold_Futures_Value_Lakh": 9587.50,
    "Gold_Futures_NAV_Pct": 1.60,
    "Silver_Futures_Value_Lakh": 1455.23,
    "Silver_Futures_NAV_Pct": 0.24
}

m = nippon_log["Report_Month"] == "2025-05"

for col, value in v.items():
    nippon_log.loc[m, col] = value

nippon_log.to_csv(output_file, index=False)

print(nippon_log.loc[
    m,
    ["Report_Month", "Gold_Futures_Value_Lakh",
     "Gold_Futures_NAV_Pct", "Silver_Futures_Value_Lakh",
     "Silver_Futures_NAV_Pct"]
].to_string(index=False))

Report_Month  Gold_Futures_Value_Lakh  Gold_Futures_NAV_Pct  Silver_Futures_Value_Lakh  Silver_Futures_NAV_Pct
     2025-05                   9587.5                   1.6                    1455.23                    0.24


In [36]:
v = {
    "Gold_Futures_Value_Lakh": 2841.06,
    "Gold_Futures_NAV_Pct": 0.51,
    "Silver_Futures_Value_Lakh": 863.25,
    "Silver_Futures_NAV_Pct": 0.15
}

m = nippon_log["Report_Month"] == "2025-04"

for col, value in v.items():
    nippon_log.loc[m, col] = value

nippon_log.to_csv(output_file, index=False)

print(nippon_log.loc[
    m,
    ["Report_Month", "Gold_Futures_Value_Lakh",
     "Gold_Futures_NAV_Pct", "Silver_Futures_Value_Lakh",
     "Silver_Futures_NAV_Pct"]
].to_string(index=False))

Report_Month  Gold_Futures_Value_Lakh  Gold_Futures_NAV_Pct  Silver_Futures_Value_Lakh  Silver_Futures_NAV_Pct
     2025-04                  2841.06                  0.51                     863.25                    0.15


In [37]:
m = nippon_log["Report_Month"] == "2025-03"

note = (
    "Gold and silver futures/options not identified in matching rows; "
    "verify before treating as absent."
)

def add_review_note(existing):
    existing = "" if pd.isna(existing) else str(existing)
    if note in existing:
        return existing
    return f"{existing}; {note}" if existing else note

nippon_log.loc[m, "Review_Notes"] = (
    nippon_log.loc[m, "Review_Notes"].apply(add_review_note)
)

nippon_log.to_csv(output_file, index=False)

print(nippon_log.loc[
    m,
    ["Report_Month", "Gold_Futures_Value_Lakh",
     "Gold_Futures_NAV_Pct", "Silver_Futures_Value_Lakh",
     "Silver_Futures_NAV_Pct", "Review_Notes"]
].to_string(index=False))

Report_Month  Gold_Futures_Value_Lakh  Gold_Futures_NAV_Pct  Silver_Futures_Value_Lakh  Silver_Futures_NAV_Pct                                                                                       Review_Notes
     2025-03                      NaN                   NaN                        NaN                     NaN Gold and silver futures/options not identified in matching rows; verify before treating as absent.


In [38]:
v = {
    "Gold_Futures_Value_Lakh": 3200.32,
    "Gold_Futures_NAV_Pct": 0.65,
    "Review_Notes": "Silver futures not identified in matching rows; verify before treating as absent."
}

m = nippon_log["Report_Month"] == "2025-02"

for col, value in v.items():
    nippon_log.loc[m, col] = value

nippon_log.to_csv(output_file, index=False)

print(nippon_log.loc[
    m,
    ["Report_Month", "Gold_Futures_Value_Lakh",
     "Gold_Futures_NAV_Pct", "Silver_Futures_Value_Lakh",
     "Silver_Futures_NAV_Pct", "Review_Notes"]
].to_string(index=False))

Report_Month  Gold_Futures_Value_Lakh  Gold_Futures_NAV_Pct  Silver_Futures_Value_Lakh  Silver_Futures_NAV_Pct                                                                      Review_Notes
     2025-02                  3200.32                  0.65                        NaN                     NaN Silver futures not identified in matching rows; verify before treating as absent.


In [39]:
v = {
    "Gold_Futures_Value_Lakh": 4440.58,
    "Gold_Futures_NAV_Pct": 0.89,
    "Silver_Futures_Value_Lakh": 1875.89,
    "Silver_Futures_NAV_Pct": 0.38
}

m = nippon_log["Report_Month"] == "2025-01"

for col, value in v.items():
    nippon_log.loc[m, col] = value

nippon_log.to_csv(output_file, index=False)

print(nippon_log.loc[
    m,
    ["Report_Month", "Gold_Futures_Value_Lakh",
     "Gold_Futures_NAV_Pct", "Silver_Futures_Value_Lakh",
     "Silver_Futures_NAV_Pct"]
].to_string(index=False))

Report_Month  Gold_Futures_Value_Lakh  Gold_Futures_NAV_Pct  Silver_Futures_Value_Lakh  Silver_Futures_NAV_Pct
     2025-01                  4440.58                  0.89                    1875.89                    0.38


In [40]:
v = {
    "Gold_Futures_Value_Lakh": 12049.44,
    "Gold_Futures_NAV_Pct": 2.48,
    "Silver_Futures_Value_Lakh": 2538.48,
    "Silver_Futures_NAV_Pct": 0.52
}

m = nippon_log["Report_Month"] == "2024-12"

for col, value in v.items():
    nippon_log.loc[m, col] = value

nippon_log.to_csv(output_file, index=False)

print(nippon_log.loc[
    m,
    ["Report_Month", "Gold_Futures_Value_Lakh",
     "Gold_Futures_NAV_Pct", "Silver_Futures_Value_Lakh",
     "Silver_Futures_NAV_Pct"]
].to_string(index=False))

Report_Month  Gold_Futures_Value_Lakh  Gold_Futures_NAV_Pct  Silver_Futures_Value_Lakh  Silver_Futures_NAV_Pct
     2024-12                 12049.44                  2.48                    2538.48                    0.52


In [41]:
# Month: [Gold futures value, Gold % NAV, Silver futures value, Silver % NAV]
futures_data = {
    "2024-11": [16891.03, 3.61, 1012.42, 0.22],
    "2024-10": [19293.78, 4.29, 3662.22, 0.81],
    "2024-09": [16105.14, 3.71, 2721.57, 0.63],
    "2024-08": [14895.09, 3.64, 1022.52, 0.25],
    "2024-07": [12259.28, 3.18, 7022.06, 1.82],
    "2024-06": [7372.95, 2.05, 7655.67, 2.13],
    "2024-05": [2945.19, 0.89, 5329.37, 1.61],
    "2024-04": [140.83, 0.05, 3856.59, 1.24],
    "2024-03": [None, None, 3219.56, 1.11],
    "2024-02": [None, None, 2587.43, 0.96],
    "2024-01": [None, None, 1475.29, 0.58],
}

columns = [
    "Gold_Futures_Value_Lakh",
    "Gold_Futures_NAV_Pct",
    "Silver_Futures_Value_Lakh",
    "Silver_Futures_NAV_Pct"
]

for month, values in futures_data.items():
    mask = nippon_log["Report_Month"] == month

    for column, value in zip(columns, values):
        nippon_log.loc[mask, column] = value

nippon_log.to_csv(output_file, index=False)

print("Updated months:", len(futures_data))
print(
    nippon_log[nippon_log["Report_Month"].isin(futures_data)]
    [["Report_Month"] + columns]
    .sort_values("Report_Month", ascending=False)
    .to_string(index=False)
)

Updated months: 11
Report_Month  Gold_Futures_Value_Lakh  Gold_Futures_NAV_Pct  Silver_Futures_Value_Lakh  Silver_Futures_NAV_Pct
     2024-11                 16891.03                  3.61                    1012.42                    0.22
     2024-10                 19293.78                  4.29                    3662.22                    0.81
     2024-09                 16105.14                  3.71                    2721.57                    0.63
     2024-08                 14895.09                  3.64                    1022.52                    0.25
     2024-07                 12259.28                  3.18                    7022.06                    1.82
     2024-06                  7372.95                  2.05                    7655.67                    2.13
     2024-05                  2945.19                  0.89                    5329.37                    1.61
     2024-04                   140.83                  0.05                    3856.59       

In [42]:
from pathlib import Path
import zipfile
import openpyxl
import xlrd

nippon_dir = PROJECT_DIR / "01_RAW_DATA" / "Nippon_India"

jan_files = [
    p for p in nippon_dir.iterdir()
    if "2024" in p.name and "JAN" in p.name.upper()
]

print("January 2024 files found:")
for p in jan_files:
    print(p.name)

January 2024 files found:
MONTHLY-PORTFOLIO-JAN-2024.xls


In [43]:
if not jan_files:
    print("No matching file found. Check the filename in the Nippon_India folder.")
else:
    source_file = jan_files[0]

    if zipfile.is_zipfile(source_file):
        with open(source_file, "rb") as stream:
            wb = openpyxl.load_workbook(
                stream, read_only=True, data_only=True
            )

        sheet_name = next(
            (s for s in wb.sheetnames if s.upper() == "MF"),
            None
        )

        if sheet_name is None:
            print("MF sheet not found. Available sheets:", wb.sheetnames)
        else:
            ws = wb[sheet_name]
            rows = list(ws.iter_rows(values_only=True))

            for i, row in enumerate(rows):
                text = " ".join(
                    str(x) for x in row if x is not None
                ).upper()

                if "SILVER" in text:
                    print(f"Row {i + 1}: {row}")

    else:
        wb = xlrd.open_workbook(str(source_file))
        sheet_name = next(
            (s for s in wb.sheet_names()
             if s.upper() == "MF"),
            None
        )

        if sheet_name is None:
            print("MF sheet not found. Available sheets:", wb.sheet_names())
        else:
            ws = wb.sheet_by_name(sheet_name)

            for i in range(ws.nrows):
                row = ws.row_values(i)
                text = " ".join(
                    str(x) for x in row if x != ""
                ).upper()

                if "SILVER" in text:
                    print(f"Row {i + 1}: {row}")

ValueError: seek of closed file

In [44]:
import zipfile
import openpyxl

source_file = jan_files[0]

if zipfile.is_zipfile(source_file):
    with open(source_file, "rb") as stream:
        wb = openpyxl.load_workbook(
            stream,
            read_only=True,
            data_only=True
        )

        sheet_name = next(
            (s for s in wb.sheetnames if s.upper() == "MF"),
            None
        )

        if sheet_name is None:
            print("MF sheet not found.")
            print("Available sheets:", wb.sheetnames)
        else:
            ws = wb[sheet_name]

            for i, row in enumerate(
                ws.iter_rows(values_only=True), start=1
            ):
                text = " ".join(
                    str(x) for x in row if x is not None
                ).upper()

                if "SILVER" in text:
                    print(f"Row {i}: {row}")

        wb.close()
else:
    print("This is not a ZIP-based workbook. Use the xlrd reader.")

Row 7: ('SILVERN24B72000C', None, 'SILVER 72000 Call February 2024 Option (1 kg)', None, 1110, 17.24, 0.0001, None, None, None, None)
Row 129: (None, 'Long', 'FUTCOM_SILVER_05/03/2024', 'Others', 1920, 1387.14, 0.0055, None, None, None, None)
Row 130: (None, 'Long', 'FUTCOM_SILVER_03/05/2024', 'Others', 120, 88.15, 0.0003, None, None, None, None)


In [45]:
# Inspect source rows for months with missing/unverified futures values

from pathlib import Path
import zipfile
import openpyxl
import xlrd
import pandas as pd

PROJECT_DIR = Path.cwd().parent
NIPPON_DIR = PROJECT_DIR / "01_RAW_DATA" / "Nippon_India"
OUTPUT_DIR = PROJECT_DIR / "02_CLEAN_DATA"
output_file = OUTPUT_DIR / "Nippon_India_Monthly_Holdings_Log.csv"

nippon_log = pd.read_csv(
    output_file,
    dtype={"Report_Month": str}
)

# Months where one or both futures values still need source verification
months_to_check = [
    "2024-01", "2024-02", "2024-03",
    "2025-02", "2025-03", "2025-08",
    "2026-02", "2026-03", "2026-05",
    "2026-06"
]

files = sorted(
    p for p in NIPPON_DIR.iterdir()
    if p.is_file() and p.suffix.lower() in [".xls", ".xlsx"]
)

def read_sheet_rows(file_path):
    """Read the MF sheet from either modern or legacy Excel format."""
    if zipfile.is_zipfile(file_path):
        with open(file_path, "rb") as stream:
            wb = openpyxl.load_workbook(
                stream,
                read_only=True,
                data_only=True
            )
            sheet_name = next(
                (s for s in wb.sheetnames if s.upper() == "MF"),
                None
            )

            if sheet_name is None:
                wb.close()
                return []

            rows = list(wb[sheet_name].iter_rows(values_only=True))
            wb.close()
            return rows

    # Genuine legacy .xls file
    wb = xlrd.open_workbook(str(file_path))
    sheet_name = next(
        (s for s in wb.sheet_names() if s.upper() == "MF"),
        None
    )

    if sheet_name is None:
        return []

    sheet = wb.sheet_by_name(sheet_name)
    return [
        tuple(sheet.cell_value(r, c) for c in range(sheet.ncols))
        for r in range(sheet.nrows)
    ]

# Read each workbook and identify the month from the existing log's
# report dates / source filename, then print futures-related rows.
for month in months_to_check:
    log_row = nippon_log.loc[nippon_log["Report_Month"] == month]

    if log_row.empty:
        print(f"\n{month}: month not found in the CSV.")
        continue

    report_date = str(log_row.iloc[0]["Report_Date"])[:10]
    month_year = month.split("-")
    year, month_num = int(month_year[0]), int(month_year[1])

    print(f"\n{'=' * 70}\nMONTH: {month} | Report date: {report_date}")

    # Search all source workbooks; report dates inside the workbook
    # will help confirm which file belongs to this month.
    for file_path in files:
        try:
            rows = read_sheet_rows(file_path)
        except Exception as exc:
            print(f"Could not read {file_path.name}: {exc}")
            continue

        if not rows:
            continue

        # Find the report month/year from the workbook's first rows.
        header_text = " ".join(
            str(value)
            for row in rows[:12]
            for value in row
            if value is not None
        ).upper()

        month_names = [
            "", "JANUARY", "FEBRUARY", "MARCH", "APRIL",
            "MAY", "JUNE", "JULY", "AUGUST", "SEPTEMBER",
            "OCTOBER", "NOVEMBER", "DECEMBER"
        ]

        # Match month and year using either numeric or written month.
        written_month = month_names[month_num]
        if str(year) not in header_text or written_month not in header_text:
            continue

        print(f"\nSource file: {file_path.name}")

        for row_num, row in enumerate(rows, start=1):
            row_text = " ".join(
                str(value) for value in row if value is not None
            ).upper()

            is_gold_silver = "GOLD" in row_text or "SILVER" in row_text
            is_derivative = (
                "FUTCOM" in row_text
                or "FUTURE" in row_text
                or "OPTION" in row_text
            )

            if is_gold_silver and is_derivative:
                print(f"Row {row_num}: {row}")


MONTH: 2024-01 | Report date: 2024-01-31

Source file: MONTHLY-PORTFOLIO-JAN-2024.xls
Row 7: ('SILVERN24B72000C', None, 'SILVER 72000 Call February 2024 Option (1 kg)', None, 1110, 17.24, 0.0001, None, None, None, None)
Row 129: (None, 'Long', 'FUTCOM_SILVER_05/03/2024', 'Others', 1920, 1387.14, 0.0055, None, None, None, None)
Row 130: (None, 'Long', 'FUTCOM_SILVER_03/05/2024', 'Others', 120, 88.15, 0.0003, None, None, None, None)

MONTH: 2024-02 | Report date: 2024-02-29

Source file: MONTHLY-PORTFOLIO-JAN-2024.xls
Row 7: ('SILVERN24B72000C', None, 'SILVER 72000 Call February 2024 Option (1 kg)', None, 1110, 17.24, 0.0001, None, None, None, None)
Row 129: (None, 'Long', 'FUTCOM_SILVER_05/03/2024', 'Others', 1920, 1387.14, 0.0055, None, None, None, None)
Row 130: (None, 'Long', 'FUTCOM_SILVER_03/05/2024', 'Others', 120, 88.15, 0.0003, None, None, None, None)

Source file: MONTHLY-PORTFOLIO-REPORT-FEB-24.xls
Row 125: (None, 'Long', 'FUTCOM_SILVER_03/05/2024', 'Others', 3630, 2587.43, 0

In [46]:
import pandas as pd
from pathlib import Path

PROJECT_DIR = Path.cwd().parent
OUTPUT_DIR = PROJECT_DIR / "02_CLEAN_DATA"
output_file = OUTPUT_DIR / "Nippon_India_Monthly_Holdings_Log.csv"

nippon_log = pd.read_csv(
    output_file,
    dtype={"Report_Month": str}
)

# Source-verified values: month -> (gold value, gold % NAV,
#                                 silver value, silver % NAV)
verified_updates = {
    "2024-01": (None, None, 1475.29, 0.58),
    "2024-02": (None, None, 2587.43, 0.96),
    "2024-03": (None, None, 3219.56, 1.11),
    "2025-02": (3200.32, 0.65, None, None),
    "2025-03": (4354.42, 0.82, None, None),
    "2026-02": (22694.56, 1.69, None, None),
    "2026-05": (26389.40, 1.70, None, None),
    "2026-06": (23375.08, 1.46, None, None),
}

for month, values in verified_updates.items():
    mask = nippon_log["Report_Month"] == month

    if mask.sum() != 1:
        print(f"Skipped {month}: expected exactly one row.")
        continue

    gold_value, gold_pct, silver_value, silver_pct = values

    # Update only values verified in the source.
    if gold_value is not None:
        nippon_log.loc[mask, "Gold_Futures_Value_Lakh"] = gold_value
        nippon_log.loc[mask, "Gold_Futures_NAV_Pct"] = gold_pct

    if silver_value is not None:
        nippon_log.loc[mask, "Silver_Futures_Value_Lakh"] = silver_value
        nippon_log.loc[mask, "Silver_Futures_NAV_Pct"] = silver_pct

nippon_log.to_csv(output_file, index=False)

print("Saved updated log:", output_file)
print("\nUpdated rows:")
print(
    nippon_log.loc[
        nippon_log["Report_Month"].isin(verified_updates.keys()),
        [
            "Report_Month",
            "Gold_Futures_Value_Lakh",
            "Gold_Futures_NAV_Pct",
            "Silver_Futures_Value_Lakh",
            "Silver_Futures_NAV_Pct",
        ],
    ].to_string(index=False)
)

Saved updated log: c:\Users\rocks\OneDrive\Desktop\AMC Project Data\02_CLEAN_DATA\Nippon_India_Monthly_Holdings_Log.csv

Updated rows:
Report_Month  Gold_Futures_Value_Lakh  Gold_Futures_NAV_Pct  Silver_Futures_Value_Lakh  Silver_Futures_NAV_Pct
     2024-01                      NaN                   NaN                    1475.29                    0.58
     2024-02                      NaN                   NaN                    2587.43                    0.96
     2024-03                      NaN                   NaN                    3219.56                    1.11
     2025-02                  3200.32                  0.65                        NaN                     NaN
     2025-03                  4354.42                  0.82                        NaN                     NaN
     2026-02                 22694.56                  1.69                        NaN                     NaN
     2026-05                 26389.40                  1.70                        NaN  

In [47]:
import pandas as pd
from pathlib import Path

PROJECT_DIR = Path.cwd().parent
output_file = (
    PROJECT_DIR
    / "02_CLEAN_DATA"
    / "Nippon_India_Monthly_Holdings_Log.csv"
)

nippon_log = pd.read_csv(
    output_file,
    dtype={"Report_Month": str}
)

expected_months = pd.period_range(
    "2024-01", "2026-09", freq="M"
).astype(str)

actual_months = nippon_log["Report_Month"].astype(str)

print("Total rows:", len(nippon_log))
print("Unique months:", actual_months.nunique())
print("Duplicate months:", actual_months.duplicated().sum())
print("Missing months:", sorted(set(expected_months) - set(actual_months)))
print("Unexpected months:", sorted(set(actual_months) - set(expected_months)))

print("\nETF values recorded:")
print("Gold ETF:", nippon_log["Gold_ETF_Value_Lakh"].notna().sum())
print("Silver ETF:", nippon_log["Silver_ETF_Value_Lakh"].notna().sum())

print("\nFutures values recorded:")
print("Gold futures:", nippon_log["Gold_Futures_Value_Lakh"].notna().sum())
print("Silver futures:", nippon_log["Silver_Futures_Value_Lakh"].notna().sum())

Total rows: 33
Unique months: 33
Duplicate months: 0
Missing months: []
Unexpected months: []

ETF values recorded:
Gold ETF: 33
Silver ETF: 27

Futures values recorded:
Gold futures: 28
Silver futures: 25


In [48]:
# Display months where gold or silver futures values are blank

futures_columns = [
    "Gold_Futures_Value_Lakh",
    "Silver_Futures_Value_Lakh"
]

for column in futures_columns:
    missing = nippon_log.loc[
        nippon_log[column].isna(),
        ["Report_Month", "Report_Date", column]
    ]

    print(f"\n{column}: {len(missing)} missing values")
    print(missing.to_string(index=False))


Gold_Futures_Value_Lakh: 5 missing values
Report_Month Report_Date  Gold_Futures_Value_Lakh
     2024-01  2024-01-31                      NaN
     2024-02  2024-02-29                      NaN
     2024-03  2024-03-31                      NaN
     2025-08  2025-08-31                      NaN
     2026-03  2026-03-31                      NaN

Silver_Futures_Value_Lakh: 8 missing values
Report_Month Report_Date  Silver_Futures_Value_Lakh
     2025-02  2025-02-28                        NaN
     2025-03  2025-03-31                        NaN
     2025-08  2025-08-31                        NaN
     2026-02  2026-02-28                        NaN
     2026-03  2026-03-31                        NaN
     2026-04  2026-04-30                        NaN
     2026-05  2026-05-31                        NaN
     2026-06  2026-06-30                        NaN


In [49]:
from pathlib import Path
import zipfile
import openpyxl
import xlrd

NIPPON_DIR = Path.cwd().parent / "01_RAW_DATA" / "Nippon_India"

files_to_check = [
    "MONTHLY-PORTFOLIO-JAN-2024.xls",
    "MONTHLY-PORTFOLIO-REPORT-FEB-24.xls",
    "NIMF-MONTHLY-PORTFOLIO-REPORT-March-24.xls",
    "NIMF-MONTHLY-PORTFOLIO-31-Aug-25.xls",
    "NIMF-MONTHLY-PORTFOLIO-31-Mar-26.xls",
]

for filename in files_to_check:
    source_file = NIPPON_DIR / filename
    print(f"\n{'=' * 70}\nFILE: {filename}")

    if not source_file.exists():
        print("File not found; check the exact filename.")
        continue

    if zipfile.is_zipfile(source_file):
        with open(source_file, "rb") as stream:
            wb = openpyxl.load_workbook(
                stream, read_only=True, data_only=True
            )
            sheet_name = next(
                (s for s in wb.sheetnames if s.upper() == "MF"), None
            )

            if sheet_name is None:
                print("MF sheet not found.")
            else:
                for row_num, row in enumerate(
                    wb[sheet_name].iter_rows(values_only=True), start=1
                ):
                    text = " ".join(
                        str(v) for v in row if v is not None
                    ).upper()
                    if "FUTCOM_GOLD" in text or "FUTCOM_SILVER" in text:
                        print(f"Row {row_num}: {row}")
            wb.close()
    else:
        wb = xlrd.open_workbook(str(source_file))
        sheet_name = next(
            (s for s in wb.sheet_names() if s.upper() == "MF"), None
        )

        if sheet_name is None:
            print("MF sheet not found.")
            continue

        ws = wb.sheet_by_name(sheet_name)
        for r in range(ws.nrows):
            row = tuple(
                ws.cell_value(r, c) for c in range(ws.ncols)
            )
            text = " ".join(
                str(v) for v in row if v is not None and v != ""
            ).upper()
            if "FUTCOM_GOLD" in text or "FUTCOM_SILVER" in text:
                print(f"Row {r + 1}: {row}")


FILE: MONTHLY-PORTFOLIO-JAN-2024.xls
Row 129: (None, 'Long', 'FUTCOM_SILVER_05/03/2024', 'Others', 1920, 1387.14, 0.0055, None, None, None, None)
Row 130: (None, 'Long', 'FUTCOM_SILVER_03/05/2024', 'Others', 120, 88.15, 0.0003, None, None, None, None)

FILE: MONTHLY-PORTFOLIO-REPORT-FEB-24.xls
Row 125: (None, 'Long', 'FUTCOM_SILVER_03/05/2024', 'Others', 3630, 2587.43, 0.0096, None)

FILE: NIMF-MONTHLY-PORTFOLIO-REPORT-March-24.xls
Row 130: ('', 'Long', 'FUTCOM_SILVER_03/05/2024', 'Others', 4290.0, 3219.56, 0.0111, '')

FILE: NIMF-MONTHLY-PORTFOLIO-31-Aug-25.xls

FILE: NIMF-MONTHLY-PORTFOLIO-31-Mar-26.xls


In [50]:
from pathlib import Path
import zipfile
import openpyxl
import xlrd

NIPPON_DIR = Path.cwd().parent / "01_RAW_DATA" / "Nippon_India"

files_to_check = [
    "NIMF-MONTHLY-PORTFOLIO-28-Feb-25.xls",
    "NIMF-MONTHLY-PORTFOLIO-31-Mar-25.xls",
    "NIMF-MONTHLY-PORTFOLIO-28-Feb-26.xls",
    "NIMF-MONTHLY-PORTFOLIO-31-May-26.xls",
    "NIMF-MONTHLY-PORTFOLIO-30-Jun-26.xls",
]

for filename in files_to_check:
    source_file = NIPPON_DIR / filename
    print(f"\n{'=' * 70}\nFILE: {filename}")

    if not source_file.exists():
        print("File not found; check the exact filename.")
        continue

    if zipfile.is_zipfile(source_file):
        with open(source_file, "rb") as stream:
            wb = openpyxl.load_workbook(
                stream, read_only=True, data_only=True
            )
            sheet_name = next(
                (s for s in wb.sheetnames if s.upper() == "MF"), None
            )

            if sheet_name:
                for row_num, row in enumerate(
                    wb[sheet_name].iter_rows(values_only=True), start=1
                ):
                    text = " ".join(
                        str(v) for v in row if v is not None
                    ).upper()
                    if "SILVER" in text:
                        print(f"Row {row_num}: {row}")
            else:
                print("MF sheet not found.")

            wb.close()
    else:
        wb = xlrd.open_workbook(str(source_file))
        sheet_name = next(
            (s for s in wb.sheet_names() if s.upper() == "MF"), None
        )

        if sheet_name:
            ws = wb.sheet_by_name(sheet_name)
            for r in range(ws.nrows):
                row = tuple(
                    ws.cell_value(r, c) for c in range(ws.ncols)
                )
                text = " ".join(
                    str(v) for v in row if v is not None and v != ""
                ).upper()
                if "SILVER" in text:
                    print(f"Row {r + 1}: {row}")
        else:
            print("MF sheet not found.")


FILE: NIMF-MONTHLY-PORTFOLIO-28-Feb-25.xls
Row 1: ('RLMF806', 'Nippon India Multi Asset Allocation Fund (An open ended scheme investing in Equity, Debt \nand Exchange Traded Commodity Derivatives and \nGold ETF & Silver ETF)', '', '', '', '', '', 'Index')
Row 141: ('149758', 'INF204KC1402', 'Nippon India Silver ETF', '', 16869201.0, 15259.88, 0.0308, '')
Row 165: ('', '', '', '', '', 'BENCHMARK NAME - 50% OF BSE 500 TRI, 20% OF MSCI WORLD INDEX TRI, 15% OF CRISIL SHORT TERM BOND INDEX 10% OF DOMESTIC PRICES OF GOLD & 5% OF DOMESTIC PRICES OF SILVER', '', '')

FILE: NIMF-MONTHLY-PORTFOLIO-31-Mar-25.xls
Row 1: ('RLMF806', 'Nippon India Multi Asset Allocation Fund (An open ended scheme investing in Equity, Debt \nand Exchange Traded Commodity Derivatives and \nGold ETF & Silver ETF)', None, None, None, None, None, 'Index')
Row 142: ('149758', 'INF204KC1402', 'Nippon India Silver ETF', None, 16869201, 16464.34, 0.0309, None)
Row 166: (None, None, None, None, None, 'BENCHMARK NAME - 50% OF

In [51]:
from pathlib import Path
import xlrd

source_file = (
    Path.cwd().parent
    / "01_RAW_DATA"
    / "Nippon_India"
    / "NIMF-MONTHLY-PORTFOLIO-28-Feb-25.xls"
)

wb = xlrd.open_workbook(str(source_file))
ws = wb.sheet_by_name("MF")

for r in range(129, min(170, ws.nrows)):
    row = tuple(ws.cell_value(r, c) for c in range(ws.ncols))
    print(f"Row {r + 1}: {row}")

Row 130: ('', '', '(b) Privately placed / Unlisted', '', '', 'NIL', 'NIL', 'NIL')
Row 131: ('', '', 'Subtotal', '', '', 'NIL', 'NIL', 'NIL')
Row 132: ('', '', '(c) Securitised Debt', '', '', '', '', '')
Row 133: ('', '', 'Subtotal', '', '', 'NIL', 'NIL', '')
Row 134: ('', '', 'Total', '', '', 78279.95, 0.1582, '')
Row 135: ('', '', 'Money Market Instruments', '', '', '', '', '')
Row 136: ('', '', 'Triparty Repo/ Reverse Repo Instrument', '', '', '', '', '')
Row 137: ('', '', 'Triparty Repo', '', '', 15737.9, 0.0318, '')
Row 138: ('', '', 'Total', '', '', 15737.9, 0.0318, '')
Row 139: ('', '', 'Exchange Traded Funds', '', '', '', '', '')
Row 140: ('BMGB51ME', 'INF204KB17I5', 'Nippon India ETF Gold Bees', '', 86261865.0, 61332.19, 0.1239, '')
Row 141: ('149758', 'INF204KC1402', 'Nippon India Silver ETF', '', 16869201.0, 15259.88, 0.0308, '')
Row 142: ('', '', 'Total', '', '', 76592.07, 0.1547, '')
Row 143: ('', '', 'OTHERS', '', '', '', '', '')
Row 144: ('FNOMGN', '', 'Cash Margin - Deri

In [53]:
from pathlib import Path
import zipfile
import openpyxl
import xlrd

NIPPON_DIR = Path.cwd().parent / "01_RAW_DATA" / "Nippon_India"

files_to_check = [
    "NIMF-MONTHLY-PORTFOLIO-31-Mar-25.xls",
    "NIMF-MONTHLY-PORTFOLIO-31-Aug-25.xls",
]

for filename in files_to_check:
    source_file = NIPPON_DIR / filename
    print(f"\n{'=' * 65}\nFILE: {filename}")

    if not source_file.exists():
        print("File not found. Check the filename.")
        continue

    # Detect the workbook's actual format.
    if zipfile.is_zipfile(source_file):
        with open(source_file, "rb") as stream:
            wb = openpyxl.load_workbook(
                stream,
                read_only=True,
                data_only=True
            )

            sheet_name = next(
                (s for s in wb.sheetnames if s.upper() == "MF"),
                None
            )

            if sheet_name is None:
                print("MF sheet not found.")
                wb.close()
                continue

            rows = list(
                wb[sheet_name].iter_rows(values_only=True)
            )
            wb.close()

    else:
        wb = xlrd.open_workbook(str(source_file))
        sheet_name = next(
            (s for s in wb.sheet_names() if s.upper() == "MF"),
            None
        )

        if sheet_name is None:
            print("MF sheet not found.")
            continue

        ws = wb.sheet_by_name(sheet_name)
        rows = [
            tuple(ws.cell_value(r, c) for c in range(ws.ncols))
            for r in range(ws.nrows)
        ]

    # Print the commodity futures section.
    in_futures_section = False
    found_section = False

    for row_num, row in enumerate(rows, start=1):
        row_text = " ".join(
            str(value)
            for value in row
            if value is not None and value != ""
        ).upper()

        if "DETAILS OF COMMODITY FUTURE" in row_text:
            in_futures_section = True
            found_section = True
            print(f"Row {row_num}: {row}")
            continue

        if in_futures_section:
            if "DISCLOSURE OF YIELD TO CALL" in row_text:
                print(f"Row {row_num}: {row}")
                break

            if any(value not in ("", None) for value in row):
                print(f"Row {row_num}: {row}")

    if not found_section:
        print("Commodity futures section heading not found.")


FILE: NIMF-MONTHLY-PORTFOLIO-31-Mar-25.xls
Row 156: (None, 'Details of Commodity Future / Index Future', None, None, None, None, None, None)
Row 157: (None, 'Long/Short', 'Name of the Instrument', 'Industry', 'Quantity.', 'Market/Fair Value (Rs. in Lacs).', ' % to NAV.', None)
Row 158: (None, 'Long', 'FUTCOM_GOLD_05/06/2025', 'Others', 4800, 4354.42, 0.0082, None)
Row 161: (None, 'Disclosure of Yield to Call', None, None, None, None, None, None)

FILE: NIMF-MONTHLY-PORTFOLIO-31-Aug-25.xls
Row 171: (None, 'Details of Commodity Future / Index Future', None, None, None, None, None, None)
Row 172: (None, 'Long/Short', 'Name of the Instrument', 'Industry', 'Quantity.', 'Market/Fair Value (Rs. in Lacs).', ' % to NAV.', None)
Row 173: (None, 'Long', 'FUTCOM_CRUDEOIL_19/09/2025', 'Others', 20000, 1130.8, 0.0016, None)
Row 176: (None, 'Disclosure of Yield to Call', None, None, None, None, None, None)


In [54]:
from pathlib import Path
import zipfile
import openpyxl
import xlrd

source_file = (
    Path.cwd().parent
    / "01_RAW_DATA"
    / "Nippon_India"
    / "NIMF-MONTHLY-PORTFOLIO-31-Mar-26.xls"
)

if zipfile.is_zipfile(source_file):
    with open(source_file, "rb") as stream:
        wb = openpyxl.load_workbook(
            stream, read_only=True, data_only=True
        )
        sheet_name = next(
            (s for s in wb.sheetnames if s.upper() == "MF"),
            None
        )
        if sheet_name:
            rows = list(
                wb[sheet_name].iter_rows(values_only=True)
            )
        else:
            rows = []
        wb.close()
else:
    wb = xlrd.open_workbook(str(source_file))
    sheet_name = next(
        (s for s in wb.sheet_names() if s.upper() == "MF"),
        None
    )
    if sheet_name:
        ws = wb.sheet_by_name(sheet_name)
        rows = [
            tuple(ws.cell_value(r, c) for c in range(ws.ncols))
            for r in range(ws.nrows)
        ]
    else:
        rows = []

in_section = False
found = False

for row_num, row in enumerate(rows, start=1):
    row_text = " ".join(
        str(v) for v in row if v is not None and v != ""
    ).upper()

    if "DETAILS OF COMMODITY FUTURE" in row_text:
        in_section = True
        found = True
        print(f"Row {row_num}: {row}")
        continue

    if in_section:
        if "DISCLOSURE OF YIELD TO CALL" in row_text:
            print(f"Row {row_num}: {row}")
            break
        if any(v not in ("", None) for v in row):
            print(f"Row {row_num}: {row}")

if not found:
    print("Commodity futures section heading not found.")

Row 208: ('', 'Details of Commodity Future / Index Future', '', '', '', '', '', '')
Row 209: ('', 'Long/Short', 'Name of the Instrument', 'Industry', 'Quantity.', 'Market/Fair Value (Rs. in Lacs).', ' % to NAV.', '')
Row 210: ('', 'Long', 'FUTCOM_CRUDEOIL_18/05/2026', 'Others', 100000.0, 8981.0, 0.0068, '')
Row 211: ('', 'Long', 'FUTCOM_CRUDEOIL_20/04/2026', 'Others', 25000.0, 2391.75, 0.0018, '')
Row 213: ('', 'Disclosure of Yield to Call', '', '', '', '', '', '')


In [55]:
# Update the March 2026 review note
month = "2026-03"

mask = nippon_log["Report_Month"].astype(str).eq(month)

if mask.sum() != 1:
    print(f"Expected exactly one row for {month}, found {mask.sum()}. No changes made.")
else:
    note = (
        "Commodity futures section inspected; crude oil futures reported, "
        "but gold and silver futures not identified."
    )

    existing_note = nippon_log.loc[mask, "Review_Notes"].iloc[0]
    existing_note = "" if pd.isna(existing_note) else str(existing_note)

    if note not in existing_note:
        updated_note = f"{existing_note}; {note}" if existing_note.strip() else note
        nippon_log.loc[mask, "Review_Notes"] = updated_note

    nippon_log.to_csv(output_file, index=False)

    print("March 2026 review note updated and CSV saved.")
    print(nippon_log.loc[mask, ["Report_Month", "Review_Notes"]].to_string(index=False))

March 2026 review note updated and CSV saved.
Report_Month                                                                                                                                                                                             Review_Notes
     2026-03 Gold and silver futures not identified in matching rows; verify before treating as absent.; Commodity futures section inspected; crude oil futures reported, but gold and silver futures not identified.


In [56]:

import zipfile
import openpyxl
import xlrd
from pathlib import Path

months_to_check = {
    "2025-08": "NIMF-MONTHLY-PORTFOLIO-31-Aug-25.xls",
    "2026-04": "NIMF-MONTHLY-PORTFOLIO-30-Apr-26.xls",
    "2026-05": "NIMF-MONTHLY-PORTFOLIO-31-May-26.xls",
    "2026-06": "NIMF-MONTHLY-PORTFOLIO-30-Jun-26.xls",
}

nippon_dir = RAW_DATA_DIR / "Nippon_India"

for month, filename in months_to_check.items():
    source_file = nippon_dir / filename
    print(f"\n{'=' * 70}\nMONTH: {month}\nFILE: {filename}")

    if not source_file.exists():
        print("FILE NOT FOUND — check the exact filename.")
        continue

    if zipfile.is_zipfile(source_file):
        with open(source_file, "rb") as stream:
            wb = openpyxl.load_workbook(
                stream, read_only=True, data_only=True
            )
            sheet_name = next(
                (s for s in wb.sheetnames if s.upper() == "MF"), None
            )
            rows = (
                list(wb[sheet_name].iter_rows(values_only=True))
                if sheet_name else []
            )
            wb.close()
    else:
        wb = xlrd.open_workbook(str(source_file))
        sheet_name = next(
            (s for s in wb.sheet_names() if s.upper() == "MF"), None
        )
        if sheet_name:
            sheet = wb.sheet_by_name(sheet_name)
            rows = [
                tuple(sheet.cell_value(r, c) for c in range(sheet.ncols))
                for r in range(sheet.nrows)
            ]
        else:
            rows = []

    if not rows:
        print("MF worksheet not found or empty.")
        continue

    start = next(
        (i for i, row in enumerate(rows)
         if any("Details of Commodity Future / Index Future" in str(v)
                for v in row if v is not None)),
        None
    )

    if start is None:
        print("Commodity futures section heading not found.")
        continue

    for i in range(start, len(rows)):
        row_text = " | ".join(str(v) for v in rows[i] if v is not None)
        print(f"Row {i + 1}: {row_text}")
        if i > start and "Disclosure of Yield to Call" in row_text:
            break



MONTH: 2025-08
FILE: NIMF-MONTHLY-PORTFOLIO-31-Aug-25.xls
Row 171: Details of Commodity Future / Index Future
Row 172: Long/Short | Name of the Instrument | Industry | Quantity. | Market/Fair Value (Rs. in Lacs). |  % to NAV.
Row 173: Long | FUTCOM_CRUDEOIL_19/09/2025 | Others | 20000 | 1130.8 | 0.0016
Row 174: 
Row 175: 
Row 176: Disclosure of Yield to Call

MONTH: 2026-04
FILE: NIMF-MONTHLY-PORTFOLIO-30-Apr-26.xls
FILE NOT FOUND — check the exact filename.

MONTH: 2026-05
FILE: NIMF-MONTHLY-PORTFOLIO-31-May-26.xls
Row 222: Details of Commodity Future / Index Future
Row 223: Long/Short | Name of the Instrument | Industry | Quantity. | Market/Fair Value (Rs. in Lacs). |  % to NAV.
Row 224: Long | FUTCOM_GOLD_05/08/2026 | Others | 16400 | 26389.4 | 0.017
Row 225: Long | FUTCOM_CRUDEOIL_18/06/2026 | Others | 210000 | 17390.1 | 0.0112
Row 226: 
Row 227: Disclosure of Yield to Call

MONTH: 2026-06
FILE: NIMF-MONTHLY-PORTFOLIO-30-Jun-26.xls
Row 220: Details of Commodity Future / Index Futu

In [57]:

from pathlib import Path

nippon_dir = RAW_DATA_DIR / "Nippon_India"

for file in sorted(nippon_dir.iterdir()):
    if file.is_file() and (
        "apr" in file.name.lower()
        or "april" in file.name.lower()
    ):
        print(file.name)


NIMF-MONTHLY-PORTFOLIO-30-April-25.xls
NIMF-MONTHLY-PORTFOLIO-30-April-26.xls
NIMF-MONTHLY-PORTFOLIO-APRIL-2024.xls


In [58]:

import zipfile
import openpyxl
import xlrd

source_file = RAW_DATA_DIR / "Nippon_India" / "NIMF-MONTHLY-PORTFOLIO-30-April-26.xls"

if zipfile.is_zipfile(source_file):
    with open(source_file, "rb") as stream:
        wb = openpyxl.load_workbook(
            stream, read_only=True, data_only=True
        )
        sheet_name = next(
            (s for s in wb.sheetnames if s.upper() == "MF"), None
        )
        rows = (
            list(wb[sheet_name].iter_rows(values_only=True))
            if sheet_name else []
        )
        wb.close()
else:
    wb = xlrd.open_workbook(str(source_file))
    sheet_name = next(
        (s for s in wb.sheet_names() if s.upper() == "MF"), None
    )
    if sheet_name:
        sheet = wb.sheet_by_name(sheet_name)
        rows = [
            tuple(sheet.cell_value(r, c) for c in range(sheet.ncols))
            for r in range(sheet.nrows)
        ]
    else:
        rows = []

start = next(
    (i for i, row in enumerate(rows)
     if any("Details of Commodity Future / Index Future" in str(v)
            for v in row if v is not None)),
    None
)

if start is None:
    print("Commodity futures section not found.")
else:
    for i in range(start, len(rows)):
        row_text = " | ".join(str(v) for v in rows[i] if v is not None)
        print(f"Row {i + 1}: {row_text}")
        if i > start and "Disclosure of Yield to Call" in row_text:
            break


Row 217: Details of Commodity Future / Index Future
Row 218: Long/Short | Name of the Instrument | Industry | Quantity. | Market/Fair Value (Rs. in Lacs). |  % to NAV.
Row 219: Long | FUTCOM_GOLD_05/06/2026 | Others | 24000 | 36266.64 | 0.0246
Row 220: Long | FUTCOM_CRUDEOIL_18/06/2026 | Others | 170000 | 15927.3 | 0.0108
Row 221: 
Row 222: 
Row 223: Disclosure of Yield to Call


In [59]:

import pandas as pd

updates = {
    "2025-08": (
        "Commodity futures section inspected; crude oil futures reported, "
        "but gold and silver futures not identified."
    ),
    "2026-04": (
        "Commodity futures section inspected; gold futures of Rs. 36266.64 "
        "lakh (2.46% of NAV) and crude oil futures reported; "
        "silver futures not identified."
    ),
    "2026-05": (
        "Commodity futures section inspected; gold futures of Rs. 26389.40 "
        "lakh (1.70% of NAV) and crude oil futures reported; "
        "silver futures not identified."
    ),
    "2026-06": (
        "Commodity futures section inspected; gold futures of Rs. 23375.08 "
        "lakh (1.46% of NAV) and crude oil futures reported; "
        "silver futures not identified."
    ),
}

for month, note in updates.items():
    mask = nippon_log["Report_Month"].astype(str).eq(month)

    if mask.sum() != 1:
        raise ValueError(
            f"Expected exactly one row for {month}; found {mask.sum()}. "
            "No CSV saved. Check the month entries."
        )

    existing = nippon_log.loc[mask, "Review_Notes"].iloc[0]
    existing = "" if pd.isna(existing) else str(existing)

    if note not in existing:
        nippon_log.loc[mask, "Review_Notes"] = (
            f"{existing}; {note}" if existing.strip() else note
        )

nippon_log.to_csv(output_file, index=False)

print("Review notes updated and CSV saved.")
print(
    nippon_log.loc[
        nippon_log["Report_Month"].astype(str).isin(updates),
        ["Report_Month", "Review_Notes"]
    ].to_string(index=False)
)


Review notes updated and CSV saved.
Report_Month                                                                                                                                                                                                                             Review_Notes
     2025-08                                 Gold and silver futures not identified in matching rows; verify before treating as absent.; Commodity futures section inspected; crude oil futures reported, but gold and silver futures not identified.
     2026-04 Silver futures not identified in keyword search; verify before treating as absent.; Commodity futures section inspected; gold futures of Rs. 36266.64 lakh (2.46% of NAV) and crude oil futures reported; silver futures not identified.
     2026-05 Silver futures not identified in keyword search; verify before treating as absent.; Commodity futures section inspected; gold futures of Rs. 26389.40 lakh (1.70% of NAV) and crude oil futures reported; silver futu

In [60]:

import pandas as pd

# Reload the saved CSV so we validate the actual file
nippon_log = pd.read_csv(output_file)

expected_months = pd.period_range(
    start="2024-01", end="2026-09", freq="M"
).astype(str)

actual_months = nippon_log["Report_Month"].astype(str)

print("Total rows:", len(nippon_log))
print("Unique months:", actual_months.nunique())
print("Duplicate month rows:", actual_months.duplicated().sum())
print("Missing months:", sorted(set(expected_months) - set(actual_months)))
print("Unexpected months:", sorted(set(actual_months) - set(expected_months)))

print("\nMissing values by column:")
columns_to_check = [
    "Gold_ETF_Value_Lakh",
    "Silver_ETF_Value_Lakh",
    "Gold_Futures_Value_Lakh",
    "Silver_Futures_Value_Lakh",
]

for col in columns_to_check:
    print(f"{col}: {nippon_log[col].isna().sum()}")

print("\nGold futures source-value check:")
for month, expected_value in {
    "2026-04": 36266.64,
    "2026-05": 26389.40,
    "2026-06": 23375.08,
}.items():
    row = nippon_log.loc[actual_months.eq(month), col if False else "Gold_Futures_Value_Lakh"]
    print(f"{month}: recorded={row.tolist()}, expected={expected_value}")

print("\nValidation complete. No changes made.")


Total rows: 33
Unique months: 33
Duplicate month rows: 0
Missing months: []
Unexpected months: []

Missing values by column:
Gold_ETF_Value_Lakh: 0
Silver_ETF_Value_Lakh: 6
Gold_Futures_Value_Lakh: 5
Silver_Futures_Value_Lakh: 8

Gold futures source-value check:
2026-04: recorded=[36266.64], expected=36266.64
2026-05: recorded=[26389.4], expected=26389.4
2026-06: recorded=[23375.08], expected=23375.08

Validation complete. No changes made.


In [61]:

missing_silver_months = nippon_log.loc[
    nippon_log["Silver_ETF_Value_Lakh"].isna(),
    ["Report_Month", "Report_Date", "Silver_ETF_Name", "Review_Notes"]
]

print(missing_silver_months.to_string(index=False))


Report_Month Report_Date Silver_ETF_Name                                                                         Review_Notes
     2024-01  2024-01-31             NaN No direct silver ETF identified in keyword search; verify before treating as absent.
     2024-02  2024-02-29             NaN No direct silver ETF identified in keyword search; verify before treating as absent.
     2024-03  2024-03-31             NaN No direct silver ETF identified in keyword search; verify before treating as absent.
     2024-04  2024-04-30             NaN No direct silver ETF identified in keyword search; verify before treating as absent.
     2024-05  2024-05-31             NaN No direct silver ETF identified in keyword search; verify before treating as absent.
     2024-06  2024-06-30             NaN No direct silver ETF identified in keyword search; verify before treating as absent.


In [62]:

import zipfile
import openpyxl
import xlrd
from pathlib import Path

nippon_dir = RAW_DATA_DIR / "Nippon_India"

files_to_check = {
    "2024-01": "MONTHLY-PORTFOLIO-JAN-2024.xls",
    "2024-02": "MONTHLY-PORTFOLIO-REPORT-FEB-24.xls",
    "2024-03": "NIMF-MONTHLY-PORTFOLIO-REPORT-March-24.xls",
    "2024-04": "NIMF-MONTHLY-PORTFOLIO-APRIL-2024.xls",
    "2024-05": "MONTHLY-PORTFOLIO-REPORT-MAY-24.xls",
    "2024-06": "NIMF-MONTHLY-PORTFOLIO-REPORT-June-24.xls",
}

for month, filename in files_to_check.items():
    source_file = nippon_dir / filename
    print(f"\n{'=' * 65}\n{month}: {filename}")

    if not source_file.exists():
        print("FILE NOT FOUND — verify the filename.")
        continue

    if zipfile.is_zipfile(source_file):
        with open(source_file, "rb") as stream:
            wb = openpyxl.load_workbook(
                stream, read_only=True, data_only=True
            )
            sheet_name = next(
                (s for s in wb.sheetnames if s.upper() == "MF"), None
            )
            rows = (
                list(wb[sheet_name].iter_rows(values_only=True))
                if sheet_name else []
            )
            wb.close()
    else:
        wb = xlrd.open_workbook(str(source_file))
        sheet_name = next(
            (s for s in wb.sheet_names() if s.upper() == "MF"), None
        )
        if sheet_name:
            sheet = wb.sheet_by_name(sheet_name)
            rows = [
                tuple(sheet.cell_value(r, c) for c in range(sheet.ncols))
                for r in range(sheet.nrows)
            ]
        else:
            rows = []

    if not rows:
        print("MF worksheet not found or empty.")
        continue

    matches = []
    for i, row in enumerate(rows):
        text = " | ".join(str(v) for v in row if v is not None)
        if "silver" in text.lower() or "benchmark" in text.lower():
            matches.append((i + 1, text))

    if matches:
        for row_num, text in matches:
            print(f"Row {row_num}: {text}")
    else:
        print("No silver or benchmark keyword matches found.")



2024-01: MONTHLY-PORTFOLIO-JAN-2024.xls
Row 7: SILVERN24B72000C | SILVER 72000 Call February 2024 Option (1 kg) | 1110 | 17.24 | 0.0001
Row 129: Long | FUTCOM_SILVER_05/03/2024 | Others | 1920 | 1387.14 | 0.0055
Row 130: Long | FUTCOM_SILVER_03/05/2024 | Others | 120 | 88.15 | 0.0003
Row 132: BENCHMARK NAME - CRISIL SHORT TERM BOND INDEX, DOMESTIC PRICE OF GOLD, MSCI WORLD INDEX TRI, S&P BSE 500 TRI
Row 133: SCHEME RISK-O-METER | BENCHMARK RISK-O-METER

2024-02: MONTHLY-PORTFOLIO-REPORT-FEB-24.xls
Row 125: Long | FUTCOM_SILVER_03/05/2024 | Others | 3630 | 2587.43 | 0.0096
Row 128: BENCHMARK NAME - CRISIL SHORT TERM BOND INDEX, DOMESTIC PRICE OF GOLD, MSCI WORLD INDEX TRI, S&P BSE 500 TRI
Row 129: SCHEME RISK-O-METER | BENCHMARK RISK-O-METER

2024-03: NIMF-MONTHLY-PORTFOLIO-REPORT-March-24.xls
Row 7: SILVERN24D72000P |  | SILVER 72000 Put April 2024 Option (1 kg) |  | 1530.0 | 5.68 | $0.00% | 
Row 130:  | Long | FUTCOM_SILVER_03/05/2024 | Others | 4290.0 | 3219.56 | 0.0111 | 
Row 133: 

In [63]:

nippon_dir = RAW_DATA_DIR / "Nippon_India"

for file in sorted(nippon_dir.iterdir()):
    name = file.name.lower()
    if file.is_file() and (
        "may" in name or "june" in name or "jun-" in name
    ):
        print(file.name)


NIMF-MONTHLY-PORTFOLIO-30-Jun-26.xls
NIMF-MONTHLY-PORTFOLIO-30-June-25.xls
NIMF-MONTHLY-PORTFOLIO-31-May-25.xls
NIMF-MONTHLY-PORTFOLIO-31-May-26.xls
NIMF-MONTHLY-PORTFOLIO-June-24.xls
NIMF-MONTHLY-PORTFOLIO-May-24.xls


In [64]:

files_to_check = {
    "2024-05": "NIMF-MONTHLY-PORTFOLIO-May-24.xls",
    "2024-06": "NIMF-MONTHLY-PORTFOLIO-June-24.xls",
}

for month, filename in files_to_check.items():
    source_file = nippon_dir / filename
    print(f"\n{'=' * 65}\n{month}: {filename}")

    if zipfile.is_zipfile(source_file):
        with open(source_file, "rb") as stream:
            wb = openpyxl.load_workbook(
                stream, read_only=True, data_only=True
            )
            sheet_name = next(
                (s for s in wb.sheetnames if s.upper() == "MF"), None
            )
            rows = (
                list(wb[sheet_name].iter_rows(values_only=True))
                if sheet_name else []
            )
            wb.close()
    else:
        wb = xlrd.open_workbook(str(source_file))
        sheet_name = next(
            (s for s in wb.sheet_names() if s.upper() == "MF"), None
        )
        if sheet_name:
            sheet = wb.sheet_by_name(sheet_name)
            rows = [
                tuple(sheet.cell_value(r, c) for c in range(sheet.ncols))
                for r in range(sheet.nrows)
            ]
        else:
            rows = []

    matches = []
    for i, row in enumerate(rows):
        text = " | ".join(str(v) for v in row if v is not None)
        if "silver" in text.lower() or "benchmark" in text.lower():
            matches.append((i + 1, text))

    if matches:
        for row_num, text in matches:
            print(f"Row {row_num}: {text}")
    else:
        print("No silver or benchmark keyword matches found.")



2024-05: NIMF-MONTHLY-PORTFOLIO-May-24.xls
Row 7: SILVER 85000 Put June 2024 Option (1 kg) | 4980 | 19.67 | 0.0001
Row 140: Long | FUTCOM_SILVER_05/07/2024 | Others | 5820 | 5329.37 | 0.0161
Row 145: BENCHMARK NAME - 50% OF BSE 500 TRI, 20% OF MSCI WORLD INDEX TRI, 15% OF CRISIL SHORT TERM BOND INDEX & 15% OF DOMESTIC PRICES OF GOLD
Row 146: SCHEME RISK-O-METER | BENCHMARK RISK-O-METER

2024-06: NIMF-MONTHLY-PORTFOLIO-June-24.xls
Row 137: Long | FUTCOM_SILVER_05/09/2024 | Others | 8550 | 7655.67 | 0.0213
Row 141: BENCHMARK NAME - 50% OF BSE 500 TRI, 20% OF MSCI WORLD INDEX TRI, 15% OF CRISIL SHORT TERM BOND INDEX & 15% OF DOMESTIC PRICES OF GOLD
Row 142: SCHEME RISK-O-METER | BENCHMARK RISK-O-METER


In [65]:

import pandas as pd

# Add columns only if they do not already exist
new_columns = {
    "Silver_Options_Value_Lakh": pd.NA,
    "Silver_Options_NAV_Pct": pd.NA,
    "Silver_Options_Review": "",
}

for column, default_value in new_columns.items():
    if column not in nippon_log.columns:
        nippon_log[column] = default_value

# Source-reported option entries for January–June 2024
option_updates = {
    "2024-01": {
        "value": 17.24,
        "nav_pct": 0.01,
        "review": "Silver call option identified; source % to NAV 0.01%.",
    },
    "2024-02": {
        "value": pd.NA,
        "nav_pct": pd.NA,
        "review": "No silver option identified in matching rows; blank is not treated as zero.",
    },
    "2024-03": {
        "value": 5.68,
        "nav_pct": pd.NA,
        "review": "Silver put option identified; source % to NAV is malformed ($0.00%); percentage left blank.",
    },
    "2024-04": {
        "value": 65.15,
        "nav_pct": 0.02,
        "review": "REVIEW REQUIRED: instrument code contains SILVER but description says CRUDEOIL; value ₹65.15 lakh and source % to NAV 0.02% recorded as an unverified candidate, not confirmed silver exposure.",
    },
    "2024-05": {
        "value": 19.67,
        "nav_pct": 0.01,
        "review": "Silver put option identified; source % to NAV 0.01%.",
    },
    "2024-06": {
        "value": pd.NA,
        "nav_pct": pd.NA,
        "review": "No silver option identified in matching rows; blank is not treated as zero.",
    },
}

# Verify each month occurs exactly once before making changes
for month in option_updates:
    count = nippon_log["Report_Month"].astype(str).eq(month).sum()
    if count != 1:
        raise ValueError(
            f"Expected exactly one row for {month}; found {count}. "
            "No CSV saved."
        )

for month, data in option_updates.items():
    mask = nippon_log["Report_Month"].astype(str).eq(month)
    nippon_log.loc[mask, "Silver_Options_Value_Lakh"] = data["value"]
    nippon_log.loc[mask, "Silver_Options_NAV_Pct"] = data["nav_pct"]
    nippon_log.loc[mask, "Silver_Options_Review"] = data["review"]

# Save the updated log
nippon_log.to_csv(output_file, index=False)

# Reload and verify the saved file
nippon_log = pd.read_csv(output_file)

print("Saved updated Nippon India log.")
print(
    nippon_log.loc[
        nippon_log["Report_Month"].astype(str).isin(option_updates),
        [
            "Report_Month",
            "Silver_Futures_Value_Lakh",
            "Silver_Options_Value_Lakh",
            "Silver_Options_NAV_Pct",
            "Silver_Options_Review",
        ],
    ].to_string(index=False)
)
print("\nTotal rows:", len(nippon_log))
print("Unique months:", nippon_log["Report_Month"].nunique())


Saved updated Nippon India log.
Report_Month  Silver_Futures_Value_Lakh  Silver_Options_Value_Lakh  Silver_Options_NAV_Pct                                                                                                                                                                           Silver_Options_Review
     2024-01                    1475.29                      17.24                    0.01                                                                                                                                           Silver call option identified; source % to NAV 0.01%.
     2024-02                    2587.43                        NaN                     NaN                                                                                                                     No silver option identified in matching rows; blank is not treated as zero.
     2024-03                    3219.56                       5.68                     NaN                             

In [66]:

nippon_log = pd.read_csv(output_file)

print("Total rows:", len(nippon_log))
print("Unique months:", nippon_log["Report_Month"].nunique())
print(
    "Duplicate months:",
    nippon_log["Report_Month"].astype(str).duplicated().sum()
)

print("\nRequired silver columns:")
for col in [
    "Silver_ETF_Value_Lakh",
    "Silver_Futures_Value_Lakh",
    "Silver_Options_Value_Lakh",
    "Silver_Options_NAV_Pct",
    "Silver_Options_Review",
]:
    print(f"{col}: {'Present' if col in nippon_log.columns else 'MISSING'}")

print("\nMissing futures values:")
for col in ["Gold_Futures_Value_Lakh", "Silver_Futures_Value_Lakh"]:
    missing_months = nippon_log.loc[
        nippon_log[col].isna(), "Report_Month"
    ].astype(str).tolist()
    print(f"{col}: {len(missing_months)} missing — {missing_months}")

print("\nOptions values recorded:", nippon_log["Silver_Options_Value_Lakh"].notna().sum())
print("Validation complete; no changes made.")


Total rows: 33
Unique months: 33
Duplicate months: 0

Required silver columns:
Silver_ETF_Value_Lakh: Present
Silver_Futures_Value_Lakh: Present
Silver_Options_Value_Lakh: Present
Silver_Options_NAV_Pct: Present
Silver_Options_Review: Present

Missing futures values:
Gold_Futures_Value_Lakh: 5 missing — ['2024-01', '2024-02', '2024-03', '2025-08', '2026-03']
Silver_Futures_Value_Lakh: 8 missing — ['2025-02', '2025-03', '2025-08', '2026-02', '2026-03', '2026-04', '2026-05', '2026-06']

Options values recorded: 4
Validation complete; no changes made.


In [69]:
from zipfile import is_zipfile
from pathlib import Path
import openpyxl
import xlrd

file_path = RAW_DATA_DIR / "Nippon_India" / "NIMF-MONTHLY-PORTFOLIO-APRIL-2024.xls"

if not file_path.exists():
    print("File not found:", file_path)

elif is_zipfile(file_path):
    # Read the ZIP-based workbook through an open binary stream.
    with open(file_path, "rb") as file_stream:
        workbook = openpyxl.load_workbook(
            file_stream,
            read_only=False,
            data_only=True
        )

        print("Worksheets:", workbook.sheetnames)
        sheet = workbook["MF"]

        matches = 0
        for row_num, row in enumerate(sheet.iter_rows(values_only=True), start=1):
            row_text = " | ".join(
                str(value) for value in row if value is not None
            )

            if "SILVERN24D72000P" in row_text or "CRUDEOIL 6800 Put" in row_text:
                print(f"\nSource row {row_num}:")
                print(row)
                matches += 1

        print("\nMatching rows:", matches)
        workbook.close()

else:
    # Read a genuine legacy XLS workbook.
    workbook = xlrd.open_workbook(str(file_path))
    sheet = workbook.sheet_by_name("MF")

    matches = 0
    for row_num in range(sheet.nrows):
        row = sheet.row_values(row_num)
        row_text = " | ".join(str(value) for value in row)

        if "SILVERN24D72000P" in row_text or "CRUDEOIL 6800 Put" in row_text:
            print(f"\nSource row {row_num + 1}:")
            print(row)
            matches += 1

    print("\nMatching rows:", matches)

Worksheets: ['Index', 'GF', 'GS', 'IP', 'IF', 'ST', 'BF', 'MI', 'PS', 'CP', 'PH', 'ME', 'NE', 'NI', 'EO', 'LF', 'FR', 'SD', 'SE', 'SH', 'LQ', 'TS', 'LE', 'IV', 'LP', 'IM', 'DC', 'UB', 'K5', '9B', 'EA', 'QP', 'GL', 'NF', 'SC', 'SF', 'AF', 'GD', 'CE', 'CF', 'RD', 'CB', 'JE', 'SX', 'RW', 'RG', 'BP', 'ES', 'NX', 'UE', 'LT', 'NL', 'NB', 'JZ', 'PU', 'GB', 'LB', 'BB', 'SB', 'HS', 'IB', 'CC', 'ON', 'NM', 'RJ', 'G6', 'SN', 'NZ', 'MF', 'NS', 'NC', 'PP', 'AL', 'NP', 'NV', 'NY', 'NG', 'NH', 'LC', 'HG', 'ET', 'NO', 'SI', 'SO', 'HK', 'CN', 'NN', 'W5', 'ND', 'NJ', 'NK', 'NQ', 'W8', 'NU', 'NW', 'U7', 'U8', 'IT', 'IX', 'BX']

Source row 7:
('SILVERN24D72000P', None, 'CRUDEOIL 6800 Put May 2024 Option (1 kg)', None, 50000, 65.15, 0.0002, None, None, None, None)

Matching rows: 1


In [70]:
import pandas as pd

output_file = OUTPUT_DIR / "Nippon_India_Monthly_Holdings_Log.csv"

nippon_log = pd.read_csv(output_file)

mask = nippon_log["Report_Month"].astype(str).eq("2024-04")

nippon_log.loc[mask, "Silver_Options_Review"] = (
    "REVIEW REQUIRED: source row has instrument code SILVERN24D72000P "
    "but description CRUDEOIL 6800 Put May 2024 Option (1 kg). "
    "Value ₹65.15 lakh and reported NAV percentage 0.02% retained "
    "as an unverified candidate; exclude from confirmed silver-option "
    "exposure until independently verified."
)

nippon_log.to_csv(output_file, index=False)

print("April 2024 review note updated.")
print(
    nippon_log.loc[
        mask,
        [
            "Report_Month",
            "Silver_Options_Value_Lakh",
            "Silver_Options_NAV_Pct",
            "Silver_Options_Review"
        ]
    ].to_string(index=False)
)

April 2024 review note updated.
Report_Month  Silver_Options_Value_Lakh  Silver_Options_NAV_Pct                                                                                                                                                                                                                                                                           Silver_Options_Review
     2024-04                      65.15                    0.02 REVIEW REQUIRED: source row has instrument code SILVERN24D72000P but description CRUDEOIL 6800 Put May 2024 Option (1 kg). Value ₹65.15 lakh and reported NAV percentage 0.02% retained as an unverified candidate; exclude from confirmed silver-option exposure until independently verified.


In [71]:
# Identify months where gold or silver futures values are missing.
gold_missing = nippon_log["Gold_Futures_Value_Lakh"].isna()
silver_missing = nippon_log["Silver_Futures_Value_Lakh"].isna()

for idx in nippon_log.index:
    additions = []

    if gold_missing.loc[idx]:
        additions.append(
            "Gold futures value blank: no matching gold futures identified "
            "in the inspected source section; blank does not mean zero."
        )

    if silver_missing.loc[idx]:
        additions.append(
            "Silver futures value blank: no matching silver futures identified "
            "in the inspected source section; blank does not mean zero."
        )

    if additions:
        existing = nippon_log.at[idx, "Review_Notes"]

        if pd.isna(existing):
            existing = ""

        existing = str(existing).strip()

        for note in additions:
            if note not in existing:
                existing = f"{existing} {note}".strip()

        nippon_log.at[idx, "Review_Notes"] = existing

# Save the updated log.
nippon_log.to_csv(output_file, index=False)

print("Missing-futures review notes standardized.")
print(
    nippon_log.loc[
        gold_missing | silver_missing,
        [
            "Report_Month",
            "Gold_Futures_Value_Lakh",
            "Silver_Futures_Value_Lakh",
            "Review_Notes"
        ]
    ].to_string(index=False)
)

Missing-futures review notes standardized.
Report_Month  Gold_Futures_Value_Lakh  Silver_Futures_Value_Lakh                                                                                                                                                                                                                                                                                                                                                                                                                                                   Review_Notes
     2024-01                      NaN                    1475.29                                                                                                                                                                                                                                                  No direct silver ETF identified in keyword search; verify before treating as absent. Gold futures value blank: no matching gold futures ide

In [72]:
import pandas as pd

output_file = OUTPUT_DIR / "Nippon_India_Monthly_Holdings_Log.csv"
check_df = pd.read_csv(output_file)

expected_months = pd.period_range("2024-01", "2026-09", freq="M").astype(str)
actual_months = check_df["Report_Month"].astype(str)

required_columns = [
    "Silver_ETF_Value_Lakh",
    "Silver_Futures_Value_Lakh",
    "Silver_Options_Value_Lakh",
    "Silver_Options_NAV_Pct",
    "Silver_Options_Review",
]

print("ROWS:", len(check_df))
print("UNIQUE MONTHS:", actual_months.nunique())
print("DUPLICATE MONTHS:", actual_months.duplicated().sum())
print("MISSING MONTHS:", sorted(set(expected_months) - set(actual_months)))
print("UNEXPECTED MONTHS:", sorted(set(actual_months) - set(expected_months)))

print("\nREQUIRED COLUMNS:")
for col in required_columns:
    print(f"{col}: {'Present' if col in check_df.columns else 'MISSING'}")

print("\nRECORDED VALUES:")
for col in [
    "Gold_ETF_Value_Lakh",
    "Silver_ETF_Value_Lakh",
    "Gold_Futures_Value_Lakh",
    "Silver_Futures_Value_Lakh",
    "Silver_Options_Value_Lakh",
]:
    print(f"{col}: {check_df[col].notna().sum()} of {len(check_df)}")

april = check_df.loc[actual_months.eq("2024-04")]

print("\nAPRIL 2024 OPTION REVIEW:")
if not april.empty:
    print(april[["Silver_Options_Value_Lakh", "Silver_Options_NAV_Pct",
                 "Silver_Options_Review"]].to_string(index=False))
else:
    print("April 2024 record not found.")

print("\nVALIDATION COMPLETE — no changes made.")

ROWS: 33
UNIQUE MONTHS: 33
DUPLICATE MONTHS: 0
MISSING MONTHS: []
UNEXPECTED MONTHS: []

REQUIRED COLUMNS:
Silver_ETF_Value_Lakh: Present
Silver_Futures_Value_Lakh: Present
Silver_Options_Value_Lakh: Present
Silver_Options_NAV_Pct: Present
Silver_Options_Review: Present

RECORDED VALUES:
Gold_ETF_Value_Lakh: 33 of 33
Silver_ETF_Value_Lakh: 27 of 33
Gold_Futures_Value_Lakh: 28 of 33
Silver_Futures_Value_Lakh: 25 of 33
Silver_Options_Value_Lakh: 4 of 33

APRIL 2024 OPTION REVIEW:
 Silver_Options_Value_Lakh  Silver_Options_NAV_Pct                                                                                                                                                                                                                                                                           Silver_Options_Review
                     65.15                    0.02 REVIEW REQUIRED: source row has instrument code SILVERN24D72000P but description CRUDEOIL 6800 Put May 2024 Option (1 kg). Valu

In [73]:
from pathlib import Path
from zipfile import is_zipfile
import openpyxl
import xlrd

nippon_dir = RAW_DATA_DIR / "Nippon_India"

files_to_check = [
    ("2024-01", "MONTHLY-PORTFOLIO-JAN-2024.xls"),
    ("2024-02", "MONTHLY-PORTFOLIO-REPORT-FEB-24.xls"),
    ("2024-03", "NIMF-MONTHLY-PORTFOLIO-REPORT-March-24.xls"),
    ("2024-04", "NIMF-MONTHLY-PORTFOLIO-APRIL-2024.xls"),
    ("2024-05", "NIMF-MONTHLY-PORTFOLIO-May-24.xls"),
    ("2024-06", "NIMF-MONTHLY-PORTFOLIO-June-24.xls"),
]

keywords = ["silver", "benchmark", "etf", "exchange traded"]

for month, filename in files_to_check:
    file_path = nippon_dir / filename

    print("\n" + "=" * 90)
    print(f"MONTH: {month} | FILE: {filename}")

    if not file_path.exists():
        print("FILE NOT FOUND — verify the filename.")
        continue

    if is_zipfile(file_path):
        with open(file_path, "rb") as file_stream:
            workbook = openpyxl.load_workbook(
                file_stream, read_only=False, data_only=True
            )
            sheet = workbook["MF"]

            for row_num, row in enumerate(
                sheet.iter_rows(values_only=True), start=1
            ):
                row_text = " | ".join(
                    str(value) for value in row if value is not None
                )

                if any(word in row_text.lower() for word in keywords):
                    print(f"Row {row_num}: {row}")

            workbook.close()

    else:
        workbook = xlrd.open_workbook(str(file_path))
        sheet = workbook.sheet_by_name("MF")

        for row_num in range(sheet.nrows):
            row = sheet.row_values(row_num)
            row_text = " | ".join(str(value) for value in row)

            if any(word in row_text.lower() for word in keywords):
                print(f"Row {row_num + 1}: {row}")


MONTH: 2024-01 | FILE: MONTHLY-PORTFOLIO-JAN-2024.xls
Row 1: ('RLMF806', 'Nippon India Multi Asset Fund (An open ended scheme investing in Equity, Debt and Exchange Traded Commodity Derivatives and Gold ETF)', None, None, None, None, None, 'Index', None, None, None)
Row 7: ('SILVERN24B72000C', None, 'SILVER 72000 Call February 2024 Option (1 kg)', None, 1110, 17.24, 0.0001, None, None, None, None)
Row 77: (None, None, 'Listed Foreign Securities/Overseas ETFs', None, None, None, None, None, None, None, None)
Row 78: ('14740454USD', 'US4642863926', 'iShares MSCI World ETF', None, 380148, 42384.23, 0.1667, None, None, None, None)
Row 107: (None, None, 'Exchange Traded Funds', None, None, None, None, None, None, None, None)
Row 108: ('BMGB51ME', 'INF204KB17I5', 'Nippon India ETF Gold Bees', None, 62360000, 33269.06, 0.1308, None, None, None, None)
Row 129: (None, 'Long', 'FUTCOM_SILVER_05/03/2024', 'Others', 1920, 1387.14, 0.0055, None, None, None, None)
Row 130: (None, 'Long', 'FUTCOM_SI

In [74]:
from pathlib import Path
from zipfile import is_zipfile
import openpyxl
import xlrd

# Months where one or both futures values are missing
missing_months = [
    "2024-01", "2024-02", "2024-03",
    "2025-02", "2025-03", "2025-08",
    "2026-02", "2026-03", "2026-04",
    "2026-05", "2026-06"
]

# Find each month's source file using the month in its filename or report date.
# Print all rows containing futures-related instrument codes.
nippon_dir = RAW_DATA_DIR / "Nippon_India"

for month in missing_months:
    print("\n" + "=" * 80)
    print("MONTH:", month)

    matching_files = [
        p for p in nippon_dir.iterdir()
        if p.is_file() and p.suffix.lower() in [".xls", ".xlsx"]
        and month.split("-")[0] in p.name
    ]

    print("Potential files:")
    for p in matching_files:
        print(" ", p.name)


MONTH: 2024-01
Potential files:
  MONTHLY-PORTFOLIO-JAN-2024.xls
  NIMF-MONTHLY-PORTFOLIO-30-Nov-2024.xls
  NIMF-MONTHLY-PORTFOLIO-APRIL-2024.xls

MONTH: 2024-02
Potential files:
  MONTHLY-PORTFOLIO-JAN-2024.xls
  NIMF-MONTHLY-PORTFOLIO-30-Nov-2024.xls
  NIMF-MONTHLY-PORTFOLIO-APRIL-2024.xls

MONTH: 2024-03
Potential files:
  MONTHLY-PORTFOLIO-JAN-2024.xls
  NIMF-MONTHLY-PORTFOLIO-30-Nov-2024.xls
  NIMF-MONTHLY-PORTFOLIO-APRIL-2024.xls

MONTH: 2025-02
Potential files:

MONTH: 2025-03
Potential files:

MONTH: 2025-08
Potential files:

MONTH: 2026-02
Potential files:

MONTH: 2026-03
Potential files:

MONTH: 2026-04
Potential files:

MONTH: 2026-05
Potential files:

MONTH: 2026-06
Potential files:


In [75]:
# Display all source workbook filenames in the Nippon India folder.
nippon_dir = RAW_DATA_DIR / "Nippon_India"

source_files = sorted(
    [
        p for p in nippon_dir.iterdir()
        if p.is_file() and p.suffix.lower() in [".xls", ".xlsx"]
    ],
    key=lambda p: p.name.lower()
)

print("Total source workbooks:", len(source_files))

for file_path in source_files:
    print(file_path.name)

Total source workbooks: 33
MONTHLY-PORTFOLIO-JAN-2024.xls
MONTHLY-PORTFOLIO-REPORT-FEB-24.xls
NIMF-MONTHLY-PORTFOLIO-28-Feb-25.xls
NIMF-MONTHLY-PORTFOLIO-28-Feb-26.xls
NIMF-MONTHLY-PORTFOLIO-30-April-25.xls
NIMF-MONTHLY-PORTFOLIO-30-April-26.xls
NIMF-MONTHLY-PORTFOLIO-30-Jun-26.xls
NIMF-MONTHLY-PORTFOLIO-30-June-25.xls
NIMF-MONTHLY-PORTFOLIO-30-Nov-2024.xls
NIMF-MONTHLY-PORTFOLIO-30-Sep-24.xls
NIMF-MONTHLY-PORTFOLIO-30-Sep-25.xls
NIMF-MONTHLY-PORTFOLIO-30-Sep-26.xls
NIMF-MONTHLY-PORTFOLIO-31-Aug-25.xls
NIMF-MONTHLY-PORTFOLIO-31-Aug-26.xls
NIMF-MONTHLY-PORTFOLIO-31-Dec-24.xls
NIMF-MONTHLY-PORTFOLIO-31-Dec-25.xls
NIMF-MONTHLY-PORTFOLIO-31-Jan-26.xls
NIMF-MONTHLY-PORTFOLIO-31-July-24.xls
NIMF-MONTHLY-PORTFOLIO-31-July-25.xls
NIMF-MONTHLY-PORTFOLIO-31-July-26.xls
NIMF-MONTHLY-PORTFOLIO-31-Mar-25.xls
NIMF-MONTHLY-PORTFOLIO-31-Mar-26.xls
NIMF-MONTHLY-PORTFOLIO-31-May-25.xls
NIMF-MONTHLY-PORTFOLIO-31-May-26.xls
NIMF-MONTHLY-PORTFOLIO-31-Oct-24.xls
NIMF-MONTHLY-PORTFOLIO-31-Oct-25.xls
NIMF-MON

In [76]:
from zipfile import is_zipfile
import openpyxl
import xlrd

files_by_month = {
    "2024-01": "MONTHLY-PORTFOLIO-JAN-2024.xls",
    "2024-02": "MONTHLY-PORTFOLIO-REPORT-FEB-24.xls",
    "2024-03": "NIMF-MONTHLY-PORTFOLIO-REPORT-March-24.xls",
    "2025-02": "NIMF-MONTHLY-PORTFOLIO-28-Feb-25.xls",
    "2025-03": "NIMF-MONTHLY-PORTFOLIO-31-Mar-25.xls",
    "2025-08": "NIMF-MONTHLY-PORTFOLIO-31-Aug-25.xls",
    "2026-02": "NIMF-MONTHLY-PORTFOLIO-28-Feb-26.xls",
    "2026-03": "NIMF-MONTHLY-PORTFOLIO-31-Mar-26.xls",
    "2026-04": "NIMF-MONTHLY-PORTFOLIO-30-April-26.xls",
    "2026-05": "NIMF-MONTHLY-PORTFOLIO-31-May-26.xls",
    "2026-06": "NIMF-MONTHLY-PORTFOLIO-30-Jun-26.xls",
}

for month, filename in files_by_month.items():
    file_path = nippon_dir / filename
    print("\n" + "=" * 90)
    print(f"EXPECTED MONTH: {month} | FILE: {filename}")

    if is_zipfile(file_path):
        with open(file_path, "rb") as stream:
            wb = openpyxl.load_workbook(
                stream, read_only=False, data_only=True
            )
            ws = wb["MF"]
            rows = list(ws.iter_rows(values_only=True))

        print("First 3 rows:")
        for row in rows[:3]:
            print(row)

        print("Matching commodity rows:")
        for row_num, row in enumerate(rows, start=1):
            row_text = " | ".join(
                str(value) for value in row if value is not None
            ).upper()

            if any(term in row_text for term in ["FUTCOM", "GOLD", "SILVER"]):
                print(f"Row {row_num}: {row}")

        wb.close()

    else:
        wb = xlrd.open_workbook(str(file_path))
        ws = wb.sheet_by_name("MF")

        print("First 3 rows:")
        for row_num in range(min(3, ws.nrows)):
            print(ws.row_values(row_num))

        print("Matching commodity rows:")
        for row_num in range(ws.nrows):
            row = ws.row_values(row_num)
            row_text = " | ".join(str(value) for value in row).upper()

            if any(term in row_text for term in ["FUTCOM", "GOLD", "SILVER"]):
                print(f"Row {row_num + 1}: {row}")


EXPECTED MONTH: 2024-01 | FILE: MONTHLY-PORTFOLIO-JAN-2024.xls
First 3 rows:
('RLMF806', 'Nippon India Multi Asset Fund (An open ended scheme investing in Equity, Debt and Exchange Traded Commodity Derivatives and Gold ETF)', None, None, None, None, None, 'Index', None, None, None)
(None, 'Monthly Portfolio Statement as on January 31,2024', None, None, None, None, None, None, None, None, None)
(None, None, None, None, None, None, None, None, None, None, None)
Matching commodity rows:
Row 1: ('RLMF806', 'Nippon India Multi Asset Fund (An open ended scheme investing in Equity, Debt and Exchange Traded Commodity Derivatives and Gold ETF)', None, None, None, None, None, 'Index', None, None, None)
Row 7: ('SILVERN24B72000C', None, 'SILVER 72000 Call February 2024 Option (1 kg)', None, 1110, 17.24, 0.0001, None, None, None, None)
Row 108: ('BMGB51ME', 'INF204KB17I5', 'Nippon India ETF Gold Bees', None, 62360000, 33269.06, 0.1308, None, None, None, None)
Row 129: (None, 'Long', 'FUTCOM_SILVE

In [77]:
# Inspect the section surrounding the separate GOLD entries.
files_to_inspect = {
    "2026-04": ("NIMF-MONTHLY-PORTFOLIO-30-April-26.xls", 245, 265),
    "2026-05": ("NIMF-MONTHLY-PORTFOLIO-31-May-26.xls", 250, 270),
    "2026-06": ("NIMF-MONTHLY-PORTFOLIO-30-Jun-26.xls", 250, 270),
}

for month, (filename, start_row, end_row) in files_to_inspect.items():
    file_path = nippon_dir / filename

    print("\n" + "=" * 80)
    print(f"{month}: rows {start_row}–{end_row}")

    if is_zipfile(file_path):
        with open(file_path, "rb") as stream:
            wb = openpyxl.load_workbook(
                stream, read_only=False, data_only=True
            )
            ws = wb["MF"]

            for row_num in range(start_row, min(end_row, ws.max_row) + 1):
                print(f"Row {row_num}: {tuple(ws.iter_rows(
                    min_row=row_num, max_row=row_num, values_only=True
                ))[0]}")

            wb.close()
    else:
        wb = xlrd.open_workbook(str(file_path))
        ws = wb.sheet_by_name("MF")

        for row_num in range(start_row - 1, min(end_row, ws.nrows)):
            print(f"Row {row_num + 1}: {ws.row_values(row_num)}")


2026-04: rows 245–265
Row 245: ('RLMF806', '(7)  During the period, the portfolio turnover ratio is 0.73 times.', None, None, None, None, None, None)
Row 246: (None, '(8) The details of repo transactions of the scheme in corporate debt securities : Nil', None, None, None, None, None, None)
Row 247: (None, None, None, None, None, None, None, None)
Row 248: ('Derivatives disclosure Table', None, None, None, None, None, None, None)
Row 249: ('A.', 'Hedging positions through futures as at 30 April 2026: Nil.', None, None, None, None, None, None)
Row 250: (None, 'Total % of existing assets hedged through futures: Nil.', None, None, None, None, None, None)
Row 251: (None, 'For the Month ended 30 April 2026 following details specified for hedging transactions through futures which have been squared off/expired: Nil.', None, None, None, None, None, None)
Row 252: (None, None, None, None, None, None, None, None)
Row 253: ('B. ', 'Other than hedging position through futures as at 30 April 2026:

In [78]:
# Inspect August 2025 derivatives disclosures.
# This is read-only: it does not change your source file or CSV.

filename = "NIMF-MONTHLY-PORTFOLIO-31-Aug-25.xls"
file_path = nippon_dir / filename

if is_zipfile(file_path):
    with open(file_path, "rb") as stream:
        wb = openpyxl.load_workbook(
            stream, read_only=False, data_only=True
        )
        ws = wb["MF"]

        rows = list(ws.iter_rows(values_only=True))
        section_row = next(
            (
                i for i, row in enumerate(rows)
                if any(
                    isinstance(value, str)
                    and "Derivatives disclosure Table" in value
                    for value in row
                )
            ),
            None
        )

        if section_row is None:
            print("Derivatives disclosure Table not found.")
        else:
            for i in range(section_row, min(section_row + 46, len(rows))):
                print(f"Row {i + 1}: {rows[i]}")

        wb.close()

else:
    wb = xlrd.open_workbook(str(file_path))
    ws = wb.sheet_by_name("MF")

    section_row = next(
        (
            i for i in range(ws.nrows)
            if any(
                isinstance(value, str)
                and "Derivatives disclosure Table" in value
                for value in ws.row_values(i)
            )
        ),
        None
    )

    if section_row is None:
        print("Derivatives disclosure Table not found.")
    else:
        for i in range(section_row, min(section_row + 46, ws.nrows)):
            print(f"Row {i + 1}: {ws.row_values(i)}")

Derivatives disclosure Table not found.


In [79]:
# Search August 2025 MF worksheet for derivatives-related rows.

filename = "NIMF-MONTHLY-PORTFOLIO-31-Aug-25.xls"
file_path = nippon_dir / filename

keywords = ["futures", "underlying", "margin", "GOLD", "SILVER"]

if is_zipfile(file_path):
    with open(file_path, "rb") as stream:
        wb = openpyxl.load_workbook(
            stream, read_only=False, data_only=True
        )
        ws = wb["MF"]

        for row_num, row in enumerate(
            ws.iter_rows(values_only=True), start=1
        ):
            row_text = " | ".join(
                str(value) for value in row if value is not None
            )

            if any(word.lower() in row_text.lower() for word in keywords):
                print(f"Row {row_num}: {row_text}")

        wb.close()

else:
    wb = xlrd.open_workbook(str(file_path))
    ws = wb.sheet_by_name("MF")

    for row_idx in range(ws.nrows):
        row = ws.row_values(row_idx)
        row_text = " | ".join(
            str(value) for value in row if value not in (None, "")
        )

        if any(word.lower() in row_text.lower() for word in keywords):
            print(f"Row {row_idx + 1}: {row_text}")

Row 1: RLMF806 | Nippon India Multi Asset Allocation Fund (An open ended scheme investing in Equity, Debt 
and Exchange Traded Commodity Derivatives and 
Gold ETF & Silver ETF) | Index
Row 144: LIGS22 | INE1U3R15011 | Liquid Gold Series** | CRISIL AAA(SO) | 119.999999 | 2900.57 | 0.0042 | 0.092423
Row 154: BMGB51ME | INF204KB17I5 | Nippon India ETF Gold Bees | 91266971 | 77887.23 | 0.1119
Row 155: 149758 | INF204KC1402 | Nippon India Silver ETF | 18869201 | 21431.64 | 0.0308
Row 158: FNOMGN | Cash Margin - Derivatives | 4275 | 0.0061
Row 159: CCILMGN | Cash Margin - CCIL | 12.51 | $0.00%
Row 181: BENCHMARK NAME - 50% OF BSE 500 TRI, 20% OF MSCI WORLD INDEX TRI, 15% OF CRISIL SHORT TERM BOND INDEX 10% OF DOMESTIC PRICES OF GOLD & 5% OF DOMESTIC PRICES OF SILVER


In [80]:
# Search February 2025 for futures and derivatives disclosure information.
# This cell only reads the source workbook.

filename = "NIMF-MONTHLY-PORTFOLIO-28-Feb-25.xls"
file_path = nippon_dir / filename

keywords = [
    "futures",
    "underlying",
    "margin",
    "derivatives disclosure",
    "GOLD",
    "SILVER"
]

if is_zipfile(file_path):
    with open(file_path, "rb") as stream:
        wb = openpyxl.load_workbook(
            stream, read_only=False, data_only=True
        )
        ws = wb["MF"]

        for row_num, row in enumerate(
            ws.iter_rows(values_only=True), start=1
        ):
            row_text = " | ".join(
                str(value) for value in row if value is not None
            )

            if any(word.lower() in row_text.lower() for word in keywords):
                print(f"Row {row_num}: {row_text}")

        wb.close()

else:
    wb = xlrd.open_workbook(str(file_path))
    ws = wb.sheet_by_name("MF")

    for row_idx in range(ws.nrows):
        row = ws.row_values(row_idx)
        row_text = " | ".join(
            str(value) for value in row if value not in (None, "")
        )

        if any(word.lower() in row_text.lower() for word in keywords):
            print(f"Row {row_idx + 1}: {row_text}")

Row 1: RLMF806 | Nippon India Multi Asset Allocation Fund (An open ended scheme investing in Equity, Debt 
and Exchange Traded Commodity Derivatives and 
Gold ETF & Silver ETF) | Index
Row 140: BMGB51ME | INF204KB17I5 | Nippon India ETF Gold Bees | 86261865.0 | 61332.19 | 0.1239
Row 141: 149758 | INF204KC1402 | Nippon India Silver ETF | 16869201.0 | 15259.88 | 0.0308
Row 144: FNOMGN | Cash Margin - Derivatives | 1575.0 | 0.0032
Row 145: CCILMGN | Cash Margin - CCIL | 5.9 | $0.00%
Row 157: Long | FUTCOM_GOLD_04/04/2025 | Others | 3800.0 | 3200.32 | 0.0065
Row 165: BENCHMARK NAME - 50% OF BSE 500 TRI, 20% OF MSCI WORLD INDEX TRI, 15% OF CRISIL SHORT TERM BOND INDEX 10% OF DOMESTIC PRICES OF GOLD & 5% OF DOMESTIC PRICES OF SILVER


In [81]:
# Inspect March 2025 derivatives-related rows.

filename = "NIMF-MONTHLY-PORTFOLIO-31-Mar-25.xls"
file_path = nippon_dir / filename

keywords = [
    "futures",
    "underlying",
    "margin",
    "derivatives disclosure",
    "GOLD",
    "SILVER"
]

if is_zipfile(file_path):
    with open(file_path, "rb") as stream:
        wb = openpyxl.load_workbook(
            stream, read_only=False, data_only=True
        )
        ws = wb["MF"]

        for row_num, row in enumerate(
            ws.iter_rows(values_only=True), start=1
        ):
            row_text = " | ".join(
                str(value) for value in row if value is not None
            )

            if any(word.lower() in row_text.lower() for word in keywords):
                print(f"Row {row_num}: {row_text}")

        wb.close()

else:
    wb = xlrd.open_workbook(str(file_path))
    ws = wb.sheet_by_name("MF")

    for row_idx in range(ws.nrows):
        row = ws.row_values(row_idx)
        row_text = " | ".join(
            str(value) for value in row if value not in (None, "")
        )

        if any(word.lower() in row_text.lower() for word in keywords):
            print(f"Row {row_idx + 1}: {row_text}")

Row 1: RLMF806 | Nippon India Multi Asset Allocation Fund (An open ended scheme investing in Equity, Debt 
and Exchange Traded Commodity Derivatives and 
Gold ETF & Silver ETF) | Index
Row 133: LIGS22 | INE1U3R15011 | Liquid Gold Series** | CRISIL AAA(SO) | 120 | 4212.85 | 0.0079 | 0.098948
Row 141: BMGB51ME | INF204KB17I5 | Nippon India ETF Gold Bees | 86261865 | 63971.8 | 0.12
Row 142: 149758 | INF204KC1402 | Nippon India Silver ETF | 16869201 | 16464.34 | 0.0309
Row 145: FNOMGN | Cash Margin - Derivatives | 2094.819 | 0.003930141886185531
Row 146: CCILMGN | Cash Margin - CCIL | 6.63 | $0.00%
Row 158: Long | FUTCOM_GOLD_05/06/2025 | Others | 4800 | 4354.42 | 0.0082
Row 166: BENCHMARK NAME - 50% OF BSE 500 TRI, 20% OF MSCI WORLD INDEX TRI, 15% OF CRISIL SHORT TERM BOND INDEX 10% OF DOMESTIC PRICES OF GOLD & 5% OF DOMESTIC PRICES OF SILVER


In [82]:
# Search February 2026 for futures and derivatives-related entries.

filename = "NIMF-MONTHLY-PORTFOLIO-28-Feb-26.xls"
file_path = nippon_dir / filename

keywords = [
    "futures",
    "underlying",
    "margin",
    "derivatives disclosure",
    "GOLD",
    "SILVER"
]

if is_zipfile(file_path):
    with open(file_path, "rb") as stream:
        wb = openpyxl.load_workbook(
            stream, read_only=False, data_only=True
        )
        ws = wb["MF"]

        for row_num, row in enumerate(
            ws.iter_rows(values_only=True), start=1
        ):
            row_text = " | ".join(
                str(value) for value in row if value is not None
            )

            if any(word.lower() in row_text.lower() for word in keywords):
                print(f"Row {row_num}: {row_text}")

        wb.close()

else:
    wb = xlrd.open_workbook(str(file_path))
    ws = wb.sheet_by_name("MF")

    for row_idx in range(ws.nrows):
        row = ws.row_values(row_idx)
        row_text = " | ".join(
            str(value) for value in row if value not in (None, "")
        )

        if any(word.lower() in row_text.lower() for word in keywords):
            print(f"Row {row_idx + 1}: {row_text}")

Row 1: RLMF806 | Nippon India Multi Asset Allocation Fund (An open ended scheme investing in Equity, Debt 
and Exchange Traded Commodity Derivatives and 
Gold ETF & Silver ETF) | Index
Row 162: LIGS27 | INE2R5L15013 | Liquid Gold Series** | CRISIL AAA(SO) | 571 | 4988.75 | 0.0037 | 0.093123
Row 163: LIGS22 | INE1U3R15011 | Liquid Gold Series** | CRISIL AAA(SO) | 119.999996 | 1497.66 | 0.0011 | 0.094925
Row 179: BMGB51ME | INF204KB17I5 | Nippon India ETF Gold Bees | 97272043 | 128010.01 | 0.0953
Row 180: 149758 | INF204KC1402 | Nippon India Silver ETF | 20900450 | 52838.43 | 0.0393
Row 183: FNOMGN | Cash Margin - Derivatives | 18875 | 0.014
Row 184: CCILMGN | Cash Margin - CCIL | 42.96 | $0.00%
Row 198: Long | FUTCOM_GOLD_02/04/2026 | Others | 14000 | 22694.56 | 0.0169
Row 207: BENCHMARK NAME - 50% OF BSE 500 TRI, 20% OF MSCI WORLD INDEX TRI, 15% OF CRISIL SHORT TERM BOND INDEX 10% OF DOMESTIC PRICES OF GOLD & 5% OF DOMESTIC PRICES OF SILVER


In [83]:
# Search March 2026 for futures and derivatives-related entries.

filename = "NIMF-MONTHLY-PORTFOLIO-31-Mar-26.xls"
file_path = nippon_dir / filename

keywords = [
    "futures",
    "underlying",
    "margin",
    "derivatives disclosure",
    "GOLD",
    "SILVER"
]

if is_zipfile(file_path):
    with open(file_path, "rb") as stream:
        wb = openpyxl.load_workbook(
            stream, read_only=False, data_only=True
        )
        ws = wb["MF"]

        for row_num, row in enumerate(
            ws.iter_rows(values_only=True), start=1
        ):
            row_text = " | ".join(
                str(value) for value in row if value is not None
            )

            if any(word.lower() in row_text.lower() for word in keywords):
                print(f"Row {row_num}: {row_text}")

        wb.close()

else:
    wb = xlrd.open_workbook(str(file_path))
    ws = wb.sheet_by_name("MF")

    for row_idx in range(ws.nrows):
        row = ws.row_values(row_idx)
        row_text = " | ".join(
            str(value) for value in row if value not in (None, "")
        )

        if any(word.lower() in row_text.lower() for word in keywords):
            print(f"Row {row_idx + 1}: {row_text}")

Row 1: RLMF806 | Nippon India Multi Asset Allocation Fund (An open ended scheme investing in Equity, Debt 
and Exchange Traded Commodity Derivatives and 
Gold ETF & Silver ETF) | Index
Row 174: LIGS27 | INE2R5L15013 | Liquid Gold Series** | CRISIL AAA(SO) | 570.999999 | 4663.06 | 0.0035 | 0.098122
Row 175: LIGS22 | INE1U3R15011 | Liquid Gold Series** | CRISIL AAA(SO) | 119.999996 | 1375.22 | 0.001 | 0.099225
Row 189: BMGB51ME | INF204KB17I5 | Nippon India ETF Gold Bees | 97272043.0 | 117883.99 | 0.0897
Row 190: 149758 | INF204KC1402 | Nippon India Silver ETF | 20959845.0 | 46698.68 | 0.0355
Row 195: FNOMGN | Cash Margin - Derivatives | 18875.0 | 0.0144
Row 196: CCILMGN | Cash Margin - CCIL | 34.04 | $0.00%
Row 218: BENCHMARK NAME - 50% OF BSE 500 TRI, 20% OF MSCI WORLD INDEX TRI, 15% OF CRISIL SHORT TERM BOND INDEX 10% OF DOMESTIC PRICES OF GOLD & 5% OF DOMESTIC PRICES OF SILVER


In [84]:
# Search March 2026 for futures and derivatives-related entries.

filename = "NIMF-MONTHLY-PORTFOLIO-31-Mar-26.xls"
file_path = nippon_dir / filename

keywords = [
    "futures",
    "underlying",
    "margin",
    "derivatives disclosure",
    "GOLD",
    "SILVER"
]

if is_zipfile(file_path):
    with open(file_path, "rb") as stream:
        wb = openpyxl.load_workbook(
            stream, read_only=False, data_only=True
        )
        ws = wb["MF"]

        for row_num, row in enumerate(
            ws.iter_rows(values_only=True), start=1
        ):
            row_text = " | ".join(
                str(value) for value in row if value is not None
            )

            if any(word.lower() in row_text.lower() for word in keywords):
                print(f"Row {row_num}: {row_text}")

        wb.close()

else:
    wb = xlrd.open_workbook(str(file_path))
    ws = wb.sheet_by_name("MF")

    for row_idx in range(ws.nrows):
        row = ws.row_values(row_idx)
        row_text = " | ".join(
            str(value) for value in row if value not in (None, "")
        )

        if any(word.lower() in row_text.lower() for word in keywords):
            print(f"Row {row_idx + 1}: {row_text}")

Row 1: RLMF806 | Nippon India Multi Asset Allocation Fund (An open ended scheme investing in Equity, Debt 
and Exchange Traded Commodity Derivatives and 
Gold ETF & Silver ETF) | Index
Row 174: LIGS27 | INE2R5L15013 | Liquid Gold Series** | CRISIL AAA(SO) | 570.999999 | 4663.06 | 0.0035 | 0.098122
Row 175: LIGS22 | INE1U3R15011 | Liquid Gold Series** | CRISIL AAA(SO) | 119.999996 | 1375.22 | 0.001 | 0.099225
Row 189: BMGB51ME | INF204KB17I5 | Nippon India ETF Gold Bees | 97272043.0 | 117883.99 | 0.0897
Row 190: 149758 | INF204KC1402 | Nippon India Silver ETF | 20959845.0 | 46698.68 | 0.0355
Row 195: FNOMGN | Cash Margin - Derivatives | 18875.0 | 0.0144
Row 196: CCILMGN | Cash Margin - CCIL | 34.04 | $0.00%
Row 218: BENCHMARK NAME - 50% OF BSE 500 TRI, 20% OF MSCI WORLD INDEX TRI, 15% OF CRISIL SHORT TERM BOND INDEX 10% OF DOMESTIC PRICES OF GOLD & 5% OF DOMESTIC PRICES OF SILVER


In [85]:
# Update Nippon India review notes after derivatives verification.
# Existing holdings values are left unchanged.

from pathlib import Path
import pandas as pd

output_file = OUTPUT_DIR / "Nippon_India_Monthly_Holdings_Log.csv"
nippon_log = pd.read_csv(output_file)

review_updates = {
    "2025-02": (
        "Derivatives disclosure checked: gold futures confirmed in holdings "
        "section; silver futures not identified in matching rows. "
        "Cash Margin - Derivatives and Cash Margin - CCIL are separate "
        "margin entries, not futures market values."
    ),
    "2025-03": (
        "Derivatives-related rows checked: gold futures confirmed in holdings "
        "section; silver futures not identified in matching rows. "
        "Cash Margin - Derivatives is separate from futures market value. "
        "Liquid Gold Series remains a separate security pending classification."
    ),
    "2025-08": (
        "Matching holdings rows checked: gold and silver futures not identified; "
        "crude-oil futures reported. Cash Margin - Derivatives is a general "
        "margin entry and does not establish gold/silver futures exposure."
    ),
    "2026-02": (
        "Derivatives-related rows checked: gold futures confirmed in holdings "
        "section; silver futures not identified in matching rows. "
        "Cash Margin - Derivatives is separate from futures market value. "
        "Liquid Gold Series entries remain separate securities."
    ),
    "2026-03": (
        "Matching holdings and commodity-futures rows checked: gold and silver "
        "futures not identified; crude-oil futures reported. "
        "Cash Margin - Derivatives does not establish gold/silver futures exposure. "
        "Liquid Gold Series entries remain separate securities."
    ),
    "2026-04": (
        "Derivatives disclosure checked: non-hedging GOLD and CRUDEOIL futures "
        "are reported separately. The reported GOLD margin is not the futures "
        "market value and must not be added to Gold_Futures_Value_Lakh."
    ),
    "2026-05": (
        "Derivatives disclosure checked: non-hedging GOLD and CRUDEOIL futures "
        "are reported separately. The reported GOLD margin is not the futures "
        "market value and must not be added to Gold_Futures_Value_Lakh."
    ),
    "2026-06": (
        "Derivatives disclosure checked: non-hedging GOLD and CRUDEOIL futures "
        "are reported separately. The reported GOLD margin is not the futures "
        "market value and must not be added to Gold_Futures_Value_Lakh."
    ),
}

if "Review_Notes" not in nippon_log.columns:
    raise KeyError("Review_Notes column is missing. No changes were saved.")

for month, note in review_updates.items():
    mask = nippon_log["Report_Month"].astype(str).eq(month)

    if mask.sum() != 1:
        raise ValueError(
            f"Expected exactly one row for {month}; found {mask.sum()}."
        )

    current = nippon_log.loc[mask, "Review_Notes"].iloc[0]
    current = "" if pd.isna(current) else str(current)

    if note not in current:
        separator = " " if current.strip() else ""
        nippon_log.loc[mask, "Review_Notes"] = current + separator + note

nippon_log.to_csv(output_file, index=False)

# Verify the saved file and ensure row count/month coverage remain intact.
verified_log = pd.read_csv(output_file)

print("Updated and saved:", output_file)
print("Rows:", len(verified_log))
print("Unique months:", verified_log["Report_Month"].nunique())
print("Duplicate months:", verified_log["Report_Month"].duplicated().sum())
print("\nUpdated review notes:")

print(
    verified_log.loc[
        verified_log["Report_Month"].isin(review_updates),
        ["Report_Month", "Review_Notes"]
    ].to_string(index=False)
)

Updated and saved: c:\Users\rocks\OneDrive\Desktop\AMC Project Data\02_CLEAN_DATA\Nippon_India_Monthly_Holdings_Log.csv
Rows: 33
Unique months: 33
Duplicate months: 0

Updated review notes:
Report_Month                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               Review_Notes
     2025-02                                                                                      

In [87]:
# Find the actual September 2026 source filename.

from pathlib import Path

print("Nippon folder:", nippon_dir)
print("Folder exists:", nippon_dir.exists())

if nippon_dir.exists():
    matches = sorted(
        file for file in nippon_dir.iterdir()
        if file.is_file()
        and ("september" in file.name.lower()
             or "sep" in file.name.lower())
        and "2026" in file.name
    )

    if matches:
        print("\nMatching files:")
        for file in matches:
            print(file.name)
    else:
        print("\nNo September/Sep 2026 files found.")
        print("\nFiles containing 2026:")
        for file in sorted(nippon_dir.iterdir()):
            if file.is_file() and "2026" in file.name:
                print(file.name)
else:
    print("The Nippon folder path itself could not be found.")

Nippon folder: c:\Users\rocks\OneDrive\Desktop\AMC Project Data\01_RAW_DATA\Nippon_India
Folder exists: True

No September/Sep 2026 files found.

Files containing 2026:


In [88]:
# Search Nippon India folder and all subfolders for Excel files.

excel_files = sorted(
    file
    for file in nippon_dir.rglob("*")
    if file.is_file()
    and file.suffix.lower() in [".xls", ".xlsx", ".xlsm"]
)

print("Total Excel files found:", len(excel_files))

for file in excel_files:
    print(file.relative_to(nippon_dir))

Total Excel files found: 33
MONTHLY-PORTFOLIO-JAN-2024.xls
MONTHLY-PORTFOLIO-REPORT-FEB-24.xls
NIMF-MONTHLY-PORTFOLIO-28-Feb-25.xls
NIMF-MONTHLY-PORTFOLIO-28-Feb-26.xls
NIMF-MONTHLY-PORTFOLIO-30-April-25.xls
NIMF-MONTHLY-PORTFOLIO-30-April-26.xls
NIMF-MONTHLY-PORTFOLIO-30-Jun-26.xls
NIMF-MONTHLY-PORTFOLIO-30-June-25.xls
NIMF-MONTHLY-PORTFOLIO-30-Nov-2024.xls
NIMF-MONTHLY-PORTFOLIO-30-Sep-24.xls
NIMF-MONTHLY-PORTFOLIO-30-Sep-25.xls
NIMF-MONTHLY-PORTFOLIO-30-Sep-26.xls
NIMF-MONTHLY-PORTFOLIO-31-Aug-25.xls
NIMF-MONTHLY-PORTFOLIO-31-Aug-26.xls
NIMF-MONTHLY-PORTFOLIO-31-Dec-24.xls
NIMF-MONTHLY-PORTFOLIO-31-Dec-25.xls
NIMF-MONTHLY-PORTFOLIO-31-Jan-26.xls
NIMF-MONTHLY-PORTFOLIO-31-July-24.xls
NIMF-MONTHLY-PORTFOLIO-31-July-25.xls
NIMF-MONTHLY-PORTFOLIO-31-July-26.xls
NIMF-MONTHLY-PORTFOLIO-31-Mar-25.xls
NIMF-MONTHLY-PORTFOLIO-31-Mar-26.xls
NIMF-MONTHLY-PORTFOLIO-31-May-25.xls
NIMF-MONTHLY-PORTFOLIO-31-May-26.xls
NIMF-MONTHLY-PORTFOLIO-31-Oct-24.xls
NIMF-MONTHLY-PORTFOLIO-31-Oct-25.xls
NIMF-MO

In [89]:
# Inspect the correct September 2026 Nippon India workbook.

filename = "NIMF-MONTHLY-PORTFOLIO-30-Sep-26.xls"
file_path = nippon_dir / filename

if is_zipfile(file_path):
    with open(file_path, "rb") as stream:
        wb = openpyxl.load_workbook(
            stream, read_only=False, data_only=True
        )
        ws = wb["MF"]

        print("Worksheet:", ws.title)
        print("Total rows:", ws.max_row)
        print("Total columns:", ws.max_column)

        for row_num in range(1, min(16, ws.max_row) + 1):
            values = next(
                ws.iter_rows(
                    min_row=row_num,
                    max_row=row_num,
                    values_only=True
                )
            )
            print(f"Row {row_num}: {values}")

        wb.close()

else:
    wb = xlrd.open_workbook(str(file_path))
    ws = wb.sheet_by_name("MF")

    print("Worksheet:", ws.name)
    print("Total rows:", ws.nrows)
    print("Total columns:", ws.ncols)

    for row_idx in range(min(16, ws.nrows)):
        print(f"Row {row_idx + 1}: {ws.row_values(row_idx)}")

Worksheet: MF
Total rows: 313
Total columns: 8
Row 1: ('RLMF806', 'Nippon India Multi Asset Allocation Fund (An open ended scheme investing in Equity, Debt \nand Exchange Traded Commodity Derivatives and \nGold ETF & Silver ETF)', None, None, None, None, None, 'Index')
Row 2: (None, 'Monthly Portfolio Statement as on September 30,2026', None, None, None, None, None, None)
Row 3: (None, None, None, None, None, None, None, None)
Row 4: (None, 'ISIN', 'Name of the Instrument', 'Industry / Rating', 'Quantity', 'Market/Fair Value\n( Rs. in Lacs)', '% to NAV', 'YIELD')
Row 5: (None, None, 'Equity & Equity related', None, None, None, None, None)
Row 6: (None, None, '(a) Listed / awaiting listing on Stock Exchanges', None, None, None, None, None)
Row 7: ('IBCL05', 'INE090A01021', 'ICICI Bank Limited', 'Banks', 4694329, 62044.95, 0.0344, None)
Row 8: ('HDFB03', 'INE040A01034', 'HDFC Bank Limited', 'Banks', 8263057, 58560.28, 0.0325, None)
Row 9: ('UTIB02', 'INE238A01034', 'Axis Bank Limited', '

In [90]:
# Identify section headings in the September 2026 MF worksheet.
# This cell only reads the source workbook.

filename = "NIMF-MONTHLY-PORTFOLIO-30-Sep-26.xls"
file_path = nippon_dir / filename

if is_zipfile(file_path):
    with open(file_path, "rb") as stream:
        wb = openpyxl.load_workbook(
            stream, read_only=False, data_only=True
        )
        ws = wb["MF"]

        for row_num, row in enumerate(
            ws.iter_rows(values_only=True), start=1
        ):
            values = [str(v).strip() for v in row if v is not None]
            if not values:
                continue

            # Print rows likely to contain section headings.
            if (
                len(values) <= 3
                or any(
                    term in " ".join(values).lower()
                    for term in [
                        "equity & equity related",
                        "debt",
                        "money market",
                        "gold",
                        "silver",
                        "commodity",
                        "derivative",
                        "benchmark",
                        "net current assets",
                        "cash",
                    ]
                )
            ):
                print(f"Row {row_num}: {tuple(row)}")

        wb.close()

else:
    wb = xlrd.open_workbook(str(file_path))
    ws = wb.sheet_by_name("MF")

    for row_idx in range(ws.nrows):
        row = ws.row_values(row_idx)
        values = [str(v).strip() for v in row if v not in (None, "")]
        if not values:
            continue

        if (
            len(values) <= 3
            or any(
                term in " ".join(values).lower()
                for term in [
                    "equity & equity related",
                    "debt",
                    "money market",
                    "gold",
                    "silver",
                    "commodity",
                    "derivative",
                    "benchmark",
                    "net current assets",
                    "cash",
                ]
            )
        ):
            print(f"Row {row_idx + 1}: {tuple(row)}")

Row 1: ('RLMF806', 'Nippon India Multi Asset Allocation Fund (An open ended scheme investing in Equity, Debt \nand Exchange Traded Commodity Derivatives and \nGold ETF & Silver ETF)', None, None, None, None, None, 'Index')
Row 2: (None, 'Monthly Portfolio Statement as on September 30,2026', None, None, None, None, None, None)
Row 5: (None, None, 'Equity & Equity related', None, None, None, None, None)
Row 6: (None, None, '(a) Listed / awaiting listing on Stock Exchanges', None, None, None, None, None)
Row 96: (None, None, 'REIT', None, None, None, None, None)
Row 98: (None, None, 'Subtotal', None, None, 1050137.61, 0.5822, None)
Row 101: (None, None, 'Total', None, None, 1050137.61, 0.5822, None)
Row 102: (None, None, 'InvIT', None, None, None, None, None)
Row 108: (None, None, 'Subtotal', None, None, 41419.41, 0.0229, None)
Row 109: (None, None, 'Total', None, None, 41419.41, 0.0229, None)
Row 110: (None, None, 'Listed Foreign Securities/Overseas ETFs', None, None, None, None, None)
R

In [91]:
# Inspect the listed equity holdings in Nippon India's September 2026 report.

filename = "NIMF-MONTHLY-PORTFOLIO-30-Sep-26.xls"
file_path = nippon_dir / filename

if is_zipfile(file_path):
    with open(file_path, "rb") as stream:
        wb = openpyxl.load_workbook(
            stream, read_only=False, data_only=True
        )
        ws = wb["MF"]

        print("Row | Instrument Name | Industry | Market Value (₹ lakh) | % NAV")

        for row_num in range(7, 96):
            row = next(
                ws.iter_rows(
                    min_row=row_num,
                    max_row=row_num,
                    values_only=True
                )
            )

            name = row[2]
            industry = row[3]
            value = row[5]
            nav_pct = row[6]

            if name is not None and str(name).strip().lower() not in {
                "subtotal", "total"
            }:
                print(
                    f"{row_num} | {name} | {industry} | "
                    f"{value} | {nav_pct}"
                )

        wb.close()

else:
    wb = xlrd.open_workbook(str(file_path))
    ws = wb.sheet_by_name("MF")

    print("Row | Instrument Name | Industry | Market Value (₹ lakh) | % NAV")

    for row_idx in range(6, min(95, ws.nrows)):
        row = ws.row_values(row_idx)

        name = row[2]
        industry = row[3]
        value = row[5]
        nav_pct = row[6]

        if name not in (None, "") and str(name).strip().lower() not in {
            "subtotal", "total"
        }:
            print(
                f"{row_idx + 1} | {name} | {industry} | "
                f"{value} | {nav_pct}"
            )

Row | Instrument Name | Industry | Market Value (₹ lakh) | % NAV
7 | ICICI Bank Limited | Banks | 62044.95 | 0.0344
8 | HDFC Bank Limited | Banks | 58560.28 | 0.0325
9 | Axis Bank Limited | Banks | 43734.85 | 0.0242
10 | State Bank of India | Banks | 30534.34 | 0.0169
11 | Eternal Limited | Retailing | 29694.57 | 0.0165
12 | Reliance Industries Limited | Petroleum Products | 26862.45 | 0.0149
13 | Mahindra & Mahindra Limited | Automobiles | 25915.75 | 0.0144
14 | Info Edge (India) Limited | Retailing | 25594.81 | 0.0142
15 | Hindustan Aeronautics Limited | Aerospace & Defense | 24380.36 | 0.0135
16 | Infosys Limited | IT - Software | 24296.61 | 0.0135
17 | Hindustan Unilever Limited | Diversified FMCG | 23982.05 | 0.0133
18 | HDFC Life Insurance Company Limited | Insurance | 22921.7 | 0.0127
19 | Trent Limited | Retailing | 22372.71 | 0.0124
20 | HDFC Asset Management Company Limited | Capital Markets | 22183.39 | 0.0123
21 | Kotak Mahindra Bank Limited | Banks | 20850 | 0.0116
22 | NT

In [92]:
import pandas as pd
from pathlib import Path

# Locate and load the existing Nippon India log
project_dir = Path.cwd().parent
output_file = (
    project_dir
    / "02_CLEAN_DATA"
    / "Nippon_India_Monthly_Holdings_Log.csv"
)

nippon_log = pd.read_csv(output_file)

print("File found:", output_file.exists())
print("Total rows:", len(nippon_log))
print("Unique months:", nippon_log["Report_Month"].nunique())
print("Duplicate months:", nippon_log["Report_Month"].duplicated().sum())

# Columns requiring verification
check_columns = [
    "Gold_ETF_Value_Lakh",
    "Silver_ETF_Value_Lakh",
    "Gold_Futures_Value_Lakh",
    "Silver_Futures_Value_Lakh",
    "Silver_Options_Value_Lakh",
]

print("\nMissing values by column:")
print(nippon_log[check_columns].isna().sum())

print("\nMonths with missing futures or options values:")
display(
    nippon_log.loc[
        nippon_log[
            [
                "Gold_Futures_Value_Lakh",
                "Silver_Futures_Value_Lakh",
                "Silver_Options_Value_Lakh",
            ]
        ].isna().any(axis=1),
        [
            "Report_Month",
            "Gold_Futures_Value_Lakh",
            "Silver_Futures_Value_Lakh",
            "Silver_Options_Value_Lakh",
            "Review_Notes",
        ],
    ].sort_values("Report_Month")
)

File found: True
Total rows: 33
Unique months: 33
Duplicate months: 0

Missing values by column:
Gold_ETF_Value_Lakh           0
Silver_ETF_Value_Lakh         6
Gold_Futures_Value_Lakh       5
Silver_Futures_Value_Lakh     8
Silver_Options_Value_Lakh    29
dtype: int64

Months with missing futures or options values:


,Report_Month,Gold_Futures_Value_Lakh,Silver_Futures_Value_Lakh,Silver_Options_Value_Lakh,Review_Notes
0,2024-01,NaN,1475.29,17.24,No direct silver ETF identified in keyword sea...
1,2024-02,NaN,2587.43,NaN,No direct silver ETF identified in keyword sea...
2,2024-03,NaN,3219.56,5.68,No direct silver ETF identified in keyword sea...
5,2024-06,7372.95,7655.67,NaN,No direct silver ETF identified in keyword sea...
6,2024-07,12259.28,7022.06,NaN,NaN
7,2024-08,14895.09,1022.52,NaN,NaN
8,2024-09,16105.14,2721.57,NaN,NaN
9,2024-10,19293.78,3662.22,NaN,NaN
10,2024-11,16891.03,1012.42,NaN,NaN
11,2024-12,12049.44,2538.48,NaN,NaN


In [93]:
# Show every month with a missing gold or silver futures value.

futures_missing = nippon_log[
    nippon_log["Gold_Futures_Value_Lakh"].isna()
    | nippon_log["Silver_Futures_Value_Lakh"].isna()
][
    [
        "Report_Month",
        "Gold_Futures_Value_Lakh",
        "Gold_Futures_NAV_Pct",
        "Silver_Futures_Value_Lakh",
        "Silver_Futures_NAV_Pct",
        "Review_Notes",
    ]
].sort_values("Report_Month")

print("Months requiring futures verification:", len(futures_missing))
display(futures_missing.to_string(index=False))

Months requiring futures verification: 11


'Report_Month  Gold_Futures_Value_Lakh  Gold_Futures_NAV_Pct  Silver_Futures_Value_Lakh  Silver_Futures_NAV_Pct                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               Review_Notes\n     2024-01                      NaN                   NaN                    1475.29                    0.58                                                                              

In [94]:
# Search the January, February and March 2024 Nippon reports
# for gold-related entries in the MF worksheet.

from zipfile import is_zipfile
import openpyxl
import xlrd

files_to_check = [
    "MONTHLY-PORTFOLIO-JAN-2024.xls",
    "MONTHLY-PORTFOLIO-REPORT-FEB-24.xls",
    "NIMF-MONTHLY-PORTFOLIO-REPORT-March-24.xls",
]

for filename in files_to_check:
    file_path = nippon_dir / filename
    print(f"\n{'=' * 25}\nFILE: {filename}\n{'=' * 25}")

    if not file_path.exists():
        print("File not found — check the exact filename.")
        continue

    if is_zipfile(file_path):
        with open(file_path, "rb") as stream:
            wb = openpyxl.load_workbook(
                stream, read_only=True, data_only=True
            )
            ws = wb["MF"]

            for row_num, row in enumerate(
                ws.iter_rows(values_only=True), start=1
            ):
                text = " ".join(
                    str(value) for value in row
                    if value is not None
                ).lower()

                if "gold" in text and "benchmark" not in text:
                    print(f"Row {row_num}: {tuple(row)}")

            wb.close()

    else:
        wb = xlrd.open_workbook(str(file_path))
        ws = wb.sheet_by_name("MF")

        for row_idx in range(ws.nrows):
            row = ws.row_values(row_idx)
            text = " ".join(
                str(value) for value in row
                if value not in (None, "")
            ).lower()

            if "gold" in text and "benchmark" not in text:
                print(f"Row {row_idx + 1}: {tuple(row)}")


FILE: MONTHLY-PORTFOLIO-JAN-2024.xls
Row 1: ('RLMF806', 'Nippon India Multi Asset Fund (An open ended scheme investing in Equity, Debt and Exchange Traded Commodity Derivatives and Gold ETF)', None, None, None, None, None, 'Index', None, None, None)
Row 108: ('BMGB51ME', 'INF204KB17I5', 'Nippon India ETF Gold Bees', None, 62360000, 33269.06, 0.1308, None, None, None, None)

FILE: MONTHLY-PORTFOLIO-REPORT-FEB-24.xls
Row 1: ('RLMF806', 'Nippon India Multi Asset Fund (An open ended scheme investing in Equity, Debt and Exchange Traded Commodity Derivatives and Gold ETF)', None, None, None, None, None, 'Index')
Row 104: ('BMGB51ME', 'INF204KB17I5', 'Nippon India ETF Gold Bees', None, 65580197, 34613.23, 0.1281, None)

FILE: NIMF-MONTHLY-PORTFOLIO-REPORT-March-24.xls
Row 1: ('RLMF806', 'Nippon India Multi Asset Fund (An open ended scheme investing in Equity, Debt and Exchange Traded Commodity Derivatives and Gold ETF)', '', '', '', '', '', 'Index')
Row 109: ('BMGB51ME', 'INF204KB17I5', 'Nip

In [95]:
# Inspect commodity-futures sections in the first three 2024 reports.

for filename in files_to_check:
    file_path = nippon_dir / filename
    print(f"\n{'=' * 20}\n{filename}\n{'=' * 20}")

    if is_zipfile(file_path):
        with open(file_path, "rb") as stream:
            wb = openpyxl.load_workbook(
                stream, read_only=True, data_only=True
            )

            for ws in wb.worksheets:
                for row_num, row in enumerate(
                    ws.iter_rows(values_only=True), start=1
                ):
                    text = " ".join(
                        str(value) for value in row
                        if value is not None
                    ).lower()

                    if "commodity future" in text or (
                        "futcom_gold" in text
                    ):
                        print(
                            f"Sheet {ws.title}, row {row_num}: "
                            f"{tuple(row)}"
                        )

            wb.close()

    else:
        wb = xlrd.open_workbook(str(file_path))

        for ws in wb.sheets():
            for row_idx in range(ws.nrows):
                row = ws.row_values(row_idx)
                text = " ".join(
                    str(value) for value in row
                    if value not in (None, "")
                ).lower()

                if "commodity future" in text or "futcom_gold" in text:
                    print(
                        f"Sheet {ws.name}, row {row_idx + 1}: "
                        f"{tuple(row)}"
                    )


MONTHLY-PORTFOLIO-JAN-2024.xls
Sheet MF, row 127: (None, 'Details of Commodity Future / Index Future', None, None, None, None, None, None, None, None, None)

MONTHLY-PORTFOLIO-REPORT-FEB-24.xls
Sheet MF, row 123: (None, 'Details of Commodity Future / Index Future', None, None, None, None, None, None)

NIMF-MONTHLY-PORTFOLIO-REPORT-March-24.xls
Sheet MF, row 128: ('', 'Details of Commodity Future / Index Future', '', '', '', '', '', '')


In [96]:
# Display the commodity-futures section and the rows beneath it.

for filename in files_to_check:
    file_path = nippon_dir / filename
    print(f"\n{'=' * 20}\n{filename}\n{'=' * 20}")

    if is_zipfile(file_path):
        with open(file_path, "rb") as stream:
            wb = openpyxl.load_workbook(
                stream, read_only=True, data_only=True
            )
            ws = wb["MF"]

            for row_num, row in enumerate(
                ws.iter_rows(values_only=True), start=1
            ):
                text = " ".join(
                    str(value) for value in row
                    if value is not None
                ).lower()

                if "details of commodity future" in text:
                    for next_row in range(
                        row_num, min(row_num + 13, ws.max_row + 1)
                    ):
                        values = next(
                            ws.iter_rows(
                                min_row=next_row,
                                max_row=next_row,
                                values_only=True
                            )
                        )
                        print(f"Row {next_row}: {values}")
                    break

            wb.close()

    else:
        wb = xlrd.open_workbook(str(file_path))
        ws = wb.sheet_by_name("MF")

        for row_idx in range(ws.nrows):
            row = ws.row_values(row_idx)
            text = " ".join(
                str(value) for value in row
                if value not in (None, "")
            ).lower()

            if "details of commodity future" in text:
                for next_idx in range(
                    row_idx, min(row_idx + 13, ws.nrows)
                ):
                    print(
                        f"Row {next_idx + 1}: "
                        f"{ws.row_values(next_idx)}"
                    )
                break


MONTHLY-PORTFOLIO-JAN-2024.xls
Row 127: (None, 'Details of Commodity Future / Index Future', None, None, None, None, None, None, None, None, None)
Row 128: (None, 'Long/Short', 'Name of the Instrument', 'Industry', 'Quantity.', 'Market/Fair Value (Rs. in Lacs).', ' % to NAV.', None, None, None, None)
Row 129: (None, 'Long', 'FUTCOM_SILVER_05/03/2024', 'Others', 1920, 1387.14, 0.0055, None, None, None, None)
Row 130: (None, 'Long', 'FUTCOM_SILVER_03/05/2024', 'Others', 120, 88.15, 0.0003, None, None, None, None)
Row 131: (None, None, None, None, None, None, None, None, None, None, None)
Row 132: (None, None, None, None, None, 'BENCHMARK NAME - CRISIL SHORT TERM BOND INDEX, DOMESTIC PRICE OF GOLD, MSCI WORLD INDEX TRI, S&P BSE 500 TRI', None, None, None, None, None)
Row 133: (None, 'SCHEME RISK-O-METER', None, None, None, 'BENCHMARK RISK-O-METER', None, None, None, None, None)

MONTHLY-PORTFOLIO-REPORT-FEB-24.xls
Row 123: (None, 'Details of Commodity Future / Index Future', None, None, 

In [97]:
# Record the verified commodity-futures section findings
# for January, February and March 2024.

review_updates_2024 = {
    "2024-01": (
        "Commodity-futures section verified: silver futures total "
        "₹1,475.29 lakh (0.58% NAV); no gold futures entry appears "
        "in this section. Gold futures remains unrecorded, not zero."
    ),
    "2024-02": (
        "Commodity-futures section verified: silver futures "
        "₹2,587.43 lakh (0.96% NAV) and crude-oil futures "
        "₹481.96 lakh. No gold futures entry appears in this section; "
        "gold futures remains unrecorded, not zero."
    ),
    "2024-03": (
        "Commodity-futures section verified: silver futures "
        "₹3,219.56 lakh (1.11% NAV) and crude-oil futures "
        "₹1,222.89 lakh. No gold futures entry appears in this section; "
        "gold futures remains unrecorded, not zero."
    ),
}

for month, note in review_updates_2024.items():
    mask = nippon_log["Report_Month"].astype(str).eq(month)

    if mask.sum() != 1:
        raise ValueError(
            f"Expected exactly one row for {month}; found {mask.sum()}."
        )

    current = nippon_log.loc[mask, "Review_Notes"].iloc[0]
    current = "" if pd.isna(current) else str(current)

    if note not in current:
        separator = " " if current.strip() else ""
        nippon_log.loc[mask, "Review_Notes"] = current + separator + note

nippon_log.to_csv(output_file, index=False)

print("Updated review notes for January–March 2024.")
print("Rows:", len(nippon_log))
print("Unique months:", nippon_log["Report_Month"].nunique())

Updated review notes for January–March 2024.
Rows: 33
Unique months: 33


In [98]:
# Verify the commodity-futures sections for August 2025 and March 2026.

files_to_check_next = [
    "NIMF-MONTHLY-PORTFOLIO-31-Aug-25.xls",
    "NIMF-MONTHLY-PORTFOLIO-31-Mar-26.xls",
]

for filename in files_to_check_next:
    file_path = nippon_dir / filename
    print(f"\n{'=' * 25}\nFILE: {filename}\n{'=' * 25}")

    if not file_path.exists():
        print("File not found.")
        continue

    if is_zipfile(file_path):
        with open(file_path, "rb") as stream:
            wb = openpyxl.load_workbook(
                stream, read_only=True, data_only=True
            )
            ws = wb["MF"]

            found = False

            for row_num, row in enumerate(
                ws.iter_rows(values_only=True), start=1
            ):
                text = " ".join(
                    str(v) for v in row if v is not None
                ).lower()

                if "details of commodity future" in text:
                    found = True
                    for next_row in range(
                        row_num, min(row_num + 16, ws.max_row + 1)
                    ):
                        values = next(
                            ws.iter_rows(
                                min_row=next_row,
                                max_row=next_row,
                                values_only=True
                            )
                        )
                        print(f"Row {next_row}: {values}")
                    break

            if not found:
                print("Commodity-futures heading not found in MF sheet.")

            wb.close()

    else:
        wb = xlrd.open_workbook(str(file_path))
        ws = wb.sheet_by_name("MF")
        found = False

        for row_idx in range(ws.nrows):
            row = ws.row_values(row_idx)
            text = " ".join(
                str(v) for v in row if v not in (None, "")
            ).lower()

            if "details of commodity future" in text:
                found = True
                for next_idx in range(
                    row_idx, min(row_idx + 16, ws.nrows)
                ):
                    print(
                        f"Row {next_idx + 1}: "
                        f"{ws.row_values(next_idx)}"
                    )
                break

        if not found:
            print("Commodity-futures heading not found in MF sheet.")


FILE: NIMF-MONTHLY-PORTFOLIO-31-Aug-25.xls
Row 171: (None, 'Details of Commodity Future / Index Future', None, None, None, None, None, None)
Row 172: (None, 'Long/Short', 'Name of the Instrument', 'Industry', 'Quantity.', 'Market/Fair Value (Rs. in Lacs).', ' % to NAV.', None)
Row 173: (None, 'Long', 'FUTCOM_CRUDEOIL_19/09/2025', 'Others', 20000, 1130.8, 0.0016, None)
Row 174: (None, None, None, None, None, None, None, None)
Row 175: (None, None, None, None, None, None, None, None)
Row 176: (None, 'Disclosure of Yield to Call', None, None, None, None, None, None)
Row 177: (None, '“With Reference to “ AMFI Best Practices Guidelines Circular No. 91/ 2020 – 21 on \xa0Valuation of AT-1 Bonds and Tier 2 Bonds” disclosure of Yield to Call \xa0as follows”', None, None, None, None, None, None)
Row 178: (None, None, None, None, None, None, None, None)
Row 179: (None, 'isin_code', 'security_name', 'Yield to Call', None, None, None, None)
Row 180: (None, 'INE062A08256', '6.24% SBI NCDSr2 BSIII T

In [99]:
# Record verified commodity-futures findings for August 2025
# and March 2026.

review_updates = {
    "2025-08": (
        "Commodity-futures section verified: crude-oil futures "
        "₹1,130.80 lakh (0.16% NAV) reported. No gold or silver "
        "futures entry appears in this section; their values remain "
        "unrecorded, not zero."
    ),
    "2026-03": (
        "Commodity-futures section verified: crude-oil futures "
        "₹8,981 lakh (0.68% NAV) and ₹2,391.75 lakh (0.18% NAV) "
        "reported. No gold or silver futures entry appears in this "
        "section; their values remain unrecorded, not zero."
    ),
}

for month, note in review_updates.items():
    mask = nippon_log["Report_Month"].astype(str).eq(month)

    if mask.sum() != 1:
        raise ValueError(
            f"Expected exactly one row for {month}; found {mask.sum()}."
        )

    current = nippon_log.loc[mask, "Review_Notes"].iloc[0]
    current = "" if pd.isna(current) else str(current)

    if note not in current:
        separator = " " if current.strip() else ""
        nippon_log.loc[mask, "Review_Notes"] = current + separator + note

nippon_log.to_csv(output_file, index=False)

print("Verification notes saved.")
print("Rows:", len(nippon_log))
print("Unique months:", nippon_log["Report_Month"].nunique())

Verification notes saved.
Rows: 33
Unique months: 33


In [100]:
# Inspect gold/silver and derivatives-related rows
# in the February 2025 and February 2026 Nippon reports.

from pathlib import Path
import openpyxl
import xlrd
from zipfile import is_zipfile

files_to_check = [
    "NIMF-MONTHLY-PORTFOLIO-REPORT-FEB-25.xls",
    "NIMF-MONTHLY-PORTFOLIO-REPORT-FEB-26.xls",
]

keywords = [
    "silver", "gold", "commodity future",
    "index future", "derivative", "margin"
]

for filename in files_to_check:
    file_path = nippon_dir / filename

    print(f"\n{'=' * 30}")
    print(f"FILE: {filename}")
    print(f"{'=' * 30}")

    if not file_path.exists():
        print("File not found. Check the exact filename in nippon_dir.")
        continue

    if is_zipfile(file_path):
        with open(file_path, "rb") as stream:
            wb = openpyxl.load_workbook(
                stream, read_only=True, data_only=True
            )
            ws = wb["MF"]

            for row_num, row in enumerate(
                ws.iter_rows(values_only=True), start=1
            ):
                values = tuple(row)
                row_text = " ".join(
                    str(v) for v in values if v is not None
                ).lower()

                if any(word in row_text for word in keywords):
                    print(f"Row {row_num}: {values}")

            wb.close()

    else:
        wb = xlrd.open_workbook(str(file_path))
        ws = wb.sheet_by_name("MF")

        for row_idx in range(ws.nrows):
            values = ws.row_values(row_idx)
            row_text = " ".join(
                str(v) for v in values if v not in (None, "")
            ).lower()

            if any(word in row_text for word in keywords):
                print(f"Row {row_idx + 1}: {values}")


FILE: NIMF-MONTHLY-PORTFOLIO-REPORT-FEB-25.xls
File not found. Check the exact filename in nippon_dir.

FILE: NIMF-MONTHLY-PORTFOLIO-REPORT-FEB-26.xls
File not found. Check the exact filename in nippon_dir.


In [101]:
# Find the exact February report filenames in the Nippon folder.

for file_path in sorted(nippon_dir.iterdir()):
    if file_path.is_file() and (
        "feb" in file_path.name.lower()
        or "february" in file_path.name.lower()
    ):
        print(file_path.name)

MONTHLY-PORTFOLIO-REPORT-FEB-24.xls
NIMF-MONTHLY-PORTFOLIO-28-Feb-25.xls
NIMF-MONTHLY-PORTFOLIO-28-Feb-26.xls


In [102]:
# Inspect the exact February 2025 and February 2026 Nippon reports.

files_to_check = [
    "NIMF-MONTHLY-PORTFOLIO-28-Feb-25.xls",
    "NIMF-MONTHLY-PORTFOLIO-28-Feb-26.xls",
]

keywords = [
    "silver", "gold", "commodity future",
    "index future", "derivative", "margin"
]

for filename in files_to_check:
    file_path = nippon_dir / filename

    print(f"\n{'=' * 30}")
    print(f"FILE: {filename}")
    print(f"{'=' * 30}")

    if is_zipfile(file_path):
        with open(file_path, "rb") as stream:
            wb = openpyxl.load_workbook(
                stream, read_only=True, data_only=True
            )
            ws = wb["MF"]

            for row_num, row in enumerate(
                ws.iter_rows(values_only=True), start=1
            ):
                values = tuple(row)
                row_text = " ".join(
                    str(v) for v in values if v is not None
                ).lower()

                if any(word in row_text for word in keywords):
                    print(f"Row {row_num}: {values}")

            wb.close()

    else:
        wb = xlrd.open_workbook(str(file_path))
        ws = wb.sheet_by_name("MF")

        for row_idx in range(ws.nrows):
            values = ws.row_values(row_idx)
            row_text = " ".join(
                str(v) for v in values if v not in (None, "")
            ).lower()

            if any(word in row_text for word in keywords):
                print(f"Row {row_idx + 1}: {values}")


FILE: NIMF-MONTHLY-PORTFOLIO-28-Feb-25.xls
Row 1: ['RLMF806', 'Nippon India Multi Asset Allocation Fund (An open ended scheme investing in Equity, Debt \nand Exchange Traded Commodity Derivatives and \nGold ETF & Silver ETF)', '', '', '', '', '', 'Index']
Row 140: ['BMGB51ME', 'INF204KB17I5', 'Nippon India ETF Gold Bees', '', 86261865.0, 61332.19, 0.1239, '']
Row 141: ['149758', 'INF204KC1402', 'Nippon India Silver ETF', '', 16869201.0, 15259.88, 0.0308, '']
Row 144: ['FNOMGN', '', 'Cash Margin - Derivatives', '', '', 1575.0, 0.0032, '']
Row 145: ['CCILMGN', '', 'Cash Margin - CCIL', '', '', 5.9, '$0.00%', '']
Row 155: ['', 'Details of Commodity Future / Index Future', '', '', '', '', '', '']
Row 157: ['', 'Long', 'FUTCOM_GOLD_04/04/2025', 'Others', 3800.0, 3200.32, 0.0065, '']
Row 165: ['', '', '', '', '', 'BENCHMARK NAME - 50% OF BSE 500 TRI, 20% OF MSCI WORLD INDEX TRI, 15% OF CRISIL SHORT TERM BOND INDEX 10% OF DOMESTIC PRICES OF GOLD & 5% OF DOMESTIC PRICES OF SILVER', '', '']

F

In [103]:
# Save the February 2025 and February 2026 inspection findings.

review_updates = {
    "2025-02": (
        "February report rechecked: gold futures ₹3,200.32 lakh "
        "(0.65% NAV) confirmed; silver futures not identified in "
        "the matching commodity-futures section. Cash Margin - "
        "Derivatives (₹1,575 lakh) is separate from futures market value."
    ),
    "2026-02": (
        "February report rechecked: gold futures ₹22,694.56 lakh "
        "(1.69% NAV) confirmed; silver futures not identified in "
        "the matching commodity-futures section. Cash Margin - "
        "Derivatives (₹18,875 lakh) is separate from futures market value. "
        "Liquid Gold Series securities remain separate from the Gold ETF."
    ),
}

for month, note in review_updates.items():
    mask = nippon_log["Report_Month"].astype(str).eq(month)

    if mask.sum() != 1:
        raise ValueError(
            f"Expected exactly one row for {month}; found {mask.sum()}."
        )

    current = nippon_log.loc[mask, "Review_Notes"].iloc[0]
    current = "" if pd.isna(current) else str(current)

    if note not in current:
        separator = " " if current.strip() else ""
        nippon_log.loc[mask, "Review_Notes"] = current + separator + note

nippon_log.to_csv(output_file, index=False)

print("February 2025 and February 2026 review notes saved.")
print("Rows:", len(nippon_log))
print("Unique months:", nippon_log["Report_Month"].nunique())

February 2025 and February 2026 review notes saved.
Rows: 33
Unique months: 33


In [104]:
# Find the exact monthly report filenames for April–June 2026.

for file_path in sorted(nippon_dir.iterdir()):
    name = file_path.name.lower()

    if file_path.is_file() and (
        ("apr" in name or "april" in name
         or "may" in name or "jun" in name or "june" in name)
        and ("26" in name or "2026" in name)
    ):
        print(file_path.name)

NIMF-MONTHLY-PORTFOLIO-30-April-26.xls
NIMF-MONTHLY-PORTFOLIO-30-Jun-26.xls
NIMF-MONTHLY-PORTFOLIO-31-May-26.xls


In [105]:
# Inspect the exact April–June 2026 Nippon India reports.

files_to_check = [
    "NIMF-MONTHLY-PORTFOLIO-30-April-26.xls",
    "NIMF-MONTHLY-PORTFOLIO-31-May-26.xls",
    "NIMF-MONTHLY-PORTFOLIO-30-Jun-26.xls",
]

keywords = [
    "silver", "gold", "commodity future",
    "index future", "derivative", "margin"
]

for filename in files_to_check:
    file_path = nippon_dir / filename

    print(f"\n{'=' * 35}")
    print(f"FILE: {filename}")
    print(f"{'=' * 35}")

    if is_zipfile(file_path):
        with open(file_path, "rb") as stream:
            wb = openpyxl.load_workbook(
                stream, read_only=True, data_only=True
            )
            ws = wb["MF"]

            for row_num, row in enumerate(
                ws.iter_rows(values_only=True), start=1
            ):
                values = tuple(row)
                row_text = " ".join(
                    str(v) for v in values if v is not None
                ).lower()

                if any(word in row_text for word in keywords):
                    print(f"Row {row_num}: {values}")

            wb.close()

    else:
        wb = xlrd.open_workbook(str(file_path))
        ws = wb.sheet_by_name("MF")

        for row_idx in range(ws.nrows):
            values = ws.row_values(row_idx)
            row_text = " ".join(
                str(v) for v in values if v not in (None, "")
            ).lower()

            if any(word in row_text for word in keywords):
                print(f"Row {row_idx + 1}: {values}")


FILE: NIMF-MONTHLY-PORTFOLIO-30-April-26.xls
Row 1: ('RLMF806', 'Nippon India Multi Asset Allocation Fund (An open ended scheme investing in Equity, Debt \nand Exchange Traded Commodity Derivatives and \nGold ETF & Silver ETF)', None, None, None, None, None, 'Index')
Row 183: ('LIGS27', 'INE2R5L15013', 'Liquid Gold Series**', 'CRISIL AAA(SO)', 570.999998, 4449.85, 0.003, 0.0975)
Row 184: ('LIGS22', 'INE1U3R15011', 'Liquid Gold Series**', 'CRISIL AAA(SO)', 119.999995, 1271.42, 0.0009, 0.09615)
Row 201: ('BMGB51ME', 'INF204KB17I5', 'Nippon India ETF Gold Bees', None, 97272043, 120062.88, 0.0815, None)
Row 202: ('149758', 'INF204KC1402', 'Nippon India Silver ETF', None, 20959845, 47683.65, 0.0324, None)
Row 205: ('FNOMGN', None, 'Cash Margin - Derivatives', None, None, 18875, 0.0128, None)
Row 206: ('CCILMGN', None, 'Cash Margin - CCIL', None, None, 116.34, 0.0001, None)
Row 217: (None, 'Details of Commodity Future / Index Future', None, None, None, None, None, None)
Row 219: (None, 'Lon

In [106]:
# Record the April–June 2026 derivatives verification findings.

review_updates = {
    "2026-04": (
        "Commodity-futures section verified: GOLD futures "
        "₹36,266.64 lakh (2.46% NAV); silver futures not identified "
        "in this section. Derivatives-disclosure GOLD margin "
        "₹3,496.098 lakh is margin, not futures market value. "
        "Liquid Gold Series remains a separate security."
    ),
    "2026-05": (
        "Commodity-futures section verified: GOLD futures "
        "₹26,389.40 lakh (1.70% NAV); silver futures not identified "
        "in this section. Derivatives-disclosure GOLD margin "
        "₹2,441.03955 lakh is margin, not futures market value. "
        "Liquid Gold Series remains a separate security."
    ),
    "2026-06": (
        "Commodity-futures section verified: GOLD futures "
        "₹23,375.08 lakh (1.46% NAV); silver futures not identified "
        "in this section. Derivatives-disclosure GOLD margin "
        "₹2,162.11655 lakh is margin, not futures market value. "
        "Liquid Gold Series remains a separate security."
    ),
}

for month, note in review_updates.items():
    mask = nippon_log["Report_Month"].astype(str).eq(month)

    if mask.sum() != 1:
        raise ValueError(
            f"Expected exactly one row for {month}; found {mask.sum()}."
        )

    current = nippon_log.loc[mask, "Review_Notes"].iloc[0]
    current = "" if pd.isna(current) else str(current)

    if note not in current:
        separator = " " if current.strip() else ""
        nippon_log.loc[mask, "Review_Notes"] = current + separator + note

nippon_log.to_csv(output_file, index=False)

print("April–June 2026 review notes saved.")
print("Rows:", len(nippon_log))
print("Unique months:", nippon_log["Report_Month"].nunique())

April–June 2026 review notes saved.
Rows: 33
Unique months: 33


In [107]:
# Recheck the ambiguous April 2024 instrument and its context.

filename = "NIMF-MONTHLY-PORTFOLIO-APRIL-2024.xls"
file_path = nippon_dir / filename

if not file_path.exists():
    print("File not found:", filename)
else:
    if is_zipfile(file_path):
        with open(file_path, "rb") as stream:
            wb = openpyxl.load_workbook(
                stream, read_only=True, data_only=True
            )
            ws = wb["MF"]

            for row_num, row in enumerate(
                ws.iter_rows(values_only=True), start=1
            ):
                if 1 <= row_num <= 12 or 125 <= row_num <= 145:
                    print(f"Row {row_num}: {tuple(row)}")

            wb.close()
    else:
        wb = xlrd.open_workbook(str(file_path))
        ws = wb.sheet_by_name("MF")

        for row_idx in range(ws.nrows):
            if row_idx < 12 or 124 <= row_idx <= 144:
                print(
                    f"Row {row_idx + 1}: "
                    f"{ws.row_values(row_idx)}"
                )

Row 1: ('RLMF806', 'Nippon India Multi Asset Fund (An open ended scheme investing in Equity, Debt and Exchange Traded Commodity Derivatives and Gold ETF)', None, None, None, None, None, 'Index', None, None, None)
Row 2: (None, 'Monthly Portfolio Statement as on April 30,2024', None, None, None, None, None, None, None, None, None)
Row 3: (None, None, None, None, None, None, None, None, None, None, None)
Row 4: (None, 'ISIN', 'Name of the Instrument', 'Industry / Rating', 'Quantity', 'Market/Fair Value\n( Rs. in Lacs)', '% to NAV', 'YIELD', None, None, None)
Row 5: (None, None, 'Derivatives', None, None, None, None, None, None, None, None)
Row 6: (None, None, 'Commodity Options', None, None, None, None, None, None, None, None)
Row 7: ('SILVERN24D72000P', None, 'CRUDEOIL 6800 Put May 2024 Option (1 kg)', None, 50000, 65.15, 0.0002, None, None, None, None)
Row 8: (None, '', 'Total', '', '', 65.15, 0.0002, None, None, None, None)
Row 9: (None, None, None, None, None, None, None, None, None,

In [108]:
# Inspect the May 2024 derivatives and commodity-futures sections.

filename = "NIMF-MONTHLY-PORTFOLIO-May-24.xls"
file_path = nippon_dir / filename

if not file_path.exists():
    print("File not found:", filename)
else:
    if is_zipfile(file_path):
        with open(file_path, "rb") as stream:
            wb = openpyxl.load_workbook(
                stream, read_only=True, data_only=True
            )
            ws = wb["MF"]

            for row_num, row in enumerate(
                ws.iter_rows(values_only=True), start=1
            ):
                if row_num <= 12 or 130 <= row_num <= 150:
                    print(f"Row {row_num}: {tuple(row)}")

            wb.close()
    else:
        wb = xlrd.open_workbook(str(file_path))
        ws = wb.sheet_by_name("MF")

        for row_idx in range(ws.nrows):
            if row_idx < 12 or 129 <= row_idx <= 149:
                print(
                    f"Row {row_idx + 1}: "
                    f"{ws.row_values(row_idx)}"
                )

Row 1: ('RLMF806', 'Nippon India Multi Asset Fund (An open ended scheme investing in Equity, Debt and Exchange Traded Commodity Derivatives and Gold ETF)', None, None, None, None, None, 'Index', None, None)
Row 2: (None, 'Monthly Portfolio Statement as on May 31,2024', None, None, None, None, None, None, None, None)
Row 3: (None, None, None, None, None, None, None, None, None, None)
Row 4: (None, 'ISIN', 'Name of the Instrument', 'Industry / Rating', 'Quantity', 'Market/Fair Value\n( Rs. in Lacs)', '% to NAV', 'YIELD', None, None)
Row 5: (None, None, 'Derivatives', None, None, None, None, None, None, None)
Row 6: (None, None, 'Commodity Options', None, None, None, None, None, None, None)
Row 7: (None, None, 'SILVER 85000 Put June 2024 Option (1 kg)', None, 4980, 19.67, 0.0001, None, None, None)
Row 8: (None, None, 'Total', None, None, 19.67, 0.0001, None, None, None)
Row 9: (None, None, None, None, None, None, None, None, None, None)
Row 10: (None, None, 'Equity & Equity related', None

In [109]:
# Inspect the June 2024 derivatives and commodity-futures sections.

filename = "NIMF-MONTHLY-PORTFOLIO-June-24.xls"
file_path = nippon_dir / filename

if not file_path.exists():
    print("File not found:", filename)
else:
    if is_zipfile(file_path):
        with open(file_path, "rb") as stream:
            wb = openpyxl.load_workbook(
                stream, read_only=True, data_only=True
            )
            ws = wb["MF"]

            for row_num, row in enumerate(
                ws.iter_rows(values_only=True), start=1
            ):
                if row_num <= 12 or 125 <= row_num <= 150:
                    print(f"Row {row_num}: {tuple(row)}")

            wb.close()
    else:
        wb = xlrd.open_workbook(str(file_path))
        ws = wb.sheet_by_name("MF")

        for row_idx in range(ws.nrows):
            if row_idx < 12 or 124 <= row_idx <= 149:
                print(
                    f"Row {row_idx + 1}: "
                    f"{ws.row_values(row_idx)}"
                )

Row 1: ('RLMF806', 'Nippon India Multi Asset Fund (An open ended scheme investing in Equity, Debt and Exchange Traded Commodity Derivatives and Gold ETF)', None, None, None, None, None, 'Index', None, None, None)
Row 2: (None, 'Monthly Portfolio Statement as on June 30,2024', None, None, None, None, None, None, None, None, None)
Row 3: (None, None, None, None, None, None, None, None, None, None, None)
Row 4: (None, 'ISIN', 'Name of the Instrument', 'Industry / Rating', 'Quantity', 'Market/Fair Value\n( Rs. in Lacs)', '% to NAV', 'YIELD', None, None, None)
Row 5: (None, None, 'Equity & Equity related', None, None, None, None, None, None, None, None)
Row 6: (None, None, '(a) Listed / awaiting listing on Stock Exchanges', None, None, None, None, None, None, None, None)
Row 7: ('HDFB03', 'INE040A01034', 'HDFC Bank Limited', 'Banks', 685000, 11534.03, 0.032, None, None, None, None)
Row 8: ('IBCL05', 'INE090A01021', 'ICICI Bank Limited', 'Banks', 850000, 10196.6, 0.0283, None, None, None, No

In [110]:
# Save the June 2024 commodity-futures verification note.

month = "2024-06"
note = (
    "Commodity-futures section verified: silver futures "
    "₹7,655.67 lakh (2.13% NAV), gold futures ₹7,372.95 lakh "
    "(2.05% NAV), and crude-oil futures ₹1,905.40 lakh "
    "(0.53% NAV). No silver option identified in the inspected rows; "
    "blank does not mean zero."
)

mask = nippon_log["Report_Month"].astype(str).eq(month)

if mask.sum() != 1:
    raise ValueError(
        f"Expected exactly one row for {month}; found {mask.sum()}."
    )

current = nippon_log.loc[mask, "Review_Notes"].iloc[0]
current = "" if pd.isna(current) else str(current)

if note not in current:
    separator = " " if current.strip() else ""
    nippon_log.loc[mask, "Review_Notes"] = current + separator + note

nippon_log.to_csv(output_file, index=False)

print("June 2024 review note saved.")
print("Rows:", len(nippon_log))
print("Unique months:", nippon_log["Report_Month"].nunique())

June 2024 review note saved.
Rows: 33
Unique months: 33


In [111]:
# Inspect the March 2025 Nippon India report.

filename = "NIMF-MONTHLY-PORTFOLIO-REPORT-March-24.xls"
file_path = nippon_dir / filename

if not file_path.exists():
    print("File not found:", filename)
else:
    if is_zipfile(file_path):
        with open(file_path, "rb") as stream:
            wb = openpyxl.load_workbook(
                stream, read_only=True, data_only=True
            )
            ws = wb["MF"]

            for row_num, row in enumerate(
                ws.iter_rows(values_only=True), start=1
            ):
                values = tuple(row)
                text = " ".join(
                    str(v) for v in values if v is not None
                ).lower()

                if (
                    row_num <= 12
                    or 125 <= row_num <= 170
                    or "liquid gold series" in text
                ):
                    print(f"Row {row_num}: {values}")

            wb.close()
    else:
        wb = xlrd.open_workbook(str(file_path))
        ws = wb.sheet_by_name("MF")

        for row_idx in range(ws.nrows):
            values = ws.row_values(row_idx)
            text = " ".join(
                str(v) for v in values if v not in (None, "")
            ).lower()

            if (
                row_idx < 12
                or 124 <= row_idx <= 169
                or "liquid gold series" in text
            ):
                print(f"Row {row_idx + 1}: {values}")

Row 1: ['RLMF806', 'Nippon India Multi Asset Fund (An open ended scheme investing in Equity, Debt and Exchange Traded Commodity Derivatives and Gold ETF)', '', '', '', '', '', 'Index']
Row 2: ['', 'Monthly Portfolio Statement as on March 31,2024', '', '', '', '', '', '']
Row 3: ['', '', '', '', '', '', '', '']
Row 4: ['', 'ISIN', 'Name of the Instrument', 'Industry / Rating', 'Quantity', 'Market/Fair Value\n( Rs. in Lacs)', '% to NAV', 'YIELD']
Row 5: ['', '', 'Derivatives', '', '', '', '', '']
Row 6: ['', '', 'Commodity Options', '', '', '', '', '']
Row 7: ['SILVERN24D72000P', '', 'SILVER 72000 Put April 2024 Option (1 kg)', '', 1530.0, 5.68, '$0.00%', '']
Row 8: ['', '', 'Total', '', '', 5.68, 0.0, '']
Row 9: ['', '', 'Equity & Equity related', '', '', '', '', '']
Row 10: ['', '', '(a) Listed / awaiting listing on Stock Exchanges', '', '', '', '', '']
Row 11: ['HDFB03', 'INE040A01034', 'HDFC Bank Limited', 'Banks', 660000.0, 9556.14, 0.0329, '']
Row 12: ['IBCL05', 'INE090A01021', 'IC

In [112]:
# Find the exact March report filenames in the Nippon folder.

for file_path in sorted(nippon_dir.iterdir()):
    name = file_path.name.lower()

    if file_path.is_file() and (
        "march" in name or "mar-" in name or "mar_" in name
    ):
        print(file_path.name)

NIMF-MONTHLY-PORTFOLIO-31-Mar-25.xls
NIMF-MONTHLY-PORTFOLIO-31-Mar-26.xls
NIMF-MONTHLY-PORTFOLIO-REPORT-March-24.xls


In [113]:
from zipfile import is_zipfile

filename = "NIMF-MONTHLY-PORTFOLIO-31-Mar-25.xls"
file_path = nippon_dir / filename

if not file_path.exists():
    raise FileNotFoundError(f"File not found: {file_path}")

if is_zipfile(file_path):
    with open(file_path, "rb") as stream:
        wb = openpyxl.load_workbook(
            stream, read_only=True, data_only=True
        )
        ws = wb["MF"]

        for row_num, row in enumerate(
            ws.iter_rows(values_only=True), start=1
        ):
            values = tuple(row)
            text = " ".join(
                str(v) for v in values if v is not None
            ).lower()

            if (
                row_num <= 5
                or 125 <= row_num <= 175
                or "liquid gold series" in text
                or (
                    "silver" in text
                    and "benchmark" not in text
                )
            ):
                print(f"Row {row_num}: {values}")

        wb.close()

else:
    wb = xlrd.open_workbook(str(file_path))
    ws = wb.sheet_by_name("MF")

    for row_idx in range(ws.nrows):
        values = ws.row_values(row_idx)
        text = " ".join(
            str(v) for v in values if v not in (None, "")
        ).lower()

        if (
            row_idx < 5
            or 124 <= row_idx <= 174
            or "liquid gold series" in text
            or (
                "silver" in text
                and "benchmark" not in text
            )
        ):
            print(f"Row {row_idx + 1}: {values}")

Row 1: ('RLMF806', 'Nippon India Multi Asset Allocation Fund (An open ended scheme investing in Equity, Debt \nand Exchange Traded Commodity Derivatives and \nGold ETF & Silver ETF)', None, None, None, None, None, 'Index')
Row 2: (None, 'Monthly Portfolio Statement as on March 31,2025', None, None, None, None, None, None)
Row 3: (None, None, None, None, None, None, None, None)
Row 4: (None, 'ISIN', 'Name of the Instrument', 'Industry / Rating', 'Quantity', 'Market/Fair Value\n( Rs. in Lacs)', '% to NAV', 'YIELD')
Row 5: (None, None, 'Equity & Equity related', None, None, None, None, None)
Row 125: ('SBAI203', 'INE062A08256', '6.24% State Bank of India**', 'CRISIL AAA', 50, 496.51, 0.0009, 0.063845)
Row 126: (None, None, 'Zero Coupon Bonds', None, None, None, None, None)
Row 127: ('NHIT23', 'INE0H7R07066', 'National Highways Infra Trust**', 'CARE AAA', 1750, 901.48, 0.0017, 0.07799)
Row 128: ('NHIT24', 'INE0H7R07058', 'National Highways Infra Trust**', 'CARE AAA', 1750, 834.34, 0.0016, 

In [114]:
month = "2025-03"

note = (
    "March 2025 report verified (31-Mar-2025): Nippon India ETF Gold "
    "BeES ₹63,971.80 lakh (12.00% NAV); Nippon India Silver ETF "
    "₹16,464.34 lakh (3.09% NAV); gold futures ₹4,354.42 lakh "
    "(0.82% NAV). Silver futures not identified in the displayed "
    "commodity-futures section; blank does not mean zero. Liquid Gold "
    "Series ₹4,212.85 lakh (0.79% NAV) is a separate CRISIL AAA(SO)-"
    "rated security and is not classified as a Gold ETF. Benchmark "
    "allocations excluded from actual holdings."
)

mask = nippon_log["Report_Month"].astype(str).eq(month)

if mask.sum() != 1:
    raise ValueError(
        f"Expected exactly one row for {month}; found {mask.sum()}."
    )

current = nippon_log.loc[mask, "Review_Notes"].iloc[0]
current = "" if pd.isna(current) else str(current)

if note not in current:
    separator = " " if current.strip() else ""
    nippon_log.loc[mask, "Review_Notes"] = current + separator + note

nippon_log.to_csv(output_file, index=False)

print("March 2025 review note saved.")
print("Rows:", len(nippon_log))
print("Unique months:", nippon_log["Report_Month"].nunique())


March 2025 review note saved.
Rows: 33
Unique months: 33


In [115]:
# List months that still need futures verification.

futures_cols = [
    "Gold_Futures_Value_Lakh",
    "Silver_Futures_Value_Lakh"
]

for _, row in nippon_log.iterrows():
    missing = [
        col for col in futures_cols
        if pd.isna(row[col])
    ]

    if missing:
        print(
            f"\nMonth: {row['Report_Month']}"
            f"\nBlank fields: {', '.join(missing)}"
            f"\nReview notes: {row['Review_Notes']}"
        )


Month: 2024-01
Blank fields: Gold_Futures_Value_Lakh
Review notes: No direct silver ETF identified in keyword search; verify before treating as absent. Gold futures value blank: no matching gold futures identified in the inspected source section; blank does not mean zero. Commodity-futures section verified: silver futures total ₹1,475.29 lakh (0.58% NAV); no gold futures entry appears in this section. Gold futures remains unrecorded, not zero.

Month: 2024-02
Blank fields: Gold_Futures_Value_Lakh
Review notes: No direct silver ETF identified in keyword search; verify before treating as absent. Gold futures value blank: no matching gold futures identified in the inspected source section; blank does not mean zero. Commodity-futures section verified: silver futures ₹2,587.43 lakh (0.96% NAV) and crude-oil futures ₹481.96 lakh. No gold futures entry appears in this section; gold futures remains unrecorded, not zero.

Month: 2024-03
Blank fields: Gold_Futures_Value_Lakh
Review notes: No di

In [116]:
# Search January–March 2024 Nippon reports for gold-related entries.

from zipfile import is_zipfile

files_to_check = {
    "2024-01": "MONTHLY-PORTFOLIO-JAN-2024.xls",
    "2024-02": "MONTHLY-PORTFOLIO-REPORT-FEB-24.xls",
    "2024-03": "NIMF-MONTHLY-PORTFOLIO-REPORT-March-24.xls",
}

for month, filename in files_to_check.items():
    file_path = nippon_dir / filename
    print(f"\n{'=' * 70}\n{month}: {filename}")

    if not file_path.exists():
        print("FILE NOT FOUND")
        continue

    if is_zipfile(file_path):
        with open(file_path, "rb") as stream:
            wb = openpyxl.load_workbook(
                stream, read_only=True, data_only=True
            )
            ws = wb["MF"]

            for row_num, row in enumerate(
                ws.iter_rows(values_only=True), start=1
            ):
                values = tuple(row)
                text = " ".join(
                    str(v) for v in values if v is not None
                ).lower()

                if "gold" in text and "benchmark" not in text:
                    print(f"Row {row_num}: {values}")

            wb.close()
    else:
        wb = xlrd.open_workbook(str(file_path))
        ws = wb.sheet_by_name("MF")

        for row_idx in range(ws.nrows):
            values = ws.row_values(row_idx)
            text = " ".join(
                str(v) for v in values if v not in (None, "")
            ).lower()

            if "gold" in text and "benchmark" not in text:
                print(f"Row {row_idx + 1}: {values}")


2024-01: MONTHLY-PORTFOLIO-JAN-2024.xls
Row 1: ('RLMF806', 'Nippon India Multi Asset Fund (An open ended scheme investing in Equity, Debt and Exchange Traded Commodity Derivatives and Gold ETF)', None, None, None, None, None, 'Index', None, None, None)
Row 108: ('BMGB51ME', 'INF204KB17I5', 'Nippon India ETF Gold Bees', None, 62360000, 33269.06, 0.1308, None, None, None, None)

2024-02: MONTHLY-PORTFOLIO-REPORT-FEB-24.xls
Row 1: ('RLMF806', 'Nippon India Multi Asset Fund (An open ended scheme investing in Equity, Debt and Exchange Traded Commodity Derivatives and Gold ETF)', None, None, None, None, None, 'Index')
Row 104: ('BMGB51ME', 'INF204KB17I5', 'Nippon India ETF Gold Bees', None, 65580197, 34613.23, 0.1281, None)

2024-03: NIMF-MONTHLY-PORTFOLIO-REPORT-March-24.xls
Row 1: ['RLMF806', 'Nippon India Multi Asset Fund (An open ended scheme investing in Equity, Debt and Exchange Traded Commodity Derivatives and Gold ETF)', '', '', '', '', '', 'Index']
Row 109: ['BMGB51ME', 'INF204KB17

In [117]:
gold_search_notes = {
    "2024-01": (
        "Additional gold-keyword search completed across the MF sheet: "
        "the only gold-related match outside the benchmark was Nippon "
        "India ETF Gold BeES, ₹33,269.06 lakh (13.08% NAV). No separate "
        "gold-futures entry identified in the search results; blank "
        "remains unrecorded, not zero."
    ),
    "2024-02": (
        "Additional gold-keyword search completed across the MF sheet: "
        "the only gold-related match outside the benchmark was Nippon "
        "India ETF Gold BeES, ₹34,613.23 lakh (12.81% NAV). No separate "
        "gold-futures entry identified in the search results; blank "
        "remains unrecorded, not zero."
    ),
    "2024-03": (
        "Additional gold-keyword search completed across the MF sheet: "
        "the only gold-related match outside the benchmark was Nippon "
        "India ETF Gold BeES, ₹37,408.00 lakh (12.88% NAV). No separate "
        "gold-futures entry identified in the search results; blank "
        "remains unrecorded, not zero."
    ),
}

for month, note in gold_search_notes.items():
    mask = nippon_log["Report_Month"].astype(str).eq(month)

    if mask.sum() != 1:
        raise ValueError(
            f"Expected exactly one row for {month}; found {mask.sum()}."
        )

    current = nippon_log.loc[mask, "Review_Notes"].iloc[0]
    current = "" if pd.isna(current) else str(current)

    if note not in current:
        separator = " " if current.strip() else ""
        nippon_log.loc[mask, "Review_Notes"] = (
            current + separator + note
        )

nippon_log.to_csv(output_file, index=False)

print("January–March 2024 gold-search review notes saved.")
print("Rows:", len(nippon_log))
print("Unique months:", nippon_log["Report_Month"].nunique())

January–March 2024 gold-search review notes saved.
Rows: 33
Unique months: 33


In [118]:
# Final structural and review-flag check for Nippon India.

expected_months = pd.period_range(
    start="2024-01",
    end="2026-09",
    freq="M"
).astype(str).tolist()

actual_months = nippon_log["Report_Month"].astype(str).tolist()

print("Total rows:", len(nippon_log))
print("Unique months:", nippon_log["Report_Month"].nunique())
print("Duplicate months:", nippon_log["Report_Month"].duplicated().sum())
print("Missing months:", sorted(set(expected_months) - set(actual_months)))
print("Unexpected months:", sorted(set(actual_months) - set(expected_months)))

print("\nMissing values in key ETF/futures fields:")
key_cols = [
    "Gold_ETF_Value_Lakh",
    "Silver_ETF_Value_Lakh",
    "Gold_Futures_Value_Lakh",
    "Silver_Futures_Value_Lakh",
    "Silver_Options_Value_Lakh",
]
print(nippon_log[key_cols].isna().sum())

print("\nReview notes needing attention:")
keywords = [
    "REVIEW REQUIRED",
    "malformed",
    "Liquid Gold Series",
    "verify before treating as absent",
]

for _, row in nippon_log.iterrows():
    note = "" if pd.isna(row["Review_Notes"]) else str(row["Review_Notes"])
    matched = [word for word in keywords if word.lower() in note.lower()]

    if matched:
        print(f"\n{row['Report_Month']} — {', '.join(matched)}")
        print(note)

Total rows: 33
Unique months: 33
Duplicate months: 0
Missing months: []
Unexpected months: []

Missing values in key ETF/futures fields:
Gold_ETF_Value_Lakh           0
Silver_ETF_Value_Lakh         6
Gold_Futures_Value_Lakh       5
Silver_Futures_Value_Lakh     8
Silver_Options_Value_Lakh    29
dtype: int64

Review notes needing attention:

2024-01 — verify before treating as absent
No direct silver ETF identified in keyword search; verify before treating as absent. Gold futures value blank: no matching gold futures identified in the inspected source section; blank does not mean zero. Commodity-futures section verified: silver futures total ₹1,475.29 lakh (0.58% NAV); no gold futures entry appears in this section. Gold futures remains unrecorded, not zero. Additional gold-keyword search completed across the MF sheet: the only gold-related match outside the benchmark was Nippon India ETF Gold BeES, ₹33,269.06 lakh (13.08% NAV). No separate gold-futures entry identified in the search re

In [119]:
from zipfile import is_zipfile

filename = "NIMF-MONTHLY-PORTFOLIO-APRIL-2024.xls"
file_path = nippon_dir / filename

if not file_path.exists():
    raise FileNotFoundError(f"File not found: {file_path}")

if is_zipfile(file_path):
    with open(file_path, "rb") as stream:
        wb = openpyxl.load_workbook(
            stream, read_only=True, data_only=True
        )
        ws = wb["MF"]

        for row_num, row in enumerate(
            ws.iter_rows(values_only=True), start=1
        ):
            values = tuple(row)
            text = " ".join(
                str(v) for v in values if v is not None
            ).lower()

            if row_num <= 20 or "silver" in text or "crudeoil" in text:
                print(f"Row {row_num}: {values}")

        wb.close()
else:
    wb = xlrd.open_workbook(str(file_path))
    ws = wb.sheet_by_name("MF")

    for row_idx in range(ws.nrows):
        values = ws.row_values(row_idx)
        text = " ".join(
            str(v) for v in values if v not in (None, "")
        ).lower()

        if row_idx < 20 or "silver" in text or "crudeoil" in text:
            print(f"Row {row_idx + 1}: {values}")

Row 1: ('RLMF806', 'Nippon India Multi Asset Fund (An open ended scheme investing in Equity, Debt and Exchange Traded Commodity Derivatives and Gold ETF)', None, None, None, None, None, 'Index', None, None, None)
Row 2: (None, 'Monthly Portfolio Statement as on April 30,2024', None, None, None, None, None, None, None, None, None)
Row 3: (None, None, None, None, None, None, None, None, None, None, None)
Row 4: (None, 'ISIN', 'Name of the Instrument', 'Industry / Rating', 'Quantity', 'Market/Fair Value\n( Rs. in Lacs)', '% to NAV', 'YIELD', None, None, None)
Row 5: (None, None, 'Derivatives', None, None, None, None, None, None, None, None)
Row 6: (None, None, 'Commodity Options', None, None, None, None, None, None, None, None)
Row 7: ('SILVERN24D72000P', None, 'CRUDEOIL 6800 Put May 2024 Option (1 kg)', None, 50000, 65.15, 0.0002, None, None, None, None)
Row 8: (None, '', 'Total', '', '', 65.15, 0.0002, None, None, None, None)
Row 9: (None, None, None, None, None, None, None, None, None,

In [120]:
month = "2024-04"

note = (
    "April 2024 option discrepancy reviewed against the original "
    "report dated 30-Apr-2024. Instrument code is SILVERN24D72000P, "
    "but the description reads 'CRUDEOIL 6800 Put May 2024 Option "
    "(1 kg)'. Reported value ₹65.15 lakh and % to NAV 0.02%. "
    "Classification remains UNVERIFIED: do not include this option "
    "as confirmed silver or crude-oil exposure without independent "
    "source evidence. Separately reported silver futures ₹3,856.59 "
    "lakh (1.24% NAV) and crude-oil futures ₹3,555.76 lakh (1.14% NAV)."
)

mask = nippon_log["Report_Month"].astype(str).eq(month)

if mask.sum() != 1:
    raise ValueError(
        f"Expected exactly one row for {month}; found {mask.sum()}."
    )

current = nippon_log.loc[mask, "Review_Notes"].iloc[0]
current = "" if pd.isna(current) else str(current)

if note not in current:
    separator = " " if current.strip() else ""
    nippon_log.loc[mask, "Review_Notes"] = current + separator + note

nippon_log.to_csv(output_file, index=False)

print("April 2024 option discrepancy documented.")
print("Rows:", len(nippon_log))
print("Unique months:", nippon_log["Report_Month"].nunique())
print(
    nippon_log.loc[
        mask,
        [
            "Report_Month",
            "Silver_Options_Value_Lakh",
            "Silver_Options_NAV_Pct",
            "Silver_Options_Review",
        ],
    ].to_string(index=False)
)

April 2024 option discrepancy documented.
Rows: 33
Unique months: 33
Report_Month  Silver_Options_Value_Lakh  Silver_Options_NAV_Pct                                                                                                                                                                                                                                                                           Silver_Options_Review
     2024-04                      65.15                    0.02 REVIEW REQUIRED: source row has instrument code SILVERN24D72000P but description CRUDEOIL 6800 Put May 2024 Option (1 kg). Value ₹65.15 lakh and reported NAV percentage 0.02% retained as an unverified candidate; exclude from confirmed silver-option exposure until independently verified.


In [121]:
# Identify all months with Liquid Gold Series review notes.

mask = nippon_log["Review_Notes"].fillna("").str.contains(
    "Liquid Gold Series",
    case=False,
    regex=False
)

print(
    nippon_log.loc[
        mask,
        ["Report_Month", "Review_Notes"]
    ].to_string(index=False)
)

print("\nMonths flagged:", int(mask.sum()))

Report_Month                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   Review_Notes
     2025-03 Gold and silver

In [122]:
# Add a consistent classification note to months mentioning
# Liquid Gold Series. Preserve all existing review notes.

liquid_gold_note = (
    "CLASSIFICATION: Liquid Gold Series is recorded separately "
    "from Gold ETF holdings. The reports identify it as a "
    "CRISIL AAA(SO)-rated security. Do not add its value to "
    "Gold ETF totals based on the security name alone. "
    "Retain it as a separate security pending any further "
    "prospectus-level verification."
)

mask = nippon_log["Review_Notes"].fillna("").str.contains(
    "Liquid Gold Series",
    case=False,
    regex=False
)

for idx in nippon_log.index[mask]:
    current = nippon_log.at[idx, "Review_Notes"]
    current = "" if pd.isna(current) else str(current)

    if liquid_gold_note not in current:
        nippon_log.at[idx, "Review_Notes"] = (
            current.rstrip() + " " + liquid_gold_note
        ).strip()

nippon_log.to_csv(output_file, index=False)

print("Liquid Gold Series classification notes updated.")
print("Rows:", len(nippon_log))
print("Unique months:", nippon_log["Report_Month"].nunique())
print("Months updated:", int(mask.sum()))

Liquid Gold Series classification notes updated.
Rows: 33
Unique months: 33
Months updated: 6


In [123]:
# Find outdated generic review notes before editing them.

stale_phrases = [
    "verify before treating as absent",
    "verify before treating as zero"
]

for phrase in stale_phrases:
    mask = nippon_log["Review_Notes"].fillna("").str.contains(
        phrase,
        case=False,
        regex=False
    )

    print(f"\nPhrase: {phrase}")
    print("Count:", int(mask.sum()))

    if mask.any():
        print(
            nippon_log.loc[
                mask,
                ["Report_Month", "Review_Notes"]
            ].to_string(index=False)
        )


Phrase: verify before treating as absent
Count: 14
Report_Month                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        

In [124]:
# Remove the outdated generic wording only.
# Preserve all detailed review notes and recorded values.

phrase = "verify before treating as absent"

mask = nippon_log["Review_Notes"].fillna("").str.contains(
    phrase,
    case=False,
    regex=False
)

print("Rows containing phrase before update:", int(mask.sum()))

nippon_log["Review_Notes"] = (
    nippon_log["Review_Notes"]
    .fillna("")
    .str.replace(phrase, "", case=False, regex=False)
    .str.replace(r"\s*;\s*;", ";", regex=True)
    .str.replace(r"^\s*[;,.]\s*", "", regex=True)
    .str.replace(r"\s{2,}", " ", regex=True)
    .str.strip()
)

nippon_log.to_csv(output_file, index=False)

remaining = nippon_log["Review_Notes"].str.contains(
    phrase,
    case=False,
    regex=False
)

print("Rows still containing phrase:", int(remaining.sum()))
print("Total rows:", len(nippon_log))
print("Unique months:", nippon_log["Report_Month"].nunique())
print("Saved to:", output_file)

Rows containing phrase before update: 14
Rows still containing phrase: 0
Total rows: 33
Unique months: 33
Saved to: c:\Users\rocks\OneDrive\Desktop\AMC Project Data\02_CLEAN_DATA\Nippon_India_Monthly_Holdings_Log.csv


In [125]:
# Final structural and missing-value validation for Nippon India.

expected_months = pd.period_range(
    start="2024-01",
    end="2026-09",
    freq="M"
).astype(str)

actual_months = nippon_log["Report_Month"].astype(str)

key_columns = [
    "Gold_ETF_Value_Lakh",
    "Gold_ETF_NAV_Pct",
    "Silver_ETF_Value_Lakh",
    "Silver_ETF_NAV_Pct",
    "Gold_Futures_Value_Lakh",
    "Gold_Futures_NAV_Pct",
    "Silver_Futures_Value_Lakh",
    "Silver_Futures_NAV_Pct",
    "Silver_Options_Value_Lakh",
    "Silver_Options_NAV_Pct",
]

print("Total rows:", len(nippon_log))
print("Unique months:", actual_months.nunique())
print("Duplicate month rows:", int(actual_months.duplicated().sum()))
print("Missing months:", sorted(set(expected_months) - set(actual_months)))
print("Unexpected months:", sorted(set(actual_months) - set(expected_months)))

print("\nMissing values in key columns:")
print(nippon_log[key_columns].isna().sum().to_string())

print("\nExplicit zeros in key columns:")
print(
    (nippon_log[key_columns].eq(0).sum())
    .to_string()
)

Total rows: 33
Unique months: 33
Duplicate month rows: 0
Missing months: []
Unexpected months: []

Missing values in key columns:
Gold_ETF_Value_Lakh           0
Gold_ETF_NAV_Pct              0
Silver_ETF_Value_Lakh         6
Silver_ETF_NAV_Pct            6
Gold_Futures_Value_Lakh       5
Gold_Futures_NAV_Pct          5
Silver_Futures_Value_Lakh     8
Silver_Futures_NAV_Pct        8
Silver_Options_Value_Lakh    29
Silver_Options_NAV_Pct       30

Explicit zeros in key columns:
Gold_ETF_Value_Lakh          0
Gold_ETF_NAV_Pct             0
Silver_ETF_Value_Lakh        0
Silver_ETF_NAV_Pct           0
Gold_Futures_Value_Lakh      0
Gold_Futures_NAV_Pct         0
Silver_Futures_Value_Lakh    0
Silver_Futures_NAV_Pct       0
Silver_Options_Value_Lakh    0
Silver_Options_NAV_Pct       0


In [126]:
# Inspect the months with missing exposure values and their explanations.

check_columns = [
    "Silver_ETF_Value_Lakh",
    "Gold_Futures_Value_Lakh",
    "Silver_Futures_Value_Lakh",
    "Silver_Options_Value_Lakh",
    "Silver_Options_NAV_Pct",
]

for col in check_columns:
    mask = nippon_log[col].isna()

    print(f"\n--- {col}: {int(mask.sum())} blank values ---")

    print(
        nippon_log.loc[
            mask,
            ["Report_Month", col, "Review_Notes"]
        ].to_string(index=False)
    )


--- Silver_ETF_Value_Lakh: 6 blank values ---
Report_Month  Silver_ETF_Value_Lakh                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   Review_Notes
     2024-01                    NaN                                No direct silver ETF identified in keyword search; . Gold futures value blank: no matching gold futures identified in the inspected source section; blank does not mean zero. Commodity-futures sec

In [127]:
# Clean punctuation left behind by removing stale phrases.
# No holdings, percentages, or missing values are changed.

notes = nippon_log["Review_Notes"].fillna("")

notes = notes.str.replace(r";\s*\.", ".", regex=True)
notes = notes.str.replace(r";\s*;", ";", regex=True)
notes = notes.str.replace(r"\.\s*;", ".", regex=True)
notes = notes.str.replace(r"(?:;\s*\.)+", ".", regex=True)
notes = notes.str.replace(r"\s{2,}", " ", regex=True)
notes = notes.str.strip()

nippon_log["Review_Notes"] = notes
nippon_log.to_csv(output_file, index=False)

print("Review-note punctuation cleaned.")
print("Total rows:", len(nippon_log))
print("Unique months:", nippon_log["Report_Month"].nunique())

# Check for common leftover punctuation patterns
for pattern in [r";\s*\.", r";\s*;", r"\.\s*;"]:
    count = nippon_log["Review_Notes"].str.contains(
        pattern, regex=True, na=False
    ).sum()
    print(f"Remaining matches for {pattern}: {int(count)}")

Review-note punctuation cleaned.
Total rows: 33
Unique months: 33
Remaining matches for ;\s*\.: 0
Remaining matches for ;\s*;: 0
Remaining matches for \.\s*;: 0


In [128]:
# Inspect the four key silver-options months before final documentation.

months_to_check = [
    "2024-01",
    "2024-03",
    "2024-04",
    "2024-10",
]

columns_to_show = [
    "Report_Month",
    "Silver_Options_Value_Lakh",
    "Silver_Options_NAV_Pct",
    "Silver_Options_Review",
    "Review_Notes",
]

print(
    nippon_log.loc[
        nippon_log["Report_Month"].astype(str).isin(months_to_check),
        columns_to_show,
    ].to_string(index=False)
)

Report_Month  Silver_Options_Value_Lakh  Silver_Options_NAV_Pct                                                                                                                                                                                                                                                                           Silver_Options_Review                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 Review_N

In [129]:
from pathlib import Path

nippon_folder = PROJECT_DIR / "01_RAW_DATA" / "Nippon_India"

october_files = [
    p for p in nippon_folder.rglob("*")
    if p.is_file()
    and "OCT" in p.name.upper()
    and "2024" in p.name.upper()
]

print("October 2024 files found:", len(october_files))

for p in october_files:
    print("\nFile:", p.name)
    print("Extension:", p.suffix)

October 2024 files found: 0


In [130]:
from pathlib import Path

nippon_folder = PROJECT_DIR / "01_RAW_DATA" / "Nippon_India"

for p in sorted(nippon_folder.rglob("*")):
    if p.is_file():
        print(p.name)

MONTHLY-PORTFOLIO-JAN-2024.xls
MONTHLY-PORTFOLIO-REPORT-FEB-24.xls
NIMF-MONTHLY-PORTFOLIO-28-Feb-25.xls
NIMF-MONTHLY-PORTFOLIO-28-Feb-26.xls
NIMF-MONTHLY-PORTFOLIO-30-April-25.xls
NIMF-MONTHLY-PORTFOLIO-30-April-26.xls
NIMF-MONTHLY-PORTFOLIO-30-Jun-26.xls
NIMF-MONTHLY-PORTFOLIO-30-June-25.xls
NIMF-MONTHLY-PORTFOLIO-30-Nov-2024.xls
NIMF-MONTHLY-PORTFOLIO-30-Sep-24.xls
NIMF-MONTHLY-PORTFOLIO-30-Sep-25.xls
NIMF-MONTHLY-PORTFOLIO-30-Sep-26.xls
NIMF-MONTHLY-PORTFOLIO-31-Aug-25.xls
NIMF-MONTHLY-PORTFOLIO-31-Aug-26.xls
NIMF-MONTHLY-PORTFOLIO-31-Dec-24.xls
NIMF-MONTHLY-PORTFOLIO-31-Dec-25.xls
NIMF-MONTHLY-PORTFOLIO-31-Jan-26.xls
NIMF-MONTHLY-PORTFOLIO-31-July-24.xls
NIMF-MONTHLY-PORTFOLIO-31-July-25.xls
NIMF-MONTHLY-PORTFOLIO-31-July-26.xls
NIMF-MONTHLY-PORTFOLIO-31-Mar-25.xls
NIMF-MONTHLY-PORTFOLIO-31-Mar-26.xls
NIMF-MONTHLY-PORTFOLIO-31-May-25.xls
NIMF-MONTHLY-PORTFOLIO-31-May-26.xls
NIMF-MONTHLY-PORTFOLIO-31-Oct-24.xls
NIMF-MONTHLY-PORTFOLIO-31-Oct-25.xls
NIMF-MONTHLY-PORTFOLIO-APRIL-2024.x

In [131]:
from pathlib import Path
import pandas as pd
import zipfile
import xlrd
from io import BytesIO

oct_file = nippon_folder / "NIMF-MONTHLY-PORTFOLIO-31-Oct-24.xls"

with open(oct_file, "rb") as f:
    file_bytes = f.read()

if zipfile.is_zipfile(BytesIO(file_bytes)):
    # The .xls file is actually an XLSX-format workbook.
    xls = pd.ExcelFile(BytesIO(file_bytes), engine="openpyxl")
else:
    # Genuine legacy XLS workbook.
    xls = pd.ExcelFile(BytesIO(file_bytes), engine="xlrd")

print("Report:", oct_file.name)
print("\nWorksheet names:")
for sheet in xls.sheet_names:
    print("-", sheet)

Report: NIMF-MONTHLY-PORTFOLIO-31-Oct-24.xls

Worksheet names:
- INDEX
- GF
- GS
- IP
- IF
- ST
- BF
- MI
- PS
- CP
- PH
- ME
- NE
- NI
- EO
- LF
- FR
- SD
- SE
- SH
- LQ
- TS
- LE
- IV
- LP
- IM
- DC
- UB
- K5
- 9B
- EA
- QP
- GL
- NF
- SC
- SF
- AF
- GD
- CE
- CF
- RD
- CB
- JE
- SX
- RW
- RG
- BP
- ES
- NX
- UE
- LT
- NL
- NB
- JZ
- PU
- GB
- LB
- BB
- SB
- HS
- IB
- CC
- ON
- NM
- RJ
- G6
- SN
- NZ
- MF
- NS
- PP
- AL
- NP
- NV
- NY
- NG
- NH
- LC
- HG
- ET
- NO
- SI
- SO
- HK
- CN
- NN
- W5
- ND
- NJ
- NK
- NQ
- W8
- NU
- NW
- U7
- U8
- IT
- IX
- BX
- EW
- MT
- IJ
- ID


In [132]:
keywords = ["SILVER", "OPTION", "PUT"]

for sheet in xls.sheet_names:
    try:
        df = pd.read_excel(xls, sheet_name=sheet, header=None)
        text = df.astype(str)

        mask = text.apply(
            lambda col: col.str.contains(
                "|".join(keywords), case=False, regex=True, na=False
            )
        ).any(axis=1)

        matches = df.loc[mask]

        if not matches.empty:
            print(f"\n{'=' * 25} SHEET: {sheet} {'=' * 25}")
            print(matches.to_string(index=True, header=False))
    except Exception as e:
        print(f"Could not inspect sheet {sheet}: {e}")


========================= SHEET: INDEX =========================
56  LB  NIPPON INDIA ETF NIFTY 1D RATE LIQUID BEES (An open ended liquid scheme, listed on the Exchange in the form of an ETF, investing in Tri-Party Repo on G-Sec or T-bills /Repo & Reverse Repo with daily Dividend and compulsory Reinvestment of Income Distribution cum capital withdrawal option. Relatively Low Interest Rate Risk and Relatively Low Credit Risk.)
68  MF                                                                                                                                                                                                     Nippon India Multi Asset Allocation Fund (An open ended scheme investing in Equity, Debt \nand Exchange Traded Commodity Derivatives and \nGold ETF & Silver ETF)
81  SI                                                                                                                                                 Nippon India Silver ETF (An open ended scheme, listed

In [133]:
oct_mask = nippon_log["Report_Month"].astype(str).eq("2024-10")

nippon_log.loc[
    oct_mask, "Silver_Options_Value_Lakh"
] = 29.63

nippon_log.loc[
    oct_mask, "Silver_Options_NAV_Pct"
] = 0.01

nippon_log.loc[
    oct_mask, "Silver_Options_Review"
] = (
    "Silver 90000 Put November 2024 Option (1 kg) identified "
    "in the MF worksheet; value ₹29.63 lakh and source % to NAV 0.01%."
)

nippon_log.loc[
    oct_mask, "Review_Notes"
] = (
    nippon_log.loc[oct_mask, "Review_Notes"].fillna("").str.rstrip()
    + " October 2024 source verification: MF worksheet confirms "
    "GOLD 75000 Put November 2024 Option (10 Gm), ₹43.99 lakh "
    "(0.01% NAV), and SILVER 90000 Put November 2024 Option "
    "(1 kg), ₹29.63 lakh (0.01% NAV). Gold and silver options "
    "are recorded separately."
).str.strip()

nippon_log.to_csv(output_file, index=False)

print(
    nippon_log.loc[
        oct_mask,
        [
            "Report_Month",
            "Silver_Options_Value_Lakh",
            "Silver_Options_NAV_Pct",
            "Silver_Options_Review",
            "Review_Notes",
        ],
    ].to_string(index=False)
)

Report_Month  Silver_Options_Value_Lakh  Silver_Options_NAV_Pct                                                                                                     Silver_Options_Review                                                                                                                                                                                                                                              Review_Notes
     2024-10                      29.63                    0.01 Silver 90000 Put November 2024 Option (1 kg) identified in the MF worksheet; value ₹29.63 lakh and source % to NAV 0.01%. October 2024 source verification: MF worksheet confirms GOLD 75000 Put November 2024 Option (10 Gm), ₹43.99 lakh (0.01% NAV), and SILVER 90000 Put November 2024 Option (1 kg), ₹29.63 lakh (0.01% NAV). Gold and silver options are recorded separately.


In [134]:
# Audit silver-options documentation across all 33 months.

options_audit = nippon_log[
    [
        "Report_Month",
        "Silver_Options_Value_Lakh",
        "Silver_Options_NAV_Pct",
        "Silver_Options_Review",
    ]
].copy()

options_audit["Value_Recorded"] = (
    options_audit["Silver_Options_Value_Lakh"].notna()
)
options_audit["Review_Recorded"] = (
    options_audit["Silver_Options_Review"]
    .fillna("")
    .str.strip()
    .ne("")
)

print("MONTHLY SILVER-OPTIONS AUDIT")
print(options_audit.to_string(index=False))

print("\nSUMMARY")
print("Total months:", len(options_audit))
print("Months with a recorded value:", int(options_audit["Value_Recorded"].sum()))
print("Months with a review note:", int(options_audit["Review_Recorded"].sum()))
print(
    "Value recorded but review missing:",
    int((options_audit["Value_Recorded"] & ~options_audit["Review_Recorded"]).sum())
)

MONTHLY SILVER-OPTIONS AUDIT
Report_Month  Silver_Options_Value_Lakh  Silver_Options_NAV_Pct                                                                                                                                                                                                                                                                           Silver_Options_Review  Value_Recorded  Review_Recorded
     2024-01                      17.24                    0.01                                                                                                                                                                                                                                           Silver call option identified; source % to NAV 0.01%.            True             True
     2024-02                        NaN                     NaN                                                                                                                                          

In [135]:
july_file = nippon_folder / "NIMF-MONTHLY-PORTFOLIO-31-July-24.xls"

with open(july_file, "rb") as f:
    july_bytes = f.read()

if zipfile.is_zipfile(BytesIO(july_bytes)):
    july_xls = pd.ExcelFile(BytesIO(july_bytes), engine="openpyxl")
else:
    july_xls = pd.ExcelFile(BytesIO(july_bytes), engine="xlrd")

print("Report:", july_file.name)

for sheet in july_xls.sheet_names:
    try:
        df = pd.read_excel(july_xls, sheet_name=sheet, header=None)
        mask = df.astype(str).apply(
            lambda col: col.str.contains(
                r"SILVER|OPTION|PUT|CALL",
                case=False,
                regex=True,
                na=False
            )
        ).any(axis=1)

        matches = df.loc[mask]
        if not matches.empty:
            print(f"\n--- SHEET: {sheet} ---")
            print(matches.to_string(index=True, header=False))
    except Exception as e:
        print(f"Could not inspect {sheet}: {e}")

Report: NIMF-MONTHLY-PORTFOLIO-31-July-24.xls

--- SHEET: INDEX ---
56  LB  NIPPON INDIA ETF NIFTY 1D RATE LIQUID BEES (An open ended liquid scheme, listed on the Exchange in the form of an ETF, investing in Tri-Party Repo on G-Sec or T-bills /Repo & Reverse Repo with daily Dividend and compulsory Reinvestment of Income Distribution cum capital withdrawal option. Relatively Low Interest Rate Risk and Relatively Low Credit Risk.)
68  MF                                                                                                                                                                                                     Nippon India Multi Asset Allocation Fund (An open ended scheme investing in Equity, Debt \nand Exchange Traded Commodity Derivatives and \nGold ETF & Silver ETF)
82  SI                                                                                                                                                 Nippon India Silver ETF (An open ended scheme, list

In [136]:
july_mask = nippon_log["Report_Month"].astype(str).eq("2024-07")

nippon_log.loc[
    july_mask, "Silver_Options_Review"
] = (
    "No silver option identified in the worksheet keyword search; "
    "blank is not treated as zero. Further source verification may "
    "be needed before concluding that no silver options were held."
)

nippon_log.loc[
    july_mask, "Review_Notes"
] = (
    nippon_log.loc[july_mask, "Review_Notes"]
    .fillna("")
    .str.rstrip()
    + " July 2024 report inspected: MF worksheet shows Nippon India "
    "Silver ETF at ₹1,227.60 lakh (0.32% NAV) and silver futures "
    "at ₹7,022.06 lakh (1.82% NAV). No silver-option row was found "
    "in the keyword search; option value remains unrecorded, not zero."
).str.strip()

nippon_log.to_csv(output_file, index=False)

print(
    nippon_log.loc[
        july_mask,
        [
            "Report_Month",
            "Silver_Options_Value_Lakh",
            "Silver_Options_NAV_Pct",
            "Silver_Options_Review",
            "Review_Notes",
        ],
    ].to_string(index=False)
)

Report_Month  Silver_Options_Value_Lakh  Silver_Options_NAV_Pct                                                                                                                                                                    Silver_Options_Review                                                                                                                                                                                                                                            Review_Notes
     2024-07                        NaN                     NaN No silver option identified in the worksheet keyword search; blank is not treated as zero. Further source verification may be needed before concluding that no silver options were held. July 2024 report inspected: MF worksheet shows Nippon India Silver ETF at ₹1,227.60 lakh (0.32% NAV) and silver futures at ₹7,022.06 lakh (1.82% NAV). No silver-option row was found in the keyword search; option value remains unrecorded, not zero.


In [137]:
august_file = nippon_folder / "NIMF_MONTHLY_PORTFOLIO_31-Aug-24.xls"

with open(august_file, "rb") as f:
    august_bytes = f.read()

if zipfile.is_zipfile(BytesIO(august_bytes)):
    august_xls = pd.ExcelFile(
        BytesIO(august_bytes), engine="openpyxl"
    )
else:
    august_xls = pd.ExcelFile(
        BytesIO(august_bytes), engine="xlrd"
    )

print("Report:", august_file.name)

for sheet in august_xls.sheet_names:
    try:
        df = pd.read_excel(
            august_xls, sheet_name=sheet, header=None
        )

        mask = df.astype(str).apply(
            lambda col: col.str.contains(
                r"SILVER|OPTION|PUT|CALL",
                case=False,
                regex=True,
                na=False,
            )
        ).any(axis=1)

        matches = df.loc[mask]

        if not matches.empty:
            print(f"\n--- SHEET: {sheet} ---")
            print(matches.to_string(index=True, header=False))

    except Exception as e:
        print(f"Could not inspect {sheet}: {e}")

Report: NIMF_MONTHLY_PORTFOLIO_31-Aug-24.xls

--- SHEET: INDEX ---
56  LB  NIPPON INDIA ETF NIFTY 1D RATE LIQUID BEES (An open ended liquid scheme, listed on the Exchange in the form of an ETF, investing in Tri-Party Repo on G-Sec or T-bills /Repo & Reverse Repo with daily Dividend and compulsory Reinvestment of Income Distribution cum capital withdrawal option. Relatively Low Interest Rate Risk and Relatively Low Credit Risk.)
68  MF                                                                                                                                                                                                     Nippon India Multi Asset Allocation Fund (An open ended scheme investing in Equity, Debt \nand Exchange Traded Commodity Derivatives and \nGold ETF & Silver ETF)
82  SI                                                                                                                                                 Nippon India Silver ETF (An open ended scheme, liste

In [138]:
august_mask = nippon_log["Report_Month"].astype(str).eq("2024-08")

nippon_log.loc[
    august_mask, "Silver_Options_Review"
] = (
    "No silver option identified in the worksheet keyword search; "
    "blank is not treated as zero. Further source verification may "
    "be needed before concluding that no silver options were held."
)

nippon_log.loc[
    august_mask, "Review_Notes"
] = (
    nippon_log.loc[august_mask, "Review_Notes"]
    .fillna("")
    .str.rstrip()
    + " August 2024 report inspected: MF worksheet shows Nippon India "
    "Silver ETF at ₹7,905.32 lakh (1.93% NAV) and silver futures "
    "at ₹1,022.52 lakh (0.25% NAV). No silver-option row was found "
    "in the keyword search; option value remains unrecorded, not zero."
).str.strip()

nippon_log.to_csv(output_file, index=False)

print(
    nippon_log.loc[
        august_mask,
        [
            "Report_Month",
            "Silver_Options_Value_Lakh",
            "Silver_Options_NAV_Pct",
            "Silver_Options_Review",
            "Review_Notes",
        ],
    ].to_string(index=False)
)

Report_Month  Silver_Options_Value_Lakh  Silver_Options_NAV_Pct                                                                                                                                                                    Silver_Options_Review                                                                                                                                                                                                                                              Review_Notes
     2024-08                        NaN                     NaN No silver option identified in the worksheet keyword search; blank is not treated as zero. Further source verification may be needed before concluding that no silver options were held. August 2024 report inspected: MF worksheet shows Nippon India Silver ETF at ₹7,905.32 lakh (1.93% NAV) and silver futures at ₹1,022.52 lakh (0.25% NAV). No silver-option row was found in the keyword search; option value remains unrecorded, not zero.


In [139]:
september_file = nippon_folder / "NIMF-MONTHLY-PORTFOLIO-30-Sep-24.xls"

with open(september_file, "rb") as f:
    september_bytes = f.read()

if zipfile.is_zipfile(BytesIO(september_bytes)):
    september_xls = pd.ExcelFile(
        BytesIO(september_bytes), engine="openpyxl"
    )
else:
    september_xls = pd.ExcelFile(
        BytesIO(september_bytes), engine="xlrd"
    )

print("Report:", september_file.name)

for sheet in september_xls.sheet_names:
    try:
        df = pd.read_excel(
            september_xls, sheet_name=sheet, header=None
        )

        mask = df.astype(str).apply(
            lambda col: col.str.contains(
                r"SILVER|OPTION|PUT|CALL",
                case=False,
                regex=True,
                na=False,
            )
        ).any(axis=1)

        matches = df.loc[mask]

        if not matches.empty:
            print(f"\n--- SHEET: {sheet} ---")
            print(matches.to_string(index=True, header=False))

    except Exception as e:
        print(f"Could not inspect {sheet}: {e}")

Report: NIMF-MONTHLY-PORTFOLIO-30-Sep-24.xls

--- SHEET: INDEX ---
56  LB  NIPPON INDIA ETF NIFTY 1D RATE LIQUID BEES (An open ended liquid scheme, listed on the Exchange in the form of an ETF, investing in Tri-Party Repo on G-Sec or T-bills /Repo & Reverse Repo with daily Dividend and compulsory Reinvestment of Income Distribution cum capital withdrawal option. Relatively Low Interest Rate Risk and Relatively Low Credit Risk.)
68  MF                                                                                                                                                                                                     Nippon India Multi Asset Allocation Fund (An open ended scheme investing in Equity, Debt \nand Exchange Traded Commodity Derivatives and \nGold ETF & Silver ETF)
82  SI                                                                                                                                                 Nippon India Silver ETF (An open ended scheme, liste

In [140]:

# FINAL NIPPON VALIDATION
# This cell checks the log only; it does not change or save anything.

import pandas as pd

# 1. Check expected monthly coverage
expected_months = pd.period_range(
    start="2024-01",
    end="2026-09",
    freq="M"
).astype(str)

actual_months = (
    nippon_log["Report_Month"]
    .astype(str)
    .str.strip()
)

missing_months = sorted(set(expected_months) - set(actual_months))
unexpected_months = sorted(set(actual_months) - set(expected_months))
duplicate_months = actual_months[actual_months.duplicated()].tolist()

print("=== MONTHLY COVERAGE ===")
print("Total rows:", len(nippon_log))
print("Unique months:", actual_months.nunique())
print("Missing months:", missing_months)
print("Unexpected months:", unexpected_months)
print("Duplicate months:", duplicate_months)

# 2. Check missing values in important columns
key_columns = [
    "Gold_ETF_Value_Lakh",
    "Gold_ETF_NAV_Pct",
    "Silver_ETF_Value_Lakh",
    "Silver_ETF_NAV_Pct",
    "Gold_Futures_Value_Lakh",
    "Gold_Futures_NAV_Pct",
    "Silver_Futures_Value_Lakh",
    "Silver_Futures_NAV_Pct",
    "Silver_Options_Value_Lakh",
    "Silver_Options_NAV_Pct",
]

print("\n=== MISSING VALUES ===")
print(nippon_log[key_columns].isna().sum().to_string())

# 3. Check whether recorded silver-option values have review notes
option_values = pd.to_numeric(
    nippon_log["Silver_Options_Value_Lakh"],
    errors="coerce"
)

option_reviews = (
    nippon_log["Silver_Options_Review"]
    .fillna("")
    .astype(str)
    .str.strip()
)

missing_option_reviews = nippon_log.loc[
    option_values.notna() & option_reviews.eq(""),
    ["Report_Month", "Silver_Options_Value_Lakh"]
]

print("\n=== SILVER-OPTION REVIEW CHECK ===")
print(
    "Months with a recorded option value:",
    int(option_values.notna().sum())
)
print(
    "Recorded option values missing a review note:",
    len(missing_option_reviews)
)
print(missing_option_reviews.to_string(index=False))

# 4. Check for accidental explicit zeros in the key columns
print("\n=== EXPLICIT ZERO CHECK ===")
print(
    nippon_log[key_columns]
    .apply(lambda col: (pd.to_numeric(col, errors="coerce") == 0).sum())
    .to_string()
)


=== MONTHLY COVERAGE ===
Total rows: 33
Unique months: 33
Missing months: []
Unexpected months: []
Duplicate months: []

=== MISSING VALUES ===
Gold_ETF_Value_Lakh           0
Gold_ETF_NAV_Pct              0
Silver_ETF_Value_Lakh         6
Silver_ETF_NAV_Pct            6
Gold_Futures_Value_Lakh       5
Gold_Futures_NAV_Pct          5
Silver_Futures_Value_Lakh     8
Silver_Futures_NAV_Pct        8
Silver_Options_Value_Lakh    28
Silver_Options_NAV_Pct       29

=== SILVER-OPTION REVIEW CHECK ===
Months with a recorded option value: 5
Recorded option values missing a review note: 0
Empty DataFrame
Columns: [Report_Month, Silver_Options_Value_Lakh]
Index: []

=== EXPLICIT ZERO CHECK ===
Gold_ETF_Value_Lakh          0
Gold_ETF_NAV_Pct             0
Silver_ETF_Value_Lakh        0
Silver_ETF_NAV_Pct           0
Gold_Futures_Value_Lakh      0
Gold_Futures_NAV_Pct         0
Silver_Futures_Value_Lakh    0
Silver_Futures_NAV_Pct       0
Silver_Options_Value_Lakh    0
Silver_Options_NAV_Pct      

In [141]:

# FINAL REVIEW OF UNRECORDED VALUES
# This cell does not modify the dataset.

columns_to_review = [
    "Silver_ETF_Value_Lakh",
    "Gold_Futures_Value_Lakh",
    "Silver_Futures_Value_Lakh",
]

for col in columns_to_review:
    missing_mask = pd.to_numeric(
        nippon_log[col], errors="coerce"
    ).isna()

    print(f"\n=== {col}: MISSING IN {missing_mask.sum()} MONTHS ===")

    print(
        nippon_log.loc[
            missing_mask,
            ["Report_Month", "Review_Notes"]
        ].to_string(index=False)
    )



=== Silver_ETF_Value_Lakh: MISSING IN 6 MONTHS ===
Report_Month                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 Review_Notes
     2024-01                                No direct silver ETF identified in keyword search. Gold futures value blank: no matching gold futures identified in the inspected source section; blank does not mean zero. Commodity-futures section verified: silver futures total ₹1,475.29

In [142]:

# FINAL DOCUMENTATION CLEANUP FOR NIPPON SILVER ETF GAPS

silver_etf_missing_months = [
    "2024-01", "2024-02", "2024-03",
    "2024-04", "2024-05", "2024-06"
]

for month in silver_etf_missing_months:
    mask = nippon_log["Report_Month"].astype(str).eq(month)

    existing_notes = (
        nippon_log.loc[mask, "Review_Notes"]
        .fillna("")
        .iloc[0]
    )

    clarification = (
        " SILVER ETF DATA LIMITATION: No direct Nippon India Silver ETF "
        "holding was identified in the inspected report/keyword search "
        "for this month. The value remains blank and is not treated as "
        "zero holdings."
    )

    if "SILVER ETF DATA LIMITATION:" not in existing_notes:
        nippon_log.loc[mask, "Review_Notes"] = (
            existing_notes.rstrip() + clarification
        ).strip()

# Save the updated log
nippon_log.to_csv(output_file, index=False)

# Verify the six months
print(
    nippon_log.loc[
        nippon_log["Report_Month"].astype(str).isin(
            silver_etf_missing_months
        ),
        [
            "Report_Month",
            "Silver_ETF_Value_Lakh",
            "Silver_ETF_NAV_Pct",
            "Review_Notes",
        ],
    ][["Report_Month", "Silver_ETF_Value_Lakh",
       "Silver_ETF_NAV_Pct"]].to_string(index=False)
)

print("\nSaved to:", output_file)
print("Total rows:", len(nippon_log))
print("Unique months:", nippon_log["Report_Month"].nunique())


Report_Month  Silver_ETF_Value_Lakh  Silver_ETF_NAV_Pct
     2024-01                    NaN                 NaN
     2024-02                    NaN                 NaN
     2024-03                    NaN                 NaN
     2024-04                    NaN                 NaN
     2024-05                    NaN                 NaN
     2024-06                    NaN                 NaN

Saved to: c:\Users\rocks\OneDrive\Desktop\AMC Project Data\02_CLEAN_DATA\Nippon_India_Monthly_Holdings_Log.csv
Total rows: 33
Unique months: 33


In [143]:

import pandas as pd
from pathlib import Path
from IPython.display import display

# Locate the saved Nippon dataset
file_path = (
    Path.cwd().parent
    / "02_CLEAN_DATA"
    / "Nippon_India_Monthly_Holdings_Log.csv"
)

# Load the final saved dataset
nippon_final = pd.read_csv(file_path)

# Display dataset size and monthly coverage
print("Dataset shape:", nippon_final.shape)
print("First month:", nippon_final["Report_Month"].iloc[0])
print("Last month:", nippon_final["Report_Month"].iloc[-1])

# Display all 33 months and all columns
display(nippon_final)


Dataset shape: (33, 17)
First month: 2024-01
Last month: 2026-09


,Report_Month,Report_Date,Gold_ETF_Name,Gold_ETF_Value_Lakh,Gold_ETF_NAV_Pct,Silver_ETF_Name,Silver_ETF_Value_Lakh,Silver_ETF_NAV_Pct,Gold_Futures_Value_Lakh,Gold_Futures_NAV_Pct,Silver_Futures_Value_Lakh,Silver_Futures_NAV_Pct,Options_and_Other_Instruments,Review_Notes,Silver_Options_Value_Lakh,Silver_Options_NAV_Pct,Silver_Options_Review
0,2024-01,2024-01-31,Nippon India ETF Gold BeES,33269.06,13.08,NaN,NaN,NaN,NaN,NaN,1475.29,0.58,NaN,No direct silver ETF identified in keyword sea...,17.24,0.01,Silver call option identified; source % to NAV...
1,2024-02,2024-02-29,Nippon India ETF Gold BeES,34613.23,12.81,NaN,NaN,NaN,NaN,NaN,2587.43,0.96,NaN,No direct silver ETF identified in keyword sea...,NaN,NaN,No silver option identified in matching rows; ...
2,2024-03,2024-03-31,Nippon India ETF Gold BeES,37408.00,12.88,NaN,NaN,NaN,NaN,NaN,3219.56,1.11,NaN,No direct silver ETF identified in keyword sea...,5.68,NaN,Silver put option identified; source % to NAV ...
3,2024-04,2024-04-30,Nippon India ETF Gold BeES,41142.82,13.19,NaN,NaN,NaN,140.83,0.05,3856.59,1.24,NaN,No direct silver ETF identified in keyword sea...,65.15,0.02,REVIEW REQUIRED: source row has instrument cod...
4,2024-05,2024-05-31,Nippon India ETF Gold BeES,41419.90,12.50,NaN,NaN,NaN,2945.19,0.89,5329.37,1.61,NaN,No direct silver ETF identified in keyword sea...,19.67,0.01,Silver put option identified; source % to NAV ...
5,2024-06,2024-06-30,Nippon India ETF Gold BeES,40973.87,11.38,NaN,NaN,NaN,7372.95,2.05,7655.67,2.13,NaN,No direct silver ETF identified in keyword sea...,NaN,NaN,No silver option identified in matching rows; ...
6,2024-07,2024-07-31,Nippon India ETF Gold BeES,39656.06,10.29,Nippon India Silver ETF,1227.60,0.32,12259.28,3.18,7022.06,1.82,NaN,July 2024 report inspected: MF worksheet shows...,NaN,NaN,No silver option identified in the worksheet k...
7,2024-08,2024-08-31,Nippon India ETF Gold BeES,40994.15,10.01,Nippon India Silver ETF,7905.32,1.93,14895.09,3.64,1022.52,0.25,NaN,August 2024 report inspected: MF worksheet sho...,NaN,NaN,No silver option identified in the worksheet k...
8,2024-09,2024-09-30,Nippon India ETF Gold BeES,42947.22,9.89,Nippon India Silver ETF,8554.37,1.97,16105.14,3.71,2721.57,0.63,NaN,NaN,NaN,NaN,NaN
9,2024-10,2024-10-31,Nippon India ETF Gold BeES,45224.67,10.05,Nippon India Silver ETF,9159.70,2.04,19293.78,4.29,3662.22,0.81,NaN,October 2024 source verification: MF worksheet...,29.63,0.01,Silver 90000 Put November 2024 Option (1 kg) i...


In [1]:

from pathlib import Path

PROJECT_DIR = Path.cwd().parent
HDFC_RAW_DIR = PROJECT_DIR / "01_RAW_DATA" / "HDFC"

hdfc_files = sorted(
    file for file in HDFC_RAW_DIR.iterdir()
    if file.is_file()
)

print("HDFC raw folder:", HDFC_RAW_DIR)
print("Total files:", len(hdfc_files))

for file in hdfc_files:
    print(file.name)


HDFC raw folder: c:\Users\rocks\OneDrive\Desktop\AMC Project Data\01_RAW_DATA\HDFC
Total files: 33
Monthly HDFC Multi Asset Allocation Fund - 30 September 2026_0.xlsx
Monthly HDFC Multi Asset Allocation Fund - 31 August 2026 (1).xlsx
Monthly HDFC Multi-Asset Allocation Fund - 28 February 2026.xlsx
Monthly HDFC Multi-Asset Allocation Fund - 30 April 2026.xlsx
Monthly HDFC Multi-Asset Allocation Fund - 30 June 2026.xlsx
Monthly HDFC Multi-Asset Allocation Fund - 31 December 2025.xlsx
Monthly HDFC Multi-Asset Allocation Fund - 31 January 2026.xlsx
Monthly HDFC Multi-Asset Allocation Fund - 31 July 2026.xlsx
Monthly HDFC Multi-Asset Allocation Fund - 31 March 2026.xlsx
Monthly HDFC Multi-Asset Allocation Fund - 31 May 2026.xlsx
Monthly HDFC Multi-Asset Fund - 28 February 2025.xlsx
Monthly HDFC Multi-Asset Fund - 29 February 2024.xlsx
Monthly HDFC Multi-Asset Fund - 30 April 2024.xlsx
Monthly HDFC Multi-Asset Fund - 30 April 2025.xlsx
Monthly HDFC Multi-Asset Fund - 30 June 2024.xlsx
Monthl

In [2]:

import re
from collections import Counter
import pandas as pd

# Extract month and year from each HDFC filename
month_map = {
    "January": 1, "February": 2, "March": 3,
    "April": 4, "May": 5, "June": 6,
    "July": 7, "August": 8, "September": 9,
    "October": 10, "November": 11, "December": 12
}

found_months = []

for file in hdfc_files:
    match = re.search(
        r"(\d{1,2})\s+([A-Za-z]+)\s+(\d{4})",
        file.name
    )

    if match:
        day, month_name, year = match.groups()
        month_number = month_map.get(month_name)

        if month_number:
            found_months.append(
                f"{int(year):04d}-{month_number:02d}"
            )

expected_months = pd.period_range(
    "2024-01", "2026-09", freq="M"
).astype(str).tolist()

counts = Counter(found_months)

print("Files:", len(hdfc_files))
print("Dates extracted:", len(found_months))
print("Missing months:", sorted(set(expected_months) - set(found_months)))
print(
    "Duplicate months:",
    sorted(month for month, count in counts.items() if count > 1)
)
print("Unexpected months:", sorted(set(found_months) - set(expected_months)))


Files: 33
Dates extracted: 33
Missing months: []
Duplicate months: []
Unexpected months: []


In [3]:

from openpyxl import load_workbook

# Select the September 2026 HDFC report
hdfc_sample = next(
    file for file in hdfc_files
    if "30 September 2026" in file.name
)

workbook = load_workbook(
    hdfc_sample,
    read_only=True,
    data_only=True
)

print("Sample file:", hdfc_sample.name)
print("\nWorksheets:", workbook.sheetnames)

# Preview the first 15 rows of each worksheet
for sheet in workbook.worksheets:
    print(f"\n--- Sheet: {sheet.title} ---")

    for row in sheet.iter_rows(
        min_row=1,
        max_row=15,
        values_only=True
    ):
        values = [
            value for value in row
            if value is not None
        ]

        if values:
            print(values)

workbook.close()


Sample file: Monthly HDFC Multi Asset Allocation Fund - 30 September 2026_0.xlsx

Worksheets: ['MY2005', 'DerivativeHDFC Multi Asset Allo', 'DerivativeMY2005']

--- Sheet: MY2005 ---
['HDFC Multi Asset Allocation Fund (An open ended scheme investing in Equity and Equity related instruments, Debt & Money Market Instruments and Gold). (Formerly known as HDFC Multi-Asset Allocation Fund)', 'HDFC Multi Asset Allocation Fund (An open ended scheme investing in Equity and Equity related instruments, Debt & Money Market Instruments and Gold)', 'HDFC Multi Asset Allocation Fund (An open ended scheme investing in Equity and Equity related instruments, Debt & Money Market Instruments and Gold)', 'HDFC Multi Asset Allocation Fund (An open ended scheme investing in Equity and Equity related instruments, Debt & Money Market Instruments and Gold)', 'HDFC Multi Asset Allocation Fund (An open ended scheme investing in Equity and Equity related instruments, Debt & Money Market Instruments and Gold)', 'H

In [4]:

from openpyxl import load_workbook

workbook = load_workbook(
    hdfc_sample,
    read_only=True,
    data_only=True
)

sheet = workbook["MY2005"]

print("Gold/Silver-related rows in:", hdfc_sample.name)

for row_number, row in enumerate(
    sheet.iter_rows(values_only=True),
    start=1
):
    row_text = " ".join(
        str(value) for value in row if value is not None
    )

    if "gold" in row_text.lower() or "silver" in row_text.lower():
        print(f"Row {row_number}: {row_text}")

workbook.close()


Gold/Silver-related rows in: Monthly HDFC Multi Asset Allocation Fund - 30 September 2026_0.xlsx
Row 1: HDFC Multi Asset Allocation Fund (An open ended scheme investing in Equity and Equity related instruments, Debt & Money Market Instruments and Gold). (Formerly known as HDFC Multi-Asset Allocation Fund) HDFC Multi Asset Allocation Fund (An open ended scheme investing in Equity and Equity related instruments, Debt & Money Market Instruments and Gold) HDFC Multi Asset Allocation Fund (An open ended scheme investing in Equity and Equity related instruments, Debt & Money Market Instruments and Gold) HDFC Multi Asset Allocation Fund (An open ended scheme investing in Equity and Equity related instruments, Debt & Money Market Instruments and Gold) HDFC Multi Asset Allocation Fund (An open ended scheme investing in Equity and Equity related instruments, Debt & Money Market Instruments and Gold) HDFC Multi Asset Allocation Fund (An open ended scheme investing in Equity and Equity related ins

In [5]:

from openpyxl import load_workbook
import pandas as pd

hdfc_holdings_rows = []

for file in hdfc_files:
    workbook = load_workbook(
        file,
        read_only=True,
        data_only=True
    )

    if "MY2005" not in workbook.sheetnames:
        print("WARNING: MY2005 sheet missing:", file.name)
        workbook.close()
        continue

    sheet = workbook["MY2005"]

    for row_number, row in enumerate(
        sheet.iter_rows(values_only=True),
        start=1
    ):
        values = list(row)
        row_text = " ".join(
            str(value) for value in values
            if value is not None
        ).lower()

        # Look for actual holding rows, not titles or benchmark references
        if (
            ("gold exchange traded fund" in row_text
             or "gold etf" in row_text
             or "silver etf" in row_text
             or "silver exchange traded fund" in row_text)
            and not row_text.startswith("portfolio as on")
            and "benchmark" not in row_text
            and "riskometer" not in row_text
        ):
            hdfc_holdings_rows.append({
                "Source_File": file.name,
                "Worksheet": "MY2005",
                "Source_Row": row_number,
                "Row_Values": " | ".join(
                    str(value) for value in values
                    if value is not None
                )
            })

    workbook.close()

hdfc_holdings_check = pd.DataFrame(hdfc_holdings_rows)

print("Gold/Silver candidate rows found:", len(hdfc_holdings_check))
display(hdfc_holdings_check)


Gold/Silver candidate rows found: 41


,Source_File,Worksheet,Source_Row,Row_Values
0,Monthly HDFC Multi Asset Allocation Fund - 30 ...,MY2005,145,| | INF179KC1981 | | HDFC Gold Exchange Trade...
1,Monthly HDFC Multi Asset Allocation Fund - 30 ...,MY2005,146,| | INF179KC1DI2 | | HDFC Silver ETF | | 500...
2,Monthly HDFC Multi Asset Allocation Fund - 31 ...,MY2005,152,| | INF179KC1981 | | HDFC Gold Exchange Trade...
3,Monthly HDFC Multi Asset Allocation Fund - 31 ...,MY2005,153,| | INF179KC1DI2 | | HDFC Silver ETF | | 500...
4,Monthly HDFC Multi-Asset Allocation Fund - 28 ...,MY2005,153,| | INF179KC1981 | | HDFC Gold Exchange Trade...
5,Monthly HDFC Multi-Asset Allocation Fund - 28 ...,MY2005,154,| INF179KC1DI2 | | HDFC Silver ETF | | 4800...
6,Monthly HDFC Multi-Asset Allocation Fund - 30 ...,MY2005,151,| | INF179KC1981 | | HDFC Gold Exchange Trade...
7,Monthly HDFC Multi-Asset Allocation Fund - 30 ...,MY2005,152,| INF179KC1DI2 | | HDFC Silver ETF | | 1000...
8,Monthly HDFC Multi-Asset Allocation Fund - 30 ...,MY2005,154,| | INF179KC1981 | | HDFC Gold Exchange Trade...
9,Monthly HDFC Multi-Asset Allocation Fund - 30 ...,MY2005,155,| INF179KC1DI2 | | HDFC Silver ETF | | 1000...


In [6]:

from openpyxl import load_workbook
import pandas as pd
import re

hdfc_metal_records = []

for file in hdfc_files:
    # Extract report date from the filename
    match = re.search(
        r"(\d{1,2})\s+([A-Za-z]+)\s+(\d{4})",
        file.name
    )

    report_date = pd.to_datetime(
        " ".join(match.groups()),
        format="%d %B %Y",
        errors="coerce"
    ) if match else pd.NaT

    workbook = load_workbook(
        file, read_only=True, data_only=True
    )
    sheet = workbook["MY2005"]

    for row_number, row in enumerate(
        sheet.iter_rows(values_only=True), start=1
    ):
        values = list(row)

        # Instrument name is in column D in the inspected report.
        name = values[3] if len(values) > 3 else None
        if not isinstance(name, str):
            continue

        name_lower = name.lower().strip()

        if "gold exchange traded fund" in name_lower or "gold etf" in name_lower:
            metal = "Gold"
        elif "silver etf" in name_lower or "silver exchange traded fund" in name_lower:
            metal = "Silver"
        else:
            continue

        # Columns G and H: market value (₹ lakh) and % of NAV.
        market_value = values[6] if len(values) > 6 else None
        nav_pct = values[7] if len(values) > 7 else None

        hdfc_metal_records.append({
            "Report_Date": report_date,
            "Report_Month": (
                report_date.strftime("%Y-%m")
                if pd.notna(report_date) else None
            ),
            "Metal": metal,
            "Instrument_Name": name.strip(),
            "Market_Value_Lakh": market_value,
            "NAV_Pct": nav_pct,
            "Source_File": file.name,
            "Source_Row": row_number
        })

    workbook.close()

hdfc_metals = pd.DataFrame(hdfc_metal_records)
hdfc_metals = hdfc_metals.sort_values(
    ["Report_Date", "Metal"], na_position="last"
)

print("Extracted rows:", len(hdfc_metals))
print("\nCounts by metal:")
display(hdfc_metals.groupby("Metal").size().rename("Rows_Found").to_frame())

print("\nExtracted records:")
display(hdfc_metals)


Extracted rows: 41

Counts by metal:


,Rows_Found
Metal,
Gold,33
Silver,8



Extracted records:


,Report_Date,Report_Month,Metal,Instrument_Name,Market_Value_Lakh,NAV_Pct,Source_File,Source_Row
31,2024-01-31,2024-01,Gold,HDFC Gold Exchange Traded Fund,27197.83,11.30,Monthly HDFC Multi-Asset Fund - 31 January 202...,114
19,2024-02-29,2024-02,Gold,HDFC Gold Exchange Traded Fund,26949.13,10.66,Monthly HDFC Multi-Asset Fund - 29 February 20...,115
35,2024-03-31,2024-03,Gold,HDFC Gold Exchange Traded Fund,29003.39,10.98,Monthly HDFC Multi-Asset Fund - 31 March 2024....,117
20,2024-04-30,2024-04,Gold,HDFC Gold Exchange Traded Fund,31266.56,11.17,Monthly HDFC Multi-Asset Fund - 30 April 2024....,122
37,2024-05-31,2024-05,Gold,HDFC Gold Exchange Traded Fund,31425.73,10.85,Monthly HDFC Multi-Asset Fund - 31 May 2024.xlsx,122
22,2024-06-30,2024-06,Gold,HDFC Gold Exchange Traded Fund,32796.06,10.44,Monthly HDFC Multi-Asset Fund - 30 June 2024.xlsx,131
33,2024-07-31,2024-07,Gold,HDFC Gold Exchange Traded Fund,36686.47,10.96,Monthly HDFC Multi-Asset Fund - 31 July 2024.xlsx,131
28,2024-08-31,2024-08,Gold,HDFC Gold Exchange Traded Fund,37886.24,10.75,Monthly HDFC Multi-Asset Fund - 31 August 2024...,135
26,2024-09-30,2024-09,Gold,HDFC Gold Exchange Traded Fund,42154.96,11.39,Monthly HDFC Multi-Asset Fund - 30 September 2...,131
39,2024-10-31,2024-10,Gold,HDFC Gold Exchange Traded Fund,44295.82,11.78,Monthly HDFC Multi-Asset Fund - 31 October 202...,131


In [7]:

all_months = pd.period_range("2024-01", "2026-09", freq="M").astype(str)

gold_months = set(
    hdfc_metals.loc[hdfc_metals["Metal"] == "Gold", "Report_Month"]
)
silver_months = set(
    hdfc_metals.loc[hdfc_metals["Metal"] == "Silver", "Report_Month"]
)

print("Gold months missing:", sorted(set(all_months) - gold_months))
print("Silver months missing:", sorted(set(all_months) - silver_months))

print("\nDuplicate month-metal records:")
display(
    hdfc_metals[
        hdfc_metals.duplicated(
            subset=["Report_Month", "Metal"], keep=False
        )
    ].sort_values(["Report_Month", "Metal"])
)

print("\nRecords with missing market value or NAV percentage:")
display(
    hdfc_metals[
        hdfc_metals["Market_Value_Lakh"].isna()
        | hdfc_metals["NAV_Pct"].isna()
    ]
)


Gold months missing: []
Silver months missing: ['2024-01', '2024-02', '2024-03', '2024-04', '2024-05', '2024-06', '2024-07', '2024-08', '2024-09', '2024-10', '2024-11', '2024-12', '2025-01', '2025-02', '2025-03', '2025-04', '2025-05', '2025-06', '2025-07', '2025-08', '2025-09', '2025-10', '2025-11', '2025-12', '2026-01']

Duplicate month-metal records:


,Report_Date,Report_Month,Metal,Instrument_Name,Market_Value_Lakh,NAV_Pct,Source_File,Source_Row



Records with missing market value or NAV percentage:


,Report_Date,Report_Month,Metal,Instrument_Name,Market_Value_Lakh,NAV_Pct,Source_File,Source_Row


In [9]:

import re
import pandas as pd

silver_candidates = []

for file in hdfc_files:
    # Extract the report date from the filename
    match = re.search(
        r"(\d{1,2})\s+([A-Za-z]+)\s+(\d{4})",
        file.name
    )

    if not match:
        print("Date not found:", file.name)
        continue

    report_date = pd.to_datetime(
        " ".join(match.groups()),
        format="%d %B %Y",
        errors="coerce"
    )

    if pd.isna(report_date):
        print("Could not parse date:", file.name)
        continue

    # Read the main holdings worksheet
    df = pd.read_excel(
        file,
        sheet_name="MY2005",
        header=None
    )

    # Search every row for the word "silver"
    for row_idx, row in df.iterrows():
        row_text = " | ".join(
            str(value)
            for value in row
            if pd.notna(value)
        )

        if "silver" in row_text.lower():
            silver_candidates.append({
                "Report_Month": report_date.strftime("%Y-%m"),
                "Source_File": file.name,
                "Source_Row": row_idx + 1,
                "Row_Text": row_text
            })

silver_candidates_df = pd.DataFrame(silver_candidates)

if not silver_candidates_df.empty:
    display(
        silver_candidates_df.sort_values(
            ["Report_Month", "Source_Row"]
        )
    )
else:
    print("No rows containing 'silver' were found.")


,Report_Month,Source_File,Source_Row,Row_Text
10,2025-12,Monthly HDFC Multi-Asset Allocation Fund - 31 ...,242,13) Riskometer based on Scheme Portfolio and P...
11,2026-01,Monthly HDFC Multi-Asset Allocation Fund - 31 ...,248,13) Riskometer based on Scheme Portfolio and P...
4,2026-02,Monthly HDFC Multi-Asset Allocation Fund - 28 ...,154,INF179KC1DI2 | HDFC Silver ETF | 480000 | 1215...
5,2026-02,Monthly HDFC Multi-Asset Allocation Fund - 28 ...,257,13) Riskometer based on Scheme Portfolio and P...
14,2026-03,Monthly HDFC Multi-Asset Allocation Fund - 31 ...,154,INF179KC1DI2 | HDFC Silver ETF | 1000000 | 216...
15,2026-03,Monthly HDFC Multi-Asset Allocation Fund - 31 ...,252,13) Riskometer based on Scheme Portfolio and P...
6,2026-04,Monthly HDFC Multi-Asset Allocation Fund - 30 ...,152,INF179KC1DI2 | HDFC Silver ETF | 1000000 | 227...
7,2026-04,Monthly HDFC Multi-Asset Allocation Fund - 30 ...,252,13) Riskometer based on Scheme Portfolio and P...
16,2026-05,Monthly HDFC Multi-Asset Allocation Fund - 31 ...,154,INF179KC1DI2 | HDFC Silver ETF | 1000000 | 249...
17,2026-05,Monthly HDFC Multi-Asset Allocation Fund - 31 ...,256,13) Riskometer based on Scheme Portfolio and P...


In [10]:

isin_matches = []

for file in hdfc_files:
    match = re.search(
        r"(\d{1,2})\s+([A-Za-z]+)\s+(\d{4})",
        file.name
    )

    if not match:
        continue

    report_date = pd.to_datetime(
        " ".join(match.groups()),
        format="%d %B %Y",
        errors="coerce"
    )

    if pd.isna(report_date):
        continue

    df = pd.read_excel(
        file,
        sheet_name="MY2005",
        header=None
    )

    for row_idx, row in df.iterrows():
        row_text = " | ".join(
            str(value) for value in row if pd.notna(value)
        )

        if "INF179KC1DI2" in row_text.upper():
            isin_matches.append({
                "Report_Month": report_date.strftime("%Y-%m"),
                "Source_File": file.name,
                "Source_Row": row_idx + 1,
                "Row_Text": row_text
            })

isin_matches_df = pd.DataFrame(isin_matches)

print("Reports with the silver ETF ISIN:", len(isin_matches_df))
display(
    isin_matches_df.sort_values("Report_Month")
    if not isin_matches_df.empty
    else isin_matches_df
)


Reports with the silver ETF ISIN: 8


,Report_Month,Source_File,Source_Row,Row_Text
2,2026-02,Monthly HDFC Multi-Asset Allocation Fund - 28 ...,154,INF179KC1DI2 | HDFC Silver ETF | 480000 | 1215...
6,2026-03,Monthly HDFC Multi-Asset Allocation Fund - 31 ...,154,INF179KC1DI2 | HDFC Silver ETF | 1000000 | 216...
3,2026-04,Monthly HDFC Multi-Asset Allocation Fund - 30 ...,152,INF179KC1DI2 | HDFC Silver ETF | 1000000 | 227...
7,2026-05,Monthly HDFC Multi-Asset Allocation Fund - 31 ...,154,INF179KC1DI2 | HDFC Silver ETF | 1000000 | 249...
4,2026-06,Monthly HDFC Multi-Asset Allocation Fund - 30 ...,155,INF179KC1DI2 | HDFC Silver ETF | 1000000 | 213...
5,2026-07,Monthly HDFC Multi-Asset Allocation Fund - 31 ...,154,INF179KC1DI2 | HDFC Silver ETF | 5000000 | 103...
1,2026-08,Monthly HDFC Multi Asset Allocation Fund - 31 ...,153,| | INF179KC1DI2 | HDFC Silver ETF | 5000000 |...
0,2026-09,Monthly HDFC Multi Asset Allocation Fund - 30 ...,146,| | INF179KC1DI2 | HDFC Silver ETF | 5000000 |...


In [12]:

from pathlib import Path

PROJECT_DIR = Path.cwd().parent
OUTPUT_DIR = PROJECT_DIR / "02_CLEAN_DATA"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project folder:", PROJECT_DIR)
print("Output folder:", OUTPUT_DIR)
print("Output folder exists:", OUTPUT_DIR.exists())


Project folder: c:\Users\rocks\OneDrive\Desktop\AMC Project Data
Output folder: c:\Users\rocks\OneDrive\Desktop\AMC Project Data\02_CLEAN_DATA
Output folder exists: True


In [13]:

hdfc_log_path = OUTPUT_DIR / "HDFC_Monthly_Metal_Holdings_Log.csv"

hdfc_metals = hdfc_metals.sort_values(
    ["Report_Month", "Metal"]
).reset_index(drop=True)

hdfc_metals.to_csv(hdfc_log_path, index=False)

print("Saved:", hdfc_log_path)
print("Total records:", len(hdfc_metals))
print("Gold records:", (hdfc_metals["Metal"] == "Gold").sum())
print("Silver records:", (hdfc_metals["Metal"] == "Silver").sum())


Saved: c:\Users\rocks\OneDrive\Desktop\AMC Project Data\02_CLEAN_DATA\HDFC_Monthly_Metal_Holdings_Log.csv
Total records: 41
Gold records: 33
Silver records: 8


In [14]:

import pandas as pd

hdfc_cleaned = pd.read_csv(
    OUTPUT_DIR / "HDFC_Monthly_Metal_Holdings_Log.csv"
)

display(hdfc_cleaned)


,Report_Date,Report_Month,Metal,Instrument_Name,Market_Value_Lakh,NAV_Pct,Source_File,Source_Row
0,2024-01-31,2024-01,Gold,HDFC Gold Exchange Traded Fund,27197.83,11.30,Monthly HDFC Multi-Asset Fund - 31 January 202...,114
1,2024-02-29,2024-02,Gold,HDFC Gold Exchange Traded Fund,26949.13,10.66,Monthly HDFC Multi-Asset Fund - 29 February 20...,115
2,2024-03-31,2024-03,Gold,HDFC Gold Exchange Traded Fund,29003.39,10.98,Monthly HDFC Multi-Asset Fund - 31 March 2024....,117
3,2024-04-30,2024-04,Gold,HDFC Gold Exchange Traded Fund,31266.56,11.17,Monthly HDFC Multi-Asset Fund - 30 April 2024....,122
4,2024-05-31,2024-05,Gold,HDFC Gold Exchange Traded Fund,31425.73,10.85,Monthly HDFC Multi-Asset Fund - 31 May 2024.xlsx,122
5,2024-06-30,2024-06,Gold,HDFC Gold Exchange Traded Fund,32796.06,10.44,Monthly HDFC Multi-Asset Fund - 30 June 2024.xlsx,131
6,2024-07-31,2024-07,Gold,HDFC Gold Exchange Traded Fund,36686.47,10.96,Monthly HDFC Multi-Asset Fund - 31 July 2024.xlsx,131
7,2024-08-31,2024-08,Gold,HDFC Gold Exchange Traded Fund,37886.24,10.75,Monthly HDFC Multi-Asset Fund - 31 August 2024...,135
8,2024-09-30,2024-09,Gold,HDFC Gold Exchange Traded Fund,42154.96,11.39,Monthly HDFC Multi-Asset Fund - 30 September 2...,131
9,2024-10-31,2024-10,Gold,HDFC Gold Exchange Traded Fund,44295.82,11.78,Monthly HDFC Multi-Asset Fund - 31 October 202...,131


In [15]:

print("HDFC metal log columns:")
print(hdfc_cleaned.columns.tolist())

print("\nFirst 5 rows:")
display(hdfc_cleaned.head())


HDFC metal log columns:
['Report_Date', 'Report_Month', 'Metal', 'Instrument_Name', 'Market_Value_Lakh', 'NAV_Pct', 'Source_File', 'Source_Row']

First 5 rows:


,Report_Date,Report_Month,Metal,Instrument_Name,Market_Value_Lakh,NAV_Pct,Source_File,Source_Row
0,2024-01-31,2024-01,Gold,HDFC Gold Exchange Traded Fund,27197.83,11.30,Monthly HDFC Multi-Asset Fund - 31 January 202...,114
1,2024-02-29,2024-02,Gold,HDFC Gold Exchange Traded Fund,26949.13,10.66,Monthly HDFC Multi-Asset Fund - 29 February 20...,115
2,2024-03-31,2024-03,Gold,HDFC Gold Exchange Traded Fund,29003.39,10.98,Monthly HDFC Multi-Asset Fund - 31 March 2024....,117
3,2024-04-30,2024-04,Gold,HDFC Gold Exchange Traded Fund,31266.56,11.17,Monthly HDFC Multi-Asset Fund - 30 April 2024....,122
4,2024-05-31,2024-05,Gold,HDFC Gold Exchange Traded Fund,31425.73,10.85,Monthly HDFC Multi-Asset Fund - 31 May 2024.xlsx,122


In [16]:

# Select the latest HDFC monthly report
hdfc_sample_file = hdfc_files[-1]

# Read the main holdings worksheet
hdfc_sample_df = pd.read_excel(
    hdfc_sample_file,
    sheet_name="MY2005",
    header=None
)

print("Report:", hdfc_sample_file.name)
print("Worksheet dimensions:", hdfc_sample_df.shape)

# Display the beginning of the report
display(hdfc_sample_df.iloc[:12, :12])


Report: Monthly HDFC Multi-Asset Fund - 31 October 2025.xlsx
Worksheet dimensions: (261, 12)


,0,1,2,3,4,5,6,7,8,9,10,11
0,HDFC Multi-Asset Fund (An open ended scheme in...,HDFC Multi-Asset Fund (An open ended scheme in...,HDFC Multi-Asset Fund (An open ended scheme in...,HDFC Multi-Asset Fund (An open ended scheme in...,HDFC Multi-Asset Fund (An open ended scheme in...,HDFC Multi-Asset Fund (An open ended scheme in...,HDFC Multi-Asset Fund (An open ended scheme in...,HDFC Multi-Asset Fund (An open ended scheme in...,HDFC Multi-Asset Fund (An open ended scheme in...,HDFC Multi-Asset Fund (An open ended scheme in...,Income,Hybrid
1,Portfolio as on 31-Oct-2025,Portfolio as on 31-Oct-2025,Portfolio as on 31-Oct-2025,Portfolio as on 31-Oct-2025,Portfolio as on 31-Oct-2025,Portfolio as on 31-Oct-2025,Portfolio as on 31-Oct-2025,Portfolio as on 31-Oct-2025,Portfolio as on 31-Oct-2025,Portfolio as on 31-Oct-2025,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,ISIN,Coupon (%),Name Of the Instrument,Industry+ /Rating,Quantity,Market/ Fair Value (Rs. in Lacs.),% to NAV,Yield,~YTC (AT1/Tier 2 bonds),Derivative\n% to NAV,Unhedged\n% to NAV
5,NaN,EQUITY & EQUITY RELATED,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,NaN,(a) Listed / awaiting listing on Stock Exchanges,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,NaN,Equity,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,|,INE090A01021,NaN,ICICI Bank Ltd.,Banks,2225700,29942.34,5.82,NaN,NaN,2.89,2.94
9,|,INE002A01018,NaN,Reliance Industries Ltd.,Petroleum Products,2016500,29973.26,5.82,NaN,NaN,3.68,2.17


In [17]:

# Display the rows that appear to contain section headings
for row_idx, row in hdfc_sample_df.iterrows():
    values = [
        str(value).strip()
        for value in row
        if pd.notna(value) and str(value).strip() not in ["", "|"]
    ]

    if values and len(values) <= 3:
        print(f"Excel row {row_idx + 1}: {' | '.join(values)}")


Excel row 6: EQUITY & EQUITY RELATED
Excel row 7: (a) Listed / awaiting listing on Stock Exchanges
Excel row 8: Equity
Excel row 81: Sub Total | 338230.20999999996 | 65.67999999999995
Excel row 82: Total | 338230.20999999996 | 65.67999999999995
Excel row 83: DEBT INSTRUMENTS
Excel row 84: (a) Listed / awaiting listing on Stock Exchanges
Excel row 85: Government Securities (Central/State)
Excel row 99: Non-Convertible debentures / Bonds
Excel row 120: Sub Total | 60523.90999999999 | 11.76
Excel row 121: (c) Securitized Debt Instruments
Excel row 122: Securitized Debt Instruments
Excel row 124: Sub Total | 2551 | 0.5
Excel row 125: Total | 63074.90999999999 | 12.26
Excel row 126: UNITS ISSUED BY REIT & INVIT
Excel row 127: Units issued by InvIT
Excel row 129: Units issued by ReIT
Excel row 133: Sub Total | 11125.15 | 2.16
Excel row 134: Total | 11125.15 | 2.16
Excel row 135: MONEY MARKET INSTRUMENTS
Excel row 136: Certificate Of Deposit (CD)
Excel row 138: Treasury bills
Excel row 140: T

In [18]:

# Show rows 5 to 152 (Excel rows 6 to 153)
# Columns B to H contain the main instrument details and values.

display(
    hdfc_sample_df.iloc[5:153, 1:8]
)


,1,2,3,4,5,6,7
5,EQUITY & EQUITY RELATED,NaN,NaN,NaN,NaN,NaN,NaN
6,(a) Listed / awaiting listing on Stock Exchanges,NaN,NaN,NaN,NaN,NaN,NaN
7,Equity,NaN,NaN,NaN,NaN,NaN,NaN
8,INE090A01021,NaN,ICICI Bank Ltd.,Banks,2225700,29942.34,5.82
9,INE002A01018,NaN,Reliance Industries Ltd.,Petroleum Products,2016500,29973.26,5.82
...,...,...,...,...,...,...,...
148,Sub Total,NaN,NaN,NaN,NaN,61310.83,11.91
149,Total,NaN,NaN,NaN,NaN,61310.83,11.91
150,NaN,NaN,NaN,NaN,NaN,NaN,NaN
151,Grand Total,NaN,NaN,NaN,NaN,514908.32,100.0


In [19]:

# Extract candidate holdings from the sample report

candidate_holdings = hdfc_sample_df.iloc[5:152].copy()

candidate_holdings = candidate_holdings.rename(
    columns={
        1: "ISIN",
        3: "Instrument_Name",
        4: "Industry_or_Rating",
        5: "Quantity",
        6: "Market_Value_Lakh",
        7: "NAV_Pct",
    }
)

# Keep rows with an instrument name and numeric market value
candidate_holdings["Market_Value_Lakh"] = pd.to_numeric(
    candidate_holdings["Market_Value_Lakh"],
    errors="coerce"
)

candidate_holdings["NAV_Pct"] = pd.to_numeric(
    candidate_holdings["NAV_Pct"],
    errors="coerce"
)

candidate_holdings = candidate_holdings[
    candidate_holdings["Instrument_Name"].notna()
    & candidate_holdings["Market_Value_Lakh"].notna()
].copy()

# Remove obvious subtotal and total rows
candidate_holdings = candidate_holdings[
    ~candidate_holdings["Instrument_Name"]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(["subtotal", "sub total", "total", "grand total"])
]

candidate_holdings["Source_Row"] = candidate_holdings.index + 1

display(
    candidate_holdings[
        [
            "ISIN",
            "Instrument_Name",
            "Industry_or_Rating",
            "Quantity",
            "Market_Value_Lakh",
            "NAV_Pct",
            "Source_Row",
        ]
    ]
)

print("Candidate holding rows:", len(candidate_holdings))
print(
    "Candidate market value total (₹ lakh):",
    round(candidate_holdings["Market_Value_Lakh"].sum(), 2)
)


,ISIN,Instrument_Name,Industry_or_Rating,Quantity,Market_Value_Lakh,NAV_Pct,Source_Row
8,INE090A01021,ICICI Bank Ltd.,Banks,2225700,29942.34,5.82,9
9,INE002A01018,Reliance Industries Ltd.,Petroleum Products,2016500,29973.26,5.82,10
10,INE040A01034,HDFC Bank Ltd.£,Banks,2641200,26076.57,5.06,11
11,INE397D01024,Bharti Airtel Ltd.,Telecom - Services,731925,15037.40,2.92,12
12,INE062A01020,State Bank of India,Banks,1500000,14055.00,2.73,13
...,...,...,...,...,...,...,...
136,INE160A16QS0,Punjab National Bank^,CRISIL - A1+,500,2480.59,0.48,137
138,IN002025Y263,182 Days TBILL MAT 260326^,Sovereign,5000000,4893.20,0.95,139
140,NaN,TREPS - Tri-party Repo,NaN,NaN,33793.43,6.56,141
145,INF179KC1981,HDFC Gold Exchange Traded Fund,NaN,57946747,59824.22,11.62,146


Candidate holding rows: 115
Candidate market value total (₹ lakh): 514908.32


In [20]:

print("Number of candidate rows:", len(candidate_holdings))

print("\nRows with missing ISIN:")
display(
    candidate_holdings[
        candidate_holdings["ISIN"].isna()
    ][
        [
            "Instrument_Name",
            "Market_Value_Lakh",
            "NAV_Pct",
            "Source_Row"
        ]
    ]
)

print("\nPotential summary or cash-related rows:")
display(
    candidate_holdings[
        candidate_holdings["Instrument_Name"]
        .astype(str)
        .str.contains(
            r"subtotal|grand total|^total$|net current assets|treps",
            case=False,
            regex=True
        )
    ][
        [
            "Instrument_Name",
            "Market_Value_Lakh",
            "NAV_Pct",
            "Source_Row"
        ]
    ]
)


Number of candidate rows: 115

Rows with missing ISIN:


,Instrument_Name,Market_Value_Lakh,NAV_Pct,Source_Row
140,TREPS - Tri-party Repo,33793.43,6.56,141
147,Net Current Assets,1486.61,0.29,148



Potential summary or cash-related rows:


,Instrument_Name,Market_Value_Lakh,NAV_Pct,Source_Row
140,TREPS - Tri-party Repo,33793.43,6.56,141
147,Net Current Assets,1486.61,0.29,148


In [21]:

import pandas as pd
from pathlib import Path

validation_results = []

for file_path in hdfc_files:
    try:
        excel_file = pd.ExcelFile(file_path)

        # Use the same holdings sheet as the sample report.
        sheet_name = "MY2005"

        if sheet_name not in excel_file.sheet_names:
            validation_results.append({
                "File": file_path.name,
                "Status": "Holdings sheet not found",
                "Candidate_Rows": None,
                "Market_Value_Lakh": None
            })
            continue

        df = pd.read_excel(file_path, sheet_name=sheet_name, header=None)

        candidates = df.iloc[5:152].copy()
        candidates = candidates.rename(columns={
            1: "ISIN",
            3: "Instrument_Name",
            6: "Market_Value_Lakh",
            7: "NAV_Pct"
        })

        candidates["Market_Value_Lakh"] = pd.to_numeric(
            candidates["Market_Value_Lakh"], errors="coerce"
        )

        candidates = candidates[
            candidates["Instrument_Name"].notna()
            & candidates["Market_Value_Lakh"].notna()
        ].copy()

        candidates = candidates[
            ~candidates["Instrument_Name"].astype(str).str.strip().str.lower()
            .isin(["subtotal", "sub total", "total", "grand total"])
        ]

        validation_results.append({
            "File": file_path.name,
            "Status": "OK",
            "Candidate_Rows": len(candidates),
            "Market_Value_Lakh": round(
                candidates["Market_Value_Lakh"].sum(), 2
            )
        })

    except Exception as e:
        validation_results.append({
            "File": file_path.name,
            "Status": f"Error: {e}",
            "Candidate_Rows": None,
            "Market_Value_Lakh": None
        })

hdfc_validation = pd.DataFrame(validation_results)

display(hdfc_validation)

print("Reports checked:", len(hdfc_validation))
print("Reports with OK status:", (hdfc_validation["Status"] == "OK").sum())
print("Reports requiring review:", (hdfc_validation["Status"] != "OK").sum())


,File,Status,Candidate_Rows,Market_Value_Lakh
0,Monthly HDFC Multi Asset Allocation Fund - 30 ...,OK,118,578243.16
1,Monthly HDFC Multi Asset Allocation Fund - 31 ...,OK,120,582839.00
2,Monthly HDFC Multi-Asset Allocation Fund - 28 ...,OK,117,515642.97
3,Monthly HDFC Multi-Asset Allocation Fund - 30 ...,OK,120,582483.14
4,Monthly HDFC Multi-Asset Allocation Fund - 30 ...,OK,121,503580.81
5,Monthly HDFC Multi-Asset Allocation Fund - 31 ...,OK,117,545966.16
6,Monthly HDFC Multi-Asset Allocation Fund - 31 ...,OK,117,565243.71
7,Monthly HDFC Multi-Asset Allocation Fund - 31 ...,OK,120,513147.00
8,Monthly HDFC Multi-Asset Allocation Fund - 31 ...,OK,120,490530.61
9,Monthly HDFC Multi-Asset Allocation Fund - 31 ...,OK,120,505379.73


Reports checked: 33
Reports with OK status: 33
Reports requiring review: 0


In [22]:

import pandas as pd

reconciliation = []

for file_path in hdfc_files:
    df = pd.read_excel(file_path, sheet_name="MY2005", header=None)

    # Locate the Grand Total row using the instrument-name column (D).
    names = df.iloc[:, 3].astype(str).str.strip()
    grand_total_rows = df.index[
        names.str.fullmatch("Grand Total", case=False, na=False)
    ].tolist()

    if not grand_total_rows:
        reconciliation.append({
            "File": file_path.name,
            "Extracted_Total_Lakh": None,
            "Grand_Total_Lakh": None,
            "Difference_Lakh": None,
            "Status": "Grand Total not found"
        })
        continue

    # Use the last matching Grand Total row, if there is more than one.
    gt_row = grand_total_rows[-1]
    reported_total = pd.to_numeric(df.iloc[gt_row, 6], errors="coerce")

    # Recreate the candidate extraction.
    candidates = df.iloc[5:152].copy()
    candidates = candidates.rename(columns={
        3: "Instrument_Name",
        6: "Market_Value_Lakh"
    })

    candidates["Market_Value_Lakh"] = pd.to_numeric(
        candidates["Market_Value_Lakh"], errors="coerce"
    )
    candidates = candidates[
        candidates["Instrument_Name"].notna()
        & candidates["Market_Value_Lakh"].notna()
    ].copy()

    candidates = candidates[
        ~candidates["Instrument_Name"].astype(str).str.strip().str.lower()
        .isin(["subtotal", "sub total", "total", "grand total"])
    ]

    extracted_total = candidates["Market_Value_Lakh"].sum()
    difference = extracted_total - reported_total

    reconciliation.append({
        "File": file_path.name,
        "Extracted_Total_Lakh": round(extracted_total, 2),
        "Grand_Total_Lakh": round(reported_total, 2),
        "Difference_Lakh": round(difference, 2),
        "Status": "MATCH" if abs(difference) < 0.02 else "REVIEW"
    })

hdfc_reconciliation = pd.DataFrame(reconciliation)

display(hdfc_reconciliation)
print("\nReconciliation summary:")
print(hdfc_reconciliation["Status"].value_counts(dropna=False))


,File,Extracted_Total_Lakh,Grand_Total_Lakh,Difference_Lakh,Status
0,Monthly HDFC Multi Asset Allocation Fund - 30 ...,None,None,None,Grand Total not found
1,Monthly HDFC Multi Asset Allocation Fund - 31 ...,None,None,None,Grand Total not found
2,Monthly HDFC Multi-Asset Allocation Fund - 28 ...,None,None,None,Grand Total not found
3,Monthly HDFC Multi-Asset Allocation Fund - 30 ...,None,None,None,Grand Total not found
4,Monthly HDFC Multi-Asset Allocation Fund - 30 ...,None,None,None,Grand Total not found
5,Monthly HDFC Multi-Asset Allocation Fund - 31 ...,None,None,None,Grand Total not found
6,Monthly HDFC Multi-Asset Allocation Fund - 31 ...,None,None,None,Grand Total not found
7,Monthly HDFC Multi-Asset Allocation Fund - 31 ...,None,None,None,Grand Total not found
8,Monthly HDFC Multi-Asset Allocation Fund - 31 ...,None,None,None,Grand Total not found
9,Monthly HDFC Multi-Asset Allocation Fund - 31 ...,None,None,None,Grand Total not found



Reconciliation summary:
Status
Grand Total not found    33
Name: count, dtype: int64


In [23]:

# Inspect the bottom section of the sample HDFC report
display(
    hdfc_sample_df.iloc[140:160, 0:8]
)


,0,1,2,3,4,5,6,7
140,NaN,NaN,NaN,TREPS - Tri-party Repo,NaN,NaN,33793.43,6.56
141,NaN,Sub Total,NaN,NaN,NaN,NaN,41167.22,7.99
142,NaN,Total,NaN,NaN,NaN,NaN,41167.22,7.99
143,NaN,OTHERS,NaN,NaN,NaN,NaN,NaN,NaN
144,NaN,Mutual Fund Units,NaN,NaN,NaN,NaN,NaN,NaN
145,|,INF179KC1981,NaN,HDFC Gold Exchange Traded Fund,NaN,57946747,59824.22,11.62
146,NaN,Net Current Assets,NaN,NaN,NaN,NaN,NaN,NaN
147,NaN,NaN,NaN,Net Current Assets,NaN,NaN,1486.61,0.29
148,NaN,Sub Total,NaN,NaN,NaN,NaN,61310.83,11.91
149,NaN,Total,NaN,NaN,NaN,NaN,61310.83,11.91


In [24]:

reconciliation = []

for file_path in hdfc_files:
    df = pd.read_excel(file_path, sheet_name="MY2005", header=None)

    # Grand Total is in column 1; market value is in column 6.
    labels = df.iloc[:, 1].astype(str).str.strip()
    gt_rows = df.index[
        labels.str.fullmatch("Grand Total", case=False, na=False)
    ].tolist()

    if not gt_rows:
        reconciliation.append({
            "File": file_path.name,
            "Extracted_Total_Lakh": None,
            "Grand_Total_Lakh": None,
            "Difference_Lakh": None,
            "Status": "Grand Total not found"
        })
        continue

    gt_row = gt_rows[-1]
    reported_total = pd.to_numeric(df.iloc[gt_row, 6], errors="coerce")

    # Apply the candidate extraction rules.
    candidates = df.iloc[5:152].copy()
    candidates = candidates.rename(columns={
        3: "Instrument_Name",
        6: "Market_Value_Lakh"
    })

    candidates["Market_Value_Lakh"] = pd.to_numeric(
        candidates["Market_Value_Lakh"], errors="coerce"
    )

    candidates = candidates[
        candidates["Instrument_Name"].notna()
        & candidates["Market_Value_Lakh"].notna()
    ].copy()

    candidates = candidates[
        ~candidates["Instrument_Name"].astype(str).str.strip().str.lower()
        .isin(["subtotal", "sub total", "total", "grand total"])
    ]

    extracted_total = candidates["Market_Value_Lakh"].sum()
    difference = extracted_total - reported_total

    reconciliation.append({
        "File": file_path.name,
        "Extracted_Total_Lakh": round(extracted_total, 2),
        "Grand_Total_Lakh": round(reported_total, 2),
        "Difference_Lakh": round(difference, 2),
        "Status": (
            "MATCH"
            if pd.notna(difference) and abs(difference) < 0.02
            else "REVIEW"
        )
    })

hdfc_reconciliation = pd.DataFrame(reconciliation)

display(hdfc_reconciliation)

print("\nReconciliation summary:")
print(hdfc_reconciliation["Status"].value_counts(dropna=False))


,File,Extracted_Total_Lakh,Grand_Total_Lakh,Difference_Lakh,Status
0,Monthly HDFC Multi Asset Allocation Fund - 30 ...,578243.16,578181.98,61.18,REVIEW
1,Monthly HDFC Multi Asset Allocation Fund - 31 ...,582839.00,603125.58,-20286.58,REVIEW
2,Monthly HDFC Multi-Asset Allocation Fund - 28 ...,515642.97,588377.95,-72734.98,REVIEW
3,Monthly HDFC Multi-Asset Allocation Fund - 30 ...,582483.14,588705.30,-6222.16,REVIEW
4,Monthly HDFC Multi-Asset Allocation Fund - 30 ...,503580.81,588140.78,-84559.97,REVIEW
5,Monthly HDFC Multi-Asset Allocation Fund - 31 ...,545966.16,545966.16,0.00,MATCH
6,Monthly HDFC Multi-Asset Allocation Fund - 31 ...,565243.71,571360.81,-6117.10,REVIEW
7,Monthly HDFC Multi-Asset Allocation Fund - 31 ...,513147.00,597991.64,-84844.64,REVIEW
8,Monthly HDFC Multi-Asset Allocation Fund - 31 ...,490530.61,555581.53,-65050.92,REVIEW
9,Monthly HDFC Multi-Asset Allocation Fund - 31 ...,505379.73,588760.66,-83380.93,REVIEW



Reconciliation summary:
Status
MATCH     22
REVIEW    11
Name: count, dtype: int64


In [25]:

review_files = hdfc_reconciliation[
    hdfc_reconciliation["Status"] == "REVIEW"
].copy()

display(
    review_files[
        [
            "File",
            "Extracted_Total_Lakh",
            "Grand_Total_Lakh",
            "Difference_Lakh"
        ]
    ].to_string(index=False)
)

print("\nNumber of reports requiring review:", len(review_files))


'                                                               File  Extracted_Total_Lakh  Grand_Total_Lakh  Difference_Lakh\nMonthly HDFC Multi Asset Allocation Fund - 30 September 2026_0.xlsx             578243.16         578181.98            61.18\n Monthly HDFC Multi Asset Allocation Fund - 31 August 2026 (1).xlsx             582839.00         603125.58        -20286.58\n   Monthly HDFC Multi-Asset Allocation Fund - 28 February 2026.xlsx             515642.97         588377.95        -72734.98\n      Monthly HDFC Multi-Asset Allocation Fund - 30 April 2026.xlsx             582483.14         588705.30         -6222.16\n       Monthly HDFC Multi-Asset Allocation Fund - 30 June 2026.xlsx             503580.81         588140.78        -84559.97\n    Monthly HDFC Multi-Asset Allocation Fund - 31 January 2026.xlsx             565243.71         571360.81         -6117.10\n       Monthly HDFC Multi-Asset Allocation Fund - 31 July 2026.xlsx             513147.00         597991.64        -8


Number of reports requiring review: 11


In [26]:

from pathlib import Path
import pandas as pd

june_file = next(
    f for f in hdfc_files
    if f.name == "Monthly HDFC Multi-Asset Allocation Fund - 30 June 2026.xlsx"
)

june_df = pd.read_excel(june_file, sheet_name="MY2005", header=None)

# Find the Grand Total row in column B (index 1).
labels = june_df.iloc[:, 1].astype(str).str.strip()
gt_rows = june_df.index[
    labels.str.fullmatch("Grand Total", case=False, na=False)
].tolist()

print("Grand Total row index:", gt_rows)

if gt_rows:
    gt_row = gt_rows[-1]
    display(june_df.iloc[max(0, gt_row - 12):gt_row + 3, 0:8])
else:
    print("Grand Total label not found in column B.")
    display(june_df.iloc[-25:, 0:8])


Grand Total row index: [160]


,0,1,2,3,4,5,6,7
148,NaN,NaN,NaN,TREPS - Tri-party Repo,NaN,NaN,7027.58,1.19
149,NaN,Sub Total,NaN,NaN,NaN,NaN,12790.6,2.17
150,NaN,Total,NaN,NaN,NaN,NaN,12790.6,2.17
151,NaN,OTHERS,NaN,NaN,NaN,NaN,NaN,NaN
152,NaN,Mutual Fund Units,NaN,NaN,NaN,NaN,NaN,NaN
153,|,INF179KC1981,NaN,HDFC Gold Exchange Traded Fund,NaN,53833490,64470.99,10.96
154,NaN,INF179KC1DI2,NaN,HDFC Silver ETF,NaN,1000000,2132.5,0.36
155,NaN,Net Current Assets,NaN,NaN,NaN,NaN,NaN,NaN
156,NaN,NaN,NaN,Net Current Assets,NaN,NaN,17956.48,3.02
157,NaN,Sub Total,NaN,NaN,NaN,NaN,84559.97,14.34


In [27]:
# Use the June 2026 report to test the corrected extraction boundary

gt_row = gt_rows[-1]

candidate_holdings = june_df.iloc[5:gt_row].copy()

candidate_holdings = candidate_holdings.rename(columns={
    1: "ISIN",
    3: "Instrument_Name",
    4: "Industry_or_Rating",
    5: "Quantity",
    6: "Market_Value_Lakh",
    7: "NAV_Pct"
})

candidate_holdings["Market_Value_Lakh"] = pd.to_numeric(
    candidate_holdings["Market_Value_Lakh"], errors="coerce"
)

candidate_holdings["NAV_Pct"] = pd.to_numeric(
    candidate_holdings["NAV_Pct"], errors="coerce"
)

candidate_holdings = candidate_holdings[
    candidate_holdings["Instrument_Name"].notna()
    & candidate_holdings["Market_Value_Lakh"].notna()
].copy()

candidate_holdings = candidate_holdings[
    ~candidate_holdings["Instrument_Name"]
    .astype(str).str.strip().str.lower()
    .isin(["subtotal", "sub total", "total", "grand total"])
]

extracted_total = candidate_holdings["Market_Value_Lakh"].sum()
grand_total = pd.to_numeric(june_df.iloc[gt_row, 6], errors="coerce")

print(f"Extracted holdings total: ₹{extracted_total:,.2f} lakh")
print(f"Reported Grand Total:     ₹{grand_total:,.2f} lakh")
print(f"Difference:               ₹{extracted_total - grand_total:,.2f} lakh")

display(
    candidate_holdings[
        candidate_holdings["Instrument_Name"].str.contains(
            "Gold ETF|Silver ETF|Net Current Assets",
            case=False,
            na=False
        )
    ][["ISIN", "Instrument_Name", "Market_Value_Lakh", "NAV_Pct"]]
)

Extracted holdings total: ₹588,140.78 lakh
Reported Grand Total:     ₹588,140.78 lakh
Difference:               ₹0.00 lakh


,ISIN,Instrument_Name,Market_Value_Lakh,NAV_Pct
154,INF179KC1DI2,HDFC Silver ETF,2132.50,0.36
156,NaN,Net Current Assets,17956.48,3.02


In [28]:
import pandas as pd

validation_results = []

for file in hdfc_files:
    try:
        df = pd.read_excel(file, sheet_name="MY2005", header=None)

        # Find Grand Total in column B (index 1)
        labels = df.iloc[:, 1].astype(str).str.strip()
        gt_rows = df.index[
            labels.str.fullmatch("Grand Total", case=False, na=False)
        ].tolist()

        if not gt_rows:
            validation_results.append({
                "Source_File": file.name,
                "Extracted_Total_Lakh": None,
                "Grand_Total_Lakh": None,
                "Difference_Lakh": None,
                "Status": "REVIEW - Grand Total not found"
            })
            continue

        gt_row = gt_rows[-1]
        holdings = df.iloc[5:gt_row].copy()

        holdings = holdings.rename(columns={
            1: "ISIN",
            3: "Instrument_Name",
            4: "Industry_or_Rating",
            5: "Quantity",
            6: "Market_Value_Lakh",
            7: "NAV_Pct"
        })

        holdings["Market_Value_Lakh"] = pd.to_numeric(
            holdings["Market_Value_Lakh"], errors="coerce"
        )

        holdings = holdings[
            holdings["Instrument_Name"].notna()
            & holdings["Market_Value_Lakh"].notna()
        ]

        holdings = holdings[
            ~holdings["Instrument_Name"].astype(str).str.strip().str.lower()
            .isin(["subtotal", "sub total", "total", "grand total"])
        ]

        extracted_total = holdings["Market_Value_Lakh"].sum()
        grand_total = pd.to_numeric(df.iloc[gt_row, 6], errors="coerce")
        difference = extracted_total - grand_total

        validation_results.append({
            "Source_File": file.name,
            "Extracted_Total_Lakh": round(extracted_total, 2),
            "Grand_Total_Lakh": round(grand_total, 2),
            "Difference_Lakh": round(difference, 2),
            "Status": "MATCH" if abs(difference) < 0.01 else "REVIEW"
        })

    except Exception as e:
        validation_results.append({
            "Source_File": file.name,
            "Extracted_Total_Lakh": None,
            "Grand_Total_Lakh": None,
            "Difference_Lakh": None,
            "Status": f"ERROR - {str(e)[:100]}"
        })

hdfc_validation_corrected = pd.DataFrame(validation_results)

print("Validation summary:")
display(hdfc_validation_corrected["Status"].value_counts().to_frame("Reports"))

print("Reports requiring review:")
display(
    hdfc_validation_corrected[
        hdfc_validation_corrected["Status"] != "MATCH"
    ][["Source_File", "Extracted_Total_Lakh",
       "Grand_Total_Lakh", "Difference_Lakh", "Status"]]
)

Validation summary:


,Reports
Status,
MATCH,32
REVIEW,1


Reports requiring review:


,Source_File,Extracted_Total_Lakh,Grand_Total_Lakh,Difference_Lakh,Status
13,Monthly HDFC Multi-Asset Fund - 30 April 2025....,423873.0,424057.35,-184.35,REVIEW


In [29]:
april_file = next(
    f for f in hdfc_files
    if f.name == "Monthly HDFC Multi-Asset Fund - 30 April 2025.xlsx"
)

april_df = pd.read_excel(april_file, sheet_name="MY2005", header=None)

labels = april_df.iloc[:, 1].astype(str).str.strip()
gt_rows = april_df.index[
    labels.str.fullmatch("Grand Total", case=False, na=False)
].tolist()

gt_row = gt_rows[-1]

print("Grand Total row index:", gt_row)
print("Reported Grand Total:", april_df.iloc[gt_row, 6])

display(april_df.iloc[max(0, gt_row - 25):gt_row + 1, 0:8])

Grand Total row index: 150
Reported Grand Total: 424057.34999999974


,0,1,2,3,4,5,6,7
125,NaN,Units issued by InvIT,NaN,NaN,NaN,NaN,NaN,NaN
126,NaN,INE0GGX23010,NaN,POWERGRID Infrastructure Investment Trust,Power,438658,351.01,0.08
127,NaN,Units issued by ReIT,NaN,NaN,NaN,NaN,NaN,NaN
128,NaN,INE041025011,NaN,Embassy Office Parks REIT,Realty,1246270,4757.88,1.12
129,NaN,INE0FDU25010,NaN,BROOKFIELD INDIA REAL ESTATE TRUST,Realty,1000000,2894.5,0.68
130,NaN,INE0NDH25011,NaN,Nexus Select Trust REIT,Realty,1300000,1685.45,0.4
131,NaN,Sub Total,NaN,NaN,NaN,NaN,9688.84,2.28
132,NaN,Total,NaN,NaN,NaN,NaN,9688.84,2.28
133,NaN,MONEY MARKET INSTRUMENTS,NaN,NaN,NaN,NaN,NaN,NaN
134,NaN,Certificate Of Deposit (CD),NaN,NaN,NaN,NaN,NaN,NaN


In [30]:
# Inspect the OPTIONS section and its surrounding rows
display(april_df.iloc[140:150, 0:12])

,0,1,2,3,4,5,6,7,8,9,10,11
140,NaN,OTHERS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
141,NaN,Mutual Fund Units,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
142,|,INF179KC1981,NaN,HDFC Gold Exchange Traded Fund,NaN,57946747,47336.7,11.16,NaN,NaN,NaN,NaN
143,NaN,Net Current Assets,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
144,NaN,NaN,NaN,Net Current Assets,NaN,NaN,489.76,0.17,NaN,NaN,NaN,NaN
145,NaN,OPTIONS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
146,NaN,NaN,NaN,NaN,NaN,150000,184.35,0.04,NaN,NaN,NaN,NaN
147,NaN,Sub Total,NaN,NaN,NaN,NaN,48010.81,11.37,NaN,NaN,NaN,NaN
148,NaN,Total,NaN,NaN,NaN,NaN,48010.81,11.37,NaN,NaN,NaN,NaN
149,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [31]:
unnamed_rows = []

for file in hdfc_files:
    df = pd.read_excel(file, sheet_name="MY2005", header=None)

    labels = df.iloc[:, 1].astype(str).str.strip()
    gt_rows = df.index[
        labels.str.fullmatch("Grand Total", case=False, na=False)
    ].tolist()

    if not gt_rows:
        continue

    gt_row = gt_rows[-1]

    # Check rows before Grand Total where market value exists
    values = pd.to_numeric(df.iloc[:gt_row, 6], errors="coerce")
    names = df.iloc[:gt_row, 3].astype(str).str.strip()

    mask = values.notna() & (
        names.eq("") | names.str.lower().eq("nan")
    )

    for idx in df.index[:gt_row][mask]:
        unnamed_rows.append({
            "Source_File": file.name,
            "Row_Index": idx,
            "ISIN_or_Label": df.iloc[idx, 1],
            "Market_Value_Lakh": values.loc[idx],
            "NAV_Pct": df.iloc[idx, 7],
            "Previous_Row_Label": df.iloc[idx - 1, 1]
                if idx > 0 else None
        })

unnamed_df = pd.DataFrame(unnamed_rows)

print("Number of unnamed rows with market values:", len(unnamed_df))
display(unnamed_df)

Number of unnamed rows with market values: 356


,Source_File,Row_Index,ISIN_or_Label,Market_Value_Lakh,NAV_Pct,Previous_Row_Label
0,Monthly HDFC Multi Asset Allocation Fund - 30 ...,80,Sub Total,399437.96,69.09,INE201P01022
1,Monthly HDFC Multi Asset Allocation Fund - 30 ...,85,Sub Total,9444.98,1.64,INE0NDH25011
2,Monthly HDFC Multi Asset Allocation Fund - 30 ...,86,Total,408882.94,70.73,Sub Total
3,Monthly HDFC Multi Asset Allocation Fund - 30 ...,121,Sub Total,59286.29,10.21,INE261F08EF5
4,Monthly HDFC Multi Asset Allocation Fund - 30 ...,125,Sub Total,2390.10,0.41,INE2I7F15012
...,...,...,...,...,...,...
351,Monthly HDFC Multi-Asset Fund - 31 October 202...,133,Total,11125.15,2.16,Sub Total
352,Monthly HDFC Multi-Asset Fund - 31 October 202...,141,Sub Total,41167.22,7.99,NaN
353,Monthly HDFC Multi-Asset Fund - 31 October 202...,142,Total,41167.22,7.99,Sub Total
354,Monthly HDFC Multi-Asset Fund - 31 October 202...,148,Sub Total,61310.83,11.91,NaN


In [32]:
# Remove subtotal/total rows and inspect genuinely unnamed value rows

summary_labels = {
    "subtotal", "sub total", "total", "grand total"
}

unnamed_review = unnamed_df.copy()

unnamed_review["Label_Clean"] = (
    unnamed_review["ISIN_or_Label"]
    .astype(str).str.strip().str.lower()
)

unnamed_review = unnamed_review[
    ~unnamed_review["Label_Clean"].isin(summary_labels)
].copy()

unnamed_review = unnamed_review.drop(columns=["Label_Clean"])

print("Unnamed rows after removing summary rows:", len(unnamed_review))
display(unnamed_review)

Unnamed rows after removing summary rows: 1


,Source_File,Row_Index,ISIN_or_Label,Market_Value_Lakh,NAV_Pct,Previous_Row_Label
157,Monthly HDFC Multi-Asset Fund - 30 April 2025....,146,NaN,184.35,0.04,OPTIONS


In [35]:

import re
import pandas as pd
from pathlib import Path

# --------------------------------------------------
# 1. SET UP OUTPUT PATHS
# --------------------------------------------------

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

holdings_path = OUTPUT_DIR / "HDFC_Full_Monthly_Holdings_Log.csv"
checks_path = OUTPUT_DIR / "HDFC_Holdings_Reconciliation.csv"

all_holdings = []
extraction_checks = []

summary_labels = {
    "subtotal",
    "sub total",
    "total",
    "grand total"
}

# --------------------------------------------------
# 2. PROCESS EVERY HDFC REPORT
# --------------------------------------------------

for file in hdfc_files:

    try:
        df = pd.read_excel(
            file,
            sheet_name="MY2005",
            header=None
        )

        # Extract report date from filename.
        # Supports dates such as 30 April 2025,
        # including filenames with suffixes like _0 or (1).

        date_match = re.search(
            r"(\d{1,2}\s+[A-Za-z]+\s+\d{4})",
            file.name
        )

        if date_match:
            report_date = pd.to_datetime(
                date_match.group(1),
                format="%d %B %Y",
                errors="coerce"
            )
        else:
            report_date = pd.NaT

        # Find Grand Total in column B (index 1).

        labels = df.iloc[:, 1].astype(str).str.strip()

        gt_rows = df.index[
            labels.str.fullmatch(
                "Grand Total",
                case=False,
                na=False
            )
        ].tolist()

        if not gt_rows:
            extraction_checks.append({
                "Source_File": file.name,
                "Report_Date": report_date,
                "Report_Month": (
                    str(report_date.to_period("M"))
                    if pd.notna(report_date) else None
                ),
                "Extracted_Total_Lakh": None,
                "Grand_Total_Lakh": None,
                "Difference_Lakh": None,
                "Status": "REVIEW - Grand Total not found"
            })
            continue

        gt_row = gt_rows[-1]

        # Reported Grand Total is in column G (index 6).

        grand_total = pd.to_numeric(
            df.iloc[gt_row, 6],
            errors="coerce"
        )

        # --------------------------------------------------
        # 3. EXTRACT HOLDINGS BEFORE GRAND TOTAL
        # --------------------------------------------------

        holdings = df.iloc[5:gt_row].copy()

        holdings = holdings.rename(columns={
            1: "ISIN_or_Label",
            3: "Instrument_Name",
            4: "Industry_or_Rating",
            5: "Quantity",
            6: "Market_Value_Lakh",
            7: "NAV_Pct",
            8: "Yield",
            9: "YTC",
            10: "Derivative_Pct_NAV",
            11: "Unhedged_Pct_NAV"
        })

        # Convert numeric fields.

        for col in [
            "Quantity",
            "Market_Value_Lakh",
            "NAV_Pct",
            "Yield",
            "YTC",
            "Derivative_Pct_NAV",
            "Unhedged_Pct_NAV"
        ]:
            holdings[col] = pd.to_numeric(
                holdings[col],
                errors="coerce"
            )

        # Retain rows with a reported market value.

        holdings = holdings[
            holdings["Market_Value_Lakh"].notna()
        ].copy()

        # Clean text fields.

        for col in [
            "ISIN_or_Label",
            "Instrument_Name",
            "Industry_or_Rating"
        ]:
            holdings[col] = (
                holdings[col]
                .astype("string")
                .str.strip()
            )

        # Remove section subtotal/total rows.

        isin_label = (
            holdings["ISIN_or_Label"]
            .fillna("")
            .str.lower()
        )

        instrument_name = (
            holdings["Instrument_Name"]
            .fillna("")
            .str.lower()
        )

        holdings = holdings[
            ~isin_label.isin(summary_labels)
            & ~instrument_name.isin(summary_labels)
        ].copy()

        # Preserve genuine rows whose instrument name is blank.
        # The confirmed April 2025 example belongs to OPTIONS.

        blank_name = (
            holdings["Instrument_Name"].isna()
            | holdings["Instrument_Name"].eq("")
        )

        options_row = (
            blank_name
            & holdings["ISIN_or_Label"]
                .fillna("")
                .str.upper()
                .eq("OPTIONS")
        )

        holdings.loc[
            options_row,
            "Instrument_Name"
        ] = "Unidentified holding - OPTIONS"

        # Any other unnamed rows remain explicitly unidentified.
        other_blank_name = (
            holdings["Instrument_Name"].isna()
            | holdings["Instrument_Name"].eq("")
        )

        holdings.loc[
            other_blank_name,
            "Instrument_Name"
        ] = "Unidentified holding - review"

        # Add report metadata.

        holdings["Report_Date"] = report_date

        holdings["Report_Month"] = (
            str(report_date.to_period("M"))
            if pd.notna(report_date)
            else None
        )

        holdings["Source_File"] = file.name

        # Excel row numbers are one-based.
        holdings["Source_Row"] = holdings.index + 1

        all_holdings.append(holdings)

        # --------------------------------------------------
        # 4. RECONCILE EXTRACTED TOTAL WITH REPORTED TOTAL
        # --------------------------------------------------

        extracted_total = holdings[
            "Market_Value_Lakh"
        ].sum()

        difference = (
            extracted_total - grand_total
            if pd.notna(grand_total)
            else None
        )

        if pd.isna(report_date):
            status = "REVIEW - Invalid or missing report date"
        elif pd.isna(grand_total):
            status = "REVIEW - Grand Total value missing"
        elif abs(difference) < 0.01:
            status = "MATCH"
        else:
            status = "REVIEW - Total mismatch"

        extraction_checks.append({
            "Source_File": file.name,
            "Report_Date": report_date,
            "Report_Month": (
                str(report_date.to_period("M"))
                if pd.notna(report_date)
                else None
            ),
            "Extracted_Total_Lakh": round(
                extracted_total, 2
            ),
            "Grand_Total_Lakh": (
                round(grand_total, 2)
                if pd.notna(grand_total)
                else None
            ),
            "Difference_Lakh": (
                round(difference, 2)
                if difference is not None
                and pd.notna(difference)
                else None
            ),
            "Status": status
        })

    except Exception as e:
        extraction_checks.append({
            "Source_File": file.name,
            "Report_Date": None,
            "Report_Month": None,
            "Extracted_Total_Lakh": None,
            "Grand_Total_Lakh": None,
            "Difference_Lakh": None,
            "Status": f"ERROR - {str(e)[:150]}"
        })

# --------------------------------------------------
# 5. COMBINE RESULTS
# --------------------------------------------------

hdfc_extraction_checks = pd.DataFrame(
    extraction_checks
)

if all_holdings:
    hdfc_full_holdings = pd.concat(
        all_holdings,
        ignore_index=True
    )
else:
    hdfc_full_holdings = pd.DataFrame()

# --------------------------------------------------
# 6. SAVE CSV FILES
# --------------------------------------------------

hdfc_full_holdings.to_csv(
    holdings_path,
    index=False
)

hdfc_extraction_checks.to_csv(
    checks_path,
    index=False
)

# --------------------------------------------------
# 7. DISPLAY VALIDATION RESULTS
# --------------------------------------------------

print("HDFC EXTRACTION SUMMARY")
print("-----------------------")
print("Source reports:", len(hdfc_files))
print("Reports checked:", len(hdfc_extraction_checks))
print("Extracted holding rows:", len(hdfc_full_holdings))

if not hdfc_full_holdings.empty:
    print(
        "Reports with valid dates:",
        hdfc_extraction_checks["Report_Date"].notna().sum()
    )

    print(
        "Unique report months:",
        hdfc_full_holdings["Report_Month"].nunique()
    )

    print("\nHoldings by report month:")
    display(
        hdfc_full_holdings.groupby("Report_Month")
        .size()
        .reset_index(name="Holding_Rows")
        .sort_values("Report_Month")
    )

print("\nReconciliation status:")
display(
    hdfc_extraction_checks["Status"]
    .value_counts()
    .rename_axis("Status")
    .reset_index(name="Reports")
)

print("\nReports requiring review:")
display(
    hdfc_extraction_checks[
        hdfc_extraction_checks["Status"] != "MATCH"
    ]
)

print("\nUnidentified holdings:")
if not hdfc_full_holdings.empty:
    display(
        hdfc_full_holdings[
            hdfc_full_holdings["Instrument_Name"]
            .str.contains(
                "Unidentified",
                case=False,
                na=False
            )
        ][[
            "Report_Month",
            "ISIN_or_Label",
            "Instrument_Name",
            "Market_Value_Lakh",
            "NAV_Pct",
            "Source_File",
            "Source_Row"
        ]]
    )

print("\nSaved files:")
print(holdings_path)
print(checks_path)


HDFC EXTRACTION SUMMARY
-----------------------
Source reports: 33
Reports checked: 33
Extracted holding rows: 3694
Reports with valid dates: 33
Unique report months: 33

Holdings by report month:


,Report_Month,Holding_Rows
0,2024-01,88
1,2024-02,89
2,2024-03,91
3,2024-04,96
4,2024-05,96
5,2024-06,102
6,2024-07,102
7,2024-08,106
8,2024-09,109
9,2024-10,105



Reconciliation status:


,Status,Reports
0,MATCH,33



Reports requiring review:


,Source_File,Report_Date,Report_Month,Extracted_Total_Lakh,Grand_Total_Lakh,Difference_Lakh,Status



Unidentified holdings:


,Report_Month,ISIN_or_Label,Instrument_Name,Market_Value_Lakh,NAV_Pct,Source_File,Source_Row
1631,2025-04,<NA>,Unidentified holding - review,184.35,0.04,Monthly HDFC Multi-Asset Fund - 30 April 2025....,147



Saved files:
c:\Users\rocks\OneDrive\Desktop\AMC Project Data\02_CLEAN_DATA\HDFC_Full_Monthly_Holdings_Log.csv
c:\Users\rocks\OneDrive\Desktop\AMC Project Data\02_CLEAN_DATA\HDFC_Holdings_Reconciliation.csv


In [36]:

import pandas as pd

# Load the saved full holdings log and existing metal-only log
hdfc_full = pd.read_csv(
    OUTPUT_DIR / "HDFC_Full_Monthly_Holdings_Log.csv"
)

hdfc_metals_check = pd.read_csv(
    OUTPUT_DIR / "HDFC_Monthly_Metal_Holdings_Log.csv"
)

# Identify Gold/Silver ETF rows in the full holdings log
metal_mask = hdfc_full["Instrument_Name"].str.contains(
    r"Gold.*ETF|Silver.*ETF",
    case=False,
    na=False
)

hdfc_full_metals = hdfc_full.loc[
    metal_mask,
    [
        "Report_Month",
        "Instrument_Name",
        "Market_Value_Lakh",
        "NAV_Pct",
        "Source_File"
    ]
].copy()

print("Gold/Silver ETF rows in full log:", len(hdfc_full_metals))
print("Rows in existing metal-only log:", len(hdfc_metals_check))

print("\nMetal ETF rows by month:")
display(
    hdfc_full_metals.groupby(
        ["Report_Month", "Instrument_Name"]
    ).size().reset_index(name="Rows")
)

print("\nExtracted Gold/Silver ETF entries:")
display(
    hdfc_full_metals.sort_values(
        ["Report_Month", "Instrument_Name"]
    )
)


Gold/Silver ETF rows in full log: 8
Rows in existing metal-only log: 41

Metal ETF rows by month:


,Report_Month,Instrument_Name,Rows
0,2026-02,HDFC Silver ETF,1
1,2026-03,HDFC Silver ETF,1
2,2026-04,HDFC Silver ETF,1
3,2026-05,HDFC Silver ETF,1
4,2026-06,HDFC Silver ETF,1
5,2026-07,HDFC Silver ETF,1
6,2026-08,HDFC Silver ETF,1
7,2026-09,HDFC Silver ETF,1



Extracted Gold/Silver ETF entries:


,Report_Month,Instrument_Name,Market_Value_Lakh,NAV_Pct,Source_File
361,2026-02,HDFC Silver ETF,1215.79,0.21,Monthly HDFC Multi-Asset Allocation Fund - 28 ...
1089,2026-03,HDFC Silver ETF,2167.10,0.39,Monthly HDFC Multi-Asset Allocation Fund - 31 ...
483,2026-04,HDFC Silver ETF,2277.50,0.39,Monthly HDFC Multi-Asset Allocation Fund - 30 ...
1212,2026-05,HDFC Silver ETF,2497.50,0.42,Monthly HDFC Multi-Asset Allocation Fund - 31 ...
607,2026-06,HDFC Silver ETF,2132.50,0.36,Monthly HDFC Multi-Asset Allocation Fund - 30 ...
966,2026-07,HDFC Silver ETF,10330.00,1.73,Monthly HDFC Multi-Asset Allocation Fund - 31 ...
241,2026-08,HDFC Silver ETF,11150.50,1.85,Monthly HDFC Multi Asset Allocation Fund - 31 ...
113,2026-09,HDFC Silver ETF,10481.50,1.81,Monthly HDFC Multi Asset Allocation Fund - 30 ...


In [37]:

# Search the full HDFC holdings log for gold and silver instruments

metal_names = hdfc_full[
    hdfc_full["Instrument_Name"].str.contains(
        r"gold|silver",
        case=False,
        na=False
    )
].copy()

print("Total rows containing Gold or Silver:", len(metal_names))

print("\nInstrument name counts:")
display(
    metal_names["Instrument_Name"]
    .value_counts()
    .rename_axis("Instrument_Name")
    .reset_index(name="Rows")
)

print("\nMonthly entries:")
display(
    metal_names[
        [
            "Report_Month",
            "ISIN_or_Label",
            "Instrument_Name",
            "Market_Value_Lakh",
            "NAV_Pct"
        ]
    ].sort_values(["Report_Month", "Instrument_Name"])
)


Total rows containing Gold or Silver: 41

Instrument name counts:


,Instrument_Name,Rows
0,HDFC Gold Exchange Traded Fund,33
1,HDFC Silver ETF,8



Monthly entries:


,Report_Month,ISIN_or_Label,Instrument_Name,Market_Value_Lakh,NAV_Pct
2721,2024-01,INF179KC1981,HDFC Gold Exchange Traded Fund,27197.83,11.30
1417,2024-02,INF179KC1981,HDFC Gold Exchange Traded Fund,26949.13,10.66
3144,2024-03,INF179KC1981,HDFC Gold Exchange Traded Fund,29003.39,10.98
1513,2024-04,INF179KC1981,HDFC Gold Exchange Traded Fund,31266.56,11.17
3356,2024-05,INF179KC1981,HDFC Gold Exchange Traded Fund,31425.73,10.85
1732,2024-06,INF179KC1981,HDFC Gold Exchange Traded Fund,32796.06,10.44
2938,2024-07,INF179KC1981,HDFC Gold Exchange Traded Fund,36686.47,10.96
2407,2024-08,INF179KC1981,HDFC Gold Exchange Traded Fund,37886.24,10.75
2182,2024-09,INF179KC1981,HDFC Gold Exchange Traded Fund,42154.96,11.39
3577,2024-10,INF179KC1981,HDFC Gold Exchange Traded Fund,44295.82,11.78


In [38]:
import pandas as pd
from pathlib import Path

full_log_path = OUTPUT_DIR / "HDFC_Full_Monthly_Holdings_Log.csv"

hdfc_full = pd.read_csv(full_log_path)

# 1. Check report-month coverage
expected_months = pd.period_range(
    start="2024-01",
    end="2026-09",
    freq="M"
).astype(str)

actual_months = sorted(
    hdfc_full["Report_Month"].dropna().astype(str).unique()
)

print("Total extracted rows:", len(hdfc_full))
print("Unique report months:", len(actual_months))
print("Missing months:", sorted(set(expected_months) - set(actual_months)))
print("Unexpected months:", sorted(set(actual_months) - set(expected_months)))

# 2. Check duplicate records using source file and source row
duplicate_mask = hdfc_full.duplicated(
    subset=["Source_File", "Source_Row"],
    keep=False
)

print("\nRows involved in duplicate source records:", duplicate_mask.sum())

# 3. Review unidentified holdings
if "Instrument_Name" in hdfc_full.columns:
    unidentified_mask = hdfc_full["Instrument_Name"].astype(str).str.contains(
        "unidentified|unknown|review",
        case=False,
        na=False
    )

    unidentified = hdfc_full.loc[
        unidentified_mask,
        [
            "Report_Month",
            "Instrument_Name",
            "Market_Value_Lakh",
            "NAV_Pct",
            "Source_File",
            "Source_Row"
        ]
    ]

    print("\nUnidentified holdings:", len(unidentified))
    display(unidentified)

# 4. Show the saved file's columns
print("\nColumns in full holdings log:")
print(hdfc_full.columns.tolist())

Total extracted rows: 3694
Unique report months: 33
Missing months: []
Unexpected months: []

Rows involved in duplicate source records: 0

Unidentified holdings: 1


,Report_Month,Instrument_Name,Market_Value_Lakh,NAV_Pct,Source_File,Source_Row
1631,2025-04,Unidentified holding - review,184.35,0.04,Monthly HDFC Multi-Asset Fund - 30 April 2025....,147



Columns in full holdings log:
['0', 'ISIN_or_Label', '2', 'Instrument_Name', 'Industry_or_Rating', 'Quantity', 'Market_Value_Lakh', 'NAV_Pct', 'Yield', 'YTC', 'Derivative_Pct_NAV', 'Unhedged_Pct_NAV', 'Report_Date', 'Report_Month', 'Source_File', 'Source_Row']


In [39]:
from pathlib import Path
import pandas as pd

full_log_path = OUTPUT_DIR / "HDFC_Full_Monthly_Holdings_Log.csv"

hdfc_full = pd.read_csv(full_log_path)

# Correct the two column headers inherited from the source spreadsheet.
hdfc_full = hdfc_full.rename(columns={
    "0": "Source_Separator",
    "ISIN_or_Label": "ISIN_or_Label",
    "2": "Coupon",
})

# Save the corrected cleaned log.
hdfc_full.to_csv(full_log_path, index=False)

print("Updated file:", full_log_path)
print("\nCorrected columns:")
print(hdfc_full.columns.tolist())

print("\nUnidentified holdings retained for review:")
display(
    hdfc_full[
        hdfc_full["Instrument_Name"].astype(str).str.contains(
            "unidentified|unknown|review",
            case=False,
            na=False
        )
    ][
        [
            "Report_Month",
            "Instrument_Name",
            "Market_Value_Lakh",
            "NAV_Pct",
            "Source_File",
            "Source_Row",
        ]
    ]
)

Updated file: c:\Users\rocks\OneDrive\Desktop\AMC Project Data\02_CLEAN_DATA\HDFC_Full_Monthly_Holdings_Log.csv

Corrected columns:
['Source_Separator', 'ISIN_or_Label', 'Coupon', 'Instrument_Name', 'Industry_or_Rating', 'Quantity', 'Market_Value_Lakh', 'NAV_Pct', 'Yield', 'YTC', 'Derivative_Pct_NAV', 'Unhedged_Pct_NAV', 'Report_Date', 'Report_Month', 'Source_File', 'Source_Row']

Unidentified holdings retained for review:


,Report_Month,Instrument_Name,Market_Value_Lakh,NAV_Pct,Source_File,Source_Row
1631,2025-04,Unidentified holding - review,184.35,0.04,Monthly HDFC Multi-Asset Fund - 30 April 2025....,147


In [40]:
import pandas as pd

hdfc_path = OUTPUT_DIR / "HDFC_Full_Monthly_Holdings_Log.csv"

hdfc_full = pd.read_csv(hdfc_path)

# Display the first 20 rows
display(hdfc_full.head(20))

,Source_Separator,ISIN_or_Label,Coupon,Instrument_Name,Industry_or_Rating,Quantity,Market_Value_Lakh,NAV_Pct,Yield,YTC,Derivative_Pct_NAV,Unhedged_Pct_NAV,Report_Date,Report_Month,Source_File,Source_Row
0,|,INE040A01034,NaN,HDFC Bank Ltd.£,Banks,5599983.0,39687.08,6.86,NaN,NaN,0.00,6.86,2026-09-30,2026-09,Monthly HDFC Multi Asset Allocation Fund - 30 ...,9
1,|,INE002A01018,NaN,Reliance Industries Ltd.,Petroleum Products,2366500.0,28090.36,4.86,NaN,NaN,1.22,3.64,2026-09-30,2026-09,Monthly HDFC Multi Asset Allocation Fund - 30 ...,10
2,|,INE090A01021,NaN,ICICI Bank Ltd.,Banks,2073100.0,27400.16,4.74,NaN,NaN,0.05,4.28,2026-09-30,2026-09,Monthly HDFC Multi Asset Allocation Fund - 30 ...,11
3,|,INE397D01024,NaN,Bharti Airtel Ltd.,Telecom - Services,951850.0,16724.00,2.89,NaN,NaN,1.13,1.47,2026-09-30,2026-09,Monthly HDFC Multi Asset Allocation Fund - 30 ...,12
4,|,INE237A01036,NaN,Kotak Mahindra Bank Limited,Banks,3600000.0,15012.00,2.60,NaN,NaN,0.01,1.95,2026-09-30,2026-09,Monthly HDFC Multi Asset Allocation Fund - 30 ...,13
5,|,INE238A01034,NaN,Axis Bank Ltd.,Banks,1200000.0,14712.00,2.54,NaN,NaN,0.00,2.54,2026-09-30,2026-09,Monthly HDFC Multi Asset Allocation Fund - 30 ...,14
6,|,INE585B01010,NaN,Maruti Suzuki India Limited,Automobiles,93800.0,11225.05,1.94,NaN,NaN,0.00,1.94,2026-09-30,2026-09,Monthly HDFC Multi Asset Allocation Fund - 30 ...,15
7,|,INE733E01010,NaN,NTPC Limited,Power,3351500.0,10816.97,1.87,NaN,NaN,0.75,1.12,2026-09-30,2026-09,Monthly HDFC Multi Asset Allocation Fund - 30 ...,16
8,NaN,INE044A01036,NaN,Sun Pharmaceutical Industries Ltd.,Pharmaceuticals & Biotechnology,546750.0,9932.26,1.72,NaN,NaN,0.14,1.58,2026-09-30,2026-09,Monthly HDFC Multi Asset Allocation Fund - 30 ...,17
9,NaN,INE361B01024,NaN,Divis Laboratories Ltd.,Pharmaceuticals & Biotechnology,100000.0,9291.50,1.61,NaN,NaN,0.00,1.61,2026-09-30,2026-09,Monthly HDFC Multi Asset Allocation Fund - 30 ...,18


In [41]:
print("Dataset shape:", hdfc_full.shape)

print("\nColumn names:")
print(hdfc_full.columns.tolist())

print("\nData types and non-null counts:")
hdfc_full.info()

Dataset shape: (3694, 16)

Column names:
['Source_Separator', 'ISIN_or_Label', 'Coupon', 'Instrument_Name', 'Industry_or_Rating', 'Quantity', 'Market_Value_Lakh', 'NAV_Pct', 'Yield', 'YTC', 'Derivative_Pct_NAV', 'Unhedged_Pct_NAV', 'Report_Date', 'Report_Month', 'Source_File', 'Source_Row']

Data types and non-null counts:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3694 entries, 0 to 3693
Data columns (total 16 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Source_Separator    330 non-null    object 
 1   ISIN_or_Label       3608 non-null   object 
 2   Coupon              962 non-null    float64
 3   Instrument_Name     3694 non-null   object 
 4   Industry_or_Rating  3571 non-null   object 
 5   Quantity            3628 non-null   float64
 6   Market_Value_Lakh   3694 non-null   float64
 7   NAV_Pct             3673 non-null   float64
 8   Yield               1088 non-null   float64
 9   YTC                 57 n

In [42]:
display(
    hdfc_full[
        [
            "Report_Month",
            "ISIN_or_Label",
            "Instrument_Name",
            "Industry_or_Rating",
            "Market_Value_Lakh",
            "NAV_Pct",
        ]
    ].head(30)
)

,Report_Month,ISIN_or_Label,Instrument_Name,Industry_or_Rating,Market_Value_Lakh,NAV_Pct
0,2026-09,INE040A01034,HDFC Bank Ltd.£,Banks,39687.08,6.86
1,2026-09,INE002A01018,Reliance Industries Ltd.,Petroleum Products,28090.36,4.86
2,2026-09,INE090A01021,ICICI Bank Ltd.,Banks,27400.16,4.74
3,2026-09,INE397D01024,Bharti Airtel Ltd.,Telecom - Services,16724.00,2.89
4,2026-09,INE237A01036,Kotak Mahindra Bank Limited,Banks,15012.00,2.60
5,2026-09,INE238A01034,Axis Bank Ltd.,Banks,14712.00,2.54
6,2026-09,INE585B01010,Maruti Suzuki India Limited,Automobiles,11225.05,1.94
7,2026-09,INE733E01010,NTPC Limited,Power,10816.97,1.87
8,2026-09,INE044A01036,Sun Pharmaceutical Industries Ltd.,Pharmaceuticals & Biotechnology,9932.26,1.72
9,2026-09,INE361B01024,Divis Laboratories Ltd.,Pharmaceuticals & Biotechnology,9291.50,1.61


In [43]:
import pandas as pd
from IPython.display import display

# Load the cleaned HDFC Gold/Silver dataset
hdfc_cleaned_path = OUTPUT_DIR / "HDFC_Monthly_Metal_Holdings_Log.csv"
hdfc_cleaned = pd.read_csv(hdfc_cleaned_path)

# Sort by month and metal
hdfc_cleaned = hdfc_cleaned.sort_values(
    ["Report_Month", "Metal"]
).reset_index(drop=True)

# Display the dataset
print("HDFC Gold/Silver Cleaned Dataset")
print("Total rows:", len(hdfc_cleaned))
print("Total columns:", len(hdfc_cleaned.columns))
print("\nColumns:")
print(hdfc_cleaned.columns.tolist())

display(hdfc_cleaned)

HDFC Gold/Silver Cleaned Dataset
Total rows: 41
Total columns: 8

Columns:
['Report_Date', 'Report_Month', 'Metal', 'Instrument_Name', 'Market_Value_Lakh', 'NAV_Pct', 'Source_File', 'Source_Row']


,Report_Date,Report_Month,Metal,Instrument_Name,Market_Value_Lakh,NAV_Pct,Source_File,Source_Row
0,2024-01-31,2024-01,Gold,HDFC Gold Exchange Traded Fund,27197.83,11.30,Monthly HDFC Multi-Asset Fund - 31 January 202...,114
1,2024-02-29,2024-02,Gold,HDFC Gold Exchange Traded Fund,26949.13,10.66,Monthly HDFC Multi-Asset Fund - 29 February 20...,115
2,2024-03-31,2024-03,Gold,HDFC Gold Exchange Traded Fund,29003.39,10.98,Monthly HDFC Multi-Asset Fund - 31 March 2024....,117
3,2024-04-30,2024-04,Gold,HDFC Gold Exchange Traded Fund,31266.56,11.17,Monthly HDFC Multi-Asset Fund - 30 April 2024....,122
4,2024-05-31,2024-05,Gold,HDFC Gold Exchange Traded Fund,31425.73,10.85,Monthly HDFC Multi-Asset Fund - 31 May 2024.xlsx,122
5,2024-06-30,2024-06,Gold,HDFC Gold Exchange Traded Fund,32796.06,10.44,Monthly HDFC Multi-Asset Fund - 30 June 2024.xlsx,131
6,2024-07-31,2024-07,Gold,HDFC Gold Exchange Traded Fund,36686.47,10.96,Monthly HDFC Multi-Asset Fund - 31 July 2024.xlsx,131
7,2024-08-31,2024-08,Gold,HDFC Gold Exchange Traded Fund,37886.24,10.75,Monthly HDFC Multi-Asset Fund - 31 August 2024...,135
8,2024-09-30,2024-09,Gold,HDFC Gold Exchange Traded Fund,42154.96,11.39,Monthly HDFC Multi-Asset Fund - 30 September 2...,131
9,2024-10-31,2024-10,Gold,HDFC Gold Exchange Traded Fund,44295.82,11.78,Monthly HDFC Multi-Asset Fund - 31 October 202...,131


In [44]:
display(
    hdfc_cleaned[
        ["Report_Month", "Metal", "Instrument_Name",
         "Market_Value_Lakh", "NAV_Pct"]
    ]
)

,Report_Month,Metal,Instrument_Name,Market_Value_Lakh,NAV_Pct
0,2024-01,Gold,HDFC Gold Exchange Traded Fund,27197.83,11.30
1,2024-02,Gold,HDFC Gold Exchange Traded Fund,26949.13,10.66
2,2024-03,Gold,HDFC Gold Exchange Traded Fund,29003.39,10.98
3,2024-04,Gold,HDFC Gold Exchange Traded Fund,31266.56,11.17
4,2024-05,Gold,HDFC Gold Exchange Traded Fund,31425.73,10.85
5,2024-06,Gold,HDFC Gold Exchange Traded Fund,32796.06,10.44
6,2024-07,Gold,HDFC Gold Exchange Traded Fund,36686.47,10.96
7,2024-08,Gold,HDFC Gold Exchange Traded Fund,37886.24,10.75
8,2024-09,Gold,HDFC Gold Exchange Traded Fund,42154.96,11.39
9,2024-10,Gold,HDFC Gold Exchange Traded Fund,44295.82,11.78


In [45]:
import pandas as pd
from pathlib import Path
from IPython.display import display

# Locate the cleaned HDFC Gold/Silver dataset
metal_file = OUTPUT_DIR / "HDFC_Monthly_Metal_Holdings_Log.csv"

# Load the dataset
hdfc_metals_clean = pd.read_csv(metal_file)

# Display the dataset
print("Dataset shape:", hdfc_metals_clean.shape)
print("\nColumn names:")
print(hdfc_metals_clean.columns.tolist())

print("\nCleaned Gold/Silver dataset:")
display(hdfc_metals_clean)

# Basic validation
print("\nRecords by metal:")
display(hdfc_metals_clean["Metal"].value_counts().rename_axis("Metal").to_frame("Records"))

print("\nRecords by month and metal:")
display(
    hdfc_metals_clean
    .groupby(["Report_Month", "Metal"])
    .size()
    .unstack(fill_value=0)
)

print("\nDuplicate month-metal records:",
      hdfc_metals_clean.duplicated(["Report_Month", "Metal"]).sum())

print("\nMissing values by column:")
display(hdfc_metals_clean.isna().sum().to_frame("Missing_Values"))

print("\nSaved file:")
print(metal_file)

Dataset shape: (41, 8)

Column names:
['Report_Date', 'Report_Month', 'Metal', 'Instrument_Name', 'Market_Value_Lakh', 'NAV_Pct', 'Source_File', 'Source_Row']

Cleaned Gold/Silver dataset:


,Report_Date,Report_Month,Metal,Instrument_Name,Market_Value_Lakh,NAV_Pct,Source_File,Source_Row
0,2024-01-31,2024-01,Gold,HDFC Gold Exchange Traded Fund,27197.83,11.30,Monthly HDFC Multi-Asset Fund - 31 January 202...,114
1,2024-02-29,2024-02,Gold,HDFC Gold Exchange Traded Fund,26949.13,10.66,Monthly HDFC Multi-Asset Fund - 29 February 20...,115
2,2024-03-31,2024-03,Gold,HDFC Gold Exchange Traded Fund,29003.39,10.98,Monthly HDFC Multi-Asset Fund - 31 March 2024....,117
3,2024-04-30,2024-04,Gold,HDFC Gold Exchange Traded Fund,31266.56,11.17,Monthly HDFC Multi-Asset Fund - 30 April 2024....,122
4,2024-05-31,2024-05,Gold,HDFC Gold Exchange Traded Fund,31425.73,10.85,Monthly HDFC Multi-Asset Fund - 31 May 2024.xlsx,122
5,2024-06-30,2024-06,Gold,HDFC Gold Exchange Traded Fund,32796.06,10.44,Monthly HDFC Multi-Asset Fund - 30 June 2024.xlsx,131
6,2024-07-31,2024-07,Gold,HDFC Gold Exchange Traded Fund,36686.47,10.96,Monthly HDFC Multi-Asset Fund - 31 July 2024.xlsx,131
7,2024-08-31,2024-08,Gold,HDFC Gold Exchange Traded Fund,37886.24,10.75,Monthly HDFC Multi-Asset Fund - 31 August 2024...,135
8,2024-09-30,2024-09,Gold,HDFC Gold Exchange Traded Fund,42154.96,11.39,Monthly HDFC Multi-Asset Fund - 30 September 2...,131
9,2024-10-31,2024-10,Gold,HDFC Gold Exchange Traded Fund,44295.82,11.78,Monthly HDFC Multi-Asset Fund - 31 October 202...,131



Records by metal:


,Records
Metal,
Gold,33
Silver,8



Records by month and metal:


Metal,Gold,Silver
Report_Month,,
2024-01,1,0
2024-02,1,0
2024-03,1,0
2024-04,1,0
2024-05,1,0
2024-06,1,0
2024-07,1,0
2024-08,1,0
2024-09,1,0



Duplicate month-metal records: 0

Missing values by column:


,Missing_Values
Report_Date,0
Report_Month,0
Metal,0
Instrument_Name,0
Market_Value_Lakh,0
NAV_Pct,0
Source_File,0
Source_Row,0



Saved file:
c:\Users\rocks\OneDrive\Desktop\AMC Project Data\02_CLEAN_DATA\HDFC_Monthly_Metal_Holdings_Log.csv


In [46]:
hdfc_gold_silver = pd.read_csv(
    OUTPUT_DIR / "HDFC_Monthly_Metal_Holdings_Log.csv"
)

display(hdfc_gold_silver)

,Report_Date,Report_Month,Metal,Instrument_Name,Market_Value_Lakh,NAV_Pct,Source_File,Source_Row
0,2024-01-31,2024-01,Gold,HDFC Gold Exchange Traded Fund,27197.83,11.30,Monthly HDFC Multi-Asset Fund - 31 January 202...,114
1,2024-02-29,2024-02,Gold,HDFC Gold Exchange Traded Fund,26949.13,10.66,Monthly HDFC Multi-Asset Fund - 29 February 20...,115
2,2024-03-31,2024-03,Gold,HDFC Gold Exchange Traded Fund,29003.39,10.98,Monthly HDFC Multi-Asset Fund - 31 March 2024....,117
3,2024-04-30,2024-04,Gold,HDFC Gold Exchange Traded Fund,31266.56,11.17,Monthly HDFC Multi-Asset Fund - 30 April 2024....,122
4,2024-05-31,2024-05,Gold,HDFC Gold Exchange Traded Fund,31425.73,10.85,Monthly HDFC Multi-Asset Fund - 31 May 2024.xlsx,122
5,2024-06-30,2024-06,Gold,HDFC Gold Exchange Traded Fund,32796.06,10.44,Monthly HDFC Multi-Asset Fund - 30 June 2024.xlsx,131
6,2024-07-31,2024-07,Gold,HDFC Gold Exchange Traded Fund,36686.47,10.96,Monthly HDFC Multi-Asset Fund - 31 July 2024.xlsx,131
7,2024-08-31,2024-08,Gold,HDFC Gold Exchange Traded Fund,37886.24,10.75,Monthly HDFC Multi-Asset Fund - 31 August 2024...,135
8,2024-09-30,2024-09,Gold,HDFC Gold Exchange Traded Fund,42154.96,11.39,Monthly HDFC Multi-Asset Fund - 30 September 2...,131
9,2024-10-31,2024-10,Gold,HDFC Gold Exchange Traded Fund,44295.82,11.78,Monthly HDFC Multi-Asset Fund - 31 October 202...,131


In [48]:

print("Columns in hdfc_reconciliation:")
print(hdfc_reconciliation.columns.tolist())

display(hdfc_reconciliation.head())


Columns in hdfc_reconciliation:
['File', 'Extracted_Total_Lakh', 'Grand_Total_Lakh', 'Difference_Lakh', 'Status']


,File,Extracted_Total_Lakh,Grand_Total_Lakh,Difference_Lakh,Status
0,Monthly HDFC Multi Asset Allocation Fund - 30 ...,578243.16,578181.98,61.18,REVIEW
1,Monthly HDFC Multi Asset Allocation Fund - 31 ...,582839.00,603125.58,-20286.58,REVIEW
2,Monthly HDFC Multi-Asset Allocation Fund - 28 ...,515642.97,588377.95,-72734.98,REVIEW
3,Monthly HDFC Multi-Asset Allocation Fund - 30 ...,582483.14,588705.30,-6222.16,REVIEW
4,Monthly HDFC Multi-Asset Allocation Fund - 30 ...,503580.81,588140.78,-84559.97,REVIEW


In [49]:

# Show HDFC reports that need reconciliation review

hdfc_review = hdfc_reconciliation.loc[
    hdfc_reconciliation["Status"] != "MATCH"
].copy()

display(hdfc_review)


,File,Extracted_Total_Lakh,Grand_Total_Lakh,Difference_Lakh,Status
0,Monthly HDFC Multi Asset Allocation Fund - 30 ...,578243.16,578181.98,61.18,REVIEW
1,Monthly HDFC Multi Asset Allocation Fund - 31 ...,582839.00,603125.58,-20286.58,REVIEW
2,Monthly HDFC Multi-Asset Allocation Fund - 28 ...,515642.97,588377.95,-72734.98,REVIEW
3,Monthly HDFC Multi-Asset Allocation Fund - 30 ...,582483.14,588705.30,-6222.16,REVIEW
4,Monthly HDFC Multi-Asset Allocation Fund - 30 ...,503580.81,588140.78,-84559.97,REVIEW
6,Monthly HDFC Multi-Asset Allocation Fund - 31 ...,565243.71,571360.81,-6117.10,REVIEW
7,Monthly HDFC Multi-Asset Allocation Fund - 31 ...,513147.00,597991.64,-84844.64,REVIEW
8,Monthly HDFC Multi-Asset Allocation Fund - 31 ...,490530.61,555581.53,-65050.92,REVIEW
9,Monthly HDFC Multi-Asset Allocation Fund - 31 ...,505379.73,588760.66,-83380.93,REVIEW
13,Monthly HDFC Multi-Asset Fund - 30 April 2025....,423873.00,424057.35,-184.35,REVIEW


In [50]:
display(
    hdfc_reconciliation.loc[
        hdfc_reconciliation["Status"] != "MATCH"
    ].sort_values("File").reset_index(drop=True)
)

,File,Extracted_Total_Lakh,Grand_Total_Lakh,Difference_Lakh,Status
0,Monthly HDFC Multi Asset Allocation Fund - 30 ...,578243.16,578181.98,61.18,REVIEW
1,Monthly HDFC Multi Asset Allocation Fund - 31 ...,582839.00,603125.58,-20286.58,REVIEW
2,Monthly HDFC Multi-Asset Allocation Fund - 28 ...,515642.97,588377.95,-72734.98,REVIEW
3,Monthly HDFC Multi-Asset Allocation Fund - 30 ...,582483.14,588705.30,-6222.16,REVIEW
4,Monthly HDFC Multi-Asset Allocation Fund - 30 ...,503580.81,588140.78,-84559.97,REVIEW
5,Monthly HDFC Multi-Asset Allocation Fund - 31 ...,565243.71,571360.81,-6117.10,REVIEW
6,Monthly HDFC Multi-Asset Allocation Fund - 31 ...,513147.00,597991.64,-84844.64,REVIEW
7,Monthly HDFC Multi-Asset Allocation Fund - 31 ...,490530.61,555581.53,-65050.92,REVIEW
8,Monthly HDFC Multi-Asset Allocation Fund - 31 ...,505379.73,588760.66,-83380.93,REVIEW
9,Monthly HDFC Multi-Asset Fund - 30 April 2025....,423873.00,424057.35,-184.35,REVIEW


In [51]:
# Select the June 2026 HDFC report
june_file = next(
    f for f in hdfc_files
    if f.name == "Monthly HDFC Multi-Asset Allocation Fund - 30 June 2026.xlsx"
)

# Read the full holdings sheet
june_df = pd.read_excel(
    june_file,
    sheet_name="MY2005",
    header=None
)

# Find the Grand Total row
labels = june_df.iloc[:, 1].astype(str).str.strip()

gt_rows = june_df.index[
    labels.str.fullmatch("Grand Total", case=False, na=False)
].tolist()

print("Grand Total row index:", gt_rows)

# Display rows immediately before and after Grand Total
if gt_rows:
    gt_row = gt_rows[-1]
    display(june_df.iloc[max(0, gt_row - 15):gt_row + 3, 0:8])
else:
    print("Grand Total not found in column B.")
    display(june_df.tail(25))

Grand Total row index: [160]


,0,1,2,3,4,5,6,7
145,NaN,INE692A16KU1,NaN,Union Bank of India^,ICRA - A1+,500,2409.65,0.41
146,NaN,INE028A16KX2,NaN,Bank of Baroda^,CARE - A1+,200,966.11,0.16
147,NaN,TREPS - Tri-party Repo,NaN,NaN,NaN,NaN,NaN,NaN
148,NaN,NaN,NaN,TREPS - Tri-party Repo,NaN,NaN,7027.58,1.19
149,NaN,Sub Total,NaN,NaN,NaN,NaN,12790.6,2.17
150,NaN,Total,NaN,NaN,NaN,NaN,12790.6,2.17
151,NaN,OTHERS,NaN,NaN,NaN,NaN,NaN,NaN
152,NaN,Mutual Fund Units,NaN,NaN,NaN,NaN,NaN,NaN
153,|,INF179KC1981,NaN,HDFC Gold Exchange Traded Fund,NaN,53833490,64470.99,10.96
154,NaN,INF179KC1DI2,NaN,HDFC Silver ETF,NaN,1000000,2132.5,0.36


In [52]:
# Check where the Grand Total appears in every HDFC report
hdfc_grand_total_rows = []

for file in hdfc_files:
    df = pd.read_excel(file, sheet_name="MY2005", header=None)

    labels = df.iloc[:, 1].astype(str).str.strip()
    matches = df.index[
        labels.str.fullmatch("Grand Total", case=False, na=False)
    ].tolist()

    hdfc_grand_total_rows.append({
        "File": file.name,
        "Grand_Total_Row_Index": matches[-1] if matches else None,
        "Grand_Total_Found": bool(matches)
    })

hdfc_grand_total_check = pd.DataFrame(hdfc_grand_total_rows)

display(
    hdfc_grand_total_check.sort_values(
        "Grand_Total_Row_Index",
        na_position="first"
    ).reset_index(drop=True)
)

print("Reports checked:", len(hdfc_grand_total_check))
print(
    "Reports without Grand Total:",
    (~hdfc_grand_total_check["Grand_Total_Found"]).sum()
)

,File,Grand_Total_Row_Index,Grand_Total_Found
0,Monthly HDFC Multi-Asset Fund - 31 January 202...,119,True
1,Monthly HDFC Multi-Asset Fund - 29 February 20...,120,True
2,Monthly HDFC Multi-Asset Fund - 31 March 2024....,122,True
3,Monthly HDFC Multi-Asset Fund - 30 April 2024....,127,True
4,Monthly HDFC Multi-Asset Fund - 31 May 2024.xlsx,127,True
5,Monthly HDFC Multi-Asset Fund - 31 October 202...,136,True
6,Monthly HDFC Multi-Asset Fund - 30 June 2024.xlsx,136,True
7,Monthly HDFC Multi-Asset Fund - 31 July 2024.xlsx,136,True
8,Monthly HDFC Multi-Asset Fund - 30 November 20...,138,True
9,Monthly HDFC Multi-Asset Fund - 31 August 2024...,140,True


Reports checked: 33
Reports without Grand Total: 0


In [53]:
# Extract June 2026 holdings up to its actual Grand Total row

gt_row = gt_rows[-1]

june_holdings_test = june_df.iloc[5:gt_row].copy()

june_holdings_test = june_holdings_test.rename(columns={
    1: "ISIN_or_Label",
    3: "Instrument_Name",
    4: "Industry_or_Rating",
    5: "Quantity",
    6: "Market_Value_Lakh",
    7: "NAV_Pct"
})

june_holdings_test["Market_Value_Lakh"] = pd.to_numeric(
    june_holdings_test["Market_Value_Lakh"], errors="coerce"
)

june_holdings_test["NAV_Pct"] = pd.to_numeric(
    june_holdings_test["NAV_Pct"], errors="coerce"
)

# Keep rows with an instrument name and a numeric market value
june_holdings_test = june_holdings_test[
    june_holdings_test["Instrument_Name"].notna()
    & june_holdings_test["Market_Value_Lakh"].notna()
].copy()

# Exclude subtotal and total rows
excluded_labels = [
    "subtotal", "sub total", "total", "grand total"
]

june_holdings_test = june_holdings_test[
    ~june_holdings_test["Instrument_Name"]
    .astype(str).str.strip().str.lower().isin(excluded_labels)
]

extracted_total = june_holdings_test["Market_Value_Lakh"].sum()
reported_total = pd.to_numeric(june_df.iloc[gt_row, 6], errors="coerce")

print(f"Extracted total: ₹{extracted_total:,.2f} lakh")
print(f"Reported Grand Total: ₹{reported_total:,.2f} lakh")
print(f"Difference: ₹{extracted_total - reported_total:,.2f} lakh")
print("Extracted holding rows:", len(june_holdings_test))

display(
    june_holdings_test[
        ["ISIN_or_Label", "Instrument_Name", "Market_Value_Lakh", "NAV_Pct"]
    ].tail(12)
)

Extracted total: ₹588,140.78 lakh
Reported Grand Total: ₹588,140.78 lakh
Difference: ₹0.00 lakh
Extracted holding rows: 124


,ISIN_or_Label,Instrument_Name,Market_Value_Lakh,NAV_Pct
128,INE261F08EN9,National Bank for Agri & Rural Dev.^,497.67,0.08
129,INE261F08EF5,National Bank for Agri & Rural Dev.,250.92,0.04
133,INE2I7F15012,Siddhivinayak Securitisation Trust (Originator...,2470.67,0.42
138,INE0GGX23010,POWERGRID Infrastructure Investment Trust,1504.24,0.26
139,INE2PB023011,RAAJMARG INFRA INVESTMENT TRUST,148.23,0.03
144,INE160A16UV6,Punjab National Bank^,2387.26,0.41
145,INE692A16KU1,Union Bank of India^,2409.65,0.41
146,INE028A16KX2,Bank of Baroda^,966.11,0.16
148,NaN,TREPS - Tri-party Repo,7027.58,1.19
153,INF179KC1981,HDFC Gold Exchange Traded Fund,64470.99,10.96


In [54]:
import pandas as pd

hdfc_reconciliation_dynamic = []

for file in hdfc_files:
    df = pd.read_excel(file, sheet_name="MY2005", header=None)

    # Find the Grand Total row in column B
    labels = df.iloc[:, 1].astype(str).str.strip()
    gt_rows = df.index[
        labels.str.fullmatch("Grand Total", case=False, na=False)
    ].tolist()

    if not gt_rows:
        hdfc_reconciliation_dynamic.append({
            "File": file.name,
            "Extracted_Total_Lakh": None,
            "Grand_Total_Lakh": None,
            "Difference_Lakh": None,
            "Status": "GRAND TOTAL NOT FOUND"
        })
        continue

    gt_row = gt_rows[-1]

    # Read all rows before Grand Total
    holdings = df.iloc[5:gt_row].copy()

    # Market value is in column G (index 6)
    holdings[6] = pd.to_numeric(holdings[6], errors="coerce")

    # Keep rows with an instrument name and numeric market value
    # Instrument names can appear in column D (index 3)
    # or, for some lines, column B (index 1)
    names = holdings[3].where(
        holdings[3].notna(), holdings[1]
    ).astype(str).str.strip()

    valid_names = (
        names.notna()
        & names.ne("")
        & names.str.lower().ne("nan")
        & ~names.str.lower().isin(
            ["subtotal", "sub total", "total", "grand total"]
        )
    )

    valid = holdings[valid_names & holdings[6].notna()]

    extracted_total = valid[6].sum()
    grand_total = pd.to_numeric(df.iloc[gt_row, 6], errors="coerce")
    difference = extracted_total - grand_total

    hdfc_reconciliation_dynamic.append({
        "File": file.name,
        "Extracted_Total_Lakh": round(extracted_total, 2),
        "Grand_Total_Lakh": round(grand_total, 2),
        "Difference_Lakh": round(difference, 2),
        "Status": "MATCH" if abs(difference) < 0.02 else "REVIEW"
    })

hdfc_reconciliation_dynamic = pd.DataFrame(
    hdfc_reconciliation_dynamic
)

print("Reconciliation status:")
display(hdfc_reconciliation_dynamic["Status"].value_counts())

print("Reports still needing review:")
display(
    hdfc_reconciliation_dynamic.loc[
        hdfc_reconciliation_dynamic["Status"] != "MATCH"
    ].sort_values("File")
)

Reconciliation status:


Status
MATCH     32
REVIEW     1
Name: count, dtype: int64

Reports still needing review:


,File,Extracted_Total_Lakh,Grand_Total_Lakh,Difference_Lakh,Status
13,Monthly HDFC Multi-Asset Fund - 30 April 2025....,423873.0,424057.35,-184.35,REVIEW


In [55]:

# Find the April 2025 HDFC report
april_file = next(
    f for f in hdfc_files
    if f.name == "Monthly HDFC Multi-Asset Fund - 30 April 2025.xlsx"
)

# Load the main holdings sheet
april_df = pd.read_excel(
    april_file,
    sheet_name="MY2005",
    header=None
)

# Find the Grand Total row
labels = april_df.iloc[:, 1].astype(str).str.strip()
gt_rows = april_df.index[
    labels.str.fullmatch("Grand Total", case=False, na=False)
].tolist()

print("Grand Total row:", gt_rows)

# Display all rows immediately before and including Grand Total
if gt_rows:
    gt_row = gt_rows[-1]
    display(april_df.iloc[max(0, gt_row - 20):gt_row + 1, 0:8])


Grand Total row: [150]


,0,1,2,3,4,5,6,7
130,NaN,INE0NDH25011,NaN,Nexus Select Trust REIT,Realty,1300000,1685.45,0.4
131,NaN,Sub Total,NaN,NaN,NaN,NaN,9688.84,2.28
132,NaN,Total,NaN,NaN,NaN,NaN,9688.84,2.28
133,NaN,MONEY MARKET INSTRUMENTS,NaN,NaN,NaN,NaN,NaN,NaN
134,NaN,Certificate Of Deposit (CD),NaN,NaN,NaN,NaN,NaN,NaN
135,NaN,INE160A16QS0,NaN,Punjab National Bank^,CRISIL - A1+,500,2398.32,0.57
136,NaN,TREPS - Tri-party Repo,NaN,NaN,NaN,NaN,NaN,NaN
137,NaN,NaN,NaN,TREPS - Tri-party Repo,NaN,NaN,32180.41,7.59
138,NaN,Sub Total,NaN,NaN,NaN,NaN,34578.73,8.16
139,NaN,Total,NaN,NaN,NaN,NaN,34578.73,8.16


In [56]:

# Display the OPTIONS section with all columns
display(april_df.iloc[140:149, :12])


,0,1,2,3,4,5,6,7,8,9,10,11
140,NaN,OTHERS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
141,NaN,Mutual Fund Units,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
142,|,INF179KC1981,NaN,HDFC Gold Exchange Traded Fund,NaN,57946747,47336.7,11.16,NaN,NaN,NaN,NaN
143,NaN,Net Current Assets,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
144,NaN,NaN,NaN,Net Current Assets,NaN,NaN,489.76,0.17,NaN,NaN,NaN,NaN
145,NaN,OPTIONS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
146,NaN,NaN,NaN,NaN,NaN,150000,184.35,0.04,NaN,NaN,NaN,NaN
147,NaN,Sub Total,NaN,NaN,NaN,NaN,48010.81,11.37,NaN,NaN,NaN,NaN
148,NaN,Total,NaN,NaN,NaN,NaN,48010.81,11.37,NaN,NaN,NaN,NaN


In [57]:

# Search the full April 2025 report for option-related details
option_matches = april_df[
    april_df.astype(str).apply(
        lambda col: col.str.contains(
            r"option|call|put|derivative|Nifty|Sensex",
            case=False,
            na=False,
            regex=True
        )
    ).any(axis=1)
]

display(option_matches.iloc[:, :12])


,0,1,2,3,4,5,6,7,8,9,10,11
4,NaN,ISIN,Coupon (%),Name Of the Instrument,Industry+ /Rating,Quantity,Market/ Fair Value (Rs. in Lacs.),% to NAV,Yield,~YTC (AT1/Tier 2 bonds),Derivative\n% to NAV,Unhedged\n% to NAV
145,NaN,OPTIONS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
156,NaN,# Non Sensex Scrips,# Non Sensex Scrips,# Non Sensex Scrips,# Non Sensex Scrips,# Non Sensex Scrips,NaN,NaN,NaN,NaN,NaN,NaN
158,NaN,~ YTC i.e. Yield to Call is disclosed at secur...,~ YTC i.e. Yield to Call is disclosed at secur...,~ YTC i.e. Yield to Call is disclosed at secur...,~ YTC i.e. Yield to Call is disclosed at secur...,~ YTC i.e. Yield to Call is disclosed at secur...,NaN,NaN,NaN,NaN,NaN,NaN
159,NaN,This scheme has exposure to floating rate inst...,This scheme has exposure to floating rate inst...,This scheme has exposure to floating rate inst...,This scheme has exposure to floating rate inst...,This scheme has exposure to floating rate inst...,NaN,NaN,NaN,NaN,NaN,NaN
220,NaN,IDCW Option,16.711,16.223,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
221,NaN,IDCW Option - Direct Plan,20.598,19.98,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
222,NaN,Growth Option,69.45,67.422,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
223,NaN,Growth Option - Direct Plan,77.304,74.982,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
233,NaN,5) Total outstanding exposure in Derivative In...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [58]:

# Display the derivative-related disclosures
display(april_df.iloc[225:240, 0:12])


,0,1,2,3,4,5,6,7,8,9,10,11
225,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
226,NaN,Dividend History - Dividend declared during th...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
227,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
228,NaN,Bonus History - Bonus declared during the mont...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
229,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
230,NaN,2) Total below investment grade or default pro...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
231,NaN,3) Total investments in Foreign Securities / O...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
232,NaN,4) Repo in Corporate Debt : Nil,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
233,NaN,5) Total outstanding exposure in Derivative In...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
234,NaN,6) Portfolio Turnover Ratio : 23.13%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [59]:
import pandas as pd

april_file = next(
    f for f in hdfc_files
    if f.name == "Monthly HDFC Multi-Asset Fund - 30 April 2025.xlsx"
)

april_df = pd.read_excel(
    april_file,
    sheet_name="MY2005",
    header=None
)

# Find the Grand Total row dynamically.
labels = april_df.iloc[:, 1].astype(str).str.strip()
gt_rows = april_df.index[
    labels.str.fullmatch("Grand Total", case=False, na=False)
].tolist()

if gt_rows:
    gt_row = gt_rows[-1]

    # Display the holdings area, including the rows before Grand Total.
    display(april_df.iloc[4:gt_row + 1, 1:8])

    print("Grand Total row index:", gt_row)
    print("Reported Grand Total (lakh):", april_df.iloc[gt_row, 6])
else:
    print("Grand Total row not found.")

,1,2,3,4,5,6,7
4,ISIN,Coupon (%),Name Of the Instrument,Industry+ /Rating,Quantity,Market/ Fair Value (Rs. in Lacs.),% to NAV
5,EQUITY & EQUITY RELATED,NaN,NaN,NaN,NaN,NaN,NaN
6,(a) Listed / awaiting listing on Stock Exchanges,NaN,NaN,NaN,NaN,NaN,NaN
7,Equity,NaN,NaN,NaN,NaN,NaN,NaN
8,INE002A01018,NaN,Reliance Industries Ltd.,Petroleum Products,2016500,28331.83,6.68
...,...,...,...,...,...,...,...
146,NaN,NaN,NaN,NaN,150000,184.35,0.04
147,Sub Total,NaN,NaN,NaN,NaN,48010.81,11.37
148,Total,NaN,NaN,NaN,NaN,48010.81,11.37
149,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Grand Total row index: 150
Reported Grand Total (lakh): 424057.34999999974


In [60]:
# Show the OPTIONS section and its value-bearing row.
display(april_df.iloc[143:149, 0:12])

,0,1,2,3,4,5,6,7,8,9,10,11
143,NaN,Net Current Assets,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
144,NaN,NaN,NaN,Net Current Assets,NaN,NaN,489.76,0.17,NaN,NaN,NaN,NaN
145,NaN,OPTIONS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
146,NaN,NaN,NaN,NaN,NaN,150000,184.35,0.04,NaN,NaN,NaN,NaN
147,NaN,Sub Total,NaN,NaN,NaN,NaN,48010.81,11.37,NaN,NaN,NaN,NaN
148,NaN,Total,NaN,NaN,NaN,NaN,48010.81,11.37,NaN,NaN,NaN,NaN


In [61]:
# Inspect the option row with every column displayed.
pd.set_option("display.max_columns", None)

display(april_df.iloc[145:148].T)

,145,146,147
0,NaN,NaN,NaN
1,OPTIONS,NaN,Sub Total
2,NaN,NaN,NaN
3,NaN,NaN,NaN
4,NaN,NaN,NaN
5,NaN,150000,NaN
6,NaN,184.35,48010.81
7,NaN,0.04,11.37
8,NaN,NaN,NaN
9,NaN,NaN,NaN


In [62]:
# Check whether the option row is excluded because its instrument name is blank.
options_row = april_df.iloc[146]

print("ISIN:", options_row.iloc[1])
print("Instrument name:", options_row.iloc[3])
print("Market value (lakh):", options_row.iloc[6])
print("NAV percentage:", options_row.iloc[7])

print("\nDoes the row have a name?")
print(pd.notna(options_row.iloc[3]))

ISIN: nan
Instrument name: nan
Market value (lakh): 184.35
NAV percentage: 0.04

Does the row have a name?
False


In [63]:
# Extract rows up to the dynamically identified Grand Total.
gt_row = gt_rows[-1]

test = april_df.iloc[5:gt_row].copy()

test["Market_Value_Lakh"] = pd.to_numeric(
    test.iloc[:, 6], errors="coerce"
)

labels = test.iloc[:, 1].astype(str).str.strip().str.lower()

# Exclude summary rows; retain value-bearing rows with blank names.
test = test[
    test["Market_Value_Lakh"].notna()
    & ~labels.isin(["sub total", "subtotal", "total", "grand total"])
].copy()

extracted_total = test["Market_Value_Lakh"].sum()
reported_total = pd.to_numeric(april_df.iloc[gt_row, 6], errors="coerce")

print(f"Extracted total: ₹{extracted_total:,.2f} lakh")
print(f"Reported Grand Total: ₹{reported_total:,.2f} lakh")
print(f"Difference: ₹{extracted_total - reported_total:,.2f} lakh")
print(f"Rows retained: {len(test)}")

# Show rows with a value but no instrument name for review.
unnamed = test[
    test.iloc[:, 3].isna()
    & test["Market_Value_Lakh"].notna()
]
display(unnamed.iloc[:, [1, 3, 6, 7]])

Extracted total: ₹424,057.35 lakh
Reported Grand Total: ₹424,057.35 lakh
Difference: ₹0.00 lakh
Rows retained: 117


,1,3,6,7
146,NaN,NaN,184.35,0.04


In [64]:
# Find rows with a market value but no instrument name
# across all HDFC reports.

unnamed_value_rows = []

for file in hdfc_files:
    try:
        df = pd.read_excel(file, sheet_name="MY2005", header=None)

        if df.shape[1] <= 7:
            continue

        # Locate the Grand Total row in column B.
        labels = df.iloc[:, 1].astype(str).str.strip()
        gt_rows = df.index[
            labels.str.fullmatch("Grand Total", case=False, na=False)
        ].tolist()

        if not gt_rows:
            continue

        gt_row = gt_rows[-1]

        # Inspect the portfolio section before Grand Total.
        section = df.iloc[5:gt_row].copy()
        values = pd.to_numeric(section.iloc[:, 6], errors="coerce")
        names = section.iloc[:, 3]

        mask = values.notna() & names.isna()

        for idx in section.index[mask]:
            unnamed_value_rows.append({
                "File": file.name,
                "Excel_Row": idx + 1,
                "Column_B": df.iloc[idx, 1],
                "Column_C": df.iloc[idx, 2],
                "Column_D": df.iloc[idx, 3],
                "Market_Value_Lakh": values.loc[idx],
                "NAV_Pct": pd.to_numeric(
                    pd.Series([df.iloc[idx, 7]]),
                    errors="coerce"
                ).iloc[0]
            })

    except Exception as e:
        print(f"Could not inspect {file.name}: {e}")

unnamed_value_df = pd.DataFrame(unnamed_value_rows)

print("Unnamed rows with market values:", len(unnamed_value_df))
display(unnamed_value_df)

Unnamed rows with market values: 356


,File,Excel_Row,Column_B,Column_C,Column_D,Market_Value_Lakh,NAV_Pct
0,Monthly HDFC Multi Asset Allocation Fund - 30 ...,81,Sub Total,NaN,NaN,399437.96,69.09
1,Monthly HDFC Multi Asset Allocation Fund - 30 ...,86,Sub Total,NaN,NaN,9444.98,1.64
2,Monthly HDFC Multi Asset Allocation Fund - 30 ...,87,Total,NaN,NaN,408882.94,70.73
3,Monthly HDFC Multi Asset Allocation Fund - 30 ...,122,Sub Total,NaN,NaN,59286.29,10.21
4,Monthly HDFC Multi Asset Allocation Fund - 30 ...,126,Sub Total,NaN,NaN,2390.10,0.41
...,...,...,...,...,...,...,...
351,Monthly HDFC Multi-Asset Fund - 31 October 202...,134,Total,NaN,NaN,11125.15,2.16
352,Monthly HDFC Multi-Asset Fund - 31 October 202...,142,Sub Total,NaN,NaN,41167.22,7.99
353,Monthly HDFC Multi-Asset Fund - 31 October 202...,143,Total,NaN,NaN,41167.22,7.99
354,Monthly HDFC Multi-Asset Fund - 31 October 202...,149,Sub Total,NaN,NaN,61310.83,11.91


In [65]:
# Remove subtotal and total rows from the unnamed-value review.
labels = unnamed_value_df["Column_B"].astype(str).str.strip().str.lower()

unnamed_review = unnamed_value_df[
    ~labels.isin(["sub total", "subtotal", "total", "grand total"])
].copy()

print("Unnamed rows after excluding totals:", len(unnamed_review))
display(unnamed_review.head(30))

Unnamed rows after excluding totals: 1


,File,Excel_Row,Column_B,Column_C,Column_D,Market_Value_Lakh,NAV_Pct
157,Monthly HDFC Multi-Asset Fund - 30 April 2025....,147,NaN,NaN,NaN,184.35,0.04


In [66]:
# Inspect the April 2025 report around the unnamed value-bearing row.

april_file = next(
    f for f in hdfc_files
    if f.name == "Monthly HDFC Multi-Asset Fund - 30 April 2025.xlsx"
)

april_df = pd.read_excel(
    april_file,
    sheet_name="MY2005",
    header=None
)

# Excel row 147 corresponds to Python index 146.
display(april_df.iloc[140:150, 0:12])

,0,1,2,3,4,5,6,7,8,9,10,11
140,NaN,OTHERS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
141,NaN,Mutual Fund Units,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
142,|,INF179KC1981,NaN,HDFC Gold Exchange Traded Fund,NaN,57946747,47336.7,11.16,NaN,NaN,NaN,NaN
143,NaN,Net Current Assets,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
144,NaN,NaN,NaN,Net Current Assets,NaN,NaN,489.76,0.17,NaN,NaN,NaN,NaN
145,NaN,OPTIONS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
146,NaN,NaN,NaN,NaN,NaN,150000,184.35,0.04,NaN,NaN,NaN,NaN
147,NaN,Sub Total,NaN,NaN,NaN,NaN,48010.81,11.37,NaN,NaN,NaN,NaN
148,NaN,Total,NaN,NaN,NaN,NaN,48010.81,11.37,NaN,NaN,NaN,NaN
149,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [67]:
# Review unnamed value-bearing rows and identify those under OPTIONS.

options_review = []

for file in hdfc_files:
    try:
        df = pd.read_excel(file, sheet_name="MY2005", header=None)

        labels = df.iloc[:, 1].astype(str).str.strip()
        gt_rows = df.index[
            labels.str.fullmatch("Grand Total", case=False, na=False)
        ].tolist()

        if not gt_rows:
            continue

        gt_row = gt_rows[-1]

        for idx in range(5, gt_row):
            name = df.iloc[idx, 3]
            value = pd.to_numeric(
                pd.Series([df.iloc[idx, 6]]), errors="coerce"
            ).iloc[0]

            if pd.isna(name) and pd.notna(value):
                previous_label = (
                    str(df.iloc[idx - 1, 1]).strip().upper()
                    if idx > 5 else ""
                )

                options_review.append({
                    "File": file.name,
                    "Excel_Row": idx + 1,
                    "Previous_Section_Label": previous_label,
                    "Market_Value_Lakh": value,
                    "NAV_Pct": pd.to_numeric(
                        pd.Series([df.iloc[idx, 7]]),
                        errors="coerce"
                    ).iloc[0]
                })

    except Exception as e:
        print(f"Could not inspect {file.name}: {e}")

options_review_df = pd.DataFrame(options_review)

display(options_review_df)

,File,Excel_Row,Previous_Section_Label,Market_Value_Lakh,NAV_Pct
0,Monthly HDFC Multi Asset Allocation Fund - 30 ...,81,INE201P01022,399437.96,69.09
1,Monthly HDFC Multi Asset Allocation Fund - 30 ...,86,INE0NDH25011,9444.98,1.64
2,Monthly HDFC Multi Asset Allocation Fund - 30 ...,87,SUB TOTAL,408882.94,70.73
3,Monthly HDFC Multi Asset Allocation Fund - 30 ...,122,INE261F08EF5,59286.29,10.21
4,Monthly HDFC Multi Asset Allocation Fund - 30 ...,126,INE2I7F15012,2390.10,0.41
...,...,...,...,...,...
351,Monthly HDFC Multi-Asset Fund - 31 October 202...,134,SUB TOTAL,11125.15,2.16
352,Monthly HDFC Multi-Asset Fund - 31 October 202...,142,NAN,41167.22,7.99
353,Monthly HDFC Multi-Asset Fund - 31 October 202...,143,SUB TOTAL,41167.22,7.99
354,Monthly HDFC Multi-Asset Fund - 31 October 202...,149,NAN,61310.83,11.91


In [68]:
# Inspect the June 2026 report's final portfolio rows.
june_file = next(
    f for f in hdfc_files
    if f.name == "Monthly HDFC Multi-Asset Allocation Fund - 30 June 2026.xlsx"
)

june_df = pd.read_excel(june_file, sheet_name="MY2005", header=None)

# Find the Grand Total row using the label in column B.
labels = june_df.iloc[:, 1].astype(str).str.strip()
gt_rows = june_df.index[
    labels.str.fullmatch("Grand Total", case=False, na=False)
].tolist()

print("Grand Total row indices:", gt_rows)

if gt_rows:
    gt_row = gt_rows[-1]

    # Display the final 25 rows before Grand Total.
    display(june_df.iloc[max(0, gt_row - 25):gt_row + 1, 0:8])
else:
    print("Grand Total not found in column B.")
    display(june_df.iloc[-35:, 0:8])

Grand Total row indices: [160]


,0,1,2,3,4,5,6,7
135,NaN,Total,NaN,NaN,NaN,NaN,79901.64,13.6
136,NaN,UNITS ISSUED BY INVIT,NaN,NaN,NaN,NaN,NaN,NaN
137,NaN,Units issued by InvIT,NaN,NaN,NaN,NaN,NaN,NaN
138,NaN,INE0GGX23010,NaN,POWERGRID Infrastructure Investment Trust,Power,1603841,1504.24,0.26
139,NaN,INE2PB023011,NaN,RAAJMARG INFRA INVESTMENT TRUST,Transport Infrastructure,129388,148.23,0.03
140,NaN,Sub Total,NaN,NaN,NaN,NaN,1652.47,0.29
141,NaN,Total,NaN,NaN,NaN,NaN,1652.47,0.29
142,NaN,MONEY MARKET INSTRUMENTS,NaN,NaN,NaN,NaN,NaN,NaN
143,NaN,Certificate Of Deposit (CD),NaN,NaN,NaN,NaN,NaN,NaN
144,NaN,INE160A16UV6,NaN,Punjab National Bank^,CARE - A1+,500,2387.26,0.41


In [69]:
# Test dynamic extraction on the June 2026 HDFC report.
import pandas as pd

# Find the Grand Total row dynamically.
labels = june_df.iloc[:, 1].astype(str).str.strip()
gt_rows = june_df.index[
    labels.str.fullmatch("Grand Total", case=False, na=False)
].tolist()

if not gt_rows:
    print("Grand Total not found. Do not continue with this report.")
else:
    gt_row = gt_rows[-1]

    # Include all rows from the first holdings row up to Grand Total.
    test = june_df.iloc[5:gt_row].copy()

    test = test.rename(columns={
        1: "ISIN",
        3: "Instrument_Name",
        4: "Industry_or_Rating",
        5: "Quantity",
        6: "Market_Value_Lakh",
        7: "NAV_Pct"
    })

    test["Market_Value_Lakh"] = pd.to_numeric(
        test["Market_Value_Lakh"], errors="coerce"
    )
    test["NAV_Pct"] = pd.to_numeric(
        test["NAV_Pct"], errors="coerce"
    )

    # Keep rows with a numeric market value and either an instrument name
    # or an ISIN; exclude subtotal/total rows.
    test = test[
        test["Market_Value_Lakh"].notna()
        & (
            test["Instrument_Name"].notna()
            | test["ISIN"].notna()
        )
    ].copy()

    labels_to_exclude = (
        test["Instrument_Name"].fillna("").astype(str).str.strip().str.lower()
    )
    isin_to_exclude = (
        test["ISIN"].fillna("").astype(str).str.strip().str.lower()
    )

    test = test[
        ~labels_to_exclude.isin(["subtotal", "sub total", "total", "grand total"])
        & ~isin_to_exclude.isin(["subtotal", "sub total", "total", "grand total"])
    ].copy()

    extracted_total = test["Market_Value_Lakh"].sum()
    reported_total = pd.to_numeric(
        june_df.iloc[gt_row, 6], errors="coerce"
    )

    print(f"Extracted total: ₹{extracted_total:,.2f} lakh")
    print(f"Reported Grand Total: ₹{reported_total:,.2f} lakh")
    print(f"Difference: ₹{extracted_total - reported_total:,.2f} lakh")
    print(f"Rows retained: {len(test)}")

    display(
        test[[
            "ISIN", "Instrument_Name", "Market_Value_Lakh", "NAV_Pct"
        ]].tail(12)
    )

Extracted total: ₹588,140.78 lakh
Reported Grand Total: ₹588,140.78 lakh
Difference: ₹0.00 lakh
Rows retained: 124


,ISIN,Instrument_Name,Market_Value_Lakh,NAV_Pct
128,INE261F08EN9,National Bank for Agri & Rural Dev.^,497.67,0.08
129,INE261F08EF5,National Bank for Agri & Rural Dev.,250.92,0.04
133,INE2I7F15012,Siddhivinayak Securitisation Trust (Originator...,2470.67,0.42
138,INE0GGX23010,POWERGRID Infrastructure Investment Trust,1504.24,0.26
139,INE2PB023011,RAAJMARG INFRA INVESTMENT TRUST,148.23,0.03
144,INE160A16UV6,Punjab National Bank^,2387.26,0.41
145,INE692A16KU1,Union Bank of India^,2409.65,0.41
146,INE028A16KX2,Bank of Baroda^,966.11,0.16
148,NaN,TREPS - Tri-party Repo,7027.58,1.19
153,INF179KC1981,HDFC Gold Exchange Traded Fund,64470.99,10.96


In [71]:

import pandas as pd

validation_results = []
extraction_review_rows = []

for file in hdfc_files:
    try:
        df = pd.read_excel(file, sheet_name="MY2005", header=None)

        # Find the report's Grand Total row.
        labels = df.iloc[:, 1].fillna("").astype(str).str.strip()
        gt_rows = df.index[
            labels.str.fullmatch("Grand Total", case=False, na=False)
        ].tolist()

        if not gt_rows:
            validation_results.append({
                "File": file.name,
                "Extracted_Total_Lakh": None,
                "Grand_Total_Lakh": None,
                "Difference_Lakh": None,
                "Status": "REVIEW - Grand Total not found"
            })
            continue

        gt_row = gt_rows[-1]
        reported_total = pd.to_numeric(df.iloc[gt_row, 6], errors="coerce")

        # Use all rows between the header and Grand Total.
        test = df.iloc[5:gt_row].copy()
        test["Instrument_Name"] = test.iloc[:, 3]
        test["ISIN"] = test.iloc[:, 1]
        test["Market_Value_Lakh"] = pd.to_numeric(
            test.iloc[:, 6], errors="coerce"
        )
        test["NAV_Pct"] = pd.to_numeric(
            test.iloc[:, 7], errors="coerce"
        )

        # Keep rows with a numeric market value.
        test = test[test["Market_Value_Lakh"].notna()].copy()

        # Remove subtotal and total rows using both label columns.
        name_labels = (
            test["Instrument_Name"]
            .fillna("").astype(str).str.strip().str.lower()
        )
        isin_labels = (
            test["ISIN"].fillna("").astype(str).str.strip().str.lower()
        )
        total_labels = {
            "subtotal", "sub total", "total", "grand total"
        }
        test = test[
            ~name_labels.isin(total_labels)
            & ~isin_labels.isin(total_labels)
        ].copy()

        extracted_total = test["Market_Value_Lakh"].sum()
        difference = extracted_total - reported_total

        # Record any valued row without an instrument name for review.
        unnamed = test[test["Instrument_Name"].isna()]
        for _, row in unnamed.iterrows():
            extraction_review_rows.append({
                "File": file.name,
                "Excel_Row": row.name + 1,
                "ISIN": row["ISIN"],
                "Market_Value_Lakh": row["Market_Value_Lakh"],
                "NAV_Pct": row["NAV_Pct"]
            })

        validation_results.append({
            "File": file.name,
            "Extracted_Total_Lakh": extracted_total,
            "Grand_Total_Lakh": reported_total,
            "Difference_Lakh": difference,
            "Status": (
                "MATCH"
                if pd.notna(reported_total)
                and abs(difference) < 0.02
                else "REVIEW"
            )
        })

    except Exception as e:
        validation_results.append({
            "File": file.name,
            "Extracted_Total_Lakh": None,
            "Grand_Total_Lakh": None,
            "Difference_Lakh": None,
            "Status": f"ERROR - {e}"
        })

hdfc_validation = pd.DataFrame(validation_results)
hdfc_unnamed_review = pd.DataFrame(extraction_review_rows)

print("Validation summary:")
print(hdfc_validation["Status"].value_counts(dropna=False))

display(
    hdfc_validation[hdfc_validation["Status"] != "MATCH"]
    .sort_values("File")
)
print("Rows with values but no instrument name:", len(hdfc_unnamed_review))
display(hdfc_unnamed_review.head(20))


Validation summary:
Status
MATCH    33
Name: count, dtype: int64


,File,Extracted_Total_Lakh,Grand_Total_Lakh,Difference_Lakh,Status


Rows with values but no instrument name: 1


,File,Excel_Row,ISIN,Market_Value_Lakh,NAV_Pct
0,Monthly HDFC Multi-Asset Fund - 30 April 2025....,147,NaN,184.35,0.04


In [72]:

# Inspect the unnamed holding and surrounding rows in the April 2025 report.
april_file = next(
    f for f in hdfc_files
    if f.name == "Monthly HDFC Multi-Asset Fund - 30 April 2025.xlsx"
)

april_df = pd.read_excel(
    april_file,
    sheet_name="MY2005",
    header=None
)

# Excel rows 142–150 correspond to Python indices 141–149.
display(april_df.iloc[141:150, 0:12])


,0,1,2,3,4,5,6,7,8,9,10,11
141,NaN,Mutual Fund Units,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
142,|,INF179KC1981,NaN,HDFC Gold Exchange Traded Fund,NaN,57946747,47336.7,11.16,NaN,NaN,NaN,NaN
143,NaN,Net Current Assets,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
144,NaN,NaN,NaN,Net Current Assets,NaN,NaN,489.76,0.17,NaN,NaN,NaN,NaN
145,NaN,OPTIONS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
146,NaN,NaN,NaN,NaN,NaN,150000,184.35,0.04,NaN,NaN,NaN,NaN
147,NaN,Sub Total,NaN,NaN,NaN,NaN,48010.81,11.37,NaN,NaN,NaN,NaN
148,NaN,Total,NaN,NaN,NaN,NaN,48010.81,11.37,NaN,NaN,NaN,NaN
149,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [73]:

# Display all columns for the unnamed options row.
display(april_df.iloc[146, :].to_frame(name="Value"))

# Search the report for option-related descriptions and notes.
for col in range(min(12, april_df.shape[1])):
    text_col = april_df.iloc[:, col].fillna("").astype(str)
    matches = text_col[
        text_col.str.contains(
            "option|call|put|derivative|futures",
            case=False,
            regex=True
        )
    ]
    if not matches.empty:
        print(f"\nMatches in column {col}:")
        display(april_df.loc[matches.index, [col]])


,Value
0,NaN
1,NaN
2,NaN
3,NaN
4,NaN
5,150000
6,184.35
7,0.04
8,NaN
9,NaN



Matches in column 1:


,1
145,OPTIONS
158,~ YTC i.e. Yield to Call is disclosed at secur...
159,This scheme has exposure to floating rate inst...
220,IDCW Option
221,IDCW Option - Direct Plan
222,Growth Option
223,Growth Option - Direct Plan
233,5) Total outstanding exposure in Derivative In...
238,10) Total outstanding exposure in Derivative I...



Matches in column 2:


,2
158,~ YTC i.e. Yield to Call is disclosed at secur...
159,This scheme has exposure to floating rate inst...


...

,4
158,~ YTC i.e. Yield to Call is disclosed at secur...
159,This scheme has exposure to floating rate inst...



Matches in column 5:


,5
158,~ YTC i.e. Yield to Call is disclosed at secur...
159,This scheme has exposure to floating rate inst...



Matches in column 10:


,10
4,Derivative\n% to NAV


In [74]:
# Inspect the April 2025 options section and its surrounding rows.
display(april_df.iloc[140:150, 0:12])

,0,1,2,3,4,5,6,7,8,9,10,11
140,NaN,OTHERS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
141,NaN,Mutual Fund Units,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
142,|,INF179KC1981,NaN,HDFC Gold Exchange Traded Fund,NaN,57946747,47336.7,11.16,NaN,NaN,NaN,NaN
143,NaN,Net Current Assets,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
144,NaN,NaN,NaN,Net Current Assets,NaN,NaN,489.76,0.17,NaN,NaN,NaN,NaN
145,NaN,OPTIONS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
146,NaN,NaN,NaN,NaN,NaN,150000,184.35,0.04,NaN,NaN,NaN,NaN
147,NaN,Sub Total,NaN,NaN,NaN,NaN,48010.81,11.37,NaN,NaN,NaN,NaN
148,NaN,Total,NaN,NaN,NaN,NaN,48010.81,11.37,NaN,NaN,NaN,NaN
149,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [75]:
options_review = []

for file in hdfc_files:
    df = pd.read_excel(file, sheet_name="MY2005", header=None)

    # Locate the OPTIONS section in column B.
    labels = df.iloc[:, 1].fillna("").astype(str).str.strip().str.upper()
    option_rows = df.index[labels.eq("OPTIONS")].tolist()

    for start in option_rows:
        # Inspect rows after OPTIONS until a subtotal/total or a new section.
        for i in range(start + 1, min(start + 10, len(df))):
            label = str(df.iloc[i, 1]).strip().upper()
            if label in {"SUB TOTAL", "SUBTOTAL", "TOTAL", "GRAND TOTAL"}:
                break

            value = pd.to_numeric(df.iloc[i, 6], errors="coerce")
            name = df.iloc[i, 3]

            if pd.notna(value) and pd.isna(name):
                options_review.append({
                    "File": file.name,
                    "Excel_Row": i + 1,
                    "Market_Value_Lakh": value,
                    "NAV_Pct": pd.to_numeric(df.iloc[i, 7], errors="coerce")
                })

options_review_df = pd.DataFrame(options_review)
display(options_review_df)
print("Unidentified option rows found:", len(options_review_df))

,File,Excel_Row,Market_Value_Lakh,NAV_Pct
0,Monthly HDFC Multi-Asset Fund - 30 April 2025....,147,184.35,0.04


Unidentified option rows found: 1


In [76]:

# Check the June 2026 report's actual Grand Total row

june_file = next(
    f for f in hdfc_files
    if f.name == "Monthly HDFC Multi-Asset Allocation Fund - 30 June 2026.xlsx"
)

june_df = pd.read_excel(
    june_file,
    sheet_name="MY2005",
    header=None
)

# Find the Grand Total row in column B
labels = june_df.iloc[:, 1].astype(str).str.strip()

gt_rows = june_df.index[
    labels.str.fullmatch("Grand Total", case=False, na=False)
].tolist()

print("Grand Total row index:", gt_rows)

if gt_rows:
    gt_row = gt_rows[-1]
    display(june_df.iloc[max(0, gt_row - 12):gt_row + 3, 0:8])
else:
    print("Grand Total not found in column B.")
    display(june_df.iloc[-25:, 0:8])


Grand Total row index: [160]


,0,1,2,3,4,5,6,7
148,NaN,NaN,NaN,TREPS - Tri-party Repo,NaN,NaN,7027.58,1.19
149,NaN,Sub Total,NaN,NaN,NaN,NaN,12790.6,2.17
150,NaN,Total,NaN,NaN,NaN,NaN,12790.6,2.17
151,NaN,OTHERS,NaN,NaN,NaN,NaN,NaN,NaN
152,NaN,Mutual Fund Units,NaN,NaN,NaN,NaN,NaN,NaN
153,|,INF179KC1981,NaN,HDFC Gold Exchange Traded Fund,NaN,53833490,64470.99,10.96
154,NaN,INF179KC1DI2,NaN,HDFC Silver ETF,NaN,1000000,2132.5,0.36
155,NaN,Net Current Assets,NaN,NaN,NaN,NaN,NaN,NaN
156,NaN,NaN,NaN,Net Current Assets,NaN,NaN,17956.48,3.02
157,NaN,Sub Total,NaN,NaN,NaN,NaN,84559.97,14.34


In [78]:

# Test dynamic extraction on the June 2026 report

# Find the Grand Total row
gt_row = gt_rows[-1]

# Extract rows after the header and before Grand Total
june_holdings = june_df.iloc[5:gt_row].copy()

june_holdings = june_holdings.rename(columns={
    1: "ISIN",
    3: "Instrument_Name",
    4: "Industry_or_Rating",
    5: "Quantity",
    6: "Market_Value_Lakh",
    7: "NAV_Pct"
})

# Convert market value and NAV percentage to numbers
june_holdings["Market_Value_Lakh"] = pd.to_numeric(
    june_holdings["Market_Value_Lakh"], errors="coerce"
)
june_holdings["NAV_Pct"] = pd.to_numeric(
    june_holdings["NAV_Pct"], errors="coerce"
)

# Keep rows with an instrument name and a numeric market value
june_holdings = june_holdings[
    june_holdings["Instrument_Name"].notna()
    & june_holdings["Market_Value_Lakh"].notna()
].copy()

# Remove subtotal and total rows
june_holdings = june_holdings[
    ~june_holdings["Instrument_Name"]
    .astype(str).str.strip().str.lower()
    .isin(["subtotal", "sub total", "total", "grand total"])
]

# Read the reported Grand Total from column G
reported_total = pd.to_numeric(
    june_df.iloc[gt_row, 6], errors="coerce"
)

extracted_total = june_holdings["Market_Value_Lakh"].sum()

print("Extracted holding rows:", len(june_holdings))
print("Extracted total (₹ lakh):", round(extracted_total, 2))
print("Reported Grand Total (₹ lakh):", round(reported_total, 2))
print("Difference (₹ lakh):", round(extracted_total - reported_total, 2))

display(
    june_holdings[
        ["ISIN", "Instrument_Name", "Market_Value_Lakh", "NAV_Pct"]
    ].tail(10)
)


Extracted holding rows: 124
Extracted total (₹ lakh): 588140.78
Reported Grand Total (₹ lakh): 588140.78
Difference (₹ lakh): 0.0


,ISIN,Instrument_Name,Market_Value_Lakh,NAV_Pct
133,INE2I7F15012,Siddhivinayak Securitisation Trust (Originator...,2470.67,0.42
138,INE0GGX23010,POWERGRID Infrastructure Investment Trust,1504.24,0.26
139,INE2PB023011,RAAJMARG INFRA INVESTMENT TRUST,148.23,0.03
144,INE160A16UV6,Punjab National Bank^,2387.26,0.41
145,INE692A16KU1,Union Bank of India^,2409.65,0.41
146,INE028A16KX2,Bank of Baroda^,966.11,0.16
148,NaN,TREPS - Tri-party Repo,7027.58,1.19
153,INF179KC1981,HDFC Gold Exchange Traded Fund,64470.99,10.96
154,INF179KC1DI2,HDFC Silver ETF,2132.50,0.36
156,NaN,Net Current Assets,17956.48,3.02


In [79]:

# Validate dynamic extraction against the Grand Total in every HDFC report

import pandas as pd

hdfc_dynamic_validation = []

for file in hdfc_files:
    try:
        df = pd.read_excel(file, sheet_name="MY2005", header=None)

        # Locate Grand Total in column B
        labels = df.iloc[:, 1].astype(str).str.strip()
        gt_rows = df.index[
            labels.str.fullmatch("Grand Total", case=False, na=False)
        ].tolist()

        if not gt_rows:
            hdfc_dynamic_validation.append({
                "File": file.name,
                "Extracted_Total_Lakh": None,
                "Grand_Total_Lakh": None,
                "Difference_Lakh": None,
                "Status": "GRAND TOTAL NOT FOUND"
            })
            continue

        gt_row = gt_rows[-1]

        # Extract the holdings before Grand Total
        holdings = df.iloc[5:gt_row].copy()
        holdings[6] = pd.to_numeric(holdings[6], errors="coerce")

        # Keep rows with an instrument name and numeric market value
        valid = (
            holdings[3].notna()
            & holdings[6].notna()
        )

        # Exclude subtotal and total rows
        excluded_labels = [
            "subtotal", "sub total", "total", "grand total"
        ]
        valid &= ~holdings[3].astype(str).str.strip().str.lower().isin(
            excluded_labels
        )

        extracted_total = holdings.loc[valid, 6].sum()
        reported_total = pd.to_numeric(df.iloc[gt_row, 6], errors="coerce")
        difference = extracted_total - reported_total

        hdfc_dynamic_validation.append({
            "File": file.name,
            "Extracted_Total_Lakh": round(extracted_total, 2),
            "Grand_Total_Lakh": round(reported_total, 2),
            "Difference_Lakh": round(difference, 2),
            "Status": (
                "MATCH"
                if pd.notna(difference) and abs(difference) < 0.01
                else "REVIEW"
            )
        })

    except Exception as e:
        hdfc_dynamic_validation.append({
            "File": file.name,
            "Extracted_Total_Lakh": None,
            "Grand_Total_Lakh": None,
            "Difference_Lakh": None,
            "Status": f"ERROR: {str(e)[:100]}"
        })

hdfc_dynamic_validation = pd.DataFrame(hdfc_dynamic_validation)

print("Validation summary:")
display(hdfc_dynamic_validation["Status"].value_counts())

print("Reports requiring review:")
display(
    hdfc_dynamic_validation[
        hdfc_dynamic_validation["Status"] != "MATCH"
    ].sort_values("File")
)


Validation summary:


Status
MATCH     32
REVIEW     1
Name: count, dtype: int64

Reports requiring review:


,File,Extracted_Total_Lakh,Grand_Total_Lakh,Difference_Lakh,Status
13,Monthly HDFC Multi-Asset Fund - 30 April 2025....,423873.0,424057.35,-184.35,REVIEW


In [80]:
# Inspect the April 2025 OPTIONS section and nearby rows.
display(april_df.iloc[140:150, 0:12])

,0,1,2,3,4,5,6,7,8,9,10,11
140,NaN,OTHERS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
141,NaN,Mutual Fund Units,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
142,|,INF179KC1981,NaN,HDFC Gold Exchange Traded Fund,NaN,57946747,47336.7,11.16,NaN,NaN,NaN,NaN
143,NaN,Net Current Assets,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
144,NaN,NaN,NaN,Net Current Assets,NaN,NaN,489.76,0.17,NaN,NaN,NaN,NaN
145,NaN,OPTIONS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
146,NaN,NaN,NaN,NaN,NaN,150000,184.35,0.04,NaN,NaN,NaN,NaN
147,NaN,Sub Total,NaN,NaN,NaN,NaN,48010.81,11.37,NaN,NaN,NaN,NaN
148,NaN,Total,NaN,NaN,NaN,NaN,48010.81,11.37,NaN,NaN,NaN,NaN
149,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [81]:

# Find the April 2025 report and inspect the unidentified options row.
april_file = next(
    f for f in hdfc_files
    if f.name == "Monthly HDFC Multi-Asset Fund - 30 April 2025.xlsx"
)

april_df = pd.read_excel(
    april_file,
    sheet_name="MY2005",
    header=None
)

display(april_df.iloc[144:148, :])


,0,1,2,3,4,5,6,7,8,9,10,11
144,NaN,NaN,NaN,Net Current Assets,NaN,NaN,489.76,0.17,NaN,NaN,NaN,NaN
145,NaN,OPTIONS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
146,NaN,NaN,NaN,NaN,NaN,150000,184.35,0.04,NaN,NaN,NaN,NaN
147,NaN,Sub Total,NaN,NaN,NaN,NaN,48010.81,11.37,NaN,NaN,NaN,NaN


In [82]:

# Document the unresolved April 2025 options entry.
april_options_note = {
    "AMC": "HDFC",
    "Report_Month": "2025-04",
    "Section": "OPTIONS",
    "Instrument_Name": None,
    "ISIN": None,
    "Quantity": 150000,
    "Market_Value_Lakh": 184.35,
    "NAV_Pct": 0.04,
    "Review_Status": "Unidentified options-related entry",
    "Review_Notes": (
        "The source report does not provide an instrument name or ISIN "
        "for this row. Retained as an unresolved entry; not classified "
        "as Gold, Silver, or AI/Technology."
    )
}

display(pd.DataFrame([april_options_note]))


,AMC,Report_Month,Section,Instrument_Name,ISIN,Quantity,Market_Value_Lakh,NAV_Pct,Review_Status,Review_Notes
0,HDFC,2025-04,OPTIONS,None,None,150000,184.35,0.04,Unidentified options-related entry,The source report does not provide an instrume...


In [83]:

# Check the Grand Total row in every HDFC report.
hdfc_grand_total_check = []

for file in hdfc_files:
    df = pd.read_excel(file, sheet_name="MY2005", header=None)

    labels = df.iloc[:, 1].astype(str).str.strip()
    matches = df.index[
        labels.str.fullmatch("Grand Total", case=False, na=False)
    ].tolist()

    hdfc_grand_total_check.append({
        "File": file.name,
        "Grand_Total_Found": len(matches) > 0,
        "Grand_Total_Row": matches[-1] + 1 if matches else None
    })

hdfc_grand_total_check_df = pd.DataFrame(hdfc_grand_total_check)

print("Reports checked:", len(hdfc_grand_total_check_df))
print("Grand Total found:")
display(hdfc_grand_total_check_df["Grand_Total_Found"].value_counts())

print("Reports where Grand Total was not found:")
display(
    hdfc_grand_total_check_df[
        ~hdfc_grand_total_check_df["Grand_Total_Found"]
    ]
)


Reports checked: 33
Grand Total found:


Grand_Total_Found
True    33
Name: count, dtype: int64

Reports where Grand Total was not found:


,File,Grand_Total_Found,Grand_Total_Row


In [84]:

# Extract candidate holdings from all 33 HDFC reports.
hdfc_all_holdings = []
hdfc_extraction_checks = []

for file in hdfc_files:
    df = pd.read_excel(file, sheet_name="MY2005", header=None)

    labels = df.iloc[:, 1].astype(str).str.strip()
    gt_rows = df.index[
        labels.str.fullmatch("Grand Total", case=False, na=False)
    ].tolist()

    if not gt_rows:
        hdfc_extraction_checks.append({
            "File": file.name,
            "Status": "REVIEW - Grand Total missing"
        })
        continue

    gt_row = gt_rows[-1]

    # The holdings table starts after the report header.
    section = df.iloc[5:gt_row].copy()

    section = section.rename(columns={
        1: "ISIN",
        3: "Instrument_Name",
        4: "Industry_or_Rating",
        5: "Quantity",
        6: "Market_Value_Lakh",
        7: "NAV_Pct"
    })

    section["Market_Value_Lakh"] = pd.to_numeric(
        section["Market_Value_Lakh"], errors="coerce"
    )
    section["NAV_Pct"] = pd.to_numeric(
        section["NAV_Pct"], errors="coerce"
    )

    # Keep rows with a numeric value and either an instrument name
    # or an ISIN. This preserves identifiable holdings without
    # treating blank section headings as investments.
    section = section[
        section["Market_Value_Lakh"].notna()
        & (
            section["Instrument_Name"].notna()
            | section["ISIN"].notna()
        )
    ].copy()

    # Exclude subtotal and total rows.
    total_labels = {"subtotal", "sub total", "total", "grand total"}
    section = section[
        ~section["Instrument_Name"]
        .astype(str).str.strip().str.lower().isin(total_labels)
    ].copy()

    section["AMC"] = "HDFC"
    section["Report_Month"] = file.name
    section["Source_File"] = file.name
    section["Source_Row"] = section.index + 1

    hdfc_all_holdings.append(section[[
        "AMC", "Report_Month", "ISIN", "Instrument_Name",
        "Industry_or_Rating", "Quantity", "Market_Value_Lakh",
        "NAV_Pct", "Source_File", "Source_Row"
    ]])

    grand_total = pd.to_numeric(df.iloc[gt_row, 6], errors="coerce")
    extracted_total = section["Market_Value_Lakh"].sum()

    hdfc_extraction_checks.append({
        "File": file.name,
        "Extracted_Rows": len(section),
        "Extracted_Total_Lakh": round(extracted_total, 2),
        "Grand_Total_Lakh": grand_total,
        "Difference_Lakh": round(extracted_total - grand_total, 2),
        "Status": (
            "MATCH"
            if pd.notna(grand_total)
            and abs(extracted_total - grand_total) <= 0.02
            else "REVIEW"
        )
    })

hdfc_holdings_df = pd.concat(hdfc_all_holdings, ignore_index=True)
hdfc_extraction_checks_df = pd.DataFrame(hdfc_extraction_checks)

print("Reports processed:", len(hdfc_extraction_checks_df))
print("Extracted holding rows:", len(hdfc_holdings_df))
print("\nReconciliation summary:")
display(hdfc_extraction_checks_df["Status"].value_counts())

print("\nReports requiring review:")
display(
    hdfc_extraction_checks_df[
        hdfc_extraction_checks_df["Status"] != "MATCH"
    ]
)

print("\nSample extracted holdings:")
display(hdfc_holdings_df.head(10))


Reports processed: 33
Extracted holding rows: 4048

Reconciliation summary:


Status
REVIEW    33
Name: count, dtype: int64


Reports requiring review:


,File,Extracted_Rows,Extracted_Total_Lakh,Grand_Total_Lakh,Difference_Lakh,Status
0,Monthly HDFC Multi Asset Allocation Fund - 30 ...,133,1734545.94,578181.98,1156363.96,REVIEW
1,Monthly HDFC Multi Asset Allocation Fund - 31 ...,134,1809376.74,603125.58,1206251.16,REVIEW
2,Monthly HDFC Multi-Asset Allocation Fund - 28 ...,133,1765133.85,588377.95,1176755.90,REVIEW
3,Monthly HDFC Multi-Asset Allocation Fund - 30 ...,133,1766115.90,588705.30,1177410.60,REVIEW
4,Monthly HDFC Multi-Asset Allocation Fund - 30 ...,136,1764422.34,588140.78,1176281.56,REVIEW
5,Monthly HDFC Multi-Asset Allocation Fund - 31 ...,128,1637898.48,545966.16,1091932.32,REVIEW
6,Monthly HDFC Multi-Asset Allocation Fund - 31 ...,131,1717014.57,571360.81,1145653.76,REVIEW
7,Monthly HDFC Multi-Asset Allocation Fund - 31 ...,135,1793974.92,597991.64,1195983.28,REVIEW
8,Monthly HDFC Multi-Asset Allocation Fund - 31 ...,135,1669922.43,555581.53,1114340.90,REVIEW
9,Monthly HDFC Multi-Asset Allocation Fund - 31 ...,135,1775332.91,588760.66,1186572.25,REVIEW



Sample extracted holdings:


,AMC,Report_Month,ISIN,Instrument_Name,Industry_or_Rating,Quantity,Market_Value_Lakh,NAV_Pct,Source_File,Source_Row
0,HDFC,Monthly HDFC Multi Asset Allocation Fund - 30 ...,INE040A01034,HDFC Bank Ltd.£,Banks,5599983,39687.08,6.86,Monthly HDFC Multi Asset Allocation Fund - 30 ...,9
1,HDFC,Monthly HDFC Multi Asset Allocation Fund - 30 ...,INE002A01018,Reliance Industries Ltd.,Petroleum Products,2366500,28090.36,4.86,Monthly HDFC Multi Asset Allocation Fund - 30 ...,10
2,HDFC,Monthly HDFC Multi Asset Allocation Fund - 30 ...,INE090A01021,ICICI Bank Ltd.,Banks,2073100,27400.16,4.74,Monthly HDFC Multi Asset Allocation Fund - 30 ...,11
3,HDFC,Monthly HDFC Multi Asset Allocation Fund - 30 ...,INE397D01024,Bharti Airtel Ltd.,Telecom - Services,951850,16724.00,2.89,Monthly HDFC Multi Asset Allocation Fund - 30 ...,12
4,HDFC,Monthly HDFC Multi Asset Allocation Fund - 30 ...,INE237A01036,Kotak Mahindra Bank Limited,Banks,3600000,15012.00,2.60,Monthly HDFC Multi Asset Allocation Fund - 30 ...,13
5,HDFC,Monthly HDFC Multi Asset Allocation Fund - 30 ...,INE238A01034,Axis Bank Ltd.,Banks,1200000,14712.00,2.54,Monthly HDFC Multi Asset Allocation Fund - 30 ...,14
6,HDFC,Monthly HDFC Multi Asset Allocation Fund - 30 ...,INE585B01010,Maruti Suzuki India Limited,Automobiles,93800,11225.05,1.94,Monthly HDFC Multi Asset Allocation Fund - 30 ...,15
7,HDFC,Monthly HDFC Multi Asset Allocation Fund - 30 ...,INE733E01010,NTPC Limited,Power,3351500,10816.97,1.87,Monthly HDFC Multi Asset Allocation Fund - 30 ...,16
8,HDFC,Monthly HDFC Multi Asset Allocation Fund - 30 ...,INE044A01036,Sun Pharmaceutical Industries Ltd.,Pharmaceuticals & Biotechnology,546750,9932.26,1.72,Monthly HDFC Multi Asset Allocation Fund - 30 ...,17
9,HDFC,Monthly HDFC Multi Asset Allocation Fund - 30 ...,INE361B01024,Divis Laboratories Ltd.,Pharmaceuticals & Biotechnology,100000,9291.50,1.61,Monthly HDFC Multi Asset Allocation Fund - 30 ...,18


In [86]:
# Convert the list of extracted holding tables into one DataFrame.
hdfc_all_holdings_df = pd.concat(
    hdfc_all_holdings,
    ignore_index=True
)

# Inspect the October 2025 report.
sample_file_name = "Monthly HDFC Multi-Asset Fund - 31 October 2025.xlsx"

sample_extracted = hdfc_all_holdings_df[
    hdfc_all_holdings_df["Source_File"].eq(sample_file_name)
].copy()

print("Extracted rows:", len(sample_extracted))
print(
    "Extracted market value (₹ lakh):",
    sample_extracted["Market_Value_Lakh"].sum()
)

display(
    sample_extracted[
        [
            "Source_Row",
            "ISIN",
            "Instrument_Name",
            "Market_Value_Lakh",
            "NAV_Pct"
        ]
    ].tail(25)
)

Extracted rows: 126
Extracted market value (₹ lakh): 1544724.9600000002


,Source_Row,ISIN,Instrument_Name,Market_Value_Lakh,NAV_Pct
4023,114,INE028A08315,Bank of Baroda (Tier 2 - Basel III)^,500.57,0.10
4024,115,INE261F08EN9,National Bank for Agri & Rural Dev.^,507.81,0.10
4025,116,INE134E08NN2,Power Finance Corporation Ltd.,508.68,0.10
4026,117,INE134E08NO0,Power Finance Corporation Ltd.^,511.70,0.10
4027,118,INE020B08FE4,REC Limited.^,511.54,0.10
4028,119,INE261F08EF5,National Bank for Agri & Rural Dev.^,253.53,0.05
4029,120,Sub Total,NaN,60523.91,11.76
4030,123,INE2I7F15012,Siddhivinayak Securitisation Trust PTC ISD 150...,2551.00,0.50
4031,124,Sub Total,NaN,2551.00,0.50
4032,125,Total,NaN,63074.91,12.26


In [87]:
# Remove summary rows so investments are not counted twice.
total_labels = {
    "subtotal", "sub total", "total", "grand total"
}

instrument_names = (
    section["Instrument_Name"]
    .astype(str)
    .str.strip()
    .str.lower()
)

isin_values = (
    section["ISIN"]
    .astype(str)
    .str.strip()
    .str.lower()
)

section = section[
    ~instrument_names.isin(total_labels)
    & ~isin_values.isin(total_labels)
].copy()

In [89]:

# Check what hdfc_all_holdings contains.
print("Type:", type(hdfc_all_holdings))
print("Number of items:", len(hdfc_all_holdings))

# Convert the list of holding tables into one DataFrame.
hdfc_all_holdings = pd.concat(
    hdfc_all_holdings,
    ignore_index=True
)

print("\nConverted type:", type(hdfc_all_holdings))
print("Total extracted rows:", len(hdfc_all_holdings))


Type: <class 'list'>
Number of items: 33

Converted type: <class 'pandas.core.frame.DataFrame'>
Total extracted rows: 4048


In [90]:

# Inspect the October 2025 report.
sample_file_name = "Monthly HDFC Multi-Asset Fund - 31 October 2025.xlsx"

sample_extracted = hdfc_all_holdings[
    hdfc_all_holdings["Source_File"].eq(sample_file_name)
].copy()

print("Extracted rows:", len(sample_extracted))
print(
    "Extracted market value (₹ lakh):",
    round(sample_extracted["Market_Value_Lakh"].sum(), 2)
)

print("\nDuplicate instrument rows:")
display(
    sample_extracted[
        sample_extracted.duplicated(
            subset=["ISIN", "Instrument_Name"],
            keep=False
        )
    ][["ISIN", "Instrument_Name", "Market_Value_Lakh", "Source_Row"]]
)


Extracted rows: 126
Extracted market value (₹ lakh): 1544724.96

Duplicate instrument rows:


,ISIN,Instrument_Name,Market_Value_Lakh,Source_Row
3994,Sub Total,NaN,338230.21,81
3995,Total,NaN,338230.21,82
4029,Sub Total,NaN,60523.91,120
4031,Sub Total,NaN,2551.00,124
4032,Total,NaN,63074.91,125
4037,Sub Total,NaN,11125.15,133
4038,Total,NaN,11125.15,134
4042,Sub Total,NaN,41167.22,142
4043,Total,NaN,41167.22,143
4046,Sub Total,NaN,61310.83,149


In [91]:

# Remove summary rows using both the instrument name and ISIN columns.
total_labels = {"subtotal", "sub total", "total", "grand total"}

instrument_names = (
    section["Instrument_Name"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

isin_values = (
    section["ISIN"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

section = section[
    ~instrument_names.isin(total_labels)
    & ~isin_values.isin(total_labels)
].copy()


In [93]:
print("Current type:", type(hdfc_all_holdings))
print("Current rows:", len(hdfc_all_holdings))
print("Columns:", hdfc_all_holdings.columns.tolist())

Current type: <class 'pandas.core.frame.DataFrame'>
Current rows: 4048
Columns: ['AMC', 'Report_Month', 'ISIN', 'Instrument_Name', 'Industry_or_Rating', 'Quantity', 'Market_Value_Lakh', 'NAV_Pct', 'Source_File', 'Source_Row']


In [94]:

# Check whether summary rows still exist in the extracted data.
for column in ["ISIN", "Instrument_Name"]:
    print(f"\nSummary labels found in {column}:")
    values = (
        hdfc_all_holdings[column]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.lower()
    )
    print(values[values.isin(
        ["subtotal", "sub total", "total", "grand total"]
    )].value_counts())



Summary labels found in ISIN:
ISIN
sub total    190
total        165
Name: count, dtype: int64

Summary labels found in Instrument_Name:
Series([], Name: count, dtype: int64)


In [95]:

# Remove summary rows without removing genuine investments.

summary_labels = ["sub total", "subtotal", "total", "grand total"]

isin_text = (
    hdfc_all_holdings["ISIN"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

name_text = (
    hdfc_all_holdings["Instrument_Name"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

hdfc_all_holdings_clean = hdfc_all_holdings[
    ~isin_text.isin(summary_labels)
    & ~name_text.isin(summary_labels)
].copy()

print("Rows before cleaning:", len(hdfc_all_holdings))
print("Rows after removing summary rows:", len(hdfc_all_holdings_clean))
print("Reports covered:", hdfc_all_holdings_clean["Source_File"].nunique())

# Check that no summary rows remain.
print("\nRemaining summary labels in ISIN:")
print(
    hdfc_all_holdings_clean["ISIN"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
    .value_counts()
    .reindex(summary_labels, fill_value=0)
)


Rows before cleaning: 4048
Rows after removing summary rows: 3693
Reports covered: 33

Remaining summary labels in ISIN:
ISIN
sub total      0
subtotal       0
total          0
grand total    0
Name: count, dtype: int64


In [96]:

# Compare cleaned holdings with the Grand Total in each report.

from pathlib import Path
import pandas as pd

reconciliation = []

for file in hdfc_files:
    df = pd.read_excel(file, sheet_name="MY2005", header=None)

    # Find the Grand Total row in column B.
    labels = df.iloc[:, 1].astype(str).str.strip()
    gt_rows = df.index[
        labels.str.fullmatch("Grand Total", case=False, na=False)
    ].tolist()

    if not gt_rows:
        reconciliation.append({
            "File": file.name,
            "Extracted_Total_Lakh": None,
            "Grand_Total_Lakh": None,
            "Difference_Lakh": None,
            "Status": "GRAND TOTAL NOT FOUND"
        })
        continue

    grand_total = pd.to_numeric(
        df.iloc[gt_rows[-1], 6], errors="coerce"
    )

    extracted_total = hdfc_all_holdings_clean.loc[
        hdfc_all_holdings_clean["Source_File"].eq(file.name),
        "Market_Value_Lakh"
    ].sum()

    difference = extracted_total - grand_total

    reconciliation.append({
        "File": file.name,
        "Extracted_Total_Lakh": round(extracted_total, 2),
        "Grand_Total_Lakh": round(grand_total, 2),
        "Difference_Lakh": round(difference, 2),
        "Status": (
            "MATCH"
            if pd.notna(grand_total) and abs(difference) <= 0.02
            else "REVIEW"
        )
    })

hdfc_reconciliation_clean = pd.DataFrame(reconciliation)

print(hdfc_reconciliation_clean["Status"].value_counts())
display(
    hdfc_reconciliation_clean[
        hdfc_reconciliation_clean["Status"] != "MATCH"
    ]
)


Status
MATCH     32
REVIEW     1
Name: count, dtype: int64


,File,Extracted_Total_Lakh,Grand_Total_Lakh,Difference_Lakh,Status
13,Monthly HDFC Multi-Asset Fund - 30 April 2025....,423873.0,424057.35,-184.35,REVIEW


In [97]:

# Inspect the April 2025 report around its Grand Total.
from IPython.display import display

april_file = next(
    f for f in hdfc_files
    if f.name == "Monthly HDFC Multi-Asset Fund - 30 April 2025.xlsx"
)

april_df = pd.read_excel(
    april_file,
    sheet_name="MY2005",
    header=None
)

labels = april_df.iloc[:, 1].astype(str).str.strip()
gt_rows = april_df.index[
    labels.str.fullmatch("Grand Total", case=False, na=False)
].tolist()

print("Grand Total row index:", gt_rows)

if gt_rows:
    gt_row = gt_rows[-1]
    display(april_df.iloc[max(0, gt_row - 20):gt_row + 3, 0:8])


Grand Total row index: [150]


,0,1,2,3,4,5,6,7
130,NaN,INE0NDH25011,NaN,Nexus Select Trust REIT,Realty,1300000,1685.45,0.4
131,NaN,Sub Total,NaN,NaN,NaN,NaN,9688.84,2.28
132,NaN,Total,NaN,NaN,NaN,NaN,9688.84,2.28
133,NaN,MONEY MARKET INSTRUMENTS,NaN,NaN,NaN,NaN,NaN,NaN
134,NaN,Certificate Of Deposit (CD),NaN,NaN,NaN,NaN,NaN,NaN
135,NaN,INE160A16QS0,NaN,Punjab National Bank^,CRISIL - A1+,500,2398.32,0.57
136,NaN,TREPS - Tri-party Repo,NaN,NaN,NaN,NaN,NaN,NaN
137,NaN,NaN,NaN,TREPS - Tri-party Repo,NaN,NaN,32180.41,7.59
138,NaN,Sub Total,NaN,NaN,NaN,NaN,34578.73,8.16
139,NaN,Total,NaN,NaN,NaN,NaN,34578.73,8.16


In [98]:

# Inspect the options row in the April 2025 report.
display(april_df.iloc[146].to_frame(name="Value"))


,Value
0,NaN
1,NaN
2,NaN
3,NaN
4,NaN
5,150000
6,184.35
7,0.04
8,NaN
9,NaN


In [99]:
# Inspect the original April 2025 report around the options row.
display(april_df.iloc[140:151, 0:12])

,0,1,2,3,4,5,6,7,8,9,10,11
140,NaN,OTHERS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
141,NaN,Mutual Fund Units,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
142,|,INF179KC1981,NaN,HDFC Gold Exchange Traded Fund,NaN,57946747,47336.7,11.16,NaN,NaN,NaN,NaN
143,NaN,Net Current Assets,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
144,NaN,NaN,NaN,Net Current Assets,NaN,NaN,489.76,0.17,NaN,NaN,NaN,NaN
145,NaN,OPTIONS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
146,NaN,NaN,NaN,NaN,NaN,150000,184.35,0.04,NaN,NaN,NaN,NaN
147,NaN,Sub Total,NaN,NaN,NaN,NaN,48010.81,11.37,NaN,NaN,NaN,NaN
148,NaN,Total,NaN,NaN,NaN,NaN,48010.81,11.37,NaN,NaN,NaN,NaN
149,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [100]:
# Find the Grand Total row and inspect the complete holdings section.
labels = june_df.iloc[:, 1].astype(str).str.strip()
gt_rows = june_df.index[
    labels.str.fullmatch("Grand Total", case=False, na=False)
].tolist()

if gt_rows:
    gt_row = gt_rows[-1]
    print("Grand Total row index:", gt_row)
    print("Grand Total market value (₹ lakh):", june_df.iloc[gt_row, 6])

    # Display the rows from the start of the holdings through Grand Total.
    display(june_df.iloc[5:gt_row + 1, 1:8])
else:
    print("Grand Total row not found.")

Grand Total row index: 160
Grand Total market value (₹ lakh): 588140.78


,1,2,3,4,5,6,7
5,EQUITY & EQUITY RELATED,NaN,NaN,NaN,NaN,NaN,NaN
6,(a) Listed / awaiting listing on Stock Exchanges,NaN,NaN,NaN,NaN,NaN,NaN
7,Equity,NaN,NaN,NaN,NaN,NaN,NaN
8,INE002A01018,NaN,Reliance Industries Ltd.,Petroleum Products,2366500,30620.14,5.21
9,INE090A01021,NaN,ICICI Bank Ltd.,Banks,2073100,28509.27,4.85
...,...,...,...,...,...,...,...
156,NaN,NaN,Net Current Assets,NaN,NaN,17956.48,3.02
157,Sub Total,NaN,NaN,NaN,NaN,84559.97,14.34
158,Total,NaN,NaN,NaN,NaN,84559.97,14.34
159,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [101]:
import pandas as pd

# Find the Grand Total row in column B.
labels = june_df.iloc[:, 1].astype(str).str.strip()
gt_rows = june_df.index[
    labels.str.fullmatch("Grand Total", case=False, na=False)
].tolist()

if not gt_rows:
    print("Grand Total row not found.")
else:
    gt_row = gt_rows[-1]

    # Extract from the holdings section up to, but not including, Grand Total.
    section = june_df.iloc[5:gt_row].copy()

    section = section.rename(columns={
        1: "ISIN",
        3: "Instrument_Name",
        4: "Industry_or_Rating",
        5: "Quantity",
        6: "Market_Value_Lakh",
        7: "NAV_Pct"
    })

    # Convert the financial columns to numeric values.
    section["Market_Value_Lakh"] = pd.to_numeric(
        section["Market_Value_Lakh"], errors="coerce"
    )
    section["NAV_Pct"] = pd.to_numeric(
        section["NAV_Pct"], errors="coerce"
    )

    # Remove subtotal and total rows.
    summary_labels = {"subtotal", "sub total", "total", "grand total"}

    isin_text = section["ISIN"].fillna("").astype(str).str.strip().str.lower()
    name_text = section["Instrument_Name"].fillna("").astype(str).str.strip().str.lower()

    section = section[
        ~isin_text.isin(summary_labels)
        & ~name_text.isin(summary_labels)
        & section["Market_Value_Lakh"].notna()
    ].copy()

    extracted_total = section["Market_Value_Lakh"].sum()
    grand_total = float(june_df.iloc[gt_row, 6])

    print("Grand Total row index:", gt_row)
    print("Extracted holdings rows:", len(section))
    print("Extracted value (₹ lakh):", round(extracted_total, 2))
    print("Report Grand Total (₹ lakh):", round(grand_total, 2))
    print("Difference (₹ lakh):", round(extracted_total - grand_total, 2))

    display(section[[
        "ISIN", "Instrument_Name", "Industry_or_Rating",
        "Market_Value_Lakh", "NAV_Pct"
    ]].tail(15))

Grand Total row index: 160
Extracted holdings rows: 124
Extracted value (₹ lakh): 588140.78
Report Grand Total (₹ lakh): 588140.78
Difference (₹ lakh): 0.0


,ISIN,Instrument_Name,Industry_or_Rating,Market_Value_Lakh,NAV_Pct
125,INE134E08NN2,Power Finance Corporation Ltd.^,CRISIL - AAA,502.58,0.09
126,INE134E08NO0,Power Finance Corporation Ltd.^,CRISIL - AAA,504.85,0.09
127,INE020B08FE4,REC Limited.^,CRISIL - AAA,505.21,0.09
128,INE261F08EN9,National Bank for Agri & Rural Dev.^,CRISIL - AAA,497.67,0.08
129,INE261F08EF5,National Bank for Agri & Rural Dev.,ICRA - AAA,250.92,0.04
133,INE2I7F15012,Siddhivinayak Securitisation Trust (Originator...,CRISIL - AAA(SO),2470.67,0.42
138,INE0GGX23010,POWERGRID Infrastructure Investment Trust,Power,1504.24,0.26
139,INE2PB023011,RAAJMARG INFRA INVESTMENT TRUST,Transport Infrastructure,148.23,0.03
144,INE160A16UV6,Punjab National Bank^,CARE - A1+,2387.26,0.41
145,INE692A16KU1,Union Bank of India^,ICRA - A1+,2409.65,0.41


In [102]:
# Reconcile each HDFC report using its own Grand Total row.
reconciliation = []

for file in hdfc_files:
    try:
        df = pd.read_excel(file, sheet_name="MY2005", header=None)

        labels = df.iloc[:, 1].astype(str).str.strip()
        gt_rows = df.index[
            labels.str.fullmatch("Grand Total", case=False, na=False)
        ].tolist()

        if not gt_rows:
            reconciliation.append({
                "File": file.name,
                "Extracted_Total_Lakh": None,
                "Grand_Total_Lakh": None,
                "Difference_Lakh": None,
                "Status": "GRAND TOTAL NOT FOUND"
            })
            continue

        gt_row = gt_rows[-1]
        section = df.iloc[5:gt_row].copy()

        names = section.iloc[:, 3].fillna("").astype(str).str.strip().str.lower()
        isins = section.iloc[:, 1].fillna("").astype(str).str.strip().str.lower()

        summary_labels = {"subtotal", "sub total", "total", "grand total"}

        values = pd.to_numeric(section.iloc[:, 6], errors="coerce")

        valid_rows = (
            ~names.isin(summary_labels)
            & ~isins.isin(summary_labels)
            & values.notna()
        )

        extracted_total = values[valid_rows].sum()
        grand_total = pd.to_numeric(df.iloc[gt_row, 6], errors="coerce")
        difference = extracted_total - grand_total

        reconciliation.append({
            "File": file.name,
            "Extracted_Total_Lakh": round(extracted_total, 2),
            "Grand_Total_Lakh": round(grand_total, 2),
            "Difference_Lakh": round(difference, 2),
            "Status": "MATCH" if abs(difference) <= 0.02 else "REVIEW"
        })

    except Exception as e:
        reconciliation.append({
            "File": file.name,
            "Extracted_Total_Lakh": None,
            "Grand_Total_Lakh": None,
            "Difference_Lakh": None,
            "Status": f"ERROR: {e}"
        })

hdfc_reconciliation_dynamic = pd.DataFrame(reconciliation)

print(hdfc_reconciliation_dynamic["Status"].value_counts())
display(
    hdfc_reconciliation_dynamic[
        hdfc_reconciliation_dynamic["Status"] != "MATCH"
    ]
)

Status
MATCH    33
Name: count, dtype: int64


,File,Extracted_Total_Lakh,Grand_Total_Lakh,Difference_Lakh,Status


In [103]:
# Reconcile each HDFC report using its own Grand Total row.
reconciliation = []

for file in hdfc_files:
    try:
        df = pd.read_excel(file, sheet_name="MY2005", header=None)

        labels = df.iloc[:, 1].astype(str).str.strip()
        gt_rows = df.index[
            labels.str.fullmatch("Grand Total", case=False, na=False)
        ].tolist()

        if not gt_rows:
            reconciliation.append({
                "File": file.name,
                "Extracted_Total_Lakh": None,
                "Grand_Total_Lakh": None,
                "Difference_Lakh": None,
                "Status": "GRAND TOTAL NOT FOUND"
            })
            continue

        gt_row = gt_rows[-1]
        section = df.iloc[5:gt_row].copy()

        names = section.iloc[:, 3].fillna("").astype(str).str.strip().str.lower()
        isins = section.iloc[:, 1].fillna("").astype(str).str.strip().str.lower()

        summary_labels = {"subtotal", "sub total", "total", "grand total"}

        values = pd.to_numeric(section.iloc[:, 6], errors="coerce")

        valid_rows = (
            ~names.isin(summary_labels)
            & ~isins.isin(summary_labels)
            & values.notna()
        )

        extracted_total = values[valid_rows].sum()
        grand_total = pd.to_numeric(df.iloc[gt_row, 6], errors="coerce")
        difference = extracted_total - grand_total

        reconciliation.append({
            "File": file.name,
            "Extracted_Total_Lakh": round(extracted_total, 2),
            "Grand_Total_Lakh": round(grand_total, 2),
            "Difference_Lakh": round(difference, 2),
            "Status": "MATCH" if abs(difference) <= 0.02 else "REVIEW"
        })

    except Exception as e:
        reconciliation.append({
            "File": file.name,
            "Extracted_Total_Lakh": None,
            "Grand_Total_Lakh": None,
            "Difference_Lakh": None,
            "Status": f"ERROR: {e}"
        })

hdfc_reconciliation_dynamic = pd.DataFrame(reconciliation)

print(hdfc_reconciliation_dynamic["Status"].value_counts())
display(
    hdfc_reconciliation_dynamic[
        hdfc_reconciliation_dynamic["Status"] != "MATCH"
    ]
)

Status
MATCH    33
Name: count, dtype: int64


,File,Extracted_Total_Lakh,Grand_Total_Lakh,Difference_Lakh,Status


In [104]:
from pathlib import Path
import pandas as pd

output_file = OUTPUT_DIR / "HDFC_Monthly_Holdings_Log.csv"

hdfc_all_holdings_clean.to_csv(output_file, index=False)

print("Saved:", output_file)
print("Rows:", len(hdfc_all_holdings_clean))
print("Reports:", hdfc_all_holdings_clean["Source_File"].nunique())
print("Months:", hdfc_all_holdings_clean["Report_Month"].nunique())

display(hdfc_all_holdings_clean.head())

Saved: c:\Users\rocks\OneDrive\Desktop\AMC Project Data\02_CLEAN_DATA\HDFC_Monthly_Holdings_Log.csv
Rows: 3693
Reports: 33
Months: 33


,AMC,Report_Month,ISIN,Instrument_Name,Industry_or_Rating,Quantity,Market_Value_Lakh,NAV_Pct,Source_File,Source_Row
0,HDFC,Monthly HDFC Multi Asset Allocation Fund - 30 ...,INE040A01034,HDFC Bank Ltd.£,Banks,5599983,39687.08,6.86,Monthly HDFC Multi Asset Allocation Fund - 30 ...,9
1,HDFC,Monthly HDFC Multi Asset Allocation Fund - 30 ...,INE002A01018,Reliance Industries Ltd.,Petroleum Products,2366500,28090.36,4.86,Monthly HDFC Multi Asset Allocation Fund - 30 ...,10
2,HDFC,Monthly HDFC Multi Asset Allocation Fund - 30 ...,INE090A01021,ICICI Bank Ltd.,Banks,2073100,27400.16,4.74,Monthly HDFC Multi Asset Allocation Fund - 30 ...,11
3,HDFC,Monthly HDFC Multi Asset Allocation Fund - 30 ...,INE397D01024,Bharti Airtel Ltd.,Telecom - Services,951850,16724.00,2.89,Monthly HDFC Multi Asset Allocation Fund - 30 ...,12
4,HDFC,Monthly HDFC Multi Asset Allocation Fund - 30 ...,INE237A01036,Kotak Mahindra Bank Limited,Banks,3600000,15012.00,2.60,Monthly HDFC Multi Asset Allocation Fund - 30 ...,13


In [105]:
# Check whether all 33 months are covered correctly.
print("Unique Report_Month values:",
      hdfc_all_holdings_clean["Report_Month"].nunique())

display(
    hdfc_all_holdings_clean[
        ["Report_Month", "Source_File"]
    ]
    .drop_duplicates()
    .sort_values("Source_File")
    .to_string(index=False)
)

Unique Report_Month values: 33


'                                                       Report_Month                                                         Source_File\nMonthly HDFC Multi Asset Allocation Fund - 30 September 2026_0.xlsx Monthly HDFC Multi Asset Allocation Fund - 30 September 2026_0.xlsx\n Monthly HDFC Multi Asset Allocation Fund - 31 August 2026 (1).xlsx  Monthly HDFC Multi Asset Allocation Fund - 31 August 2026 (1).xlsx\n   Monthly HDFC Multi-Asset Allocation Fund - 28 February 2026.xlsx    Monthly HDFC Multi-Asset Allocation Fund - 28 February 2026.xlsx\n      Monthly HDFC Multi-Asset Allocation Fund - 30 April 2026.xlsx       Monthly HDFC Multi-Asset Allocation Fund - 30 April 2026.xlsx\n       Monthly HDFC Multi-Asset Allocation Fund - 30 June 2026.xlsx        Monthly HDFC Multi-Asset Allocation Fund - 30 June 2026.xlsx\n   Monthly HDFC Multi-Asset Allocation Fund - 31 December 2025.xlsx    Monthly HDFC Multi-Asset Allocation Fund - 31 December 2025.xlsx\n    Monthly HDFC Multi-Asset Allocation 

In [106]:
import re
from datetime import datetime
import pandas as pd

# Extract the report date from each original filename.
def extract_report_month(filename):
    match = re.search(
        r"(\d{1,2})\s+([A-Za-z]+)\s+(\d{4})",
        str(filename)
    )

    if match:
        day, month_name, year = match.groups()
        report_date = datetime.strptime(
            f"{day} {month_name} {year}", "%d %B %Y"
        )
        return report_date.strftime("%Y-%m")

    return None

hdfc_all_holdings_clean["Report_Month"] = (
    hdfc_all_holdings_clean["Source_File"]
    .apply(extract_report_month)
)

# Check the month coverage.
actual_months = sorted(
    hdfc_all_holdings_clean["Report_Month"].dropna().unique()
)

expected_months = pd.period_range(
    start="2024-01",
    end="2026-09",
    freq="M"
).astype(str).tolist()

missing_months = sorted(set(expected_months) - set(actual_months))
unexpected_months = sorted(set(actual_months) - set(expected_months))

print("Unique months:", len(actual_months))
print("Missing months:", missing_months)
print("Unexpected months:", unexpected_months)
print("Rows with unparsed month:",
      hdfc_all_holdings_clean["Report_Month"].isna().sum())

Unique months: 33
Missing months: []
Unexpected months: []
Rows with unparsed month: 0


In [107]:
from pathlib import Path

output_file = OUTPUT_DIR / "HDFC_Monthly_Holdings_Log.csv"

# Arrange rows by month and then source row.
hdfc_all_holdings_clean = (
    hdfc_all_holdings_clean
    .sort_values(["Report_Month", "Source_Row"])
    .reset_index(drop=True)
)

# Save the cleaned holdings log.
hdfc_all_holdings_clean.to_csv(output_file, index=False)

print("Saved to:", output_file)
print("Total holdings rows:", len(hdfc_all_holdings_clean))
print("Reports covered:", hdfc_all_holdings_clean["Source_File"].nunique())
print("Months covered:", hdfc_all_holdings_clean["Report_Month"].nunique())

Saved to: c:\Users\rocks\OneDrive\Desktop\AMC Project Data\02_CLEAN_DATA\HDFC_Monthly_Holdings_Log.csv
Total holdings rows: 3693
Reports covered: 33
Months covered: 33


In [108]:
# Check for repeated instruments within each monthly report.
duplicate_holdings = hdfc_all_holdings_clean[
    hdfc_all_holdings_clean.duplicated(
        subset=["Report_Month", "ISIN", "Instrument_Name"],
        keep=False
    )
].sort_values(["Report_Month", "ISIN", "Instrument_Name"])

print("Rows involved in duplicate instrument entries:", len(duplicate_holdings))
display(duplicate_holdings.head(20))

Rows involved in duplicate instrument entries: 0


,AMC,Report_Month,ISIN,Instrument_Name,Industry_or_Rating,Quantity,Market_Value_Lakh,NAV_Pct,Source_File,Source_Row


In [109]:
# Find gold and silver related entries in the full HDFC holdings log.
metal_keywords = r"gold|silver"

hdfc_metal_check = hdfc_all_holdings_clean[
    hdfc_all_holdings_clean["Instrument_Name"]
    .astype(str)
    .str.contains(metal_keywords, case=False, na=False)
].copy()

display(
    hdfc_metal_check[
        [
            "Report_Month",
            "ISIN",
            "Instrument_Name",
            "Market_Value_Lakh",
            "NAV_Pct",
            "Source_File"
        ]
    ].sort_values(["Report_Month", "Instrument_Name"])
)

print("Metal-related rows:", len(hdfc_metal_check))
print("Unique report months:", hdfc_metal_check["Report_Month"].nunique())

,Report_Month,ISIN,Instrument_Name,Market_Value_Lakh,NAV_Pct,Source_File
86,2024-01,INF179KC1981,HDFC Gold Exchange Traded Fund,27197.83,11.30,Monthly HDFC Multi-Asset Fund - 31 January 202...
175,2024-02,INF179KC1981,HDFC Gold Exchange Traded Fund,26949.13,10.66,Monthly HDFC Multi-Asset Fund - 29 February 20...
266,2024-03,INF179KC1981,HDFC Gold Exchange Traded Fund,29003.39,10.98,Monthly HDFC Multi-Asset Fund - 31 March 2024....
362,2024-04,INF179KC1981,HDFC Gold Exchange Traded Fund,31266.56,11.17,Monthly HDFC Multi-Asset Fund - 30 April 2024....
458,2024-05,INF179KC1981,HDFC Gold Exchange Traded Fund,31425.73,10.85,Monthly HDFC Multi-Asset Fund - 31 May 2024.xlsx
560,2024-06,INF179KC1981,HDFC Gold Exchange Traded Fund,32796.06,10.44,Monthly HDFC Multi-Asset Fund - 30 June 2024.xlsx
662,2024-07,INF179KC1981,HDFC Gold Exchange Traded Fund,36686.47,10.96,Monthly HDFC Multi-Asset Fund - 31 July 2024.xlsx
768,2024-08,INF179KC1981,HDFC Gold Exchange Traded Fund,37886.24,10.75,Monthly HDFC Multi-Asset Fund - 31 August 2024...
873,2024-09,INF179KC1981,HDFC Gold Exchange Traded Fund,42154.96,11.39,Monthly HDFC Multi-Asset Fund - 30 September 2...
982,2024-10,INF179KC1981,HDFC Gold Exchange Traded Fund,44295.82,11.78,Monthly HDFC Multi-Asset Fund - 31 October 202...


Metal-related rows: 41
Unique report months: 33


In [110]:
# Compare metal-related holdings in the full log with the separate metal log.

full_metals = hdfc_all_holdings_clean[
    hdfc_all_holdings_clean["Instrument_Name"]
    .astype(str)
    .str.contains(r"gold|silver", case=False, na=False)
].copy()

full_metals["Report_Month"] = full_metals["Report_Month"].astype(str)

metal_log = pd.read_csv(OUTPUT_DIR / "HDFC_Monthly_Metal_Holdings_Log.csv")
metal_log["Report_Month"] = metal_log["Report_Month"].astype(str)

# Standardise names for a reliable comparison.
full_metals["Instrument_Name"] = (
    full_metals["Instrument_Name"].astype(str).str.strip()
)
metal_log["Instrument_Name"] = (
    metal_log["Instrument_Name"].astype(str).str.strip()
)

comparison = full_metals[
    ["Report_Month", "ISIN", "Instrument_Name", "Market_Value_Lakh", "NAV_Pct"]
].merge(
    metal_log[
        ["Report_Month", "Instrument_Name", "Market_Value_Lakh", "NAV_Pct"]
    ],
    on=["Report_Month", "Instrument_Name"],
    how="outer",
    suffixes=("_Full_Log", "_Metal_Log"),
    indicator=True
)

print("Comparison rows:", len(comparison))
print("\nMatch status:")
print(comparison["_merge"].value_counts())

display(
    comparison[
        (comparison["_merge"] != "both")
        | (
            (comparison["Market_Value_Lakh_Full_Log"]
             - comparison["Market_Value_Lakh_Metal_Log"]).abs() > 0.01
        )
    ]
)

Comparison rows: 41

Match status:
_merge
both          41
left_only      0
right_only     0
Name: count, dtype: int64


,Report_Month,ISIN,Instrument_Name,Market_Value_Lakh_Full_Log,NAV_Pct_Full_Log,Market_Value_Lakh_Metal_Log,NAV_Pct_Metal_Log,_merge


In [111]:
# Check that every expected month is present.
expected_months = pd.period_range(
    start="2024-01",
    end="2026-09",
    freq="M"
).astype(str)

actual_months = sorted(
    hdfc_all_holdings_clean["Report_Month"].dropna().unique()
)

missing_months = sorted(set(expected_months) - set(actual_months))
unexpected_months = sorted(set(actual_months) - set(expected_months))

# Check duplicate instruments within each month.
duplicates = hdfc_all_holdings_clean[
    hdfc_all_holdings_clean.duplicated(
        subset=["Report_Month", "ISIN", "Instrument_Name"],
        keep=False
    )
].sort_values(["Report_Month", "Instrument_Name"])

print("Unique months:", len(actual_months))
print("Missing months:", missing_months)
print("Unexpected months:", unexpected_months)
print("Unparsed months:", hdfc_all_holdings_clean["Report_Month"].isna().sum())
print("Duplicate instrument rows:", len(duplicates))

if not duplicates.empty:
    display(duplicates.head(20))

Unique months: 33
Missing months: []
Unexpected months: []
Unparsed months: 0
Duplicate instrument rows: 0


In [112]:
# List ICICI Prudential source files.
icici_folder = PROJECT_DIR / "01_RAW_DATA" / "ICICI_Prudential"

icici_files = sorted(
    f for f in icici_folder.iterdir()
    if f.is_file()
)

print("ICICI Prudential source files:", len(icici_files))

for f in icici_files[:10]:
    print(f.name)

if len(icici_files) > 10:
    print("...and", len(icici_files) - 10, "more files")

ICICI Prudential source files: 33
ICICI Prudential Multi Asset Allocation Fund-aug-2026.xlsx
ICICI Prudential Multi Asset Allocation Fund-sep-2026.xlsx
ICICI Prudential Multi-Asset Fund-apr-2024.xlsx
ICICI Prudential Multi-Asset Fund-apr-2025.xlsx
ICICI Prudential Multi-Asset Fund-apr-2026.xlsx
ICICI Prudential Multi-Asset Fund-aug-2024.xlsx
ICICI Prudential Multi-Asset Fund-aug-2025.xlsx
ICICI Prudential Multi-Asset Fund-dec-2024.xlsx
ICICI Prudential Multi-Asset Fund-dec-2025.xlsx
ICICI Prudential Multi-Asset Fund-feb-2024.xlsx
...and 23 more files


In [113]:
import re
import pandas as pd

# Match the month and year at the end of each filename.
month_map = {
    "jan": 1, "feb": 2, "mar": 3, "apr": 4,
    "may": 5, "jun": 6, "jul": 7, "aug": 8,
    "sep": 9, "oct": 10, "nov": 11, "dec": 12
}

file_months = []

for f in icici_files:
    match = re.search(
        r"[- ]([a-z]{3})[- ](20\d{2})\.xlsx$",
        f.name,
        flags=re.IGNORECASE
    )

    if match:
        month_text = match.group(1).lower()
        year = int(match.group(2))

        if month_text in month_map:
            file_months.append({
                "File": f.name,
                "Report_Month": f"{year}-{month_map[month_text]:02d}"
            })
    else:
        print("Filename needs review:", f.name)

icici_coverage = pd.DataFrame(file_months)

expected_months = pd.period_range(
    start="2024-01",
    end="2026-09",
    freq="M"
).astype(str)

actual_months = icici_coverage["Report_Month"].tolist()

missing_months = sorted(set(expected_months) - set(actual_months))
unexpected_months = sorted(set(actual_months) - set(expected_months))

duplicate_months = icici_coverage[
    icici_coverage.duplicated("Report_Month", keep=False)
].sort_values("Report_Month")

print("Files parsed:", len(icici_coverage))
print("Unique months:", icici_coverage["Report_Month"].nunique())
print("Missing months:", missing_months)
print("Unexpected months:", unexpected_months)
print("Duplicate-month files:", len(duplicate_months))

if not duplicate_months.empty:
    display(duplicate_months)

display(icici_coverage.sort_values("Report_Month"))

Filename needs review: ICICI Prudential Multi-Asset Fund-july-2024.xlsx
Filename needs review: ICICI Prudential Multi-Asset Fund-june-2024.xlsx
Filename needs review: ICICI Prudential Multi-Asset Fund-june-2025.xlsx
Filename needs review: ICICI Prudential Multi-Asset Fund-June-2026.xlsx
Files parsed: 29
Unique months: 29
Missing months: ['2024-06', '2024-07', '2025-06', '2026-06']
Unexpected months: []
Duplicate-month files: 0


,File,Report_Month
12,ICICI Prudential Multi-Asset Fund-jan-2024.xlsx,2024-01
9,ICICI Prudential Multi-Asset Fund-feb-2024.xlsx,2024-02
17,ICICI Prudential Multi-Asset Fund-mar-2024.xlsx,2024-03
2,ICICI Prudential Multi-Asset Fund-apr-2024.xlsx,2024-04
20,ICICI Prudential Multi-Asset Fund-may-2024.xlsx,2024-05
5,ICICI Prudential Multi-Asset Fund-aug-2024.xlsx,2024-08
27,ICICI Prudential Multi-Asset Fund-sep-2024.xlsx,2024-09
25,ICICI Prudential Multi-Asset Fund-oct-2024.xlsx,2024-10
23,ICICI Prudential Multi-Asset Fund-nov-2024.xlsx,2024-11
7,ICICI Prudential Multi-Asset Fund-dec-2024.xlsx,2024-12


In [114]:
import re
import pandas as pd

month_map = {
    "jan": 1, "january": 1,
    "feb": 2, "february": 2,
    "mar": 3, "march": 3,
    "apr": 4, "april": 4,
    "may": 5,
    "jun": 6, "june": 6,
    "jul": 7, "july": 7,
    "aug": 8, "august": 8,
    "sep": 9, "september": 9,
    "oct": 10, "october": 10,
    "nov": 11, "november": 11,
    "dec": 12, "december": 12
}

file_months = []

for f in icici_files:
    match = re.search(
        r"[- ]([a-z]+)[- ](20\d{2})\.xlsx$",
        f.name,
        flags=re.IGNORECASE
    )

    if match:
        month_text = match.group(1).lower()
        year = int(match.group(2))

        if month_text in month_map:
            file_months.append({
                "File": f.name,
                "Report_Month": f"{year}-{month_map[month_text]:02d}"
            })
        else:
            print("Unrecognized month:", f.name)
    else:
        print("Filename needs review:", f.name)

icici_coverage = pd.DataFrame(file_months)

expected_months = pd.period_range(
    start="2024-01",
    end="2026-09",
    freq="M"
).astype(str)

actual_months = icici_coverage["Report_Month"].tolist()

missing_months = sorted(set(expected_months) - set(actual_months))
unexpected_months = sorted(set(actual_months) - set(expected_months))

duplicate_months = icici_coverage[
    icici_coverage.duplicated("Report_Month", keep=False)
].sort_values("Report_Month")

print("Files parsed:", len(icici_coverage))
print("Unique months:", icici_coverage["Report_Month"].nunique())
print("Missing months:", missing_months)
print("Unexpected months:", unexpected_months)
print("Duplicate-month files:", len(duplicate_months))

if not duplicate_months.empty:
    display(duplicate_months)

display(icici_coverage.sort_values("Report_Month"))

Files parsed: 33
Unique months: 33
Missing months: []
Unexpected months: []
Duplicate-month files: 0


,File,Report_Month
12,ICICI Prudential Multi-Asset Fund-jan-2024.xlsx,2024-01
9,ICICI Prudential Multi-Asset Fund-feb-2024.xlsx,2024-02
21,ICICI Prudential Multi-Asset Fund-mar-2024.xlsx,2024-03
2,ICICI Prudential Multi-Asset Fund-apr-2024.xlsx,2024-04
24,ICICI Prudential Multi-Asset Fund-may-2024.xlsx,2024-05
18,ICICI Prudential Multi-Asset Fund-june-2024.xlsx,2024-06
17,ICICI Prudential Multi-Asset Fund-july-2024.xlsx,2024-07
5,ICICI Prudential Multi-Asset Fund-aug-2024.xlsx,2024-08
31,ICICI Prudential Multi-Asset Fund-sep-2024.xlsx,2024-09
29,ICICI Prudential Multi-Asset Fund-oct-2024.xlsx,2024-10


In [115]:
# Select the September 2026 ICICI report.
icici_sample_file = next(
    f for f in icici_files
    if "sep-2026" in f.name.lower()
)

print("Sample report:", icici_sample_file.name)

# List the worksheet names in the report.
xls = pd.ExcelFile(icici_sample_file)

print("\nWorksheets:")
for sheet in xls.sheet_names:
    print("-", sheet)

Sample report: ICICI Prudential Multi Asset Allocation Fund-sep-2026.xlsx

Worksheets:
- MULTI
- Derivative


In [116]:
import pandas as pd

icici_sample_file = next(
    f for f in icici_files
    if f.name == "ICICI Prudential Multi Asset Allocation Fund-sep-2026.xlsx"
)

icici_sample_df = pd.read_excel(
    icici_sample_file,
    sheet_name="MULTI",
    header=None
)

print("Sheet shape:", icici_sample_df.shape)

display(icici_sample_df.head(30))

Sheet shape: (376, 10)


,0,1,2,3,4,5,6,7,8,9
0,NaN,ICICI Prudential Mutual Fund,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,ICICI Prudential Multi Asset Allocation Fund,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,"Portfolio as on Sep 30,2026",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,Company/Issuer/Instrument Name,ISIN,Coupon,Industry/Rating,Quantity,Exposure/Market Value(Rs.Lakh),% to Nav,Yield of the instrument,Yield to Call @
4,NaN,Equity & Equity Related Instruments (Note -1),NaN,NaN,NaN,NaN,5762358.53,0.685111,NaN,NaN
5,NaN,Units of Real Estate Investment Trust (REITs),NaN,NaN,NaN,NaN,82027.3,0.009753,NaN,NaN
6,NaN,EMBASSY OFFICE PARKS REIT,INE041025011,NaN,Units of Real Estate Investment Trust (REITs),10213194,44368.16,0.005275,NaN,NaN
7,NaN,Brookfield India Real Estate Trust REIT,INE0FDU25010,NaN,Units of Real Estate Investment Trust (REITs),6556751,22032.65,0.00262,NaN,NaN
8,NaN,Nexus Select Trust,INE0NDH25011,NaN,Units of Real Estate Investment Trust (REITs),4779301,7995.77,0.000951,NaN,NaN
9,NaN,MINDSPACE BUSINESS PARKS REIT,INE0CCU25019,NaN,Units of Real Estate Investment Trust (REITs),1524132,7630.72,0.000907,NaN,NaN


In [117]:
# Find rows labelled "Grand Total" in the instrument-name column.
labels = icici_sample_df.iloc[:, 1].astype(str).str.strip()

grand_total_rows = icici_sample_df.index[
    labels.str.fullmatch("Grand Total", case=False, na=False)
].tolist()

print("Grand Total row indices:", grand_total_rows)

if grand_total_rows:
    gt_row = grand_total_rows[-1]
    display(
        icici_sample_df.iloc[
            max(0, gt_row - 10):gt_row + 4,
            1:9
        ]
    )
else:
    print("Grand Total not found. Showing the final 25 rows:")
    display(icici_sample_df.iloc[-25:, 1:9])

Grand Total row indices: []
Grand Total not found. Showing the final 25 rows:


,1,2,3,4,5,6,7,8
351,For the Instrument/security whose final ISIN i...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
352,Debt instruments having structured obligations...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
353,@ As per AMFI Best Practices Guidelines Circul...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
354,For Instances of Deviation In valuation Of Sec...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
355,As per AMFI Best Practices Guidelines Circular...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
356,This scheme has exposure to floating rate inst...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
357,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
358,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
359,Scheme Riskometer,NaN,NaN,NaN,NaN,NaN,NaN,NaN
360,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [118]:
# Search the entire sheet for portfolio-total labels
for col in icici_sample_df.columns:
    text_col = icici_sample_df[col].astype(str).str.strip()

    matches = text_col[
        text_col.str.contains(
            r"grand\s*total|net\s*assets|total\s*value|total\s*investments",
            case=False,
            na=False,
            regex=True
        )
    ]

    if not matches.empty:
        print(f"\nColumn {col}:")
        for idx, value in matches.items():
            print(f"Row index {idx}: {value}")

        display(
            icici_sample_df.iloc[
                max(0, matches.index[0] - 2):
                min(len(icici_sample_df), matches.index[-1] + 3)
            ]
        )


Column 1:
Row index 292: Total Net Assets
Row index 329: 2. Total Value and Percentage of Non Traded/Unlisted/Unquoted/Illiquid shares


,0,1,2,3,4,5,6,7,8,9
290,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
291,NaN,Net Current Assets,NaN,NaN,NaN,NaN,-52898.006626,-0.006289,NaN,NaN
292,NaN,Total Net Assets,NaN,NaN,NaN,NaN,8410836.187578,1.0,NaN,NaN
293,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
294,NaN,INTEREST RATE SWAPS (At Notional Value),NaN,NaN,NaN,NaN,105000,0.012484,NaN,NaN
295,NaN,Interest Rate Swaps- MD -27-Feb-2029 (Pay floa...,NaN,NaN,NaN,NaN,2500,0.000297,NaN,NaN
296,NaN,Interest Rate Swaps- MD -27-Feb-2029 (Pay floa...,NaN,NaN,NaN,NaN,10000,0.001189,NaN,NaN
297,NaN,Interest Rate Swaps- MD -21-Feb-2029 (Pay floa...,NaN,NaN,NaN,NaN,10000,0.001189,NaN,NaN
298,NaN,Interest Rate Swaps- MD -26-Feb-2029 (Pay floa...,NaN,NaN,NaN,NaN,10000,0.001189,NaN,NaN
299,NaN,Interest Rate Swaps- MD -26-Feb-2029 (Pay floa...,NaN,NaN,NaN,NaN,10000,0.001189,NaN,NaN


In [119]:
# Inspect the main holdings table boundaries
display(icici_sample_df.iloc[0:20, 1:9])
display(icici_sample_df.iloc[275:294, 1:9])

,1,2,3,4,5,6,7,8
0,ICICI Prudential Mutual Fund,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ICICI Prudential Multi Asset Allocation Fund,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"Portfolio as on Sep 30,2026",NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Company/Issuer/Instrument Name,ISIN,Coupon,Industry/Rating,Quantity,Exposure/Market Value(Rs.Lakh),% to Nav,Yield of the instrument
4,Equity & Equity Related Instruments (Note -1),NaN,NaN,NaN,NaN,5762358.53,0.685111,NaN
5,Units of Real Estate Investment Trust (REITs),NaN,NaN,NaN,NaN,82027.3,0.009753,NaN
6,EMBASSY OFFICE PARKS REIT,INE041025011,NaN,Units of Real Estate Investment Trust (REITs),10213194,44368.16,0.005275,NaN
7,Brookfield India Real Estate Trust REIT,INE0FDU25010,NaN,Units of Real Estate Investment Trust (REITs),6556751,22032.65,0.00262,NaN
8,Nexus Select Trust,INE0NDH25011,NaN,Units of Real Estate Investment Trust (REITs),4779301,7995.77,0.000951,NaN
9,MINDSPACE BUSINESS PARKS REIT,INE0CCU25019,NaN,Units of Real Estate Investment Trust (REITs),1524132,7630.72,0.000907,NaN


,1,2,3,4,5,6,7,8
275,Reverse Repo,NaN,NaN,NaN,NaN,49999.71,0.005945,NaN
276,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
277,Units of Infrastructure Investment Trusts (Inv...,NaN,NaN,NaN,NaN,27500.24,0.00327,NaN
278,IRB Infrastructure Developers Ltd.,INE183W23014,NaN,Units of Infrastructure Investment Trusts,15079364,9683.97,0.001151,NaN
279,Cube Highways Trust,INE0NR623014,NaN,Units of Infrastructure Investment Trusts,4200000,6514.2,0.000775,NaN
280,Vertis Infrastructure Trust,INE0KXY23015,NaN,Units of Infrastructure Investment Trusts,5100000,6069,0.000722,NaN
281,Capital Infra Trust InvIT,INE0Z8Z23013,NaN,Units of Infrastructure Investment Trusts,4762415,3667.54,0.000436,NaN
282,Raajmarg Infra Investment Trust,INE2PB023011,NaN,Units of Infrastructure Investment Trusts,1293932,1565.53,0.000186,NaN
283,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
284,Units of an Alternative Investment Fund (AIF),NaN,NaN,NaN,NaN,Nil,Nil,NaN


In [120]:
# List likely section headings in the main ICICI holdings table
for idx in range(4, 293):
    name = str(icici_sample_df.iloc[idx, 1]).strip()

    if name != "nan":
        isin = icici_sample_df.iloc[idx, 2]
        value = icici_sample_df.iloc[idx, 6]

        # Headings generally have no ISIN and may contain no numeric value
        if pd.isna(isin) and pd.isna(value):
            print(idx, name)

In [121]:
# Find potential gold, silver, and ETF holdings across the report
for idx in range(len(icici_sample_df)):
    name = str(icici_sample_df.iloc[idx, 1])
    industry = str(icici_sample_df.iloc[idx, 4])

    if any(word in (name + " " + industry).lower()
           for word in ["gold", "silver", "etf"]):
        print(
            idx,
            name,
            "| Industry:", industry,
            "| Value (₹ lakh):", icici_sample_df.iloc[idx, 6],
            "| % to NAV:", icici_sample_df.iloc[idx, 7]
        )

135 Foreign Securities/Overseas ETFs | Industry: nan | Value (₹ lakh): 13893.4 | % to NAV: 0.0016518452731
273 ICICI Prudential Gold ETF | Industry: Mutual Fund | Value (₹ lakh): 846910.49 | % to NAV: 0.1006927814443
319 Gold (1 KG-1000 GMS) Commodity December 2026 Future $$ | Industry: Gold | Value (₹ lakh): 163939.6 | % to NAV: 0.0194914746101
342 5. Total market value of Investment in Foreign Securities / ADRs / GDRs / Overseas ETFs as on Sep 30,2026 | Industry: nan | Value (₹ lakh): nan | % to NAV: nan
375 Benchmark name - Nifty 200 TRI (65%) + Nifty Composite Debt Index (25%) + Domestic Price of Gold (6%) + Domestic Price of Silver (1%) + iCOMDEX Composite Index (3%) | Industry: nan | Value (₹ lakh): nan | % to NAV: nan


In [122]:
for file in sorted(icici_files)[:5]:
    xl = pd.ExcelFile(file)
    print(f"\nFile: {file.name}")

    if "MULTI" in xl.sheet_names:
        df = pd.read_excel(file, sheet_name="MULTI", header=None)
        print("MULTI sheet shape:", df.shape)
        print("Total Net Assets row:",
              df.iloc[:, 1].astype(str).str.strip()
                .eq("Total Net Assets").to_numpy().nonzero()[0].tolist())


File: ICICI Prudential Multi Asset Allocation Fund-aug-2026.xlsx
MULTI sheet shape: (400, 10)
Total Net Assets row: [316]

File: ICICI Prudential Multi Asset Allocation Fund-sep-2026.xlsx
MULTI sheet shape: (376, 10)
Total Net Assets row: [292]

File: ICICI Prudential Multi-Asset Fund-apr-2024.xlsx
MULTI sheet shape: (294, 10)
Total Net Assets row: [219]

File: ICICI Prudential Multi-Asset Fund-apr-2025.xlsx
MULTI sheet shape: (368, 10)
Total Net Assets row: [267]

File: ICICI Prudential Multi-Asset Fund-apr-2026.xlsx
MULTI sheet shape: (425, 10)
Total Net Assets row: [323]


In [123]:
icici_layout_check = []

for file in sorted(icici_files):
    xl = pd.ExcelFile(file)

    if "MULTI" not in xl.sheet_names:
        icici_layout_check.append({
            "File": file.name,
            "Sheet_Rows": None,
            "Sheet_Columns": None,
            "Total_Net_Assets_Row": None,
            "Status": "MULTI sheet missing"
        })
        continue

    df = pd.read_excel(file, sheet_name="MULTI", header=None)

    labels = df.iloc[:, 1].astype(str).str.strip()
    total_rows = df.index[labels.eq("Total Net Assets")].tolist()

    icici_layout_check.append({
        "File": file.name,
        "Sheet_Rows": len(df),
        "Sheet_Columns": len(df.columns),
        "Total_Net_Assets_Row": total_rows[0] if total_rows else None,
        "Status": "OK" if total_rows and len(df.columns) >= 8 else "REVIEW"
    })

icici_layout_check = pd.DataFrame(icici_layout_check)

print("Reports checked:", len(icici_layout_check))
print("\nStatus counts:")
display(icici_layout_check["Status"].value_counts().to_frame("Count"))

print("\nReports requiring review:")
display(icici_layout_check[icici_layout_check["Status"] != "OK"])

Reports checked: 33

Status counts:


,Count
Status,
OK,33



Reports requiring review:


,File,Sheet_Rows,Sheet_Columns,Total_Net_Assets_Row,Status


In [124]:
# Inspect the Derivative sheet in the September 2026 report
derivative_df = pd.read_excel(
    icici_sample_file,
    sheet_name="Derivative",
    header=None
)

print("Derivative sheet shape:", derivative_df.shape)
display(derivative_df.head(20))

Derivative sheet shape: (136, 8)


,0,1,2,3,4,5,6,7
0,NaN,DERIVATIVE DISCLOSURE,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1,Hedging Positions through Futures as on Sep 3...,A. Hedging Positions through Futures as on M...,A. Hedging Positions through Futures as on M...,A. Hedging Positions through Futures as on M...,A. Hedging Positions through Futures as on M...,NaN,NaN
3,NaN,Scheme,Underlying Security Name,Long/(Short),Future price when purchased (in Rs.),Current price of the contract (in Rs.),Margin Maintained (Rs. in Lakhs),NaN
4,NaN,DYNMIC,Alkem Laboratories Ltd,(Short),5284.014865,5171.5,2220.341681,NaN
5,NaN,DYNMIC,IndusInd Bank Ltd.,(Short),913.943423,901.8,14209.040636,NaN
6,NaN,DYNMIC,LIFE INSURANCE CORPORATION OF INDIA,(Short),410.3262,399.6,130.53432,NaN
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2,Other than Hedging Positions through Futures a...,B. Other than Hedging Positions through Future...,B. Other than Hedging Positions through Future...,B. Other than Hedging Positions through Future...,B. Other than Hedging Positions through Future...,NaN,NaN
9,NaN,Scheme,Underlying Security Name,Long/(Short),Future price when purchased (in Rs.),Current price of the contract (in Rs.),Margin Maintained (Rs. in Lakhs),NaN


In [125]:
# Search the derivative sheet for metal-related positions
for idx in range(len(derivative_df)):
    row_text = " ".join(
        derivative_df.iloc[idx].fillna("").astype(str).tolist()
    )

    if any(word in row_text.lower() for word in ["gold", "silver", "commodity"]):
        print(f"Row {idx}: {row_text}")

Row 12:  DYNMIC GOLD* Long 15418595.908 14903600 15241.545 
Row 14:  *Exchange Traded Commodity Derivatives      
Row 44:  **Includes Exchange Traded Commodity Derivatives      


In [126]:
# Prepare main holdings rows from the September 2026 report
icici_main = icici_sample_df.copy()

icici_main["Instrument_Name"] = icici_main.iloc[:, 1].astype("string").str.strip()
icici_main["ISIN"] = icici_main.iloc[:, 2]
icici_main["Industry_or_Rating"] = icici_main.iloc[:, 4]
icici_main["Market_Value_Lakh"] = pd.to_numeric(
    icici_main.iloc[:, 6], errors="coerce"
)
icici_main["NAV_Pct_Raw"] = pd.to_numeric(
    icici_main.iloc[:, 7], errors="coerce"
)
icici_main["Source_Row"] = icici_main.index

# Keep rows with names and numeric values, excluding the portfolio total
icici_main = icici_main[
    icici_main["Instrument_Name"].notna()
    & icici_main["Market_Value_Lakh"].notna()
    & icici_main["Instrument_Name"].ne("Total Net Assets")
].copy()

# Review likely metal ETF or metal-related rows
metal_mask = (
    icici_main["Instrument_Name"].str.contains(
        r"gold|silver|etf", case=False, na=False
    )
    | icici_main["Industry_or_Rating"].astype("string").str.contains(
        r"gold|silver", case=False, na=False
    )
)

display(
    icici_main.loc[
        metal_mask,
        [
            "Source_Row",
            "Instrument_Name",
            "ISIN",
            "Industry_or_Rating",
            "Market_Value_Lakh",
            "NAV_Pct_Raw",
        ],
    ]
)

,Source_Row,Instrument_Name,ISIN,Industry_or_Rating,Market_Value_Lakh,NAV_Pct_Raw
135,135,Foreign Securities/Overseas ETFs,NaN,NaN,13893.40,0.001652
273,273,ICICI Prudential Gold ETF,INF109KC1NT3,Mutual Fund,846910.49,0.100693
319,319,Gold (1 KG-1000 GMS) Commodity December 2026 F...,NaN,Gold,163939.60,0.019491


In [127]:
# Compare extracted main-table values with reported Total Net Assets
reported_total = pd.to_numeric(
    icici_sample_df.loc[
        icici_sample_df.iloc[:, 1].astype(str).str.strip()
        .eq("Total Net Assets"),
        6
    ],
    errors="coerce"
).iloc[0]

extracted_total = icici_main["Market_Value_Lakh"].sum()

print(f"Reported Total Net Assets: ₹{reported_total:,.2f} lakh")
print(f"Extracted row-value sum:   ₹{extracted_total:,.2f} lakh")
print(f"Difference:                ₹{extracted_total - reported_total:,.2f} lakh")
print(f"Rows extracted:             {len(icici_main)}")

Reported Total Net Assets: ₹8,410,836.19 lakh
Extracted row-value sum:   ₹24,534,653.64 lakh
Difference:                ₹16,123,817.45 lakh
Rows extracted:             274


In [128]:
# Inspect extracted rows without an ISIN
no_isin = icici_main[icici_main["ISIN"].isna()]

display(
    no_isin[
        [
            "Source_Row",
            "Instrument_Name",
            "Industry_or_Rating",
            "Market_Value_Lakh",
            "NAV_Pct_Raw",
        ]
    ].sort_values("Source_Row")
)

,Source_Row,Instrument_Name,Industry_or_Rating,Market_Value_Lakh,NAV_Pct_Raw
4,4,Equity & Equity Related Instruments (Note -1),NaN,5.762359e+06,0.685111
5,5,Units of Real Estate Investment Trust (REITs),NaN,8.202730e+04,0.009753
11,11,Listed / Awaiting Listing On Stock Exchanges,NaN,5.645610e+06,0.671165
132,132,Coal India Ltd. (Covered call) $$,Consumable Fuels,-1.683000e+01,NaN
133,133,Interglobe Aviation Ltd. (Covered call) $$,Transport Services,-4.728000e+01,NaN
135,135,Foreign Securities/Overseas ETFs,NaN,1.389340e+04,0.001652
138,138,Compulsory Convertible Debenture,NaN,2.082801e+04,0.002476
143,143,Debt Instruments,NaN,9.210161e+05,0.109504
145,145,Listed / Awaiting Listing On Stock Exchanges,NaN,8.883308e+05,0.105617
147,147,Government Securities,NaN,4.240657e+05,0.050419


In [129]:
# Inspect the end of the main holdings and the start of derivative sections
display(
    icici_sample_df.iloc[265:322, 1:8]
)

,1,2,3,4,5,6,7
265,91 Days Treasury Bills,IN002026X255,NaN,SOV,20000000,19750.02,0.002348
266,182 Days Treasury Bills,IN002026Y022,NaN,SOV,17500000,17465,0.002076
267,364 Days Treasury Bills,IN002025Z286,NaN,SOV,7100000,7092.93,0.000843
268,364 Days Treasury Bills,IN002025Z344,NaN,SOV,5000000,4964.38,0.00059
269,364 Days Treasury Bills,IN002025Z294,NaN,SOV,2100000,2095.8,0.000249
270,182 Days Treasury Bills,IN002026Y014,NaN,SOV,203800,203.6,^
271,NaN,NaN,NaN,NaN,NaN,NaN,NaN
272,Units of Mutual Fund,NaN,NaN,NaN,NaN,846910.49,0.100693
273,ICICI Prudential Gold ETF,INF109KC1NT3,NaN,Mutual Fund,671777975,846910.49,0.100693
274,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [130]:
# Inspect the rows around the end of the main portfolio holdings
display(icici_sample_df.iloc[270:294, 1:8])

,1,2,3,4,5,6,7
270,182 Days Treasury Bills,IN002026Y014,NaN,SOV,203800,203.6,^
271,NaN,NaN,NaN,NaN,NaN,NaN,NaN
272,Units of Mutual Fund,NaN,NaN,NaN,NaN,846910.49,0.100693
273,ICICI Prudential Gold ETF,INF109KC1NT3,NaN,Mutual Fund,671777975,846910.49,0.100693
274,NaN,NaN,NaN,NaN,NaN,NaN,NaN
275,Reverse Repo,NaN,NaN,NaN,NaN,49999.71,0.005945
276,NaN,NaN,NaN,NaN,NaN,NaN,NaN
277,Units of Infrastructure Investment Trusts (Inv...,NaN,NaN,NaN,NaN,27500.24,0.00327
278,IRB Infrastructure Developers Ltd.,INE183W23014,NaN,Units of Infrastructure Investment Trusts,15079364,9683.97,0.001151
279,Cube Highways Trust,INE0NR623014,NaN,Units of Infrastructure Investment Trusts,4200000,6514.2,0.000775


In [131]:
# Inspect all rows in the main holdings section
# Columns: instrument name, ISIN, industry/rating, market value

display(icici_sample_df.iloc[4:293, [1, 2, 4, 6]].to_string())

"                                                                1             2                                                 4               6\n4                   Equity & Equity Related Instruments (Note -1)           NaN                                               NaN      5762358.53\n5                   Units of Real Estate Investment Trust (REITs)           NaN                                               NaN         82027.3\n6                                       EMBASSY OFFICE PARKS REIT  INE041025011     Units of Real Estate Investment Trust (REITs)        44368.16\n7                         Brookfield India Real Estate Trust REIT  INE0FDU25010     Units of Real Estate Investment Trust (REITs)        22032.65\n8                                              Nexus Select Trust  INE0NDH25011     Units of Real Estate Investment Trust (REITs)         7995.77\n9                                   MINDSPACE BUSINESS PARKS REIT  INE0CCU25019     Units of Real Estate Investment T

In [132]:
import pandas as pd

# Main portfolio rows only; stop before derivative disclosures
icici_test = icici_sample_df.iloc[4:292].copy()

icici_test = icici_test.rename(columns={
    1: "Instrument_Name",
    2: "ISIN",
    4: "Industry_or_Rating",
    6: "Market_Value_Lakh"
})

icici_test["Market_Value_Lakh"] = pd.to_numeric(
    icici_test["Market_Value_Lakh"], errors="coerce"
)

# Retain individual securities with an ISIN,
# plus valid portfolio line items without an ISIN
valid_no_isin = [
    "Reverse Repo",
    "TREPS",
    "Cash Margin - Derivatives",
    "Net Current Assets"
]

name = icici_test["Instrument_Name"].astype(str).str.strip()

keep = (
    (icici_test["ISIN"].notna() & icici_test["Market_Value_Lakh"].notna())
    | (name.isin(valid_no_isin) & icici_test["Market_Value_Lakh"].notna())
    | (
        name.str.contains(r"\(Covered call\)", case=False, na=False)
        & icici_test["Market_Value_Lakh"].notna()
    )
)

icici_test_holdings = icici_test.loc[keep].copy()

print("Individual holding rows:", len(icici_test_holdings))
print(
    "Selected holdings total (₹ lakh):",
    round(icici_test_holdings["Market_Value_Lakh"].sum(), 2)
)
print(
    "Reported Total Net Assets (₹ lakh):",
    round(float(icici_sample_df.iloc[292, 6]), 2)
)
print(
    "Difference (₹ lakh):",
    round(
        icici_test_holdings["Market_Value_Lakh"].sum()
        - float(icici_sample_df.iloc[292, 6]),
        2
    )
)

Individual holding rows: 239
Selected holdings total (₹ lakh): 8410836.19
Reported Total Net Assets (₹ lakh): 8410836.19
Difference (₹ lakh): 0.0


In [133]:
# Validate the selected holdings against Total Net Assets
# across all ICICI monthly reports

icici_validation_results = []

for file in icici_files:
    df = pd.read_excel(file, sheet_name="MULTI", header=None)

    # Locate the Total Net Assets row
    total_rows = df.index[
        df.iloc[:, 1].astype(str).str.strip().eq("Total Net Assets")
    ]

    if len(total_rows) != 1:
        icici_validation_results.append({
            "Source_File": file.name,
            "Status": "REVIEW: Total Net Assets row issue"
        })
        continue

    total_row = total_rows[0]
    reported_total = pd.to_numeric(
        df.iloc[total_row, 6], errors="coerce"
    )

    # Select only rows before Total Net Assets
    main = df.iloc[4:total_row].copy()
    main = main.rename(columns={
        1: "Instrument_Name",
        2: "ISIN",
        4: "Industry_or_Rating",
        6: "Market_Value_Lakh"
    })

    main["Market_Value_Lakh"] = pd.to_numeric(
        main["Market_Value_Lakh"], errors="coerce"
    )

    names = main["Instrument_Name"].astype(str).str.strip()

    valid_no_isin = [
        "Reverse Repo",
        "TREPS",
        "Cash Margin - Derivatives",
        "Net Current Assets"
    ]

    keep = (
        (
            main["ISIN"].notna()
            & main["Market_Value_Lakh"].notna()
        )
        | (
            names.isin(valid_no_isin)
            & main["Market_Value_Lakh"].notna()
        )
        | (
            names.str.contains(
                r"\(Covered call\)", case=False, na=False
            )
            & main["Market_Value_Lakh"].notna()
        )
    )

    selected_total = main.loc[
        keep, "Market_Value_Lakh"
    ].sum()

    difference = selected_total - reported_total

    icici_validation_results.append({
        "Source_File": file.name,
        "Selected_Rows": int(keep.sum()),
        "Selected_Total_Lakh": round(selected_total, 2),
        "Reported_Total_Lakh": round(reported_total, 2),
        "Difference_Lakh": round(difference, 2),
        "Status": "OK" if abs(difference) < 0.02 else "REVIEW"
    })

icici_validation = pd.DataFrame(icici_validation_results)

print(icici_validation["Status"].value_counts())
display(
    icici_validation[
        icici_validation["Status"] != "OK"
    ]
)

Status
OK        24
REVIEW     9
Name: count, dtype: int64


,Source_File,Selected_Rows,Selected_Total_Lakh,Reported_Total_Lakh,Difference_Lakh,Status
2,ICICI Prudential Multi-Asset Fund-apr-2024.xlsx,159,3952784.49,3953459.36,-674.87,REVIEW
4,ICICI Prudential Multi-Asset Fund-apr-2026.xlsx,261,8300870.45,8354735.20,-53864.75,REVIEW
11,ICICI Prudential Multi-Asset Fund-feb-2026.xlsx,281,8248769.20,8304478.01,-55708.81,REVIEW
14,ICICI Prudential Multi-Asset Fund-jan-2026.xlsx,245,8019227.14,8076822.66,-57595.52,REVIEW
18,ICICI Prudential Multi-Asset Fund-june-2024.xlsx,176,4392195.93,4392878.17,-682.24,REVIEW
22,ICICI Prudential Multi-Asset Fund-mar-2025.xlsx,182,5532933.19,5536032.23,-3099.04,REVIEW
23,ICICI Prudential Multi-Asset Fund-mar-2026.xlsx,247,7714351.98,7765826.74,-51474.76,REVIEW
24,ICICI Prudential Multi-Asset Fund-may-2024.xlsx,179,4115641.82,4115951.52,-309.70,REVIEW
26,ICICI Prudential Multi-Asset Fund-may-2026.xlsx,250,8361805.61,8416517.78,-54712.17,REVIEW


In [134]:
# Inspect rows excluded from the April 2026 holdings selection

file = next(
    f for f in icici_files
    if "apr-2026" in f.name.lower()
)

df = pd.read_excel(file, sheet_name="MULTI", header=None)

total_row = df.index[
    df.iloc[:, 1].astype(str).str.strip().eq("Total Net Assets")
][0]

main = df.iloc[4:total_row].copy()
main = main.rename(columns={
    1: "Instrument_Name",
    2: "ISIN",
    4: "Industry_or_Rating",
    6: "Market_Value_Lakh"
})

main["Market_Value_Lakh"] = pd.to_numeric(
    main["Market_Value_Lakh"], errors="coerce"
)

names = main["Instrument_Name"].astype(str).str.strip()

valid_no_isin = [
    "Reverse Repo",
    "TREPS",
    "Cash Margin - Derivatives",
    "Net Current Assets"
]

keep = (
    (
        main["ISIN"].notna()
        & main["Market_Value_Lakh"].notna()
    )
    | (
        names.isin(valid_no_isin)
        & main["Market_Value_Lakh"].notna()
    )
    | (
        names.str.contains(
            r"\(Covered call\)", case=False, na=False
        )
        & main["Market_Value_Lakh"].notna()
    )
)

excluded = main.loc[
    (~keep) & main["Market_Value_Lakh"].notna(),
    ["Instrument_Name", "ISIN", "Industry_or_Rating",
     "Market_Value_Lakh"]
].copy()

excluded.insert(0, "Source_Row", excluded.index + 1)

print("Total value of excluded numeric rows (₹ lakh):",
      round(excluded["Market_Value_Lakh"].sum(), 2))

display(excluded)

Total value of excluded numeric rows (₹ lakh): 15496329.24


,Source_Row,Instrument_Name,ISIN,Industry_or_Rating,Market_Value_Lakh
4,5,Equity & Equity Related Instruments (Note -1),NaN,NaN,5.861242e+06
6,7,Listed / Awaiting Listing On Stock Exchanges,NaN,NaN,5.751702e+06
136,137,Nifty 50 Index Put option $$,NaN,Index Options,6.986900e+02
137,138,Nifty 50 Index Put option $$,NaN,Index Options,6.242900e+02
155,156,Compulsory Convertible Debenture,NaN,NaN,1.755120e+04
158,159,Units of Real Estate Investment Trust (REITs),NaN,NaN,7.929232e+04
164,165,Foreign Securities/Overseas ETFs,NaN,NaN,1.269643e+04
169,170,Debt Instruments,NaN,NaN,6.208867e+05
171,172,Listed / Awaiting Listing On Stock Exchanges,NaN,NaN,5.774041e+05
173,174,Government Securities,NaN,NaN,3.026765e+05


In [135]:
# Inspect the Gold and Mutual Fund sections in April 2026

display(
    df.iloc[295:312, [1, 2, 4, 6, 7]].rename(columns={
        1: "Instrument_Name",
        2: "ISIN",
        4: "Industry_or_Rating",
        6: "Market_Value_Lakh",
        7: "NAV_Ratio"
    })
)

,Instrument_Name,ISIN,Industry_or_Rating,Market_Value_Lakh,NAV_Ratio
295,91 Days Treasury Bills,IN002025X489,SOV,13236.64,0.001584
296,182 Days Treasury Bills,IN002025Y370,SOV,12825.8,0.001535
297,182 Days Treasury Bills,IN002025Y420,SOV,7411.61,0.000887
298,364 Days Treasury Bills,IN002025Z112,SOV,3181.59,0.000381
299,91 Days Treasury Bills,IN002026X016,SOV,1881.36,0.000225
300,NaN,NaN,NaN,NaN,NaN
301,Gold,NaN,NaN,52541.77,0.006289
302,Gold (995 Purity) ^,NaN,Gold,29925.44,0.003582
303,Gold (995 Purity) ^,NaN,Gold,22616.33,0.002707
304,NaN,NaN,NaN,NaN,NaN


In [136]:
# Inspect all numeric rows without an ISIN
no_isin = main[
    main["ISIN"].isna()
    & main["Market_Value_Lakh"].notna()
].copy()

no_isin.insert(0, "Source_Row", no_isin.index + 1)

display(
    no_isin[
        ["Source_Row", "Instrument_Name",
         "Industry_or_Rating", "Market_Value_Lakh"]
    ]
)

,Source_Row,Instrument_Name,Industry_or_Rating,Market_Value_Lakh
4,5,Equity & Equity Related Instruments (Note -1),NaN,5.861242e+06
6,7,Listed / Awaiting Listing On Stock Exchanges,NaN,5.751702e+06
136,137,Nifty 50 Index Put option $$,Index Options,6.986900e+02
137,138,Nifty 50 Index Put option $$,Index Options,6.242900e+02
138,139,Interglobe Aviation Ltd. (Covered call) $$,Transport Services,-3.163000e+01
139,140,Tata Motors Ltd. (Covered call) $$,Automobiles,-6.240000e+01
140,141,ITC Ltd. (Covered call) $$,Diversified Fmcg,-8.160000e+01
141,142,ITC Ltd. (Covered call) $$,Diversified Fmcg,-8.280000e+01
142,143,Sun Pharmaceutical Industries Ltd. (Covered ca...,Pharmaceuticals & Biotechnology,-8.701000e+01
143,144,Bharti Airtel Ltd. (Covered call) $$,Telecom - Services,-9.102000e+01


In [137]:
# Show the full main holdings section around the reconciliation boundary
display(
    df.iloc[285:total_row + 1, [1, 2, 4, 6, 7]].rename(columns={
        1: "Instrument_Name",
        2: "ISIN",
        4: "Industry_or_Rating",
        6: "Market_Value_Lakh",
        7: "NAV_Ratio"
    })
)

,Instrument_Name,ISIN,Industry_or_Rating,Market_Value_Lakh,NAV_Ratio
285,Canara Bank **,INE476A16G44,CRISIL A1+,4741.12,0.000567
286,NaN,NaN,NaN,NaN,NaN
287,Commercial Papers,NaN,NaN,19885.32,0.00238
288,Aditya Birla Housing Finance Ltd. **,INE831R14FU4,CRISIL A1+,14984.88,0.001794
289,Panatone Finvest Ltd. **,INE116F14232,CRISIL A1+,4900.44,0.000587
290,NaN,NaN,NaN,NaN,NaN
291,Treasury Bills,NaN,NaN,160677.3,0.019232
292,364 Days Treasury Bills,IN002025Z062,SOV,68942.11,0.008252
293,91 Days Treasury Bills,IN002026X032,SOV,37058.06,0.004436
294,364 Days Treasury Bills,IN002025Z153,SOV,16140.13,0.001932


In [138]:
print("Selected total:", round(main.loc[keep, "Market_Value_Lakh"].sum(), 2))
print("Reported total:", round(pd.to_numeric(df.iloc[total_row, 6]), 2))
print("Difference:", round(
    main.loc[keep, "Market_Value_Lakh"].sum()
    - pd.to_numeric(df.iloc[total_row, 6]), 2
))

Selected total: 8300870.45
Reported total: 8354735.2
Difference: -53864.75


In [139]:
# Show rows with blank ISIN that are currently excluded,
# excluding obvious category subtotal rows.

excluded_no_isin = main.loc[
    (~keep)
    & main["ISIN"].isna()
    & main["Market_Value_Lakh"].notna()
].copy()

excluded_no_isin["Name_Clean"] = (
    excluded_no_isin["Instrument_Name"]
    .astype(str)
    .str.strip()
)

subtotal_terms = [
    "equity & equity related instruments",
    "listed / awaiting listing on stock exchanges",
    "compulsory convertible debenture",
    "units of real estate investment trust",
    "foreign securities/overseas etfs",
    "debt instruments",
    "government securities",
    "non-convertible debentures",
    "securitized debt instruments",
    "money market instruments",
    "certificate of deposits",
    "commercial papers",
    "treasury bills",
    "gold",
    "units of mutual fund",
    "units of infrastructure investment trusts",
    "others"
]

is_subtotal = excluded_no_isin["Name_Clean"].str.lower().apply(
    lambda name: any(term in name for term in subtotal_terms)
)

display(
    excluded_no_isin.loc[
        ~is_subtotal,
        ["Instrument_Name", "Industry_or_Rating", "Market_Value_Lakh"]
    ]
)

,Instrument_Name,Industry_or_Rating,Market_Value_Lakh
136,Nifty 50 Index Put option $$,Index Options,698.69
137,Nifty 50 Index Put option $$,Index Options,624.29
260,Privately Placed/Unlisted,NaN,402.56


In [140]:
display(
    df.iloc[252:267, [1, 2, 4, 6, 7]].rename(columns={
        1: "Instrument_Name",
        2: "ISIN",
        4: "Industry_or_Rating",
        6: "Market_Value_Lakh",
        7: "NAV_Ratio"
    })
)

,Instrument_Name,ISIN,Industry_or_Rating,Market_Value_Lakh,NAV_Ratio
252,India Universal Trust AL1 **,INE16J715035,FITCH AAA(SO),14820.87,0.001774
253,India Universal Trust AL2 **,INE1CBK15037,CRISIL AAA(SO),7278.4,0.000871
254,Sansar Trust **,INE14LF15016,CRISIL AAA(SO),7182,0.00086
255,India Universal Trust AL1 **,INE16J715027,FITCH AAA(SO),6108.35,0.000731
256,India Universal Trust AL2 **,INE1CBK15029,CRISIL AAA(SO),5386.65,0.000645
257,India Universal Trust AL2 **,INE1CBK15011,CRISIL AAA(SO),1472.94,0.000176
258,India Universal Trust AL1 **,INE16J715019,FITCH AAA(SO),830.86,0.000099
259,NaN,NaN,NaN,NaN,NaN
260,Privately Placed/Unlisted,NaN,NaN,402.56,^
261,Non Convertible Preference Shares,NaN,NaN,NaN,NaN


In [141]:
# Compare the selected NAV percentage sum with the reported total NAV
main["NAV_Ratio"] = pd.to_numeric(df.iloc[4:total_row, 7], errors="coerce")

print(
    "Selected market value (₹ lakh):",
    round(main.loc[keep, "Market_Value_Lakh"].sum(), 2)
)
print(
    "Selected NAV ratio:",
    round(main.loc[keep, "NAV_Ratio"].sum(), 6)
)
print(
    "Reported NAV ratio:",
    pd.to_numeric(df.iloc[total_row, 7], errors="coerce")
)

Selected market value (₹ lakh): 8300870.45
Selected NAV ratio: 0.993693
Reported NAV ratio: 0.9999999999882618


In [142]:
# Inspect excluded rows with non-zero market values and NAV ratios
excluded_check = main.loc[
    (~keep) & main["Market_Value_Lakh"].notna(),
    ["Instrument_Name", "ISIN", "Industry_or_Rating",
     "Market_Value_Lakh", "NAV_Ratio"]
].copy()

display(
    excluded_check.sort_values(
        "Market_Value_Lakh", ascending=False
    )
)

,Instrument_Name,ISIN,Industry_or_Rating,Market_Value_Lakh,NAV_Ratio
4,Equity & Equity Related Instruments (Note -1),NaN,NaN,5.861242e+06,0.701782
6,Listed / Awaiting Listing On Stock Exchanges,NaN,NaN,5.751702e+06,0.688671
305,Units of Mutual Fund,NaN,NaN,8.590471e+05,0.102822
169,Debt Instruments,NaN,NaN,6.208867e+05,0.074267
171,Listed / Awaiting Listing On Stock Exchanges,NaN,NaN,5.774041e+05,0.069111
270,Money Market Instruments,NaN,NaN,4.675695e+05,0.055965
173,Government Securities,NaN,NaN,3.026765e+05,0.036228
272,Certificate of Deposits,NaN,NaN,2.870069e+05,0.034353
212,Non-Convertible debentures / Bonds,NaN,NaN,2.747275e+05,0.032883
291,Treasury Bills,NaN,NaN,1.606773e+05,0.019232


In [143]:
# Check the effect of adding identified individual holdings without an ISIN
extra_rows = main.index.isin([136, 137, 302, 303])

print(
    "Additional market value (₹ lakh):",
    round(main.loc[extra_rows, "Market_Value_Lakh"].sum(), 2)
)

print(
    "Additional NAV ratio:",
    round(main.loc[extra_rows, "NAV_Ratio"].sum(), 6)
)

print(
    "Revised market value (₹ lakh):",
    round(
        main.loc[keep, "Market_Value_Lakh"].sum()
        + main.loc[extra_rows, "Market_Value_Lakh"].sum(),
        2
    )
)

print(
    "Revised NAV ratio:",
    round(
        main.loc[keep, "NAV_Ratio"].sum()
        + main.loc[extra_rows, "NAV_Ratio"].sum(),
        6
    )
)

Additional market value (₹ lakh): 53864.75
Additional NAV ratio: 0.006447
Revised market value (₹ lakh): 8354735.2
Revised NAV ratio: 1.00014


In [144]:
display(
    main.loc[
        main.index.isin([136, 137, 302, 303]),
        [
            "Instrument_Name",
            "ISIN",
            "Industry_or_Rating",
            "Market_Value_Lakh",
            "NAV_Ratio"
        ]
    ]
)

,Instrument_Name,ISIN,Industry_or_Rating,Market_Value_Lakh,NAV_Ratio
136,Nifty 50 Index Put option $$,NaN,Index Options,698.69,0.000084
137,Nifty 50 Index Put option $$,NaN,Index Options,624.29,0.000075
302,Gold (995 Purity) ^,NaN,Gold,29925.44,0.003582
303,Gold (995 Purity) ^,NaN,Gold,22616.33,0.002707


In [145]:
# List excluded rows that are not obvious category subtotals
excluded_no_isin = main.loc[
    (~keep)
    & main["ISIN"].isna()
    & main["Market_Value_Lakh"].notna()
].copy()

excluded_no_isin["Instrument_Name"] = (
    excluded_no_isin["Instrument_Name"].astype(str).str.strip()
)

display(
    excluded_no_isin[
        [
            "Instrument_Name",
            "Industry_or_Rating",
            "Market_Value_Lakh",
            "NAV_Ratio"
        ]
    ]
)

,Instrument_Name,Industry_or_Rating,Market_Value_Lakh,NAV_Ratio
4,Equity & Equity Related Instruments (Note -1),NaN,5.861242e+06,0.701782
6,Listed / Awaiting Listing On Stock Exchanges,NaN,5.751702e+06,0.688671
136,Nifty 50 Index Put option $$,Index Options,6.986900e+02,0.000084
137,Nifty 50 Index Put option $$,Index Options,6.242900e+02,0.000075
155,Compulsory Convertible Debenture,NaN,1.755120e+04,0.002101
158,Units of Real Estate Investment Trust (REITs),NaN,7.929232e+04,0.009491
164,Foreign Securities/Overseas ETFs,NaN,1.269643e+04,0.001520
169,Debt Instruments,NaN,6.208867e+05,0.074267
171,Listed / Awaiting Listing On Stock Exchanges,NaN,5.774041e+05,0.069111
173,Government Securities,NaN,3.026765e+05,0.036228


In [147]:

import pandas as pd
from pathlib import Path

validation_results = []

# Known no-ISIN individual holdings or valid accounting lines
valid_no_isin = [
    "Reverse Repo",
    "TREPS",
    "Cash Margin - Derivatives",
    "Net Current Assets",
    "Gold (995 Purity) ^",
    "Nifty 50 Index Put option $$",
]

for file_path in icici_files:
    file_path = Path(file_path)
    df = pd.read_excel(file_path, sheet_name="MULTI", header=None)

    # Locate Total Net Assets dynamically
    names = df.iloc[:, 1].astype(str).str.strip()
    total_matches = df.index[
        names.str.fullmatch("Total Net Assets", case=False, na=False)
    ]

    if len(total_matches) == 0:
        validation_results.append({
            "Source_File": file_path.name,
            "Status": "REVIEW - Total Net Assets not found",
        })
        continue

    total_row = total_matches[0]
    main = df.iloc[4:total_row].copy()

    main = main.rename(columns={
        1: "Instrument_Name",
        2: "ISIN",
        4: "Industry_or_Rating",
        5: "Quantity",
        6: "Market_Value_Lakh",
        7: "NAV_Ratio",
    })

    main["Instrument_Name"] = (
        main["Instrument_Name"].astype(str).str.strip()
    )
    main["Market_Value_Lakh"] = pd.to_numeric(
        main["Market_Value_Lakh"], errors="coerce"
    )

    # Ordinary individual holdings with an ISIN
    has_isin = (
        main["ISIN"].notna()
        & main["ISIN"].astype(str).str.strip().ne("")
        & main["ISIN"].astype(str).str.lower().ne("nan")
    )

    # Valid individual holdings/accounting rows without an ISIN
    no_isin_valid = main["Instrument_Name"].isin(valid_no_isin)

    # Covered-call rows are individual derivative positions
    covered_call = main["Instrument_Name"].str.contains(
        r"\(Covered call\)", case=False, na=False
    )

    keep = (
        (has_isin | no_isin_valid | covered_call)
        & main["Market_Value_Lakh"].notna()
    )

    selected_total = main.loc[keep, "Market_Value_Lakh"].sum()
    reported_total = pd.to_numeric(
        df.iloc[total_row, 6], errors="coerce"
    )

    difference = selected_total - reported_total
    tolerance = 1.0  # ₹1 lakh tolerance for report reconciliation

    validation_results.append({
        "Report_Month": file_path.stem,
        "Source_File": file_path.name,
        "Selected_Rows": int(keep.sum()),
        "Selected_Value_Lakh": round(selected_total, 2),
        "Reported_Total_Lakh": round(reported_total, 2),
        "Difference_Lakh": round(difference, 2),
        "Status": "OK" if abs(difference) <= tolerance else "REVIEW",
    })

validation_df = pd.DataFrame(validation_results)

display(validation_df)
print("\nStatus summary:")
print(validation_df["Status"].value_counts(dropna=False))


,Report_Month,Source_File,Selected_Rows,Selected_Value_Lakh,Reported_Total_Lakh,Difference_Lakh,Status
0,ICICI Prudential Multi Asset Allocation Fund-a...,ICICI Prudential Multi Asset Allocation Fund-a...,262,8783309.61,8783309.61,0.00,OK
1,ICICI Prudential Multi Asset Allocation Fund-s...,ICICI Prudential Multi Asset Allocation Fund-s...,239,8410836.19,8410836.19,0.00,OK
2,ICICI Prudential Multi-Asset Fund-apr-2024,ICICI Prudential Multi-Asset Fund-apr-2024.xlsx,159,3952784.49,3953459.36,-674.87,REVIEW
3,ICICI Prudential Multi-Asset Fund-apr-2025,ICICI Prudential Multi-Asset Fund-apr-2025.xlsx,213,5748450.97,5748450.97,-0.00,OK
4,ICICI Prudential Multi-Asset Fund-apr-2026,ICICI Prudential Multi-Asset Fund-apr-2026.xlsx,265,8354735.20,8354735.20,0.00,OK
5,ICICI Prudential Multi-Asset Fund-aug-2024,ICICI Prudential Multi-Asset Fund-aug-2024.xlsx,166,4820126.03,4820126.03,0.00,OK
6,ICICI Prudential Multi-Asset Fund-aug-2025,ICICI Prudential Multi-Asset Fund-aug-2025.xlsx,204,6477024.09,6477024.09,0.00,OK
7,ICICI Prudential Multi-Asset Fund-dec-2024,ICICI Prudential Multi-Asset Fund-dec-2024.xlsx,224,5102745.82,5102745.82,-0.00,OK
8,ICICI Prudential Multi-Asset Fund-dec-2025,ICICI Prudential Multi-Asset Fund-dec-2025.xlsx,248,7817911.37,7817911.37,-0.00,OK
9,ICICI Prudential Multi-Asset Fund-feb-2024,ICICI Prudential Multi-Asset Fund-feb-2024.xlsx,130,3470383.61,3470383.61,0.00,OK



Status summary:
Status
OK        31
REVIEW     2
Name: count, dtype: int64


In [148]:

from pathlib import Path
import pandas as pd

file_path = next(
    p for p in icici_files
    if "apr-2024" in Path(p).name.lower()
)

df = pd.read_excel(file_path, sheet_name="MULTI", header=None)

names = df.iloc[:, 1].astype(str).str.strip()
total_row = df.index[
    names.str.fullmatch("Total Net Assets", case=False, na=False)
][0]

main = df.iloc[4:total_row].copy()
main = main.rename(columns={
    1: "Instrument_Name",
    2: "ISIN",
    4: "Industry_or_Rating",
    5: "Quantity",
    6: "Market_Value_Lakh",
    7: "NAV_Ratio",
})

main["Instrument_Name"] = (
    main["Instrument_Name"].astype(str).str.strip()
)
main["Market_Value_Lakh"] = pd.to_numeric(
    main["Market_Value_Lakh"], errors="coerce"
)

has_isin = (
    main["ISIN"].notna()
    & main["ISIN"].astype(str).str.strip().ne("")
    & main["ISIN"].astype(str).str.lower().ne("nan")
)

valid_no_isin = [
    "Reverse Repo", "TREPS", "Cash Margin - Derivatives",
    "Net Current Assets", "Gold (995 Purity) ^",
    "Nifty 50 Index Put option $$"
]

no_isin_valid = main["Instrument_Name"].isin(valid_no_isin)
covered_call = main["Instrument_Name"].str.contains(
    r"\(Covered call\)", case=False, na=False
)

keep = (
    (has_isin | no_isin_valid | covered_call)
    & main["Market_Value_Lakh"].notna()
)

display(
    main.loc[
        (~keep) & main["Market_Value_Lakh"].notna(),
        [
            "Instrument_Name", "ISIN",
            "Industry_or_Rating", "Market_Value_Lakh"
        ]
    ]
)


,Instrument_Name,ISIN,Industry_or_Rating,Market_Value_Lakh
4,Equity & Equity Related Instruments (Note -1),NaN,NaN,2.615217e+06
6,Listed / Awaiting Listing On Stock Exchanges,NaN,NaN,2.602816e+06
88,Nifty 50 Index Call option $$,NaN,Index Options,4.532700e+02
89,Nifty 50 Index Call option $$,NaN,Index Options,1.455700e+02
90,Nifty 50 Index Call option $$,NaN,Index Options,7.603000e+01
109,Compulsory Convertible Debenture,NaN,NaN,1.240100e+04
117,Debt Instruments,NaN,NaN,3.668330e+05
119,Listed / Awaiting Listing On Stock Exchanges,NaN,NaN,3.668330e+05
121,Government Securities,NaN,NaN,1.891888e+05
133,Non-Convertible debentures / Bonds,NaN,NaN,1.776442e+05


In [149]:
valid_no_isin = [
    "Reverse Repo",
    "TREPS",
    "Cash Margin - Derivatives",
    "Net Current Assets",
    "Gold (995 Purity) ^",
    "Nifty 50 Index Put option $$",
    "Nifty 50 Index Call option $$",
]

In [150]:
display(
    validation_df.loc[
        validation_df["Status"] == "REVIEW",
        [
            "Source_File",
            "Selected_Value_Lakh",
            "Reported_Total_Lakh",
            "Difference_Lakh"
        ]
    ]
)

,Source_File,Selected_Value_Lakh,Reported_Total_Lakh,Difference_Lakh
2,ICICI Prudential Multi-Asset Fund-apr-2024.xlsx,3952784.49,3953459.36,-674.87
24,ICICI Prudential Multi-Asset Fund-may-2024.xlsx,4115713.95,4115951.52,-237.57


In [151]:

from pathlib import Path
import pandas as pd

file_path = next(
    p for p in icici_files
    if "may-2024" in Path(p).name.lower()
)

df = pd.read_excel(file_path, sheet_name="MULTI", header=None)

names = df.iloc[:, 1].astype(str).str.strip()
total_row = df.index[
    names.str.fullmatch("Total Net Assets", case=False, na=False)
][0]

main = df.iloc[4:total_row].copy()
main = main.rename(columns={
    1: "Instrument_Name",
    2: "ISIN",
    4: "Industry_or_Rating",
    6: "Market_Value_Lakh",
    7: "NAV_Ratio",
})

main["Instrument_Name"] = (
    main["Instrument_Name"].astype(str).str.strip()
)
main["Market_Value_Lakh"] = pd.to_numeric(
    main["Market_Value_Lakh"], errors="coerce"
)

has_isin = (
    main["ISIN"].notna()
    & main["ISIN"].astype(str).str.strip().ne("")
    & main["ISIN"].astype(str).str.lower().ne("nan")
)

valid_no_isin = [
    "Reverse Repo", "TREPS", "Cash Margin - Derivatives",
    "Net Current Assets", "Gold (995 Purity) ^",
    "Nifty 50 Index Put option $$",
    "Nifty 50 Index Call option $$"
]

no_isin_valid = main["Instrument_Name"].isin(valid_no_isin)
covered_call = main["Instrument_Name"].str.contains(
    r"\(Covered call\)", case=False, na=False
)

keep = (
    (has_isin | no_isin_valid | covered_call)
    & main["Market_Value_Lakh"].notna()
)

display(
    main.loc[
        (~keep) & main["Market_Value_Lakh"].notna(),
        [
            "Instrument_Name", "ISIN",
            "Industry_or_Rating", "Market_Value_Lakh"
        ]
    ]
)


,Instrument_Name,ISIN,Industry_or_Rating,Market_Value_Lakh
4,Equity & Equity Related Instruments (Note -1),NaN,NaN,2.754917e+06
6,Listed / Awaiting Listing On Stock Exchanges,NaN,NaN,2.740074e+06
125,Compulsory Convertible Debenture,NaN,NaN,1.269737e+04
128,Foreign Securities/Overseas ETFs,NaN,NaN,2.145610e+03
133,Debt Instruments,NaN,NaN,3.966919e+05
135,Listed / Awaiting Listing On Stock Exchanges,NaN,NaN,3.966919e+05
137,Government Securities,NaN,NaN,1.941930e+05
150,Non-Convertible debentures / Bonds,NaN,NaN,2.024988e+05
198,Money Market Instruments,NaN,NaN,1.281337e+05
204,Treasury Bills,NaN,NaN,1.281337e+05


In [152]:
# Inspect context around the Gold row and nearby holdings
display(
    main.iloc[210:220][
        [
            "Instrument_Name",
            "ISIN",
            "Industry_or_Rating",
            "Market_Value_Lakh",
            "NAV_Ratio"
        ]
    ]
)

,Instrument_Name,ISIN,Industry_or_Rating,Market_Value_Lakh,NAV_Ratio
214,Gold,NaN,NaN,72.13,^
215,Gold (995 Purity) ^,NaN,Gold,72.13,^
216,nan,NaN,NaN,NaN,NaN
217,Units of Mutual Fund,NaN,NaN,195119.12,0.047406
218,ICICI PRUDENTIAL SILVER ETF,INF109KC1Y56,Mutual Fund,112667.68,0.027373
219,ICICI Prudential Gold ETF,INF109KC1NT3,Mutual Fund,82451.44,0.020032
220,nan,NaN,NaN,NaN,NaN
221,Units of Infrastructure Investment Trusts (Inv...,NaN,NaN,6808.00,0.001654
222,India Infrastructure Trust,INE05KD23015,Units of Infrastructure Investment Trusts,6808.00,0.001654
223,nan,NaN,NaN,NaN,NaN


In [153]:
display(
    main.loc[
        (~keep)
        & main["Market_Value_Lakh"].notna()
        & main["Industry_or_Rating"].notna(),
        [
            "Instrument_Name",
            "ISIN",
            "Industry_or_Rating",
            "Market_Value_Lakh",
            "NAV_Ratio"
        ]
    ]
)

,Instrument_Name,ISIN,Industry_or_Rating,Market_Value_Lakh,NAV_Ratio


In [154]:
main["NAV_Ratio"] = pd.to_numeric(
    main["NAV_Ratio"], errors="coerce"
)

print(
    "Selected NAV ratio:",
    round(main.loc[keep, "NAV_Ratio"].sum(), 6)
)

print(
    "Reported NAV ratio:",
    pd.to_numeric(
        df.iloc[total_row, 7], errors="coerce"
    )
)

Selected NAV ratio: 1.00043
Reported NAV ratio: 0.9999999999939477


In [155]:
print("May 2024 status: REVIEW")
print("Selected holdings NAV ratio: 1.00043")
print("Reported NAV ratio: 1.00000")
print("Market value difference: -₹237.57 lakh")
print("Note: Difference remains unexplained; no adjustment made.")

May 2024 status: REVIEW
Selected holdings NAV ratio: 1.00043
Reported NAV ratio: 1.00000
Market value difference: -₹237.57 lakh
Note: Difference remains unexplained; no adjustment made.


In [156]:
from pathlib import Path
import pandas as pd

validation_results = []

valid_no_isin = [
    "Reverse Repo",
    "TREPS",
    "Cash Margin - Derivatives",
    "Net Current Assets",
    "Gold (995 Purity) ^",
    "Nifty 50 Index Put option $$",
    "Nifty 50 Index Call option $$",
]

for file_path in icici_files:
    file_path = Path(file_path)
    df = pd.read_excel(file_path, sheet_name="MULTI", header=None)

    names = df.iloc[:, 1].astype(str).str.strip()
    total_rows = df.index[
        names.str.fullmatch("Total Net Assets", case=False, na=False)
    ]

    if len(total_rows) == 0:
        validation_results.append({
            "Report": file_path.name,
            "Status": "REVIEW",
            "Difference_Lakh": None,
            "Note": "Total Net Assets row not found"
        })
        continue

    total_row = total_rows[0]
    main = df.iloc[4:total_row].copy()

    main = main.rename(columns={
        1: "Instrument_Name",
        2: "ISIN",
        4: "Industry_or_Rating",
        6: "Market_Value_Lakh",
        7: "NAV_Ratio",
    })

    main["Instrument_Name"] = (
        main["Instrument_Name"].astype(str).str.strip()
    )
    main["Market_Value_Lakh"] = pd.to_numeric(
        main["Market_Value_Lakh"], errors="coerce"
    )

    isin = main["ISIN"].astype(str).str.strip()
    has_isin = (
        main["ISIN"].notna()
        & isin.ne("")
        & isin.str.lower().ne("nan")
    )

    name = main["Instrument_Name"]

    keep = (
        (
            has_isin
            | name.isin(valid_no_isin)
            | name.str.contains(r"\(Covered call\)", case=False, na=False)
        )
        & main["Market_Value_Lakh"].notna()
    )

    selected_total = main.loc[keep, "Market_Value_Lakh"].sum()
    reported_total = pd.to_numeric(
        df.iloc[total_row, 6], errors="coerce"
    )

    difference = selected_total - reported_total

    validation_results.append({
        "Report": file_path.name,
        "Selected_Rows": int(keep.sum()),
        "Selected_Total_Lakh": round(selected_total, 2),
        "Reported_Total_Lakh": round(reported_total, 2),
        "Difference_Lakh": round(difference, 2),
        "Status": "OK" if abs(difference) < 1 else "REVIEW"
    })

validation_df = pd.DataFrame(validation_results)

display(validation_df)
print("\nStatus counts:")
print(validation_df["Status"].value_counts())

,Report,Selected_Rows,Selected_Total_Lakh,Reported_Total_Lakh,Difference_Lakh,Status
0,ICICI Prudential Multi Asset Allocation Fund-a...,262,8783309.61,8783309.61,0.0,OK
1,ICICI Prudential Multi Asset Allocation Fund-s...,239,8410836.19,8410836.19,0.0,OK
2,ICICI Prudential Multi-Asset Fund-apr-2024.xlsx,162,3953459.36,3953459.36,-0.0,OK
3,ICICI Prudential Multi-Asset Fund-apr-2025.xlsx,213,5748450.97,5748450.97,-0.0,OK
4,ICICI Prudential Multi-Asset Fund-apr-2026.xlsx,265,8354735.20,8354735.20,0.0,OK
5,ICICI Prudential Multi-Asset Fund-aug-2024.xlsx,166,4820126.03,4820126.03,0.0,OK
6,ICICI Prudential Multi-Asset Fund-aug-2025.xlsx,204,6477024.09,6477024.09,0.0,OK
7,ICICI Prudential Multi-Asset Fund-dec-2024.xlsx,224,5102745.82,5102745.82,-0.0,OK
8,ICICI Prudential Multi-Asset Fund-dec-2025.xlsx,248,7817911.37,7817911.37,-0.0,OK
9,ICICI Prudential Multi-Asset Fund-feb-2024.xlsx,130,3470383.61,3470383.61,0.0,OK



Status counts:
Status
OK    33
Name: count, dtype: int64


In [157]:
from pathlib import Path
import pandas as pd
import re

valid_no_isin = [
    "Reverse Repo",
    "TREPS",
    "Cash Margin - Derivatives",
    "Net Current Assets",
    "Gold (995 Purity) ^",
    "Nifty 50 Index Put option $$",
    "Nifty 50 Index Call option $$",
]

all_holdings = []

for file_path in icici_files:
    file_path = Path(file_path)
    df = pd.read_excel(file_path, sheet_name="MULTI", header=None)

    names = df.iloc[:, 1].astype(str).str.strip()
    total_rows = df.index[
        names.str.fullmatch("Total Net Assets", case=False, na=False)
    ]

    if len(total_rows) == 0:
        print(f"Skipped: Total Net Assets not found in {file_path.name}")
        continue

    total_row = total_rows[0]
    main = df.iloc[4:total_row].copy()

    main = main.rename(columns={
        1: "Instrument_Name",
        2: "ISIN",
        4: "Industry_or_Rating",
        5: "Quantity",
        6: "Market_Value_Lakh",
        7: "NAV_Ratio",
    })

    main["Instrument_Name"] = main["Instrument_Name"].astype(str).str.strip()

    isin = main["ISIN"].astype(str).str.strip()
    has_isin = (
        main["ISIN"].notna()
        & isin.ne("")
        & isin.str.lower().ne("nan")
    )

    name = main["Instrument_Name"]

    keep = (
        (
            has_isin
            | name.isin(valid_no_isin)
            | name.str.contains(r"\(Covered call\)", case=False, na=False)
        )
        & pd.to_numeric(main["Market_Value_Lakh"], errors="coerce").notna()
    )

    holdings = main.loc[keep].copy()

    holdings["Market_Value_Lakh"] = pd.to_numeric(
        holdings["Market_Value_Lakh"], errors="coerce"
    )
    holdings["NAV_Pct"] = (
        pd.to_numeric(holdings["NAV_Ratio"], errors="coerce") * 100
    )

    # Extract month and year from the report filename
    filename = file_path.stem.lower()
    match = re.search(
        r"(jan|january|feb|february|mar|march|apr|april|may|"
        r"jun|june|jul|july|aug|august|sep|sept|september|"
        r"oct|october|nov|november|dec|december)[-_ ](20\d{2})",
        filename
    )

    if match:
        month_text, year = match.groups()
        month_number = pd.to_datetime(month_text, format="%B").month \
            if len(month_text) > 3 else pd.to_datetime(
                month_text[:3], format="%b"
            ).month
        report_month = f"{year}-{month_number:02d}"
    else:
        report_month = ""

    holdings["AMC"] = "ICICI Prudential"
    holdings["Report_Month"] = report_month
    holdings["Source_File"] = file_path.name
    holdings["Source_Row"] = holdings.index + 1

    holdings = holdings.rename(columns={
        "ISIN": "ISIN",
        "Instrument_Name": "Instrument_Name",
        "Quantity": "Quantity",
    })

    all_holdings.append(holdings[[
        "AMC",
        "Report_Month",
        "ISIN",
        "Instrument_Name",
        "Industry_or_Rating",
        "Quantity",
        "Market_Value_Lakh",
        "NAV_Pct",
        "Source_File",
        "Source_Row",
    ]])

icici_holdings_log = pd.concat(all_holdings, ignore_index=True)

output_path = OUTPUT_DIR / "ICICI_Prudential_Monthly_Holdings_Log.csv"
icici_holdings_log.to_csv(output_path, index=False)

print("Rows extracted:", len(icici_holdings_log))
print("Unique report months:", icici_holdings_log["Report_Month"].nunique())
print("Missing report months:", icici_holdings_log["Report_Month"].eq("").sum())
print("Saved to:", output_path)
display(icici_holdings_log.head(10))

Rows extracted: 6943
Unique report months: 33
Missing report months: 0
Saved to: c:\Users\rocks\OneDrive\Desktop\AMC Project Data\02_CLEAN_DATA\ICICI_Prudential_Monthly_Holdings_Log.csv


,AMC,Report_Month,ISIN,Instrument_Name,Industry_or_Rating,Quantity,Market_Value_Lakh,NAV_Pct,Source_File,Source_Row
0,ICICI Prudential,2026-08,INE040A01034,HDFC Bank Ltd.,Banks,69199542,490624.75,5.585876,ICICI Prudential Multi Asset Allocation Fund-a...,8
1,ICICI Prudential,2026-08,INE0J1Y01017,Life Insurance Corporation of India,Insurance,80107788,334850.55,3.812351,ICICI Prudential Multi Asset Allocation Fund-a...,9
2,ICICI Prudential,2026-08,INE090A01021,ICICI Bank Ltd.,Banks,22730775,330505.47,3.762881,ICICI Prudential Multi Asset Allocation Fund-a...,10
3,ICICI Prudential,2026-08,INE002A01018,Reliance Industries Ltd.,Petroleum Products,22432703,286465.62,3.261477,ICICI Prudential Multi Asset Allocation Fund-a...,11
4,ICICI Prudential,2026-08,INE918I01026,Bajaj Finserv Ltd.,Finance,11233749,227034.07,2.584835,ICICI Prudential Multi Asset Allocation Fund-a...,12
5,ICICI Prudential,2026-08,INE238A01034,Axis Bank Ltd.,Banks,16068164,208886.13,2.378217,ICICI Prudential Multi Asset Allocation Fund-a...,13
6,ICICI Prudential,2026-08,INE646L01027,Interglobe Aviation Ltd.,Transport Services,3146458,164685.61,1.874984,ICICI Prudential Multi Asset Allocation Fund-a...,14
7,ICICI Prudential,2026-08,INE009A01021,Infosys Ltd.,It - Software,13357602,151448.49,1.724276,ICICI Prudential Multi Asset Allocation Fund-a...,15
8,ICICI Prudential,2026-08,INE154A01025,ITC Ltd.,Diversified Fmcg,54664918,139668.87,1.590162,ICICI Prudential Multi Asset Allocation Fund-a...,16
9,ICICI Prudential,2026-08,INE018E01016,SBI Cards & Payment Services Ltd.,Finance,20910198,134818.50,1.534940,ICICI Prudential Multi Asset Allocation Fund-a...,17


In [158]:
# Final quality check for ICICI Prudential holdings log

print("Total rows:", len(icici_holdings_log))
print("Unique months:", icici_holdings_log["Report_Month"].nunique())
print(
    "Missing months:",
    sorted(
        set(pd.period_range("2024-01", "2026-09", freq="M").astype(str))
        - set(icici_holdings_log["Report_Month"].unique())
    )
)

print("\nMissing values in key fields:")
display(
    icici_holdings_log[
        [
            "Report_Month",
            "Instrument_Name",
            "Market_Value_Lakh",
            "NAV_Pct",
            "Source_File"
        ]
    ].isna().sum()
)

print("\nRows per month:")
display(
    icici_holdings_log.groupby("Report_Month").size().reset_index(
        name="Holding_Rows"
    )
)

Total rows: 6943
Unique months: 33
Missing months: []

Missing values in key fields:


Report_Month           0
Instrument_Name        0
Market_Value_Lakh      0
NAV_Pct              567
Source_File            0
dtype: int64


Rows per month:


,Report_Month,Holding_Rows
0,2024-01,132
1,2024-02,130
2,2024-03,141
3,2024-04,162
4,2024-05,181
5,2024-06,181
6,2024-07,191
7,2024-08,166
8,2024-09,196
9,2024-10,165


In [159]:
# Inspect rows with missing NAV percentages
missing_nav = icici_holdings_log[
    icici_holdings_log["NAV_Pct"].isna()
]

print("Rows with missing NAV_Pct:", len(missing_nav))

display(
    missing_nav[
        [
            "Report_Month",
            "ISIN",
            "Instrument_Name",
            "Market_Value_Lakh",
            "Source_File",
            "Source_Row"
        ]
    ].head(20)
)

Rows with missing NAV_Pct: 567


,Report_Month,ISIN,Instrument_Name,Market_Value_Lakh,Source_File,Source_Row
118,2026-08,INE00WC01027,Affle India Ltd.,230.19,ICICI Prudential Multi Asset Allocation Fund-a...,126
119,2026-08,INE296A01032,Bajaj Finance Ltd.,59.51,ICICI Prudential Multi Asset Allocation Fund-a...,127
120,2026-08,NaN,Coal India Ltd.(Covered call) $$,-9.32,ICICI Prudential Multi Asset Allocation Fund-a...,128
121,2026-08,NaN,Asian Paints Ltd.(Covered call) $$,-14.35,ICICI Prudential Multi Asset Allocation Fund-a...,129
122,2026-08,NaN,HDFC Life Insurance Company Ltd(Covered call) $$,-15.68,ICICI Prudential Multi Asset Allocation Fund-a...,130
123,2026-08,NaN,Titan Company Ltd.(Covered call) $$,-29.49,ICICI Prudential Multi Asset Allocation Fund-a...,131
124,2026-08,NaN,State Bank Of India(Covered call) $$,-39.78,ICICI Prudential Multi Asset Allocation Fund-a...,132
125,2026-08,NaN,Sun Pharmaceutical Industries Ltd.(Covered cal...,-43.79,ICICI Prudential Multi Asset Allocation Fund-a...,133
126,2026-08,NaN,Tech Mahindra Ltd.(Covered call) $$,-44.55,ICICI Prudential Multi Asset Allocation Fund-a...,134
127,2026-08,NaN,State Bank Of India(Covered call) $$,-50.63,ICICI Prudential Multi Asset Allocation Fund-a...,135


In [160]:
# Inspect the original NAV-ratio cells for the missing values
missing_nav = icici_holdings_log[
    icici_holdings_log["NAV_Pct"].isna()
]

for _, row in missing_nav.head(10).iterrows():
    file_path = next(
        p for p in icici_files
        if Path(p).name == row["Source_File"]
    )

    source_df = pd.read_excel(
        file_path,
        sheet_name="MULTI",
        header=None
    )

    source_row = int(row["Source_Row"]) - 1

    print(
        row["Report_Month"],
        "|",
        row["Instrument_Name"],
        "| Original NAV cell:",
        repr(source_df.iloc[source_row, 7])
    )

2026-08 | Affle India Ltd. | Original NAV cell: '^'
2026-08 | Bajaj Finance Ltd. | Original NAV cell: '^'
2026-08 | Coal India Ltd.(Covered call) $$ | Original NAV cell: '^'
2026-08 | Asian Paints Ltd.(Covered call) $$ | Original NAV cell: '^'
2026-08 | HDFC Life Insurance Company Ltd(Covered call) $$ | Original NAV cell: '^'
2026-08 | Titan Company Ltd.(Covered call) $$ | Original NAV cell: '^'
2026-08 | State Bank Of India(Covered call) $$ | Original NAV cell: '^'
2026-08 | Sun Pharmaceutical Industries Ltd.(Covered call) $$ | Original NAV cell: '^'
2026-08 | Tech Mahindra Ltd.(Covered call) $$ | Original NAV cell: '^'
2026-08 | State Bank Of India(Covered call) $$ | Original NAV cell: '^'


In [161]:
# Preserve the source meaning of missing NAV percentages
icici_holdings_log["NAV_Pct_Note"] = ""

missing_nav_mask = icici_holdings_log["NAV_Pct"].isna()

icici_holdings_log.loc[
    missing_nav_mask, "NAV_Pct_Note"
] = "Source NAV cell contains '^'; percentage unavailable as numeric value"

# Save the updated log
output_path = OUTPUT_DIR / "ICICI_Prudential_Monthly_Holdings_Log.csv"
icici_holdings_log.to_csv(output_path, index=False)

print("Rows with source '^' in NAV percentage:", missing_nav_mask.sum())
print("Remaining missing numeric NAV percentages:",
      icici_holdings_log["NAV_Pct"].isna().sum())
print("Saved updated log to:", output_path)

Rows with source '^' in NAV percentage: 567
Remaining missing numeric NAV percentages: 567
Saved updated log to: c:\Users\rocks\OneDrive\Desktop\AMC Project Data\02_CLEAN_DATA\ICICI_Prudential_Monthly_Holdings_Log.csv


In [162]:
from pathlib import Path

SBI_DIR = PROJECT_DIR / "01_RAW_DATA" / "SBI"

sbi_files = sorted(
    p for p in SBI_DIR.iterdir()
    if p.is_file() and p.suffix.lower() in [".xlsx", ".xls", ".xlsm"]
)

print("Total SBI Excel reports:", len(sbi_files))

for p in sbi_files:
    print(p.name)

Total SBI Excel reports: 33
SBI-MULTI-ASSET-ALLOCATION-FUND---MARCH-2024.xlsx
SBI-MULTI-ASSET-ALLOCATION-FUND-feb-2024.xlsx
SBI-MULTI-ASSET-ALLOCATION-FUND-jan-2024.xlsx
SBI-MULTI-ASSET-ALLOCATION-FUND-MONTHLY-PORTFOLIO---APRIL-2024.xlsx
SBI-MULTI-ASSET-ALLOCATION-FUND-MONTHLY-PORTFOLIO---APRIL-2025.xlsx
SBI-MULTI-ASSET-ALLOCATION-FUND-MONTHLY-PORTFOLIO---APRIL-2026.xlsx
SBI-MULTI-ASSET-ALLOCATION-FUND-MONTHLY-PORTFOLIO---AUGUST-2024.xlsx
SBI-MULTI-ASSET-ALLOCATION-FUND-MONTHLY-PORTFOLIO---AUGUST-2025.xlsx
SBI-MULTI-ASSET-ALLOCATION-FUND-MONTHLY-PORTFOLIO---AUGUST-2026.xlsx
SBI-MULTI-ASSET-ALLOCATION-FUND-MONTHLY-PORTFOLIO---DECEMBER-2024.xlsx
SBI-MULTI-ASSET-ALLOCATION-FUND-MONTHLY-PORTFOLIO---DECEMBER-2025.xlsx
SBI-MULTI-ASSET-ALLOCATION-FUND-MONTHLY-PORTFOLIO---FEBRUARY-2025.xlsx
SBI-MULTI-ASSET-ALLOCATION-FUND-MONTHLY-PORTFOLIO---FEBRUARY-2026.xlsx
SBI-MULTI-ASSET-ALLOCATION-FUND-MONTHLY-PORTFOLIO---JANUARY-2025.xlsx
SBI-MULTI-ASSET-ALLOCATION-FUND-MONTHLY-PORTFOLIO---JANUARY-2026.

In [163]:
import re
import pandas as pd
from pathlib import Path

month_map = {
    "JANUARY": 1, "FEBRUARY": 2, "MARCH": 3,
    "APRIL": 4, "MAY": 5, "JUNE": 6,
    "JULY": 7, "AUGUST": 8, "SEPTEMBER": 9,
    "OCTOBER": 10, "NOVEMBER": 11, "DECEMBER": 12
}

sbi_months = []

for p in sbi_files:
    match = re.search(
        r"(JANUARY|FEBRUARY|MARCH|APRIL|MAY|JUNE|JULY|AUGUST|"
        r"SEPTEMBER|OCTOBER|NOVEMBER|DECEMBER)[-_ ]+(20\d{2})",
        p.stem.upper()
    )

    if match:
        month_name, year = match.groups()
        sbi_months.append(f"{year}-{month_map[month_name]:02d}")
    else:
        print("Could not identify month:", p.name)

expected_months = set(
    pd.period_range("2024-01", "2026-09", freq="M").astype(str)
)
actual_months = pd.Series(sbi_months)

print("Files:", len(sbi_files))
print("Unique months:", actual_months.nunique())
print("Missing months:", sorted(expected_months - set(sbi_months)))
print("Duplicate months:", actual_months[actual_months.duplicated()].tolist())

Could not identify month: SBI-MULTI-ASSET-ALLOCATION-FUND-feb-2024.xlsx
Could not identify month: SBI-MULTI-ASSET-ALLOCATION-FUND-jan-2024.xlsx
Files: 33
Unique months: 31
Missing months: ['2024-01', '2024-02']
Duplicate months: []


In [164]:
import re
import pandas as pd
from pathlib import Path

month_map = {
    "JANUARY": 1, "FEBRUARY": 2, "MARCH": 3,
    "APRIL": 4, "MAY": 5, "JUNE": 6,
    "JULY": 7, "AUGUST": 8, "SEPTEMBER": 9,
    "OCTOBER": 10, "NOVEMBER": 11, "DECEMBER": 12
}

sbi_months = []

for p in sbi_files:
    match = re.search(
        r"(JANUARY|FEBRUARY|MARCH|APRIL|MAY|JUNE|JULY|AUGUST|"
        r"SEPTEMBER|OCTOBER|NOVEMBER|DECEMBER)[-_ ]+(20\d{2})",
        p.stem,
        flags=re.IGNORECASE
    )

    if match:
        month_name, year = match.groups()
        sbi_months.append(
            f"{year}-{month_map[month_name.upper()]:02d}"
        )
    else:
        print("Could not identify month:", p.name)

expected_months = set(
    pd.period_range("2024-01", "2026-09", freq="M").astype(str)
)
actual_months = pd.Series(sbi_months)

print("Files:", len(sbi_files))
print("Unique months:", actual_months.nunique())
print("Missing months:", sorted(expected_months - set(sbi_months)))
print("Duplicate months:", actual_months[actual_months.duplicated()].tolist())

Could not identify month: SBI-MULTI-ASSET-ALLOCATION-FUND-feb-2024.xlsx
Could not identify month: SBI-MULTI-ASSET-ALLOCATION-FUND-jan-2024.xlsx
Files: 33
Unique months: 31
Missing months: ['2024-01', '2024-02']
Duplicate months: []


In [165]:

import re
from pathlib import Path
import pandas as pd

RAW_DIR = PROJECT_DIR / "01_RAW_DATA" / "SBI"

month_map = {
    "JAN": 1, "FEB": 2, "MAR": 3, "APR": 4,
    "MAY": 5, "JUN": 6, "JUL": 7, "AUG": 8,
    "SEP": 9, "SEPT": 9, "OCT": 10, "NOV": 11,
    "DEC": 12
}

month_pattern = (
    r"(JAN(?:UARY)?|FEB(?:RUARY)?|MAR(?:CH)?|APR(?:IL)?|"
    r"MAY|JUN(?:E)?|JUL(?:Y)?|AUG(?:UST)?|"
    r"SEPT?(?:EMBER)?|OCT(?:OBER)?|NOV(?:EMBER)?|DEC(?:EMBER)?)"
    r"[-_ ]+(20\d{2})"
)

records = []

for file in RAW_DIR.glob("*.xlsx"):
    match = re.search(month_pattern, file.stem, re.IGNORECASE)

    if match:
        month_name = match.group(1).upper()
        year = int(match.group(2))
        month_key = f"{year}-{month_map[month_name[:4] if month_name.startswith('SEPT') else month_name[:3]]:02d}"
        records.append({
            "File": file.name,
            "Report_Month": month_key
        })
    else:
        print("Could not identify month:", file.name)

coverage = pd.DataFrame(records)
expected_months = pd.period_range("2024-01", "2026-09", freq="M").astype(str)

print("Files:", len(list(RAW_DIR.glob("*.xlsx"))))
print("Unique months:", coverage["Report_Month"].nunique())
print("Missing months:", sorted(set(expected_months) - set(coverage["Report_Month"])))
print(
    "Duplicate months:",
    coverage.loc[
        coverage["Report_Month"].duplicated(keep=False), "Report_Month"
    ].tolist()
)


Files: 33
Unique months: 33
Missing months: []
Duplicate months: []


In [166]:

import pandas as pd
from pathlib import Path

RAW_DIR = PROJECT_DIR / "01_RAW_DATA" / "SBI"

sample_file = next(
    f for f in RAW_DIR.glob("*.xlsx")
    if "MARCH-2024" in f.name.upper()
)

xls = pd.ExcelFile(sample_file)

print("Sample file:", sample_file.name)
print("Sheets:", xls.sheet_names)

for sheet in xls.sheet_names:
    df = pd.read_excel(sample_file, sheet_name=sheet, header=None)
    print(f"\nSheet: {sheet} | Shape: {df.shape}")
    print(df.head(15).to_string(index=False, header=False))


Sample file: SBI-MULTI-ASSET-ALLOCATION-FUND---MARCH-2024.xlsx
Sheets: ['SMAAF']

Sheet: SMAAF | Shape: (171, 11)
NaN      NaN                                           NaN                             NaN                NaN      NaN                          NaN      NaN   NaN           NaN             NaN
NaN      NaN                               SBI Mutual Fund                             101                NaN      NaN                          NaN      NaN   NaN Back to Index             NaN
NaN      NaN                                 SCHEME NAME : SBI Multi Asset Allocation Fund                NaN      NaN                          NaN      NaN   NaN           NaN             NaN
NaN      NaN                   PORTFOLIO STATEMENT AS ON :             2024-03-31 00:00:00                NaN      NaN                          NaN      NaN   NaN           NaN             NaN
NaN      NaN                                           NaN                             NaN                NaN     

In [167]:

sbi_sample_df = pd.read_excel(sample_file, sheet_name="SMAAF", header=None)

print(sbi_sample_df.tail(35).to_string(index=True, header=False))


136 NaN          NaN                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            a) Mutual Fund Units / Exchange Traded Funds           NaN                                                                                                                                       NaN        NaN       NaN    NaN  NaN  NaN  NaN
137 NaN     417210.0                                                                                    

In [168]:

# Display the main portfolio table, including section headings and totals
print(sbi_sample_df.iloc[5:160, 2:8].to_string(index=True, header=False))


5                      Name of the Instrument / Issuer          ISIN                  Rating / Industry^   Quantity  Market value\n(Rs. in Lakhs)  % to AUM
6                                                  NaN           NaN                                 NaN        NaN                           NaN       NaN
7                              EQUITY & EQUITY RELATED           NaN                                 NaN        NaN                           NaN       NaN
8        a) Listed/awaiting listing on Stock Exchanges           NaN                                 NaN        NaN                           NaN       NaN
9                                    GAIL (India) Ltd.  INE129A01019                                 Gas    4350297                       7876.21      1.86
10                       Steel Authority of India Ltd.  INE114A01011                      Ferrous Metals    5630100                       7558.41      1.79
11                              Tata Technologies Ltd.  INE142M0

In [169]:

import pandas as pd

df = sbi_sample_df.copy()

# Relevant columns in the SBI report
holdings = df.iloc[:, [2, 3, 4, 5, 6, 7]].copy()
holdings.columns = [
    "Instrument_Name", "ISIN", "Industry_or_Rating",
    "Quantity", "Market_Value_Lakh", "NAV_Pct"
]

holdings["Instrument_Name"] = holdings["Instrument_Name"].astype("string").str.strip()
holdings["Market_Value_Lakh"] = pd.to_numeric(
    holdings["Market_Value_Lakh"], errors="coerce"
)
holdings["NAV_Pct"] = pd.to_numeric(holdings["NAV_Pct"], errors="coerce")

# Keep rows with an instrument name and numeric market value
holdings = holdings[
    holdings["Instrument_Name"].notna()
    & holdings["Market_Value_Lakh"].notna()
].copy()

# Remove subtotal/total rows
holdings = holdings[
    ~holdings["Instrument_Name"].str.fullmatch(
        r"total|grand total \(aum\)", case=False, na=False
    )
]

# Remove section headings and NIL entries
holdings = holdings[
    ~holdings["Instrument_Name"].str.contains(
        r"^EQUITY|^DEBT INSTRUMENTS|^MONEY MARKET INSTRUMENTS|"
        r"^OTHERS$|^Other Current Assets|^a\)|^b\)|^c\)|^d\)|^e\)|^f\)|^GRAND TOTAL",
        case=False, na=False
    )
]

grand_total = pd.to_numeric(df.iloc[:, 6], errors="coerce")
grand_total = grand_total[
    df.iloc[:, 2].astype("string").str.contains(
        "GRAND TOTAL", case=False, na=False
    )
].iloc[0]

print("Extracted rows:", len(holdings))
print("Extracted value (₹ lakh):", round(holdings["Market_Value_Lakh"].sum(), 2))
print("Reported Grand Total (₹ lakh):", round(grand_total, 2))
print("Difference (₹ lakh):", round(holdings["Market_Value_Lakh"].sum() - grand_total, 2))

print("\nGold/Silver/TREPS check:")
print(
    holdings[
        holdings["Instrument_Name"].str.contains(
            "Gold ETF|Silver ETF|TREPS", case=False, na=False
        )
    ][["Instrument_Name", "Market_Value_Lakh", "NAV_Pct"]].to_string(index=False)
)


Extracted rows: 88
Extracted value (₹ lakh): 422978.7
Reported Grand Total (₹ lakh): 422978.7
Difference (₹ lakh): -0.0

Gold/Silver/TREPS check:
        Instrument_Name  Market_Value_Lakh  NAV_Pct
           SBI Gold ETF           60272.10    14.25
Nippon India Silver ETF            4345.20     1.03
                  TREPS           22725.36     5.37


In [170]:

import re
import pandas as pd
from pathlib import Path

RAW_DIR = PROJECT_DIR / "01_RAW_DATA" / "SBI"
OUTPUT_DIR = PROJECT_DIR / "02_CLEAN_DATA"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

month_map = {
    "JAN": 1, "FEB": 2, "MAR": 3, "APR": 4,
    "MAY": 5, "JUN": 6, "JUL": 7, "AUG": 8,
    "SEP": 9, "SEPT": 9, "OCT": 10, "NOV": 11, "DEC": 12
}
month_pattern = (
    r"(JAN(?:UARY)?|FEB(?:RUARY)?|MAR(?:CH)?|APR(?:IL)?|MAY|"
    r"JUN(?:E)?|JUL(?:Y)?|AUG(?:UST)?|SEPT?(?:EMBER)?|"
    r"OCT(?:OBER)?|NOV(?:EMBER)?|DEC(?:EMBER)?)[-_ ]+(20\d{2})"
)

all_holdings = []
validation = []

for file in sorted(RAW_DIR.glob("*.xlsx")):
    match = re.search(month_pattern, file.stem, re.IGNORECASE)
    if not match:
        validation.append({
            "Source_File": file.name,
            "Report_Month": None,
            "Extracted_Total_Lakh": None,
            "Reported_Total_Lakh": None,
            "Difference_Lakh": None,
            "Status": "REVIEW - month not identified"
        })
        continue

    month_name = match.group(1).upper()
    month_number = month_map[
        month_name[:4] if month_name.startswith("SEPT") else month_name[:3]
    ]
    report_month = f"{match.group(2)}-{month_number:02d}"

    try:
        xls = pd.ExcelFile(file)
        sheet = next(
            (s for s in xls.sheet_names
             if "SMAAF" in s.upper()),
            xls.sheet_names[0]
        )
        df = pd.read_excel(file, sheet_name=sheet, header=None)

        # Locate the reported Grand Total (AUM)
        grand_mask = df.iloc[:, 2].astype("string").str.contains(
            r"GRAND TOTAL\s*\(AUM\)", case=False, na=False, regex=True
        )
        if not grand_mask.any():
            validation.append({
                "Source_File": file.name,
                "Report_Month": report_month,
                "Extracted_Total_Lakh": None,
                "Reported_Total_Lakh": None,
                "Difference_Lakh": None,
                "Status": "REVIEW - Grand Total not found"
            })
            continue

        reported_total = pd.to_numeric(
            df.loc[grand_mask, 6], errors="coerce"
        ).dropna()
        if reported_total.empty:
            validation.append({
                "Source_File": file.name,
                "Report_Month": report_month,
                "Extracted_Total_Lakh": None,
                "Reported_Total_Lakh": None,
                "Difference_Lakh": None,
                "Status": "REVIEW - Grand Total not numeric"
            })
            continue
        reported_total = float(reported_total.iloc[0])

        h = df.iloc[:, [2, 3, 4, 5, 6, 7]].copy()
        h.columns = [
            "Instrument_Name", "ISIN", "Industry_or_Rating",
            "Quantity", "Market_Value_Lakh", "NAV_Pct"
        ]
        h["Instrument_Name"] = h["Instrument_Name"].astype("string").str.strip()
        h["Market_Value_Lakh"] = pd.to_numeric(
            h["Market_Value_Lakh"], errors="coerce"
        )
        h["NAV_Pct"] = pd.to_numeric(h["NAV_Pct"], errors="coerce")

        h = h[
            h["Instrument_Name"].notna()
            & h["Market_Value_Lakh"].notna()
        ].copy()

        # Remove total rows and section headings, not valid instruments
        h = h[
            ~h["Instrument_Name"].str.fullmatch(
                r"total|grand total\s*\(aum\)", case=False, na=False
            )
        ]
        h = h[
            ~h["Instrument_Name"].str.contains(
                r"^EQUITY|^DEBT INSTRUMENTS|^MONEY MARKET INSTRUMENTS|"
                r"^OTHERS$|^Other Current Assets|^a\)|^b\)|^c\)|"
                r"^d\)|^e\)|^f\)|^GRAND TOTAL",
                case=False, na=False
            )
        ]

        extracted_total = float(h["Market_Value_Lakh"].sum())
        difference = extracted_total - reported_total
        status = "OK" if abs(difference) <= 0.02 else "REVIEW"

        h["AMC"] = "SBI"
        h["Report_Month"] = report_month
        h["Source_File"] = file.name
        h["Source_Row"] = h.index + 1
        all_holdings.append(h)

        validation.append({
            "Source_File": file.name,
            "Report_Month": report_month,
            "Extracted_Total_Lakh": round(extracted_total, 2),
            "Reported_Total_Lakh": round(reported_total, 2),
            "Difference_Lakh": round(difference, 2),
            "Status": status
        })

    except Exception as e:
        validation.append({
            "Source_File": file.name,
            "Report_Month": report_month,
            "Extracted_Total_Lakh": None,
            "Reported_Total_Lakh": None,
            "Difference_Lakh": None,
            "Status": f"REVIEW - {type(e).__name__}: {e}"
        })

if all_holdings:
    holdings_log = pd.concat(all_holdings, ignore_index=True)
    columns = [
        "AMC", "Report_Month", "ISIN", "Instrument_Name",
        "Industry_or_Rating", "Quantity", "Market_Value_Lakh",
        "NAV_Pct", "Source_File", "Source_Row"
    ]
    holdings_log = holdings_log[columns]
    holdings_log.to_csv(
        OUTPUT_DIR / "SBI_Monthly_Holdings_Log.csv", index=False
    )

validation_df = pd.DataFrame(validation).sort_values(
    "Report_Month", na_position="last"
)
validation_df.to_csv(
    OUTPUT_DIR / "SBI_Extraction_Validation.csv", index=False
)

print("Extracted rows:", len(holdings_log) if all_holdings else 0)
print("Months processed:", validation_df["Report_Month"].nunique())
print("\nValidation status:")
print(validation_df["Status"].value_counts(dropna=False))
print("\nMonths requiring review:")
print(validation_df[validation_df["Status"] != "OK"].to_string(index=False))
print("\nSaved:")
print(OUTPUT_DIR / "SBI_Monthly_Holdings_Log.csv")
print(OUTPUT_DIR / "SBI_Extraction_Validation.csv")


Extracted rows: 3453
Months processed: 33

Validation status:
Status
OK        22
REVIEW    11
Name: count, dtype: int64

Months requiring review:
                                                            Source_File Report_Month  Extracted_Total_Lakh  Reported_Total_Lakh  Difference_Lakh Status
  SBI-MULTI-ASSET-ALLOCATION-FUND-MONTHLY-PORTFOLIO---OCTOBER-2025.xlsx      2025-10            1142495.96           1130622.96         11873.00 REVIEW
 SBI-MULTI-ASSET-ALLOCATION-FUND-MONTHLY-PORTFOLIO---NOVEMBER-2025.xlsx      2025-11            1214068.35           1201168.35         12900.00 REVIEW
 SBI-MULTI-ASSET-ALLOCATION-FUND-MONTHLY-PORTFOLIO---DECEMBER-2025.xlsx      2025-12            1307698.98           1303289.75          4409.23 REVIEW
  SBI-MULTI-ASSET-ALLOCATION-FUND-MONTHLY-PORTFOLIO---JANUARY-2026.xlsx      2026-01            1520534.06           1494380.18         26153.88 REVIEW
    SBI-MULTI-ASSET-ALLOCATION-FUND-MONTHLY-PORTFOLIO---MARCH-2026.xlsx      2026-03         

In [171]:

from pathlib import Path
import pandas as pd

RAW_DIR = PROJECT_DIR / "01_RAW_DATA" / "SBI"

sep_file = next(
    f for f in RAW_DIR.glob("*.xlsx")
    if "SEPTEMBER-2026" in f.name.upper()
)

xls = pd.ExcelFile(sep_file)
print("File:", sep_file.name)
print("Sheets:", xls.sheet_names)

sep_df = pd.read_excel(sep_file, sheet_name=xls.sheet_names[0], header=None)

print("\nShape:", sep_df.shape)

grand_mask = sep_df.astype("string").apply(
    lambda col: col.str.contains(
        r"GRAND TOTAL", case=False, na=False, regex=True
    )
).any(axis=1)

print("\nGrand Total rows:")
print(sep_df.loc[grand_mask].to_string(index=True, header=False))

print("\nLast 40 rows:")
print(sep_df.tail(40).to_string(index=True, header=False))

print("\nLargest values in column G (index 6):")
print(
    sep_df.assign(
        _value=pd.to_numeric(sep_df.iloc[:, 6], errors="coerce")
    ).nlargest(15, "_value").drop(columns="_value").to_string(index=True, header=False)
)


File: SBI-MULTI-ASSET-ALLOCATION-FUND-MONTHLY-PORTFOLIO---SEPTEMBER-2026.xlsx
Sheets: ['SMAAF']

Shape: (327, 11)

Grand Total rows:
200 NaN NaN  GRAND TOTAL (AUM)  NaN  NaN  NaN  2104418.61  100.0  NaN  NaN  NaN

Last 40 rows:
287 NaN NaN                                                                                                                                                                                             NaN       NaN                                                                                                                                   NaN                                         NaN                                   NaN  NaN  NaN  NaN  NaN
288 NaN NaN                     For the period 01-September-2026 to 30-September-2026, the following details specified for hedging transactions through options which have already been exercised/expired :       NaN                                                                                                             

In [173]:

import re
import pandas as pd
from pathlib import Path

# --------------------------------------------------
# 1. Folders
# --------------------------------------------------
RAW_DIR = PROJECT_DIR / "01_RAW_DATA" / "SBI"
OUTPUT_DIR = PROJECT_DIR / "02_CLEAN_DATA"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

HOLDINGS_PATH = OUTPUT_DIR / "SBI_Monthly_Holdings_Log.csv"
VALIDATION_PATH = OUTPUT_DIR / "SBI_Extraction_Validation.csv"

# --------------------------------------------------
# 2. Identify report month from filename
# --------------------------------------------------
month_map = {
    "JAN": 1, "FEB": 2, "MAR": 3, "APR": 4,
    "MAY": 5, "JUN": 6, "JUL": 7, "AUG": 8,
    "SEP": 9, "SEPT": 9, "OCT": 10, "NOV": 11,
    "DEC": 12
}

month_pattern = (
    r"(JAN(?:UARY)?|FEB(?:RUARY)?|MAR(?:CH)?|APR(?:IL)?|MAY|"
    r"JUN(?:E)?|JUL(?:Y)?|AUG(?:UST)?|SEPT?(?:EMBER)?|"
    r"OCT(?:OBER)?|NOV(?:EMBER)?|DEC(?:EMBER)?)[-_ ]+(20\d{2})"
)

all_holdings = []
validation = []

# --------------------------------------------------
# 3. Process each monthly report
# --------------------------------------------------
for file in sorted(RAW_DIR.glob("*.xlsx")):

    # Identify month
    match = re.search(month_pattern, file.stem, re.IGNORECASE)

    if not match:
        validation.append({
            "Source_File": file.name,
            "Report_Month": None,
            "Extracted_Rows": None,
            "Extracted_Total_Lakh": None,
            "Reported_Total_Lakh": None,
            "Difference_Lakh": None,
            "Status": "REVIEW - month not identified"
        })
        continue

    month_name = match.group(1).upper()
    year = int(match.group(2))

    month_key = (
        month_name[:4]
        if month_name.startswith("SEPT")
        else month_name[:3]
    )
    report_month = f"{year}-{month_map[month_key]:02d}"

    try:
        # Read workbook and select the portfolio sheet
        xls = pd.ExcelFile(file)

        sheet = next(
            (s for s in xls.sheet_names if "SMAAF" in s.upper()),
            xls.sheet_names[0]
        )

        df = pd.read_excel(file, sheet_name=sheet, header=None)

        # --------------------------------------------------
        # 4. Find Grand Total (AUM)
        # --------------------------------------------------
        grand_mask = df.iloc[:, 2].astype("string").str.contains(
            r"GRAND TOTAL\s*\(AUM\)",
            case=False,
            na=False,
            regex=True
        )

        if not grand_mask.any():
            validation.append({
                "Source_File": file.name,
                "Report_Month": report_month,
                "Extracted_Rows": None,
                "Extracted_Total_Lakh": None,
                "Reported_Total_Lakh": None,
                "Difference_Lakh": None,
                "Status": "REVIEW - Grand Total not found"
            })
            continue

        grand_row = df.index[grand_mask][0]

        reported_values = pd.to_numeric(
            df.loc[grand_mask, 6], errors="coerce"
        ).dropna()

        if reported_values.empty:
            validation.append({
                "Source_File": file.name,
                "Report_Month": report_month,
                "Extracted_Rows": None,
                "Extracted_Total_Lakh": None,
                "Reported_Total_Lakh": None,
                "Difference_Lakh": None,
                "Status": "REVIEW - Grand Total not numeric"
            })
            continue

        reported_total = float(reported_values.iloc[0])

        # --------------------------------------------------
        # 5. Extract only rows before Grand Total
        # --------------------------------------------------
        main_df = df.iloc[:grand_row].copy()

        h = main_df.iloc[:, [2, 3, 4, 5, 6, 7]].copy()

        h.columns = [
            "Instrument_Name",
            "ISIN",
            "Industry_or_Rating",
            "Quantity",
            "Market_Value_Lakh",
            "NAV_Pct"
        ]

        h["Instrument_Name"] = (
            h["Instrument_Name"].astype("string").str.strip()
        )

        h["Market_Value_Lakh"] = pd.to_numeric(
            h["Market_Value_Lakh"], errors="coerce"
        )

        h["NAV_Pct"] = pd.to_numeric(
            h["NAV_Pct"], errors="coerce"
        )

        # Keep named rows with numeric market value
        h = h[
            h["Instrument_Name"].notna()
            & h["Instrument_Name"].ne("")
            & h["Market_Value_Lakh"].notna()
        ].copy()

        # Remove section headings and subtotal rows
        exclude_pattern = (
            r"^EQUITY|^DEBT INSTRUMENTS|^MONEY MARKET INSTRUMENTS|"
            r"^OTHERS$|^OTHER CURRENT ASSETS|"
            r"^[a-h]\)|^TOTAL$|^GRAND TOTAL"
        )

        h = h[
            ~h["Instrument_Name"].str.contains(
                exclude_pattern, case=False, na=False, regex=True
            )
        ].copy()

        # Record source information
        h["AMC"] = "SBI"
        h["Report_Month"] = report_month
        h["Source_File"] = file.name
        h["Source_Row"] = h.index + 1

        # --------------------------------------------------
        # 6. Reconcile extracted total with reported AUM
        # --------------------------------------------------
        extracted_total = float(h["Market_Value_Lakh"].sum())
        difference = extracted_total - reported_total

        status = (
            "OK"
            if abs(difference) <= 0.02
            else "REVIEW"
        )

        validation.append({
            "Source_File": file.name,
            "Report_Month": report_month,
            "Extracted_Rows": len(h),
            "Extracted_Total_Lakh": round(extracted_total, 2),
            "Reported_Total_Lakh": round(reported_total, 2),
            "Difference_Lakh": round(difference, 2),
            "Status": status
        })

        all_holdings.append(h)

    except Exception as e:
        validation.append({
            "Source_File": file.name,
            "Report_Month": report_month,
            "Extracted_Rows": None,
            "Extracted_Total_Lakh": None,
            "Reported_Total_Lakh": None,
            "Difference_Lakh": None,
            "Status": f"REVIEW - {type(e).__name__}: {e}"
        })

# --------------------------------------------------
# 7. Save holdings log
# --------------------------------------------------
columns = [
    "AMC",
    "Report_Month",
    "ISIN",
    "Instrument_Name",
    "Industry_or_Rating",
    "Quantity",
    "Market_Value_Lakh",
    "NAV_Pct",
    "Source_File",
    "Source_Row"
]

if all_holdings:
    holdings_log = pd.concat(all_holdings, ignore_index=True)
    holdings_log = holdings_log[columns]

    holdings_log.to_csv(HOLDINGS_PATH, index=False)
else:
    holdings_log = pd.DataFrame(columns=columns)

# --------------------------------------------------
# 8. Save validation report
# --------------------------------------------------
validation_df = pd.DataFrame(validation)

if not validation_df.empty:
    validation_df = validation_df.sort_values(
        "Report_Month", na_position="last"
    )

validation_df.to_csv(VALIDATION_PATH, index=False)

# --------------------------------------------------
# 9. Display results
# --------------------------------------------------
print("SBI EXTRACTION SUMMARY")
print("-" * 40)
print("Excel files found:", len(list(RAW_DIR.glob("*.xlsx"))))
print("Months processed:", validation_df["Report_Month"].nunique())
print("Extracted holdings rows:", len(holdings_log))

print("\nValidation status:")
print(validation_df["Status"].value_counts(dropna=False))

print("\nMonths requiring review:")
print(
    validation_df[
        validation_df["Status"] != "OK"
    ].to_string(index=False)
)

print("\nGold, Silver and TREPS entries:")
if not holdings_log.empty:
    metal_check = holdings_log[
        holdings_log["Instrument_Name"].str.contains(
            r"gold|silver|treps",
            case=False,
            na=False
        )
    ]

    print(
        metal_check[
            ["Report_Month", "Instrument_Name",
             "Market_Value_Lakh", "NAV_Pct"]
        ].to_string(index=False)
    )

print("\nSaved files:")
print(HOLDINGS_PATH)
print(VALIDATION_PATH)


SBI EXTRACTION SUMMARY
----------------------------------------
Excel files found: 33
Months processed: 33
Extracted holdings rows: 3396

Validation status:
Status
OK    33
Name: count, dtype: int64

Months requiring review:
Empty DataFrame
Columns: [Source_File, Report_Month, Extracted_Rows, Extracted_Total_Lakh, Reported_Total_Lakh, Difference_Lakh, Status]
Index: []

Gold, Silver and TREPS entries:
Report_Month         Instrument_Name  Market_Value_Lakh  NAV_Pct
     2024-03            SBI Gold ETF           60272.10    14.25
     2024-03 Nippon India Silver ETF            4345.20     1.03
     2024-03                   TREPS           22725.36     5.37
     2024-02            SBI Gold ETF           41331.94    10.65
     2024-02                   TREPS           20256.40     5.22
     2024-01            SBI Gold ETF           41819.72    11.53
     2024-01                   TREPS           33359.94     9.20
     2024-04            SBI Gold ETF           64856.00    13.87
     2024-

In [174]:

import pandas as pd
from IPython.display import display

# Load the cleaned SBI holdings dataset
sbi_df = pd.read_csv(
    PROJECT_DIR / "02_CLEAN_DATA" / "SBI_Monthly_Holdings_Log.csv"
)

# Display the first 10 rows
display(sbi_df.head(10))


,AMC,Report_Month,ISIN,Instrument_Name,Industry_or_Rating,Quantity,Market_Value_Lakh,NAV_Pct,Source_File,Source_Row
0,SBI,2024-03,INE129A01019,GAIL (India) Ltd.,Gas,4350297.0,7876.21,1.86,SBI-MULTI-ASSET-ALLOCATION-FUND---MARCH-2024.xlsx,10
1,SBI,2024-03,INE114A01011,Steel Authority of India Ltd.,Ferrous Metals,5630100.0,7558.41,1.79,SBI-MULTI-ASSET-ALLOCATION-FUND---MARCH-2024.xlsx,11
2,SBI,2024-03,INE142M01025,Tata Technologies Ltd.,IT - Services,711574.0,7262.32,1.72,SBI-MULTI-ASSET-ALLOCATION-FUND---MARCH-2024.xlsx,12
3,SBI,2024-03,INE075A01022,Wipro Ltd.,IT - Software,1500000.0,7201.50,1.70,SBI-MULTI-ASSET-ALLOCATION-FUND---MARCH-2024.xlsx,13
4,SBI,2024-03,INE062A01020,State Bank of India,Banks,950000.0,7147.33,1.69,SBI-MULTI-ASSET-ALLOCATION-FUND---MARCH-2024.xlsx,14
5,SBI,2024-03,INE726G01019,ICICI Prudential Life Insurance Company Ltd.,Insurance,1012032.0,6159.73,1.46,SBI-MULTI-ASSET-ALLOCATION-FUND---MARCH-2024.xlsx,15
6,SBI,2024-03,INE376G01013,Biocon Ltd.,Pharmaceuticals & Biotechnology,2099318.0,5545.35,1.31,SBI-MULTI-ASSET-ALLOCATION-FUND---MARCH-2024.xlsx,16
7,SBI,2024-03,INE388Y01029,FSN E-Commerce Ventures Ltd.,Retailing,3300000.0,5344.35,1.26,SBI-MULTI-ASSET-ALLOCATION-FUND---MARCH-2024.xlsx,17
8,SBI,2024-03,INE040A01034,HDFC Bank Ltd.,Banks,361000.0,5226.92,1.24,SBI-MULTI-ASSET-ALLOCATION-FUND---MARCH-2024.xlsx,18
9,SBI,2024-03,INE022Q01020,Indian Energy Exchange Ltd.,Capital Markets,3500000.0,4702.25,1.11,SBI-MULTI-ASSET-ALLOCATION-FUND---MARCH-2024.xlsx,19


In [175]:

# Dataset dimensions
print("Number of rows:", sbi_df.shape[0])
print("Number of columns:", sbi_df.shape[1])

# Column names and data types
sbi_df.info()

# Summary statistics
display(sbi_df.describe(include="all").T)


Number of rows: 3396
Number of columns: 10
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3396 entries, 0 to 3395
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   AMC                 3396 non-null   object 
 1   Report_Month        3396 non-null   object 
 2   ISIN                3327 non-null   object 
 3   Instrument_Name     3396 non-null   object 
 4   Industry_or_Rating  3327 non-null   object 
 5   Quantity            3327 non-null   float64
 6   Market_Value_Lakh   3396 non-null   float64
 7   NAV_Pct             3377 non-null   float64
 8   Source_File         3396 non-null   object 
 9   Source_Row          3396 non-null   int64  
dtypes: float64(3), int64(1), object(6)
memory usage: 265.4+ KB


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
AMC,3396,1,SBI,3396,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Report_Month,3396,33,2026-09,133,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ISIN,3327,265,INF200KA16D8,33,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Instrument_Name,3396,206,Muthoot Finance Ltd.,88,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Industry_or_Rating,3327,60,Banks,278,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Quantity,3327.0,NaN,NaN,NaN,3344378.856628,9982969.002181,45.0,12500.0,461361.0,2587138.5,107591000.0
Market_Value_Lakh,3396.0,NaN,NaN,NaN,10105.833263,13664.95957,-50213.46,3215.0875,6316.335,11960.155,187750.05
NAV_Pct,3377.0,NaN,NaN,NaN,0.977199,1.20279,-3.06,0.38,0.68,1.18,14.25
Source_File,3396,33,SBI-MULTI-ASSET-ALLOCATION-FUND-MONTHLY-PORTFO...,133,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Source_Row,3396.0,NaN,NaN,NaN,70.922261,41.628505,10.0,35.0,63.0,102.0,198.0


In [176]:

display(
    sbi_df[
        [
            "Report_Month",
            "ISIN",
            "Instrument_Name",
            "Industry_or_Rating",
            "Market_Value_Lakh",
            "NAV_Pct"
        ]
    ].head(20)
)


,Report_Month,ISIN,Instrument_Name,Industry_or_Rating,Market_Value_Lakh,NAV_Pct
0,2024-03,INE129A01019,GAIL (India) Ltd.,Gas,7876.21,1.86
1,2024-03,INE114A01011,Steel Authority of India Ltd.,Ferrous Metals,7558.41,1.79
2,2024-03,INE142M01025,Tata Technologies Ltd.,IT - Services,7262.32,1.72
3,2024-03,INE075A01022,Wipro Ltd.,IT - Software,7201.50,1.70
4,2024-03,INE062A01020,State Bank of India,Banks,7147.33,1.69
5,2024-03,INE726G01019,ICICI Prudential Life Insurance Company Ltd.,Insurance,6159.73,1.46
6,2024-03,INE376G01013,Biocon Ltd.,Pharmaceuticals & Biotechnology,5545.35,1.31
7,2024-03,INE388Y01029,FSN E-Commerce Ventures Ltd.,Retailing,5344.35,1.26
8,2024-03,INE040A01034,HDFC Bank Ltd.,Banks,5226.92,1.24
9,2024-03,INE022Q01020,Indian Energy Exchange Ltd.,Capital Markets,4702.25,1.11


In [177]:

gold_silver = sbi_df[
    sbi_df["Instrument_Name"].str.contains(
        r"gold|silver",
        case=False,
        na=False
    )
]

display(gold_silver)


,AMC,Report_Month,ISIN,Instrument_Name,Industry_or_Rating,Quantity,Market_Value_Lakh,NAV_Pct,Source_File,Source_Row
84,SBI,2024-03,INF200KA16D8,SBI Gold ETF,Mutual Fund,103241000.0,60272.10,14.25,SBI-MULTI-ASSET-ALLOCATION-FUND---MARCH-2024.xlsx,138
85,SBI,2024-03,INF204KC1402,Nippon India Silver ETF,Mutual Fund,6000000.0,4345.20,1.03,SBI-MULTI-ASSET-ALLOCATION-FUND---MARCH-2024.xlsx,139
173,SBI,2024-02,INF200KA16D8,SBI Gold ETF,Mutual Fund,76216000.0,41331.94,10.65,SBI-MULTI-ASSET-ALLOCATION-FUND-feb-2024.xlsx,139
256,SBI,2024-01,INF200KA16D8,SBI Gold ETF,Mutual Fund,76216000.0,41819.72,11.53,SBI-MULTI-ASSET-ALLOCATION-FUND-jan-2024.xlsx,135
343,SBI,2024-04,INF200KA16D8,SBI Gold ETF,Mutual Fund,103241000.0,64856.00,13.87,SBI-MULTI-ASSET-ALLOCATION-FUND-MONTHLY-PORTFO...,138
...,...,...,...,...,...,...,...,...,...,...
3258,SBI,2025-09,INF200KB1217,SBI Silver ETF,Mutual Fund/ETF,51296178.0,72301.96,6.90,SBI-MULTI-ASSET-ALLOCATION-FUND-MONTHLY-PORTFO...,149
3259,SBI,2025-09,INF200KA16D8,SBI Gold ETF,Mutual Fund/ETF,37241000.0,36920.73,3.52,SBI-MULTI-ASSET-ALLOCATION-FUND-MONTHLY-PORTFO...,150
3260,SBI,2025-09,INF204KC1402,Nippon India Silver ETF,Mutual Fund/ETF,17775074.0,24186.54,2.31,SBI-MULTI-ASSET-ALLOCATION-FUND-MONTHLY-PORTFO...,151
3391,SBI,2026-09,INF200KA16D8,SBI Gold ETF,Mutual Fund/ETF,107591000.0,135145.06,6.42,SBI-MULTI-ASSET-ALLOCATION-FUND-MONTHLY-PORTFO...,182


In [178]:

from pathlib import Path

KOTAK_RAW_DIR = PROJECT_DIR / "01_RAW_DATA" / "Kotak"

kotak_files = sorted(KOTAK_RAW_DIR.glob("*.xlsx"))

print("Kotak Excel reports found:", len(kotak_files))

for file in kotak_files:
    print(file.name)


Kotak Excel reports found: 10
ConsolidatedSEBIPortfolioApril2026.xlsx
ConsolidatedSEBIPortfolioAugust2026.xlsx
ConsolidatedSEBIPortfolioDecember2025.xlsx
ConsolidatedSEBIPortfolioFebruary2026.xlsx
ConsolidatedSEBIPortfolioJanuary2026.xlsx
ConsolidatedSEBIPortfolioJuly2026.xlsx
ConsolidatedSEBIPortfolioJune2026.xlsx
ConsolidatedSEBIPortfolioMarch2026.xlsx
ConsolidatedSEBIPortfolioMay2026.xlsx
ConsolidatedSEBIPortfolioSeptember2026.xlsx


In [179]:

import pandas as pd

expected_months = pd.period_range(
    start="2024-01",
    end="2026-09",
    freq="M"
)

available_months = []

for file in kotak_files:
    match = pd.Series([file.stem]).str.extract(
        r"(?i)(January|February|March|April|May|June|July|August|September|October|November|December)(\d{4})"
    )

    if match.notna().all(axis=None):
        month_name = match.iloc[0, 0].capitalize()
        year = int(match.iloc[0, 1])

        month_number = pd.to_datetime(
            month_name, format="%B"
        ).month

        available_months.append(
            pd.Period(f"{year}-{month_number:02d}", freq="M")
        )

available_months = set(available_months)
missing_months = [
    str(month)
    for month in expected_months
    if month not in available_months
]

print("Available months:", len(available_months))
print("Missing months:", len(missing_months))
print("\nMissing months:")
print(missing_months)


Available months: 10
Missing months: 23

Missing months:
['2024-01', '2024-02', '2024-03', '2024-04', '2024-05', '2024-06', '2024-07', '2024-08', '2024-09', '2024-10', '2024-11', '2024-12', '2025-01', '2025-02', '2025-03', '2025-04', '2025-05', '2025-06', '2025-07', '2025-08', '2025-09', '2025-10', '2025-11']


In [180]:

import pandas as pd

sample_file = KOTAK_RAW_DIR / "ConsolidatedSEBIPortfolioSeptember2026.xlsx"

xls = pd.ExcelFile(sample_file)

print("Sheet names:")
print(xls.sheet_names)

for sheet in xls.sheet_names[:5]:
    preview = pd.read_excel(
        sample_file,
        sheet_name=sheet,
        header=None,
        nrows=12
    )

    print(f"\n--- Sheet: {sheet} ---")
    display(preview)


Sheet names:
['V3I', 'TIF', 'TCH', 'TAL', 'STF', 'SRF', 'SPO', 'SOF', 'SIL', 'SIF', 'SEF', 'ROF', 'QOF', 'Q3T', 'Q3I', 'PBT', 'OVR', 'NVF', 'NTF', 'NSI', 'NNF', 'NNE', 'NMX', 'NMT', 'NLV', 'NIF', 'NFS', 'NEF', 'NCI', 'NBI', 'NAS', 'N10', 'N5E', 'N1E', 'MUC', 'MTF', 'MSO', 'MSC', 'MNT', 'MMI', 'MIF', 'MID', 'MFO', 'MFF', 'MDF', 'MCF', 'MAF', 'M5I', 'M3E', 'LVI', 'LTF', 'LIQ', 'LDT', 'KWD', 'KSF', 'KRE', 'KQT', 'KPF', 'KOP', 'KMN', 'KMI', 'KLD', 'KIP', 'KIE', 'KGS', 'KGI', 'KFL', 'KFE', 'KEO', 'KCB', 'KBC', 'KBA', 'KAM', 'K30', 'ITF', 'HLC', 'GTF', 'GSF', 'GOF', 'GIF', 'GEM', 'FOF', 'V04', 'FLX', 'FLT', 'FLR', 'ESG', 'EME', 'ELS', 'DYF', 'DOO', 'CTF', 'CRO', 'CPL', 'CON', 'CMP', 'CHE', 'BTF', 'BST', 'BSI', 'BON', 'BIF', 'BHI', 'BFS', 'BEF', 'BAL', 'B14', 'B13', 'B12', 'B11', 'B10', 'B08', 'B06', 'B03', 'B02', 'B01', 'ALF', 'AL3', 'A50', 'Common Notes', 'Scheme']

--- Sheet: V3I ---


,0,1,2,3,4,5,6,7,8
0,NaN,NaN,Portfolio of Kotak Nifty200 Value 30 Index Fun...,NaN,NaN,NaN,NaN,NaN,NaN
1,Name of Instrument,NaN,NaN,ISIN Code,Industry,Yield,Quantity,Market Value (Rs.in Lacs),% to Net Assets
2,Equity & Equity related,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,Listed/Awaiting listing on Stock Exchange,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,,Coal India Limited,INE522F01014,Consumable Fuels,NaN,15970,67.87,5.24
5,NaN,,BHARAT PETROLEUM CORPORATION LTD.,INE029A01011,Petroleum Products,NaN,22347,67.76,5.23
6,NaN,,Oil And Natural Gas Corporation Ltd.,INE213A01029,Oil,NaN,29395,66.37,5.13
7,NaN,,STATE BANK OF INDIA.,INE062A01020,Banks,NaN,6819,65.43,5.06
8,NaN,,ITC Ltd.,INE154A01025,Diversified FMCG,NaN,24298,63.84,4.93
9,NaN,,NTPC LTD,INE733E01010,Power,NaN,19759,63.77,4.93



--- Sheet: TIF ---


,0,1,2,3,4,5,6,7,8
0,NaN,NaN,Portfolio of Kotak Nifty India Tourism Index F...,NaN,NaN,NaN,NaN,NaN,NaN
1,Name of Instrument,NaN,NaN,ISIN Code,Industry,Yield,Quantity,Market Value (Rs.in Lacs),% to Net Assets
2,Equity & Equity related,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,Listed/Awaiting listing on Stock Exchange,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,,Inter Globe Aviation Ltd,INE646L01027,Transport Services,NaN,14990,747.1,20.46
5,NaN,,INDIAN HOTELS CO LTD,INE053A01029,Leisure Services,NaN,101882,742.36,20.33
6,NaN,,GMR AIRPORTS LIMITED,INE776C01039,Transport Infrastructure,NaN,698153,664.29,18.19
7,NaN,,ITC HOTELS LIMITED,INE379A01028,Leisure Services,NaN,225387,369.93,10.13
8,NaN,,Jubilant Foodworks Limited,INE797F01020,Leisure Services,NaN,78994,360.77,9.88
9,NaN,,Indian Railway Catering And Tourism Corporatio...,INE335Y01020,Leisure Services,NaN,60679,276.33,7.57



--- Sheet: TCH ---


,0,1,2,3,4,5,6,7,8
0,NaN,NaN,Portfolio of Kotak Technology Fund as on 30-Se...,NaN,NaN,NaN,NaN,NaN,NaN
1,Name of Instrument,NaN,NaN,ISIN Code,Industry,Yield,Quantity,Market Value (Rs.in Lacs),% to Net Assets
2,Equity & Equity related,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,Listed/Awaiting listing on Stock Exchange,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,,Infosys Ltd.,INE009A01021,IT - Software,NaN,775503,7709.28,16.1
5,NaN,,BHARTI AIRTEL LTD.,INE397D01024,Telecom - Services,NaN,307577,5404.13,11.29
6,NaN,,Tech Mahindra Ltd.,INE669C01036,IT - Software,NaN,249579,3803.58,7.94
7,NaN,,ETERNAL LIMITED,INE758T01015,Retailing,NaN,1093016,3497.65,7.3
8,NaN,,Persistent Systems Limited,INE262H01021,IT - Software,NaN,41776,2205.36,4.61
9,NaN,,Coforge Limited,INE591G01025,IT - Software,NaN,124800,2182.88,4.56



--- Sheet: TAL ---


,0,1,2,3,4,5,6,7,8
0,NaN,NaN,Portfolio of Kotak Transportation & Logistics ...,NaN,NaN,NaN,NaN,NaN,NaN
1,Name of Instrument,NaN,NaN,ISIN Code,Industry,Yield,Quantity,Market Value (Rs.in Lacs),% to Net Assets
2,Equity & Equity related,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,Listed/Awaiting listing on Stock Exchange,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,,Maruti Suzuki India Limited,INE585B01010,Automobiles,NaN,34300,4104.68,7.54
5,NaN,,Hero MotoCorp Ltd.,INE158A01026,Automobiles,NaN,72000,3773.52,6.93
6,NaN,,INDO-MIM LTD,INE084101034,Industrial Manufacturing,NaN,270000,3354.62,6.16
7,NaN,,Inter Globe Aviation Ltd,INE646L01027,Transport Services,NaN,61500,3065.16,5.63
8,NaN,,Mahindra & Mahindra Ltd.,INE101A01026,Automobiles,NaN,102800,3032.6,5.57
9,NaN,,Bajaj Auto Ltd.,INE917I01010,Automobiles,NaN,24300,2642.38,4.85



--- Sheet: STF ---


,0,1,2,3,4,5,6,7,8
0,NaN,NaN,Portfolio of Kotak BSE Sensex ETF as on 30-Sep...,NaN,NaN,NaN,NaN,NaN,NaN
1,Name of Instrument,NaN,NaN,ISIN Code,Industry,Yield,Quantity,Market Value (Rs.in Lacs),% to Net Assets
2,Equity & Equity related,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,Listed/Awaiting listing on Stock Exchange,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,,HDFC BANK LTD.,INE040A01034,Banks,NaN,40518,287.56,12.59
5,NaN,,ICICI BANK LTD.,INE090A01021,Banks,NaN,19058,252.04,11.03
6,NaN,,RELIANCE INDUSTRIES LTD.,INE002A01018,Petroleum Products,NaN,17610,209.12,9.15
7,NaN,,BHARTI AIRTEL LTD.,INE397D01024,Telecom - Services,NaN,8286,145.68,6.38
8,NaN,,Larsen and Toubro Ltd.,INE018A01030,Construction,NaN,3106,116.63,5.11
9,NaN,,STATE BANK OF INDIA.,INE062A01020,Banks,NaN,11031,105.97,4.64


In [181]:

import pandas as pd

for file in sorted(KOTAK_RAW_DIR.glob("*.xlsx")):
    xls = pd.ExcelFile(file)
    matches = []

    for sheet in xls.sheet_names:
        preview = pd.read_excel(
            file,
            sheet_name=sheet,
            header=None,
            nrows=2
        )

        title_text = " ".join(
            preview.iloc[0].dropna().astype(str).tolist()
        )

        if (
            "multi asset allocation fund" in title_text.lower()
            or "multi asset allocation" in sheet.lower()
        ):
            matches.append((sheet, title_text))

    print(f"\nFILE: {file.name}")
    print("MATCHES:", matches)



FILE: ConsolidatedSEBIPortfolioApril2026.xlsx
MATCHES: [('MAF', 'Portfolio of Kotak Multi Asset Allocation Fund as on 30-Apr-2026')]

FILE: ConsolidatedSEBIPortfolioAugust2026.xlsx
MATCHES: [('MAF', 'Portfolio of Kotak Multi Asset Allocation Fund as on 31-Aug-2026')]

FILE: ConsolidatedSEBIPortfolioDecember2025.xlsx
MATCHES: [('MAF', 'Portfolio of Kotak Multi Asset Allocation Fund as on 31-Dec-2025')]

FILE: ConsolidatedSEBIPortfolioFebruary2026.xlsx
MATCHES: [('MAF', 'Portfolio of Kotak Multi Asset Allocation Fund as on 28-Feb-2026')]

FILE: ConsolidatedSEBIPortfolioJanuary2026.xlsx
MATCHES: [('MAF', 'Portfolio of Kotak Multi Asset Allocation Fund as on 31-Jan-2026')]

FILE: ConsolidatedSEBIPortfolioJuly2026.xlsx
MATCHES: [('MAF', 'Portfolio of Kotak Multi Asset Allocation Fund as on 31-Jul-2026')]

FILE: ConsolidatedSEBIPortfolioJune2026.xlsx
MATCHES: [('MAF', 'Portfolio of Kotak Multi Asset Allocation Fund as on 30-Jun-2026')]

FILE: ConsolidatedSEBIPortfolioMarch2026.xlsx
MATCHES:

In [182]:

import pandas as pd

sample_file = KOTAK_RAW_DIR / "ConsolidatedSEBIPortfolioSeptember2026.xlsx"

maf_df = pd.read_excel(
    sample_file,
    sheet_name="MAF",
    header=None
)

print("Sheet shape:", maf_df.shape)

print("\nFirst 20 rows:")
display(maf_df.head(20))

print("\nLast 25 rows:")
display(maf_df.tail(25))


Sheet shape: (210, 9)

First 20 rows:


,0,1,2,3,4,5,6,7,8
0,NaN,NaN,Portfolio of Kotak Multi Asset Allocation Fund...,NaN,NaN,NaN,NaN,NaN,NaN
1,Name of Instrument,NaN,NaN,ISIN Code,Industry / Rating,Yield,Quantity,Market Value (Rs.in Lacs),% to Net Assets
2,Equity & Equity related,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,Listed/Awaiting listing on Stock Exchange,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,,STATE BANK OF INDIA.,INE062A01020,Banks,NaN,5190434,49802.21,3.27
5,NaN,,INDO-MIM LTD,INE084101034,Industrial Manufacturing,NaN,3571317,44371.83,2.91
6,NaN,,ITC Ltd.,INE154A01025,Diversified FMCG,NaN,14750000,38755.63,2.54
7,NaN,,NTPC LTD,INE733E01010,Power,NaN,11843500,38224.9,2.51
8,NaN,,ETERNAL LIMITED,INE758T01015,Retailing,NaN,11882040,38022.53,2.5
9,NaN,,Maruti Suzuki India Limited,INE585B01010,Automobiles,NaN,305000,36499.35,2.4



Last 25 rows:


,0,1,2,3,4,5,6,7,8
185,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
186,9,** Thinly traded/non-traded securities- Fair v...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
187,NaN,Limited in accordance with guidelines on valu...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
188,NaN,Exchange board of India and approved by the Tr...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
189,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
190,10,SO: Structured Obligations FRB: Floating Rate ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
191,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
192,Scheme,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
193,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
194,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
